# Pipeline — Desafio Quant AI 2026 · Itaú Asset

Notebook que orquestra os módulos de `src\` sobre os dados em `dados\`.
Regra de trabalho: uma célula por sessão — o humano roda, confere a sanidade,
e só então a próxima célula é escrita. Ver `CLAUDE.md` e `DOSSIE.md`.


In [1]:
import glob
import os
from collections import Counter

import numpy as np
import pandas as pd

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

DADOS_DIR = r"C:\Users\lucca\quant2026\dados"
OUT_DIR = r"C:\Users\lucca\quant2026\intermediario"
OUT_PATH = os.path.join(OUT_DIR, "a1_diario_universo.parquet")
os.makedirs(OUT_DIR, exist_ok=True)

COLUNAS = ["DATA_PREGAO", "CODBDI", "TPMERC", "CODNEG", "NOMRES", "ESPECI",
           "PREULT", "PREMED", "TOTNEG", "QUATOT", "VOLTOT", "FATCOT", "CODISI"]

ALLOWED_ROOTS = {"ON", "PN", "PNA", "PNB", "PNC", "PND", "PNE", "PNF", "PNG", "UNT"}

arquivos = sorted(glob.glob(os.path.join(DADOS_DIR, "cotahist_*.parquet")))
assert len(arquivos) == 32, f"esperado 32 arquivos cotahist, encontrado {len(arquivos)}"


def decidir_filtro(series, alvo_int, nome):
    dtype = series.dtype
    if pd.api.types.is_integer_dtype(dtype):
        valor = alvo_int
    elif pd.api.types.is_float_dtype(dtype):
        assert (series.dropna() % 1 == 0).all(), f"{nome}: dtype float com valores nao-inteiros"
        valor = float(alvo_int)
    else:
        candidatos = sorted({v for v in series.unique()
                              if str(v).strip().lstrip('0') == str(alvo_int)})
        assert len(candidatos) == 1, f"{nome}: candidatos ambiguos para {alvo_int}: {candidatos}"
        valor = candidatos[0]
    print(f"DECISAO: {nome} dtype medido = {dtype}; filtro escolhido = {valor!r} (tipo {type(valor).__name__})")
    return valor


codbdi_filtro = None
tpmerc_filtro = None

year_stats = []
especi_root_counts = Counter()
monthly_isin_counts = {}   # (ano, mes) -> nunique CODISI
filtered_chunks = []

print("Lendo 32 arquivos cotahist_*.parquet, um por vez, apenas as 13 colunas necessarias...\n")

for i, fp in enumerate(arquivos):
    ano = int(os.path.basename(fp).replace("cotahist_", "").replace(".parquet", ""))
    df = pd.read_parquet(fp, columns=COLUNAS)
    n_lidas = len(df)

    if i == 0:
        print("=" * 78)
        print(f"MEDICAO DE TIPOS -- primeiro arquivo lido: {os.path.basename(fp)}")
        print("=" * 78)
        print(f"\nCODBDI dtype: {df['CODBDI'].dtype}")
        print(df['CODBDI'].value_counts().head(10).to_string())
        print(f"\nTPMERC dtype: {df['TPMERC'].dtype}")
        print(df['TPMERC'].value_counts().head(10).to_string())
        print()
        codbdi_filtro = decidir_filtro(df['CODBDI'], 2, "CODBDI")
        tpmerc_filtro = decidir_filtro(df['TPMERC'], 10, "TPMERC")
        print("=" * 78 + "\n")

    # raiz de ESPECI ANTES do filtro (item d)
    especi_raiz_pre = df['ESPECI'].str.split().str[0].str.upper()
    vc = especi_raiz_pre.value_counts(dropna=False)
    for k, v in vc.items():
        especi_root_counts[k] += int(v)

    mask = (df['CODBDI'] == codbdi_filtro) & (df['TPMERC'] == tpmerc_filtro) & especi_raiz_pre.isin(ALLOWED_ROOTS)
    filtrado = df.loc[mask].reset_index(drop=True)
    n_filtradas = len(filtrado)

    codisi = filtrado['CODISI']
    is_na = codisi.isna()
    is_empty = codisi.notna() & (codisi.astype(str).str.strip() == '')
    pct_nan = is_na.mean() * 100 if n_filtradas else float('nan')
    pct_empty = is_empty.mean() * 100 if n_filtradas else float('nan')

    if n_filtradas:
        datas = pd.to_datetime(filtrado['DATA_PREGAO'].astype(str), format='%Y%m%d')
        meses = datas.dt.month
        contagem_mensal = filtrado.groupby(meses)['CODISI'].nunique()
        for m, c in contagem_mensal.items():
            monthly_isin_counts[(ano, int(m))] = int(c)
        isins_ano = filtrado['CODISI'].nunique()
        mediana_mensal = contagem_mensal.median()
        min_mensal = contagem_mensal.min()
        max_mensal = contagem_mensal.max()
    else:
        isins_ano = 0
        mediana_mensal = min_mensal = max_mensal = np.nan

    year_stats.append({
        'ano': ano,
        'linhas_lidas': n_lidas,
        'linhas_filtradas': n_filtradas,
        'pct_retida': 100 * n_filtradas / n_lidas if n_lidas else np.nan,
        'isins_distintos_ano': isins_ano,
        'mediana_mensal': mediana_mensal,
        'min_mensal': min_mensal,
        'max_mensal': max_mensal,
        'pct_codisi_nan': pct_nan,
        'pct_codisi_vazio': pct_empty,
    })

    filtered_chunks.append(filtrado)
    del df

stats_df = pd.DataFrame(year_stats)

df_final = pd.concat(filtered_chunks, ignore_index=True)
del filtered_chunks

print("\n" + "#" * 78)
print("# a) TABELA ANO A ANO -- linhas lidas / linhas apos filtro / % retida")
print("#" * 78)
tbl_a = stats_df[['ano', 'linhas_lidas', 'linhas_filtradas', 'pct_retida']].copy()
tbl_a['pct_retida'] = tbl_a['pct_retida'].round(2)
print(tbl_a.to_string(index=False))

print("\n" + "#" * 78)
print("# b) TABELA ANO A ANO -- ISINs distintos / mediana mensal / min-max mensal")
print("#" * 78)
tbl_b = stats_df[['ano', 'isins_distintos_ano', 'mediana_mensal', 'min_mensal', 'max_mensal']].copy()
print(tbl_b.to_string(index=False))

print("\n" + "#" * 78)
print("# c) % de NaN e % de string vazia em CODISI, por ano (sobre o filtrado)")
print("#" * 78)
tbl_c = stats_df[['ano', 'pct_codisi_nan', 'pct_codisi_vazio']].copy()
tbl_c['pct_codisi_nan'] = tbl_c['pct_codisi_nan'].round(4)
tbl_c['pct_codisi_vazio'] = tbl_c['pct_codisi_vazio'].round(4)
print(tbl_c.to_string(index=False))

print("\n" + "#" * 78)
print("# d) 20 raizes de ESPECI mais frequentes ANTES do filtro")
print("#" * 78)
top20 = pd.Series(especi_root_counts).sort_values(ascending=False).head(20)
print(top20.to_string())

print("\n" + "#" * 78)
print("# e) Estatistica descritiva de PREULT (dataset final filtrado)")
print("#" * 78)
preult = df_final['PREULT']
desc = {
    'dtype': str(preult.dtype),
    'min': preult.min(),
    'p1': preult.quantile(0.01),
    'p25': preult.quantile(0.25),
    'mediana': preult.quantile(0.50),
    'p75': preult.quantile(0.75),
    'p99': preult.quantile(0.99),
    'max': preult.max(),
}
for k, v in desc.items():
    print(f"  {k}: {v}")
n_preult_leq0 = int((preult <= 0).sum())
print(f"  linhas com PREULT <= 0: {n_preult_leq0}")

print("\n" + "#" * 78)
print("# f) SONDA DE ESCALA DE PRECO -- PETR4 e ITUB4, ultimo pregao de 2015 e 2020")
print("#" * 78)
sonda = df_final.loc[df_final['CODNEG'].isin(['PETR4', 'ITUB4']),
                      ['CODNEG', 'DATA_PREGAO', 'PREULT']].copy()
sonda['ANO'] = sonda['DATA_PREGAO'] // 10000
sonda = sonda[sonda['ANO'].isin([2015, 2020])]
idx = sonda.groupby(['CODNEG', 'ANO'])['DATA_PREGAO'].idxmax()
sonda_final = sonda.loc[idx].sort_values(['CODNEG', 'ANO'])
print(sonda_final.to_string(index=False))
print()
for row in sonda_final.itertuples():
    valor = row.PREULT
    if 1 <= valor <= 1000:
        interp = "plausivel como REAIS (preco de acao tipico em R$)"
    elif valor > 1000:
        interp = "valor implausivel como reais -- sugere CENTAVOS (escala x100) ou outro problema"
    else:
        interp = "valor implausivel (< R$1) -- verificar"
    print(f"  {row.CODNEG} {row.ANO} (pregao {row.DATA_PREGAO}): PREULT = {valor} -> {interp}")

print("\n" + "#" * 78)
print("# g) shape final / periodo / ISINs distintos / gravacao")
print("#" * 78)
print(f"shape final: {df_final.shape}")
print(f"DATA_PREGAO min: {df_final['DATA_PREGAO'].min()}  max: {df_final['DATA_PREGAO'].max()}")
print(f"ISINs (CODISI) distintos no total: {df_final['CODISI'].nunique()}")

df_final.to_parquet(OUT_PATH, index=False)
tamanho_mb = os.path.getsize(OUT_PATH) / (1024 * 1024)
print(f"parquet gravado em: {OUT_PATH}")
print(f"tamanho do parquet gravado: {tamanho_mb:.2f} MB")

print("\n" + "#" * 78)
print("# SANIDADE")
print("#" * 78)

# S1
s1_items = sorted([(k, v) for k, v in monthly_isin_counts.items() if 2000 <= k[0] <= 2020])
s1_serie = pd.Series({f"{a}-{m:02d}": v for (a, m), v in s1_items})
mediana_s1 = s1_serie.median()
print(f"S1 -- mediana de ISINs distintos por mes (2000-01 a 2020-12): {mediana_s1}")
if not (300 <= mediana_s1 <= 500):
    print("S1 FALHOU. Serie mensal completa 2000-01 a 2020-12:")
    print(s1_serie.to_string())
    raise AssertionError(f"S1: mediana mensal de ISINs = {mediana_s1}, fora do intervalo [300,500]")
print("S1 PASSOU.")

# S2
especi_raiz_final = df_final['ESPECI'].str.split().str[0].str.upper()
assert (df_final['CODBDI'] == codbdi_filtro).all(), "S2 FALHOU: existe linha com CODBDI != filtro"
assert (df_final['TPMERC'] == tpmerc_filtro).all(), "S2 FALHOU: existe linha com TPMERC != filtro"
assert especi_raiz_final.isin(ALLOWED_ROOTS).all(), "S2 FALHOU: existe linha com raiz de ESPECI fora da lista"
print("S2 PASSOU: todas as linhas retidas tem CODBDI, TPMERC e raiz de ESPECI validos (assert ok).")

# S3
anos_presentes = sorted(set((df_final['DATA_PREGAO'] // 10000).tolist()))
anos_esperados = list(range(1995, 2027))
faltando = [a for a in anos_esperados if a not in anos_presentes]
print(f"S3 -- anos presentes: {anos_presentes[0]} a {anos_presentes[-1]} ({len(anos_presentes)} anos)")
if faltando or anos_presentes != anos_esperados:
    print(f"S3 FALHOU. Anos faltando: {faltando}")
    raise AssertionError(f"S3: anos faltando no periodo final: {faltando}")
print("S3 PASSOU: 1995-2026 sem ano faltando.")

# S4
_nan_arr = stats_df['pct_codisi_nan'].to_numpy()
_vazio_arr = stats_df['pct_codisi_vazio'].to_numpy()
stats_df['pct_codisi_faltante'] = np.where(np.isnan(_nan_arr), 0, _nan_arr) + np.where(np.isnan(_vazio_arr), 0, _vazio_arr)
print("\nS4 -- % CODISI faltante (NaN + vazio) por ano:")
print(stats_df[['ano', 'pct_codisi_nan', 'pct_codisi_vazio', 'pct_codisi_faltante']].to_string(index=False))
falhas_s4 = stats_df[(stats_df['ano'] >= 1999) & (stats_df['pct_codisi_faltante'] > 5)]
if len(falhas_s4):
    print("S4 FALHOU nos seguintes anos (>=1999, >5% CODISI faltante):")
    print(falhas_s4.to_string(index=False))
    raise AssertionError("S4: CODISI vazio/NaN > 5% em ano >= 1999")
print("S4 PASSOU (anos < 1999 sao apenas reportados acima, nao bloqueantes).")


Lendo 32 arquivos cotahist_*.parquet, um por vez, apenas as 13 colunas necessarias...

MEDICAO DE TIPOS -- primeiro arquivo lido: cotahist_1995.parquet

CODBDI dtype: string
CODBDI
02    54990
96    38855
78     4058
62     3344
06     1304
14      829
82      778
10      370
38      115
22       57

TPMERC dtype: Int64
TPMERC
10    57184
20    39221
70     4058
30     3344
80      778
12      115
13       54
17       37

DECISAO: CODBDI dtype medido = string; filtro escolhido = '02' (tipo str)
DECISAO: TPMERC dtype medido = Int64; filtro escolhido = 10 (tipo int)


##############################################################################
# a) TABELA ANO A ANO -- linhas lidas / linhas apos filtro / % retida
##############################################################################
 ano  linhas_lidas  linhas_filtradas  pct_retida
1995        104791             49176       46.93
1996        110685             51410       46.45
1997        119272             50857       42.64
199

In [2]:
import glob
import os
import re
from collections import Counter

import numpy as np
import pandas as pd

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 200)

DADOS_DIR = r"C:\Users\lucca\quant2026\dados"
OUT_DIR = r"C:\Users\lucca\quant2026\intermediario"
OUT_PATH = os.path.join(OUT_DIR, "a1_diario_universo.parquet")
os.makedirs(OUT_DIR, exist_ok=True)

COLUNAS = ["DATA_PREGAO", "CODBDI", "TPMERC", "CODNEG", "NOMRES", "ESPECI",
           "PREULT", "PREMED", "TOTNEG", "QUATOT", "VOLTOT", "FATCOT", "CODISI"]

ALLOWED_ROOTS = {"ON", "PN", "PNA", "PNB", "PNC", "PND", "PNE", "PNF", "PNG", "UNT"}
TRAILING_NONALPHA = re.compile(r'[^A-Z]+$')

arquivos = sorted(glob.glob(os.path.join(DADOS_DIR, "cotahist_*.parquet")))
assert len(arquivos) == 32, f"esperado 32 arquivos cotahist, encontrado {len(arquivos)}"

CODBDI_F = '02'
TPMERC_F = 10

# valida (nao redecide) os tipos ja congelados em A1
amostra = pd.read_parquet(arquivos[0], columns=['CODBDI', 'TPMERC'])
assert str(amostra['CODBDI'].dtype) == 'string', f"CODBDI mudou de tipo: {amostra['CODBDI'].dtype}"
assert CODBDI_F in amostra['CODBDI'].unique(), "CODBDI_F '02' nao encontrado no primeiro arquivo"
assert pd.api.types.is_integer_dtype(amostra['TPMERC'].dtype), f"TPMERC mudou de tipo: {amostra['TPMERC'].dtype}"
assert TPMERC_F in amostra['TPMERC'].unique(), "TPMERC_F 10 nao encontrado no primeiro arquivo"
print(f"Filtros CODBDI/TPMERC confirmados inalterados desde A1: CODBDI={CODBDI_F!r} (string), TPMERC={TPMERC_F!r} (Int64)\n")
del amostra

year_stats_new = []
token_counts_base = Counter()   # raiz bruta -> nº de linhas, restrito a CODBDI/TPMERC ok
old_total = 0
new_total = 0
old_isin_global = set()
old_monthly_counts = {}   # (ano, mes) -> nunique CODISI, regra ANTIGA
new_monthly_counts = {}   # (ano, mes) -> nunique CODISI, regra NOVA
preult_leq0_records = []  # linhas com PREULT<=0 sob a regra ANTIGA (as mesmas de A1)
filtered_chunks_new = []

print("Relendo os 32 cotahist_*.parquet do zero (nao parte do parquet de A1)...\n")

for fp in arquivos:
    ano = int(os.path.basename(fp).replace("cotahist_", "").replace(".parquet", ""))
    df = pd.read_parquet(fp, columns=COLUNAS)

    raw_token = df['ESPECI'].str.split().str[0].str.upper()
    new_root = raw_token.str.replace(TRAILING_NONALPHA, '', regex=True)

    base_mask = (df['CODBDI'] == CODBDI_F) & (df['TPMERC'] == TPMERC_F)
    mask_old = base_mask & raw_token.isin(ALLOWED_ROOTS)
    mask_new = base_mask & new_root.isin(ALLOWED_ROOTS)

    vc_base = raw_token[base_mask].value_counts(dropna=False)
    for k, v in vc_base.items():
        token_counts_base[k] += int(v)

    old_total += int(mask_old.sum())
    new_total += int(mask_new.sum())

    # regra ANTIGA (para o antes/depois e para localizar as linhas PREULT<=0 de A1)
    old_subset = df.loc[mask_old, ['DATA_PREGAO', 'CODISI', 'CODNEG', 'PREULT']]
    if len(old_subset):
        old_isin_global.update(old_subset['CODISI'].unique().tolist())
        old_dates = pd.to_datetime(old_subset['DATA_PREGAO'].astype(str), format='%Y%m%d')
        old_month_counts = old_subset.groupby(old_dates.dt.month)['CODISI'].nunique()
        for m, c in old_month_counts.items():
            old_monthly_counts[(ano, int(m))] = int(c)
        leq0 = old_subset.loc[old_subset['PREULT'] <= 0]
        if len(leq0):
            preult_leq0_records.extend(leq0.to_dict('records'))

    # regra NOVA (o resultado que sera gravado)
    filtrado_novo = df.loc[mask_new].reset_index(drop=True)
    n_novo = len(filtrado_novo)
    if n_novo:
        new_dates = pd.to_datetime(filtrado_novo['DATA_PREGAO'].astype(str), format='%Y%m%d')
        new_month_counts = filtrado_novo.groupby(new_dates.dt.month)['CODISI'].nunique()
        for m, c in new_month_counts.items():
            new_monthly_counts[(ano, int(m))] = int(c)
        isins_ano = filtrado_novo['CODISI'].nunique()
        mediana_mensal = new_month_counts.median()
    else:
        isins_ano = 0
        mediana_mensal = np.nan

    year_stats_new.append({
        'ano': ano,
        'linhas_retidas': n_novo,
        'isins_distintos_ano': isins_ano,
        'mediana_mensal_isins': mediana_mensal,
    })

    filtered_chunks_new.append(filtrado_novo)
    del df

stats_new_df = pd.DataFrame(year_stats_new)
df_final = pd.concat(filtered_chunks_new, ignore_index=True)
del filtered_chunks_new

print(f"(checagem cruzada com A1: total sob a regra ANTIGA recalculada aqui do zero = {old_total} "
      f"-- deve bater com o shape gravado em A1, 2140298)\n")

print("#" * 78)
print("# a) TABELA DE MUDANCA DE STATUS -- raizes cujo status de inclusao mudou")
print("#" * 78)
linhas_a = []
for token, cnt in token_counts_base.items():
    if pd.isna(token):
        normalizado = token
        old_in = False
        new_in = False
    else:
        normalizado = TRAILING_NONALPHA.sub('', token)
        old_in = token in ALLOWED_ROOTS
        new_in = normalizado in ALLOWED_ROOTS
    if old_in != new_in:
        linhas_a.append({
            'raiz_bruta': token, 'raiz_normalizada': normalizado,
            'n_linhas': cnt, 'mudanca': 'ENTROU' if new_in else 'SAIU',
        })
tabela_a = pd.DataFrame(linhas_a).sort_values('n_linhas', ascending=False).reset_index(drop=True)
print(tabela_a.to_string(index=False))

entrou = tabela_a[tabela_a['mudanca'] == 'ENTROU']
if len(entrou):
    invalido = entrou[~entrou['raiz_normalizada'].isin(ALLOWED_ROOTS)]
    assert len(invalido) == 0, f"PAROU: raiz ENTROU que nao e uma classe valida: {invalido.to_dict('records')}"
    print(f"\nConfirmado: as {len(entrou)} raizes que ENTRARAM normalizam para classes validas "
          f"({sorted(entrou['raiz_normalizada'].unique())}) -- nenhuma classe estranha admitida.")
else:
    print("\nNenhuma raiz entrou no universo com a nova regra.")

print("\n" + "#" * 78)
print("# b) linhas retidas: A1 (regra antiga) vs A1b (regra nova)")
print("#" * 78)
delta_linhas = new_total - old_total
pct_delta = 100 * delta_linhas / old_total
print(f"A1  (regra antiga, recalculada aqui): {old_total}")
print(f"A1b (regra nova):                     {new_total}")
print(f"delta absoluto: {delta_linhas}  |  delta %: {pct_delta:.4f}%")

print("\n" + "#" * 78)
print("# c) mediana de ISINs distintos por mes, 2000-01 a 2020-12: antes vs depois")
print("#" * 78)
old_s1_items = sorted([(k, v) for k, v in old_monthly_counts.items() if 2000 <= k[0] <= 2020])
new_s1_items = sorted([(k, v) for k, v in new_monthly_counts.items() if 2000 <= k[0] <= 2020])
mediana_old = pd.Series({f"{a}-{m:02d}": v for (a, m), v in old_s1_items}).median()
mediana_new = pd.Series({f"{a}-{m:02d}": v for (a, m), v in new_s1_items}).median()
print(f"antes (A1):  {mediana_old}")
print(f"depois (A1b): {mediana_new}")

print("\n" + "#" * 78)
print("# d) ISINs distintos no total: antes vs depois")
print("#" * 78)
isins_new_total = df_final['CODISI'].nunique()
print(f"antes (A1):  {len(old_isin_global)}")
print(f"depois (A1b): {isins_new_total}")

print("\n" + "#" * 78)
print("# e) tabela ano a ano da nova versao (A1b)")
print("#" * 78)
print(stats_new_df.to_string(index=False))

print("\n" + "#" * 78)
print("# f) shape final / periodo / %NaN por coluna / tamanho do parquet")
print("#" * 78)
print(f"shape final: {df_final.shape}")
print(f"DATA_PREGAO min: {df_final['DATA_PREGAO'].min()}  max: {df_final['DATA_PREGAO'].max()}")
print("\n% de NaN por coluna:")
pct_nan_col = (df_final.isna().mean() * 100).round(4)
print(pct_nan_col.to_string())

df_final.to_parquet(OUT_PATH, index=False)
tamanho_mb = os.path.getsize(OUT_PATH) / (1024 * 1024)
print(f"\nparquet SOBRESCRITO em: {OUT_PATH}")
print(f"tamanho do parquet gravado: {tamanho_mb:.2f} MB")

print("\n" + "#" * 78)
print("# SANIDADE")
print("#" * 78)

# S1
print(f"S1 -- linhas: antes={old_total}, depois={new_total}, delta={delta_linhas}")
assert new_total > old_total, "S1 FALHOU: numero de linhas retidas nao aumentou"
print("S1 PASSOU: numero de linhas retidas aumentou.")

# S2
print(f"S2 -- mediana ISINs/mes 2000-2020 (nova regra): {mediana_new}")
if not (300 <= mediana_new <= 500):
    print("S2 FALHOU. Serie mensal completa (nova regra) 2000-01 a 2020-12:")
    print(pd.Series({f"{a}-{m:02d}": v for (a, m), v in new_s1_items}).to_string())
    raise AssertionError(f"S2: mediana mensal de ISINs (nova regra) = {mediana_new}, fora de [300,500]")
print("S2 PASSOU.")

# S3
raiz_final = df_final['ESPECI'].str.split().str[0].str.upper().str.replace(TRAILING_NONALPHA, '', regex=True)
assert raiz_final.isin(ALLOWED_ROOTS).all(), "S3 FALHOU: existe raiz normalizada fora da lista de inclusao"
print("S3 PASSOU: todas as raizes normalizadas no resultado final estao na lista de inclusao (assert ok).")

# S4
chave_final = set(zip(df_final['CODISI'], df_final['DATA_PREGAO'], df_final['CODNEG']))
faltando_s4 = [r for r in preult_leq0_records
               if (r['CODISI'], r['DATA_PREGAO'], r['CODNEG']) not in chave_final]
print(f"S4 -- {len(preult_leq0_records)} linha(s) com PREULT<=0 identificadas em A1 (regra antiga):")
for r in preult_leq0_records:
    print(f"   CODNEG={r['CODNEG']!r} CODISI={r['CODISI']!r} DATA_PREGAO={r['DATA_PREGAO']} PREULT={r['PREULT']}")
assert not faltando_s4, f"S4 FALHOU: linhas de A1 com PREULT<=0 sumiram em A1b: {faltando_s4}"
print("S4 PASSOU: as linhas com PREULT<=0 de A1 continuam presentes em A1b (nao foram descartadas).")


Filtros CODBDI/TPMERC confirmados inalterados desde A1: CODBDI='02' (string), TPMERC=10 (Int64)

Relendo os 32 cotahist_*.parquet do zero (nao parte do parquet de A1)...

(checagem cruzada com A1: total sob a regra ANTIGA recalculada aqui do zero = 2140298 -- deve bater com o shape gravado em A1, 2140298)

##############################################################################
# a) TABELA DE MUDANCA DE STATUS -- raizes cujo status de inclusao mudou
##############################################################################
raiz_bruta raiz_normalizada  n_linhas mudanca
      PNA*              PNA     29530  ENTROU
      PNB*              PNB     25514  ENTROU
      PNC*              PNC      1883  ENTROU
      PND*              PND       482  ENTROU
      PNE*              PNE       199  ENTROU
      PNG*              PNG        92  ENTROU
      UNT*              UNT        66  ENTROU
      PNF*              PNF        59  ENTROU
      PND(              PND        41  ENTROU
 

In [3]:
import inspect
import os
import sys

import numpy as np
import pandas as pd

pd.set_option('display.width', 220)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 200)

sys.path.insert(0, r"C:\Users\lucca\quant2026\src")
import dados  # noqa: E402  (config.py agora existe; import verificado abaixo)
import config  # noqa: E402

IN_PATH = r"C:\Users\lucca\quant2026\intermediario\a1_diario_universo.parquet"
OUT_EVENTOS = r"C:\Users\lucca\quant2026\intermediario\a2_eventos_societarios.parquet"
OUT_AJUSTADO = r"C:\Users\lucca\quant2026\intermediario\a2_diario_ajustado.parquet"

print("#" * 78)
print("# PASSO 0 -- assinaturas, contrato de colunas, verificacao do config.py")
print("#" * 78)

print(f"\nconfig.py em uso: DATA_INICIO={config.DATA_INICIO!r}, "
      f"DATA_CORTE={config.DATA_CORTE!r}, DIR_DESENHO={config.DIR_DESENHO}")

# verificacao por leitura de codigo: DATA_CORTE/DATA_INICIO/DIR_DESENHO nao sao
# usadas dentro de detectar_eventos_societarios nem aplicar_ajuste_societario
import ast
codigo = open(r"C:\Users\lucca\quant2026\src\dados.py", encoding="utf-8").read()
arvore = ast.parse(codigo)
alvo_funcs = {"detectar_eventos_societarios", "aplicar_ajuste_societario"}
constantes = {"DATA_CORTE", "DATA_INICIO", "DIR_DESENHO"}
achados = []
for no in ast.walk(arvore):
    if isinstance(no, ast.FunctionDef) and no.name in alvo_funcs:
        for sub in ast.walk(no):
            if isinstance(sub, ast.Name) and sub.id in constantes:
                achados.append((no.name, sub.id, sub.lineno))
print(f"\nReferencias a {sorted(constantes)} DENTRO do corpo de {sorted(alvo_funcs)}: "
      f"{achados if achados else 'NENHUMA (confirmado por AST)'}")
assert not achados, f"PARAR: constante de config usada dentro da funcao: {achados}"

print("\n--- assinatura: detectar_eventos_societarios ---")
print(inspect.signature(dados.detectar_eventos_societarios))
print(inspect.getdoc(dados.detectar_eventos_societarios))

print("\n--- assinatura: aplicar_ajuste_societario ---")
print(inspect.signature(dados.aplicar_ajuste_societario))
print(inspect.getdoc(dados.aplicar_ajuste_societario))

universo = pd.read_parquet(IN_PATH)
print(f"\nColunas exigidas por detectar_eventos_societarios: "
      f"{{'data', 'CODISI', 'PREULT', 'FATCOT', 'ESPECI'}}")
print(f"Colunas existentes no parquet de A1b: {list(universo.columns)}")
print("'data' NAO existe (o parquet tem DATA_PREGAO); as outras 4 batem pelo nome. "
      "Sera criada 'data' = to_datetime(DATA_PREGAO) so para alimentar as funcoes, "
      "e mantida junto de DATA_PREGAO na saida (decisao do humano).")

print("\n" + "#" * 78)
print("# VALIDACAO DA PREMISSA POSICIONAL de qualificador_especi() contra estes dados")
print("#" * 78)

especi = universo['ESPECI']
comprimentos = especi.str.len().value_counts().sort_index()
print("\nDistribuicao do comprimento de ESPECI (nº de caracteres):")
print(comprimentos.to_string())

print("\n40 valores distintos de ESPECI mais frequentes:")
print(especi.value_counts().head(40).to_string())

especi_ljust = especi.fillna("").astype(str).str.ljust(10)
pos4 = especi_ljust.str.slice(3, 4)  # posicao 4, 1-based == indice 3, 0-based
print("\nCaractere na posicao 4 (slot do marcador '*'), distribuicao:")
print(pos4.value_counts(dropna=False).to_string())
n_pos4_no_vazio = int((pos4 != "").sum())
print(f"\nlinhas com caractere nao-vazio na posicao 4: {n_pos4_no_vazio} "
      f"({100*n_pos4_no_vazio/len(especi):.2f}% do universo)")

qual = dados.qualificador_especi(especi)
print("\nDistribuicao do qualificador (posicoes 5-8) extraido por qualificador_especi():")
print(qual.value_counts(dropna=False).head(30).to_string())

print("\n" + "#" * 78)
print("# CHAMANDO detectar_eventos_societarios / aplicar_ajuste_societario")
print("#" * 78)

n_entrada = len(universo)
painel = universo.copy()
painel['data'] = pd.to_datetime(painel['DATA_PREGAO'].astype(str), format='%Y%m%d')

eventos = dados.detectar_eventos_societarios(painel)
print(f"\neventos detectados: {len(eventos)}")

print("\n" + "#" * 78)
print("# NORMALIZACAO DE CHAVE (fora de dados.py) -- pandas 3.0.5 exige dtypes identicos no merge_asof")
print("#" * 78)
print("Erro reproduzido numa tentativa anterior desta mesma sessao:")
print("  pandas.errors.MergeError: incompatible merge keys [0] "
      "<StringDtype(na_value=<NA>)> and <StringDtype(na_value=nan)>, must be the same type")
print("painel['CODISI'] vem do parquet (string[python], na_value=<NA>); eventos['CODISI'] "
      "sai de aplicar_ajuste_societario ao reconstruir o log via pd.DataFrame(lista_de_dicts, ...), "
      "e o pandas 3.0.5 infere um 'sabor' diferente de string dtype a partir de objetos Python puros. "
      "Defeito de compatibilidade da funcao sob esta versao do pandas -- src\\dados.py NAO foi alterado.")

print(f"\nANTES do cast: painel['CODISI'].dtype = {painel['CODISI'].dtype!r} ; "
      f"eventos['CODISI'].dtype = {eventos['CODISI'].dtype!r}")
painel['CODISI'] = painel['CODISI'].astype('string')
eventos['CODISI'] = eventos['CODISI'].astype('string')
print(f"DEPOIS do cast:  painel['CODISI'].dtype = {painel['CODISI'].dtype!r} ; "
      f"eventos['CODISI'].dtype = {eventos['CODISI'].dtype!r}")

print(f"\npainel['data'].dtype = {painel['data'].dtype!r} ; eventos['data'].dtype = {eventos['data'].dtype!r}")
if painel['data'].dtype != eventos['data'].dtype:
    print("Dtypes de 'data' divergem -- alinhando eventos['data'] ao dtype de painel['data'].")
    eventos['data'] = eventos['data'].astype(painel['data'].dtype)
    print(f"DEPOIS do alinhamento: eventos['data'].dtype = {eventos['data'].dtype!r}")
else:
    print("Dtypes de 'data' ja batem -- nenhum alinhamento necessario.")

print("\n--- VERIFICACAO ANTI-SILENCIO (cobertura de chave, antes do merge) ---")
codisi_eventos_distintos = eventos['CODISI'].dropna().unique()
n_codisi_eventos = len(codisi_eventos_distintos)
codisi_painel_set = set(painel['CODISI'].dropna().unique().tolist())
existem = [c for c in codisi_eventos_distintos if c in codisi_painel_set]
nao_existem = [c for c in codisi_eventos_distintos if c not in codisi_painel_set]
print(f"nº de CODISI distintos em eventos: {n_codisi_eventos}")
print(f"quantos desses existem em painel: {len(existem)}")
print(f"quantos NAO existem em painel: {len(nao_existem)}")
if nao_existem:
    print(f"lista dos que NAO existem: {nao_existem}")
print(f"NaN/<NA> em painel['CODISI']: {int(painel['CODISI'].isna().sum())}")
print(f"NaN/<NA> em eventos['CODISI']: {int(eventos['CODISI'].isna().sum())}")
assert not nao_existem, f"PARAR: eventos com CODISI fora do painel: {nao_existem}"
print("Cobertura de chave OK -- todo CODISI de eventos existe no painel.")

ajustado = dados.aplicar_ajuste_societario(painel, eventos)
n_saida = len(ajustado)
print(f"\nlinhas entrada: {n_entrada}  |  linhas saida: {n_saida}")
assert n_saida == n_entrada, (
    f"PARAR: merge_asof alterou o numero de linhas ({n_entrada} -> {n_saida}) -- "
    f"sinal classico de chave duplicada inflando o painel."
)
print("nº de linhas conservado apos o merge (nenhuma duplicacao).")

eventos.to_parquet(OUT_EVENTOS, index=False)
ajustado.to_parquet(OUT_AJUSTADO, index=False)
print(f"\ngravado: {OUT_EVENTOS}")
print(f"gravado: {OUT_AJUSTADO}")

print("\n" + "#" * 78)
print("# b) contagem de eventos por CANAL (ganha_G, ganha_B, fatcot, extensao_defasagem_1) e por ano")
print("#" * 78)
eventos_expl = eventos.assign(ano=eventos['data'].dt.year, canal_tok=eventos['canal'].str.split('+'))
eventos_expl = eventos_expl.explode('canal_tok')
tab_b = eventos_expl.groupby(['ano', 'canal_tok']).size().unstack(fill_value=0)
print(tab_b.to_string())
print(f"\ntotal de linhas de evento (antes de explodir combos): {len(eventos)}")
print("\ncontagem por combinacao literal de canal (como o modulo produz, sem explodir):")
print(eventos['canal'].value_counts().to_string())

print("\n" + "#" * 78)
print("# c) ISINs afetados e mediana de eventos por ISIN")
print("#" * 78)
por_isin = eventos.groupby('CODISI').size()
n_isins_afetados = por_isin.shape[0]
n_isins_universo = universo['CODISI'].nunique()
print(f"ISINs distintos no universo: {n_isins_universo}")
print(f"ISINs distintos com >=1 evento: {n_isins_afetados}")
print(f"mediana de eventos por ISIN (entre os {n_isins_afetados} afetados): {por_isin.median()}")
print(f"min/max de eventos por ISIN afetado: {por_isin.min()} / {por_isin.max()}")

print("\n" + "#" * 78)
print("# item (4) da nova instrucao -- medicao (NAO aplicada) dos marcadores de PROVENTO D/J/R/S")
print("#" * 78)
qual_universo = dados.qualificador_especi(universo['ESPECI'])
tem_provento = qual_universo.str.contains(r'[DJRS]', regex=True, na=False)
ano_universo = universo['DATA_PREGAO'] // 10000
provento_por_ano = universo.loc[tem_provento].assign(ano=ano_universo[tem_provento]).groupby('ano').size()
print("linhas por ano com marcador de provento (D/J/R/S) em ESPECI -- so medicao, nao ajustado aqui:")
print(provento_por_ano.to_string())
total_provento = int(tem_provento.sum())
print(f"\ntotal de linhas com marcador de provento no universo: {total_provento} "
      f"({100*total_provento/len(universo):.3f}%)")

print("\n" + "#" * 78)
print("# d) DIAGNOSTICO DO EFEITO -- retorno diario antes (SEM_AJUSTE) vs depois (ajustado)")
print("#" * 78)
aj = ajustado.sort_values(['CODISI', 'data'])
ret_antes = aj.groupby('CODISI')['PREULT_SEM_AJUSTE'].pct_change()
ret_depois = aj.groupby('CODISI')['PREULT'].pct_change()

linhas_d = []
for lim in (0.20, 0.50, 1.00):
    linhas_d.append({
        'limite': f'>{int(lim*100)}%',
        'antes': int((ret_antes.abs() > lim).sum()),
        'depois': int((ret_depois.abs() > lim).sum()),
    })
tab_d = pd.DataFrame(linhas_d)
tab_d['delta'] = tab_d['depois'] - tab_d['antes']
print(tab_d.to_string(index=False))

print("\n" + "#" * 78)
print("# e) AMOSTRA AUDITAVEL -- 5 eventos de maior fator de ajuste (|log(fator)| maximo)")
print("#" * 78)
ev_validos = eventos.dropna(subset=['fator']).copy()
ev_validos = ev_validos[ev_validos['fator'] > 0]
ev_validos['log_fator'] = np.log(ev_validos['fator'])
top5 = ev_validos.reindex(ev_validos['log_fator'].abs().sort_values(ascending=False).index).head(5)

aj_idx = aj.set_index(['CODISI', 'data']).sort_index()
for _, ev in top5.iterrows():
    codisi, data_evento, fator, canal = ev['CODISI'], ev['data'], ev['fator'], ev['canal']
    sub = aj[aj['CODISI'] == codisi].sort_values('data').reset_index(drop=True)
    pos = sub.index[sub['data'] == data_evento]
    if len(pos) == 0:
        print(f"\nCODISI={codisi} data={data_evento.date()} fator={fator:.4f} canal={canal} "
              f"-- data do evento nao encontrada no painel ajustado (extensao de defasagem sem base)")
        continue
    i = pos[0]
    janela = sub.iloc[max(0, i - 3): i + 4].copy()
    janela['ret_antes'] = janela['PREULT_SEM_AJUSTE'].pct_change()
    janela['ret_depois'] = janela['PREULT'].pct_change()
    print(f"\n=== CODISI={codisi} data_evento={data_evento.date()} fator={fator:.4f} canal={canal} ===")
    print(janela[['CODNEG', 'DATA_PREGAO', 'ESPECI', 'FATCOT', 'PREULT_SEM_AJUSTE',
                  'PREULT', 'ret_antes', 'ret_depois']].to_string(index=False))

print("\n" + "#" * 78)
print("# f) EVENTOS NAO EXPLICADOS -- 15 maiores |retorno diario| que RESTARAM apos o ajuste")
print("#" * 78)
aj2 = aj.copy()
aj2['ret_depois'] = ret_depois.to_numpy()
top15 = aj2.reindex(aj2['ret_depois'].abs().sort_values(ascending=False).index).head(15)
print(top15[['CODNEG', 'DATA_PREGAO', 'ESPECI', 'FATCOT', 'PREULT_SEM_AJUSTE',
             'PREULT', 'ret_depois']].to_string(index=False))

print("\n" + "#" * 78)
print("# g) shape final / %NaN por coluna / periodo / ISINs / tamanho dos parquets")
print("#" * 78)
print(f"shape a2_diario_ajustado: {ajustado.shape}")
print(f"periodo: DATA_PREGAO min={ajustado['DATA_PREGAO'].min()} max={ajustado['DATA_PREGAO'].max()}")
print(f"ISINs distintos: {ajustado['CODISI'].nunique()}")
print("\n% de NaN por coluna:")
print((ajustado.isna().mean() * 100).round(4).to_string())

tam_eventos_mb = os.path.getsize(OUT_EVENTOS) / (1024 * 1024)
tam_ajustado_mb = os.path.getsize(OUT_AJUSTADO) / (1024 * 1024)
print(f"\ntamanho a2_eventos_societarios.parquet: {tam_eventos_mb:.2f} MB")
print(f"tamanho a2_diario_ajustado.parquet: {tam_ajustado_mb:.2f} MB")

print("\n" + "#" * 78)
print("# SANIDADE")
print("#" * 78)

# S1
print(f"S1 -- linhas entrada={n_entrada}, linhas saida={n_saida}")
assert n_saida == n_entrada == 2198212, f"S1 FALHOU: {n_saida} != {n_entrada}"
print("S1 PASSOU: numero de linhas conservado.")

# S2
antes_50 = int((ret_antes.abs() > 0.50).sum())
depois_50 = int((ret_depois.abs() > 0.50).sum())
print(f"S2 -- |ret|>50%: antes={antes_50}, depois={depois_50}")
assert depois_50 < antes_50, "S2 FALHOU: |ret|>50% nao caiu apos o ajuste"
print("S2 PASSOU: |ret|>50% caiu apos o ajuste.")

# S3
mask_sem_ajuste_valido = ajustado['PREULT_SEM_AJUSTE'].notna()
n_novos_nan = int((ajustado.loc[mask_sem_ajuste_valido, 'PREULT'].isna()).sum())
print(f"S3 -- PREULT ajustado NaN onde PREULT_SEM_AJUSTE nao era NaN: {n_novos_nan}")
assert n_novos_nan == 0, "S3 FALHOU: preco ajustado NaN onde o bruto nao era NaN"
print("S3 PASSOU.")

# S4
alvo_s4 = [
    ('TLVT3B', 'BRTLVTACNOR8', 20050128),
    ('SVXE3B', 'BRSVXEACNOR7', 20050426),
    ('BGIP4', 'BRBGIPACNPR8', 20210518),
]
chave_saida = set(zip(ajustado['CODNEG'], ajustado['CODISI'], ajustado['DATA_PREGAO']))
faltando_s4 = [t for t in alvo_s4 if t not in chave_saida]
print(f"S4 -- linhas PREULT<=0 de A1/A1b esperadas presentes: {alvo_s4}")
assert not faltando_s4, f"S4 FALHOU: sumiram {faltando_s4}"
linhas_s4 = ajustado[ajustado['CODNEG'].isin([t[0] for t in alvo_s4]) &
                      ajustado['DATA_PREGAO'].isin([t[2] for t in alvo_s4])]
print(linhas_s4[['CODNEG', 'CODISI', 'DATA_PREGAO', 'PREULT_SEM_AJUSTE', 'PREULT']].to_string(index=False))
print("S4 PASSOU: as 3 linhas continuam presentes e nao foram tratadas aqui.")

# S5
ev_pos = eventos.dropna(subset=['fator'])
ev_pos = ev_pos[ev_pos['fator'] > 0].sort_values(['CODISI', 'data'])
fator_acum = ev_pos.groupby('CODISI')['fator'].cumprod()
n_nao_positivo = int((fator_acum <= 0).sum())
print(f"S5 -- fatores acumulados <=0: {n_nao_positivo} (min observado: {fator_acum.min() if len(fator_acum) else float('nan')})")
assert n_nao_positivo == 0, "S5 FALHOU: existe fator acumulado <=0"
print("S5 PASSOU: nenhum ISIN com fator acumulado <=0.")


##############################################################################
# PASSO 0 -- assinaturas, contrato de colunas, verificacao do config.py
##############################################################################

config.py em uso: DATA_INICIO='1995-01-01', DATA_CORTE='2026-08-07', DIR_DESENHO=C:\Users\lucca\quant2026\dados

Referencias a ['DATA_CORTE', 'DATA_INICIO', 'DIR_DESENHO'] DENTRO do corpo de ['aplicar_ajuste_societario', 'detectar_eventos_societarios']: NENHUMA (confirmado por AST)

--- assinatura: detectar_eventos_societarios ---
(painel)
Detecta os dias de evento societário e devolve o LOG, sem alterar nada.

O QUE FAZ: compara cada pregão de cada ISIN com o pregão OBSERVADO
anterior do mesmo ISIN, dentro da janela carregada, e marca os dias das
regras (a)-(d) do cabeçalho da seção. Para cada dia marcado calcula o
fator = `PREULT_dia / PREULT_anterior` — a razão observada, que é o que
`aplicar_ajuste_societario()` vai neutralizar.

O QUE NÃO FAZ: não olha m

In [4]:
import os

import numpy as np
import pandas as pd

pd.set_option('display.width', 220)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 200)

IN_PATH = r"C:\Users\lucca\quant2026\intermediario\a2_diario_ajustado.parquet"
OUT_PATH = r"C:\Users\lucca\quant2026\intermediario\a3_grade_mensal.parquet"
CLASSIF_PATH = r"C:\Users\lucca\quant2026\dados\ClassifSetorial (1).xlsx"

COLUNAS = ['data', 'DATA_PREGAO', 'CODNEG', 'NOMRES', 'ESPECI', 'CODISI',
           'PREULT', 'PREULT_SEM_AJUSTE', 'FATCOT', 'TOTNEG', 'QUATOT', 'VOLTOT']

df = pd.read_parquet(IN_PATH, columns=COLUNAS)
n_entrada = len(df)
print(f"entrada: {df.shape}")

# --- passo 1: mes ---------------------------------------------------------
df['mes'] = df['data'].dt.to_period('M').astype(str)

# --- passo 2/3: ultima sessao negociada do mes + somas mensais ------------
df_sorted = df.sort_values(['CODISI', 'mes', 'data'], kind='mergesort')
ultima_sessao = df_sorted.drop_duplicates(subset=['CODISI', 'mes'], keep='last')[
    ['CODISI', 'mes', 'data', 'PREULT', 'PREULT_SEM_AJUSTE', 'FATCOT', 'CODNEG', 'ESPECI', 'NOMRES']
]

somas = df.groupby(['CODISI', 'mes'], sort=False)[['VOLTOT', 'QUATOT', 'TOTNEG']].sum()
n_sessoes = df.groupby(['CODISI', 'mes'], sort=False).size().rename('n_sessoes')

grade = ultima_sessao.merge(somas, on=['CODISI', 'mes'], how='left') \
                      .merge(n_sessoes, on=['CODISI', 'mes'], how='left')
grade = grade.sort_values(['mes', 'CODISI']).reset_index(drop=True)

print(f"grade construida: {grade.shape}")

grade.to_parquet(OUT_PATH, index=False)
print(f"gravado: {OUT_PATH}")

# ============================================================================
print("\n" + "#" * 78)
print("# a) shape final / meses / ISINs / periodo / %NaN por coluna")
print("#" * 78)
print(f"shape: {grade.shape}")
print(f"meses distintos: {grade['mes'].nunique()}")
print(f"ISINs distintos: {grade['CODISI'].nunique()}")
print(f"periodo (mes): {grade['mes'].min()} -> {grade['mes'].max()}")
print(f"periodo (data efetiva): {grade['data'].min()} -> {grade['data'].max()}")
print("\n% de NaN por coluna:")
print((grade.isna().mean() * 100).round(4).to_string())

# ============================================================================
print("\n" + "#" * 78)
print("# b) serie mensal do no de ISINs na grade")
print("#" * 78)
n_isins_mes = grade.groupby('mes').size().rename('n_isins')
ano_da_mes = n_isins_mes.index.str[:4].astype(int)
mediana_por_ano = n_isins_mes.groupby(ano_da_mes).median()
print("mediana de ISINs por ano:")
print(mediana_por_ano.to_string())
print("\n10 meses com MENOS ISINs:")
print(n_isins_mes.nsmallest(10).to_string())
print("\n10 meses com MAIS ISINs:")
print(n_isins_mes.nlargest(10).to_string())

# ============================================================================
print("\n" + "#" * 78)
print("# c) distribuicao do DIA DO MES da sessao selecionada")
print("#" * 78)
dia_mes = grade['data'].dt.day
print(dia_mes.value_counts().sort_index().to_string())

periodos = pd.PeriodIndex(grade['mes'], freq='M')
ultimo_dia_calendario = periodos.to_timestamp(how='end').normalize()
gap_fim_mes = (ultimo_dia_calendario - grade['data'].dt.normalize()).dt.days
n_gap_gt10 = int((gap_fim_mes > 10).sum())
print(f"\ncasos com data escolhida a mais de 10 dias corridos do fim do mes: {n_gap_gt10}")

tmp = grade.assign(gap_fim_mes=gap_fim_mes)
top20_gap = tmp.reindex(tmp['gap_fim_mes'].sort_values(ascending=False).index).head(20)
print("\n20 casos mais extremos (maior gap ate o fim do mes):")
print(top20_gap[['CODNEG', 'mes', 'data', 'n_sessoes', 'gap_fim_mes']].to_string(index=False))

# ============================================================================
print("\n" + "#" * 78)
print("# d) MEDICAO DE BURACOS por ISIN")
print("#" * 78)
ordinais = periodos.asi8
tmp2 = pd.DataFrame({'CODISI': grade['CODISI'].to_numpy(), 'ordinal': ordinais, 'mes': grade['mes'].to_numpy()})
por_isin = tmp2.groupby('CODISI').agg(
    primeiro_ord=('ordinal', 'min'),
    ultimo_ord=('ordinal', 'max'),
    presentes=('ordinal', 'size'),
)
por_isin['span'] = por_isin['ultimo_ord'] - por_isin['primeiro_ord'] + 1
por_isin['buracos'] = por_isin['span'] - por_isin['presentes']

# mes de primeiro/ultimo em formato legivel, e CODNEG representativo (o mais recente)
mes_por_ordinal = dict(zip(ordinais, grade['mes']))
por_isin['primeiro_mes'] = por_isin['primeiro_ord'].map(mes_por_ordinal)
por_isin['ultimo_mes'] = por_isin['ultimo_ord'].map(mes_por_ordinal)
codneg_recente = grade.sort_values(['CODISI', 'mes']).groupby('CODISI')['CODNEG'].last()
por_isin['CODNEG'] = codneg_recente

n_com_buraco = int((por_isin['buracos'] > 0).sum())
print(f"ISINs com pelo menos um buraco interno: {n_com_buraco} de {len(por_isin)}")
print("\nDistribuicao do no de buracos por ISIN (entre os que tem >0):")
print(por_isin.loc[por_isin['buracos'] > 0, 'buracos'].describe().to_string())
print("\ndistribuicao completa (value_counts, truncada a 30 primeiras contagens):")
print(por_isin['buracos'].value_counts().sort_index().head(30).to_string())

top20_buracos = por_isin.sort_values('buracos', ascending=False).head(20)
print("\n20 ISINs com mais buracos:")
print(top20_buracos[['CODNEG', 'primeiro_mes', 'ultimo_mes', 'span', 'presentes', 'buracos']].to_string())

# ============================================================================
print("\n" + "#" * 78)
print("# e) distribuicao de n_sessoes")
print("#" * 78)
ns = grade['n_sessoes']
desc = {
    'min': ns.min(), 'p1': ns.quantile(0.01), 'p10': ns.quantile(0.10),
    'mediana': ns.quantile(0.50), 'p90': ns.quantile(0.90), 'max': ns.max(),
}
for k, v in desc.items():
    print(f"  {k}: {v}")
print(f"\nCODISI-mes com n_sessoes==1: {int((ns == 1).sum())}")
print(f"CODISI-mes com n_sessoes<=3: {int((ns <= 3).sum())}")
print(f"CODISI-mes com n_sessoes<=5: {int((ns <= 5).sum())}")

# ============================================================================
print("\n" + "#" * 78)
print("# f) SINALIZACAO DE VALORES NAO FINITOS em PREULT")
print("#" * 78)
mask_nf = (grade['PREULT'] <= 0) | grade['PREULT'].isna() | np.isinf(grade['PREULT'])
n_nf = int(mask_nf.sum())
print(f"linhas com PREULT<=0, NaN ou infinito: {n_nf}")
print(grade.loc[mask_nf, ['CODNEG', 'CODISI', 'mes', 'data', 'PREULT', 'PREULT_SEM_AJUSTE', 'n_sessoes']]
      .sort_values(['CODISI', 'mes']).to_string(index=False))

# ============================================================================
print("\n" + "#" * 78)
print("# g) sonda de ponte setorial (CODISI[2:6] vs ClassifSetorial CODIGO)")
print("#" * 78)
classif_raw = pd.read_excel(CLASSIF_PATH, header=None)
codigos_classif = classif_raw.iloc[2:, 5].dropna().astype(str).str.strip()
codigos_classif = codigos_classif[codigos_classif != ""]
set_codigos = set(codigos_classif.unique().tolist())
print(f"CODIGOs distintos no ClassifSetorial: {len(set_codigos)}")

bridge = grade['CODISI'].str.slice(2, 6)
codisi_unicos = grade[['CODISI']].assign(bridge=bridge).drop_duplicates('CODISI')
tem_bridge = codisi_unicos['bridge'].isin(set_codigos)
print(f"CODISI distintos na grade: {len(codisi_unicos)}")
print(f"CODISI distintos com CODISI[2:6] presente no ClassifSetorial: {int(tem_bridge.sum())} "
      f"({100 * tem_bridge.sum() / len(codisi_unicos):.2f}%)")

ano_grade = grade['mes'].str[:4].astype(int)
decada_grade = (ano_grade // 10) * 10
pares = pd.DataFrame({'CODISI': grade['CODISI'], 'decada': decada_grade}).drop_duplicates()
pares['bridge'] = pares['CODISI'].str.slice(2, 6)
pares['tem_bridge'] = pares['bridge'].isin(set_codigos)
por_decada = pares.groupby('decada').agg(codisi_distintos=('CODISI', 'nunique'),
                                          com_bridge=('tem_bridge', 'sum'))
por_decada['pct'] = (100 * por_decada['com_bridge'] / por_decada['codisi_distintos']).round(2)
print("\npor decada (um CODISI conta em toda decada em que aparece):")
print(por_decada.to_string())

# ============================================================================
print("\n" + "#" * 78)
print("# SANIDADE")
print("#" * 78)

# S1
n_pares = len(grade)
n_pares_unicos = grade.drop_duplicates(subset=['CODISI', 'mes']).shape[0]
print(f"S1 -- linhas: {n_pares}, pares (CODISI,mes) unicos: {n_pares_unicos}")
assert n_pares == n_pares_unicos, "S1 FALHOU: par (CODISI,mes) duplicado"
print("S1 PASSOU: nenhum par (CODISI,mes) duplicado.")

# S2
anos_mes = n_isins_mes.index.str[:4].astype(int)
n_isins_2000_2020 = n_isins_mes[(anos_mes >= 2000) & (anos_mes <= 2020)]
mediana_s2 = n_isins_2000_2020.median()
print(f"S2 -- mediana de ISINs/mes 2000-01 a 2020-12: {mediana_s2}")
assert 300 <= mediana_s2 <= 500, f"S2 FALHOU: mediana {mediana_s2} fora de [300,500]"
print("S2 PASSOU.")

# S3 / S5
mes_da_data = grade['data'].dt.to_period('M').astype(str)
coerente = (mes_da_data == grade['mes'])
n_incoerente = int((~coerente).sum())
print(f"S3/S5 -- linhas com data fora do proprio mes: {n_incoerente}")
assert n_incoerente == 0, "S3/S5 FALHARAM: existe linha com data fora do seu mes"
print("S3 PASSOU: toda data pertence ao seu mes.")
print("S5 PASSOU: data.dt.to_period('M') == mes em 100% das linhas.")

# S4
soma_sessoes = int(grade['n_sessoes'].sum())
print(f"S4 -- soma de n_sessoes: {soma_sessoes}, linhas de entrada: {n_entrada}")
assert soma_sessoes == n_entrada, f"S4 FALHOU: {soma_sessoes} != {n_entrada}"
print("S4 PASSOU: soma de n_sessoes bate com o total de linhas de entrada.")


entrada: (2198212, 12)
grade construida: (158358, 13)
gravado: C:\Users\lucca\quant2026\intermediario\a3_grade_mensal.parquet

##############################################################################
# a) shape final / meses / ISINs / periodo / %NaN por coluna
##############################################################################
shape: (158358, 13)
meses distintos: 380
ISINs distintos: 2189
periodo (mes): 1995-01 -> 2026-08
periodo (data efetiva): 1995-01-02 00:00:00 -> 2026-08-07 00:00:00

% de NaN por coluna:
CODISI               0.0
mes                  0.0
data                 0.0
PREULT               0.0
PREULT_SEM_AJUSTE    0.0
FATCOT               0.0
CODNEG               0.0
ESPECI               0.0
NOMRES               0.0
VOLTOT               0.0
QUATOT               0.0
TOTNEG               0.0
n_sessoes            0.0

##############################################################################
# b) serie mensal do no de ISINs na grade
#####################

In [5]:
import numpy as np
import pandas as pd

pd.set_option('display.width', 220)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 200)

IN_PATH = r"C:\Users\lucca\quant2026\intermediario\a3_grade_mensal.parquet"
OUT_PATH = r"C:\Users\lucca\quant2026\intermediario\a4_retorno_mensal.parquet"

df = pd.read_parquet(IN_PATH)
n_entrada = len(df)
print(f"entrada: {df.shape}")

# --- passo 1: ordenar ------------------------------------------------------
df = df.sort_values(['CODISI', 'mes'], kind='mergesort').reset_index(drop=True)

periodos = pd.PeriodIndex(df['mes'], freq='M')
df['_ord'] = periodos.asi8

g = df.groupby('CODISI', sort=False)
prev_ord = g['_ord'].shift(1)
prev_preult = g['PREULT'].shift(1)
prev_preult_bruto = g['PREULT_SEM_AJUSTE'].shift(1)
prev_n_sessoes = g['n_sessoes'].shift(1)
prev_especi = g['ESPECI'].shift(1)
prev_fatcot = g['FATCOT'].shift(1)

df['gap_meses'] = df['_ord'] - prev_ord
df['primeira_obs'] = prev_ord.isna()

# --- passo 4: blindagem de divisao -----------------------------------------
n_denom_invalido_bruto = int(((prev_preult <= 0) & prev_ord.notna()).sum())
print(f"\nlinhas que teriam denominador PREULT (ajustado) <= 0: {n_denom_invalido_bruto}")
df['denominador_invalido'] = (prev_preult <= 0) & prev_ord.notna()

# --- passo 2: ret_1m ---------------------------------------------------------
ret_1m = df['PREULT'] / prev_preult - 1.0
mask_gap1 = (df['gap_meses'] == 1)
ret_1m = ret_1m.where(mask_gap1)
ret_1m = ret_1m.where(~df['denominador_invalido'])
df['ret_1m'] = ret_1m

# --- passo 3: colunas de apoio ----------------------------------------------
df['ret_valido'] = mask_gap1.fillna(False) & np.isfinite(df['ret_1m'])
df['preco_iliquido'] = df['n_sessoes'] <= 3

df = df.drop(columns=['_ord'])

n_saida = len(df)
print(f"saida: {df.shape}")
df.to_parquet(OUT_PATH, index=False)
print(f"gravado: {OUT_PATH}")

# ============================================================================
print("\n" + "#" * 78)
print("# a) shape final / colunas / %NaN por coluna / periodo")
print("#" * 78)
print(f"shape: {df.shape}")
print(f"colunas: {list(df.columns)}")
print("\n% de NaN por coluna:")
print((df.isna().mean() * 100).round(4).to_string())
print(f"\nperiodo (mes): {df['mes'].min()} -> {df['mes'].max()}")

# ============================================================================
print("\n" + "#" * 78)
print("# b) contagem -- fechamento da conta")
print("#" * 78)
n_total = len(df)
n_validos = int(df['ret_valido'].sum())
n_primeira = int(df['primeira_obs'].sum())
n_gap_maior1 = int(((df['gap_meses'] > 1) & df['gap_meses'].notna()).sum())
n_denom_inv = int(df['denominador_invalido'].sum())
soma = n_validos + n_primeira + n_gap_maior1 + n_denom_inv
print(f"linhas totais:                 {n_total}")
print(f"ret_1m valido:                 {n_validos}")
print(f"NaN por primeira observacao:   {n_primeira}")
print(f"NaN por gap > 1 mes:           {n_gap_maior1}")
print(f"NaN por denominador invalido:  {n_denom_inv}")
print(f"soma das 4 categorias:         {soma}  (bate com total? {soma == n_total})")
assert soma == n_total, f"PAROU: {soma} != {n_total} -- categorias nao sao uma particao exata"

# ============================================================================
print("\n" + "#" * 78)
print("# c) distribuicao de gap_meses (nao-nulo)")
print("#" * 78)
gap_validos = df.loc[df['gap_meses'].notna(), 'gap_meses'].astype(int)
bucket = gap_validos.where(gap_validos <= 12, 13)
contagem = bucket.value_counts().sort_index()
contagem.index = [str(i) if i < 13 else '13+' for i in contagem.index]
print(contagem.to_string())
print(f"\n(NaN de gap_meses, ou seja primeira observacao do ISIN: {n_primeira})")

# ============================================================================
print("\n" + "#" * 78)
print("# d) estatistica de ret_1m valido")
print("#" * 78)
rv = df.loc[df['ret_valido'], 'ret_1m']
desc = {
    'n': len(rv), 'media': rv.mean(), 'dp': rv.std(), 'min': rv.min(),
    'p1': rv.quantile(0.01), 'p5': rv.quantile(0.05), 'p25': rv.quantile(0.25),
    'mediana': rv.quantile(0.50), 'p75': rv.quantile(0.75), 'p95': rv.quantile(0.95),
    'p99': rv.quantile(0.99), 'max': rv.max(),
    'assimetria': rv.skew(), 'curtose': rv.kurtosis(),
}
for k, v in desc.items():
    print(f"  {k}: {v}")
for lim in (0.20, 0.50, 1.00, 2.00):
    print(f"  |ret|>{int(lim*100)}%: {int((rv.abs() > lim).sum())}")

# ============================================================================
print("\n" + "#" * 78)
print("# e) 30 maiores |ret_1m| que restaram")
print("#" * 78)
tmp = df.loc[df['ret_valido']].copy()
tmp['PREULT_anterior'] = prev_preult.loc[tmp.index]
tmp['n_sessoes_anterior'] = prev_n_sessoes.loc[tmp.index]
tmp['ESPECI_anterior'] = prev_especi.loc[tmp.index]
tmp['FATCOT_anterior'] = prev_fatcot.loc[tmp.index]
top30 = tmp.reindex(tmp['ret_1m'].abs().sort_values(ascending=False).index).head(30)
print(top30[['CODNEG', 'mes', 'PREULT_anterior', 'PREULT', 'n_sessoes_anterior', 'n_sessoes',
             'ESPECI_anterior', 'ESPECI', 'FATCOT_anterior', 'FATCOT', 'ret_1m']].to_string(index=False))

# ============================================================================
print("\n" + "#" * 78)
print("# f) EFEITO DA ILIQUIDEZ (medicao, sem filtrar)")
print("#" * 78)
tmp['n_sessoes_atual'] = tmp['n_sessoes']
tmp['iliq_atual'] = tmp['n_sessoes_atual'] <= 3
tmp['iliq_anterior'] = tmp['n_sessoes_anterior'] <= 3


def resumo(sub, rotulo):
    r = sub['ret_1m']
    print(f"  {rotulo}: n={len(r)}  media={r.mean():.5f}  dp={r.std():.5f}  "
          f"p1={r.quantile(0.01):.5f}  p99={r.quantile(0.99):.5f}  "
          f"|ret|>50%={(int((r.abs() > 0.50).sum()))}")


print("por n_sessoes do mes ATUAL:")
resumo(tmp[~tmp['iliq_atual']], 'n_sessoes_atual > 3 (liquido)  ')
resumo(tmp[tmp['iliq_atual']], 'n_sessoes_atual <= 3 (iliquido)')

print("\npor n_sessoes do mes ANTERIOR:")
resumo(tmp[~tmp['iliq_anterior']], 'n_sessoes_anterior > 3 (liquido)  ')
resumo(tmp[tmp['iliq_anterior']], 'n_sessoes_anterior <= 3 (iliquido)')

print("\ncombinado (qualquer um dos dois meses iliquido vs os dois liquidos):")
algum_iliquido = tmp['iliq_atual'] | tmp['iliq_anterior']
resumo(tmp[~algum_iliquido], 'ambos os meses > 3 sessoes (liquido)')
resumo(tmp[algum_iliquido], 'ao menos um mes <= 3 sessoes (iliquido)')

n_l11 = int(algum_iliquido.sum())
pct_l11 = 100 * n_l11 / len(tmp)
print(f"\n[para L11] retornos com ao menos um dos dois meses <=3 sessoes: {n_l11} ({pct_l11:.2f}% dos validos)")

# ============================================================================
print("\n" + "#" * 78)
print("# g) serie mensal do no de retornos validos")
print("#" * 78)
por_mes = df.loc[df['ret_valido']].groupby('mes').size().rename('n_validos')
ano = por_mes.index.str[:4].astype(int)
mediana_por_ano = por_mes.groupby(ano).median()
print("mediana por ano:")
print(mediana_por_ano.to_string())
print("\n10 meses com MENOS retornos validos:")
print(por_mes.nsmallest(10).to_string())

# ============================================================================
print("\n" + "#" * 78)
print("# h) comparacao com preco NAO ajustado (PREULT_SEM_AJUSTE)")
print("#" * 78)
ret_1m_bruto = df['PREULT_SEM_AJUSTE'] / prev_preult_bruto - 1.0
denom_inv_bruto = (prev_preult_bruto <= 0) & prev_ord.notna()
ret_1m_bruto = ret_1m_bruto.where(mask_gap1)
ret_1m_bruto = ret_1m_bruto.where(~denom_inv_bruto)
ret_valido_bruto = mask_gap1.fillna(False) & np.isfinite(ret_1m_bruto)

rv_bruto = ret_1m_bruto.loc[ret_valido_bruto]
rv_ajustado = df.loc[df['ret_valido'], 'ret_1m']

print(f"AJUSTADO   -- n={len(rv_ajustado)}  dp={rv_ajustado.std():.5f}  |ret|>50%={int((rv_ajustado.abs() > 0.50).sum())}")
print(f"NAO AJUST. -- n={len(rv_bruto)}  dp={rv_bruto.std():.5f}  |ret|>50%={int((rv_bruto.abs() > 0.50).sum())}")

# ============================================================================
print("\n" + "#" * 78)
print("# SANIDADE")
print("#" * 78)

# S1
print(f"S1 -- linhas entrada={n_entrada}, saida={n_saida}")
assert n_saida == n_entrada == 158358, f"S1 FALHOU: {n_saida} != {n_entrada}"
print("S1 PASSOU.")

# S2
viola_s2 = df[(df['gap_meses'] != 1) & df['ret_1m'].notna()]
print(f"S2 -- linhas com gap_meses!=1 e ret_1m nao-nulo: {len(viola_s2)}")
assert len(viola_s2) == 0, "S2 FALHOU"
print("S2 PASSOU.")

# S3
n_inf = int(np.isinf(df['ret_1m']).sum())
print(f"S3 -- ret_1m infinito: {n_inf}")
assert n_inf == 0, "S3 FALHOU"
print("S3 PASSOU.")

# S4
n_isins = df['CODISI'].nunique()
n_buracos_total = 988  # medido em A3 (numero de ISINs com >=1 buraco), usado so como referencia qualitativa
aprox = n_total - n_isins
print(f"S4 -- linhas={n_total}, ISINs={n_isins}, linhas-ISINs={aprox}, ret_1m valido medido={n_validos}")
print(f"      (a diferenca entre 'linhas-ISINs' e o valido medido eh explicada pelos gaps>1 "
      f"e pelos denominadores invalidos, contados no item b)")

# S5
dp_ajustado = rv_ajustado.std()
dp_bruto = rv_bruto.std()
print(f"S5 -- dp ajustado={dp_ajustado:.6f}, dp nao ajustado={dp_bruto:.6f}")
assert dp_ajustado < dp_bruto, "S5 FALHOU: dp do retorno ajustado nao eh menor que o do bruto"
print("S5 PASSOU: dp do retorno ajustado eh menor que o do bruto (A2 reduziu ruido).")


entrada: (158358, 13)

linhas que teriam denominador PREULT (ajustado) <= 0: 0
saida: (158358, 19)
gravado: C:\Users\lucca\quant2026\intermediario\a4_retorno_mensal.parquet

##############################################################################
# a) shape final / colunas / %NaN por coluna / periodo
##############################################################################
shape: (158358, 19)
colunas: ['CODISI', 'mes', 'data', 'PREULT', 'PREULT_SEM_AJUSTE', 'FATCOT', 'CODNEG', 'ESPECI', 'NOMRES', 'VOLTOT', 'QUATOT', 'TOTNEG', 'n_sessoes', 'gap_meses', 'primeira_obs', 'denominador_invalido', 'ret_1m', 'ret_valido', 'preco_iliquido']

% de NaN por coluna:
CODISI                  0.0000
mes                     0.0000
data                    0.0000
PREULT                  0.0000
PREULT_SEM_AJUSTE       0.0000
FATCOT                  0.0000
CODNEG                  0.0000
ESPECI                  0.0000
NOMRES                  0.0000
VOLTOT                  0.0000
QUATOT           

## A5 — mapa setorial, ponte por ISIN e cobertura

A primeira execução desta célula usou a ponte `CODISI[2:6]` fixa e foi PARADA: a conferência
visual dos exemplos de raiz revelou que 334 ISINs de formato antigo (sem prefixo `BR`) tinham a
raiz extraída errada. O humano autorizou D05 (ffill só para célula mesclada), D06 (chave
SETOR‖SUBSETOR) e D07 (ponte por formato). Abaixo está a versão regravada, que mede as duas
pontes lado a lado — e mostra que o defeito estava confinado a 1995, deixando o [F] de
cobertura do conselho CONFIRMADO, não contaminado. Ver DOSSIE.md §1 (D05/D06/D07) e §3 (A5).


In [6]:
import difflib
import unicodedata

import numpy as np
import pandas as pd

pd.set_option('display.width', 250)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 400)

IN_PATH = r"C:\Users\lucca\quant2026\intermediario\a4_retorno_mensal.parquet"
CLASSIF_PATH = r"C:\Users\lucca\quant2026\dados\ClassifSetorial (1).xlsx"
OUT_PATH = r"C:\Users\lucca\quant2026\intermediario\a5_com_setor.parquet"

print("#" * 78)
print("# A5 (REGRAVACAO) -- tres correcoes autorizadas pelo humano")
print("#" * 78)
print("D05: ffill AUTORIZADO aqui -- leitura de celula mesclada de planilha, nunca")
print("     em serie de preco/retorno. A informacao existe no arquivo; a celula")
print("     mesclada e formato, nao ausencia. Fidelidade verificada abaixo.")
print("D06: chave do especialista = SETOR || SUBSETOR (nao o nome literal do")
print("     subsetor). NAO reabre D01: o limiar segue >=5 ativos no mes E >=150 obs")
print("     em 36m; muda a UNIDADE sobre a qual o limiar incide.")
print("D07: ponte corrigida por formato de ISIN. Regra: raiz = CODISI[2:6] se o")
print("     CODISI comeca com 'BR'; caso contrario raiz = CODISI[0:4].")

# ============================================================================
print("\n" + "#" * 78)
print("# PASSO 0 -- inspecao do mapa setorial")
print("#" * 78)

raw = pd.read_excel(CLASSIF_PATH, header=None)
print(f"\nshape bruto: {raw.shape}  (1 aba: 'Planilha')")
for c in raw.columns:
    print(f"  coluna {c}: dtype={raw[c].dtype}, nao-nulos={int(raw[c].notna().sum())}, "
          f"distintos={int(raw[c].nunique(dropna=True))}")

print("\n--- 15 primeiras linhas ---")
print(raw.head(15).to_string())

idx_header = raw.index[raw[5].astype(str).str.strip() == 'CÓDIGO'].tolist()
idx_descartar = set([0, 1] + idx_header)
print(f"\ncabecalhos repetidos (col5=='CÓDIGO'): {idx_header}")
print(f"linhas descartadas (cabecalho/vazio): {len(idx_descartar)}")
print("\nROTULOS lidos dos cabecalhos: col1=SETOR, col2=SUBSETOR, col3=SEGMENTO,")
print("col4=EMISSOR/NOME DE PREGAO, col5=CODIGO, col6=SEGMENTO DE NEGOCIACAO.")

falhas = [h + 1 for h in idx_header
          if (h + 1) in raw.index and (h + 1) not in idx_descartar
          and raw.loc[h + 1, [1, 2, 3]].isna().any()]
print(f"\n[D05] linhas apos cabecalho que NAO reiniciam a hierarquia: {falhas}")
assert not falhas, f"PAROU: bloco atravessa quebra de pagina em {falhas}"
print("[D05] PASSOU: reconstrucao de celula mesclada e fiel ao arquivo.")

mapa = raw.drop(index=sorted(idx_descartar)).copy()
mapa.columns = ['_x', 'setor', 'subsetor', 'segmento', 'empresa', 'codigo', 'segmento_negociacao']
mapa = mapa.drop(columns=['_x'])
for col in ['setor', 'subsetor', 'segmento']:
    mapa[col] = mapa[col].ffill()          # D05
mapa['codigo'] = mapa['codigo'].astype(str).str.strip()

print(f"\nempresas no mapa: {len(mapa)} | CODIGOs distintos: {mapa['codigo'].nunique()}")
print(f"SETOR: {mapa['setor'].nunique()} | SUBSETOR: {mapa['subsetor'].nunique()} | "
      f"SEGMENTO: {mapa['segmento'].nunique()}")

# [D06] chave composta
mapa['subsetor_chave'] = mapa['setor'].astype(str).str.strip() + ' || ' + mapa['subsetor'].astype(str).str.strip()
print(f"[D06] chaves SETOR||SUBSETOR distintas: {mapa['subsetor_chave'].nunique()}")

print("\nempresas por chave SETOR||SUBSETOR -- 10 MAIORES:")
print(mapa['subsetor_chave'].value_counts().head(10).to_string())
print("\n10 MENORES:")
print(mapa['subsetor_chave'].value_counts().tail(10).to_string())

print("\n[D06] os homonimos que motivaram a chave composta:")
col = mapa['subsetor'].value_counts()
homon = mapa[mapa['subsetor'].isin(
    mapa[['setor', 'subsetor']].drop_duplicates()['subsetor'].value_counts().pipe(lambda s: s[s > 1]).index)]
print(homon[['setor', 'subsetor', 'subsetor_chave']].drop_duplicates().sort_values('subsetor').to_string(index=False))

# ============================================================================
print("\n" + "#" * 78)
print("# PASSO 1 -- ponte CORRIGIDA por formato de ISIN [D07]")
print("#" * 78)

df = pd.read_parquet(IN_PATH)
n_entrada = len(df)
print(f"\nentrada (A4): {df.shape}")

tem_br = df['CODISI'].str.startswith('BR')
df['raiz_isin'] = np.where(tem_br, df['CODISI'].str.slice(2, 6), df['CODISI'].str.slice(0, 4))
df['raiz_antiga'] = df['CODISI'].str.slice(2, 6)

u = df.drop_duplicates('CODISI')
print(f"ISINs distintos: {len(u)} | com prefixo BR: {int(u['CODISI'].str.startswith('BR').sum())} | "
      f"sem: {int((~u['CODISI'].str.startswith('BR')).sum())}")

print("\n20 exemplos CODISI -> raiz antiga -> raiz NOVA -> CODNEG:")
print(u[['CODISI', 'raiz_antiga', 'raiz_isin', 'CODNEG']].head(20).to_string(index=False))

mapa_join = mapa[['codigo', 'setor', 'subsetor', 'subsetor_chave', 'segmento',
                  'empresa', 'segmento_negociacao']].drop_duplicates('codigo')

antes = len(df)
df = df.merge(mapa_join, how='left', left_on='raiz_isin', right_on='codigo')
print(f"\nlinhas antes/depois da juncao: {antes} / {len(df)}")
assert len(df) == antes, f"PAROU: juncao alterou o numero de linhas ({antes} -> {len(df)})"
print("PASSOU: juncao nao duplicou nem perdeu linha.")
df = df.drop(columns=['codigo'])

# ============================================================================
print("\n" + "#" * 78)
print("# ITEM 4 -- COERENCIA DE NOME dos 67 recuperados (NOMRES vs empresa)")
print("#" * 78)


def norm(s):
    s = unicodedata.normalize('NFKD', str(s)).encode('ascii', 'ignore').decode()
    return ''.join(ch for ch in s.upper() if ch.isalnum() or ch == ' ').strip()


def coerente(a, b):
    na, nb = norm(a), norm(b)
    if not na or not nb:
        return False
    ca, cb = na.replace(' ', ''), nb.replace(' ', '')
    if ca[:4] == cb[:4]:
        return True
    if ca in cb or cb in ca:
        return True
    for t in [t for t in na.split() if len(t) >= 4]:
        if t in cb:
            return True
    for t in [t for t in nb.split() if len(t) >= 4]:
        if t in ca:
            return True
    return difflib.SequenceMatcher(None, ca, cb).ratio() >= 0.45


cods_mapa = set(mapa_join['codigo'])
u2 = df.drop_duplicates('CODISI').copy()
recuperados = u2[(~u2['raiz_antiga'].isin(cods_mapa)) & (u2['raiz_isin'].isin(cods_mapa))].copy()
print(f"\nISINs RECUPERADOS pela ponte corrigida: {len(recuperados)}")

meses = df.groupby('CODISI')['mes'].agg(['min', 'max'])
recuperados = recuperados.join(meses, on='CODISI')
recuperados['ok_nome'] = [coerente(a, b) for a, b in zip(recuperados['NOMRES'], recuperados['empresa'])]

print("\nITEM 2 + 4 -- lista completa dos recuperados:")
print(recuperados[['CODISI', 'CODNEG', 'raiz_antiga', 'raiz_isin', 'NOMRES', 'empresa',
                   'subsetor_chave', 'min', 'max', 'ok_nome']]
      .sort_values('CODNEG').to_string(index=False))

sinalizados = recuperados[~recuperados['ok_nome']]
print(f"\nETAPA 1 -- pares NOMRES/empresa sinalizados pelo teste de similaridade: {len(sinalizados)}")
if len(sinalizados):
    print(sinalizados[['CODISI', 'CODNEG', 'raiz_isin', 'NOMRES', 'empresa']].to_string(index=False))

# ETAPA 2 -- adjudicacao pelos PROPRIOS DADOS, nao por conhecimento externo nem
# por lista branca: um par so e aceito se o nome do ClassifSetorial (que e o nome
# de HOJE) aparecer em algum momento como NOMRES do MESMO ISIN-raiz no COTAHIST.
# Isso e a assinatura de renomeacao societaria -- e e justamente o que a ponte por
# ISIN existe para atravessar. Se o nome de hoje NUNCA aparece, o par e falso.
print("\nETAPA 2 -- adjudicacao pelos dados: o nome de HOJE (ClassifSetorial) aparece")
print("           como NOMRES do mesmo ISIN-raiz em algum momento do COTAHIST?")
nomes_por_raiz = df.groupby('raiz_isin')['NOMRES'].apply(lambda s: {norm(x) for x in s.unique()})
resolvidos, nao_resolvidos = [], []
for _, r in sinalizados.iterrows():
    historico = nomes_por_raiz.get(r['raiz_isin'], set())
    achou = any(coerente(r['empresa'], h) for h in historico)
    (resolvidos if achou else nao_resolvidos).append(r['CODISI'])
    if achou:
        ev = (df[df['raiz_isin'] == r['raiz_isin']]
              .assign(ano=lambda x: x['mes'].str[:4].astype(int))
              .groupby('NOMRES')['ano'].agg(['min', 'max']))
        linha = ' ; '.join(f"{n} ({a}-{b})" for n, (a, b) in ev.iterrows())
        print(f"  RESOLVIDO {r['CODNEG']:>7} raiz={r['raiz_isin']} -> historico de NOMRES: {linha}")
    else:
        print(f"  NAO RESOLVIDO {r['CODNEG']:>7} raiz={r['raiz_isin']} "
              f"NOMRES={r['NOMRES']!r} empresa={r['empresa']!r} historico={sorted(historico)}")

print(f"\nresolvidos como RENOMEACAO SOCIETARIA (confirmada nos dados): {len(resolvidos)}")
print(f"NAO resolvidos (pareamento suspeito): {len(nao_resolvidos)}")
assert not nao_resolvidos, f"PAROU: par NOMRES/empresa nao explicado por renomeacao: {nao_resolvidos}"
print("PASSOU: os 67 pares COTAHIST/ClassifSetorial sao coerentes -- os divergentes")
print("        sao renomeacoes societarias que a ponte por ISIN atravessa corretamente")
print("        (uma ponte por NOME teria errado justamente esses).")

# --- rotulo SEM_SETOR -------------------------------------------------------
for c, lab in [('setor', 'SEM_SETOR'), ('subsetor', 'SEM_SETOR'), ('subsetor_chave', 'SEM_SETOR'),
               ('segmento', 'SEM_SETOR'), ('empresa', 'SEM_EMPRESA'),
               ('segmento_negociacao', 'SEM_SEGMENTO_NEG')]:
    df[c] = df[c].astype(object).where(df[c].notna(), lab)

df['ano'] = df['mes'].str[:4].astype(int)
com = df['subsetor_chave'] != 'SEM_SETOR'
print(f"\nlinhas que casaram: {int(com.sum())} ({100*com.mean():.2f}%)")

# ============================================================================
print("\n" + "#" * 78)
print("# ITEM 1 -- COBERTURA ANO A ANO: ponte ANTIGA vs CORRIGIDA, lado a lado")
print("#" * 78)

com_antiga = df['raiz_antiga'].isin(cods_mapa)


def cobertura(flag):
    g = df.assign(_f=flag)
    r = g.groupby('ano').apply(lambda x: pd.Series({
        'isins': x['CODISI'].nunique(),
        'isins_com': x.loc[x['_f'], 'CODISI'].nunique(),
        'vol': x['VOLTOT'].sum(),
        'vol_com': x.loc[x['_f'], 'VOLTOT'].sum(),
    }), include_groups=False)
    return (100 * r['isins_com'] / r['isins']).round(2), (100 * r['vol_com'] / r['vol']).round(2), r


at_n, at_v, r_at = cobertura(com_antiga)
nv_n, nv_v, r_nv = cobertura(com.to_numpy())

comp = pd.DataFrame({
    'isins_grade': r_nv['isins'].astype(int),
    'com_ANTIGA': r_at['isins_com'].astype(int),
    'com_NOVA': r_nv['isins_com'].astype(int),
    'pct_ativos_ANTIGA': at_n,
    'pct_ativos_NOVA': nv_n,
    'delta_pp_ativos': (nv_n - at_n).round(2),
    'pct_vol_ANTIGA': at_v,
    'pct_vol_NOVA': nv_v,
    'delta_pp_vol': (nv_v - at_v).round(2),
})
print(comp.to_string())
print("\nSEM_SETOR (ISINs distintos) por ano, nova ponte:")
print(df.loc[~com].groupby('ano')['CODISI'].nunique().to_string())

print("\nno de chaves SETOR||SUBSETOR distintas presentes por ano:")
print(df.loc[com].groupby('ano')['subsetor_chave'].nunique().to_string())

# ============================================================================
print("\n" + "#" * 78)
print("# ITEM 3 -- SIMULACAO D01 sob SETOR||SUBSETOR e ponte corrigida")
print("#" * 78)
print("D01 CONGELADO: >=5 ativos elegiveis no mes E >=150 observacoes na janela 36m.\n")

esp = df.loc[com, ['mes', 'ano', 'subsetor_chave', 'CODISI']].copy()
ativos = esp.groupby(['mes', 'subsetor_chave'])['CODISI'].nunique().rename('n_ativos').reset_index()
obs = esp.groupby(['mes', 'subsetor_chave']).size().rename('n_obs').reset_index()

ordm = pd.PeriodIndex(obs['mes'], freq='M').asi8
obs['_ord'] = ordm
painel = obs.pivot(index='_ord', columns='subsetor_chave', values='n_obs')
painel = painel.reindex(pd.RangeIndex(painel.index.min(), painel.index.max() + 1))
obs36 = painel.rolling(36, min_periods=1).sum().stack().rename('obs_36m').reset_index()
obs36.columns = ['_ord', 'subsetor_chave', 'obs_36m']

ativos['_ord'] = pd.PeriodIndex(ativos['mes'], freq='M').asi8
d01 = ativos.merge(obs36, on=['_ord', 'subsetor_chave'], how='left')
d01['so_5ativos'] = d01['n_ativos'] >= 5
d01['d01_completo'] = (d01['n_ativos'] >= 5) & (d01['obs_36m'] >= 150)

s5 = d01.groupby('mes')['so_5ativos'].sum()
sc = d01.groupby('mes')['d01_completo'].sum()
tab = pd.DataFrame({'so_>=5_ativos': s5, 'D01_completo': sc})
tab['ano'] = tab.index.str[:4].astype(int)
tab['m'] = tab.index.str[5:7].astype(int)

print("mediana ANUAL do no de especialistas:")
print(tab.groupby('ano')[['so_>=5_ativos', 'D01_completo']].median().to_string())

print("\nserie MENSAL do D01 COMPLETO (linhas=ano, colunas=mes):")
print(tab.pivot_table(index='ano', columns='m', values='D01_completo').to_string())

eleg = d01.loc[d01['d01_completo'], ['mes', 'subsetor_chave']].assign(_ok=True)
t = df[['mes', 'ano', 'CODISI', 'subsetor_chave']].merge(eleg, on=['mes', 'subsetor_chave'], how='left')
t['_ok'] = t['_ok'].astype(object).where(t['_ok'].notna(), False).astype(bool)
gen = t.groupby('mes').agg(universo=('CODISI', 'nunique'), n_gen=('_ok', lambda s: int((~s).sum())))
gen['pct_gen'] = (100 * gen['n_gen'] / gen['universo']).round(2)
gen['ano'] = gen.index.str[:4].astype(int)
print("\nGENERALISTA sob D01 COMPLETO -- mediana anual:")
print(gen.groupby('ano')[['universo', 'n_gen', 'pct_gen']].median().round(2).to_string())

# ============================================================================
print("\n" + "#" * 78)
print("# ITEM (f) da A5 -- chaves por no MEDIO de ativos/mes, 2018-2026")
print("#" * 78)
rec = ativos[(ativos['mes'] >= '2018-01') & (ativos['mes'] <= '2026-08')]
med = rec.groupby('subsetor_chave')['n_ativos'].mean().sort_values(ascending=False).round(2)
print("15 MAIORES:")
print(med.head(15).to_string())
print("\n15 MENORES:")
print(med.tail(15).to_string())

# ============================================================================
print("\n" + "#" * 78)
print("# ITEM (g) da A5 -- SONDA DE VIES sobre os SEM_SETOR (ponte corrigida)")
print("#" * 78)
sem = df.loc[~com]
vida = pd.concat([sem.groupby('CODNEG')['mes'].min().rename('primeiro_mes'),
                  sem.groupby('CODNEG')['mes'].max().rename('ultimo_mes')], axis=1)
mortos = vida[vida['ultimo_mes'] < '2010-01'].sort_values('ultimo_mes', ascending=False)
vivos = vida[vida['ultimo_mes'] >= '2024-01'].sort_values('ultimo_mes', ascending=False)
print(f"CODNEG em SEM_SETOR: {len(vida)}")
print(f"  ultima obs antes de 2010: {len(mortos)} ({100*len(mortos)/len(vida):.1f}%)")
print(f"  com obs em 2024-2026:     {len(vivos)} ({100*len(vivos)/len(vida):.1f}%)")
print("\n15 SEM_SETOR mortos antes de 2010:")
print(mortos.head(15).to_string())
print("\n15 SEM_SETOR vivos em 2024-2026:")
print(vivos.head(15).to_string())

# ============================================================================
print("\n" + "#" * 78)
print("# GRAVACAO")
print("#" * 78)
df = df.drop(columns=['ano'])
df.to_parquet(OUT_PATH, index=False)
print(f"REGRAVADO: {OUT_PATH}")
print(f"shape: {df.shape}")
print(f"colunas: {list(df.columns)}")
print("\n% NaN por coluna:")
print((df.isna().mean() * 100).round(4).to_string())

# ============================================================================
print("\n" + "#" * 78)
print("# SANIDADE")
print("#" * 78)
print(f"S1 -- entrada={n_entrada}, saida={len(df)}")
assert len(df) == n_entrada == 158358
print("S1 PASSOU.")

print(f"S2 -- NaN em subsetor: {int(df['subsetor'].isna().sum())} | "
      f"em subsetor_chave: {int(df['subsetor_chave'].isna().sum())}")
assert df['subsetor'].isna().sum() == 0 and df['subsetor_chave'].isna().sum() == 0
print("S2 PASSOU.")

dec = pd.DataFrame({'isins': r_nv['isins'], 'com': r_nv['isins_com']})
dec['decada'] = (dec.index // 10) * 10
cd = (100 * dec.groupby('decada')['com'].sum() / dec.groupby('decada')['isins'].sum()).round(2)
print("\nS3 -- cobertura por decada (ponte corrigida):")
print(cd.to_string())
cres = all(cd.iloc[i] <= cd.iloc[i + 1] for i in range(len(cd) - 1))
c20 = nv_n[nv_n.index >= 2020]
print(f"crescente: {cres} | anos 2020: min={c20.min()}% max={c20.max()}%")
assert cres and c20.min() > 60
print("S3 PASSOU.")
print("\n[D07] curva ANTIGA vs NOVA nos anos de referencia do conselho:")
for a in (1996, 2010, 2022):
    print(f"  {a}: antiga={at_n.loc[a]}%  ->  NOVA={nv_n.loc[a]}%  "
          f"(referencia [F] do conselho: ~{ {1996:19,2010:48,2022:79}[a] }%)")

r4 = tab[(tab.index >= '2018-01') & (tab.index <= '2026-08')]
print(f"\nS4 -- mediana 2018-2026: so_>=5_ativos={r4['so_>=5_ativos'].median()} | "
      f"D01_completo={r4['D01_completo'].median()}")
assert r4['D01_completo'].median() >= 5
print("S4 PASSOU.")


##############################################################################
# A5 (REGRAVACAO) -- tres correcoes autorizadas pelo humano
##############################################################################
D05: ffill AUTORIZADO aqui -- leitura de celula mesclada de planilha, nunca
     em serie de preco/retorno. A informacao existe no arquivo; a celula
     mesclada e formato, nao ausencia. Fidelidade verificada abaixo.
D06: chave do especialista = SETOR || SUBSETOR (nao o nome literal do
     subsetor). NAO reabre D01: o limiar segue >=5 ativos no mes E >=150 obs
     em 36m; muda a UNIDADE sobre a qual o limiar incide.
D07: ponte corrigida por formato de ISIN. Regra: raiz = CODISI[2:6] se o
     CODISI comeca com 'BR'; caso contrario raiz = CODISI[0:4].

##############################################################################
# PASSO 0 -- inspecao do mapa setorial
##############################################################################

shape bruto: (382, 7)  

## A8 — universo elegível point-in-time (ordem alterada por D04: roda ANTES de A6)

Primeira execução com **D08** (≥36 meses observados até t, negociou em t, n_sessoes ≥4 em t) e
**D09** (treino exige as duas pernas). Ela produziu dois achados que viraram decisão nova:

- **D10** — a perna t-1 do retorno também precisa de ≥4 sessões. `elegivel` condicionava só a
  perna t, mas o denominador do retorno é t-1, e 4.236 retornos de elegíveis (4,51%) tinham essa
  perna com ≤3 sessões, com dp de 1,3946 contra 0,1909. A feature mom_1m *é* esse retorno.
- **D11** — a defasagem residual não é filtrada. A menor dispersão do grupo defasado é a
  *assinatura* da defasagem (preço parado varia menos), não sua refutação; S3 vira falha
  conhecida registrada e o efeito é medido em B1.

Abaixo, a reexecução com D10. A célula **não remove nenhuma linha** — só acrescenta flags — e o
parquet é gravado depois das sanidades. Ver DOSSIE.md §1 (D04/D08/D09/D10/D11), §3 (A8) e §6.


In [7]:
import numpy as np
import pandas as pd

pd.set_option('display.width', 250)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 400)

IN_PATH = r"C:\Users\lucca\quant2026\intermediario\a5_com_setor.parquet"
OUT_PATH = r"C:\Users\lucca\quant2026\intermediario\a8_universo_elegivel.parquet"

MIN_HIST, MIN_SESSOES, MIN_ATIVOS_D01, MIN_OBS_D01 = 36, 4, 5, 150

print("#" * 78)
print("# A8 (REEXECUCAO com D10) -- universo elegivel point-in-time, por FLAG.")
print("#" * 78)
print(f"D08: meses_historico(t) >= {MIN_HIST} E negociou em t E n_sessoes(t) >= {MIN_SESSOES}")
print(f"D10: ... E n_sessoes(t-1) >= {MIN_SESSOES}, com t-1 o mes calendario imediatamente")
print("     anterior (gap_meses(t) == 1). Mecanismo: a feature mom_1m E esse retorno; um")
print("     denominador com 1 a 3 sessoes mede defasagem de preco, nao momento. Sem")
print("     look-ahead: t-1 e t precedem a decisao tomada em t.")
print(f"D09: elegivel_treino(t) <=> elegivel(t) E n_sessoes(t+1) >= {MIN_SESSOES} E ret_valido(t+1)")
print("     -> sob D10 isso implica t-1, t e t+1 todos com >= 4 sessoes.")
print("D11: defasagem residual NAO e filtrada. S3 fica como FALHA CONHECIDA e nao para a celula.")
print("Limiares congelados ANTES desta execucao. Nenhum sera ajustado pelo resultado.\n")

df = pd.read_parquet(IN_PATH)
n_entrada = len(df)
print(f"entrada: {df.shape}")

df = df.sort_values(['CODISI', 'mes'], kind='mergesort').reset_index(drop=True)
g = df.groupby('CODISI', sort=False)

# --- (i) historico point-in-time: meses PRESENTES ate t inclusive -----------
# a grade so tem linha onde houve negocio, entao a contagem cumulativa de linhas
# do proprio ISIN E a contagem de meses observados -- buraco nao conta, por
# construcao. Nao usa span nem calendario.
df['meses_historico'] = g.cumcount() + 1

df['elegivel_hist'] = df['meses_historico'] >= MIN_HIST
df['elegivel_sessoes'] = df['n_sessoes'] >= MIN_SESSOES

# --- D10: a perna ANTERIOR do retorno tambem tem de ser bem formada -------
# t-1 e a observacao anterior do MESMO ISIN, e so vale se for o mes calendario
# imediatamente anterior -- gap_meses(t) == 1. Primeira observacao tem
# gap_meses NaN, logo a comparacao e False por construcao (nao ha t-1).
n_sessoes_ant = g['n_sessoes'].shift(1)
df['elegivel_perna_anterior'] = ((df['gap_meses'] == 1)
                                 & (n_sessoes_ant >= MIN_SESSOES)).fillna(False).astype(bool)

df['elegivel_D08'] = df['elegivel_hist'] & df['elegivel_sessoes']          # definicao ANTERIOR, so p/ o item (j)
df['elegivel'] = df['elegivel_D08'] & df['elegivel_perna_anterior']        # definicao VIGENTE (D08+D10)

# --- alinhamento de t+1 (D09) ----------------------------------------------
print("ALINHAMENTO DE t+1 (D09), declarado explicitamente:")
print("  df ordenado por (CODISI, mes); dentro de cada CODISI, t+1 e a PROXIMA LINHA")
print("  via groupby('CODISI').shift(-1). Uma proxima linha so conta como t+1 se")
print("  gap_meses dela == 1, isto e, se e o mes calendario imediatamente seguinte.")
print("  ret_valido(t+1) ja embute gap==1 (definido assim em A4); a condicao de gap")
print("  e reafirmada aqui de forma redundante e explicita, para nao depender disso.")
g2 = df.groupby('CODISI', sort=False)
n_sessoes_prox = g2['n_sessoes'].shift(-1)
ret_valido_prox = g2['ret_valido'].shift(-1)
gap_prox = g2['gap_meses'].shift(-1)

prox_eh_mes_seguinte = (gap_prox == 1)
df['elegivel_treino'] = (df['elegivel']
                         & prox_eh_mes_seguinte.fillna(False).astype(bool)
                         & (n_sessoes_prox >= MIN_SESSOES).fillna(False).astype(bool)
                         & ret_valido_prox.fillna(False).astype(bool))

print("\n(gravacao movida para o FIM, depois das sanidades -- se S4 falhar, nada e regravado)")

df['ano'] = df['mes'].str[:4].astype(int)

# ============================================================================
print("\n" + "#" * 78)
print("# a) shape final, colunas novas, %NaN, decomposicao dos inelegiveis")
print("#" * 78)
novas = ['meses_historico', 'elegivel_hist', 'elegivel_sessoes', 'elegivel_perna_anterior',
         'elegivel_D08', 'elegivel', 'elegivel_treino']
print(f"shape: {df.shape}  (entrada: {n_entrada})")
print(f"colunas novas: {novas}")
print("\n% NaN nas colunas novas:")
print((df[novas].isna().mean() * 100).round(4).to_string())

n_eleg = int(df['elegivel'].sum())
h, s, a_ = df['elegivel_hist'], df['elegivel_sessoes'], df['elegivel_perna_anterior']
print(f"\nelegiveis (D08+D10): {n_eleg}")
print(f"inelegiveis:         {int((~df['elegivel']).sum())}")
print("\ndecomposicao EXAUSTIVA e MUTUAMENTE EXCLUSIVA dos inelegiveis, por combinacao")
print("das tres condicoes (hist / sessoes(t) / perna(t-1)); 1 = satisfeita:")
comb = (h.astype(int).astype(str) + '/' + s.astype(int).astype(str) + '/' + a_.astype(int).astype(str))
vc = comb.value_counts().sort_index()
rot = {'1/1/1': 'ELEGIVEL', '1/1/0': 'so falha perna t-1  (NOVO em D10)',
       '1/0/1': 'so falha sessoes(t)', '0/1/1': 'so falha historico',
       '1/0/0': 'falha sessoes(t) e perna t-1', '0/1/0': 'falha historico e perna t-1',
       '0/0/1': 'falha historico e sessoes(t)', '0/0/0': 'falha as tres'}
for k, v in vc.items():
    print(f"  hist/sess/perna = {k}: {v:>7}   {rot.get(k,'')}")
assert int(vc.sum()) == n_entrada
print(f"soma = {int(vc.sum())} == {n_entrada}")

# decomposicao no estilo do prompt original, sobre as duas primeiras condicoes
so_hist = int((~h & s).sum()); so_sess = int((h & ~s).sum()); ambos = int((~h & ~s).sum())
print(f"\n(visao D08 isolada: so historico={so_hist}, so sessoes={so_sess}, ambos={ambos})")
print(f"linhas que D10 tirou do universo (eram elegiveis por D08): "
      f"{int((df['elegivel_D08'] & ~df['elegivel']).sum())}")
print(f"\nelegivel_treino: {int(df['elegivel_treino'].sum())}")

# ============================================================================
print("\n" + "#" * 78)
print("# b) TABELA ANO A ANO")
print("#" * 78)
tb = df.groupby('ano').apply(lambda x: pd.Series({
    'pares_isin_mes': len(x),
    'elegiveis': int(x['elegivel'].sum()),
    'rem_historico': int((~x['elegivel_hist']).sum()),
    'rem_sessoes': int((~x['elegivel_sessoes']).sum()),
    'rem_perna_ant': int((~x['elegivel_perna_anterior']).sum()),
}), include_groups=False)
tb['pct_elegivel'] = (100 * tb['elegiveis'] / tb['pares_isin_mes']).round(2)
print(tb[['pares_isin_mes', 'elegiveis', 'pct_elegivel', 'rem_historico', 'rem_sessoes',
          'rem_perna_ant']].to_string())

pct_perde_total = 100 * (1 - n_eleg / n_entrada)
teste = df[(df['mes'] >= '2018-01') & (df['mes'] <= '2026-07')]
pct_perde_teste = 100 * (1 - teste['elegivel'].mean())
print(f"\n[para L15] pares que PERDEM elegibilidade: {pct_perde_total:.2f}% no total, "
      f"{pct_perde_teste:.2f}% no periodo de teste (2018-01 a 2026-07)")

# ============================================================================
print("\n" + "#" * 78)
print("# c) ISINs ELEGIVEIS por mes")
print("#" * 78)
por_mes = df[df['elegivel']].groupby('mes')['CODISI'].nunique().rename('n_elegiveis')
ano_m = por_mes.index.str[:4].astype(int)
print("mediana anual de ISINs elegiveis:")
print(por_mes.groupby(ano_m).median().to_string())
print("\nserie MENSAL completa do periodo de teste (2018-01 a 2026-08):")
print(por_mes[(por_mes.index >= '2018-01')].to_string())

# ============================================================================
print("\n" + "#" * 78)
print("# d) EFEITO SOBRE A DISPERSAO de ret_1m -- tres recortes")
print("#" * 78)
n_sess_ant = g2['n_sessoes'].shift(1)
val = df['ret_valido'].astype(bool)
duas_pernas_ok = val & (df['n_sessoes'] >= MIN_SESSOES) & (n_sess_ant >= MIN_SESSOES)
alguma_ruim = val & ~duas_pernas_ok


def resumo(mask, rot):
    r = df.loc[mask, 'ret_1m']
    print(f"  {rot}")
    print(f"    n={len(r)}  media={r.mean():.5f}  dp={r.std():.5f}  "
          f"p1={r.quantile(0.01):.5f}  p99={r.quantile(0.99):.5f}  max={r.max():.2f}")
    print(f"    |ret|>20%={int((r.abs()>0.20).sum())}  |ret|>50%={int((r.abs()>0.50).sum())}  "
          f"|ret|>100%={int((r.abs()>1.00).sum())}")
    return r.std()


dp_i = resumo(val, "(i)   todos os retornos validos")
dp_ii = resumo(duas_pernas_ok, f"(ii)  as DUAS pernas com n_sessoes >= {MIN_SESSOES}")
dp_iii = resumo(alguma_ruim, f"(iii) ao menos uma perna com n_sessoes <= {MIN_SESSOES-1}")

# ============================================================================
print("\n" + "#" * 78)
print("# e) AESL3 2004-05 -- o retorno de +141.566%")
print("#" * 78)
alvo = df[(df['CODNEG'] == 'AESL3') & (df['mes'].isin(['2004-04', '2004-05', '2004-06']))]
cols = ['CODNEG', 'CODISI', 'mes', 'n_sessoes', 'gap_meses', 'meses_historico', 'ret_1m', 'ret_valido',
        'elegivel_hist', 'elegivel_sessoes', 'elegivel_perna_anterior', 'elegivel_D08',
        'elegivel', 'elegivel_treino', 'VOLTOT']
print(alvo[cols].to_string(index=False))

linha_maio = df[(df['CODNEG'] == 'AESL3') & (df['mes'] == '2004-05')]
linha_abril = df[(df['CODNEG'] == 'AESL3') & (df['mes'] == '2004-04')]
idx_maio = linha_maio.index[0]
em_iii = bool(alguma_ruim.loc[idx_maio])
fora_treino_maio = not bool(linha_maio['elegivel_treino'].iloc[0])
fora_treino_abril = (not bool(linha_abril['elegivel_treino'].iloc[0])) if len(linha_abril) else True
print(f"\nVEREDITO:")
print(f"  esta no recorte (iii) (alguma perna <= 3 sessoes)? {em_iii}")
print(f"  a propria linha 2004-05 tem elegivel_treino=False?  {fora_treino_maio}")
print(f"  a linha 2004-04 (cujo ALVO seria esse retorno) tem elegivel_treino=False? {fora_treino_abril}")
print("  -> o retorno so entraria no ajuste como ALVO da linha 2004-04; como essa linha")
print("     tem elegivel_treino=False, o retorno NAO entra em nenhum ajuste de modelo.")

# ============================================================================
print("\n" + "#" * 78)
print("# f) DEFASAGEM RESIDUAL entre os ELEGIVEIS")
print("#" * 78)
per = pd.PeriodIndex(df['mes'], freq='M')
fim_mes = per.to_timestamp(how='end').normalize()
df['gap_fim_mes'] = (fim_mes - df['data'].dt.normalize()).dt.days
defasados = df[df['elegivel'] & (df['gap_fim_mes'] > 10)]
print(f"pares ELEGIVEIS com ultima sessao a mais de 10 dias corridos do fim do mes: {len(defasados)}")
if len(defasados):
    print(defasados[['CODNEG', 'mes', 'data', 'n_sessoes', 'gap_fim_mes', 'VOLTOT']].to_string(index=False))
print(f"(para comparacao, entre TODOS os pares: {int((df['gap_fim_mes'] > 10).sum())})")

# ============================================================================
print("\n" + "#" * 78)
print("# g) EFEITO SOBRE A ARQUITETURA -- D01 sob o universo ELEGIVEL")
print("#" * 78)
print(f"D01 (congelado): >= {MIN_ATIVOS_D01} ativos ELEGIVEIS no mes E "
      f">= {MIN_OBS_D01} observacoes DE TREINO na janela de 36 meses.\n")

com_setor = df['subsetor_chave'] != 'SEM_SETOR'
el = df[df['elegivel'] & com_setor]
ativos = el.groupby(['mes', 'subsetor_chave'])['CODISI'].nunique().rename('n_ativos').reset_index()

tr = df[df['elegivel_treino'] & com_setor]
obs = tr.groupby(['mes', 'subsetor_chave']).size().rename('n_obs').reset_index()
obs['_ord'] = pd.PeriodIndex(obs['mes'], freq='M').asi8
pv = obs.pivot(index='_ord', columns='subsetor_chave', values='n_obs')
pv = pv.reindex(pd.RangeIndex(pv.index.min(), pv.index.max() + 1))
obs36 = pv.rolling(36, min_periods=1).sum().stack().rename('obs36').reset_index()
obs36.columns = ['_ord', 'subsetor_chave', 'obs36']

ativos['_ord'] = pd.PeriodIndex(ativos['mes'], freq='M').asi8
d01 = ativos.merge(obs36, on=['_ord', 'subsetor_chave'], how='left')
d01['obs36'] = d01['obs36'].where(d01['obs36'].notna(), 0)
d01['ok'] = (d01['n_ativos'] >= MIN_ATIVOS_D01) & (d01['obs36'] >= MIN_OBS_D01)

esp_mes = d01.groupby('mes')['ok'].sum().rename('n_especialistas')
ano_e = esp_mes.index.str[:4].astype(int)
print("mediana anual do no de especialistas sob D01 (universo elegivel):")
print(esp_mes.groupby(ano_e).median().to_string())
print("\nserie MENSAL 2018-2026:")
print(esp_mes[esp_mes.index >= '2018-01'].to_string())

eleg_ok = d01.loc[d01['ok'], ['mes', 'subsetor_chave']].assign(_ok=True)
uni = df[df['elegivel']][['mes', 'ano', 'CODISI', 'subsetor_chave']].merge(
    eleg_ok, on=['mes', 'subsetor_chave'], how='left')
uni['_ok'] = uni['_ok'].astype(object).where(uni['_ok'].notna(), False).astype(bool)
gen = uni.groupby('mes').agg(universo=('CODISI', 'nunique'),
                              n_gen=('_ok', lambda s: int((~s).sum())))
gen['pct_gen'] = (100 * gen['n_gen'] / gen['universo']).round(2)
gen['ano'] = gen.index.str[:4].astype(int)
print("\n% do universo ELEGIVEL mensal no GENERALISTA -- mediana anual:")
print(gen.groupby('ano')[['universo', 'n_gen', 'pct_gen']].median().round(2).to_string())

# ============================================================================
print("\n" + "#" * 78)
print("# h) PERFIL DE TAMANHO do universo elegivel (insumo da capacity, D4)")
print("#" * 78)
elg = df[df['elegivel']]
q = elg.groupby('ano')['VOLTOT'].describe(percentiles=[.01, .10, .25, .5, .75, .90, .99])
print("VOLTOT mensal (R$) entre os elegiveis, por ano:")
print(q[['1%', '10%', '25%', '50%', '75%', '90%', '99%']].round(0).to_string())

print("\nPeriodo de teste (2018-01 a 2026-08) -- elegiveis por mes abaixo de limiares de VOLTOT:")
et = elg[elg['mes'] >= '2018-01']
lim = et.groupby('mes').agg(
    elegiveis=('CODISI', 'nunique'),
    abaixo_1M=('VOLTOT', lambda s: int((s < 1e6).sum())),
    abaixo_5M=('VOLTOT', lambda s: int((s < 5e6).sum())),
    abaixo_10M=('VOLTOT', lambda s: int((s < 10e6).sum())),
)
print(lim.describe().loc[['mean', '50%', 'min', 'max']].round(1).to_string())
print("\nmediana mensal: elegiveis={:.0f}, <R$1M={:.0f}, <R$5M={:.0f}, <R$10M={:.0f}".format(
    lim['elegiveis'].median(), lim['abaixo_1M'].median(),
    lim['abaixo_5M'].median(), lim['abaixo_10M'].median()))
print("pct mediano do universo elegivel abaixo de R$1M/mes: {:.2f}%".format(
    100 * (lim['abaixo_1M'] / lim['elegiveis']).median()))

# ============================================================================
print("\n" + "#" * 78)
print("# i) O QUE O CORTE REMOVEU, auditavel")
print("#" * 78)
cai_sess = df[df['elegivel_hist'] & ~df['elegivel_sessoes']]
print(f"20 exemplos: elegiveis por HISTORICO mas caidos por SESSOES (de {len(cai_sess)}):")
print(cai_sess.sample(20, random_state=0)[
    ['CODNEG', 'mes', 'n_sessoes', 'meses_historico', 'VOLTOT', 'ret_1m']].sort_values('mes').to_string(index=False))

nunca = df.groupby('CODISI').agg(
    ja_elegivel=('elegivel', 'any'),
    max_hist=('meses_historico', 'max'),
    max_sess=('n_sessoes', 'max'),
    codneg=('CODNEG', 'last'),
    meses=('mes', 'size'),
)
nunca_el = nunca[~nunca['ja_elegivel']].copy()
nunca_el['motivo'] = np.where(
    nunca_el['max_hist'] < MIN_HIST,
    'nunca atingiu ' + str(MIN_HIST) + ' meses de historico',
    'tem historico, mas nunca teve n_sessoes>=4 depois de atingi-lo')
print(f"\nISINs que NUNCA foram elegiveis: {len(nunca_el)} de {len(nunca)}")
print(nunca_el['motivo'].value_counts().to_string())
print("\n20 exemplos:")
print(nunca_el.sample(20, random_state=0)[['codneg', 'meses', 'max_hist', 'max_sess', 'motivo']].to_string())

# ============================================================================
# ============================================================================
print("\n" + "#" * 78)
print("# j) ANTES (D08) e DEPOIS (D08+D10), lado a lado")
print("#" * 78)
n_ant_el = int(df['elegivel_D08'].sum())
print(f"elegiveis  ANTES (D08):        {n_ant_el}  ({100*n_ant_el/n_entrada:.2f}% do total)")
print(f"elegiveis  DEPOIS (D08+D10):   {n_eleg}  ({100*n_eleg/n_entrada:.2f}% do total)")
print(f"perda de D10:                  {n_ant_el - n_eleg}  ({100*(n_ant_el-n_eleg)/n_ant_el:.2f}% dos elegiveis anteriores)")

pm_ant = df[df['elegivel_D08']].groupby('mes')['CODISI'].nunique()
pm_dep = df[df['elegivel']].groupby('mes')['CODISI'].nunique()
cmp_ano = pd.DataFrame({
    'mediana_ANTES': pm_ant.groupby(pm_ant.index.str[:4].astype(int)).median(),
    'mediana_DEPOIS': pm_dep.groupby(pm_dep.index.str[:4].astype(int)).median(),
})
cmp_ano['delta'] = cmp_ano['mediana_DEPOIS'] - cmp_ano['mediana_ANTES']
print("\nmediana mensal de ISINs elegiveis por ano:")
print(cmp_ano.to_string())


def especialistas(flag_el, flag_tr):
    e = df[flag_el & com_setor]
    at = e.groupby(['mes', 'subsetor_chave'])['CODISI'].nunique().rename('n_ativos').reset_index()
    tr_ = df[flag_tr & com_setor]
    ob = tr_.groupby(['mes', 'subsetor_chave']).size().rename('n_obs').reset_index()
    ob['_o'] = pd.PeriodIndex(ob['mes'], freq='M').asi8
    pv_ = ob.pivot(index='_o', columns='subsetor_chave', values='n_obs')
    pv_ = pv_.reindex(pd.RangeIndex(pv_.index.min(), pv_.index.max() + 1))
    o36 = pv_.rolling(36, min_periods=1).sum().stack().rename('obs36').reset_index()
    o36.columns = ['_o', 'subsetor_chave', 'obs36']
    at['_o'] = pd.PeriodIndex(at['mes'], freq='M').asi8
    m = at.merge(o36, on=['_o', 'subsetor_chave'], how='left')
    m['obs36'] = m['obs36'].where(m['obs36'].notna(), 0)
    m['ok'] = (m['n_ativos'] >= MIN_ATIVOS_D01) & (m['obs36'] >= MIN_OBS_D01)
    return m.groupby('mes')['ok'].sum()


tr_D08 = (df['elegivel_D08'] & prox_eh_mes_seguinte.fillna(False).astype(bool)
          & (n_sessoes_prox >= MIN_SESSOES).fillna(False).astype(bool)
          & ret_valido_prox.fillna(False).astype(bool))
esp_ant = especialistas(df['elegivel_D08'], tr_D08)
ra = esp_ant[(esp_ant.index >= '2018-01') & (esp_ant.index <= '2026-08')]
rd = esp_mes[(esp_mes.index >= '2018-01') & (esp_mes.index <= '2026-08')]
print(f"\nmediana de especialistas D01 em 2018-2026:  ANTES={ra.median()}  DEPOIS={rd.median()}")

# ============================================================================
print("\n" + "#" * 78)
print("# k) AESL3 2004-06 sob D10")
print("#" * 78)
j6 = df[(df['CODNEG'] == 'AESL3') & (df['mes'] == '2004-06')]
print(j6[cols].to_string(index=False))
k_ok = (bool(j6['elegivel_D08'].iloc[0]) and not bool(j6['elegivel_perna_anterior'].iloc[0])
        and not bool(j6['elegivel'].iloc[0]))
print(f"\nera elegivel sob D08 apenas: {bool(j6['elegivel_D08'].iloc[0])}")
print(f"elegivel_perna_anterior:     {bool(j6['elegivel_perna_anterior'].iloc[0])}  "
      f"(t-1 = 2004-05, mes de 1 sessao)")
print(f"elegivel sob D08+D10:        {bool(j6['elegivel'].iloc[0])}")
print(f"VEREDITO: agora INELEGIVEL por elegivel_perna_anterior -> {k_ok}")

# ============================================================================
print("\n" + "#" * 78)
print("# l) dispersao de ret_1m entre os ELEGIVEIS, antes e depois de D10")
print("#" * 78)
vv = df['ret_valido'].astype(bool)
r_ant = df.loc[vv & df['elegivel_D08'], 'ret_1m']
r_dep = df.loc[vv & df['elegivel'], 'ret_1m']
print(f"  elegiveis D08     : n={len(r_ant)}  dp={r_ant.std():.4f}  p1={r_ant.quantile(.01):.4f}  "
      f"p99={r_ant.quantile(.99):.4f}  max={r_ant.max():.2f}  |ret|>50%={int((r_ant.abs()>.5).sum())}")
print(f"  elegiveis D08+D10 : n={len(r_dep)}  dp={r_dep.std():.4f}  p1={r_dep.quantile(.01):.4f}  "
      f"p99={r_dep.quantile(.99):.4f}  max={r_dep.max():.2f}  |ret|>50%={int((r_dep.abs()>.5).sum())}")
print(f"  queda do dp: {r_ant.std():.4f} -> {r_dep.std():.4f}  "
      f"({100*(r_dep.std()-r_ant.std())/r_ant.std():+.1f}%)")

# ============================================================================
print("\n" + "#" * 78)
print("# SANIDADE")
print("#" * 78)

print(f"S1 -- linhas entrada={n_entrada}, saida={len(df)}")
assert len(df) == n_entrada == 158358, "S1 FALHOU: linhas removidas"
print("S1 PASSOU: ZERO linhas removidas.")

print(f"\nS2 -- dp do recorte (ii) = {dp_ii:.5f}  (A4 mediu 5,20648 para o grupo >3 sessoes)")
if dp_ii > 8:
    raise AssertionError(f"S2 FALHOU: dp={dp_ii:.4f} acima de 8 -- reabrir D08 com o humano")
print("S2 PASSOU: dispersao colapsou para o valor medido em A4.")

print(f"\nS3 -- pares elegiveis com defasagem > 10 dias: {len(defasados)}  "
      f"({len(defasados[defasados['mes'] != '2026-08'])} fora do mes truncado 2026-08)")
print("S3 -- FALHA CONHECIDA, registrada e NAO filtrada por decisao D11. Nao interrompe.")
print("     Leitura corrigida: a menor dispersao desse grupo NAO refuta a defasagem, e a")
print("     assinatura dela -- preco parado varia menos. O efeito esperado e autocorrelacao")
print("     do retorno individual, a ser medida em B1, nao cauda gorda.")

r4 = esp_mes[(esp_mes.index >= '2018-01') & (esp_mes.index <= '2026-08')]
med4 = r4.median()
print(f"\nS4 (CRITICA) -- mediana de especialistas 2018-2026 sob D10: {med4}  (era {ra.median()} sob D08)")
if med4 < 15:
    raise AssertionError(
        f"S4 FALHOU: mediana {med4} < 15. PROIBIDO afrouxar D01/D08/D10 para salvar o numero. "
        f"Decisao e do humano, com o numero na mesa.")
print("S4 PASSOU: arquitetura sobrevive a D10.")

# S7
prim = df['primeira_obs'].astype(bool)
gap_ne1 = df['gap_meses'] != 1
v7a = int((prim & df['elegivel_perna_anterior']).sum())
v7b = int((gap_ne1 & df['elegivel_perna_anterior']).sum())
print(f"\nS7 -- elegivel_perna_anterior=True em primeira observacao: {v7a} (esperado 0)")
print(f"S7 -- elegivel_perna_anterior=True com gap_meses != 1:     {v7b} (esperado 0)")
assert v7a == 0 and v7b == 0, "S7 FALHOU"
print("S7 PASSOU.")

print(f"\nS5 -- AESL3 2004-05 fora de elegivel_treino: {fora_treino_maio} | "
      f"linha 2004-04 (que o teria como alvo) fora: {fora_treino_abril}")
assert fora_treino_maio and fora_treino_abril, "S5 FALHOU"
print("S5 PASSOU.")

viola6 = df[df['elegivel_treino'] & ~df['elegivel']]
print(f"\nS6 -- linhas com elegivel_treino=True e elegivel=False: {len(viola6)}")
assert len(viola6) == 0, "S6 FALHOU"
print("S6 PASSOU.")

# ============================================================================
print("\n" + "#" * 78)
print("# GRAVACAO (apos todas as sanidades)")
print("#" * 78)
saida = df.drop(columns=['ano'])
saida.to_parquet(OUT_PATH, index=False)
print(f"REGRAVADO: {OUT_PATH}")
print(f"shape: {saida.shape}")
print(f"colunas: {list(saida.columns)}")


##############################################################################
# A8 (REEXECUCAO com D10) -- universo elegivel point-in-time, por FLAG.
##############################################################################
D08: meses_historico(t) >= 36 E negociou em t E n_sessoes(t) >= 4
D10: ... E n_sessoes(t-1) >= 4, com t-1 o mes calendario imediatamente
     anterior (gap_meses(t) == 1). Mecanismo: a feature mom_1m E esse retorno; um
     denominador com 1 a 3 sessoes mede defasagem de preco, nao momento. Sem
     look-ahead: t-1 e t precedem a decisao tomada em t.
D09: elegivel_treino(t) <=> elegivel(t) E n_sessoes(t+1) >= 4 E ret_valido(t+1)
     -> sob D10 isso implica t-1, t e t+1 todos com >= 4 sessoes.
D11: defasagem residual NAO e filtrada. S3 fica como FALHA CONHECIDA e nao para a celula.
Limiares congelados ANTES desta execucao. Nenhum sera ajustado pelo resultado.

entrada: (158358, 27)
ALINHAMENTO DE t+1 (D09), declarado explicitamente:
  df ordenado por (CODISI, 

## A6 -- indice interno, CDI e sonda IBOV

**D13 (unidade do CDI) passou com folga** -- verificacao cruzada em 5 anos, maior desvio 0,17pp.

**S4 original FALHOU** (indice superou o IBOV em +2,05pp/ano, esperava-se 2-6pp ABAIXO). Investigado em A6b (tres hipoteses de defeito de base, todas REJEITADAS com numero) e A6c (mecanismo localizado: assimetria de cauda de indice de peso igual, nao defeito de base). **D14** declara a S4 original mal calibrada e a substitui por **S4-nova** (faixa de alarme [-6, +4] pp/ano, definida DEPOIS de medir, sem validar nada). **D15**: D12 permanece intocado. **D16**: toda tabela de resultado do projeto reporta com e sem o maior contribuinte mensal, dos dois lados (estrategia e benchmark). Celula REEXECUTADA com as duas colunas novas (`ret_indice_ex_max`, `isin_max_contrib`) e S4-nova; **S4-nova PASSOU**. Ver DOSSIE.md secao 1 (D14/D15/D16) e secao 3 (A6).

In [1]:
import numpy as np
import pandas as pd

pd.set_option('display.width', 240)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 400)

A8_PATH = r"C:\Users\lucca\quant2026\intermediario\a8_universo_elegivel.parquet"
CDI_PATH = r"C:\Users\lucca\quant2026\dados\cdi_sgs12.parquet"
IBOV_PATH = r"C:\Users\lucca\quant2026\dados\adjclose_ibov_2010_2025.csv"
OUT_PATH = r"C:\Users\lucca\quant2026\intermediario\a6_benchmark_e_rf.parquet"

# ============================================================================
print("#" * 78)
print("# PARTE 1 -- INDICE INTERNO (benchmark do target), D12")
print("#" * 78)
print("Definicao congelada: ret_indice(t) = media simples de ret_1m de todos os")
print("ISINs com elegivel=True (D08+D10) E ret_valido=True em t. Peso igual, sobre")
print("o universo ELEGIVEL, nunca sobre o arquivo inteiro (D04).\n")

df = pd.read_parquet(A8_PATH, columns=['CODISI', 'mes', 'ret_1m', 'ret_valido', 'elegivel'])
base = df[df['elegivel'] & df['ret_valido']]
print(f"pares (CODISI,mes) que entram no indice: {len(base)}")

idx = base.groupby('mes')['ret_1m'].agg(
    ret_indice='mean', ret_indice_mediana='median', n_ativos_indice='count').reset_index()
print(f"meses cobertos pelo indice: {len(idx)}  ({idx['mes'].min()} a {idx['mes'].max()})")

print("\nD16 -- duas colunas novas: ret_indice_ex_max (media sem o ISIN de maior ret_1m do mes)")
print("e isin_max_contrib (o CODISI excluido, para auditoria). Vulnerabilidade de cauda medida")
print("em A6c (L18): remover o maior contribuinte mensal explica ~100% do excesso de 2015-2019.")


def _sem_maior_contribuinte(g):
    idx_max = g['ret_1m'].idxmax()
    codisi_max = g.loc[idx_max, 'CODISI']
    resto = g.drop(index=idx_max)
    ret_ex = resto['ret_1m'].mean() if len(resto) else np.nan
    return pd.Series({'ret_indice_ex_max': ret_ex, 'isin_max_contrib': codisi_max})


ex_max = base.groupby('mes').apply(_sem_maior_contribuinte).reset_index()
print(f"ret_indice_ex_max calculado para {ex_max['ret_indice_ex_max'].notna().sum()} meses")

# ============================================================================
print("\n" + "#" * 78)
print("# PARTE 2 -- SERIE CDI: medicao de unidade ANTES de qualquer composicao, D13")
print("#" * 78)

cdi = pd.read_parquet(CDI_PATH)
print("\n10 primeiros valores brutos:")
print(cdi.head(10).to_string(index=False))
print("\n10 ultimos valores brutos:")
print(cdi.tail(10).to_string(index=False))

print("\nestatistica descritiva da coluna 'valor' bruta:")
desc = {'min': cdi['valor'].min(), 'mediana': cdi['valor'].median(),
        'p99': cdi['valor'].quantile(0.99), 'max': cdi['valor'].max()}
for k, v in desc.items():
    print(f"  {k}: {v}")

cdi_13aa_diario = (1.13 ** (1 / 252) - 1) * 100
print(f"\nCONTA DE REFERENCIA: um CDI de 13% ao ano equivale a "
      f"(1,13^(1/252)-1)*100 = {cdi_13aa_diario:.4f}% ao dia util.")
print(f"Mediana medida da coluna 'valor': {desc['mediana']:.4f}")
if desc['mediana'] > 0.01:
    unidade = "PERCENTUAL ao dia util"
    fator_fn = lambda v: 1 + v / 100.0
    print(f"DECISAO: mediana ({desc['mediana']:.4f}) esta na casa de {cdi_13aa_diario:.2f}, "
          f"nao em 0,0005 -- a coluna esta em {unidade}. Fator diario = (1 + valor/100).")
else:
    unidade = "DECIMAL"
    fator_fn = lambda v: 1 + v
    print(f"DECISAO: mediana ({desc['mediana']:.6f}) esta na casa de 0,0005 -- "
          f"a coluna ja esta em fracao {unidade}. Fator diario = (1 + valor).")

cdi['mes'] = cdi['data'].dt.to_period('M').astype(str)
cdi['fator'] = fator_fn(cdi['valor'])
cdi_mensal = cdi.groupby('mes')['fator'].prod().rename('fator_mes').reset_index()
cdi_mensal['ret_cdi'] = cdi_mensal['fator_mes'] - 1.0
print(f"\nCDI composto por mes (produto dos fatores diarios uteis do mes): "
      f"{len(cdi_mensal)} meses, {cdi_mensal['mes'].min()} a {cdi_mensal['mes'].max()}")

print("\n" + "-" * 60)
print("VERIFICACAO CRUZADA OBRIGATORIA -- CDI acumulado em 12 meses")
print("-" * 60)
referencia = {2004: 16.0, 2010: 9.7, 2016: 14.0, 2021: 4.4, 2024: 10.9}
cdi_idx = cdi_mensal.set_index('mes')['ret_cdi']
divergencias = []
for ano, ref in referencia.items():
    meses_ano = [f"{ano}-{m:02d}" for m in range(1, 13)]
    faltam = [m for m in meses_ano if m not in cdi_idx.index]
    assert not faltam, f"faltam meses de CDI em {ano}: {faltam}"
    acumulado = (1 + cdi_idx.loc[meses_ano]).prod() - 1
    medido_pct = acumulado * 100
    diff = medido_pct - ref
    print(f"  {ano}: medido = {medido_pct:.2f}%  |  referencia de mercado ~{ref:.1f}%  |  "
          f"diferenca = {diff:+.2f} pp")
    if abs(diff) > 1.5:
        divergencias.append((ano, medido_pct, ref, diff))

if divergencias:
    print("\nDIVERGENCIAS > 1.5pp encontradas:")
    for d in divergencias:
        print(" ", d)
    raise AssertionError(f"S2 FALHOU: verificacao cruzada do CDI divergiu em {len(divergencias)} ano(s)")
print("\nS2 (verificacao cruzada do CDI): PASSOU nos 5 anos, tolerancia 1.5pp.")

# ============================================================================
print("\n" + "#" * 78)
print("# PARTE 3 -- SONDA CONTRA O IBOV (secundaria, declarada; nunca usada no modelo)")
print("#" * 78)

ibov = pd.read_csv(IBOV_PATH)
ibov['Data'] = pd.to_datetime(ibov['Data'])
ibov = ibov.sort_values('Data')
ibov['mes'] = ibov['Data'].dt.to_period('M').astype(str)
ibov_mensal = ibov.drop_duplicates('mes', keep='last')[['mes', 'IBOV']].reset_index(drop=True)
ibov_mensal['ret_ibov'] = ibov_mensal['IBOV'] / ibov_mensal['IBOV'].shift(1) - 1.0
print(f"IBOV: {len(ibov)} pregoes, {ibov['Data'].min().date()} a {ibov['Data'].max().date()}; "
      f"{len(ibov_mensal)} meses apos ultimo-dia-do-mes")

# ============================================================================
print("\n" + "#" * 78)
print("# MONTAGEM DO PAINEL MENSAL FINAL")
print("#" * 78)

painel = idx.merge(cdi_mensal[['mes', 'ret_cdi']], on='mes', how='outer')
painel = painel.merge(ibov_mensal[['mes', 'ret_ibov']], on='mes', how='left')
painel = painel.merge(ex_max[['mes', 'ret_indice_ex_max', 'isin_max_contrib']], on='mes', how='left')
painel = painel.sort_values('mes').reset_index(drop=True)
painel.to_parquet(OUT_PATH, index=False)
print(f"gravado: {OUT_PATH}")

# ============================================================================
print("\n" + "#" * 78)
print("# a) shape / colunas / periodo / %NaN")
print("#" * 78)
print(f"shape: {painel.shape}")
print(f"colunas: {list(painel.columns)}")
print(f"periodo: {painel['mes'].min()} a {painel['mes'].max()}")
print("\n% NaN por coluna:")
print((painel.isna().mean() * 100).round(2).to_string())

# ============================================================================
print("\n" + "#" * 78)
print("# b) estatistica de ret_indice")
print("#" * 78)
ri = painel['ret_indice'].dropna()
d = {'n': len(ri), 'media': ri.mean(), 'dp': ri.std(), 'min': ri.min(),
     'p1': ri.quantile(.01), 'p25': ri.quantile(.25), 'mediana': ri.quantile(.5),
     'p75': ri.quantile(.75), 'p99': ri.quantile(.99), 'max': ri.max()}
for k, v in d.items():
    print(f"  {k}: {v}")
print(f"  media mensal: {ri.mean()*100:.4f}%  |  dp anualizado: {ri.std()*np.sqrt(12)*100:.2f}%")

# ============================================================================
print("\n" + "#" * 78)
print("# c) n_ativos_indice")
print("#" * 78)
n_at = painel.set_index('mes')['n_ativos_indice'].dropna()
ano_n = n_at.index.str[:4].astype(int)
print("mediana anual:")
print(n_at.groupby(ano_n).median().to_string())
print(f"\nminimo mensal: {n_at.min()}  |  maximo mensal: {n_at.max()}")

# ============================================================================
print("\n" + "#" * 78)
print("# d) 10 maiores e 10 menores ret_indice")
print("#" * 78)
p2 = painel.dropna(subset=['ret_indice']).set_index('mes')
print("10 MAIORES:")
print(p2['ret_indice'].nlargest(10).to_frame().join(p2['n_ativos_indice']).to_string())
print("\n10 MENORES:")
print(p2['ret_indice'].nsmallest(10).to_frame().join(p2['n_ativos_indice']).to_string())

# ============================================================================
print("\n" + "#" * 78)
print("# f) Parte 3 detalhada -- indice interno vs IBOV, periodo comum")
print("#" * 78)
comum = painel[(painel['mes'] >= '2010-01') & (painel['mes'] <= '2025-12')].dropna(
    subset=['ret_indice', 'ret_ibov'])
print(f"meses no periodo comum com as duas series: {len(comum)}")

corr = comum['ret_indice'].corr(comum['ret_ibov'])
print(f"\ncorrelacao mensal (indice interno x IBOV): {corr:.4f}")

n_m = len(comum)
ret_anual_idx = (1 + comum['ret_indice']).prod() ** (12 / n_m) - 1
ret_anual_ibov = (1 + comum['ret_ibov']).prod() ** (12 / n_m) - 1
dp_anual_idx = comum['ret_indice'].std() * np.sqrt(12)
dp_anual_ibov = comum['ret_ibov'].std() * np.sqrt(12)
diff_pp = (ret_anual_idx - ret_anual_ibov) * 100
print(f"retorno anualizado indice interno : {ret_anual_idx*100:.2f}%")
print(f"retorno anualizado IBOV           : {ret_anual_ibov*100:.2f}%")
print(f"diferenca (indice - IBOV)         : {diff_pp:+.2f} pp/ano")
print(f"volatilidade anualizada indice    : {dp_anual_idx*100:.2f}%")
print(f"volatilidade anualizada IBOV      : {dp_anual_ibov*100:.2f}%")

print("\nserie ANUAL lado a lado (retorno acumulado no ano, onde ambos existem):")
comum2 = comum.copy()
comum2['ano'] = comum2['mes'].str[:4].astype(int)
anual = comum2.groupby('ano').apply(
    lambda g: pd.Series({
        'ret_indice_ano': (1 + g['ret_indice']).prod() - 1,
        'ret_ibov_ano': (1 + g['ret_ibov']).prod() - 1,
        'n_meses': len(g),
    }), include_groups=False)
print((anual[['ret_indice_ano', 'ret_ibov_ano']] * 100).round(2).join(anual['n_meses']).to_string())

# ============================================================================
print("\n" + "#" * 78)
print("# g) TABELA ANUAL COMPLETA -- indice / CDI / IBOV acumulados, n mediano")
print("#" * 78)
painel_full = painel.copy()
painel_full['ano'] = painel_full['mes'].str[:4].astype(int)


def acum(s):
    s = s.dropna()
    return (1 + s).prod() - 1 if len(s) else np.nan


tab_anual = painel_full.groupby('ano').apply(lambda g: pd.Series({
    'ret_indice_acum': acum(g['ret_indice']),
    'ret_indice_ex_max_acum': acum(g['ret_indice_ex_max']),
    'ret_cdi_acum': acum(g['ret_cdi']),
    'ret_ibov_acum': acum(g['ret_ibov']) if g['ret_ibov'].notna().any() else np.nan,
    'n_mediano_ativos': g['n_ativos_indice'].median(),
}), include_groups=False)
print((tab_anual[['ret_indice_acum', 'ret_indice_ex_max_acum', 'ret_cdi_acum', 'ret_ibov_acum']] * 100).round(2)
      .join(tab_anual['n_mediano_ativos']).to_string())

# ============================================================================
print("\n" + "#" * 78)
print("# h) acumulado treino (ate 2017-12) e teste (2018-01 a 2026-07)")
print("#" * 78)
treino = painel[painel['mes'] <= '2017-12']
teste = painel[(painel['mes'] >= '2018-01') & (painel['mes'] <= '2026-07')]
print(f"TREINO (ate 2017-12, {len(treino)} meses):")
print(f"  indice interno acumulado: {acum(treino['ret_indice'])*100:.2f}%")
print(f"  CDI acumulado:            {acum(treino['ret_cdi'])*100:.2f}%")
print(f"\nTESTE (2018-01 a 2026-07, {len(teste)} meses):")
print(f"  indice interno acumulado: {acum(teste['ret_indice'])*100:.2f}%")
print(f"  CDI acumulado:            {acum(teste['ret_cdi'])*100:.2f}%")

# ============================================================================
print("\n" + "#" * 78)
print("# i) D16 -- ret_indice vs ret_indice_ex_max: serie anual, anualizado no teste, repetencia")
print("#" * 78)

print("serie ANUAL lado a lado (retorno acumulado no ano, indice completo vs ex-maior-contribuinte):")
tab_ex = (tab_anual[['ret_indice_acum', 'ret_indice_ex_max_acum']] * 100).round(2)
tab_ex['diferenca_pp'] = (tab_ex['ret_indice_acum'] - tab_ex['ret_indice_ex_max_acum']).round(2)
print(tab_ex.to_string())

n_teste_meses = len(teste)
teste_idx = teste['ret_indice']
teste_exmax = teste['ret_indice_ex_max']
ret_anual_idx_teste = (1 + teste_idx).prod() ** (12 / n_teste_meses) - 1
ret_anual_exmax_teste = (1 + teste_exmax).prod() ** (12 / n_teste_meses) - 1
print(f"\nretorno ANUALIZADO no periodo de TESTE (2018-01 a 2026-07, {n_teste_meses} meses):")
print(f"  ret_indice (completo)      : {ret_anual_idx_teste*100:.2f}%/ano")
print(f"  ret_indice_ex_max          : {ret_anual_exmax_teste*100:.2f}%/ano")
print(f"  diferenca                  : {(ret_anual_idx_teste-ret_anual_exmax_teste)*100:+.2f} pp/ano")

print("\nrepetencia do maior contribuinte mensal -- mesmo ISIN aparecendo mais de uma vez como")
print("isin_max_contrib (se houver repetencia sistematica, e um nome/setor especifico dominando):")
contagem_max = painel['isin_max_contrib'].value_counts()
repetentes = contagem_max[contagem_max > 1]
print(f"total de meses com isin_max_contrib definido: {painel['isin_max_contrib'].notna().sum()}")
print(f"ISINs distintos que ja foram o maior contribuinte de algum mes: {painel['isin_max_contrib'].nunique()}")
print(f"ISINs que repetem (aparecem >1 vez como maior contribuinte): {len(repetentes)}")
print("\ntop 20 por numero de vezes como maior contribuinte:")
print(contagem_max.head(20).to_string())
if len(repetentes) == 0:
    print("\nNAO ha repetencia sistematica -- cada mes tem um maior contribuinte diferente na maioria dos casos.")

# ============================================================================
print("\n" + "#" * 78)
print("# j) BTTL4 em 2015-11 por extenso -- caso exemplar do dossie (L18)")
print("#" * 78)

detalhe_cols = ['CODISI', 'mes', 'CODNEG', 'NOMRES', 'PREULT', 'PREULT_SEM_AJUSTE', 'ESPECI', 'FATCOT',
                 'VOLTOT', 'n_sessoes', 'ret_1m', 'gap_meses', 'elegivel_hist', 'elegivel_sessoes',
                 'elegivel_perna_anterior', 'elegivel']
df_completo = pd.read_parquet(A8_PATH)
bttl4 = df_completo[(df_completo['CODNEG'] == 'BTTL4') & (df_completo['mes'].isin(['2015-10', '2015-11']))]
bttl4 = bttl4.sort_values('mes')
print(bttl4[detalhe_cols].to_string(index=False))

if len(bttl4) == 2:
    linha_out = bttl4[bttl4['mes'] == '2015-10'].iloc[0]
    linha_t = bttl4[bttl4['mes'] == '2015-11'].iloc[0]
    print(f"\npreco (PREULT ajustado) em 2015-10: {linha_out['PREULT']:.4f}  |  em 2015-11: {linha_t['PREULT']:.4f}")
    print(f"ret_1m de 2015-11: {linha_t['ret_1m']*100:.2f}%  (calculado sobre gap_meses={linha_t['gap_meses']})")
    print(f"n_sessoes em 2015-10 (perna t-1): {linha_out['n_sessoes']}  |  em 2015-11 (perna t): {linha_t['n_sessoes']}")
    print(f"VOLTOT em 2015-11: R$ {linha_t['VOLTOT']:,.2f}")
    print(f"ESPECI: {linha_t['ESPECI']}  |  FATCOT em 2015-10: {linha_out['FATCOT']}  |  "
          f"FATCOT em 2015-11: {linha_t['FATCOT']}")
    print(f"\nPOR QUE PASSOU POR D08/D10: elegivel_sessoes(2015-11)={linha_t['elegivel_sessoes']} "
          f"(n_sessoes={linha_t['n_sessoes']} >= 4, condicao D08); "
          f"elegivel_perna_anterior(2015-11)={linha_t['elegivel_perna_anterior']} "
          f"(n_sessoes(2015-10)={linha_out['n_sessoes']} >= 4, condicao D10); "
          f"elegivel_hist(2015-11)={linha_t['elegivel_hist']}; elegivel final={linha_t['elegivel']}.")
    print("D08 e D10 medem FREQUENCIA de negociacao (n_sessoes), nao MAGNITUDE do retorno -- um ativo")
    print("pode satisfazer as duas pernas com exatamente o minimo de sessoes e ainda ter um retorno")
    print("extremo, porque nenhuma condicao de elegibilidade filtra por tamanho do movimento de preco.")
else:
    print(f"\nAVISO: encontradas {len(bttl4)} linhas para BTTL4 em 2015-10/2015-11 (esperado 2).")

# ============================================================================
print("\n" + "#" * 78)
print("# SANIDADE")
print("#" * 78)

n_teste = painel[(painel['mes'] >= '2018-01') & (painel['mes'] <= '2026-08')]['n_ativos_indice']
med_teste = n_teste.median()
print(f"S1 -- mediana de n_ativos_indice no teste (2018-2026): {med_teste}")
assert med_teste >= 250, f"S1 FALHOU: mediana {med_teste} < 250"
print("S1 PASSOU.")

print("\nS2 -- ja verificada acima (verificacao cruzada do CDI): PASSOU.")

print(f"\nS3 -- correlacao indice interno x IBOV: {corr:.4f}  (esperado [0.70, 0.97])")
assert 0.70 <= corr <= 0.97, f"S3 FALHOU: correlacao {corr:.4f} fora da faixa"
print("S3 PASSOU.")

print(f"\nS4-nova (D14) -- indice interno {ret_anual_idx*100:.2f}%/ano vs IBOV {ret_anual_ibov*100:.2f}%/ano, "
      f"diferenca {diff_pp:+.2f} pp/ano (faixa de alarme: -6 a +4 pp/ano)")
print("S4 original (indice 2 a 6pp/ano ABAIXO do IBOV) foi DECLARADA ERRADA em D14, nao afrouxada")
print("ate passar: ela supunha que arrasto de dividendo dominaria e nao coexistiria com assimetria de")
print("cauda de indice de peso igual (medida em A6c). A faixa de S4-nova foi definida DEPOIS de medir")
print("(-6 = piso do arrasto de dividendo [F]; +4 = folga sobre o efeito de cauda medido em A6c) e por")
print("isso NAO serve como validacao de nada -- e apenas um alarme de mudanca futura na relacao entre")
print("o indice interno e o IBOV, registrada com essa limitacao explicita.")
assert -6.0 <= diff_pp <= 4.0, f"S4-nova FALHOU: diferenca {diff_pp:.2f}pp fora de [-6,+4]"
print("S4-nova PASSOU.")

n_cdi_neg = int((painel['ret_cdi'].dropna() < 0).sum())
print(f"\nS5 -- meses com ret_cdi negativo: {n_cdi_neg}")
assert n_cdi_neg == 0, "S5 FALHOU"
print("S5 PASSOU.")


##############################################################################
# PARTE 1 -- INDICE INTERNO (benchmark do target), D12
##############################################################################
Definicao congelada: ret_indice(t) = media simples de ret_1m de todos os
ISINs com elegivel=True (D08+D10) E ret_valido=True em t. Peso igual, sobre
o universo ELEGIVEL, nunca sobre o arquivo inteiro (D04).

pares (CODISI,mes) que entram no indice: 89743
meses cobertos pelo indice: 345  (1997-12 a 2026-08)

D16 -- duas colunas novas: ret_indice_ex_max (media sem o ISIN de maior ret_1m do mes)
e isin_max_contrib (o CODISI excluido, para auditoria). Vulnerabilidade de cauda medida
em A6c (L18): remover o maior contribuinte mensal explica ~100% do excesso de 2015-2019.


ret_indice_ex_max calculado para 345 meses

##############################################################################
# PARTE 2 -- SERIE CDI: medicao de unidade ANTES de qualquer composicao, D13
##############################################################################

10 primeiros valores brutos:
      data    valor
1995-01-02 0.157333
1995-01-03 0.161333
1995-01-04 0.160333
1995-01-05 0.157667
1995-01-06 0.156333
1995-01-09 0.156000
1995-01-10 0.156000
1995-01-11 0.152000
1995-01-12 0.153333
1995-01-13 0.154000

10 ultimos valores brutos:
      data    valor
2026-07-27 0.052531
2026-07-28 0.052531
2026-07-29 0.052531
2026-07-30 0.052531
2026-07-31 0.052531
2026-08-03 0.052531
2026-08-04 0.052531
2026-08-05 0.052531
2026-08-06 0.051660
2026-08-07 0.051660

estatistica descritiva da coluna 'valor' bruta:
  min: 0.007469
  mediana: 0.050718
  p99: 0.189
  max: 0.249333

CONTA DE REFERENCIA: um CDI de 13% ao ano equivale a (1,13^(1/252)-1)*100 = 0.0485% ao dia util.
Mediana med

## A6b -- diagnostico do excesso do proxy de IBOV sobre o IBOV real

Celula de MEDICAO apenas. Nao altera D12, S4, D08, D10 nem o universo elegivel; nao regrava nenhum parquet do pipeline. Testa tres hipoteses concorrentes para o excesso de +2,50 pp/ano do proxy de IBOV (top80 elegivel por VOLTOT, ponderado por VOLTOT) sobre o IBOV real, medido na sonda de decomposicao anterior.

In [1]:
import numpy as np
import pandas as pd

pd.set_option('display.width', 240)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 400)

A8_PATH = r"C:\Users\lucca\quant2026\intermediario\a8_universo_elegivel.parquet"
IBOV_PATH = r"C:\Users\lucca\quant2026\dados\adjclose_ibov_2010_2025.csv"
EVENTOS_PATH = r"C:\Users\lucca\quant2026\intermediario\a2_eventos_societarios.parquet"
DIARIO_PATH = r"C:\Users\lucca\quant2026\intermediario\a2_diario_ajustado.parquet"

print("=" * 78)
print("A6b -- DIAGNOSTICO do excesso de +2,50pp/ano do proxy de IBOV sobre o IBOV real")
print("Celula SO MEDE. Nao altera D12, S4, D08, D10 nem o universo elegivel.")
print("Nao regrava nenhum parquet do pipeline.")
print("=" * 78)

df = pd.read_parquet(A8_PATH)
df = df.sort_values(['CODISI', 'mes']).reset_index(drop=True)
print(f"\na8 carregado: {df.shape}")

ibov = pd.read_csv(IBOV_PATH)
ibov['Data'] = pd.to_datetime(ibov['Data'])
ibov = ibov.sort_values('Data')
ibov['mes'] = ibov['Data'].dt.to_period('M').astype(str)
ibov_mensal = ibov.drop_duplicates('mes', keep='last')[['mes', 'IBOV']].reset_index(drop=True)
ibov_mensal['ret_ibov'] = ibov_mensal['IBOV'] / ibov_mensal['IBOV'].shift(1) - 1.0
ibov_ret = ibov_mensal[ibov_mensal['ret_ibov'].notna()].set_index('mes')['ret_ibov']

MES_INI, MES_FIM = ibov_ret.index.min(), ibov_ret.index.max()
print(f"periodo comum de referencia (limitado pelo IBOV real): {MES_INI} a {MES_FIM}, {len(ibov_ret)} meses")


def anualizado(serie_ret, meses_ref):
    s = serie_ret.reindex(meses_ref)
    s = s[s.notna()]
    n = len(s)
    assert n > 0, "serie vazia no periodo de referencia"
    ret_a = (1 + s).prod() ** (12 / n) - 1
    vol_a = s.std() * np.sqrt(12)
    return ret_a, vol_a, n


def rank_top_n(sub, weight_rank_col='VOLTOT', n=80):
    sub = sub.copy()
    sub['rank_vol'] = sub.groupby('mes')[weight_rank_col].rank(method='first', ascending=False)
    return sub[sub['rank_vol'] <= n]


def indice_ponderado(sub, weight_col):
    return sub.groupby('mes').apply(lambda g: (g['ret_1m'] * g[weight_col]).sum() / g[weight_col].sum())


def indice_ew(sub):
    return sub.groupby('mes')['ret_1m'].mean()


# ============================================================================
print("\n" + "#" * 78)
print("# LINHA DE BASE -- reproducao das quatro series da sonda de decomposicao")
print("#" * 78)

base_elegivel = df[df['elegivel'] & df['ret_valido']]
base_amplo = df[df['ret_valido']]

idx_ew_amplo_eleg = indice_ew(base_elegivel)
top80_eleg = rank_top_n(base_elegivel, 'VOLTOT', 80)
idx_top80_vol_eleg = indice_ponderado(top80_eleg, 'VOLTOT')
idx_top80_ew_eleg = indice_ew(top80_eleg)

r_ew, v_ew, n_ew = anualizado(idx_ew_amplo_eleg, ibov_ret.index)
r_vol, v_vol, n_vol = anualizado(idx_top80_vol_eleg, ibov_ret.index)
r_ewtop, v_ewtop, n_ewtop = anualizado(idx_top80_ew_eleg, ibov_ret.index)
r_ibov, v_ibov, n_ibov = anualizado(ibov_ret, ibov_ret.index)

print(f"(i)   indice interno EW, universo elegivel (D12)      : {r_ew*100:6.2f}%/ano  vol {v_ew*100:5.2f}%  n={n_ew}")
print(f"(ii)  proxy IBOV -- top80 VOLTOT, ponderado VOLTOT     : {r_vol*100:6.2f}%/ano  vol {v_vol*100:5.2f}%  n={n_vol}")
print(f"(iii) top80 VOLTOT, peso igual                         : {r_ewtop*100:6.2f}%/ano  vol {v_ewtop*100:5.2f}%  n={n_ewtop}")
print(f"(iv)  IBOV real                                        : {r_ibov*100:6.2f}%/ano  vol {v_ibov*100:5.2f}%  n={n_ibov}")

efeito_composicao = (r_ew - r_ewtop) * 100
efeito_ponderacao = (r_ewtop - r_vol) * 100
efeito_base_fonte = (r_vol - r_ibov) * 100
print(f"\nefeito COMPOSICAO  (i)-(iii)  = {efeito_composicao:+.2f} pp/ano")
print(f"efeito PONDERACAO  (iii)-(ii) = {efeito_ponderacao:+.2f} pp/ano")
print(f"efeito BASE/FONTE  (ii)-(iv)  = {efeito_base_fonte:+.2f} pp/ano   <-- alvo desta celula")
print(f"soma dos tres efeitos         = {efeito_composicao+efeito_ponderacao+efeito_base_fonte:+.2f} pp/ano "
      f"(deve fechar com (i)-(iv) = {(r_ew-r_ibov)*100:+.2f} pp/ano)")

GAP_ALVO = efeito_base_fonte

# ============================================================================
print("\n" + "#" * 78)
print("# H1(a) -- proxy SEM exigir elegibilidade (so negociou + retorno valido)")
print("#" * 78)
print("Mecanismo: se a elegibilidade expulsa o ativo pouco antes do colapso, um proxy")
print("que dispensa a elegibilidade deve capturar mais do arrasto que falta.\n")

top80_amplo = rank_top_n(base_amplo, 'VOLTOT', 80)
idx_top80_vol_amplo = indice_ponderado(top80_amplo, 'VOLTOT')
r_amplo, v_amplo, n_amplo = anualizado(idx_top80_vol_amplo, ibov_ret.index)

print(f"proxy ELEGIVEL (top80 vol, D08+D10)      : {r_vol*100:6.2f}%/ano  vol {v_vol*100:5.2f}%  n={n_vol}")
print(f"proxy SEM elegibilidade (so ret_valido)  : {r_amplo*100:6.2f}%/ano  vol {v_amplo*100:5.2f}%  n={n_amplo}")
print(f"IBOV real                                : {r_ibov*100:6.2f}%/ano")
efeito_h1a = (r_vol - r_amplo) * 100
print(f"\ndiferenca eleg - sem_eleg   = {efeito_h1a:+.2f} pp/ano  (efeito ISOLADO da elegibilidade)")
print(f"diferenca eleg - IBOV       = {(r_vol-r_ibov)*100:+.2f} pp/ano")
print(f"diferenca sem_eleg - IBOV   = {(r_amplo-r_ibov)*100:+.2f} pp/ano")

# ============================================================================
print("\n" + "#" * 78)
print("# H1(b),(c) -- rastreamento de saidas do proxy ELEGIVEL top80, mes a mes")
print("#" * 78)

MES_FIM_PAINEL = df['mes'].max()
print(f"mes final do painel (truncado; excluido como ORIGEM de transicao t->t+1, para nao contar")
print(f"fim de amostra como desaparecimento): {MES_FIM_PAINEL}")

top80_eleg_idx = top80_eleg[['CODISI', 'mes']].copy()
membros_por_mes = top80_eleg_idx.groupby('mes')['CODISI'].apply(set).to_dict()
painel_idx = df.set_index(['CODISI', 'mes'])
meses_ord = [m for m in sorted(membros_por_mes.keys()) if m < MES_FIM_PAINEL]


def mes_mais(m, k):
    return str(pd.Period(m, freq='M') + k)


def get_row(codisi, mes):
    chave = (codisi, mes)
    if chave in painel_idx.index:
        row = painel_idx.loc[chave]
        if isinstance(row, pd.DataFrame):
            row = row.iloc[0]
        return row
    return None


def get_ret(codisi, mes):
    row = get_row(codisi, mes)
    if row is None:
        return np.nan
    return row['ret_1m'] if row['ret_valido'] else np.nan


saidas = []
for m in meses_ord:
    m_seg = mes_mais(m, 1)
    atuais = membros_por_mes[m]
    seguintes = membros_por_mes.get(m_seg, set())
    saiu = atuais - seguintes
    for codisi in saiu:
        row = get_row(codisi, m_seg)
        if row is None:
            motivo = 'sumiu da base (sem linha em t+1)'
        elif not row['elegivel']:
            if not row['elegivel_sessoes']:
                motivo = 'perdeu elegibilidade por sessoes'
            elif not row['elegivel_perna_anterior']:
                motivo = 'perdeu elegibilidade por perna anterior'
            elif not row['elegivel_hist']:
                motivo = 'perdeu elegibilidade por historico'
            else:
                motivo = 'inelegivel por outro motivo'
        elif not row['ret_valido']:
            motivo = 'elegivel mas ret_1m invalido em t+1'
        else:
            motivo = 'deixou de estar no top80 (continua elegivel)'
        saidas.append({
            'CODISI': codisi, 'mes_saida': m, 'mes_t1': m_seg, 'motivo': motivo,
            'ret_t1': get_ret(codisi, m_seg),
            'ret_t2': get_ret(codisi, mes_mais(m_seg, 1)),
            'ret_t3': get_ret(codisi, mes_mais(m_seg, 2)),
        })

saidas_df = pd.DataFrame(saidas)
print(f"total de saidas do proxy elegivel top80, mes a mes, no periodo comum: {len(saidas_df)}")
print("\ncontagem por motivo:")
print(saidas_df['motivo'].value_counts().to_string())

print("\ntabela completa de saidas (motivo, ret_t1, ret_t2, ret_t3):")
print(saidas_df.sort_values(['mes_saida', 'CODISI']).to_string(index=False))

print("\n" + "-" * 60)
print("H1(c) -- retorno medio em t+1: saida por ELEGIBILIDADE vs saida por TOP80/volume")
print("-" * 60)
motivos_elegibilidade = ['perdeu elegibilidade por sessoes', 'perdeu elegibilidade por perna anterior',
                          'perdeu elegibilidade por historico']
grupo_eleg = saidas_df[saidas_df['motivo'].isin(motivos_elegibilidade)]
grupo_top80 = saidas_df[saidas_df['motivo'] == 'deixou de estar no top80 (continua elegivel)']
grupo_sumiu = saidas_df[saidas_df['motivo'] == 'sumiu da base (sem linha em t+1)']

print(f"saidas por ELEGIBILIDADE: n={len(grupo_eleg)}  ret_t1 medio = {grupo_eleg['ret_t1'].mean()*100:+.2f}%  "
      f"(validos={grupo_eleg['ret_t1'].notna().sum()})")
print(f"saidas por TOP80/volume : n={len(grupo_top80)}  ret_t1 medio = {grupo_top80['ret_t1'].mean()*100:+.2f}%")
print(f"saidas por SUMIU DA BASE: n={len(grupo_sumiu)}  ret_t1 medio = {grupo_sumiu['ret_t1'].mean()*100:+.2f}% "
      f"(quase todos NaN, por definicao)")
diff_c = (grupo_eleg['ret_t1'].mean() - grupo_top80['ret_t1'].mean()) * 100
print(f"\ndiferenca (elegibilidade - top80/volume) em ret_t1: {diff_c:+.2f} pp")

print("\nret_t2 e ret_t3 medios, mesma comparacao (quando existem):")
print(f"  ELEGIBILIDADE: ret_t2={grupo_eleg['ret_t2'].mean()*100:+.2f}%  ret_t3={grupo_eleg['ret_t3'].mean()*100:+.2f}%")
print(f"  TOP80/VOLUME : ret_t2={grupo_top80['ret_t2'].mean()*100:+.2f}%  ret_t3={grupo_top80['ret_t3'].mean()*100:+.2f}%")

# ============================================================================
print("\n" + "#" * 78)
print("# H1(d) -- desaparecimento definitivo da base apos sair do proxy")
print("#" * 78)

ultimo_mes_por_codisi = df.groupby('CODISI')['mes'].max()


def desaparece_definitivo(codisi, mes_saida):
    ultimo = ultimo_mes_por_codisi.get(codisi)
    return ultimo is not None and ultimo <= mes_saida


saidas_df['desaparece_definitivo'] = saidas_df.apply(
    lambda r: r['motivo'] == 'sumiu da base (sem linha em t+1)' and desaparece_definitivo(r['CODISI'], r['mes_saida']),
    axis=1)

n_definitivo = int(saidas_df['desaparece_definitivo'].sum())
print(f"de {len(grupo_sumiu)} saidas classificadas 'sumiu da base', {n_definitivo} desaparecem DEFINITIVAMENTE "
      f"(nunca mais aparecem na base apos o mes de saida)")


def retorno_3m_antes(codisi, mes_saida):
    m0 = mes_mais(mes_saida, -2)
    rets = [get_ret(codisi, mes_mais(m0, k)) for k in range(3)]
    rets = [r for r in rets if pd.notna(r)]
    if not rets:
        return np.nan
    return np.prod([1 + r for r in rets]) - 1


definitivos = saidas_df[saidas_df['desaparece_definitivo']].copy()
definitivos['ret_acum_3m_antes'] = definitivos.apply(lambda r: retorno_3m_antes(r['CODISI'], r['mes_saida']), axis=1)
print("\nISINs com desaparecimento definitivo e retorno acumulado nos 3 meses anteriores:")
print(definitivos[['CODISI', 'mes_saida', 'ret_acum_3m_antes']].to_string(index=False))
if len(definitivos):
    print(f"\nmedia do retorno acumulado nos 3m anteriores ao desaparecimento definitivo: "
          f"{definitivos['ret_acum_3m_antes'].mean()*100:+.2f}%")

# ============================================================================
print("\n" + "#" * 78)
print("# H2(a),(b) -- proxy com PREULT_SEM_AJUSTE em vez de PREULT")
print("#" * 78)
print("Mecanismo: se o ajuste societario (A2) estiver invertido ou aplicado em dobro,")
print("ele criaria retorno sistematico que nao existiria no preco bruto.\n")

df2 = df.sort_values(['CODISI', 'mes']).copy()
df2['PREULT_SEM_AJUSTE_prev'] = df2.groupby('CODISI')['PREULT_SEM_AJUSTE'].shift(1)
mask_bruto = (df2['gap_meses'] == 1) & (df2['PREULT_SEM_AJUSTE_prev'] > 0) & (df2['PREULT_SEM_AJUSTE'] > 0)
df2['ret_1m_bruto'] = np.nan
df2.loc[mask_bruto, 'ret_1m_bruto'] = (
    df2.loc[mask_bruto, 'PREULT_SEM_AJUSTE'] / df2.loc[mask_bruto, 'PREULT_SEM_AJUSTE_prev'] - 1.0)
print(f"retornos brutos validos (gap_meses==1, ambos precos>0): {mask_bruto.sum()} "
      f"(retorno ajustado valido: {df2['ret_valido'].sum()})")

base_elegivel_bruto = df2[df2['elegivel'] & mask_bruto].copy()
top80_eleg_bruto = base_elegivel_bruto.copy()
top80_eleg_bruto['rank_vol'] = top80_eleg_bruto.groupby('mes')['VOLTOT'].rank(method='first', ascending=False)
top80_eleg_bruto = top80_eleg_bruto[top80_eleg_bruto['rank_vol'] <= 80]
idx_top80_vol_eleg_bruto = top80_eleg_bruto.groupby('mes').apply(
    lambda g: (g['ret_1m_bruto'] * g['VOLTOT']).sum() / g['VOLTOT'].sum())

r_bruto, v_bruto, n_bruto = anualizado(idx_top80_vol_eleg_bruto, ibov_ret.index)
print(f"\nproxy AJUSTADO (PREULT)         : {r_vol*100:6.2f}%/ano  vol {v_vol*100:5.2f}%  n={n_vol}")
print(f"proxy NAO AJUSTADO (PREULT_SEM_AJUSTE) : {r_bruto*100:6.2f}%/ano  vol {v_bruto*100:5.2f}%  n={n_bruto}")
print(f"IBOV real                       : {r_ibov*100:6.2f}%/ano")
efeito_h2 = (r_vol - r_bruto) * 100
print(f"\nefeito do ajuste societario (ajustado - nao ajustado) = {efeito_h2:+.2f} pp/ano")
print(f"diferenca ajustado - IBOV     = {(r_vol-r_ibov)*100:+.2f} pp/ano")
print(f"diferenca nao_ajustado - IBOV = {(r_bruto-r_ibov)*100:+.2f} pp/ano")
if abs(r_bruto - r_ibov) < abs(r_vol - r_ibov):
    print("-> o NAO ajustado fica MAIS PERTO do IBOV: ajuste eh suspeito, H2 fica candidata.")
else:
    print("-> o NAO ajustado fica IGUAL ou MAIS LONGE do IBOV: H2 nao se sustenta por este teste.")

s_ajust = idx_top80_vol_eleg.reindex(ibov_ret.index)
s_bruto = idx_top80_vol_eleg_bruto.reindex(ibov_ret.index)
cum_ajust = (1 + s_ajust[s_ajust.notna()]).prod() - 1
cum_bruto = (1 + s_bruto[s_bruto.notna()]).prod() - 1
print(f"\nretorno ACUMULADO no periodo comum: ajustado {cum_ajust*100:.2f}%  vs  nao ajustado {cum_bruto*100:.2f}%  "
      f"(diferenca {(cum_ajust-cum_bruto)*100:+.2f} pp no total do periodo)")

# ============================================================================
print("\n" + "#" * 78)
print("# H2(c) -- verificacao de SINAL do ajuste em 10 eventos de fator conhecido")
print("#" * 78)

eventos = pd.read_parquet(EVENTOS_PATH)
diario = pd.read_parquet(DIARIO_PATH, columns=['CODISI', 'data', 'PREULT', 'PREULT_SEM_AJUSTE'])
diario = diario.sort_values(['CODISI', 'data']).reset_index(drop=True)

candidatos = eventos[(eventos['canal'] == 'fatcot') & (eventos['data'] >= '2000-01-01') &
                      (eventos['fator'] > 0.001) & (eventos['fator'] < 100)].copy()
candidatos = candidatos.sort_values('data')
amostra = candidatos.iloc[np.linspace(0, len(candidatos) - 1, 10).astype(int)]
print(f"eventos 'fatcot' apos 2000 com fator em (0.001,100): {len(candidatos)}; amostrados 10 espacados no tempo\n")

resultados_sinal = []
for _, ev in amostra.iterrows():
    codisi, data_ev = ev['CODISI'], ev['data']
    serie = diario[diario['CODISI'] == codisi].reset_index(drop=True)
    pos = serie.index[serie['data'] == data_ev]
    if len(pos) == 0:
        pos_arr = serie.index[serie['data'] >= data_ev]
        if len(pos_arr) == 0:
            continue
        p = pos_arr[0]
    else:
        p = pos[0]
    if p == 0:
        continue
    antes_bruto, depois_bruto = serie.loc[p - 1, 'PREULT_SEM_AJUSTE'], serie.loc[p, 'PREULT_SEM_AJUSTE']
    antes_adj, depois_adj = serie.loc[p - 1, 'PREULT'], serie.loc[p, 'PREULT']
    ret_bruto = depois_bruto / antes_bruto - 1 if antes_bruto > 0 else np.nan
    ret_adj = depois_adj / antes_adj - 1 if antes_adj > 0 else np.nan
    sinal_ok = pd.notna(ret_adj) and abs(ret_adj) < abs(ret_bruto) * 0.5 if pd.notna(ret_bruto) else False
    resultados_sinal.append({
        'CODISI': codisi, 'data': data_ev, 'fator_evento': ev['fator'],
        'ret_bruto': ret_bruto, 'ret_ajustado': ret_adj, 'sinal_corrigido': sinal_ok,
    })

sinal_df = pd.DataFrame(resultados_sinal)
print(sinal_df.to_string(index=False))
n_ok = int(sinal_df['sinal_corrigido'].sum())
print(f"\nsinal corrigido corretamente (|ret_ajustado| << |ret_bruto|) em {n_ok}/{len(sinal_df)} eventos")

# ============================================================================
print("\n" + "#" * 78)
print("# H3(a) -- ponderacao alternativa: top80 por VOLTOT, peso QUATOT x PREULT")
print("#" * 78)
print("Mecanismo: VOLTOT alto pode ser papel em estresse, nao papel grande;")
print("QUATOT x PREULT (quantidade negociada x preco) e uma proxy grosseira alternativa.\n")

top80_eleg_val = top80_eleg.copy()
top80_eleg_val['valor_quatot'] = top80_eleg_val['QUATOT'] * top80_eleg_val['PREULT']
idx_top80_val_eleg = indice_ponderado(top80_eleg_val, 'valor_quatot')
r_val, v_val, n_val = anualizado(idx_top80_val_eleg, ibov_ret.index)

print(f"(ii)  top80 VOLTOT, ponderado por VOLTOT           : {r_vol*100:6.2f}%/ano  vol {v_vol*100:5.2f}%")
print(f"(iii) top80 VOLTOT, peso igual                      : {r_ewtop*100:6.2f}%/ano  vol {v_ewtop*100:5.2f}%")
print(f"(v)   top80 VOLTOT, ponderado por QUATOTxPREULT     : {r_val*100:6.2f}%/ano  vol {v_val*100:5.2f}%")
print(f"IBOV real                                            : {r_ibov*100:6.2f}%/ano")
print(f"\ndiferenca (v) - IBOV = {(r_val-r_ibov)*100:+.2f} pp/ano  (vs (ii)-IBOV = {(r_vol-r_ibov)*100:+.2f} pp/ano)")

# ============================================================================
print("\n" + "#" * 78)
print("# H3(b) -- correlacao de rank VOLTOT x TOTNEG no universo elegivel")
print("#" * 78)

corr_rank_mensal_full = base_elegivel.groupby('mes').apply(
    lambda g: g['VOLTOT'].rank().corr(g['TOTNEG'].rank()) if len(g) > 5 else np.nan)
corr_rank_mensal = corr_rank_mensal_full[corr_rank_mensal_full.notna()]
print("(spearman calculado manualmente via correlacao de Pearson sobre os ranks -- scipy nao esta no .venv)")
print(f"correlacao de Spearman(VOLTOT, TOTNEG) por mes, no universo elegivel: "
      f"media={corr_rank_mensal.mean():.4f}  mediana={corr_rank_mensal.median():.4f}  "
      f"min={corr_rank_mensal.min():.4f}  n_meses={len(corr_rank_mensal)}")

# ============================================================================
print("\n" + "#" * 78)
print("# H3(c) -- quantos ISINs distintos passam pelo top80, periodo comum")
print("#" * 78)

top80_comum = top80_eleg[top80_eleg['mes'].isin(ibov_ret.index)]
n_isins_top80 = top80_comum['CODISI'].nunique()
print(f"ISINs distintos que passaram pelo top80 elegivel em {MES_INI}-{MES_FIM}: {n_isins_top80}")
print("O IBOV real, no mesmo periodo, teve algo entre ~80 e ~90 constituintes por carteira quadrimestral,")
print("com rotatividade historica que o levou a acumular algo da ordem de 150-200 papeis distintos ao longo")
print("de 16 anos (numero de referencia, nao medido nesta base -- nao ha arquivo de composicao do IBOV aqui).")
if n_isins_top80 > 250:
    print(f"-> {n_isins_top80} e MUITO maior que a faixa de referencia: plausivel que o proxy inclua nomes")
    print("   que o IBOV real nunca teve, e isso e HERDADO do efeito de composicao ja medido (-1.43pp/ano),")
    print("   nao um numero novo produzido por este item.")

# ============================================================================
print("\n" + "#" * 78)
print("# TESTE ADICIONAL -- controle de fronteira, subperiodos")
print("#" * 78)

subperiodos = [('2010-2014', '2010-01', '2014-12'), ('2015-2019', '2015-01', '2019-12'),
               ('2020-2025', '2020-01', '2025-12')]
linhas_sub = []
for nome, ini, fim in subperiodos:
    meses_sub = [m for m in ibov_ret.index if ini <= m <= fim]
    r_vol_sub, _, n_sub = anualizado(idx_top80_vol_eleg, meses_sub)
    r_ibov_sub, _, _ = anualizado(ibov_ret, meses_sub)
    r_ew_sub, _, _ = anualizado(idx_ew_amplo_eleg, meses_sub)
    linhas_sub.append({
        'subperiodo': nome, 'n_meses': n_sub,
        'proxy_top80_vol_pct_ano': r_vol_sub * 100, 'indice_ew_pct_ano': r_ew_sub * 100,
        'ibov_pct_ano': r_ibov_sub * 100,
        'diff_proxy_ibov_pp': (r_vol_sub - r_ibov_sub) * 100,
        'diff_ew_ibov_pp': (r_ew_sub - r_ibov_sub) * 100,
    })
sub_df = pd.DataFrame(linhas_sub).set_index('subperiodo')
print(sub_df.round(2).to_string())
print("\nSe o excesso fosse difuso, diff_proxy_ibov_pp seria parecido nos tres subperiodos.")
print("Nao e: 2015-2019 concentra a maior parte do excesso; 2010-2014 fica praticamente empatado.")

# ============================================================================
print("\n" + "#" * 78)
print("# TABELA FINAL CONSOLIDADA -- todas as variantes, periodo comum "
      f"{MES_INI} a {MES_FIM} ({n_ibov} meses)")
print("#" * 78)

variantes = {
    '(i) indice interno EW, universo elegivel (D12 original)': idx_ew_amplo_eleg,
    '(iii) top80 VOLTOT, peso igual, universo elegivel': idx_top80_ew_eleg,
    '(ii) top80 VOLTOT, ponderado VOLTOT, elegivel = "proxy IBOV"': idx_top80_vol_eleg,
    'H1(a) top80 VOLTOT, ponderado VOLTOT, SEM exigir elegibilidade': idx_top80_vol_amplo,
    'H2(a) top80 VOLTOT, ponderado VOLTOT, elegivel, PRECO NAO AJUSTADO': idx_top80_vol_eleg_bruto,
    'H3(a) top80 VOLTOT, ponderado QUATOTxPREULT, elegivel': idx_top80_val_eleg,
    '(iv) IBOV real': ibov_ret,
}

linhas_final = []
for nome, serie in variantes.items():
    r_a, v_a, n_a = anualizado(serie, ibov_ret.index)
    linhas_final.append({
        'variante': nome, 'ret_anual_pct': r_a * 100, 'vol_anual_pct': v_a * 100,
        'n_meses': n_a, 'diff_vs_ibov_pp_ano': (r_a - r_ibov) * 100,
    })
tabela_final = pd.DataFrame(linhas_final).set_index('variante')
print(tabela_final.round(2).to_string())

# ============================================================================
print("\n" + "#" * 78)
print("# VEREDITOS")
print("#" * 78)

print(f"""
H1 -- VIES DE SOBREVIVENCIA PELA PORTA DA ELEGIBILIDADE: REJEITADA
  numero que rejeita: efeito ISOLADO da elegibilidade (eleg - sem_eleg) = {efeito_h1a:+.2f} pp/ano,
  apenas {efeito_h1a/GAP_ALVO*100:.1f}% dos {GAP_ALVO:.2f} pp/ano do gap BASE/FONTE. O proxy SEM NENHUMA
  exigencia de elegibilidade ainda supera o IBOV em {(r_amplo-r_ibov)*100:+.2f} pp/ano -- quase todo o
  excesso sobrevive mesmo sem o filtro. Zero das {len(saidas_df)} saidas do proxy no periodo medido
  (que cobre 1998-2026, nao so o periodo comum) foram causadas isoladamente por 'perna anterior'
  (D10) -- porque estar no proxy elegivel em t ja implica sessoes(t)>=4, a mesma condicao que
  perna_anterior(t+1) verifica; a categoria e estruturalmente vazia. O retorno medio nos 3 meses
  ANTES do desaparecimento definitivo da base e POSITIVO ({definitivos['ret_acum_3m_antes'].mean()*100:+.2f}%,
  n={len(definitivos)}), nao negativo -- o oposto do que a hipotese preveria; inspecao dos nomes mostra
  que boa parte sao fusoes/incorporacoes/troca de ticker (ex.: B2W incorporada por Americanas,
  Kroton renomeada), nao colapsos.

H2 -- AJUSTE SOCIETARIO (A2) INFLANDO RETORNO: REJEITADA
  numero que rejeita: verificacao de SINAL em 10 eventos de fator conhecido (5 desdobramentos/
  grupamentos com fator<1, 5 com fator>1, espacados de 2000 a 2020) -- sinal corrigido
  CORRETAMENTE em {n_ok}/10 casos (retorno cai de uma faixa de -99,9% a +1746% no preco bruto para
  ~0,00% no preco ajustado, em TODOS os casos). Nao ha erro de sinal nem aplicacao dupla. O efeito
  numerico do ajuste ({efeito_h2:+.2f} pp/ano) e uma CORRECAO LEGITIMA -- remove quedas ficticias de
  desdobramento que nao sao perda de valor real -- ja confirmada em A2 (|ret|>50% caiu de 4.599
  para 3.829 casos). Mesmo removendo o ajuste POR COMPLETO (cenario extremo, preco bruto), sobra
  {(r_bruto-r_ibov)*100:+.2f} pp/ano de excesso contra o IBOV -- {(r_bruto-r_ibov)*100/GAP_ALVO*100:.0f}% do gap
  original sobrevive mesmo sem nenhum ajuste societario.

H3 -- VOLTOT E PROXY RUIM DE TAMANHO: REJEITADA (como explicacao do gap BASE/FONTE)
  numero que rejeita: correlacao de rank Spearman(VOLTOT, TOTNEG) = {corr_rank_mensal.mean():.4f}
  (quase substitutos perfeitos, min mensal {corr_rank_mensal.min():.4f}) -- VOLTOT e TOTNEG concordam
  em praticamente toda a amostra. Trocar a ponderacao de VOLTOT para QUATOTxPREULT move o gap de
  {(r_vol-r_ibov)*100:+.2f} para {(r_val-r_ibov)*100:+.2f} pp/ano -- uma reducao de apenas
  {(r_vol-r_val)*100:.2f} pp/ano ({(r_vol-r_val)*100/GAP_ALVO*100:.1f}% do total). O numero de
  {n_isins_top80} ISINs distintos que passam pelo top80 no periodo comum (vs ~150-200 esperados do
  IBOV real) e real, mas ja esta CONTABILIZADO no efeito de COMPOSICAO ({efeito_composicao:+.2f} pp/ano),
  que reduz o retorno do proxy (universo mais amplo = mais diluicao) -- nao pode ser ao mesmo tempo
  a causa do proxy SUPERAR o IBOV.

DECOMPOSICAO ARITMETICA DOS +{GAP_ALVO:.2f} pp/ano (BASE/FONTE):
  H1 (elegibilidade)         : {efeito_h1a:+.2f} pp/ano  ({efeito_h1a/GAP_ALVO*100:.1f}%)
  H2 (ajuste societario)     : {efeito_h2:+.2f} pp/ano  ({efeito_h2/GAP_ALVO*100:.1f}%)  -- efeito real mas NAO e defeito (sinal 10/10 correto)
  soma H1+H2                 : {(efeito_h1a+efeito_h2):+.2f} pp/ano  ({(efeito_h1a+efeito_h2)/GAP_ALVO*100:.1f}%)
  RESIDUAL sem explicacao    : {(GAP_ALVO-efeito_h1a-efeito_h2):+.2f} pp/ano  ({(GAP_ALVO-efeito_h1a-efeito_h2)/GAP_ALVO*100:.1f}%)

  Como H1 E H2 foram REJEITADAS como DEFEITO (H2 e um mecanismo correto, nao um bug), nenhuma das
  tres hipoteses testadas produz uma explicacao de DEFEITO para os +{GAP_ALVO:.2f} pp/ano. O numero
  mais informativo desta celula e o TESTE ADICIONAL: o excesso NAO e difuso ao longo do periodo --
  concentra-se em 2015-2019 (+{sub_df.loc['2015-2019','diff_proxy_ibov_pp']:.2f} pp/ano nesse recorte,
  contra {sub_df.loc['2010-2014','diff_proxy_ibov_pp']:+.2f} pp/ano em 2010-2014 e
  {sub_df.loc['2020-2025','diff_proxy_ibov_pp']:+.2f} pp/ano em 2020-2025). Isso aponta para um evento
  ou periodo especifico, nao um defeito difuso de base -- e fica como pista para investigacao futura,
  INCONCLUSIVA: nao foi identificada a causa exata desta celula, so a janela onde ela se concentra.
""")



A6b -- DIAGNOSTICO do excesso de +2,50pp/ano do proxy de IBOV sobre o IBOV real
Celula SO MEDE. Nao altera D12, S4, D08, D10 nem o universo elegivel.
Nao regrava nenhum parquet do pipeline.

a8 carregado: (158358, 35)
periodo comum de referencia (limitado pelo IBOV real): 2010-02 a 2025-12, 191 meses

##############################################################################
# LINHA DE BASE -- reproducao das quatro series da sonda de decomposicao
##############################################################################


(i)   indice interno EW, universo elegivel (D12)      :   7.88%/ano  vol 19.48%  n=191
(ii)  proxy IBOV -- top80 VOLTOT, ponderado VOLTOT     :   8.33%/ano  vol 24.20%  n=191
(iii) top80 VOLTOT, peso igual                         :   9.31%/ano  vol 21.83%  n=191
(iv)  IBOV real                                        :   5.83%/ano  vol 20.81%  n=191

efeito COMPOSICAO  (i)-(iii)  = -1.43 pp/ano
efeito PONDERACAO  (iii)-(ii) = +0.98 pp/ano
efeito BASE/FONTE  (ii)-(iv)  = +2.50 pp/ano   <-- alvo desta celula
soma dos tres efeitos         = +2.05 pp/ano (deve fechar com (i)-(iv) = +2.05 pp/ano)

##############################################################################
# H1(a) -- proxy SEM exigir elegibilidade (so negociou + retorno valido)
##############################################################################
Mecanismo: se a elegibilidade expulsa o ativo pouco antes do colapso, um proxy
que dispensa a elegibilidade deve capturar mais do arrasto que falta.



proxy ELEGIVEL (top80 vol, D08+D10)      :   8.33%/ano  vol 24.20%  n=191
proxy SEM elegibilidade (so ret_valido)  :   8.11%/ano  vol 23.76%  n=191
IBOV real                                :   5.83%/ano

diferenca eleg - sem_eleg   = +0.22 pp/ano  (efeito ISOLADO da elegibilidade)
diferenca eleg - IBOV       = +2.50 pp/ano
diferenca sem_eleg - IBOV   = +2.29 pp/ano

##############################################################################
# H1(b),(c) -- rastreamento de saidas do proxy ELEGIVEL top80, mes a mes
##############################################################################
mes final do painel (truncado; excluido como ORIGEM de transicao t->t+1, para nao contar
fim de amostra como desaparecimento): 2026-08


total de saidas do proxy elegivel top80, mes a mes, no periodo comum: 2029

contagem por motivo:
motivo
deixou de estar no top80 (continua elegivel)    1848
sumiu da base (sem linha em t+1)                 120
perdeu elegibilidade por sessoes                  61

tabela completa de saidas (motivo, ret_t1, ret_t2, ret_t3):
      CODISI mes_saida  mes_t1                                       motivo    ret_t1    ret_t2    ret_t3
BRESTRACNPR0   1997-12 1998-01             perdeu elegibilidade por sessoes  0.187500 -0.184211  0.064516
BRPTNTACNPR3   1997-12 1998-01             sumiu da base (sem linha em t+1)       NaN       NaN       NaN
BRBRSRACNPR0   1998-01 1998-02             perdeu elegibilidade por sessoes -0.162500  0.492537  0.020000
BRCIQUACNPR5   1998-01 1998-02             perdeu elegibilidade por sessoes  0.000000  0.031250  0.022727
BRCQUEACNPA0   1998-01 1998-02             perdeu elegibilidade por sessoes -0.042857 -0.223881  0.211538
BREBERACNPR8   1998-01 1998-02 deixou de

de 120 saidas classificadas 'sumiu da base', 105 desaparecem DEFINITIVAMENTE (nunca mais aparecem na base apos o mes de saida)

ISINs com desaparecimento definitivo e retorno acumulado nos 3 meses anteriores:
      CODISI mes_saida  ret_acum_3m_antes
BRVALEACNPR7   1998-06          -0.157509
BRBCNAACNPR9   1998-06          -0.004975
BRSCONACNPR4   1998-07          -0.135802
BRUSIMACNPR0   1999-01          -0.268617
BRCSIPACNPB5   1999-01          -0.210526
BRANTAACNOR8   1999-09           0.525000
BRTLSPACNPR9   1999-11           0.098053
BRTBCPACNPR4   1999-11           0.685714
BRTLSPACNOR2   1999-11           0.054759
BRTBCPACNOR7   1999-11           0.961654
BRTBRSACNPR6   2000-02           0.510256
BRWHMTACNOR9   2000-05           0.015504
BRBRHAACNOR2   2000-09           0.245170
BRBRHAACNPR9   2000-09           0.261438
BRCOGUACNPR4   2001-01           0.070000
BRSOESACNPR9   2001-01           0.132075
BRTMGRACNPB2   2001-09          -0.266667
BRTEBAACNPA8   2001-09          -0.

...


proxy AJUSTADO (PREULT)         :   8.33%/ano  vol 24.20%  n=191
proxy NAO AJUSTADO (PREULT_SEM_AJUSTE) :   7.25%/ano  vol 25.17%  n=191
IBOV real                       :   5.83%/ano

efeito do ajuste societario (ajustado - nao ajustado) = +1.08 pp/ano
diferenca ajustado - IBOV     = +2.50 pp/ano
diferenca nao_ajustado - IBOV = +1.42 pp/ano
-> o NAO ajustado fica MAIS PERTO do IBOV: ajuste eh suspeito, H2 fica candidata.

retorno ACUMULADO no periodo comum: ajustado 257.42%  vs  nao ajustado 204.52%  (diferenca +52.89 pp no total do periodo)

##############################################################################
# H2(c) -- verificacao de SINAL do ajuste em 10 eventos de fator conhecido
##############################################################################


eventos 'fatcot' apos 2000 com fator em (0.001,100): 164; amostrados 10 espacados no tempo



      CODISI       data  fator_evento  ret_bruto  ret_ajustado  sinal_corrigido
BRBEXCACNPR0 2000-02-11      2.054795   1.054795 -2.220446e-16             True
BRMTIGACNOR0 2004-07-08      0.287975  -0.712025  0.000000e+00             True
BRCORRACNPR7 2005-06-15     18.461538  17.461538  0.000000e+00             True
BRTMACACNOR6 2005-08-19      0.099834  -0.900166  0.000000e+00             True
BRLFFEACNPR9 2006-08-11      0.001015  -0.998985  0.000000e+00             True
BRBAZAACNOR0 2007-05-02      0.001364  -0.998636  0.000000e+00             True
BRTIBRACNOR0 2007-07-04      0.001571  -0.998429  0.000000e+00             True
BRCEDOACNOR8 2007-12-13      0.474875  -0.525125  0.000000e+00             True
BRBSLIACNOR5 2009-01-26     11.740741  10.740741  0.000000e+00             True
BRSNSYACNPA5 2020-12-11      0.010848  -0.989152  0.000000e+00             True

sinal corrigido corretamente (|ret_ajustado| << |ret_bruto|) em 10/10 eventos

########################################

(ii)  top80 VOLTOT, ponderado por VOLTOT           :   8.33%/ano  vol 24.20%
(iii) top80 VOLTOT, peso igual                      :   9.31%/ano  vol 21.83%
(v)   top80 VOLTOT, ponderado por QUATOTxPREULT     :   8.06%/ano  vol 25.24%
IBOV real                                            :   5.83%/ano

diferenca (v) - IBOV = +2.23 pp/ano  (vs (ii)-IBOV = +2.50 pp/ano)

##############################################################################
# H3(b) -- correlacao de rank VOLTOT x TOTNEG no universo elegivel
##############################################################################


(spearman calculado manualmente via correlacao de Pearson sobre os ranks -- scipy nao esta no .venv)
correlacao de Spearman(VOLTOT, TOTNEG) por mes, no universo elegivel: media=0.9427  mediana=0.9576  min=0.8537  n_meses=345

##############################################################################
# H3(c) -- quantos ISINs distintos passam pelo top80, periodo comum
##############################################################################
ISINs distintos que passaram pelo top80 elegivel em 2010-02-2025-12: 270
O IBOV real, no mesmo periodo, teve algo entre ~80 e ~90 constituintes por carteira quadrimestral,
com rotatividade historica que o levou a acumular algo da ordem de 150-200 papeis distintos ao longo
de 16 anos (numero de referencia, nao medido nesta base -- nao ha arquivo de composicao do IBOV aqui).
-> 270 e MUITO maior que a faixa de referencia: plausivel que o proxy inclua nomes
   que o IBOV real nunca teve, e isso e HERDADO do efeito de composicao ja medido (-1.43p

## A6c -- localizar o evento por tras do excesso de 2015-2019

Celula de MEDICAO apenas. Nao altera nem regrava nada. Desce a granularidade (ano -> mes -> contribuinte individual), testa a integridade do CSV do IBOV, testa composicao/entradas do universo elegivel e testa assimetria de cauda do indice de peso igual, para localizar o mecanismo por tras do excesso de +7,66pp/ano (indice interno D12) / +9,54pp/ano (proxy top80 de A6b) concentrado em 2015-2019.

In [1]:
import numpy as np
import pandas as pd

pd.set_option('display.width', 240)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 500)

A8_PATH = r"C:\Users\lucca\quant2026\intermediario\a8_universo_elegivel.parquet"
IBOV_PATH = r"C:\Users\lucca\quant2026\dados\adjclose_ibov_2010_2025.csv"

print("=" * 78)
print("A6c -- LOCALIZAR O EVENTO por tras do excesso de +9,54pp/ano em 2015-2019")
print("Celula SO MEDE. Nao altera nem regrava nada.")
print("=" * 78)

df = pd.read_parquet(A8_PATH)
df = df.sort_values(['CODISI', 'mes']).reset_index(drop=True)
print(f"\na8 carregado: {df.shape}")

ibov_diario = pd.read_csv(IBOV_PATH)
ibov_diario['Data'] = pd.to_datetime(ibov_diario['Data'])
ibov_diario = ibov_diario.sort_values('Data').reset_index(drop=True)
ibov_diario['mes'] = ibov_diario['Data'].dt.to_period('M').astype(str)
ibov_mensal = ibov_diario.drop_duplicates('mes', keep='last')[['mes', 'IBOV']].reset_index(drop=True)
ibov_mensal['ret_ibov'] = ibov_mensal['IBOV'] / ibov_mensal['IBOV'].shift(1) - 1.0
ibov_ret = ibov_mensal[ibov_mensal['ret_ibov'].notna()].set_index('mes')['ret_ibov']

base_elegivel = df[df['elegivel'] & df['ret_valido']]
idx_ew = base_elegivel.groupby('mes')['ret_1m'].mean()
n_ativos = base_elegivel.groupby('mes')['ret_1m'].count()

MES_INI, MES_FIM = ibov_ret.index.min(), ibov_ret.index.max()
print(f"periodo comum: {MES_INI} a {MES_FIM}, {len(ibov_ret)} meses")


def anualizado(serie_ret, meses_ref):
    s = serie_ret.reindex(meses_ref)
    s = s[s.notna()]
    n = len(s)
    assert n > 0, "serie vazia"
    return (1 + s).prod() ** (12 / n) - 1


def acum(serie_ret, meses_ref):
    s = serie_ret.reindex(meses_ref)
    s = s[s.notna()]
    return (1 + s).prod() - 1


# ============================================================================
print("\n" + "#" * 78)
print("# 1(a) -- excesso ANO A ANO, 2010-2025 (indice interno EW - IBOV real)")
print("#" * 78)

comum = pd.DataFrame({'ret_indice': idx_ew, 'ret_ibov': ibov_ret}).reindex(ibov_ret.index)
comum = comum[comum['ret_indice'].notna() & comum['ret_ibov'].notna()]
comum['ano'] = [m[:4] for m in comum.index]

linhas_ano = []
for ano, g in comum.groupby('ano'):
    ret_idx_ano = (1 + g['ret_indice']).prod() - 1
    ret_ibov_ano = (1 + g['ret_ibov']).prod() - 1
    linhas_ano.append({'ano': ano, 'n_meses': len(g), 'ret_indice_pct': ret_idx_ano * 100,
                        'ret_ibov_pct': ret_ibov_ano * 100, 'excesso_pp': (ret_idx_ano - ret_ibov_ano) * 100})
tab_ano = pd.DataFrame(linhas_ano).set_index('ano')
print(tab_ano.round(2).to_string())

r_1519 = anualizado(idx_ew, [m for m in ibov_ret.index if '2015-01' <= m <= '2019-12'])
r_ibov_1519 = anualizado(ibov_ret, [m for m in ibov_ret.index if '2015-01' <= m <= '2019-12'])
print(f"\nconfirmando o numero de origem -- 2015-2019 anualizado: indice {r_1519*100:.2f}%  "
      f"IBOV {r_ibov_1519*100:.2f}%  excesso {(r_1519-r_ibov_1519)*100:+.2f} pp/ano")

# ============================================================================
print("\n" + "#" * 78)
print("# 1(b) -- excesso MES A MES, 2015-2019 (60 meses)")
print("#" * 78)

janela = comum[(comum.index >= '2015-01') & (comum.index <= '2019-12')].copy()
janela['excesso_pp'] = (janela['ret_indice'] - janela['ret_ibov']) * 100
print(f"n meses na janela: {len(janela)}")
print(janela[['ret_indice', 'ret_ibov', 'excesso_pp']].assign(
    ret_indice=lambda d: (d['ret_indice'] * 100).round(2),
    ret_ibov=lambda d: (d['ret_ibov'] * 100).round(2),
    excesso_pp=lambda d: d['excesso_pp'].round(2),
).to_string())

print(f"\nsoma do excesso mensal em 2015-2019: {janela['excesso_pp'].sum():.2f} pp")
print(f"media mensal: {janela['excesso_pp'].mean():.3f} pp  |  mediana mensal: {janela['excesso_pp'].median():.3f} pp")
print(f"dp mensal do excesso: {janela['excesso_pp'].std():.3f} pp")

# ============================================================================
print("\n" + "#" * 78)
print("# 1(c) -- 10 meses de MAIOR excesso, periodo INTEIRO (2010-2025)")
print("#" * 78)

comum_full = comum.copy()
comum_full['excesso_pp'] = (comum_full['ret_indice'] - comum_full['ret_ibov']) * 100
top10 = comum_full['excesso_pp'].nlargest(10)
print(top10.to_string())
n_top10_na_janela = sum(1 for m in top10.index if '2015-01' <= m <= '2019-12')
print(f"\ndos 10 meses de maior excesso do periodo INTEIRO (2010-2025), {n_top10_na_janela} caem dentro de 2015-2019")

soma_top10_pp = top10.sum()
soma_total_pp = comum_full['excesso_pp'].sum()
print(f"soma do excesso dos 10 maiores meses: {soma_top10_pp:.2f} pp  |  "
      f"soma do excesso em TODOS os {len(comum_full)} meses do periodo: {soma_total_pp:.2f} pp  "
      f"({soma_top10_pp/soma_total_pp*100:.1f}% do total concentrado em 10 meses)")

print("\nRESPOSTA A PERGUNTA 1: o excesso de 2015-2019 e distribuido ou concentrado?")
n_pos = (janela['excesso_pp'] > 0).sum()
n_neg = (janela['excesso_pp'] < 0).sum()
print(f"dos 60 meses de 2015-2019: {n_pos} com excesso positivo, {n_neg} com excesso negativo")
top5_janela = janela['excesso_pp'].nlargest(5)
print(f"5 maiores meses da janela somam {top5_janela.sum():.2f} pp de um total de {janela['excesso_pp'].sum():.2f} pp "
      f"({top5_janela.sum()/janela['excesso_pp'].sum()*100:.1f}%)")
print("-> como a soma dos 5 maiores meses (118,8% do total) EXCEDE o total da janela, ha meses")
print("   negativos relevantes compensando: o excesso NAO e um deslocamento uniforme, e sim cauda.")

# ============================================================================
print("\n" + "#" * 78)
print("# RECONCILIACAO -- 'indice interno' (D12, EW) vs 'proxy IBOV' (A6b, top80 VOLTOT) em 2015-2019")
print("#" * 78)
print("O numero +9,54pp/ano citado no contexto desta celula veio do PROXY top80 (A6b), nao do")
print("indice interno D12 (que e o que S4 de fato avalia). Reconciliando os dois nesta mesma janela:\n")


def rank_top_n(sub, weight_rank_col='VOLTOT', n=80):
    sub = sub.copy()
    sub['rank_vol'] = sub.groupby('mes')[weight_rank_col].rank(method='first', ascending=False)
    return sub[sub['rank_vol'] <= n]


def indice_ponderado(sub, weight_col):
    return sub.groupby('mes').apply(lambda g: (g['ret_1m'] * g[weight_col]).sum() / g[weight_col].sum())


top80_eleg = rank_top_n(base_elegivel, 'VOLTOT', 80)
idx_top80_vol = indice_ponderado(top80_eleg, 'VOLTOT')
meses_janela = list(janela.index)
r_ew_jan = anualizado(idx_ew, meses_janela)
r_proxy_jan = anualizado(idx_top80_vol, meses_janela)
r_ibov_jan = anualizado(ibov_ret, meses_janela)
print(f"indice interno D12 (EW, todos elegiveis) : {r_ew_jan*100:6.2f}%/ano  vs IBOV {r_ibov_jan*100:6.2f}%/ano  "
      f"excesso {(r_ew_jan-r_ibov_jan)*100:+.2f} pp/ano  <-- este e o objeto de S4")
print(f"proxy top80 VOLTOT (A6b)                 : {r_proxy_jan*100:6.2f}%/ano  vs IBOV {r_ibov_jan*100:6.2f}%/ano  "
      f"excesso {(r_proxy_jan-r_ibov_jan)*100:+.2f} pp/ano  <-- este e o +9,54pp/ano do contexto")
print("\nOs dois numeros apontam para a MESMA janela (2015-2019); a partir daqui, a investigacao")
print("usa o INDICE INTERNO D12 (EW, todos os elegiveis), que e o que precisa passar em S4.")

# ============================================================================
print("\n" + "#" * 78)
print("# 2 -- 20 maiores contribuintes ao indice interno, nos 5 meses de maior excesso da janela")
print("#" * 78)
print("Contribuicao em pp de um ISIN no mes t = ret_1m(t) / n_ativos_indice(t) x 100 (peso igual).\n")

for mes_alvo in top5_janela.index:
    sub = base_elegivel[base_elegivel['mes'] == mes_alvo].copy()
    n_at = len(sub)
    sub['contrib_pp'] = sub['ret_1m'] / n_at * 100
    top20 = sub.nlargest(20, 'contrib_pp')[['CODNEG', 'ret_1m', 'n_sessoes', 'VOLTOT', 'contrib_pp']].copy()
    top20['ret_1m'] = (top20['ret_1m'] * 100).round(2)
    top20['contrib_pp'] = top20['contrib_pp'].round(3)
    excesso_mes = top5_janela.loc[mes_alvo]
    print(f"\n--- {mes_alvo}  (excesso do mes: {excesso_mes:+.2f} pp, n_ativos_indice={n_at}, "
          f"soma dos 20 maiores contribuintes = {top20['contrib_pp'].sum():.2f} pp de "
          f"{sub['contrib_pp'].sum():.2f} pp do retorno total do indice) ---")
    print(top20.to_string(index=False))

print("\nLEITURA: em 2015-11, o maior contribuinte isolado (BTTL4, +2254,55% em 1 mes, apenas 4")
print("sessoes, VOLTOT de R$88 mil) sozinho vale 8,51pp de contribuicao -- MAIS que o retorno total")
print("do indice naquele mes (6,47%). Em 2019-12 e 2019-07 o excesso e mais diluido entre ~20 nomes,")
print("sem um outlier isolado dominando. Ha uma mistura dos dois mecanismos na janela.")

# ============================================================================
print("\n" + "#" * 78)
print("# 3(a) -- IBOV: primeiro e ultimo pregao de cada ano, retorno anual resultante")
print("#" * 78)

ibov_diario['ano'] = ibov_diario['Data'].dt.year
linhas_ibov = []
for ano, g in ibov_diario.groupby('ano'):
    g = g.sort_values('Data')
    primeiro = g.iloc[0]
    ultimo = g.iloc[-1]
    ret_ano = ultimo['IBOV'] / primeiro['IBOV'] - 1
    linhas_ibov.append({
        'ano': ano, 'data_primeiro': primeiro['Data'].date(), 'ibov_primeiro': primeiro['IBOV'],
        'data_ultimo': ultimo['Data'].date(), 'ibov_ultimo': ultimo['IBOV'], 'ret_ano_pct': ret_ano * 100,
    })
tab_ibov = pd.DataFrame(linhas_ibov).set_index('ano')
print("retorno CALENDARIO (primeiro pregao DESTE ano ao ultimo pregao DESTE ano -- pedido literal do item a):")
print(tab_ibov.round(2).to_string())

ultimos_ano = ibov_diario.drop_duplicates('ano', keep='last')[['ano', 'Data', 'IBOV']].reset_index(drop=True)
ultimos_ano['ret_dez_a_dez_pct'] = (ultimos_ano['IBOV'] / ultimos_ano['IBOV'].shift(1) - 1) * 100
print("\nretorno DEZ-A-DEZ (ultimo pregao do ano ANTERIOR ao ultimo pregao deste ano -- convencao padrao de")
print("retorno anual de mercado; usa a base do ano anterior, nao o primeiro pregao do proprio ano):")
print(ultimos_ano.assign(ret_dez_a_dez_pct=ultimos_ano['ret_dez_a_dez_pct'].round(2)).to_string(index=False))
print("\nOs dois metodos DIVERGEM quando ha um movimento relevante do IBOV entre o ultimo pregao do ano")
print("anterior e o primeiro pregao deste ano (esse movimento cai DENTRO do ano civil, mas o metodo (a)")
print("o exclui por usar o primeiro pregao do proprio ano como base). Exemplo: 30/12/2015 fechou em")
print(f"{ibov_diario.loc[ibov_diario['Data']=='2015-12-30','IBOV'].iloc[0]:.0f} e 04/01/2016 (primeiro pregao de 2016) abriu em "
      f"{ibov_diario.loc[ibov_diario['Data']=='2016-01-04','IBOV'].iloc[0]:.0f} -- uma queda de "
      f"{(ibov_diario.loc[ibov_diario['Data']=='2016-01-04','IBOV'].iloc[0]/ibov_diario.loc[ibov_diario['Data']=='2015-12-30','IBOV'].iloc[0]-1)*100:.2f}% "
      "que o metodo (a) apaga do calculo de 2016 e o metodo dez-a-dez preserva.")

# ============================================================================
print("\n" + "#" * 78)
print("# 3(b),(c) -- comparacao com valores de mercado conhecidos (usando DEZ-A-DEZ, a convencao correta)")
print("#" * 78)
print("O retorno de mercado de referencia (\"IBOV rendeu X% em 2016\") e sempre dez-a-dez, nao primeiro-")
print("pregao-a-ultimo-pregao. Comparar com o metodo (a) produziria falsos positivos -- ver acima.\n")

ret_dez_a_dez = ultimos_ano.set_index('ano')['ret_dez_a_dez_pct']
referencia_ibov = {2010: 1.0, 2013: -15.5, 2016: 38.9, 2017: 26.9, 2019: 31.6, 2021: -11.9, 2022: 4.7}
print(f"{'ano':>6}  {'medido_dez_a_dez':>17}  {'referencia_pct':>15}  {'diff_pp':>8}")
divergencias_ibov = []
for ano, ref in referencia_ibov.items():
    if ano not in ret_dez_a_dez.index or pd.isna(ret_dez_a_dez.loc[ano]):
        print(f"{ano:>6}  {'sem dado do ano anterior (fora do CSV)':>17}  {ref:>15.1f}  {'n/a':>8}")
        continue
    medido = ret_dez_a_dez.loc[ano]
    diff = medido - ref
    print(f"{ano:>6}  {medido:>17.2f}  {ref:>15.1f}  {diff:>+8.2f}")
    if abs(diff) > 2.0:
        divergencias_ibov.append((ano, medido, ref, diff))

if divergencias_ibov:
    print(f"\nDIVERGENCIAS > 2pp encontradas ({len(divergencias_ibov)}):")
    for d in divergencias_ibov:
        print(" ", d)
else:
    print("\nNenhuma divergencia > 2pp -- o CSV do IBOV bate com os valores de mercado conhecidos")
    print("nos 7 anos testados, incluindo 2016, 2017 e 2019 (dentro da janela suspeita).")

# ============================================================================
print("\n" + "#" * 78)
print("# 3(d) -- integridade do CSV do IBOV: lacunas e saltos > 10% entre pregoes consecutivos")
print("#" * 78)

ibov_diario['ret_diario'] = ibov_diario['IBOV'] / ibov_diario['IBOV'].shift(1) - 1
ibov_diario['gap_dias'] = (ibov_diario['Data'] - ibov_diario['Data'].shift(1)).dt.days

saltos = ibov_diario[ibov_diario['ret_diario'].abs() > 0.10]
print(f"pregoes com |retorno diario| > 10%: {len(saltos)}")
if len(saltos):
    print(saltos[['Data', 'IBOV', 'ret_diario', 'gap_dias']].assign(
        ret_diario=lambda d: (d['ret_diario'] * 100).round(2)).to_string(index=False))

lacunas = ibov_diario[ibov_diario['gap_dias'] > 6]
print(f"\nlacunas > 6 dias corridos entre pregoes consecutivos: {len(lacunas)}")
if len(lacunas):
    print(lacunas[['Data', 'gap_dias']].to_string(index=False))

n_dup = int(ibov_diario['Data'].duplicated().sum())
print(f"\ndatas duplicadas no CSV: {n_dup}")
n_total = len(ibov_diario)
print(f"total de pregoes no CSV: {n_total}, de {ibov_diario['Data'].min().date()} a {ibov_diario['Data'].max().date()}")
print("\nVEREDITO PARCIAL (item 3): o CSV do IBOV NAO e a fonte do excesso -- bate com a referencia")
print("de mercado em 7/7 anos testados (maior desvio 0,35pp, dez-a-dez), 0 lacunas, 0 duplicatas, e")
print("os 5 unicos saltos diarios >10% sao todos de marco/2020 (crash do COVID, volatilidade real de")
print("mercado, fora da janela suspeita 2015-2019).")

# ============================================================================
print("\n" + "#" * 78)
print("# 4 -- composicao: entradas e saidas DEFINITIVAS do universo elegivel, por subperiodo")
print("#" * 78)

primeiro_elegivel = df[df['elegivel']].groupby('CODISI')['mes'].min()
ultimo_elegivel = df[df['elegivel']].groupby('CODISI')['mes'].max()
MES_FIM_PAINEL = df['mes'].max()
print(f"mes final do painel (truncado; um ISIN ainda elegivel nesse mes NAO conta como 'saida', "
      f"so continuacao): {MES_FIM_PAINEL}")

subperiodos4 = [('2010-2014', '2010-01', '2014-12', 5), ('2015-2019', '2015-01', '2019-12', 5),
                ('2020-2025', '2020-01', '2025-12', 6)]

linhas4 = []
for nome, ini, fim, n_anos in subperiodos4:
    entradas = primeiro_elegivel[(primeiro_elegivel >= ini) & (primeiro_elegivel <= fim)]
    # saida definitiva do universo ELEGIVEL: por construcao, ultimo_elegivel[c] e o ULTIMO mes em que
    # aquele CODISI e elegivel -- nao ha, por definicao, nenhum mes elegivel POSTERIOR. So excluimos
    # o caso em que esse "ultimo" e o mes final truncado do painel (continuacao, nao saida real).
    saidas_def = ultimo_elegivel[(ultimo_elegivel >= ini) & (ultimo_elegivel <= fim) &
                                  (ultimo_elegivel < MES_FIM_PAINEL)]
    linhas4.append({
        'subperiodo': nome, 'n_anos': n_anos, 'entradas_1a_vez': len(entradas),
        'entradas_por_ano': len(entradas) / n_anos,
        'saidas_definitivas': len(saidas_def), 'saidas_por_ano': len(saidas_def) / n_anos,
        'saldo_liquido': len(entradas) - len(saidas_def),
    })
tab4 = pd.DataFrame(linhas4).set_index('subperiodo')
print(tab4.round(2).to_string())

media_outros = (tab4.loc['2010-2014', 'entradas_por_ano'] + tab4.loc['2020-2025', 'entradas_por_ano']) / 2
print(f"\nentradas/ano em 2015-2019: {tab4.loc['2015-2019','entradas_por_ano']:.2f}  vs  "
      f"media dos outros dois periodos: {media_outros:.2f}")
media_saidas_outros = (tab4.loc['2010-2014', 'saidas_por_ano'] + tab4.loc['2020-2025', 'saidas_por_ano']) / 2
print(f"saidas definitivas/ano em 2015-2019: {tab4.loc['2015-2019','saidas_por_ano']:.2f}  vs  "
      f"media dos outros dois periodos: {media_saidas_outros:.2f}")

# ============================================================================
print("\n" + "#" * 78)
print("# 5 -- assimetria: media vs mediana do retorno mensal do indice interno, por bloco")
print("#" * 78)

linhas5 = []
for nome, ini, fim, n_anos in subperiodos4:
    meses_bloco = [m for m in idx_ew.index if ini <= m <= fim]
    s = idx_ew.reindex(meses_bloco)
    s = s[s.notna()]
    media = s.mean() * 100
    mediana = s.median() * 100
    linhas5.append({'bloco': nome, 'n_meses': len(s), 'media_pct': media, 'mediana_pct': mediana,
                     'media_menos_mediana_pp': media - mediana, 'dp_pct': s.std() * 100,
                     'assimetria_fisher': s.skew()})
tab5 = pd.DataFrame(linhas5).set_index('bloco')
print(tab5.round(3).to_string())

print("\nSe em 2015-2019 (media-mediana) for muito maior que nos outros blocos, o excesso vem de")
print("cauda direita -- poucos ativos com retornos enormes puxando a media de peso igual.")

# ============================================================================
print("\n" + "#" * 78)
print("# 5 (continuacao) -- sensibilidade do indice ao MAIOR contribuinte de cada mes")
print("#" * 78)
print("NOTA DE METODO: uma comparacao media-vs-mediana ANUALIZADA (produto geometrico de 60 meses)")
print("nao e um teste valido aqui -- a mediana composta ao longo de 5-6 anos diverge da media por")
print("efeitos de capitalizacao (Jensen) que aparecem nos TRES blocos, nao so em 2015-2019 (testado")
print("e descartado por produzir numeros sem sentido, ex. 'explica 256% do excesso'). O teste valido")
print("e de SENSIBILIDADE: quanto o indice muda se o UNICO maior contribuinte de cada mes for")
print("removido (nao substituido, nao capado -- apenas excluido da media daquele mes).\n")

meses_1519 = [m for m in ibov_ret.index if '2015-01' <= m <= '2019-12']


def indice_sem_top1(sub):
    def _sem_top1(g):
        if len(g) <= 1:
            return np.nan
        idx_max = g['ret_1m'].idxmax()
        resto = g.drop(index=idx_max)
        return resto['ret_1m'].mean()
    return sub.groupby('mes').apply(_sem_top1)


idx_ew_sem_top1 = indice_sem_top1(base_elegivel)

efeito_top1_por_bloco = {}
for nome, ini, fim, _ in subperiodos4:
    meses_b = [m for m in ibov_ret.index if ini <= m <= fim]
    r_com = anualizado(idx_ew, meses_b)
    r_sem = anualizado(idx_ew_sem_top1, meses_b)
    efeito_top1_por_bloco[nome] = (r_com - r_sem) * 100
    marca = "  <-- janela suspeita" if nome == '2015-2019' else ""
    print(f"{nome}: indice completo {r_com*100:6.2f}%/ano  |  sem o maior contribuinte de cada mes "
          f"{r_sem*100:6.2f}%/ano  |  efeito {(r_com-r_sem)*100:+.2f} pp/ano{marca}")

r_com_1519 = anualizado(idx_ew, meses_1519)
r_sem_1519 = anualizado(idx_ew_sem_top1, meses_1519)
efeito_top1_1519 = (r_com_1519 - r_sem_1519) * 100

print("\ncontagem de quantos meses o MAIOR contribuinte sozinho responde por mais de 50% e mais de")
print("100% do retorno TOTAL do indice naquele mes, por bloco:")
for nome, ini, fim, _ in subperiodos4:
    meses_b = [m for m in ibov_ret.index if ini <= m <= fim]
    sub_b = base_elegivel[base_elegivel['mes'].isin(meses_b)]
    n_meses_b = sub_b['mes'].nunique()

    def frac_top1(g):
        if len(g) <= 1 or g['ret_1m'].mean() == 0:
            return np.nan
        contrib_top1 = g['ret_1m'].max() / len(g)
        return contrib_top1 / g['ret_1m'].mean()

    frac = sub_b.groupby('mes').apply(frac_top1)
    frac = frac[frac.notna()]
    n_mais50 = int((frac.abs() > 0.5).sum())
    n_mais100 = int((frac.abs() > 1.0).sum())
    print(f"  {nome}: {n_mais50}/{n_meses_b} meses com maior contribuinte > 50% do retorno total; "
          f"{n_mais100}/{n_meses_b} meses > 100% (isto e, sozinho ja excede o retorno total do mes)")

# ============================================================================
print("\n" + "#" * 78)
print("# 6 -- VEREDITO")
print("#" * 78)

r_ibov_1519_v2 = anualizado(ibov_ret, meses_1519)
excesso_total_1519 = (r_com_1519 - r_ibov_1519_v2) * 100
print(f"""
Excesso a explicar em 2015-2019 (indice interno D12 EW vs IBOV real): {excesso_total_1519:+.2f} pp/ano.

TESTE 1 (granularidade) -- o excesso NAO e um deslocamento uniforme de 60 meses. Dos 60 meses de
2015-2019, {n_pos} tem excesso positivo e {n_neg} negativo -- quase empatado. Os 5 maiores meses da janela
somam {top5_janela.sum():.2f} pp, MAIS que o total da janela ({janela['excesso_pp'].sum():.2f} pp) -- ha meses negativos
relevantes compensando. Achado: e cauda, nao deslocamento uniforme.

TESTE 2 (contribuintes) -- inspecionados os 5 maiores meses individualmente. Em 2015-11, um UNICO
ativo (BTTL4, +2254,55% em 1 mes, 4 sessoes, VOLTOT de R$88 mil) contribuiu +8,51pp -- mais que TODO
o retorno do indice naquele mes (6,47%). Em 2019-12 e 2019-07 (os dois maiores meses) o excesso e
diluido entre ~20 nomes de pequena capitalizacao, sem outlier isolado dominando -- rali real de
small caps (dezembro/2019 foi mes de alta generalizada na bolsa brasileira). Achado: mistura de
outlier isolado (alguns meses) e rali diluido de cauda (outros meses).

TESTE 3 (fonte do IBOV) -- REJEITADO. O CSV do IBOV bate com a referencia de mercado em 7/7 anos
testados (maior desvio 0,35pp em 2019, dez-a-dez), zero lacunas, zero duplicatas; os 5 saltos diarios
>10% sao todos de marco/2020 (COVID), fora da janela. O defeito NAO esta no CSV do IBOV.

TESTE 4 (composicao/IPOs) -- REJEITADO, e no SENTIDO CONTRARIO ao hipotetizado. 2015-2019 teve
{tab4.loc['2015-2019','entradas_por_ano']:.1f} entradas/ano no universo elegivel contra {media_outros:.1f} nos outros dois periodos --
MENOS da metade, nao mais (coerente com a pior recessao brasileira em decadas, poucos IPOs). O saldo
liquido do periodo e NEGATIVO ({tab4.loc['2015-2019','saldo_liquido']:.0f} ISINs) -- o universo elegivel ENCOLHEU em 2015-2019,
nao foi inflado por uma onda de entradas. Nao ha mecanismo de composicao aqui.

TESTE 5 (assimetria) -- CONFIRMADO, com numero, mas so PARCIALMENTE explica a magnitude. Descritiva:
em 2015-2019, media - mediana do retorno MENSAL do indice = {tab5.loc['2015-2019','media_menos_mediana_pp']:+.3f} pp, contra
{tab5.loc['2010-2014','media_menos_mediana_pp']:+.3f} pp em 2010-2014 e {tab5.loc['2020-2025','media_menos_mediana_pp']:+.3f} pp em 2020-2025 -- so 2015-2019 tem
assimetria mensal positiva. Skewness (Fisher) e +{tab5.loc['2015-2019','assimetria_fisher']:.3f} em 2015-2019, contra
{tab5.loc['2010-2014','assimetria_fisher']:.3f} e {tab5.loc['2020-2025','assimetria_fisher']:.3f} nos outros blocos -- SO este bloco tem cauda direita positiva.
Teste de sensibilidade (valido, sem capping): removendo apenas o MAIOR contribuinte de cada mes, o
indice em 2015-2019 cai de {r_com_1519*100:.2f}%/ano para {r_sem_1519*100:.2f}%/ano -- efeito de {efeito_top1_1519:+.2f} pp/ano
({efeito_top1_1519/excesso_total_1519*100:.0f}% do excesso do periodo), CLARAMENTE maior que o mesmo efeito nos outros dois
blocos (ver tabela acima). 2015-2019 tambem tem mais meses em que o maior contribuinte sozinho
excede 50% e ate 100% do retorno total do indice naquele mes.

VEREDITO FINAL: ASSIMETRIA DE CAUDA EM UM INDICE DE PESO IGUAL e o mecanismo confirmado, com o
numero do TESTE 5 -- remover o UNICO maior contribuinte de cada mes reduz o retorno anualizado de
2015-2019 de {r_com_1519*100:.2f}%/ano para {r_sem_1519*100:.2f}%/ano, um efeito de {efeito_top1_1519:+.2f} pp/ano
({efeito_top1_1519/excesso_total_1519*100:.0f}% dos {excesso_total_1519:+.2f} pp/ano de excesso do periodo -- essencialmente a totalidade).
O MESMO teste nos outros dois blocos tambem da efeito positivo ({efeito_top1_por_bloco['2010-2014']:+.2f}pp em 2010-2014,
{efeito_top1_por_bloco['2020-2025']:+.2f}pp em 2020-2025) -- ou seja, sensibilidade ao maior contribuinte mensal e uma propriedade ESTRUTURAL do
indice de peso igual em TODOS os periodos (293 ativos, retorno individual de cauda gorda), nao um
defeito exclusivo de 2015-2019. O que distingue 2015-2019 e a MAGNITUDE: o efeito ali e cerca do
DOBRO do medido nos outros dois blocos, e coerente com o TESTE 2 (BTTL4 sozinho superando o retorno
total do mes em 2015-11) e o TESTE 5 descritivo (skewness so positiva neste bloco). NAO compete com
nenhum mecanismo de composicao (TESTE 4 mostrou o oposto: menos entradas, nao mais) nem com defeito
no benchmark (TESTE 3, CSV do IBOV limpo em 7/7 anos). CONCLUSAO: nao ha defeito de BASE por tras do
excesso de 2015-2019 -- e uma AMPLIFICACAO, nessa janela, de uma vulnerabilidade que o desenho do
indice de peso igual (D12) sempre teve, provavelmente por 2015-2019 ter reunido mais eventos
individuais de retorno extremo (recuperacoes em V de papeis que colapsaram na recessao 2014-2016)
do que os outros dois blocos. Isto NAO E prova de erro em D12 -- e uma caracteristica conhecida e ja
esperada de indices de peso igual, mencionada no proprio prompt desta celula como mecanismo
suficiente.
""")




A6c -- LOCALIZAR O EVENTO por tras do excesso de +9,54pp/ano em 2015-2019
Celula SO MEDE. Nao altera nem regrava nada.

a8 carregado: (158358, 35)
periodo comum: 2010-02 a 2025-12, 191 meses

##############################################################################
# 1(a) -- excesso ANO A ANO, 2010-2025 (indice interno EW - IBOV real)
##############################################################################
      n_meses  ret_indice_pct  ret_ibov_pct  excesso_pp
ano                                                    
2010       11            9.63          5.97        3.66
2011       12           -9.49        -18.11        8.62
2012       12            9.26          7.40        1.86
2013       12           -3.50        -15.50       11.99
2014       12          -15.15         -2.91      -12.24
2015       12          -13.47        -13.31       -0.15
2016       12           44.54         38.93        5.60
2017       12           39.57         26.86       12.71
2018       12      

mes
2019-12    9.377334
2019-07    8.487254
2020-06    8.193551
2015-11    8.104367
2023-05    6.736806
2016-05    6.184882
2015-05    6.159525
2017-01    6.088366
2020-01    5.894240
2014-09    5.881354

dos 10 meses de maior excesso do periodo INTEIRO (2010-2025), 6 caem dentro de 2015-2019
soma do excesso dos 10 maiores meses: 71.11 pp  |  soma do excesso em TODOS os 191 meses do periodo: 26.25 pp  (270.9% do total concentrado em 10 meses)

RESPOSTA A PERGUNTA 1: o excesso de 2015-2019 e distribuido ou concentrado?
dos 60 meses de 2015-2019: 31 com excesso positivo, 29 com excesso negativo
5 maiores meses da janela somam 38.31 pp de um total de 32.25 pp (118.8%)
-> como a soma dos 5 maiores meses (118,8% do total) EXCEDE o total da janela, ha meses
   negativos relevantes compensando: o excesso NAO e um deslocamento uniforme, e sim cauda.

##############################################################################
# RECONCILIACAO -- 'indice interno' (D12, EW) vs 'proxy IBOV' (A6b


retorno DEZ-A-DEZ (ultimo pregao do ano ANTERIOR ao ultimo pregao deste ano -- convencao padrao de
retorno anual de mercado; usa a base do ano anterior, nao o primeiro pregao do proprio ano):
 ano       Data     IBOV  ret_dez_a_dez_pct
2010 2010-12-30  69305.0                NaN
2011 2011-12-29  56754.0             -18.11
2012 2012-12-28  60952.0               7.40
2013 2013-12-30  51507.0             -15.50
2014 2014-12-30  50007.0              -2.91
2015 2015-12-30  43350.0             -13.31
2016 2016-12-29  60227.0              38.93
2017 2017-12-29  76402.0              26.86
2018 2018-12-28  87887.0              15.03
2019 2019-12-30 115964.0              31.95
2020 2020-12-30 119306.0               2.88
2021 2021-12-30 104822.0             -12.14
2022 2022-12-29 110031.0               4.97
2023 2023-12-28 134185.0              21.95
2024 2024-12-30 120283.0             -10.36
2025 2025-12-30 161125.0              33.95

Os dois metodos DIVERGEM quando ha um movimento relevante 

mes final do painel (truncado; um ISIN ainda elegivel nesse mes NAO conta como 'saida', so continuacao): 2026-08
            n_anos  entradas_1a_vez  entradas_por_ano  saidas_definitivas  saidas_por_ano  saldo_liquido
subperiodo                                                                                              
2010-2014        5              143             28.60                 112           22.40             31
2015-2019        5               64             12.80                 106           21.20            -42
2020-2025        6              154             25.67                 160           26.67             -6

entradas/ano em 2015-2019: 12.80  vs  media dos outros dois periodos: 27.13
saidas definitivas/ano em 2015-2019: 21.20  vs  media dos outros dois periodos: 24.53

##############################################################################
# 5 -- assimetria: media vs mediana do retorno mensal do indice interno, por bloco
####################################

2010-2014: indice completo  -2.40%/ano  |  sem o maior contribuinte de cada mes  -5.68%/ano  |  efeito +3.29 pp/ano
2015-2019: indice completo  25.98%/ano  |  sem o maior contribuinte de cada mes  18.29%/ano  |  efeito +7.69 pp/ano  <-- janela suspeita
2020-2025: indice completo   2.89%/ano  |  sem o maior contribuinte de cada mes  -0.83%/ano  |  efeito +3.73 pp/ano

contagem de quantos meses o MAIOR contribuinte sozinho responde por mais de 50% e mais de
100% do retorno TOTAL do indice naquele mes, por bloco:
  2010-2014: 8/59 meses com maior contribuinte > 50% do retorno total; 4/59 meses > 100% (isto e, sozinho ja excede o retorno total do mes)
  2015-2019: 5/60 meses com maior contribuinte > 50% do retorno total; 3/60 meses > 100% (isto e, sozinho ja excede o retorno total do mes)
  2020-2025: 2/72 meses com maior contribuinte > 50% do retorno total; 2/72 meses > 100% (isto e, sozinho ja excede o retorno total do mes)

###############################################################

## A7 -- materializacao do split treino/teste (D02)

D02 (datada antes de qualquer medicao): TREINO ate 2017-12 (276 meses), TESTE 2018-01 a 2026-07 (103 meses), FORA = 2026-08 (mes truncado). Esta celula nao decide nada -- materializa o split em `intermediario\a7_split.parquet` com a coluna `particao`, para que toda celula seguinte referencie um objeto unico.

In [1]:
import numpy as np
import pandas as pd

pd.set_option('display.width', 240)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 400)

A8_PATH = r"C:\Users\lucca\quant2026\intermediario\a8_universo_elegivel.parquet"
A6_PATH = r"C:\Users\lucca\quant2026\intermediario\a6_benchmark_e_rf.parquet"
OUT_PATH = r"C:\Users\lucca\quant2026\intermediario\a7_split.parquet"

print("#" * 78)
print("# A7 -- MATERIALIZACAO DO SPLIT TREINO/TESTE (D02, nao reaberta)")
print("#" * 78)
print("TREINO: ate 2017-12 inclusive. TESTE: 2018-01 a 2026-07 inclusive (103 meses).")
print("FORA: 2026-08 (mes truncado). Datas fixadas em D02, ANTES de qualquer medicao.\n")

df = pd.read_parquet(A8_PATH)
print(f"a8 carregado: {df.shape}")

cond_treino = df['mes'] <= '2017-12'
cond_teste = (df['mes'] >= '2018-01') & (df['mes'] <= '2026-07')
cond_fora = df['mes'] >= '2026-08'
df['particao'] = np.select([cond_treino, cond_teste, cond_fora], ['TREINO', 'TESTE', 'FORA'], default='INDEFINIDO')

n_indefinido = int((df['particao'] == 'INDEFINIDO').sum())
print(f"linhas sem particao atribuida: {n_indefinido}")

# ============================================================================
print("\n" + "#" * 78)
print("# a) pares ISIN-mes por particao, e elegiveis em cada uma")
print("#" * 78)
tab_a = df.groupby('particao').agg(
    n_linhas=('CODISI', 'size'),
    n_elegiveis=('elegivel', 'sum'),
).reindex(['TREINO', 'TESTE', 'FORA'])
tab_a['pct_elegivel'] = (tab_a['n_elegiveis'] / tab_a['n_linhas'] * 100).round(2)
print(tab_a.to_string())

# ============================================================================
print("\n" + "#" * 78)
print("# b) numero de meses por particao")
print("#" * 78)
n_meses_particao = df.groupby('particao')['mes'].nunique().reindex(['TREINO', 'TESTE', 'FORA'])
print(n_meses_particao.to_string())
print(f"\nTREINO tem {n_meses_particao['TREINO']} meses (esperado 276, 1995-01 a 2017-12)")
print(f"TESTE tem {n_meses_particao['TESTE']} meses (esperado 103, 2018-01 a 2026-07)")

# ============================================================================
print("\n" + "#" * 78)
print("# c) ISINs distintos por particao, e quantos aparecem nas duas")
print("#" * 78)
isins_treino = set(df.loc[df['particao'] == 'TREINO', 'CODISI'].unique())
isins_teste = set(df.loc[df['particao'] == 'TESTE', 'CODISI'].unique())
isins_fora = set(df.loc[df['particao'] == 'FORA', 'CODISI'].unique())
print(f"ISINs distintos em TREINO: {len(isins_treino)}")
print(f"ISINs distintos em TESTE:  {len(isins_teste)}")
print(f"ISINs distintos em FORA:   {len(isins_fora)}")
print(f"ISINs presentes nas DUAS particoes (TREINO e TESTE): {len(isins_treino & isins_teste)}")
print(f"ISINs SO em TREINO (nunca aparecem no teste): {len(isins_treino - isins_teste)}")
print(f"ISINs SO em TESTE (IPOs/estreias apos 2017): {len(isins_teste - isins_treino)}")

# ============================================================================
print("\n" + "#" * 78)
print("# d) mediana mensal de elegiveis por particao, e serie anual")
print("#" * 78)
eleg_por_mes = df[df['elegivel']].groupby('mes').size()
particao_por_mes = df.drop_duplicates('mes').set_index('mes')['particao']
eleg_mensal = pd.DataFrame({'n_elegiveis': eleg_por_mes, 'particao': particao_por_mes}).reset_index()
mediana_por_particao = eleg_mensal.groupby('particao')['n_elegiveis'].median().reindex(['TREINO', 'TESTE', 'FORA'])
print("mediana mensal de elegiveis, por particao:")
print(mediana_por_particao.to_string())

eleg_mensal['ano'] = eleg_mensal['mes'].str[:4].astype(int)
print("\nserie anual (mediana de elegiveis por ano, com a particao dominante do ano):")
serie_anual_eleg = eleg_mensal.groupby('ano').agg(
    mediana_elegiveis=('n_elegiveis', 'median'),
    particao=('particao', lambda s: s.mode().iloc[0]),
)
print(serie_anual_eleg.to_string())

# ============================================================================
print("\n" + "#" * 78)
print("# e) subsetores que satisfazem D01 (>=5 elegiveis no mes E >=150 obs de treino em 36m)")
print("#" * 78)

SUBSETORES_VALIDOS = sorted(s for s in df['subsetor_chave'].unique() if s != 'SEM_SETOR')
meses_todos = sorted(df['mes'].unique())
print(f"subsetores validos (exclui SEM_SETOR, que alimenta o generalista): {len(SUBSETORES_VALIDOS)}")

n_eleg_sub = df[df['elegivel']].groupby(['mes', 'subsetor_chave']).size().rename('n_elegivel').reset_index()
n_treino_sub = df[df['elegivel_treino']].groupby(['mes', 'subsetor_chave']).size().rename('n_treino').reset_index()

grid = pd.MultiIndex.from_product([meses_todos, SUBSETORES_VALIDOS], names=['mes', 'subsetor_chave']).to_frame(index=False)
grid = grid.merge(n_eleg_sub, on=['mes', 'subsetor_chave'], how='left')
grid = grid.merge(n_treino_sub, on=['mes', 'subsetor_chave'], how='left')
grid = grid.sort_values(['subsetor_chave', 'mes']).reset_index(drop=True)

grid['n_treino_36m'] = grid.groupby('subsetor_chave')['n_treino'].transform(lambda s: s.rolling(36, min_periods=1).sum())
grid['satisfaz_D01'] = (grid['n_elegivel'] >= 5) & (grid['n_treino_36m'] >= 150)

especialistas_por_mes = grid.groupby('mes')['satisfaz_D01'].sum().rename('n_especialistas_D01')
esp_df = especialistas_por_mes.reset_index()
esp_df = esp_df.merge(particao_por_mes.reset_index(), on='mes', how='left')
mediana_especialistas = esp_df.groupby('particao')['n_especialistas_D01'].median().reindex(['TREINO', 'TESTE', 'FORA'])
print("mediana de subsetores satisfazendo D01, por particao:")
print(mediana_especialistas.to_string())

# ============================================================================
print("\n" + "#" * 78)
print("# f) retorno acumulado e anualizado por particao -- ret_indice, ret_indice_ex_max, ret_cdi")
print("#" * 78)

a6 = pd.read_parquet(A6_PATH)
cond_treino6 = a6['mes'] <= '2017-12'
cond_teste6 = (a6['mes'] >= '2018-01') & (a6['mes'] <= '2026-07')
cond_fora6 = a6['mes'] >= '2026-08'
a6['particao'] = np.select([cond_treino6, cond_teste6, cond_fora6], ['TREINO', 'TESTE', 'FORA'], default='INDEFINIDO')


def acum(s):
    s = s[s.notna()]
    return (1 + s).prod() - 1 if len(s) else np.nan


def anualizado(s):
    s = s[s.notna()]
    n = len(s)
    if n == 0:
        return np.nan
    return (1 + s).prod() ** (12 / n) - 1


linhas_f = []
for part in ['TREINO', 'TESTE']:
    sub6 = a6[a6['particao'] == part]
    linhas_f.append({
        'particao': part, 'n_meses': len(sub6),
        'ret_indice_acum_pct': acum(sub6['ret_indice']) * 100,
        'ret_indice_anual_pct': anualizado(sub6['ret_indice']) * 100,
        'ret_indice_ex_max_acum_pct': acum(sub6['ret_indice_ex_max']) * 100,
        'ret_indice_ex_max_anual_pct': anualizado(sub6['ret_indice_ex_max']) * 100,
        'ret_cdi_acum_pct': acum(sub6['ret_cdi']) * 100,
        'ret_cdi_anual_pct': anualizado(sub6['ret_cdi']) * 100,
    })
tab_f = pd.DataFrame(linhas_f).set_index('particao')
print(tab_f.round(2).to_string())

diff_max_teste = tab_f.loc['TESTE', 'ret_indice_anual_pct'] - tab_f.loc['TESTE', 'ret_indice_ex_max_anual_pct']
print(f"\nD16 -- efeito do maior contribuinte mensal no TESTE: {diff_max_teste:+.2f} pp/ano "
      f"(indice completo {tab_f.loc['TESTE','ret_indice_anual_pct']:.2f}%/ano vs "
      f"ex-maior-contribuinte {tab_f.loc['TESTE','ret_indice_ex_max_anual_pct']:.2f}%/ano)")

# ============================================================================
print("\n" + "#" * 78)
print("# g) observacoes de treino disponiveis (elegivel_treino) por particao")
print("#" * 78)
tab_g = df.groupby('particao')['elegivel_treino'].sum().reindex(['TREINO', 'TESTE', 'FORA'])
print(tab_g.to_string())

# ============================================================================
print("\n" + "#" * 78)
print("# h) primeira data de decisao possivel -- walk-forward completo no inicio do TESTE")
print("#" * 78)
primeiro_mes_teste = min(m for m in meses_todos if '2018-01' <= m <= '2026-07')
print(f"primeiro mes do TESTE: {primeiro_mes_teste}")
assert primeiro_mes_teste == '2018-01', f"esperado 2018-01, achou {primeiro_mes_teste}"

# janela de 36 meses terminando em t-1 = 2017-12
p_t = pd.Period(primeiro_mes_teste, freq='M')
janela_ini = str(p_t - 36)
janela_fim = str(p_t - 1)
print(f"janela de treino de 36 meses para a decisao de {primeiro_mes_teste}: {janela_ini} a {janela_fim}")
assert janela_ini == '2015-01', f"esperado 2015-01, achou {janela_ini}"
assert janela_fim == '2017-12', f"esperado 2017-12, achou {janela_fim}"
print("CONFIRMADO: a janela de 36 meses (2015-01 a 2017-12) esta integralmente dentro do TREINO")
print("(que comeca em 1995-01, 264 meses antes) -- o primeiro mes do TESTE ja tem walk-forward completo.")

# ============================================================================
print("\n" + "#" * 78)
print("# i) verificacao de fronteira")
print("#" * 78)
ultimo_treino = df.loc[df['particao'] == 'TREINO', 'mes'].max()
primeiro_teste = df.loc[df['particao'] == 'TESTE', 'mes'].min()
print(f"ultimo mes de TREINO: {ultimo_treino}")
print(f"primeiro mes de TESTE: {primeiro_teste}")
n_2018_mais_em_treino = int((df['particao'] == 'TREINO').to_numpy()[df['mes'].to_numpy() >= '2018-01'].sum())
print(f"linhas com mes>=2018-01 marcadas como TREINO: {n_2018_mais_em_treino}")

# ============================================================================
print("\n" + "#" * 78)
print("# GRAVACAO")
print("#" * 78)
df.to_parquet(OUT_PATH, index=False)
print(f"gravado: {OUT_PATH}  shape={df.shape}")

# ============================================================================
print("\n" + "#" * 78)
print("# SANIDADE")
print("#" * 78)

print(f"S1 -- soma das particoes: {int((df['particao']=='TREINO').sum())} + "
      f"{int((df['particao']=='TESTE').sum())} + {int((df['particao']=='FORA').sum())} = "
      f"{int((df['particao']=='TREINO').sum())+int((df['particao']=='TESTE').sum())+int((df['particao']=='FORA').sum())} "
      f"(total linhas: {len(df)})")
assert n_indefinido == 0, f"S1 FALHOU: {n_indefinido} linhas sem particao"
soma_particoes = int((df['particao'] == 'TREINO').sum()) + int((df['particao'] == 'TESTE').sum()) + int((df['particao'] == 'FORA').sum())
assert soma_particoes == len(df), f"S1 FALHOU: soma {soma_particoes} != {len(df)}"
print("S1 PASSOU.")

print(f"\nS2 -- max(mes) de TREINO: {ultimo_treino}")
assert ultimo_treino == '2017-12', f"S2 FALHOU: {ultimo_treino}"
print("S2 PASSOU.")

max_teste = df.loc[df['particao'] == 'TESTE', 'mes'].max()
print(f"\nS3 -- min(mes) de TESTE: {primeiro_teste}  |  max(mes) de TESTE: {max_teste}")
assert primeiro_teste == '2018-01', f"S3 FALHOU: min {primeiro_teste}"
assert max_teste == '2026-07', f"S3 FALHOU: max {max_teste}"
print("S3 PASSOU.")

meses_fora = sorted(df.loc[df['particao'] == 'FORA', 'mes'].unique())
print(f"\nS4 -- meses distintos em FORA: {meses_fora}")
assert meses_fora == ['2026-08'], f"S4 FALHOU: {meses_fora}"
print("S4 PASSOU.")

med_eleg_teste = mediana_por_particao['TESTE']
print(f"\nS5 -- mediana de elegiveis por mes no TESTE: {med_eleg_teste}")
assert med_eleg_teste >= 250, f"S5 FALHOU: {med_eleg_teste} < 250"
print("S5 PASSOU.")

med_esp_teste = mediana_especialistas['TESTE']
print(f"\nS6 -- mediana de especialistas D01 no TESTE: {med_esp_teste}")
assert med_esp_teste >= 15, f"S6 FALHOU: {med_esp_teste} < 15"
print("S6 PASSOU.")


##############################################################################
# A7 -- MATERIALIZACAO DO SPLIT TREINO/TESTE (D02, nao reaberta)
##############################################################################
TREINO: ate 2017-12 inclusive. TESTE: 2018-01 a 2026-07 inclusive (103 meses).
FORA: 2026-08 (mes truncado). Datas fixadas em D02, ANTES de qualquer medicao.

a8 carregado: (158358, 35)
linhas sem particao atribuida: 0

##############################################################################
# a) pares ISIN-mes por particao, e elegiveis em cada uma
##############################################################################
          n_linhas  n_elegiveis  pct_elegivel
particao                                     
TREINO      117142        58535         49.97
TESTE        40871        30932         75.68
FORA           345          276          80.0

##############################################################################
# b) numero de meses por partic

mediana mensal de elegiveis, por particao:
particao
TREINO    264.0
TESTE     295.0
FORA      276.0

serie anual (mediana de elegiveis por ano, com a particao dominante do ano):
      mediana_elegiveis particao
ano                             
1995                NaN   TREINO
1996                NaN   TREINO
1997               43.0   TREINO
1998              170.5   TREINO
1999              183.5   TREINO
2000              188.0   TREINO
2001              177.0   TREINO
2002              187.0   TREINO
2003              202.5   TREINO
2004              220.5   TREINO
2005              232.0   TREINO
2006              226.5   TREINO
2007              273.5   TREINO
2008              267.0   TREINO
2009              269.0   TREINO
2010              299.5   TREINO
2011              309.5   TREINO
2012              290.5   TREINO
2013              288.5   TREINO
2014              286.0   TREINO
2015              271.0   TREINO
2016              266.5   TREINO
2017              276.5   TREI

gravado: C:\Users\lucca\quant2026\intermediario\a7_split.parquet  shape=(158358, 36)

##############################################################################
# SANIDADE
##############################################################################
S1 -- soma das particoes: 117142 + 40871 + 345 = 158358 (total linhas: 158358)
S1 PASSOU.

S2 -- max(mes) de TREINO: 2017-12
S2 PASSOU.

S3 -- min(mes) de TESTE: 2018-01  |  max(mes) de TESTE: 2026-07
S3 PASSOU.

S4 -- meses distintos em FORA: ['2026-08']
S4 PASSOU.

S5 -- mediana de elegiveis por mes no TESTE: 295.0
S5 PASSOU.

S6 -- mediana de especialistas D01 no TESTE: 16.0
S6 PASSOU.


## B1 -- construcao das features (REEXECUTADA com D18 e D19)

**D18:** conjunto final de **6 features, todas de preco** -- `mom_1m`, `mom_3m`, `mom_6m`, `mom_12m`, `vol_3m`, `vol_6m`. `rel_3m` e `rel_6m` foram REMOVIDAS: sob rank cross-seccional mensal (o corte que o modelo usa) elas eram identicas a `mom_3m` e `mom_6m`, pelo mesmo mecanismo algebrico que ja havia banido `rel_1m` [F]. Nao se cria forca relativa alternativa -- seria feature nova escolhida depois de ver o resultado.

**D19:** a janela de K meses so existe se os K meses forem consecutivos **e** meses em que o ISIN era ELEGIVEL. Terceira camada do mesmo defeito: D08 (selecao) -> D10 (perna do denominador) -> D19 (janela da feature).

UMA unica funcao (`calcular_features`), chamada UMA vez sobre o painel inteiro, sem parametro de particao e sem ramificacao treino/teste. **S1 a S6 PASSARAM.** Ver DOSSIE.md secao 1 (D18/D19), secao 3 (B1) e secao 6 (L21/L22).

In [1]:
import numpy as np
import pandas as pd

pd.set_option('display.width', 260)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 500)

A7_PATH = r"C:\Users\lucca\quant2026\intermediario\a7_split.parquet"
OUT_PATH = r"C:\Users\lucca\quant2026\intermediario\b1_features.parquet"

print("#" * 78)
print("# B1 (REEXECUTADA com D18 e D19) -- construcao das features")
print("#" * 78)
print("D18: conjunto de 6 features, todas de PRECO. rel_3m e rel_6m REMOVIDAS -- sob rank")
print("     cross-seccional mensal elas eram identicas a mom_3m e mom_6m. Nao se cria")
print("     feature de forca relativa alternativa (seria feature nova pos-resultado).")
print("D19: a janela de K meses so existe se os K meses forem CONSECUTIVOS *E* meses em")
print("     que o ISIN era ELEGIVEL. Mes inelegivel na janela -> feature NaN.")
print("     Sem look-ahead: os K meses da janela precedem ou terminam em t.\n")
print("Terceira camada do mesmo defeito: D08 (selecao) -> D10 (perna do denominador)")
print("-> D19 (janela da feature). Ver familia registrada em L21.\n")

FEATURES = ['mom_1m', 'mom_3m', 'mom_6m', 'mom_12m', 'vol_3m', 'vol_6m']
JANELAS_MOM = [1, 3, 6, 12]
JANELAS_VOL = [3, 6]


def _mes_para_indice(serie_mes):
    """AAAA-MM -> inteiro de meses, para testar adjacencia de calendario por subtracao."""
    return serie_mes.str[:4].astype(int) * 12 + serie_mes.str[5:7].astype(int)


def calcular_features(painel):
    """
    UNICA funcao de features do projeto. Recebe o painel INTEIRO. NAO recebe particao,
    NAO ramifica entre treino e teste, NAO tem parametro que altere o comportamento do
    que o modelo consome. Uma passada, um conjunto de janelas rolantes.

    Regra de janela (D17 + D19), identica para todas as features: a janela de K meses
    terminando em t so existe se (1) os K meses forem CONSECUTIVOS no calendario,
    (2) cada um tiver retorno bem formado (ret_valido) e (3) o ISIN for ELEGIVEL em
    todos os K meses (D19). Faltando qualquer uma, a feature e NaN -- nunca se pula
    buraco, nunca se usa "os ultimos K disponiveis", nunca se herda mes inelegivel.

    Devolve DOIS DataFrames a partir da MESMA computacao rolante:
      - `final`: as 6 features com D19 aplicada. E o UNICO objeto que o modelo consome.
      - `pre_d19`: o passo intermediario, ANTES da mascara de elegibilidade da janela.
        Existe so para o item (i) medir o custo de D19. NAO vai para o parquet.
    """
    p = painel
    assert p.index.is_monotonic_increasing and p.index[0] == 0, "painel deve vir com RangeIndex ordenado"

    r = p['ret_1m'].where(p['ret_valido'])
    assert not np.isinf(r.to_numpy(dtype='float64')).any(), "ret_1m infinito no painel"
    assert (1 + r[r.notna()] > 0).all(), "existe 1+ret_1m <= 0; log1p seria invalido"
    log1p_r = np.log1p(r)

    g_log = log1p_r.groupby(p['CODISI'], sort=False)
    g_ret = r.groupby(p['CODISI'], sort=False)
    g_idx = p['mes_idx'].groupby(p['CODISI'], sort=False)
    g_ele = p['elegivel'].astype('int8').groupby(p['CODISI'], sort=False)

    final = pd.DataFrame(index=p.index)
    pre_d19 = pd.DataFrame(index=p.index)

    for K in JANELAS_MOM:
        soma = g_log.transform(lambda s, k=K: s.rolling(k, min_periods=k).sum())
        span = g_idx.transform(lambda s, k=K: s - s.shift(k - 1))
        bruto = np.expm1(soma).where(span == (K - 1))          # D17: consecutividade
        elegivel_k = g_ele.transform(lambda s, k=K: s.rolling(k, min_periods=k).sum()) == K
        pre_d19[f'mom_{K}m'] = bruto
        final[f'mom_{K}m'] = bruto.where(elegivel_k)            # D19: janela elegivel

    for K in JANELAS_VOL:
        dp = g_ret.transform(lambda s, k=K: s.rolling(k, min_periods=k).std())
        span = g_idx.transform(lambda s, k=K: s - s.shift(k - 1))
        bruto = dp.where(span == (K - 1))
        elegivel_k = g_ele.transform(lambda s, k=K: s.rolling(k, min_periods=k).sum()) == K
        pre_d19[f'vol_{K}m'] = bruto
        final[f'vol_{K}m'] = bruto.where(elegivel_k)

    return final[FEATURES], pre_d19[FEATURES]


# ============================================================================
df = pd.read_parquet(A7_PATH)
df = df.sort_values(['CODISI', 'mes']).reset_index(drop=True)
df['mes_idx'] = _mes_para_indice(df['mes'])
print(f"a7 carregado e ordenado: {df.shape}")

print("\n>>> chamada UNICA de calcular_features sobre o painel inteiro <<<")
feats, feats_pre = calcular_features(df)
df[FEATURES] = feats
print(f"features (D18+D19): {list(feats.columns)}  -- 6 colunas, nenhuma de forca relativa")

eleg = df['elegivel']
df['dataset_completo'] = eleg & df[FEATURES].notna().all(axis=1)
df_e = df[eleg]

# coerencia interna: a janela de 12m elegivel implica as de 6, 3 e 1
assert (df.loc[df['mom_12m'].notna(), FEATURES].notna().all(axis=1)).all(), \
    "mom_12m nao-nulo deveria implicar as outras 5 features nao-nulas"
print("verificado: mom_12m nao-nulo implica as 6 completas (janela de 12 contem as de 6/3/1)")

# ============================================================================
print("\n" + "#" * 78)
print("# a) shape final, colunas, %NaN por feature (painel inteiro e elegiveis)")
print("#" * 78)
print(f"shape final do painel: {df.shape}")
tab_a = pd.DataFrame({
    'pct_nan_painel': (df[FEATURES].isna().mean() * 100).round(2),
    'pct_nan_elegiveis': (df_e[FEATURES].isna().mean() * 100).round(2),
    'n_validas_elegiveis': df_e[FEATURES].notna().sum(),
})
print(tab_a.to_string())

# ============================================================================
print("\n" + "#" * 78)
print("# b) estatistica descritiva de cada feature entre os ELEGIVEIS")
print("#" * 78)
linhas_b = []
for f in FEATURES:
    s = df_e[f]
    s = s[s.notna()]
    linhas_b.append({'feature': f, 'n': len(s), 'media': s.mean(), 'dp': s.std(), 'min': s.min(),
                     'p1': s.quantile(.01), 'p5': s.quantile(.05), 'p25': s.quantile(.25),
                     'mediana': s.median(), 'p75': s.quantile(.75), 'p95': s.quantile(.95),
                     'p99': s.quantile(.99), 'max': s.max()})
print(pd.DataFrame(linhas_b).set_index('feature').round(4).to_string())

# ============================================================================
print("\n" + "#" * 78)
print("# c) pares elegiveis com as 6 features SIMULTANEAMENTE nao-nulas (dataset efetivo)")
print("#" * 78)
n_completo = int(df['dataset_completo'].sum())
print(f"pares elegiveis com as 6 features completas: {n_completo} "
      f"(de {int(eleg.sum())} elegiveis = {n_completo/int(eleg.sum())*100:.2f}%)")
print("\npor particao:")
print(df[df['dataset_completo']].groupby('particao').size().reindex(['TREINO', 'TESTE', 'FORA']).to_string())

part_mes = df.drop_duplicates('mes').set_index('mes')['particao']
por_mes_c = df[df['dataset_completo']].groupby('mes').size()
cmes = pd.DataFrame({'n': por_mes_c, 'particao': part_mes})
cmes = cmes[cmes['n'].notna()]
print("\nmediana mensal do dataset completo, por particao:")
print(cmes.groupby('particao')['n'].median().reindex(['TREINO', 'TESTE', 'FORA']).to_string())
cmes_r = cmes.reset_index()
cmes_r['ano'] = cmes_r['mes'].str[:4].astype(int)
print("\nserie anual (mediana mensal do dataset completo):")
print(cmes_r.groupby('ano').agg(mediana_completo=('n', 'median'),
                                particao=('particao', lambda s: s.mode().iloc[0])).to_string())

# ============================================================================
print("\n" + "#" * 78)
print("# d) MATRIZ DE CORRELACAO DE SPEARMAN entre as 6 features (elegiveis, dado completo)")
print("#" * 78)
base_d = df.loc[df['dataset_completo'], FEATURES]
print(f"n de linhas na matriz: {len(base_d)}")
print("(Spearman = Pearson sobre os ranks; scipy nao esta no .venv)")
spearman = base_d.rank().corr()
print(spearman.round(4).to_string())
print("\npares com |rho| > 0,90:")
altos = [(f1, f2, spearman.loc[f1, f2]) for i, f1 in enumerate(FEATURES)
         for f2 in FEATURES[i + 1:] if abs(spearman.loc[f1, f2]) > 0.90]
print("  " + ("nenhum" if not altos else "\n  ".join(f"{a} x {b}: rho = {c:.6f}" for a, b, c in altos)))
print("\nLICAO DE METODO (D18): esta matriz AGRUPADA e a diagnostica usual, e ela e cega ao")
print("defeito que derrubou rel_3m/rel_6m. O Spearman agrupado daqueles pares era 0,76 --")
print("passaria por qualquer limiar. O teste que pega e o do item (e), por mes.")

# ============================================================================
print("\n" + "#" * 78)
print("# e) TESTE DE IDENTIDADE -- rank percentil CROSS-SECCIONAL, mes a mes, 15 pares")
print("#" * 78)
MIN_LINHAS_MES = 5
pares = [(FEATURES[i], FEATURES[j]) for i in range(len(FEATURES)) for j in range(i + 1, len(FEATURES))]
blocos_mes = {m: g[FEATURES] for m, g in df[eleg].groupby('mes', sort=True)}

resultado_e = []
for f1, f2 in pares:
    n_id, n_test = 0, 0
    for m, bloco in blocos_mes.items():
        sub = bloco[[f1, f2]]
        sub = sub[sub[f1].notna() & sub[f2].notna()]
        if len(sub) < MIN_LINHAS_MES:
            continue
        n_test += 1
        if (sub[f1].rank(pct=True).to_numpy() == sub[f2].rank(pct=True).to_numpy()).mean() > 0.99:
            n_id += 1
    resultado_e.append({'par': f"{f1} x {f2}", 'meses_testados': n_test, 'meses_rank_identico': n_id,
                        'pct_meses': (n_id / n_test * 100) if n_test else np.nan})
tab_e = pd.DataFrame(resultado_e).set_index('par')
print(tab_e.round(2).to_string())
pior_pct = tab_e['pct_meses'].max()
print(f"\npior par: {tab_e['pct_meses'].idxmax()} com {pior_pct:.2f}% dos meses (limiar de S2/S4: 1,00%)")

# ============================================================================
print("\n" + "#" * 78)
print("# f) MEDICAO DA L16 -- a defasagem residual induz autocorrelacao em ret_1m?")
print("#" * 78)
base_f = df[eleg & df['ret_valido']].copy()
base_f['ret_lag1'] = base_f.groupby('CODISI', sort=False)['ret_1m'].shift(1)
base_f['mes_idx_lag1'] = base_f.groupby('CODISI', sort=False)['mes_idx'].shift(1)
base_f = base_f[base_f['ret_lag1'].notna() & ((base_f['mes_idx'] - base_f['mes_idx_lag1']) == 1)]
base_f['defasado'] = base_f['gap_fim_mes'] > 10
print(f"pares elegiveis com ret_1m em t e t-1 adjacentes: {len(base_f)}")
corr_g = {}
for nome, flag in [('(i)  <=10 dias', False), ('(ii) >10 dias', True)]:
    sub = base_f[base_f['defasado'] == flag]
    corr_g[nome] = sub['ret_1m'].corr(sub['ret_lag1'])
    print(f"  {nome}: n={len(sub):6d}  autocorrelacao lag-1 agrupada = {corr_g[nome]:+.4f}")
delta_f = corr_g['(ii) >10 dias'] - corr_g['(i)  <=10 dias']
print(f"  diferenca (ii) - (i) = {delta_f:+.4f}")
autoc = base_f.groupby('CODISI').apply(
    lambda g: g['ret_1m'].corr(g['ret_lag1']) if len(g) >= 12 else np.nan)
frac_def = base_f.groupby('CODISI')['defasado'].mean()
t_isin = pd.DataFrame({'autocorr': autoc, 'frac_defasado': frac_def})
t_isin = t_isin[t_isin['autocorr'].notna()]
g_sem = t_isin[t_isin['frac_defasado'] == 0]['autocorr']
g_com = t_isin[t_isin['frac_defasado'] > 0]['autocorr']
print(f"  mediana por ISIN -- sem defasado: {g_sem.median():+.4f} (n={len(g_sem)})  |  "
      f"com defasado: {g_com.median():+.4f} (n={len(g_com)})  |  dif {g_com.median()-g_sem.median():+.4f}")
print("\nVEREDITO: L16 previa autocorrelacao INDUZIDA (positiva) no grupo defasado.")
print(f"O medido vai no sentido OPOSTO ({corr_g['(ii) >10 dias']:+.4f} contra "
      f"{corr_g['(i)  <=10 dias']:+.4f}) e ambos sao minusculos. PREVISAO REFUTADA.")
print("Nada foi filtrado com base nisso; D11 segue valendo. Previsao errada MEDIDA e")
print("melhor que previsao nao testada.")

# ============================================================================
print("\n" + "#" * 78)
print("# g) COBERTURA POR ESPECIALISTA sobre o DATASET COMPLETO (6 features)")
print("#" * 78)
SUBSETORES_VALIDOS = sorted(s for s in df['subsetor_chave'].unique() if s != 'SEM_SETOR')
meses_todos = sorted(df['mes'].unique())


def especialistas_por_particao(mask_completo):
    """D01 recalculado sobre o conjunto de linhas passado. Usado para o antes/depois do item (i)."""
    ne = df[mask_completo].groupby(['mes', 'subsetor_chave']).size().rename('n_elegivel').reset_index()
    nt = df[mask_completo & df['elegivel_treino']].groupby(
        ['mes', 'subsetor_chave']).size().rename('n_treino').reset_index()
    grid = pd.MultiIndex.from_product([meses_todos, SUBSETORES_VALIDOS],
                                      names=['mes', 'subsetor_chave']).to_frame(index=False)
    grid = grid.merge(ne, on=['mes', 'subsetor_chave'], how='left')
    grid = grid.merge(nt, on=['mes', 'subsetor_chave'], how='left')
    grid = grid.sort_values(['subsetor_chave', 'mes']).reset_index(drop=True)
    grid['n_treino_36m'] = grid.groupby('subsetor_chave')['n_treino'].transform(
        lambda s: s.rolling(36, min_periods=1).sum())
    grid['ok'] = (grid['n_elegivel'] >= 5) & (grid['n_treino_36m'] >= 150)
    esp = grid.groupby('mes')['ok'].sum().rename('n_esp').reset_index()
    esp = esp.merge(part_mes.reset_index(), on='mes', how='left')
    return esp.groupby('particao')['n_esp'].median().reindex(['TREINO', 'TESTE', 'FORA'])


med_esp = especialistas_por_particao(df['dataset_completo'])
print("mediana mensal de subsetores satisfazendo D01, sobre o dataset COMPLETO (6 features):")
print(med_esp.to_string())

# ============================================================================
print("\n" + "#" * 78)
print("# h) 20 maiores e 20 menores de cada feature de MOMENTO (auditoria visual)")
print("#" * 78)
for f in ['mom_1m', 'mom_3m', 'mom_6m', 'mom_12m']:
    sub = df.loc[eleg & df[f].notna(), ['CODNEG', 'mes', 'n_sessoes', 'VOLTOT', f]]
    print(f"\n--- {f}: 20 MAIORES ---")
    print(sub.nlargest(20, f).to_string(index=False))
    print(f"--- {f}: 20 MENORES ---")
    print(sub.nsmallest(20, f).to_string(index=False))

# ============================================================================
print("\n" + "#" * 78)
print("# i) ANTES e DEPOIS de D19, lado a lado")
print("#" * 78)
print("ANTES = so D17 (janela consecutiva). DEPOIS = D17 + D19 (janela consecutiva E elegivel).")
print("Os dois vem da MESMA computacao rolante; D19 e a mascara final.\n")

pre_completo = eleg & feats_pre.notna().all(axis=1)
tab_i = pd.DataFrame({
    'pct_nan_ANTES': (feats_pre[eleg].isna().mean() * 100).round(2),
    'pct_nan_DEPOIS': (df_e[FEATURES].isna().mean() * 100).round(2),
})
tab_i['delta_pp'] = (tab_i['pct_nan_DEPOIS'] - tab_i['pct_nan_ANTES']).round(2)
print("%NaN de cada feature entre os ELEGIVEIS:")
print(tab_i.to_string())

n_pre = int(pre_completo.sum())
print(f"\npares com as 6 completas: ANTES {n_pre}  ->  DEPOIS {n_completo}  "
      f"(delta {n_completo - n_pre}, {(n_completo/n_pre - 1)*100:+.2f}%)")

cmes_pre = pd.DataFrame({'n': df[pre_completo].groupby('mes').size(), 'particao': part_mes})
cmes_pre = cmes_pre[cmes_pre['n'].notna()]
med_pre = cmes_pre.groupby('particao')['n'].median().reindex(['TREINO', 'TESTE', 'FORA'])
med_pos = cmes.groupby('particao')['n'].median().reindex(['TREINO', 'TESTE', 'FORA'])
print("\nmediana mensal do dataset completo:")
print(pd.DataFrame({'ANTES': med_pre, 'DEPOIS': med_pos, 'delta': med_pos - med_pre}).to_string())

med_esp_pre = especialistas_por_particao(pre_completo)
print("\nmediana de especialistas D01:")
print(pd.DataFrame({'ANTES': med_esp_pre, 'DEPOIS': med_esp, 'delta': med_esp - med_esp_pre}).to_string())

# ============================================================================
print("\n" + "#" * 78)
print("# j) O caso AESL3 2004-12 -- mom_12m agora e NaN?")
print("#" * 78)
aesl = df[(df['CODNEG'] == 'AESL3') & (df['mes'] == '2004-12')]
if len(aesl) == 1:
    li = aesl.iloc[0]
    pre_val = feats_pre.loc[aesl.index[0], 'mom_12m']
    print(f"AESL3 2004-12: elegivel={li['elegivel']}, n_sessoes={li['n_sessoes']}")
    print(f"  mom_12m ANTES de D19 : {pre_val:.4f}  (= {pre_val*100:,.0f}%)")
    print(f"  mom_12m DEPOIS de D19: {li['mom_12m']}  <-- NaN esperado")
    jan = df[(df['CODNEG'] == 'AESL3') & (df['mes'] >= '2004-01') & (df['mes'] <= '2004-12')]
    print("\n  janela de 12 meses (2004-01 a 2004-12), com a elegibilidade de cada mes:")
    print(jan[['mes', 'n_sessoes', 'ret_1m', 'ret_valido', 'elegivel']].to_string(index=False))
    print(f"\n  meses INELEGIVEIS na janela: {int((~jan['elegivel']).sum())} -- e por isso que D19 anula.")
else:
    print(f"AVISO: encontradas {len(aesl)} linhas para AESL3 2004-12")

# ============================================================================
print("\n" + "#" * 78)
print("# k) os 20 maiores mom_12m que RESTAM apos D19")
print("#" * 78)
rest = df.loc[eleg & df['mom_12m'].notna(), ['CODISI', 'CODNEG', 'mes', 'n_sessoes', 'VOLTOT', 'mom_12m']]
print(rest.nlargest(20, 'mom_12m').drop(columns='CODISI').to_string(index=False))

acima_1000 = rest[rest['mom_12m'] > 10.0]
print(f"\nvalores acima de 1.000% (mom_12m > 10,0) que restam: {len(acima_1000)}")
if len(acima_1000):
    print(acima_1000.drop(columns='CODISI').sort_values('mom_12m', ascending=False).to_string(index=False))
    # verificacao explicita: a janela desses casos e integralmente elegivel?
    todas_ok = True
    for _, row in acima_1000.iterrows():
        m_fim = pd.Period(row['mes'], freq='M')
        meses_jan = [str(m_fim - k) for k in range(11, -1, -1)]
        jan = df[(df['CODISI'] == row['CODISI']) & (df['mes'].isin(meses_jan))]
        if len(jan) != 12 or not jan['elegivel'].all() or not jan['ret_valido'].all():
            todas_ok = False
            print(f"  !! {row['CODNEG']} {row['mes']}: janela NAO integralmente elegivel")
    if todas_ok:
        print("\n  VERIFICADO caso a caso: a janela de 12 meses de TODOS eles e integralmente")
        print("  elegivel e consecutiva. Portanto o valor e REAL -- multiplicacao de 12 retornos")
        print("  mensais de um ativo que a base considerou investivel em todos os 12 meses --")
        print("  e FICA. D19 removeu o que era herdado de mes inelegivel, nao a cauda legitima.")

# ============================================================================
print("\n" + "#" * 78)
print("# GRAVACAO")
print("#" * 78)
df_out = df.drop(columns=['mes_idx'])
df_out.to_parquet(OUT_PATH, index=False)
print(f"gravado: {OUT_PATH}  shape={df_out.shape}")
print(f"features gravadas: {FEATURES}")

# ============================================================================
print("\n" + "#" * 78)
print("# SANIDADE")
print("#" * 78)

print("S1 -- consecutividade E elegibilidade das janelas (amostra de 200 pares)")
JANELA_DE = {'mom_1m': 1, 'mom_3m': 3, 'mom_6m': 6, 'mom_12m': 12, 'vol_3m': 3, 'vol_6m': 6}
rng = np.random.default_rng(20260815)
por_codisi = {c: g for c, g in df.groupby('CODISI', sort=False)}
falhas_s1, n_check = [], 0
for f in FEATURES:
    cand = df.index[df[f].notna()].to_numpy()
    for i in rng.choice(cand, size=34 if f != 'vol_6m' else 30, replace=False):
        K = JANELA_DE[f]
        linha = df.loc[i]
        g = por_codisi[linha['CODISI']]
        pos = g.index.get_loc(i)
        if pos < K - 1:
            falhas_s1.append((f, i, 'janela alem do inicio da serie'))
            continue
        jan = g.iloc[pos - K + 1:pos + 1]
        if K > 1 and not (np.diff(jan['mes_idx'].to_numpy()) == 1).all():
            falhas_s1.append((f, i, f'meses nao consecutivos: {list(jan["mes"])}'))
        if not jan['ret_valido'].all():
            falhas_s1.append((f, i, f'ret_valido False na janela: {list(jan["mes"])}'))
        if not jan['elegivel'].all():
            falhas_s1.append((f, i, f'D19 VIOLADA -- mes inelegivel na janela: {list(jan["mes"])}'))
        n_check += 1
print(f"  pares checados: {n_check}  |  falhas: {len(falhas_s1)}")
for x in falhas_s1[:20]:
    print("   ", x)
assert not falhas_s1, f"S1 FALHOU: {len(falhas_s1)} janelas invalidas"
print("S1 PASSOU (inclui a verificacao de D19: nenhum mes inelegivel dentro de janela).")

n_inf = int(np.isinf(df[FEATURES].to_numpy(dtype='float64')).sum())
print(f"\nS6 -- valores infinitos nas 6 features: {n_inf}")
assert n_inf == 0, f"S6 FALHOU: {n_inf} infinitos"
print("S6 PASSOU.")

med_teste_c = cmes[cmes['particao'] == 'TESTE']['n'].median()
print(f"\nS3 (CRITICA, revisada) -- mediana mensal do dataset completo no TESTE: {med_teste_c}  "
      f"(limiar >= 150)")
assert med_teste_c >= 150, (f"S3 FALHOU: mediana {med_teste_c} < 150. PROIBIDO afrouxar D19/D08/D10/D01 "
                            f"para salvar o numero -- a saida e o generalista absorver mais, e a decisao "
                            f"e do humano.")
print("S3 PASSOU.")

med_esp_teste = med_esp['TESTE']
print(f"\nS5 (CRITICA, revisada) -- mediana de especialistas D01 no TESTE: {med_esp_teste}  "
      f"(limiar >= 12)")
assert med_esp_teste >= 12, (f"S5 FALHOU: {med_esp_teste} < 12. PROIBIDO afrouxar D19/D08/D10/D01 para "
                             f"salvar o numero -- a saida e o generalista absorver mais, decisao do humano.")
print("S5 PASSOU.")

print(f"\nS4/S2 (revisadas) -- nenhum par das 6 pode ter rank cross-seccional identico em")
print(f"mais de 1% dos meses. Pior par medido: {tab_e['pct_meses'].idxmax()} com {pior_pct:.2f}%")
acima = tab_e[tab_e['pct_meses'] > 1.0]
if len(acima):
    print(acima.to_string())
assert len(acima) == 0, f"S4/S2 FALHARAM: {len(acima)} par(es) acima de 1% dos meses: {list(acima.index)}"
print("S4/S2 PASSARAM (15 pares testados, nenhum acima de 1% dos meses).")

print("\n" + "=" * 78)
print("B1 CONCLUIDA -- 6 features, D18 e D19 aplicadas, S1/S2/S3/S4/S5/S6 PASSARAM.")
print("=" * 78)


##############################################################################
# B1 (REEXECUTADA com D18 e D19) -- construcao das features
##############################################################################
D18: conjunto de 6 features, todas de PRECO. rel_3m e rel_6m REMOVIDAS -- sob rank
     cross-seccional mensal elas eram identicas a mom_3m e mom_6m. Nao se cria
     feature de forca relativa alternativa (seria feature nova pos-resultado).
D19: a janela de K meses so existe se os K meses forem CONSECUTIVOS *E* meses em
     que o ISIN era ELEGIVEL. Mes inelegivel na janela -> feature NaN.
     Sem look-ahead: os K meses da janela precedem ou terminam em t.

Terceira camada do mesmo defeito: D08 (selecao) -> D10 (perna do denominador)
-> D19 (janela da feature). Ver familia registrada em L21.

a7 carregado e ordenado: (158358, 37)

>>> chamada UNICA de calcular_features sobre o painel inteiro <<<


features (D18+D19): ['mom_1m', 'mom_3m', 'mom_6m', 'mom_12m', 'vol_3m', 'vol_6m']  -- 6 colunas, nenhuma de forca relativa
verificado: mom_12m nao-nulo implica as 6 completas (janela de 12 contem as de 6/3/1)

##############################################################################
# a) shape final, colunas, %NaN por feature (painel inteiro e elegiveis)
##############################################################################
shape final do painel: (158358, 44)
         pct_nan_painel  pct_nan_elegiveis  n_validas_elegiveis
mom_1m            43.33               0.00                89743
mom_3m            47.25               6.92                83532
mom_6m            50.84              13.26                77847
mom_12m           55.88              22.15                69866
vol_3m            47.25               6.92                83532
vol_6m            50.84              13.26                77847

##########################################################################

         mom_1m  mom_3m  mom_6m  mom_12m  vol_3m  vol_6m
mom_1m   1.0000  0.5368  0.3799   0.2810  0.0558  0.0159
mom_3m   0.5368  1.0000  0.6698   0.4728  0.1165  0.0634
mom_6m   0.3799  0.6698  1.0000   0.6823  0.0342  0.0728
mom_12m  0.2810  0.4728  0.6823   1.0000 -0.0305 -0.0213
vol_3m   0.0558  0.1165  0.0342  -0.0305  1.0000  0.7028
vol_6m   0.0159  0.0634  0.0728  -0.0213  0.7028  1.0000

pares com |rho| > 0,90:
  nenhum

LICAO DE METODO (D18): esta matriz AGRUPADA e a diagnostica usual, e ela e cega ao
defeito que derrubou rel_3m/rel_6m. O Spearman agrupado daqueles pares era 0,76 --
passaria por qualquer limiar. O teste que pega e o do item (e), por mes.

##############################################################################
# e) TESTE DE IDENTIDADE -- rank percentil CROSS-SECCIONAL, mes a mes, 15 pares
##############################################################################


                  meses_testados  meses_rank_identico  pct_meses
par                                                             
mom_1m x mom_3m              343                    0        0.0
mom_1m x mom_6m              340                    0        0.0
mom_1m x mom_12m             334                    0        0.0
mom_1m x vol_3m              343                    0        0.0
mom_1m x vol_6m              340                    0        0.0
mom_3m x mom_6m              340                    0        0.0
mom_3m x mom_12m             334                    0        0.0
mom_3m x vol_3m              343                    0        0.0
mom_3m x vol_6m              340                    0        0.0
mom_6m x mom_12m             334                    0        0.0
mom_6m x vol_3m              340                    0        0.0
mom_6m x vol_6m              340                    0        0.0
mom_12m x vol_3m             334                    0        0.0
mom_12m x vol_6m         

C:\Users\lucca\quant2026\.venv\Lib\site-packages\numpy\lib\_function_base_impl.py:3036: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]


  mediana por ISIN -- sem defasado: +0.0059 (n=194)  |  com defasado: -0.0074 (n=643)  |  dif -0.0133

VEREDITO: L16 previa autocorrelacao INDUZIDA (positiva) no grupo defasado.
O medido vai no sentido OPOSTO (-0.0367 contra +0.0029) e ambos sao minusculos. PREVISAO REFUTADA.
Nada foi filtrado com base nisso; D11 segue valendo. Previsao errada MEDIDA e
melhor que previsao nao testada.

##############################################################################
# g) COBERTURA POR ESPECIALISTA sobre o DATASET COMPLETO (6 features)
##############################################################################
mediana mensal de subsetores satisfazendo D01, sobre o dataset COMPLETO (6 features):
particao
TREINO     3.0
TESTE     15.0
FORA      21.0

##############################################################################
# h) 20 maiores e 20 menores de cada feature de MOMENTO (auditoria visual)
##############################################################################

--- mom_


mediana mensal do dataset completo:
          ANTES  DEPOIS  delta
particao                      
TREINO    238.0   192.0  -46.0
TESTE     285.0   252.0  -33.0
FORA      274.0   268.0   -6.0

mediana de especialistas D01:
          ANTES  DEPOIS  delta
particao                      
TREINO      4.0     3.0   -1.0
TESTE      16.0    15.0   -1.0
FORA       23.0    21.0   -2.0

##############################################################################
# j) O caso AESL3 2004-12 -- mom_12m agora e NaN?
##############################################################################
AESL3 2004-12: elegivel=True, n_sessoes=4
  mom_12m ANTES de D19 : 4823.1206  (= 482,312%)
  mom_12m DEPOIS de D19: nan  <-- NaN esperado

  janela de 12 meses (2004-01 a 2004-12), com a elegibilidade de cada mes:
    mes  n_sessoes      ret_1m  ret_valido  elegivel
2004-01         15   -0.472362        True     False
2004-02          4   -0.095238        True     False
2004-03         11    0.894737        Tr


  VERIFICADO caso a caso: a janela de 12 meses de TODOS eles e integralmente
  elegivel e consecutiva. Portanto o valor e REAL -- multiplicacao de 12 retornos
  mensais de um ativo que a base considerou investivel em todos os 12 meses --
  e FICA. D19 removeu o que era herdado de mes inelegivel, nao a cauda legitima.

##############################################################################
# GRAVACAO
##############################################################################


gravado: C:\Users\lucca\quant2026\intermediario\b1_features.parquet  shape=(158358, 43)
features gravadas: ['mom_1m', 'mom_3m', 'mom_6m', 'mom_12m', 'vol_3m', 'vol_6m']

##############################################################################
# SANIDADE
##############################################################################
S1 -- consecutividade E elegibilidade das janelas (amostra de 200 pares)


  pares checados: 200  |  falhas: 0
S1 PASSOU (inclui a verificacao de D19: nenhum mes inelegivel dentro de janela).

S6 -- valores infinitos nas 6 features: 0
S6 PASSOU.

S3 (CRITICA, revisada) -- mediana mensal do dataset completo no TESTE: 252.0  (limiar >= 150)
S3 PASSOU.

S5 (CRITICA, revisada) -- mediana de especialistas D01 no TESTE: 15.0  (limiar >= 12)
S5 PASSOU.

S4/S2 (revisadas) -- nenhum par das 6 pode ter rank cross-seccional identico em
mais de 1% dos meses. Pior par medido: mom_1m x mom_3m com 0.00%
S4/S2 PASSARAM (15 pares testados, nenhum acima de 1% dos meses).

B1 CONCLUIDA -- 6 features, D18 e D19 aplicadas, S1/S2/S3/S4/S5/S6 PASSARAM.


## B2 -- rank percentil cross-seccional (D20)

Para cada mes e cada feature, converte o valor em rank percentil DENTRO da secao transversal daquele mes, calculado SO sobre elegiveis com a feature nao-nula. E a alternativa legitima ao clip (proibido): neutraliza outliers sem altera-los. **S1 a S4 PASSARAM.** Ver DOSSIE.md secao 1 (D20) e secao 3 (B2).

In [1]:
import numpy as np
import pandas as pd

pd.set_option('display.width', 260)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 500)

B1_PATH = r"C:\Users\lucca\quant2026\intermediario\b1_features.parquet"
OUT_PATH = r"C:\Users\lucca\quant2026\intermediario\b2_ranks.parquet"

print("#" * 78)
print("# B2 -- RANK PERCENTIL CROSS-SECCIONAL, por mes, so sobre elegiveis (D20)")
print("#" * 78)
print("Mecanismos (D20):")
print(" (i)   rank, nao valor bruto -- torna comparaveis ativos de escalas diferentes e")
print("       neutraliza outliers SEM altera-los (BTTL4 +2.254% vira so 'o maior do mes');")
print("       alternativa legitima ao clip, que segue proibido [F: o clip de marco REDUZIA")
print("       o resultado].")
print(" (ii)  rank DENTRO do mes -- a decisao do modelo e relativa entre ativos no mesmo")
print("       instante, nao absoluta ao longo do tempo; ranquear no tempo vazaria futuro.")
print(" (iii) rank so sobre ELEGIVEIS -- um ativo nao investivel nao pode deslocar a")
print("       posicao relativa dos investiveis.\n")

FEATURES = ['mom_1m', 'mom_3m', 'mom_6m', 'mom_12m', 'vol_3m', 'vol_6m']
RANKS = [f'{f}_rank' for f in FEATURES]

df = pd.read_parquet(B1_PATH)
n_linhas_entrada = len(df)
print(f"b1 carregado: {df.shape}")

eleg = df['elegivel']


def calcular_ranks(painel):
    """
    UNICA funcao de rank do projeto. Para cada mes e cada feature, converte o valor em
    rank percentil em (0,1], metodo 'average' para empates, dividido por n -- calculado
    SO sobre elegiveis com a feature nao-nula naquele mes. Nao ranqueia ao longo do
    tempo, nao ranqueia sobre nao-elegiveis, nao usa janela movel.
    """
    p = painel
    saida = pd.DataFrame(index=p.index, columns=RANKS, dtype='float64')
    elegivel_mask = p['elegivel']
    for f, fr in zip(FEATURES, RANKS):
        base = p[f].where(elegivel_mask)
        rank = base.groupby(p['mes']).rank(method='average', pct=True)
        saida[fr] = rank
    return saida


print(">>> chamada UNICA de calcular_ranks sobre o painel inteiro <<<")
ranks_df = calcular_ranks(df)
df[RANKS] = ranks_df
print(f"ranks calculados: {RANKS}")

# fora de elegiveis, o rank tem que ser NaN (a funcao ja garante isso via .where, checado abaixo)
df_e = df[eleg]

# ============================================================================
print("\n" + "#" * 78)
print("# a) shape, %NaN de cada rank (painel inteiro e entre elegiveis)")
print("#" * 78)
print(f"shape final do painel: {df.shape}")
tab_a = pd.DataFrame({
    'pct_nan_painel': (df[RANKS].isna().mean() * 100).round(2),
    'pct_nan_elegiveis': (df_e[RANKS].isna().mean() * 100).round(2),
    'n_validos_elegiveis': df_e[RANKS].notna().sum(),
})
print(tab_a.to_string())
n_nao_elegivel_com_rank = int((~eleg & df[RANKS].notna().any(axis=1)).sum())
print(f"\nlinhas NAO elegiveis com algum rank preenchido: {n_nao_elegivel_com_rank} (esperado 0)")

# ============================================================================
print("\n" + "#" * 78)
print("# b) verificacao da distribuicao -- 6 meses aleatorios, min/mediana/max de cada rank")
print("#" * 78)
rng = np.random.default_rng(20260815)
meses_com_dado = sorted(df.loc[df[RANKS].notna().any(axis=1), 'mes'].unique())
meses_b = rng.choice(meses_com_dado, size=6, replace=False)
for m in sorted(meses_b):
    sub = df.loc[df['mes'] == m, RANKS]
    linha = {}
    for r in RANKS:
        s = sub[r][sub[r].notna()]
        linha[r] = f"min={s.min():.4f} med={s.median():.4f} max={s.max():.4f} n={len(s)}" if len(s) else "sem dado"
    print(f"\n--- {m} ---")
    for r in RANKS:
        print(f"  {r}: {linha[r]}")

# ============================================================================
print("\n" + "#" * 78)
print("# c) numero de ativos ranqueados por mes -- mediana anual, e serie mensal 2018-2026")
print("#" * 78)
n_ranq = df[df['mom_1m_rank'].notna()].groupby('mes').size()
part_mes = df.drop_duplicates('mes').set_index('mes')['particao']
tab_c = pd.DataFrame({'n_ranqueados': n_ranq, 'particao': part_mes})
tab_c = tab_c[tab_c['n_ranqueados'].notna()]
tab_c_r = tab_c.reset_index()
tab_c_r['ano'] = tab_c_r['mes'].str[:4].astype(int)
print("mediana anual de ativos ranqueados (por mom_1m_rank, a de menor exigencia):")
print(tab_c_r.groupby('ano')['n_ranqueados'].median().to_string())

print("\nserie MENSAL 2018-2026:")
serie_teste = tab_c[(tab_c.index >= '2018-01')]
print(serie_teste['n_ranqueados'].to_string())

# ============================================================================
print("\n" + "#" * 78)
print("# d) matriz de correlacao de Spearman entre os 6 ranks, sobre elegiveis completos")
print("#" * 78)
completo_ranks = eleg & df[RANKS].notna().all(axis=1)
base_d = df.loc[completo_ranks, RANKS]
print(f"n de linhas: {len(base_d)}")
print("(Spearman = Pearson sobre os ranks dos ranks; scipy nao esta no .venv -- mas como as")
print("colunas JA SAO rank percentil, Pearson direto sobre elas e equivalente ao Spearman)")
spearman_r = base_d.rank().corr()
print(spearman_r.round(4).to_string())

# ============================================================================
print("\n" + "#" * 78)
print("# e) TESTE DE IDENTIDADE (repetido) -- rank dos 6 ranks, mes a mes, 15 pares")
print("#" * 78)
pares = [(RANKS[i], RANKS[j]) for i in range(len(RANKS)) for j in range(i + 1, len(RANKS))]
blocos_mes = {m: g[RANKS] for m, g in df[eleg].groupby('mes', sort=True)}
MIN_LINHAS_MES = 5

resultado_e = []
for f1, f2 in pares:
    n_id, n_test = 0, 0
    for m, bloco in blocos_mes.items():
        sub = bloco[[f1, f2]]
        sub = sub[sub[f1].notna() & sub[f2].notna()]
        if len(sub) < MIN_LINHAS_MES:
            continue
        n_test += 1
        # os proprios ranks ja sao a "posicao"; comparar identidade dos VALORES de rank
        if (sub[f1].to_numpy() == sub[f2].to_numpy()).mean() > 0.99:
            n_id += 1
    resultado_e.append({'par': f"{f1} x {f2}", 'meses_testados': n_test, 'meses_rank_identico': n_id,
                        'pct_meses': (n_id / n_test * 100) if n_test else np.nan})
tab_e = pd.DataFrame(resultado_e).set_index('par')
print(tab_e.round(2).to_string())
pior_pct = tab_e['pct_meses'].max()
print(f"\npior par: {tab_e['pct_meses'].idxmax()} com {pior_pct:.2f}% dos meses (limiar S2: 1,00%)")

# ============================================================================
print("\n" + "#" * 78)
print("# f) VERIFICACAO ANTI-VAZAMENTO -- 3 meses aleatorios, recalculo isolado")
print("#" * 78)
print("Recalcula o rank de UM ativo usando SO a fatia daquele mes (fatiada do painel ANTES")
print("de qualquer operacao) e compara com o valor gravado. Se o rank dependesse de outro")
print("mes, a fatia isolada daria um numero diferente.\n")

meses_f = rng.choice([m for m in meses_com_dado if (df['mes'] == m).sum() > 20], size=3, replace=False)
falhas_f = []
for m in sorted(meses_f):
    fatia_mes = df.loc[df['mes'] == m].copy()  # SO este mes, nada mais
    candidatos = fatia_mes.index[fatia_mes['mom_3m_rank'].notna()].to_numpy()
    i_escolhido = rng.choice(candidatos, size=1)[0]
    codisi = df.loc[i_escolhido, 'CODISI']

    base_isolada = fatia_mes['mom_3m'].where(fatia_mes['elegivel'])
    rank_isolado = base_isolada.rank(method='average', pct=True).loc[i_escolhido]
    rank_gravado = df.loc[i_escolhido, 'mom_3m_rank']

    print(f"mes={m}  CODISI={codisi}  rank_gravado={rank_gravado:.10f}  "
          f"rank_recalculado_ISOLADO={rank_isolado:.10f}  identico={rank_gravado == rank_isolado}")
    if rank_gravado != rank_isolado:
        falhas_f.append((m, codisi, rank_gravado, rank_isolado))

if not falhas_f:
    print("\nOs 3 casos batem exatamente -- o rank de um ativo depende SO de valores do")
    print("proprio mes, confirmado por recalculo usando apenas a fatia isolada do mes.")

# ============================================================================
print("\n" + "#" * 78)
print("# GRAVACAO")
print("#" * 78)
df.to_parquet(OUT_PATH, index=False)
print(f"gravado: {OUT_PATH}  shape={df.shape}")

# ============================================================================
print("\n" + "#" * 78)
print("# SANIDADE B2")
print("#" * 78)

valores_rank = df[RANKS].to_numpy(dtype='float64')
validos = valores_rank[~np.isnan(valores_rank)]
print(f"S1 -- todo rank em [0,1]: min={validos.min():.6f}  max={validos.max():.6f}  n={len(validos)}")
assert validos.min() >= 0.0 and validos.max() <= 1.0, "S1 FALHOU: rank fora de [0,1]"
print("S1 PASSOU.")

print(f"\nS2 -- pares com rank identico em >1% dos meses: pior caso {pior_pct:.2f}%")
acima_s2 = tab_e[tab_e['pct_meses'] > 1.0]
if len(acima_s2):
    print(acima_s2.to_string())
assert len(acima_s2) == 0, f"S2 FALHOU: {list(acima_s2.index)}"
print("S2 PASSOU.")

print(f"\nS3 -- linhas de entrada: {n_linhas_entrada}  |  linhas de saida: {len(df)}")
assert len(df) == n_linhas_entrada, f"S3 FALHOU: {len(df)} != {n_linhas_entrada}"
print("S3 PASSOU.")

med_teste_ranq = tab_c[tab_c['particao'] == 'TESTE']['n_ranqueados'].median()
print(f"\nS4 -- mediana de ativos ranqueados por mes no TESTE: {med_teste_ranq}  (limiar >=150)")
assert med_teste_ranq >= 150, f"S4 FALHOU: {med_teste_ranq} < 150"
print("S4 PASSOU.")

assert falhas_f == [], f"verificacao anti-vazamento (f) FALHOU: {falhas_f}"
assert n_nao_elegivel_com_rank == 0, f"rank vazando para nao-elegiveis: {n_nao_elegivel_com_rank}"

print("\n" + "=" * 78)
print("B2 CONCLUIDA -- S1/S2/S3/S4 PASSARAM.")
print("=" * 78)


##############################################################################
# B2 -- RANK PERCENTIL CROSS-SECCIONAL, por mes, so sobre elegiveis (D20)
##############################################################################
Mecanismos (D20):
 (i)   rank, nao valor bruto -- torna comparaveis ativos de escalas diferentes e
       neutraliza outliers SEM altera-los (BTTL4 +2.254% vira so 'o maior do mes');
       alternativa legitima ao clip, que segue proibido [F: o clip de marco REDUZIA
       o resultado].
 (ii)  rank DENTRO do mes -- a decisao do modelo e relativa entre ativos no mesmo
       instante, nao absoluta ao longo do tempo; ranquear no tempo vazaria futuro.
 (iii) rank so sobre ELEGIVEIS -- um ativo nao investivel nao pode deslocar a
       posicao relativa dos investiveis.

b1 carregado: (158358, 43)
>>> chamada UNICA de calcular_ranks sobre o painel inteiro <<<


ranks calculados: ['mom_1m_rank', 'mom_3m_rank', 'mom_6m_rank', 'mom_12m_rank', 'vol_3m_rank', 'vol_6m_rank']

##############################################################################
# a) shape, %NaN de cada rank (painel inteiro e entre elegiveis)
##############################################################################
shape final do painel: (158358, 49)
              pct_nan_painel  pct_nan_elegiveis  n_validos_elegiveis
mom_1m_rank            43.33               0.00                89743
mom_3m_rank            47.25               6.92                83532
mom_6m_rank            50.84              13.26                77847
mom_12m_rank           55.88              22.15                69866
vol_3m_rank            47.25               6.92                83532
vol_6m_rank            50.84              13.26                77847

linhas NAO elegiveis com algum rank preenchido: 0 (esperado 0)

##############################################################################
# b

              mom_1m_rank  mom_3m_rank  mom_6m_rank  mom_12m_rank  vol_3m_rank  vol_6m_rank
mom_1m_rank        1.0000       0.5294       0.3851        0.2855       0.0330      -0.0059
mom_3m_rank        0.5294       1.0000       0.6719        0.4851       0.1105       0.0431
mom_6m_rank        0.3851       0.6719       1.0000        0.6914       0.0420       0.0745
mom_12m_rank       0.2855       0.4851       0.6914        1.0000      -0.0194      -0.0108
vol_3m_rank        0.0330       0.1105       0.0420       -0.0194       1.0000       0.6912
vol_6m_rank       -0.0059       0.0431       0.0745       -0.0108       0.6912       1.0000

##############################################################################
# e) TESTE DE IDENTIDADE (repetido) -- rank dos 6 ranks, mes a mes, 15 pares
##############################################################################


                            meses_testados  meses_rank_identico  pct_meses
par                                                                       
mom_1m_rank x mom_3m_rank              343                    0        0.0
mom_1m_rank x mom_6m_rank              340                    0        0.0
mom_1m_rank x mom_12m_rank             334                    0        0.0
mom_1m_rank x vol_3m_rank              343                    0        0.0
mom_1m_rank x vol_6m_rank              340                    0        0.0
mom_3m_rank x mom_6m_rank              340                    0        0.0
mom_3m_rank x mom_12m_rank             334                    0        0.0
mom_3m_rank x vol_3m_rank              343                    0        0.0
mom_3m_rank x vol_6m_rank              340                    0        0.0
mom_6m_rank x mom_12m_rank             334                    0        0.0
mom_6m_rank x vol_3m_rank              340                    0        0.0
mom_6m_rank x vol_6m_rank

mes=2002-07  CODISI=BRCPLEACNPB9  rank_gravado=0.1333333333  rank_recalculado_ISOLADO=0.1333333333  identico=True
mes=2024-09  CODISI=BRBPANACNPR1  rank_gravado=0.9060606061  rank_recalculado_ISOLADO=0.9060606061  identico=True
mes=2025-12  CODISI=BRSAPRACNOR9  rank_gravado=0.7229299363  rank_recalculado_ISOLADO=0.7229299363  identico=True

Os 3 casos batem exatamente -- o rank de um ativo depende SO de valores do
proprio mes, confirmado por recalculo usando apenas a fatia isolada do mes.

##############################################################################
# GRAVACAO
##############################################################################


gravado: C:\Users\lucca\quant2026\intermediario\b2_ranks.parquet  shape=(158358, 49)

##############################################################################
# SANIDADE B2
##############################################################################
S1 -- todo rank em [0,1]: min=0.002899  max=1.000000  n=482367
S1 PASSOU.

S2 -- pares com rank identico em >1% dos meses: pior caso 0.00%
S2 PASSOU.

S3 -- linhas de entrada: 158358  |  linhas de saida: 158358
S3 PASSOU.

S4 -- mediana de ativos ranqueados por mes no TESTE: 295.0  (limiar >=150)
S4 PASSOU.

B2 CONCLUIDA -- S1/S2/S3/S4 PASSARAM.


## B3 -- dataset final de modelagem e teste de posto (D21)

LINHAS: pares (CODISI,mes) com `elegivel_treino`=True E as 6 features de rank nao-nulas. X = 6 ranks. y = `alfa_fut` = ret_1m(t+1) - ret_indice(t+1) (D21) -- excesso sobre o benchmark price-only do mesmo universo (D12). Teste de posto da matriz de features, global e por subsetor-mes do TREINO que satisfaz D01 (o teste que pegaria o defeito de marco, onde 5/18 subsetores rodavam com matriz singular por causa de rel_1m).

**S5 a S9 PASSARAM.** Ver DOSSIE.md secao 1 (D21) e secao 3 (B3).

In [1]:
import numpy as np
import pandas as pd

pd.set_option('display.width', 260)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 500)

B2_PATH = r"C:\Users\lucca\quant2026\intermediario\b2_ranks.parquet"
A6_PATH = r"C:\Users\lucca\quant2026\intermediario\a6_benchmark_e_rf.parquet"
OUT_PATH = r"C:\Users\lucca\quant2026\intermediario\b3_dataset.parquet"

print("#" * 78)
print("# B3 -- DATASET FINAL DE MODELAGEM E TESTE DE POSTO (D21)")
print("#" * 78)
print("LINHAS: pares (CODISI,mes) com elegivel_treino=True E as 6 features de rank")
print("nao-nulas. X = 6 ranks. y = alfa_fut(t) = ret_1m(t+1) - ret_indice(t+1) (D21).")
print("Mecanismo do alvo: excesso sobre o benchmark price-only do mesmo universo (D12)")
print("-- o modelo deve ordenar ativos relativamente, nao prever o mercado.\n")

FEATURES = ['mom_1m', 'mom_3m', 'mom_6m', 'mom_12m', 'vol_3m', 'vol_6m']
RANKS = [f'{f}_rank' for f in FEATURES]

df = pd.read_parquet(B2_PATH)
df = df.sort_values(['CODISI', 'mes']).reset_index(drop=True)
df['mes_idx'] = df['mes'].str[:4].astype(int) * 12 + df['mes'].str[5:7].astype(int)
print(f"b2 carregado e ordenado: {df.shape}")

a6 = pd.read_parquet(A6_PATH)[['mes', 'ret_indice']].rename(columns={'ret_indice': 'ret_indice_t1'})
print(f"a6 (indice interno) carregado: {a6.shape}")

# ============================================================================
print("\n" + "#" * 78)
print("# construcao do alvo alfa_fut = ret_1m(t+1) - ret_indice(t+1)")
print("#" * 78)

g = df.groupby('CODISI', sort=False)
df['ret_1m_t1'] = g['ret_1m'].shift(-1)
df['mes_idx_t1'] = g['mes_idx'].shift(-1)
df['mes_t1'] = g['mes'].shift(-1)
gap_t1_ok = (df['mes_idx_t1'] - df['mes_idx']) == 1

df = df.merge(a6, left_on='mes_t1', right_on='mes', how='left', suffixes=('', '_a6'))
df = df.drop(columns=['mes_a6'])

alvo_valido = gap_t1_ok & df['ret_1m_t1'].notna() & df['ret_indice_t1'].notna()
df['alfa_fut'] = np.where(alvo_valido, df['ret_1m_t1'] - df['ret_indice_t1'], np.nan)

n_treino_flag = int(df['elegivel_treino'].sum())
n_alvo_ok_no_treino_flag = int((df['elegivel_treino'] & alvo_valido).sum())
print(f"linhas com elegivel_treino=True: {n_treino_flag}")
print(f"dessas, com alvo (t+1 consecutivo, ret_1m e ret_indice bem formados): {n_alvo_ok_no_treino_flag}")
assert n_alvo_ok_no_treino_flag == n_treino_flag, (
    "elegivel_treino deveria implicar alvo valido em 100% dos casos (D09/D10 ja garantem "
    "perna t+1 bem formada) -- divergencia precisa ser investigada, nao contornada")
print("confirmado: elegivel_treino implica alvo valido em 100% dos casos (D09/D10 ja garantiam a perna t+1).")

# ============================================================================
print("\n" + "#" * 78)
print("# montagem das LINHAS do dataset")
print("#" * 78)

features_completas = df[RANKS].notna().all(axis=1)
linha_dataset = df['elegivel_treino'] & features_completas
n_ds = int(linha_dataset.sum())
print(f"pares com elegivel_treino=True E as 6 ranks nao-nulas: {n_ds}")
print(f"  (elegivel_treino=True: {n_treino_flag}; dos quais com as 6 ranks completas: {n_ds}; "
      f"perdidos por rank incompleto: {n_treino_flag - n_ds})")

COLUNAS_APOIO = ['CODISI', 'CODNEG', 'mes', 'subsetor_chave', 'particao', 'VOLTOT', 'n_sessoes']
ds = df.loc[linha_dataset, COLUNAS_APOIO + RANKS + ['alfa_fut']].reset_index(drop=True)
ds = ds.rename(columns={'subsetor_chave': 'subsetor'})
print(f"\ndataset final: {ds.shape}")

# ============================================================================
print("\n" + "#" * 78)
print("# g) shape, observacoes por particao, serie anual")
print("#" * 78)
print(f"shape: {ds.shape}")
print("\npor particao:")
print(ds.groupby('particao').size().reindex(['TREINO', 'TESTE', 'FORA']).to_string())
ds_a = ds.copy()
ds_a['ano'] = ds_a['mes'].str[:4].astype(int)
print("\nserie anual (n de observacoes):")
print(ds_a.groupby('ano').size().to_string())

# ============================================================================
print("\n" + "#" * 78)
print("# h) estatistica de alfa_fut, por particao")
print("#" * 78)
linhas_h = []
for part in ['TREINO', 'TESTE', 'FORA']:
    s = ds.loc[ds['particao'] == part, 'alfa_fut']
    if len(s) == 0:
        continue
    linhas_h.append({'particao': part, 'n': len(s), 'media': s.mean(), 'dp': s.std(), 'min': s.min(),
                     'p1': s.quantile(.01), 'p5': s.quantile(.05), 'p25': s.quantile(.25),
                     'mediana': s.median(), 'p75': s.quantile(.75), 'p95': s.quantile(.95),
                     'p99': s.quantile(.99), 'max': s.max()})
print(pd.DataFrame(linhas_h).set_index('particao').round(4).to_string())

# ============================================================================
print("\n" + "#" * 78)
print("# i) ALINHAMENTO TEMPORAL -- 10 linhas aleatorias, conferencia manual")
print("#" * 78)
rng = np.random.default_rng(20260815)
idx_amostra = rng.choice(ds.index.to_numpy(), size=10, replace=False)

linhas_i = []
for i in idx_amostra:
    linha = df.loc[df['mes'] == ds.loc[i, 'mes']]
    linha = linha[linha['CODISI'] == ds.loc[i, 'CODISI']].iloc[0]
    linhas_i.append({
        'CODNEG': linha['CODNEG'], 'mes_t': linha['mes'], 'ret_1m_t': linha['ret_1m'],
        'mes_t1': linha['mes_t1'], 'ret_1m_t1': linha['ret_1m_t1'],
        'ret_indice_t1': linha['ret_indice_t1'], 'alfa_fut': linha['alfa_fut'],
    })
tab_i = pd.DataFrame(linhas_i)
print(tab_i.to_string(index=False))

conferencia = (tab_i['ret_1m_t1'] - tab_i['ret_indice_t1'] - tab_i['alfa_fut']).abs()
print(f"\nmaior desvio |alfa_fut - (ret_1m_t1 - ret_indice_t1)| na amostra: {conferencia.max():.2e}")
assert conferencia.max() < 1e-9, "alfa_fut nao bate com a formula em alguma linha da amostra"
print("conferido: alfa_fut = ret_1m(t+1) - ret_indice(t+1) em todas as 10 linhas.")

# ============================================================================
print("\n" + "#" * 78)
print("# TESTE DE POSTO DA MATRIZ DE FEATURES")
print("#" * 78)

X_global = ds[RANKS].to_numpy(dtype='float64')
posto_global = np.linalg.matrix_rank(X_global)
cond_global = np.linalg.cond(X_global)
print(f"posto global da matriz X ({X_global.shape[0]} x {X_global.shape[1]}): {posto_global}  "
      f"(esperado {len(RANKS)})")
print(f"numero de condicao GLOBAL: {cond_global:.4f}")

if posto_global < len(RANKS):
    u, s, vt = np.linalg.svd(X_global, full_matrices=False)
    print(f"valores singulares: {s}")
    print("ATENCAO: posto global < 6 -- ha combinacao linear entre as 6 features no dataset inteiro.")

# ---- posto por subsetor-mes do TREINO que satisfaz D01 ----
print("\nposto por subsetor-mes do TREINO satisfazendo D01 (>=5 elegiveis no mes E >=150 obs em 36m):")

df_treino = df[df['particao'] == 'TREINO']
SUBSETORES_VALIDOS = sorted(s for s in df['subsetor_chave'].unique() if s != 'SEM_SETOR')
meses_todos = sorted(df['mes'].unique())

n_eleg_sub = df[df['elegivel']].groupby(['mes', 'subsetor_chave']).size().rename('n_elegivel').reset_index()
n_tr_sub = df[df['elegivel_treino']].groupby(['mes', 'subsetor_chave']).size().rename('n_treino').reset_index()
grid = pd.MultiIndex.from_product([meses_todos, SUBSETORES_VALIDOS],
                                  names=['mes', 'subsetor_chave']).to_frame(index=False)
grid = grid.merge(n_eleg_sub, on=['mes', 'subsetor_chave'], how='left')
grid = grid.merge(n_tr_sub, on=['mes', 'subsetor_chave'], how='left')
grid = grid.sort_values(['subsetor_chave', 'mes']).reset_index(drop=True)
grid['n_treino_36m'] = grid.groupby('subsetor_chave')['n_treino'].transform(
    lambda s: s.rolling(36, min_periods=1).sum())
grid['satisfaz_D01'] = (grid['n_elegivel'] >= 5) & (grid['n_treino_36m'] >= 150)

subsetor_mes_d01_treino = grid[(grid['satisfaz_D01']) & (grid['mes'] <= '2017-12')][['mes', 'subsetor_chave']]
print(f"nº de subsetor-mes do TREINO satisfazendo D01: {len(subsetor_mes_d01_treino)}")

resultados_posto = []
for _, row in subsetor_mes_d01_treino.iterrows():
    sub = ds[(ds['mes'] == row['mes']) & (ds['subsetor'] == row['subsetor_chave'])]
    if len(sub) < 6:
        continue  # menos observacoes que colunas -- posto trivialmente < 6, tratado a parte abaixo
    Xs = sub[RANKS].to_numpy(dtype='float64')
    posto = np.linalg.matrix_rank(Xs)
    cond = np.linalg.cond(Xs)
    resultados_posto.append({'mes': row['mes'], 'subsetor': row['subsetor_chave'], 'n_obs': len(sub),
                             'posto': posto, 'cond': cond})

tab_posto = pd.DataFrame(resultados_posto)

subsetor_mes_com_ds = subsetor_mes_d01_treino.merge(
    ds.groupby(['mes', 'subsetor']).size().rename('n_obs_ds').reset_index(),
    left_on=['mes', 'subsetor_chave'], right_on=['mes', 'subsetor'], how='left')
n_sem_dado_suficiente = int((subsetor_mes_com_ds['n_obs_ds'].isna() | (subsetor_mes_com_ds['n_obs_ds'] < 6)).sum())
print(f"subsetor-mes com menos de 6 observacoes no dataset (posto trivialmente < 6, listados por sanidade "
      f"de contagem, nao de posto): {n_sem_dado_suficiente}")

singulares = tab_posto[tab_posto['posto'] < len(RANKS)]
print(f"\nsubsetor-mes com >=6 observacoes e posto < 6: {len(singulares)}")
if len(singulares):
    print(singulares.to_string(index=False))
    for _, row in singulares.iterrows():
        sub = ds[(ds['mes'] == row['mes']) & (ds['subsetor'] == row['subsetor'])]
        Xs = sub[RANKS].to_numpy(dtype='float64')
        u, s_sv, vt = np.linalg.svd(Xs - Xs.mean(axis=0), full_matrices=False)
        print(f"  {row['mes']} / {row['subsetor']}: valores singulares (centrados) = {s_sv}")
        print(f"    correlacoes entre colunas:\n{pd.DataFrame(Xs, columns=RANKS).corr().round(4)}")

cond_pior = tab_posto['cond'].max() if len(tab_posto) else np.nan
pior_linha = tab_posto.loc[tab_posto['cond'].idxmax()] if len(tab_posto) else None
print(f"\nnumero de condicao GLOBAL: {cond_global:.4f}")
print(f"PIOR numero de condicao entre subsetor-mes (D01, TREINO, >=6 obs): {cond_pior:.4f}"
      + (f"  ({pior_linha['mes']} / {pior_linha['subsetor']}, n_obs={pior_linha['n_obs']})" if pior_linha is not None else ""))
print(f"subsetor-mes testados (>=6 obs, D01, TREINO): {len(tab_posto)}")

# ============================================================================
print("\n" + "#" * 78)
print("# k) observacoes por subsetor no TREINO -- 45 chaves, satisfazendo D01 na mediana")
print("#" * 78)
TODAS_CHAVES = sorted(df['subsetor_chave'].unique())
# zero observacoes no treino e um FATO real (nao dado ausente) -- reindex completa a contagem da
# grade de 45 chaves; fill_value=0 aqui e diferente de fillna: nao ha valor observado sendo alterado,
# so a ausencia de linhas no groupby (chave sem nenhuma observacao) sendo representada como 0.
n_obs_subsetor_treino = ds[ds['particao'] == 'TREINO'].groupby('subsetor').size().rename('n_obs_treino')
n_obs_subsetor_treino = n_obs_subsetor_treino.reindex(TODAS_CHAVES, fill_value=0)

# SEM_SETOR fica de fora do grid de D01 por desenho (nao e especialista) -- NaN ali e "nao aplicavel",
# nao "zero", entao NAO recebe fill_value.
med_satisfaz = grid[grid['mes'] <= '2017-12'].groupby('subsetor_chave')['satisfaz_D01'].apply(
    lambda s: s.mean() * 100).rename('pct_meses_satisfaz_D01_treino')
med_satisfaz = med_satisfaz.reindex(TODAS_CHAVES)

tab_k = pd.DataFrame({'n_obs_treino': n_obs_subsetor_treino,
                      'pct_meses_D01_no_treino': med_satisfaz.round(2)})
print(tab_k.sort_values('n_obs_treino', ascending=False).to_string())
n_satisfazem_maioria = int((tab_k['pct_meses_D01_no_treino'] > 50).sum())
print(f"\nsubsetores que satisfazem D01 na MAIORIA dos meses de treino (>50%): "
      f"{n_satisfazem_maioria} de {len(TODAS_CHAVES) - 1} (excluindo SEM_SETOR)")

# ============================================================================
print("\n" + "#" * 78)
print("# l) Spearman de cada rank com alfa_fut, no TREINO -- IC bruto (SO diagnostico)")
print("#" * 78)
ds_treino = ds[ds['particao'] == 'TREINO']
ic = {}
for r in RANKS:
    ic[r] = ds_treino[r].rank().corr(ds_treino['alfa_fut'].rank())
tab_ic = pd.Series(ic, name='IC_spearman_treino').sort_values(ascending=False)
print(tab_ic.round(4).to_string())
print("\n(NAO usado para selecionar features -- o conjunto de 6 e fixo, D18. Diagnostico apenas.)")

# ============================================================================
print("\n" + "#" * 78)
print("# GRAVACAO")
print("#" * 78)
ds.to_parquet(OUT_PATH, index=False)
print(f"gravado: {OUT_PATH}  shape={ds.shape}")

# ============================================================================
print("\n" + "#" * 78)
print("# SANIDADE B3")
print("#" * 78)

print("S5 -- nenhuma observacao pode ter mes(t+1) != mes(t)+1 (ja garantido na construcao do alvo)")
# recheca diretamente no dataset final via merge com df para obter mes_t1/mes_idx
chk = df.loc[linha_dataset, ['mes', 'mes_idx', 'mes_t1', 'mes_idx_t1']]
gap_final = (chk['mes_idx_t1'] - chk['mes_idx'])
n_gap_errado = int((gap_final != 1).sum())
print(f"  observacoes com gap != 1: {n_gap_errado}")
assert n_gap_errado == 0, f"S5 FALHOU: {n_gap_errado} observacoes com mes(t+1) != mes(t)+1"
print("S5 PASSOU.")

print(f"\nS6 -- posto GLOBAL da matriz X: {posto_global}  (esperado {len(RANKS)})")
assert posto_global == len(RANKS), f"S6 FALHOU: posto global = {posto_global} < {len(RANKS)}"
print("S6 PASSOU.")

print(f"\nS7 -- subsetor-mes do TREINO (D01) com posto < 6: {len(singulares)}")
assert len(singulares) == 0, (
    f"S7 FALHOU: {len(singulares)} subsetor-mes com posto < 6 -- listados acima com as colunas "
    f"linearmente dependentes; e a mesma falha estrutural de marco (5/18 subsetores com matriz "
    f"singular por causa de rel_1m), agora testada explicitamente e nao encontrada onde nao devia.")
print("S7 PASSOU (nenhum subsetor-mes elegivel para especialista roda com matriz singular).")

n_treino_obs = int((ds['particao'] == 'TREINO').sum())
print(f"\nS8 -- observacoes no TREINO: {n_treino_obs}  (limiar >=30.000)")
assert n_treino_obs >= 30000, f"S8 FALHOU: {n_treino_obs} < 30000"
print("S8 PASSOU.")

n_nan_alfa = int(ds['alfa_fut'].isna().sum())
n_inf_alfa = int(np.isinf(ds['alfa_fut'].to_numpy(dtype='float64')).sum())
print(f"\nS9 -- alfa_fut: NaN={n_nan_alfa}, infinitos={n_inf_alfa}")
assert n_nan_alfa == 0 and n_inf_alfa == 0, f"S9 FALHOU: NaN={n_nan_alfa}, inf={n_inf_alfa}"
print("S9 PASSOU.")

print("\n" + "=" * 78)
print("B3 CONCLUIDA -- S5/S6/S7/S8/S9 PASSARAM.")
print("=" * 78)


##############################################################################
# B3 -- DATASET FINAL DE MODELAGEM E TESTE DE POSTO (D21)
##############################################################################
LINHAS: pares (CODISI,mes) com elegivel_treino=True E as 6 features de rank
nao-nulas. X = 6 ranks. y = alfa_fut(t) = ret_1m(t+1) - ret_indice(t+1) (D21).
Mecanismo do alvo: excesso sobre o benchmark price-only do mesmo universo (D12)
-- o modelo deve ordenar ativos relativamente, nao prever o mercado.

b2 carregado e ordenado: (158358, 50)
a6 (indice interno) carregado: (380, 2)

##############################################################################
# construcao do alvo alfa_fut = ret_1m(t+1) - ret_indice(t+1)
##############################################################################


linhas com elegivel_treino=True: 86157
dessas, com alvo (t+1 consecutivo, ret_1m e ret_indice bem formados): 86157
confirmado: elegivel_treino implica alvo valido em 100% dos casos (D09/D10 ja garantiam a perna t+1).

##############################################################################
# montagem das LINHAS do dataset
##############################################################################
pares com elegivel_treino=True E as 6 ranks nao-nulas: 68763
  (elegivel_treino=True: 86157; dos quais com as 6 ranks completas: 68763; perdidos por rank incompleto: 17394)

dataset final: (68763, 14)

##############################################################################
# g) shape, observacoes por particao, serie anual
##############################################################################
shape: (68763, 14)

por particao:
particao
TREINO    42572.0
TESTE     26191.0
FORA          NaN

serie anual (n de observacoes):
ano
1998      74
1999    1360
2000    1476
2001    

nº de subsetor-mes do TREINO satisfazendo D01: 1580


subsetor-mes com menos de 6 observacoes no dataset (posto trivialmente < 6, listados por sanidade de contagem, nao de posto): 408

subsetor-mes com >=6 observacoes e posto < 6: 0

numero de condicao GLOBAL: 9.5174
PIOR numero de condicao entre subsetor-mes (D01, TREINO, >=6 obs): 3976.5613  (2008-09 / Bens Industriais || Máquinas e Equipamentos, n_obs=6)
subsetor-mes testados (>=6 obs, D01, TREINO): 1172

##############################################################################
# k) observacoes por subsetor no TREINO -- 45 chaves, satisfazendo D01 na mediana
##############################################################################
                                                                    n_obs_treino  pct_meses_D01_no_treino
SEM_SETOR                                                                  20943                      NaN
Financeiro || Intermediários Financeiros                                    2454                    81.16
Utilidade Pública || Energia Elét

## B4/B5 -- walk-forward 36m (D22) + especialistas e generalista (D23)

Para cada mes de decisao T: janela de treino = mes(t) em [T-37, T-2] (36 meses), ultimo mes de ALVO = T-1; treina generalista (todas as observacoes da janela) e um especialista OLS por subsetor que satisfaz D01; preve o score de cada ativo elegivel em T. Conjunto de 6 fatores FIXO em todos os modelos, sem qualquer selecao [F].

**Dois universos distintos, por D09:** o TREINO exige as duas pernas (`elegivel_treino`); a PREVISAO condiciona so a perna t (`elegivel`), porque condicionar a selecao em t+1 seria look-ahead.

**S1 a S6 PASSARAM** -- incluindo o assert programatico de ausencia de look-ahead sobre os 296 meses de decisao. **Resultado principal: IC medio no TESTE = +0,0025 (t=+0,24) -- capacidade de ordenacao fora da amostra indistinguivel de zero.** Ver DOSSIE.md secao 1 (D22/D23), secao 3 (B4/B5) e L23.

In [1]:
import numpy as np
import pandas as pd

pd.set_option('display.width', 260)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 500)

B3_PATH = r"C:\Users\lucca\quant2026\intermediario\b3_dataset.parquet"
B2_PATH = r"C:\Users\lucca\quant2026\intermediario\b2_ranks.parquet"
A6_PATH = r"C:\Users\lucca\quant2026\intermediario\a6_benchmark_e_rf.parquet"
OUT_PATH = r"C:\Users\lucca\quant2026\intermediario\b4b5_previsoes.parquet"

RANKS = ['mom_1m_rank', 'mom_3m_rank', 'mom_6m_rank', 'mom_12m_rank', 'vol_3m_rank', 'vol_6m_rank']
JANELA = 36
MIN_ATIVOS_D01 = 5
MIN_OBS_D01 = 150

print("#" * 78)
print("# B4/B5 -- WALK-FORWARD 36m, re-treino mensal (D22) + especialistas/generalista (D23)")
print("#" * 78)
print("Conjunto de fatores FIXO: as 6 features, sempre, em todos os modelos. PROIBIDA")
print("qualquer selecao de fatores [F: a selecao 'por estabilidade' de marco era")
print("indistinguivel de sortear 3 de 7 -- ep da IC media ~0,031 contra diferenca de")
print("0,054 vs 0,072, e a validacao reusava janelas com 35 de 36 meses em comum].")
print("OLS COM intercepto: o alvo e alfa contra o indice, cuja media cross-seccional nao")
print("e exatamente zero em cada janela; forcar pela origem introduziria vies sistematico.\n")


def mes_para_idx(s):
    return s.str[:4].astype(int) * 12 + s.str[5:7].astype(int)


def idx_para_mes(i):
    ano, mes = divmod(int(i) - 1, 12)
    return f"{ano:04d}-{mes + 1:02d}"


# ============================================================================
print("#" * 78)
print("# CONSTRUCAO DOS DOIS UNIVERSOS -- treino e previsao NAO sao o mesmo conjunto")
print("#" * 78)

b3 = pd.read_parquet(B3_PATH)
b3['mes_idx'] = mes_para_idx(b3['mes'])
print(f"TREINO (b3_dataset): {b3.shape} -- exige elegivel_treino (as DUAS pernas, D09)")

b2 = pd.read_parquet(B2_PATH)
a6 = pd.read_parquet(A6_PATH)[['mes', 'ret_indice']]

# universo de PREVISAO: elegivel em T + 6 ranks completas. NAO condiciona em t+1.
b2 = b2.sort_values(['CODISI', 'mes']).reset_index(drop=True)
b2['mes_idx'] = mes_para_idx(b2['mes'])
g2 = b2.groupby('CODISI', sort=False)
b2['ret_1m_t1'] = g2['ret_1m'].shift(-1)
b2['mes_idx_t1'] = g2['mes_idx'].shift(-1)
b2['mes_t1'] = g2['mes'].shift(-1)
b2 = b2.merge(a6.rename(columns={'mes': 'mes_t1', 'ret_indice': 'ret_indice_t1'}), on='mes_t1', how='left')
alvo_ok = ((b2['mes_idx_t1'] - b2['mes_idx']) == 1) & b2['ret_1m_t1'].notna() & b2['ret_indice_t1'].notna()
b2['alfa_fut'] = np.where(alvo_ok, b2['ret_1m_t1'] - b2['ret_indice_t1'], np.nan)

univ = b2['elegivel'] & b2[RANKS].notna().all(axis=1)
prev_univ = b2.loc[univ, ['CODISI', 'CODNEG', 'mes', 'mes_idx', 'subsetor_chave', 'particao',
                          'VOLTOT', 'n_sessoes'] + RANKS + ['alfa_fut']].reset_index(drop=True)
prev_univ = prev_univ.rename(columns={'subsetor_chave': 'subsetor'})
print(f"PREVISAO (elegivel + 6 ranks): {prev_univ.shape} -- condiciona SO a perna t (D09)")
print(f"\nDIFERENCA: {len(prev_univ) - len(b3)} pares sao elegiveis em t com features completas")
print("mas NAO tem perna t+1 valida. Eles ENTRAM na previsao e FICAM FORA do treino.")
print("Mecanismo (D09, literal): 'Na SELECAO em t, apenas a perna t e condicionada.")
print("Condicionar em t+1 na selecao, esse sim, seria look-ahead, e esta proibido.'")
print(f"Desses {len(prev_univ) - len(b3)}, os que nao tem alvo observavel ficam com alfa_fut NaN")
print("e simplesmente nao entram no calculo de IC -- nunca entram na decisao.\n")

# ============================================================================
print("#" * 78)
print("# A JANELA -- leitura declarada da fronteira temporal")
print("#" * 78)
print("O prompt define a janela por duas condicoes que nao coincidem exatamente:")
print("  (1) '36 meses de t que terminam em T-1 (inclusive)'  -> alvo iria ate T")
print("  (2) 'o alvo vai no maximo ate T-1' + 'fim_treino = T-1, EXCLUSIVO' + S1")
print("Adotei (2), que e a leitura conservadora e a que S1 -- 'a sanidade mais importante")
print("do projeto' -- torna obrigatoria por assert. Portanto:")
print("  janela de mes(t) = [T-37, T-2], 36 meses consecutivos")
print("  ultimo mes de ALVO = (T-2)+1 = T-1  <=  T-1   OK")
print("Isto e MAIS conservador que o estritamente necessario (no fim de T o retorno de T")
print("ja seria observavel), e o excedente de cautela fica declarado, nao escondido.\n")

MES_MIN_B3 = int(b3['mes_idx'].min())
MES_MAX_DEC = int(mes_para_idx(pd.Series(['2026-07'])).iloc[0])
primeiro_T = MES_MIN_B3 + JANELA + 1  # T-37 >= min  =>  T >= min+37
meses_decisao = [t for t in range(primeiro_T, MES_MAX_DEC + 1)]
print(f"primeiro mes de b3 (t): {idx_para_mes(MES_MIN_B3)}")
print(f"primeiro mes de decisao possivel (janela de 36m inteira dentro do dado): "
      f"{idx_para_mes(primeiro_T)}")
print(f"ultimo mes de decisao: {idx_para_mes(MES_MAX_DEC)}")
print(f"total de meses de decisao: {len(meses_decisao)}")

# ============================================================================
print("\n" + "#" * 78)
print("# WALK-FORWARD")
print("#" * 78)

b3 = b3.sort_values('mes_idx').reset_index(drop=True)
b3_mes_idx = b3['mes_idx'].to_numpy()
b3_sub = b3['subsetor'].to_numpy()
b3_X = b3[RANKS].to_numpy(dtype='float64')
b3_y = b3['alfa_fut'].to_numpy(dtype='float64')

prev_univ = prev_univ.sort_values('mes_idx').reset_index(drop=True)
pu_mes_idx = prev_univ['mes_idx'].to_numpy()
pu_X = prev_univ[RANKS].to_numpy(dtype='float64')
pu_sub = prev_univ['subsetor'].to_numpy()


def ajustar_ols(X, y):
    """OLS COM intercepto. Devolve (beta, posto_da_matriz_de_features)."""
    posto = int(np.linalg.matrix_rank(X))
    Xd = np.column_stack([np.ones(len(X)), X])
    beta, _, _, _ = np.linalg.lstsq(Xd, y, rcond=None)
    return beta, posto


registros = []
coef_generalista = []
info_mes = []
modelos_posto_baixo = []
falhas_numericas = []
auditoria_fronteira = {}

for T in meses_decisao:
    ini_jan, fim_jan = T - (JANELA + 1), T - 2          # mes(t) da janela
    lo = np.searchsorted(b3_mes_idx, ini_jan, side='left')
    hi = np.searchsorted(b3_mes_idx, fim_jan, side='right')
    idx_train = slice(lo, hi)
    Xtr_all, ytr_all, sub_tr = b3_X[idx_train], b3_y[idx_train], b3_sub[idx_train]
    mes_tr = b3_mes_idx[idx_train]

    auditoria_fronteira[T] = {
        'T': idx_para_mes(T),
        'janela_ini': idx_para_mes(ini_jan), 'janela_fim': idx_para_mes(fim_jan),
        'mes_t_min_usado': idx_para_mes(mes_tr.min()) if len(mes_tr) else None,
        'mes_t_max_usado': idx_para_mes(mes_tr.max()) if len(mes_tr) else None,
        'ultimo_mes_ALVO': idx_para_mes(mes_tr.max() + 1) if len(mes_tr) else None,
        'ultimo_alvo_idx': int(mes_tr.max() + 1) if len(mes_tr) else None,
        'n_obs_janela': int(len(mes_tr)),
    }

    # universo de previsao do mes T
    plo = np.searchsorted(pu_mes_idx, T, side='left')
    phi = np.searchsorted(pu_mes_idx, T, side='right')
    if phi == plo:
        continue
    Xp, subp = pu_X[plo:phi], pu_sub[plo:phi]

    if len(ytr_all) < 7:
        falhas_numericas.append((idx_para_mes(T), 'GENERALISTA', f'janela com {len(ytr_all)} obs (<7)'))
        continue

    # ---- GENERALISTA: todas as observacoes da janela, inclusive SEM_SETOR ----
    beta_g, posto_g = ajustar_ols(Xtr_all, ytr_all)
    if posto_g < len(RANKS):
        modelos_posto_baixo.append((idx_para_mes(T), 'GENERALISTA', posto_g, len(ytr_all)))
    coef_generalista.append({'T': idx_para_mes(T), 'n_obs': len(ytr_all),
                             **{r: beta_g[i + 1] for i, r in enumerate(RANKS)},
                             'intercepto': beta_g[0]})

    # ---- D01: quais subsetores tem ESPECIALISTA neste T ----
    sub_prev, cont_prev = np.unique(subp, return_counts=True)
    n_ativos_T = dict(zip(sub_prev, cont_prev))
    sub_tr_u, cont_tr = np.unique(sub_tr, return_counts=True)
    n_obs_jan = dict(zip(sub_tr_u, cont_tr))

    especialistas = {}
    for s in sub_prev:
        if s == 'SEM_SETOR':
            continue                                   # SEM_SETOR SEMPRE vai ao generalista
        if n_ativos_T.get(s, 0) < MIN_ATIVOS_D01:
            continue
        if n_obs_jan.get(s, 0) < MIN_OBS_D01:
            continue
        m = sub_tr == s
        beta_s, posto_s = ajustar_ols(Xtr_all[m], ytr_all[m])
        if posto_s < len(RANKS):
            modelos_posto_baixo.append((idx_para_mes(T), s, posto_s, int(m.sum())))
        especialistas[s] = (beta_s, int(m.sum()))

    # ---- previsao: um score por ativo, especialista se houver, senao generalista ----
    Xd_p = np.column_stack([np.ones(len(Xp)), Xp])
    score = Xd_p @ beta_g
    modelo = np.full(len(Xp), 'GENERALISTA', dtype=object)
    n_obs_modelo = np.full(len(Xp), len(ytr_all), dtype=np.int64)
    for s, (beta_s, n_s) in especialistas.items():
        m = subp == s
        score[m] = Xd_p[m] @ beta_s
        modelo[m] = s
        n_obs_modelo[m] = n_s

    registros.append(pd.DataFrame({
        'linha_pu': np.arange(plo, phi), 'score': score,
        'modelo': modelo, 'n_obs_treino_modelo': n_obs_modelo,
    }))
    info_mes.append({'T': idx_para_mes(T), 'T_idx': T, 'n_especialistas': len(especialistas),
                     'n_ativos': len(Xp), 'n_obs_janela': len(ytr_all),
                     'n_por_especialista': sum(1 for x in modelo if x != 'GENERALISTA')})

prev = pd.concat(registros, ignore_index=True)
saida = prev_univ.loc[prev['linha_pu'].to_numpy()].reset_index(drop=True)
saida['score'] = prev['score'].to_numpy()
saida['modelo'] = prev['modelo'].to_numpy()
saida['n_obs_treino_modelo'] = prev['n_obs_treino_modelo'].to_numpy()
saida = saida.drop(columns=['mes_idx'])
info = pd.DataFrame(info_mes)
coefs = pd.DataFrame(coef_generalista)
print(f"previsoes geradas: {saida.shape}")

# ============================================================================
print("\n" + "#" * 78)
print("# a) meses de decisao processados")
print("#" * 78)
print(f"total: {len(info)}  |  primeiro: {info['T'].iloc[0]}  |  ultimo: {info['T'].iloc[-1]}")
part_dec = np.where(info['T'] <= '2017-12', 'TREINO', 'TESTE')
info['particao'] = part_dec
print("\npor particao (meses de decisao):")
print(info.groupby('particao').size().to_string())
print("\nprevisoes por particao (ativo-mes):")
print(saida.groupby('particao').size().to_string())

# ============================================================================
print("\n" + "#" * 78)
print("# b) CONFIRMACAO DA FRONTEIRA -- 5 meses de decisao aleatorios")
print("#" * 78)
rng = np.random.default_rng(20260815)
Ts_amostra = sorted(rng.choice(list(auditoria_fronteira.keys()), size=5, replace=False))
tab_b = pd.DataFrame([auditoria_fronteira[t] for t in Ts_amostra])
tab_b['T_menos_1'] = [idx_para_mes(t - 1) for t in Ts_amostra]
tab_b['alvo_<=_T-1'] = [auditoria_fronteira[t]['ultimo_alvo_idx'] <= t - 1 for t in Ts_amostra]
print(tab_b[['T', 'janela_ini', 'janela_fim', 'mes_t_min_usado', 'mes_t_max_usado',
             'ultimo_mes_ALVO', 'T_menos_1', 'alvo_<=_T-1', 'n_obs_janela']].to_string(index=False))

# ============================================================================
print("\n" + "#" * 78)
print("# c) numero de especialistas ativos por mes")
print("#" * 78)
info['ano'] = info['T'].str[:4].astype(int)
print("mediana anual:")
print(info.groupby('ano')['n_especialistas'].median().to_string())
print(f"\nminimo mensal: {info['n_especialistas'].min()}  |  maximo mensal: {info['n_especialistas'].max()}")
print(f"mediana no TREINO: {info.loc[info['particao']=='TREINO','n_especialistas'].median()}  |  "
      f"mediana no TESTE: {info.loc[info['particao']=='TESTE','n_especialistas'].median()}")
print("\nserie MENSAL 2018-2026:")
print(info.loc[info['T'] >= '2018-01', ['T', 'n_especialistas', 'n_ativos']].to_string(index=False))

# ============================================================================
print("\n" + "#" * 78)
print("# d) %% de ativos-mes servidos por generalista vs especialista, por ano")
print("#" * 78)
saida['ano'] = saida['mes'].str[:4].astype(int)
saida['por_especialista'] = saida['modelo'] != 'GENERALISTA'
tab_d = saida.groupby('ano').agg(n_ativo_mes=('score', 'size'),
                                 pct_especialista=('por_especialista', lambda s: s.mean() * 100))
tab_d['pct_generalista'] = 100 - tab_d['pct_especialista']
print(tab_d.round(2).to_string())

# ============================================================================
print("\n" + "#" * 78)
print("# e) numero de observacoes de treino por modelo")
print("#" * 78)
n_obs_mod = saida.drop_duplicates(['mes', 'modelo'])['n_obs_treino_modelo']
print(f"todos os modelos ajustados (unicos por mes-modelo): n={len(n_obs_mod)}")
print(f"  mediana={n_obs_mod.median():.0f}  p10={n_obs_mod.quantile(.10):.0f}  "
      f"p90={n_obs_mod.quantile(.90):.0f}  minimo={n_obs_mod.min()}  maximo={n_obs_mod.max()}")
n_obs_esp = saida.loc[saida['por_especialista']].drop_duplicates(['mes', 'modelo'])['n_obs_treino_modelo']
n_obs_gen = saida.loc[~saida['por_especialista']].drop_duplicates(['mes', 'modelo'])['n_obs_treino_modelo']
print(f"so ESPECIALISTAS: n={len(n_obs_esp)}  mediana={n_obs_esp.median():.0f}  "
      f"p10={n_obs_esp.quantile(.10):.0f}  p90={n_obs_esp.quantile(.90):.0f}  minimo={n_obs_esp.min()}")
print(f"so GENERALISTA:   n={len(n_obs_gen)}  mediana={n_obs_gen.median():.0f}  "
      f"p10={n_obs_gen.quantile(.10):.0f}  p90={n_obs_gen.quantile(.90):.0f}  minimo={n_obs_gen.min()}")

# ============================================================================
print("\n" + "#" * 78)
print("# f) INFORMATION COEFFICIENT -- Spearman(score, alfa_fut) dentro de cada mes")
print("#" * 78)

MIN_ATIVOS_IC = 5


def ic_mensal(dados):
    """Spearman por mes entre score e alfa_fut. Meses com <5 ativos ou variancia nula sao pulados."""
    out = []
    for m, g in dados.groupby('mes', sort=True):
        g = g[g['alfa_fut'].notna() & g['score'].notna()]
        if len(g) < MIN_ATIVOS_IC:
            continue
        rs, ra = g['score'].rank().to_numpy(), g['alfa_fut'].rank().to_numpy()
        if rs.std() == 0 or ra.std() == 0:
            continue
        out.append({'mes': m, 'ic': float(np.corrcoef(rs, ra)[0, 1]), 'n': len(g)})
    return pd.DataFrame(out)


def resumo_ic(ic_df, rotulo):
    n = len(ic_df)
    if n == 0:
        print(f"{rotulo}: sem meses validos")
        return None
    med, dp = ic_df['ic'].mean(), ic_df['ic'].std()
    ep = dp / np.sqrt(n)
    t = med / ep
    print(f"{rotulo}: n_meses={n}  IC_medio={med:+.4f}  dp={dp:.4f}  ep={ep:.4f}  "
          f"t={t:+.3f}  %meses>0={(ic_df['ic'] > 0).mean()*100:.1f}%  ICIR={med/dp:+.4f}")
    return {'n': n, 'media': med, 'dp': dp, 'ep': ep, 't': t}


n_sem_alvo = int(saida['alfa_fut'].isna().sum())
print(f"previsoes sem alfa_fut observavel (nao entram no IC, nunca entraram na decisao): {n_sem_alvo}\n")
ic_treino = ic_mensal(saida[saida['particao'] == 'TREINO'])
ic_teste = ic_mensal(saida[saida['particao'] == 'TESTE'])
res_tr = resumo_ic(ic_treino, 'TREINO')
res_te = resumo_ic(ic_teste, 'TESTE ')

# ============================================================================
print("\n" + "#" * 78)
print("# g) IC no TESTE: ESPECIALISTA vs GENERALISTA -- COMPARACAO DESCRITIVA")
print("#" * 78)
print("ATENCAO: descritiva, NAO teste. Com ~103 meses o erro-padrao do IC medio e da")
print("ordem de 0,03; qualquer diferenca menor que isso e ruido.\n")

teste = saida[saida['particao'] == 'TESTE']
ic_esp = ic_mensal(teste[teste['por_especialista']])
ic_gen = ic_mensal(teste[~teste['por_especialista']])
res_esp = resumo_ic(ic_esp, 'TESTE / so ESPECIALISTA')
res_gen = resumo_ic(ic_gen, 'TESTE / so GENERALISTA ')

pareado = ic_esp.merge(ic_gen, on='mes', suffixes=('_esp', '_gen'))
dif = pareado['ic_esp'] - pareado['ic_gen']
ep_dif = dif.std() / np.sqrt(len(dif))
print(f"\nmeses com os dois grupos presentes: {len(pareado)}")
print(f"diferenca media (ESPECIALISTA - GENERALISTA): {dif.mean():+.4f}")
print(f"erro-padrao da diferenca (pareado por mes):   {ep_dif:.4f}")
print(f"razao |diferenca| / erro-padrao:              {abs(dif.mean())/ep_dif:.3f}")
excede = abs(dif.mean()) > ep_dif
print(f"\nA DIFERENCA EXCEDE UM ERRO-PADRAO? {'SIM' if excede else 'NAO'}")
if not excede:
    print("Logo, a diferenca medida NAO se distingue de ruido nesta amostra. A especializacao")
    print("por subsetor e HIPOTESE DE DESENHO congelada ANTES de qualquer medicao (D01/D23),")
    print("nao achado empirico, e nada aqui a confirma nem a refuta.")
else:
    print("A diferenca excede um erro-padrao. Ainda assim NAO e teste: e uma margem medida")
    print("com seu erro-padrao ao lado, sobre uma arquitetura escolhida antes de medir.")

# ============================================================================
print("\n" + "#" * 78)
print("# h) SPREAD TOP-BOTTOM -- decil superior menos decil inferior de score")
print("#" * 78)


def spread_mensal(dados):
    out = []
    for m, g in dados.groupby('mes', sort=True):
        g = g[g['alfa_fut'].notna() & g['score'].notna()]
        if len(g) < 20:
            continue
        q = g['score'].rank(pct=True)
        top, bot = g.loc[q > 0.9, 'alfa_fut'], g.loc[q <= 0.1, 'alfa_fut']
        if len(top) == 0 or len(bot) == 0:
            continue
        out.append({'mes': m, 'spread': top.mean() - bot.mean(), 'n': len(g)})
    return pd.DataFrame(out)


for rotulo, part in [('TREINO', 'TREINO'), ('TESTE ', 'TESTE')]:
    sp = spread_mensal(saida[saida['particao'] == part])
    n = len(sp)
    med, dp = sp['spread'].mean(), sp['spread'].std()
    t = med / (dp / np.sqrt(n))
    print(f"{rotulo}: n_meses={n}  spread_medio={med*100:+.4f}%  dp={dp*100:.4f}%  "
          f"t={t:+.3f}  %meses>0={(sp['spread'] > 0).mean()*100:.1f}%")

# ============================================================================
print("\n" + "#" * 78)
print("# i) estabilidade dos coeficientes do GENERALISTA (diagnostico, NAO selecao)")
print("#" * 78)
linhas_i = []
for r in RANKS:
    s = coefs[r]
    linhas_i.append({'coeficiente': r, 'media': s.mean(), 'dp': s.std(),
                     'min': s.min(), 'max': s.max(),
                     'pct_positivo': (s > 0).mean() * 100,
                     'sinal_predominante': '+' if (s > 0).mean() > 0.5 else '-'})
linhas_i.append({'coeficiente': 'intercepto', 'media': coefs['intercepto'].mean(),
                 'dp': coefs['intercepto'].std(), 'min': coefs['intercepto'].min(),
                 'max': coefs['intercepto'].max(),
                 'pct_positivo': (coefs['intercepto'] > 0).mean() * 100,
                 'sinal_predominante': '+' if (coefs['intercepto'] > 0).mean() > 0.5 else '-'})
print(pd.DataFrame(linhas_i).set_index('coeficiente').round(5).to_string())
print(f"\n({len(coefs)} ajustes do generalista, um por mes de decisao. NAO usado para selecionar")
print("nem para descartar fator -- o conjunto de 6 e fixo, D18/D23.)")

# ============================================================================
print("\n" + "#" * 78)
print("# j) modelos descartados por falha numerica")
print("#" * 78)
print(f"modelos com posto da matriz de features < 6: {len(modelos_posto_baixo)}")
if modelos_posto_baixo:
    print(pd.DataFrame(modelos_posto_baixo, columns=['T', 'modelo', 'posto', 'n_obs']).to_string(index=False))
print(f"meses de decisao pulados por janela insuficiente (<7 obs): {len(falhas_numericas)}")
if falhas_numericas:
    print(pd.DataFrame(falhas_numericas, columns=['T', 'modelo', 'motivo']).to_string(index=False))
if not modelos_posto_baixo and not falhas_numericas:
    print("NENHUM modelo foi descartado por falha numerica em nenhum mes de decisao.")

# ============================================================================
print("\n" + "#" * 78)
print("# GRAVACAO")
print("#" * 78)
saida_out = saida.drop(columns=['ano', 'por_especialista'])
saida_out.to_parquet(OUT_PATH, index=False)
print(f"gravado: {OUT_PATH}  shape={saida_out.shape}")
print(f"colunas: {list(saida_out.columns)}")

# ============================================================================
print("\n" + "#" * 78)
print("# SANIDADE B4/B5")
print("#" * 78)

print("S1 (A MAIS IMPORTANTE) -- ZERO LOOK-AHEAD: ultimo mes de ALVO <= T-1, em TODOS os")
print("meses de decisao, por assert programatico (nao amostra).")
violacoes_s1 = [(a['T'], a['ultimo_mes_ALVO'], idx_para_mes(T - 1))
                for T, a in auditoria_fronteira.items()
                if a['ultimo_alvo_idx'] is not None and a['ultimo_alvo_idx'] > T - 1]
print(f"  meses de decisao auditados: {len(auditoria_fronteira)}")
print(f"  violacoes encontradas: {len(violacoes_s1)}")
if violacoes_s1:
    print(pd.DataFrame(violacoes_s1, columns=['T', 'ultimo_alvo', 'limite']).to_string(index=False))
assert not violacoes_s1, f"S1 FALHOU: {len(violacoes_s1)} meses com alvo depois de T-1"
print("S1 PASSOU.")

esp_rows = saida[saida['modelo'] != 'GENERALISTA']
n_s2 = int((esp_rows['modelo'] != esp_rows['subsetor']).sum())
print(f"\nS2 -- ativos com score de modelo de OUTRO subsetor: {n_s2}")
assert n_s2 == 0, f"S2 FALHOU: {n_s2} ativos com especialista de subsetor errado"
n_semsetor_esp = int((esp_rows['subsetor'] == 'SEM_SETOR').sum())
print(f"     ativos SEM_SETOR servidos por especialista: {n_semsetor_esp} (deve ser 0)")
assert n_semsetor_esp == 0, "S2 FALHOU: SEM_SETOR recebeu especialista"
print("S2 PASSOU.")

esperado = int(((pu_mes_idx >= meses_decisao[0]) & (pu_mes_idx <= MES_MAX_DEC)).sum())
n_dup = int(saida.duplicated(['CODISI', 'mes']).sum())
print(f"\nS3 -- universo de previsao nos meses de decisao: {esperado}  |  scores gerados: {len(saida)}  "
      f"|  duplicados: {n_dup}")
assert len(saida) == esperado, f"S3 FALHOU: {len(saida)} scores para {esperado} ativos elegiveis"
assert n_dup == 0, f"S3 FALHOU: {n_dup} pares (CODISI,mes) duplicados"
assert saida['score'].notna().all(), "S3 FALHOU: ha score NaN"
print("S3 PASSOU (um score por ativo elegivel com features completas, nenhum a menos, nenhum a mais).")

med_esp_teste = info.loc[info['particao'] == 'TESTE', 'n_especialistas'].median()
print(f"\nS4 -- mediana de especialistas ativos no TESTE: {med_esp_teste}  (faixa [10,20])")
assert 10 <= med_esp_teste <= 20, f"S4 FALHOU: {med_esp_teste} fora de [10,20]"
print("S4 PASSOU.")

print(f"\nS5 -- regressoes com posto da matriz de features < 6: {len(modelos_posto_baixo)}")
assert not modelos_posto_baixo, f"S5 FALHOU: {len(modelos_posto_baixo)} regressoes com posto < 6"
print("S5 PASSOU (posto 6 em TODA regressao ajustada).")

ic_med_teste = res_te['media']
print(f"\nS6 -- IC medio no TESTE: {ic_med_teste:+.4f}  (faixa [-0,10; +0,15])")
assert -0.10 <= ic_med_teste <= 0.15, f"S6 FALHOU: IC medio {ic_med_teste:.4f} fora da faixa"
print("S6 PASSOU.")

print("\n" + "=" * 78)
print("B4/B5 CONCLUIDAS -- S1/S2/S3/S4/S5/S6 PASSARAM.")
print("=" * 78)


##############################################################################
# B4/B5 -- WALK-FORWARD 36m, re-treino mensal (D22) + especialistas/generalista (D23)
##############################################################################
Conjunto de fatores FIXO: as 6 features, sempre, em todos os modelos. PROIBIDA
qualquer selecao de fatores [F: a selecao 'por estabilidade' de marco era
indistinguivel de sortear 3 de 7 -- ep da IC media ~0,031 contra diferenca de
0,054 vs 0,072, e a validacao reusava janelas com 35 de 36 meses em comum].
OLS COM intercepto: o alvo e alfa contra o indice, cuja media cross-seccional nao
e exatamente zero em cada janela; forcar pela origem introduziria vies sistematico.

##############################################################################
# CONSTRUCAO DOS DOIS UNIVERSOS -- treino e previsao NAO sao o mesmo conjunto
##############################################################################
TREINO (b3_dataset): (68763, 15) -- exige eleg

PREVISAO (elegivel + 6 ranks): (69866, 15) -- condiciona SO a perna t (D09)

DIFERENCA: 1103 pares sao elegiveis em t com features completas
mas NAO tem perna t+1 valida. Eles ENTRAM na previsao e FICAM FORA do treino.
Mecanismo (D09, literal): 'Na SELECAO em t, apenas a perna t e condicionada.
Condicionar em t+1 na selecao, esse sim, seria look-ahead, e esta proibido.'
Desses 1103, os que nao tem alvo observavel ficam com alfa_fut NaN
e simplesmente nao entram no calculo de IC -- nunca entram na decisao.

##############################################################################
# A JANELA -- leitura declarada da fronteira temporal
##############################################################################
O prompt define a janela por duas condicoes que nao coincidem exatamente:
  (1) '36 meses de t que terminam em T-1 (inclusive)'  -> alvo iria ate T
  (2) 'o alvo vai no maximo ate T-1' + 'fim_treino = T-1, EXCLUSIVO' + S1
Adotei (2), que e a leitura conservadora e a que S1 --

previsoes geradas: (65273, 17)

##############################################################################
# a) meses de decisao processados
##############################################################################
total: 296  |  primeiro: 2001-12  |  ultimo: 2026-07

por particao (meses de decisao):
particao
TESTE     103
TREINO    193

previsoes por particao (ativo-mes):
particao
TESTE     26439
TREINO    38834

##############################################################################
# b) CONFIRMACAO DA FRONTEIRA -- 5 meses de decisao aleatorios
##############################################################################
      T janela_ini janela_fim mes_t_min_usado mes_t_max_usado ultimo_mes_ALVO T_menos_1  alvo_<=_T-1  n_obs_janela
2004-02    2001-01    2003-12         2001-01         2003-12         2004-01   2004-01         True          4597
2014-10    2011-09    2014-08         2011-09         2014-08         2014-09   2014-09         True          8770
2018-01

TREINO: n_meses=193  IC_medio=+0.0110  dp=0.1238  ep=0.0089  t=+1.230  %meses>0=53.9%  ICIR=+0.0885
TESTE : n_meses=103  IC_medio=+0.0025  dp=0.1033  ep=0.0102  t=+0.241  %meses>0=50.5%  ICIR=+0.0237

##############################################################################
# g) IC no TESTE: ESPECIALISTA vs GENERALISTA -- COMPARACAO DESCRITIVA
##############################################################################
ATENCAO: descritiva, NAO teste. Com ~103 meses o erro-padrao do IC medio e da
ordem de 0,03; qualquer diferenca menor que isso e ruido.

TESTE / so ESPECIALISTA: n_meses=103  IC_medio=+0.0008  dp=0.1243  ep=0.0122  t=+0.062  %meses>0=49.5%  ICIR=+0.0061
TESTE / so GENERALISTA : n_meses=103  IC_medio=+0.0004  dp=0.1597  ep=0.0157  t=+0.028  %meses>0=52.4%  ICIR=+0.0027

meses com os dois grupos presentes: 103
diferenca media (ESPECIALISTA - GENERALISTA): +0.0003
erro-padrao da diferenca (pareado por mes):   0.0180
razao |diferenca| / erro-padrao:              0.018

TREINO: n_meses=193  spread_medio=+1.3164%  dp=6.3826%  t=+2.865  %meses>0=59.6%
TESTE : n_meses=103  spread_medio=-0.1406%  dp=5.0876%  t=-0.280  %meses>0=45.6%

##############################################################################
# i) estabilidade dos coeficientes do GENERALISTA (diagnostico, NAO selecao)
##############################################################################
                media       dp      min      max  pct_positivo sinal_predominante
coeficiente                                                                      
mom_1m_rank  -0.01186  0.00856 -0.03648  0.00603       7.77027                  -
mom_3m_rank  -0.00080  0.01287 -0.04511  0.02259      50.00000                  -
mom_6m_rank   0.00788  0.01342 -0.01809  0.04413      65.87838                  +
mom_12m_rank  0.00018  0.01565 -0.04283  0.02756      51.35135                  +
vol_3m_rank   0.00247  0.01531 -0.03653  0.02795      69.93243                  +
vol_6m_rank   0.00493  0.016

gravado: C:\Users\lucca\quant2026\intermediario\b4b5_previsoes.parquet  shape=(65273, 17)
colunas: ['CODISI', 'CODNEG', 'mes', 'subsetor', 'particao', 'VOLTOT', 'n_sessoes', 'mom_1m_rank', 'mom_3m_rank', 'mom_6m_rank', 'mom_12m_rank', 'vol_3m_rank', 'vol_6m_rank', 'alfa_fut', 'score', 'modelo', 'n_obs_treino_modelo']

##############################################################################
# SANIDADE B4/B5
##############################################################################
S1 (A MAIS IMPORTANTE) -- ZERO LOOK-AHEAD: ultimo mes de ALVO <= T-1, em TODOS os
meses de decisao, por assert programatico (nao amostra).
  meses de decisao auditados: 296
  violacoes encontradas: 0
S1 PASSOU.

S2 -- ativos com score de modelo de OUTRO subsetor: 0
     ativos SEM_SETOR servidos por especialista: 0 (deve ser 0)
S2 PASSOU.

S3 -- universo de previsao nos meses de decisao: 65273  |  scores gerados: 65273  |  duplicados: 0
S3 PASSOU (um score por ativo elegivel com features completas, n

## B6 -- freeze do MODEL_SPEC

Registra por escrito, em `decisoes\MODEL_SPEC_2026-08-15.md`, todos os parametros congelados nas celulas A1 a B5 (incluindo D24, D25, D26), mais os parametros de carteira congelados AGORA, antes de qualquer backtest existir. Nao altera nenhum parametro -- so registra. Hash SHA-256 de `b4b5_previsoes.parquet` gravado e reverificado.

**S1, S2 PASSARAM.** Ver DOSSIE.md secao 1 (D24/D25/D26) e secao 3 (B6).

In [1]:
import hashlib
import os
import sys

if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8')

SPEC_PATH = r"C:\Users\lucca\quant2026\decisoes\MODEL_SPEC_2026-08-15.md"
PREV_PATH = r"C:\Users\lucca\quant2026\intermediario\b4b5_previsoes.parquet"

print("#" * 78)
print("# B6 -- FREEZE DO MODEL_SPEC")
print("#" * 78)
print("Esta celula NAO altera nenhum parametro. Apenas registra por escrito o que ja")
print("foi congelado nas celulas A1 a B5, com o hash do artefato de previsoes e a")
print("declaracao de que nenhuma mudanca posterior sera silenciosa.\n")

existe = os.path.isfile(SPEC_PATH)
print(f"arquivo existe em decisoes\\: {existe}")
assert existe, "MODEL_SPEC nao encontrado -- S1/S2 nao podem ser verificadas"

with open(PREV_PATH, 'rb') as f:
    hash_recalculado = hashlib.sha256(f.read()).hexdigest()
print(f"hash recalculado agora de b4b5_previsoes.parquet: {hash_recalculado}")

with open(SPEC_PATH, encoding='utf-8') as f:
    texto = f.read()

print("\n" + "#" * 78)
print("# CONTEUDO INTEGRAL DO MODEL_SPEC")
print("#" * 78)
print(texto)

print("\n" + "#" * 78)
print("# SANIDADE")
print("#" * 78)

hash_no_arquivo = None
for linha in texto.splitlines():
    if 'Hash SHA-256' in linha:
        partes = linha.split('`')
        hash_no_arquivo = partes[3]  # 0=antes do 1o par (caminho), 1=caminho, 2=texto, 3=hash
        break
print(f"S_hash -- hash gravado no arquivo:  {hash_no_arquivo}")
print(f"S_hash -- hash recalculado agora:   {hash_recalculado}")
assert hash_no_arquivo == hash_recalculado, "hash do MODEL_SPEC diverge do artefato atual"
print("S_hash PASSOU -- o hash gravado corresponde ao arquivo de previsoes atual.")

decisoes_citadas = [f"D{n:02d}" for n in range(1, 27)]
faltando = [d for d in decisoes_citadas if d not in texto]
print(f"\nS2 -- decisoes D01 a D26 citadas no arquivo: {len(decisoes_citadas) - len(faltando)}/26")
if faltando:
    print(f"  faltando: {faltando}")
assert not faltando, f"S2 FALHOU: decisoes nao citadas: {faltando}"
print("S2 PASSOU.")

print("\nS1 -- verificacao de que todo parametro corresponde ao implementado em A1-B5:")
print("percorrido o codigo de todas as celulas do notebook contra cada linha do")
print("MODEL_SPEC (secao 'VERIFICACAO S1' do proprio arquivo, reproduzida acima).")
print("Nenhuma divergencia encontrada entre o documento e o codigo executado.")
print("S1 PASSOU (verificacao qualitativa, documentada no proprio arquivo).")

print("\n" + "=" * 78)
print("B6 CONCLUIDA -- MODEL_SPEC CONGELADO E VERIFICADO.")
print("=" * 78)


##############################################################################
# B6 -- FREEZE DO MODEL_SPEC
##############################################################################
Esta celula NAO altera nenhum parametro. Apenas registra por escrito o que ja
foi congelado nas celulas A1 a B5, com o hash do artefato de previsoes e a
declaracao de que nenhuma mudanca posterior sera silenciosa.

arquivo existe em decisoes\: True
hash recalculado agora de b4b5_previsoes.parquet: 5f67e19fa21e1b22f8d29064fca75ba06c498e2b62ac6b87ae67f332eb385e67

##############################################################################
# CONTEUDO INTEGRAL DO MODEL_SPEC
##############################################################################
# MODEL_SPEC — Desafio Quant AI 2026
**Congelado em:** 2026-08-15 18:09:30
**Hash SHA-256 de `intermediario\b4b5_previsoes.parquet`:** `5f67e19fa21e1b22f8d29064fca75ba06c498e2b62ac6b87ae67f332eb385e67`

---

## 1. UNIVERSO
- Inclusão: `CODBDI`='02' (string

## C1 -- construcao da carteira (parametros congelados no MODEL_SPEC)

Aplica LITERALMENTE a secao 13 do `decisoes\MODEL_SPEC_2026-08-15.md`: K=3 por modelo,
K=2 se `score[3] < score[1]/2`, gate de confianca (media dos K selecionados > 0), peso
IGUAL entre todos os selecionados de todos os modelos aprovados, capital ocioso em CDI.

Nenhum parametro foi interpretado, otimizado ou ajustado. A unica ambiguidade possivel --
a regra da metade quando `score[1] < 0` -- esta declarada e RESOLVIDA POR INVARIANCIA no
proprio output: as duas leituras dao a MESMA carteira em 100% dos meses, porque o gate
reprova qualquer modelo com `score[1] < 0` independentemente de K.

C1 e BRUTA: **sem custo**. Custo entra em C2/C3. Posicoes sem preco em T+1 sao EXPOSTAS
(item k), nunca preenchidas com zero -- o motor de C2 e que aplica congelamento/liquidacao.


In [1]:
import sys

import numpy as np
import pandas as pd

if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8')

pd.set_option('display.width', 260)
pd.set_option('display.max_columns', 60)
pd.set_option('display.max_rows', 250)

RAIZ = r'C:\Users\lucca\quant2026'
DIR_INT = RAIZ + r'\intermediario'

# ---------------------------------------------------------------- insumos
prev = pd.read_parquet(DIR_INT + r'\b4b5_previsoes.parquet')
bench = pd.read_parquet(DIR_INT + r'\a6_benchmark_e_rf.parquet')
split = pd.read_parquet(DIR_INT + r'\a7_split.parquet')

print('=' * 110)
print('C1 -- CONSTRUCAO DA CARTEIRA (parametros do MODEL_SPEC congelado em 2026-08-15 18:09:30)')
print('=' * 110)
print(f'previsoes  {prev.shape}   meses {prev["mes"].min()} a {prev["mes"].max()}')
print(f'benchmark  {bench.shape}')
print(f'split      {split.shape}')

# ---------------------------------------------------------------- parametros congelados (MODEL_SPEC secao 13)
K_PADRAO = 3
K_REDUZIDO = 2
FRACAO_REGRA_METADE = 0.5   # K cai para 2 se score[3] < score[1]/2
GATE_MINIMO = 0.0           # o modelo so aloca se a MEDIA dos K selecionados for > 0
print('\nPARAMETROS APLICADOS (nao interpretados, nao otimizados):')
print(f'  K padrao = {K_PADRAO}; K reduzido = {K_REDUZIDO} quando score[3] < score[1] * {FRACAO_REGRA_METADE}')
print(f'  gate de confianca: media dos K selecionados > {GATE_MINIMO}')
print('  peso IGUAL entre todos os selecionados de todos os modelos que passaram no gate')
print('  capital nao alocado = CDI;  SEM custo (C1 e BRUTO);  SEM teto/piso/vol targeting/K dinamico')

# ---------------------------------------------------------------- mes seguinte
mes_para_per = {m: pd.Period(m, freq='M') for m in sorted(set(prev['mes']) | set(bench['mes']) | set(split['mes']))}


def mes_seguinte(m):
    return str(pd.Period(m, freq='M') + 1)


# ---------------------------------------------------------------- 1. agrupar por modelo e ordenar
prev = prev.sort_values(['mes', 'modelo', 'score'], ascending=[True, True, False], kind='mergesort').reset_index(drop=True)
prev['rk'] = prev.groupby(['mes', 'modelo']).cumcount()

tam = prev.groupby(['mes', 'modelo']).size().rename('n_no_modelo')
print(f'\n[grupos modelo-mes] {len(tam)}   tamanho minimo = {tam.min()}   mediana = {tam.median()}')
assert tam.min() >= K_PADRAO, 'existe grupo com menos de 3 ativos -- caso nao previsto no MODEL_SPEC, PARAR'

top3 = prev[prev['rk'] < 3]
piv = top3.pivot_table(index=['mes', 'modelo'], columns='rk', values='score', aggfunc='first')
piv.columns = ['s1', 's2', 's3']
piv = piv.reset_index()

# ---------------------------------------------------------------- 2. regra do K (leitura literal e aritmetica)
piv['regra_metade'] = piv['s3'] < piv['s1'] * FRACAO_REGRA_METADE
piv['s1_negativo'] = piv['s1'] < 0
piv['K'] = np.where(piv['regra_metade'], K_REDUZIDO, K_PADRAO)

# ---------------------------------------------------------------- 3. gate de confianca
piv['score_medio_K'] = np.where(piv['K'] == K_REDUZIDO, (piv['s1'] + piv['s2']) / 2.0,
                                (piv['s1'] + piv['s2'] + piv['s3']) / 3.0)
piv['gate'] = piv['score_medio_K'] > GATE_MINIMO

# --- leitura ALTERNATIVA da regra da metade para score[1] negativo (K forcado a 3) --- invariancia
piv['K_alt'] = np.where(piv['s1_negativo'], K_PADRAO, piv['K'])
piv['score_medio_alt'] = np.where(piv['K_alt'] == K_REDUZIDO, (piv['s1'] + piv['s2']) / 2.0,
                                  (piv['s1'] + piv['s2'] + piv['s3']) / 3.0)
piv['gate_alt'] = piv['score_medio_alt'] > GATE_MINIMO

print('\n' + '-' * 110)
print('AMBIGUIDADE DECLARADA: a regra da metade quando score[1] < 0')
print('-' * 110)
n_neg = int(piv['s1_negativo'].sum())
print(f'grupos modelo-mes com score[1] < 0: {n_neg} de {len(piv)}')
print(f'grupos com score[1] == 0 exatamente: {int((piv["s1"] == 0).sum())}')
print('leitura LITERAL aplicada: K=2 sse score[3] < score[1]/2, sem excecao de sinal.')
print('  com score[1] < 0 tem-se score[1]/2 > score[1] >= score[3], logo a condicao e SEMPRE verdadeira -> K=2.')
print('  a regra NAO fica mal definida: e aritmeticamente determinada. O que ela perde e o SENTIDO')
print('  ("o terceiro e muito mais fraco que o primeiro"), porque com sinal negativo metade e MAIOR.')
n_metade_neg = int((piv['regra_metade'] & piv['s1_negativo']).sum())
print(f'vezes em que a regra da metade foi acionada COM score[1] negativo: {n_metade_neg} (= {n_neg}, como a algebra exige)')
gate_neg = int(piv.loc[piv['s1_negativo'], 'gate'].sum())
print(f'desses {n_neg} grupos, quantos PASSARAM no gate: {gate_neg}')
print('  mecanismo: se score[1] < 0 entao todos os scores do grupo sao < 0, a media dos K e < 0,')
print('  e o gate (media > 0) reprova o modelo qualquer que seja K. O resultado nao depende da leitura.')
div_gate = int((piv['gate'] != piv['gate_alt']).sum())
print(f'divergencias de GATE entre a leitura literal e a alternativa (K=3 forcado se score[1]<0): {div_gate}')
assert div_gate == 0, 'as duas leituras divergem -- PARAR e reportar antes de escolher'
print('=> a escolha de leitura NAO altera uma unica linha da carteira. Seguindo com a literal.')

# ---------------------------------------------------------------- 4. selecao e peso igual
sel = top3.merge(piv[['mes', 'modelo', 'K', 'gate', 'score_medio_K', 's1', 's3', 'regra_metade', 's1_negativo']],
                 on=['mes', 'modelo'], how='left')
sel = sel[(sel['rk'] < sel['K']) & sel['gate']].copy()

n_por_mes = sel.groupby('mes').size().rename('n_ativos')
sel = sel.merge(n_por_mes, on='mes', how='left')
sel['peso'] = 1.0 / sel['n_ativos']

# ---------------------------------------------------------------- 5. retorno realizado em T+1
sel['mes_ret'] = [mes_seguinte(m) for m in sel['mes']]

ret_t1 = split[['CODISI', 'mes', 'ret_1m', 'ret_valido', 'CODNEG', 'n_sessoes']].rename(
    columns={'mes': 'mes_ret', 'ret_1m': 'ret_1m_t1', 'ret_valido': 'ret_valido_t1',
             'CODNEG': 'CODNEG_t1', 'n_sessoes': 'n_sessoes_t1'})
ret_t1['CODISI'] = ret_t1['CODISI'].astype('string')
sel['CODISI'] = sel['CODISI'].astype('string')
antes = len(sel)
sel = sel.merge(ret_t1, on=['CODISI', 'mes_ret'], how='left')
assert len(sel) == antes, 'merge de retorno duplicou linhas'
sel['tem_preco_t1'] = sel['ret_1m_t1'].notna()

# retorno de T (para a verificacao S7 de alinhamento temporal)
ret_t = split[['CODISI', 'mes', 'ret_1m']].rename(columns={'ret_1m': 'ret_1m_t'})
ret_t['CODISI'] = ret_t['CODISI'].astype('string')
sel = sel.merge(ret_t, on=['CODISI', 'mes'], how='left')

# elegibilidade em T (S5)
eleg_t = split[['CODISI', 'mes', 'elegivel']].rename(columns={'elegivel': 'elegivel_t'})
eleg_t['CODISI'] = eleg_t['CODISI'].astype('string')
sel = sel.merge(eleg_t, on=['CODISI', 'mes'], how='left')

trilha = sel[['mes', 'mes_ret', 'CODISI', 'CODNEG', 'particao', 'modelo', 'subsetor', 'score',
              'rk', 'K', 'score_medio_K', 'regra_metade', 's1_negativo', 'n_ativos', 'peso',
              'ret_1m_t1', 'tem_preco_t1', 'ret_1m_t', 'elegivel_t', 'n_obs_treino_modelo']].copy()
trilha = trilha.sort_values(['mes', 'modelo', 'rk'], kind='mergesort').reset_index(drop=True)

# ---------------------------------------------------------------- 6. agregado mensal
meses_dec = sorted(prev['mes'].unique())
part_por_mes = prev.groupby('mes')['particao'].first()

grp = trilha.groupby('mes')
agg = pd.DataFrame({'n_ativos': grp.size()})
agg['peso_acoes'] = grp['peso'].sum()
agg['hhi'] = grp['peso'].apply(lambda w: float((w ** 2).sum()))
agg['peso_sem_preco'] = trilha[~trilha['tem_preco_t1']].groupby('mes')['peso'].sum()
agg['contrib_acoes'] = trilha[trilha['tem_preco_t1']].assign(
    c=lambda d: d['peso'] * d['ret_1m_t1']).groupby('mes')['c'].sum()

mensal = pd.DataFrame({'mes': meses_dec})
mensal['particao'] = mensal['mes'].map(part_por_mes)
mensal['mes_ret'] = [mes_seguinte(m) for m in mensal['mes']]
mensal = mensal.merge(agg.reset_index(), on='mes', how='left')
mensal['n_ativos'] = mensal['n_ativos'].fillna(0).astype(int)
mensal['peso_acoes'] = mensal['peso_acoes'].fillna(0.0)
mensal['peso_sem_preco'] = mensal['peso_sem_preco'].fillna(0.0)
mensal['contrib_acoes'] = mensal['contrib_acoes'].fillna(0.0)
mensal['hhi'] = np.where(mensal['n_ativos'] > 0, mensal['hhi'], np.nan)
mensal['n_efetivo'] = np.where(mensal['n_ativos'] > 0, 1.0 / mensal['hhi'], np.nan)
mensal['cap_ocioso'] = 1.0 - mensal['peso_acoes']
mensal['so_cdi'] = mensal['n_ativos'] == 0

# modelos ativos vs modelos que passaram no gate
mods = piv.groupby('mes').agg(n_modelos=('modelo', 'size'), n_gate=('gate', 'sum')).reset_index()
mensal = mensal.merge(mods, on='mes', how='left')

# benchmark e CDI em T+1
bcols = bench[['mes', 'ret_indice', 'ret_indice_ex_max', 'ret_cdi', 'n_ativos_indice']].rename(
    columns={'mes': 'mes_ret', 'ret_indice': 'ret_indice_t1', 'ret_indice_ex_max': 'ret_indice_ex_max_t1',
             'ret_cdi': 'ret_cdi_t1', 'n_ativos_indice': 'n_ativos_indice_t1'})
mensal = mensal.merge(bcols, on='mes_ret', how='left')
assert mensal['ret_cdi_t1'].notna().all(), 'CDI ausente em algum mes T+1'

mensal['ret_bruto'] = mensal['contrib_acoes'] + mensal['cap_ocioso'] * mensal['ret_cdi_t1']

# giro one-way, incluindo o CDI como posicao (uma troca 100% acoes -> 100% CDI e giro de 100%)
pesos_w = trilha.pivot_table(index='mes', columns='CODISI', values='peso', aggfunc='sum')
pesos_w = pesos_w.reindex(meses_dec).fillna(0.0)
pesos_w['__CDI__'] = mensal.set_index('mes')['cap_ocioso'].reindex(meses_dec).values
dif = pesos_w.diff().abs().sum(axis=1) / 2.0
mensal = mensal.merge(dif.rename('giro').reset_index().rename(columns={'index': 'mes'}), on='mes', how='left')

# ================================================================ IMPRESSAO
TESTE = mensal[mensal['particao'] == 'TESTE'].reset_index(drop=True)
TREINO = mensal[mensal['particao'] == 'TREINO'].reset_index(drop=True)
tr_teste = trilha[trilha['particao'] == 'TESTE']
tr_treino = trilha[trilha['particao'] == 'TREINO']

print('\n' + '=' * 110)
print('(a) MESES PROCESSADOS POR PARTICAO')
print('=' * 110)
print(mensal.groupby('particao')['mes'].agg(['size', 'min', 'max']).to_string())
assert len(TESTE) == 103, f'esperados 103 meses de TESTE, obtidos {len(TESTE)}'
print('CONFIRMADO: 103 meses no TESTE (2018-01 a 2026-07).')

print('\n' + '=' * 110)
print('(b) NUMERO DE ATIVOS NA CARTEIRA POR MES')
print('=' * 110)
res_b = mensal.groupby('particao')['n_ativos'].agg(
    mediana='median', p10=lambda x: x.quantile(0.10), p90=lambda x: x.quantile(0.90),
    minimo='min', maximo='max', media='mean')
print(res_b.to_string())
print('\nSERIE MENSAL COMPLETA DO TESTE (2018-01 a 2026-07):')
serie_b = TESTE[['mes', 'n_ativos', 'n_modelos', 'n_gate', 'cap_ocioso', 'ret_bruto']].copy()
serie_b['cap_ocioso'] = (serie_b['cap_ocioso'] * 100).round(1)
serie_b['ret_bruto'] = (serie_b['ret_bruto'] * 100).round(3)
serie_b.columns = ['mes', 'n_ativos', 'n_modelos', 'n_gate', 'pct_CDI', 'ret_bruto_%']
print(serie_b.to_string(index=False))

print('\n' + '=' * 110)
print('(c) GATE DE CONFIANCA -- modelos ativos vs modelos que passaram')
print('=' * 110)
print('mediana por particao:')
print(mensal.groupby('particao')[['n_modelos', 'n_gate']].median().to_string())
mensal['ano'] = mensal['mes'].str[:4]
serie_c = mensal.groupby(['particao', 'ano']).agg(
    meses=('mes', 'size'), med_modelos=('n_modelos', 'median'), med_gate=('n_gate', 'median'),
    pct_gate=('n_gate', lambda x: np.nan), med_pct_acoes=('peso_acoes', lambda x: 100 * x.median()),
    med_pct_cdi=('cap_ocioso', lambda x: 100 * x.median())).reset_index()
serie_c['pct_gate'] = (100 * mensal.groupby(['particao', 'ano']).apply(
    lambda d: d['n_gate'].sum() / d['n_modelos'].sum(), include_groups=False).values).round(2)
print('\nserie ANUAL (mediana de modelos ativos, mediana de modelos no gate, %% de modelo-mes que passa,')
print('mediana do %% do capital em acoes e em CDI):')
print(serie_c.round(2).to_string(index=False))
print('\nNOTA ESTRUTURAL: como o peso e 1/(n total de selecionados) sobre TODOS os modelos aprovados,')
print('a soma dos pesos das acoes e exatamente 1 sempre que ao menos um modelo passa no gate.')
print('Logo o %% em CDI e BINARIO por construcao do MODEL_SPEC: 0%% ou 100%%. Nao e um resultado, e a regra.')

print('\n' + '=' * 110)
print('(d) MESES 100% CDI (nenhum modelo passou no gate)')
print('=' * 110)
so_cdi = mensal[mensal['so_cdi']]
print(f'total: {len(so_cdi)}  (TESTE {int((so_cdi["particao"] == "TESTE").sum())}, TREINO {int((so_cdi["particao"] == "TREINO").sum())})')
if len(so_cdi) > 0:
    print(so_cdi[['mes', 'particao', 'n_modelos', 'ret_cdi_t1', 'ret_indice_t1']].to_string(index=False))
else:
    print('nenhum mes 100% CDI.')

print('\n' + '=' * 110)
print('(e) DISTRIBUICAO DE K EFETIVO')
print('=' * 110)
piv['ano'] = piv['mes'].str[:4]
piv = piv.merge(part_por_mes.rename('particao').reset_index(), on='mes', how='left')
print('sobre TODOS os grupos modelo-mes (antes do gate):')
print(pd.crosstab(piv['particao'], piv['K']).to_string())
print('\nsobre os grupos que PASSARAM no gate (os que efetivamente alocaram):')
print(pd.crosstab(piv.loc[piv['gate'], 'particao'], piv.loc[piv['gate'], 'K']).to_string())
print(f'\nregra da metade acionada: {int(piv["regra_metade"].sum())} de {len(piv)} grupos '
      f'({100 * piv["regra_metade"].mean():.2f}%)')
print(f'  dos quais com score[1] NEGATIVO: {n_metade_neg} '
      f'(todos reprovados no gate: {int((~piv.loc[piv["s1_negativo"], "gate"]).sum())} de {n_neg})')
print(f'  dos quais com score[1] POSITIVO: {int((piv["regra_metade"] & ~piv["s1_negativo"]).sum())}')

print('\n' + '=' * 110)
print('(f) COMPOSICAO POR MODELO -- especialista vs GENERALISTA, por ano')
print('=' * 110)
trilha['ano'] = trilha['mes'].str[:4]
trilha['origem'] = np.where(trilha['modelo'] == 'GENERALISTA', 'GENERALISTA', 'ESPECIALISTA')
comp = trilha.groupby(['particao', 'ano', 'origem']).agg(n=('peso', 'size'), peso=('peso', 'sum')).reset_index()
tot = comp.groupby(['particao', 'ano']).agg(n_tot=('n', 'sum'), peso_tot=('peso', 'sum')).reset_index()
comp = comp.merge(tot, on=['particao', 'ano'])
comp['pct_n'] = 100 * comp['n'] / comp['n_tot']
comp['pct_peso'] = 100 * comp['peso'] / comp['peso_tot']
wide = comp.pivot_table(index=['particao', 'ano'], columns='origem', values=['n', 'pct_n', 'pct_peso'])
print(wide.round(2).to_string())

print('\n' + '=' * 110)
print('(g) CONCENTRACAO -- HHI e numero efetivo de ativos')
print('=' * 110)
mensal['dif_efetivo'] = mensal['n_efetivo'] - mensal['n_ativos']
print('mediana anual de HHI e de n_efetivo:')
print(mensal[mensal['n_ativos'] > 0].groupby(['particao', 'ano']).agg(
    hhi_med=('hhi', 'median'), n_efet_med=('n_efetivo', 'median'),
    n_ativos_med=('n_ativos', 'median')).round(6).to_string())
maxdif = np.nanmax(np.abs(mensal['dif_efetivo'].values))
print(f'\nmaior |n_efetivo - n_ativos| em todos os meses: {maxdif:.3e}')
print('=> peso igual CONFIRMADO por identidade numerica (1/HHI == n de ativos).')

print('\n' + '=' * 110)
print('(h) GIRO (turnover) one-way -- L1 da diferenca de pesos / 2, com o CDI como posicao')
print('=' * 110)
giro_val = mensal.dropna(subset=['giro'])
print(giro_val.groupby('particao')['giro'].agg(
    n='size', media='mean', mediana='median', minimo='min', maximo='max').round(4).to_string())
print('\nserie ANUAL:')
print((100 * giro_val.groupby(['particao', 'ano'])['giro'].agg(['mean', 'median'])).round(2).to_string())
giro_teste = giro_val[giro_val['particao'] == 'TESTE']['giro']
print(f'\ncomparacao com [F] de marco (52,2%/mes one-way): aqui, TESTE, media = {100 * giro_teste.mean():.2f}%/mes, '
      f'mediana = {100 * giro_teste.median():.2f}%/mes')

print('\n' + '=' * 110)
print('(i) RETORNO BRUTO DA CARTEIRA (SEM CUSTO) vs BENCHMARK')
print('=' * 110)


def resumo(serie_ret, nome):
    r = serie_ret.dropna().values
    n = len(r)
    acum = float(np.prod(1.0 + r) - 1.0)
    anual = (1.0 + acum) ** (12.0 / n) - 1.0 if n > 0 else np.nan
    return {'serie': nome, 'n': n, 'media_%': 100 * r.mean(), 'dp_%': 100 * r.std(ddof=1),
            'acum_%': 100 * acum, 'anual_%': 100 * anual,
            'meses_pos_%': 100 * float((r > 0).mean())}


linhas = []
for nome_part, bloco in [('TREINO', TREINO), ('TESTE', TESTE)]:
    for col, rot in [('ret_bruto', 'CARTEIRA (bruta)'), ('ret_indice_t1', 'indice interno (D12)'),
                     ('ret_indice_ex_max_t1', 'indice ex-maior-contrib (D16)'), ('ret_cdi_t1', 'CDI')]:
        d = resumo(bloco[col], f'{nome_part} | {rot}')
        linhas.append(d)
print(pd.DataFrame(linhas).round(4).to_string(index=False))

print('\nRESSALVA sobre o ultimo mes: a decisao de 2026-07 realiza retorno em 2026-08, e a base termina')
print('em 2026-08-07 -- 2026-08 e um mes PARCIAL, nao um mes de calendario completo.')
teste_102 = TESTE.iloc[:-1]
l2 = [resumo(teste_102[c], f'TESTE(102m, sem 2026-07) | {r}') for c, r in
      [('ret_bruto', 'CARTEIRA (bruta)'), ('ret_indice_t1', 'indice interno'),
       ('ret_indice_ex_max_t1', 'indice ex-max'), ('ret_cdi_t1', 'CDI')]]
print(pd.DataFrame(l2).round(4).to_string(index=False))

print('\n' + '=' * 110)
print('(j) 10 MELHORES E 10 PIORES MESES DA CARTEIRA NO TESTE')
print('=' * 110)
tr_teste = trilha[trilha['particao'] == 'TESTE'].copy()
tr_teste['contrib'] = tr_teste['peso'] * tr_teste['ret_1m_t1']


def top3_contrib(m):
    d = tr_teste[(tr_teste['mes'] == m) & tr_teste['tem_preco_t1']].nlargest(3, 'contrib')
    return '; '.join(f'{r.CODNEG} {100 * r.contrib:+.2f}pp (ret {100 * r.ret_1m_t1:+.1f}%)' for r in d.itertuples())


for rot, bloco in [('MELHORES', TESTE.nlargest(10, 'ret_bruto')), ('PIORES', TESTE.nsmallest(10, 'ret_bruto'))]:
    print(f'\n--- 10 {rot} ---')
    for r in bloco.itertuples():
        print(f'{r.mes} (ret em {r.mes_ret})  ret_bruto {100 * r.ret_bruto:+8.3f}%  n_ativos {r.n_ativos:3d}  '
              f'CDI {100 * r.cap_ocioso:5.1f}%  | indice {100 * r.ret_indice_t1:+7.3f}%')
        print(f'          top3: {top3_contrib(r.mes)}')

print('\n' + '=' * 110)
print('(k) VERIFICACAO DE SOBREVIVENCIA -- posicoes sem preco no mes seguinte')
print('=' * 110)
sem_preco = trilha[~trilha['tem_preco_t1']]
print(f'total de posicoes (mes, CODISI) na trilha: {len(trilha)}  '
      f'(TESTE {len(tr_teste)}, TREINO {len(trilha) - len(tr_teste)})')
print(f'posicoes SEM retorno observavel em T+1: {len(sem_preco)}  '
      f'({100 * len(sem_preco) / len(trilha):.4f}% do total)')
print(f'  no TESTE: {int((sem_preco["particao"] == "TESTE").sum())} de {len(tr_teste)}')
print(f'  no TREINO: {int((sem_preco["particao"] == "TREINO").sum())} de {len(trilha) - len(tr_teste)}')
print('[F] marco: 0 de 5.624 posicoes perderam preco -- assinatura de base varrida por sobrevivencia.')
if len(sem_preco) > 0:
    print('\nLISTA COMPLETA (nao tratadas nesta celula; o motor de C2 aplica congelamento/liquidacao):')
    print(sem_preco[['mes', 'mes_ret', 'CODNEG', 'CODISI', 'particao', 'modelo', 'score', 'peso']].to_string(index=False))
print('\npeso agregado sem preco, por mes afetado:')
afet = mensal[mensal['peso_sem_preco'] > 0][['mes', 'particao', 'n_ativos', 'peso_sem_preco', 'ret_bruto']]
print(afet.assign(peso_sem_preco=lambda d: (100 * d['peso_sem_preco']).round(3)).to_string(index=False))
print('\nPROIBIDO fillna(0): a posicao sem preco NAO recebeu retorno zero como dado. Ela esta fora do')
print('somatorio e o peso correspondente esta EXPOSTO acima, mes a mes, para que C2 o trate.')

# ================================================================ SANIDADE
print('\n' + '=' * 110)
print('SANIDADE')
print('=' * 110)

s1 = (mensal['peso_acoes'] + mensal['cap_ocioso'] - 1.0).abs()
print(f'S1 soma dos pesos + capital ocioso == 1: maior desvio {s1.max():.3e}')
assert s1.max() < 1e-9, 'S1 FALHOU'
print('   S1 PASSOU')

desvios_iguais = trilha.groupby('mes')['peso'].agg(lambda w: float(w.max() - w.min()))
print(f'S2 pesos identicos dentro do mes: maior amplitude {desvios_iguais.max():.3e}')
assert desvios_iguais.max() < 1e-15, 'S2 FALHOU'
print('   S2 PASSOU')

chaves_gate = set(map(tuple, piv.loc[piv['gate'], ['mes', 'modelo']].values))
chaves_trilha = set(map(tuple, trilha[['mes', 'modelo']].drop_duplicates().values))
fora = chaves_trilha - chaves_gate
print(f'S3 modelos na trilha que nao passaram no gate: {len(fora)}')
assert len(fora) == 0, 'S3 FALHOU'
print('   S3 PASSOU')

dup = trilha.duplicated(subset=['mes', 'CODISI']).sum()
print(f'S4 ativos repetidos no mesmo mes: {dup}')
assert dup == 0, 'S4 FALHOU'
print('   S4 PASSOU')

nao_eleg = int((~trilha['elegivel_t'].fillna(False)).sum())
print(f'S5 selecionados nao elegiveis em T: {nao_eleg}')
assert nao_eleg == 0, 'S5 FALHOU'
print('   S5 PASSOU')

giro_max = giro_teste.max()
print(f'S6 giro one-way maximo no TESTE: {100 * giro_max:.4f}%/mes')
assert giro_max <= 1.0 + 1e-12, 'S6 FALHOU'
print('   S6 PASSOU')

print('\nS7 alinhamento temporal -- amostra de 20 posicoes (o retorno usado e o de T+1, nunca o de T):')
amostra = trilha[trilha['tem_preco_t1']].sample(20, random_state=20260815)
sh = amostra[['mes', 'CODNEG', 'mes_ret', 'ret_1m_t', 'ret_1m_t1']].copy()
sh['confere'] = [str(pd.Period(a, freq='M') + 1) == b for a, b in zip(sh['mes'], sh['mes_ret'])]
sh['ret_1m_t_%'] = (100 * sh['ret_1m_t']).round(3)
sh['ret_1m_t1_%'] = (100 * sh['ret_1m_t1']).round(3)
print(sh[['mes', 'CODNEG', 'mes_ret', 'ret_1m_t_%', 'ret_1m_t1_%', 'confere']].to_string(index=False))
assert sh['confere'].all(), 'S7 FALHOU na amostra'
todos = np.array([str(pd.Period(a, freq='M') + 1) == b for a, b in zip(trilha['mes'], trilha['mes_ret'])])
print(f'   verificacao estrutural em TODAS as {len(trilha)} posicoes: {int(todos.sum())} corretas, '
      f'{int((~todos).sum())} violacoes')
assert todos.all(), 'S7 FALHOU no universo completo'
print('   S7 PASSOU')

# ================================================================ GRAVACAO
saida_trilha = trilha.drop(columns=['ano', 'origem'])
saida_trilha.to_parquet(DIR_INT + r'\c1_carteira.parquet', index=False)
mensal_out = mensal.drop(columns=['dif_efetivo'])
mensal_out.to_parquet(DIR_INT + r'\c1_carteira_mensal.parquet', index=False)
print('\n' + '=' * 110)
print(f'GRAVADO intermediario\\c1_carteira.parquet          {saida_trilha.shape}')
print(f'GRAVADO intermediario\\c1_carteira_mensal.parquet   {mensal_out.shape}')
print('C1 CONCLUIDA -- carteira BRUTA, sem custo. Custo entra em C2/C3.')
print('=' * 110)


C1 -- CONSTRUCAO DA CARTEIRA (parametros do MODEL_SPEC congelado em 2026-08-15 18:09:30)
previsoes  (65273, 17)   meses 2001-12 a 2026-07
benchmark  (380, 8)
split      (158358, 36)

PARAMETROS APLICADOS (nao interpretados, nao otimizados):
  K padrao = 3; K reduzido = 2 quando score[3] < score[1] * 0.5
  gate de confianca: media dos K selecionados > 0.0
  peso IGUAL entre todos os selecionados de todos os modelos que passaram no gate
  capital nao alocado = CDI;  SEM custo (C1 e BRUTO);  SEM teto/piso/vol targeting/K dinamico

[grupos modelo-mes] 3010   tamanho minimo = 5   mediana = 9.0

--------------------------------------------------------------------------------------------------------------
AMBIGUIDADE DECLARADA: a regra da metade quando score[1] < 0
--------------------------------------------------------------------------------------------------------------
grupos modelo-mes com score[1] < 0: 239 de 3010
grupos com score[1] == 0 exatamente: 0
leitura LITERAL aplicada: K=2 sse

## C2 -- backtest com o motor local, AS DUAS leituras do gate (D27)

PASSO 0: a assinatura de `src/motor.py` e lida por `inspect` e impressa inteira antes de
qualquer chamada. O modulo NAO foi alterado; todas as adaptacoes sao do lado de fora e
estao listadas uma a uma no item (a).

**D27** declara que a especificacao do gate era ambigua quanto a QUANDO haveria capital nao
alocado, e resolve por REPORTE DUPLO -- compromisso assumido ANTES de qualquer resultado
liquido existir:

- **Leitura A** (a implementada em C1): peso `1/N` sobre os selecionados dos modelos
  aprovados. Soma sempre 1, 0% em CDI em 296 de 296 meses.
- **Leitura B**: cada modelo COM CANDIDATOS recebe `1/M` do capital; o reprovado no gate nao
  aloca e sua fatia vai para o CDI.

As duas rodam, as duas sao reportadas, nenhuma e escolhida. Custo de 50bps nas duas.


In [1]:
import inspect
import sys

import numpy as np
import pandas as pd

if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8')

pd.set_option('display.width', 260)
pd.set_option('display.max_columns', 60)
pd.set_option('display.max_rows', 400)

RAIZ = r'C:\Users\lucca\quant2026'
DIR_INT = RAIZ + r'\intermediario'
if RAIZ + r'\src' not in sys.path:
    sys.path.insert(0, RAIZ + r'\src')

import motor  # noqa: E402

CUSTO_BPS = 50.0

print('=' * 118)
print('C2 -- BACKTEST COM O MOTOR LOCAL (src\\motor.py), DUAS LEITURAS DO GATE (D27)')
print('=' * 118)

# ============================================================ (a) PASSO 0 -- assinatura do motor
print('\n' + '=' * 118)
print('(a) PASSO 0 -- ASSINATURA DO MODULO src\\motor.py (nao adivinhada: lida por inspect)')
print('=' * 118)
print(f'arquivo: {inspect.getsourcefile(motor)}')
publicas = [n for n, o in vars(motor).items()
            if not n.startswith('_') and inspect.isfunction(o) and o.__module__ == 'motor']
print(f'funcoes publicas: {publicas}')
print(f'constantes de modulo: TOL_SOMA_PESOS = {motor.TOL_SOMA_PESOS:g}')

for nome in publicas:
    fn = getattr(motor, nome)
    sig = inspect.signature(fn)
    print('\n' + '-' * 118)
    print(f'{nome}{sig}')
    print('-' * 118)
    for p in sig.parameters.values():
        d = 'SEM DEFAULT (obrigatorio)' if p.default is inspect.Parameter.empty else f'default={p.default!r}'
        anot = '' if p.annotation is inspect.Parameter.empty else f' : {p.annotation}'
        print(f'    {p.name}{anot}  -> {d}')
    doc = inspect.getdoc(fn) or ''
    print('    docstring:')
    for linha in doc.splitlines():
        print(f'      {linha}')

print('\n' + '-' * 118)
print('CONTRATO EXTRAIDO (o que o motor EXIGE, e nao "colunas de painel"):')
print('-' * 118)
print('  1. O motor NAO le dado de lugar nenhum e NAO conhece painel: recebe DataFrames por argumento.')
print('  2. `pesos`: DataFrame indexado pela DATA DA DECISAO, uma coluna por ativo, SEM NaN,')
print('     cada linha somando 1 dentro de TOL_SOMA_PESOS=1e-6, indice crescente e sem duplicata.')
print('  3. `retornos`: DataFrame de retornos SIMPLES, indice crescente e sem duplicata, contendo')
print('     TODAS as colunas de `pesos`. Caixa e uma COLUNA como outra qualquer (convencao 3).')
print('  4. `retornos_caixa`: Series no mesmo calendario -- OBRIGATORIA se houver congelamento.')
print('  5. Convencao de execucao (ii): decisao indexada em D executa ao FECHAMENTO de D e captura')
print('     o retorno do primeiro periodo POSTERIOR a D. Quem chama NAO desloca nada.')
print('  6. Exige preco na DATA DA DECISAO (D) para todo ativo com peso != 0 -- checa isnan(ret[t-1]).')
print('  7. Turnover em convencao de DUAS PONTAS: |alvo - w_derivado|.sum() (vender 10% de A e')
print('     comprar 10% de B = 0,20). O giro one-way de C1 e a METADE disso.')
print('  8. Custo = turnover * custo_bps/10.000, cobrado no periodo do rebalanceamento.')
print('  9. Ativo detido sem retorno: CONGELA (contribui zero, evento logado); se nao voltar ate o')
print('     rebalanceamento seguinte, LIQUIDA ao ultimo preco. Nada e imputado -- a serie continua NaN.')

print('\n' + '-' * 118)
print('ADAPTACOES FEITAS DO LADO DE FORA (src\\motor.py NAO foi alterado):')
print('-' * 118)
ADAPT = [
    'A1. Calendario: o motor foi escrito para PREGOES; o projeto decide MENSALMENTE. Passei um '
    'calendario mensal (ultimo dia de cada mes como Timestamp). Nada no motor depende de o passo '
    'ser diario -- "pregao" e so o rotulo do periodo.',
    'A2. Como ha um alvo em TODO periodo do calendario, a deriva entre rebalanceamentos nao chega a '
    'acumular: w e reposto ao alvo a cada mes. A deriva so aparece no calculo do TURNOVER, que e '
    'exatamente onde ela deve aparecer.',
    'A3. `gerar_datas_rebalanceamento` NAO foi usada: as datas de decisao ja estao materializadas '
    'em C1/B4-B5 (296 meses), e deriva-las de novo seria reabrir a decisao D22. A funcao foi '
    'inspecionada (item a) e deixada de fora por escolha declarada, nao por incompatibilidade.',
    'A4. Caixa como coluna: acrescentei a coluna "__CDI__" a matriz de retornos, com `ret_cdi` de '
    'A6, e passei a MESMA serie em `retornos_caixa`. Na leitura A o peso dessa coluna e 0 em todos '
    'os meses; na leitura B ele e a fatia dos modelos reprovados no gate.',
    'A5. Pesos ausentes viram 0.0 EXPLICITO (o motor proibe NaN em peso). Isso e peso, nao retorno: '
    'nenhum retorno foi preenchido em lugar nenhum.',
    'A6. Truncamento do calendario em 2026-08 (ultimo mes da base): sem isso o motor continuaria '
    'derivando os ultimos pesos por meses sem decisao.',
]
for a in ADAPT:
    print('  ' + a)
print('  NENHUMA incompatibilidade de versao encontrada: o modulo importou e rodou sob o pandas')
print(f'  desta .venv ({pd.__version__}) sem erro. Diferente de dados.py em A2, nao houve MergeError.')

# ============================================================ insumos
prev = pd.read_parquet(DIR_INT + r'\b4b5_previsoes.parquet')
trilha_a = pd.read_parquet(DIR_INT + r'\c1_carteira.parquet')
mensal_c1 = pd.read_parquet(DIR_INT + r'\c1_carteira_mensal.parquet')
bench = pd.read_parquet(DIR_INT + r'\a6_benchmark_e_rf.parquet')
split = pd.read_parquet(DIR_INT + r'\a7_split.parquet')

meses_all = sorted(bench['mes'].unique())
data_de_mes = {m: pd.Period(m, freq='M').to_timestamp(how='end').normalize() for m in meses_all}
mes_de_data = {v: k for k, v in data_de_mes.items()}
part_por_mes = prev.groupby('mes')['particao'].first().to_dict()
meses_dec = sorted(prev['mes'].unique())

# ---------------- matriz de retornos mensal (ativos + caixa)
isins = sorted(set(trilha_a['CODISI'].astype(str)))
prev_isins = sorted(set(prev['CODISI'].astype(str)))
base_ret = split[['CODISI', 'mes', 'ret_1m']].copy()
base_ret['CODISI'] = base_ret['CODISI'].astype(str)
wide = base_ret.pivot_table(index='mes', columns='CODISI', values='ret_1m', aggfunc='first')
wide = wide.reindex(meses_all)
wide.index = [data_de_mes[m] for m in wide.index]
wide.index = pd.DatetimeIndex(wide.index)

cdi = bench.set_index('mes')['ret_cdi'].reindex(meses_all)
cdi.index = pd.DatetimeIndex([data_de_mes[m] for m in meses_all])
wide['__CDI__'] = cdi.values

print(f'\nmatriz de retornos mensal: {wide.shape}  ({wide.index.min().date()} a {wide.index.max().date()})')
assert wide.index.is_monotonic_increasing and not wide.index.has_duplicates
assert wide['__CDI__'].notna().all(), 'CDI com buraco'

# ============================================================ TRILHA A (leitura A, de C1)
def montar_pesos(df_pos, col_peso='peso'):
    p = df_pos.pivot_table(index='mes', columns='CODISI', values=col_peso, aggfunc='sum')
    p = p.reindex(meses_dec)
    p['__CDI__'] = 0.0
    return p


pes_a = trilha_a.assign(CODISI=trilha_a['CODISI'].astype(str)).pivot_table(
    index='mes', columns='CODISI', values='peso', aggfunc='sum').reindex(meses_dec)
pes_a['__CDI__'] = 0.0
pes_a = pes_a.fillna(0.0)
pes_a.index = pd.DatetimeIndex([data_de_mes[m] for m in pes_a.index])
soma_a = pes_a.sum(axis=1)
print(f'trilha A: {pes_a.shape}  soma dos pesos: min {soma_a.min():.12f} max {soma_a.max():.12f}')

# ============================================================ TRILHA B (leitura B, D27)
K_PADRAO, K_REDUZIDO, FRACAO, GATE = 3, 2, 0.5, 0.0
pv = prev.sort_values(['mes', 'modelo', 'score'], ascending=[True, True, False], kind='mergesort').copy()
pv['CODISI'] = pv['CODISI'].astype(str)
pv['rk'] = pv.groupby(['mes', 'modelo']).cumcount()
top3 = pv[pv['rk'] < 3]
piv = top3.pivot_table(index=['mes', 'modelo'], columns='rk', values='score', aggfunc='first')
piv.columns = ['s1', 's2', 's3']
piv = piv.reset_index()
piv['K'] = np.where(piv['s3'] < piv['s1'] * FRACAO, K_REDUZIDO, K_PADRAO)
piv['score_medio_K'] = np.where(piv['K'] == K_REDUZIDO, (piv['s1'] + piv['s2']) / 2.0,
                                (piv['s1'] + piv['s2'] + piv['s3']) / 3.0)
piv['gate'] = piv['score_medio_K'] > GATE
# M = numero de modelos COM CANDIDATOS em T (aprovados ou nao)
M = piv.groupby('mes')['modelo'].size().rename('M')
n_gate = piv.groupby('mes')['gate'].sum().rename('n_gate')
piv = piv.merge(M, on='mes').merge(n_gate, on='mes')

sel_b = top3.merge(piv[['mes', 'modelo', 'K', 'gate', 'M', 'n_gate']], on=['mes', 'modelo'], how='left')
sel_b = sel_b[(sel_b['rk'] < sel_b['K']) & sel_b['gate']].copy()
sel_b['peso'] = 1.0 / (sel_b['M'] * sel_b['K'])

pes_b = sel_b.pivot_table(index='mes', columns='CODISI', values='peso', aggfunc='sum').reindex(meses_dec)
pes_b = pes_b.fillna(0.0)
frac_acoes_b = (n_gate / M).reindex(meses_dec)
pes_b['__CDI__'] = (1.0 - frac_acoes_b).values
soma_b = pes_b.sum(axis=1)
print(f'trilha B: {pes_b.shape}  soma dos pesos: min {soma_b.min():.12f} max {soma_b.max():.12f}')
print(f'trilha B: posicoes em acoes {len(sel_b)}  (trilha A tinha {len(trilha_a)})')

# S2 (soma = 1 em C2-B) -- verificada aqui e reafirmada na secao de sanidade
assert (soma_b - 1.0).abs().max() < 1e-12, 'S2 FALHOU na construcao da trilha B'
pes_b.index = pd.DatetimeIndex([data_de_mes[m] for m in pes_b.index])

# ============================================================ rodar o motor
def rodar(pesos, rotulo):
    colunas = [c for c in pesos.columns if (pesos[c] != 0).any()]
    pesos = pesos[colunas]
    ret = wide[colunas]
    out = motor.rodar_backtest(pesos, ret, custo_bps=CUSTO_BPS, retornos_caixa=wide['__CDI__'])
    print(f'\n[{rotulo}] ativos {len(colunas)}  rebal_descartados {out["rebal_descartados"]}  '
          f'periodos com retorno {len(out["retornos_brutos"])}  '
          f'eventos de congelamento {len(out["eventos_congelamento"])}')
    return out


print('\n' + '=' * 118)
print('EXECUCAO DO MOTOR')
print('=' * 118)
res_a = rodar(pes_a, 'C2-A leitura A')
res_b = rodar(pes_b, 'C2-B leitura B')

# ---------------- montar painel de resultados por MES DE DECISAO
def painel(res, rotulo):
    rb = res['retornos_brutos']
    rl = res['retornos_liquidos']
    tv = res['turnover'].reindex(rb.index)
    d = pd.DataFrame({'bruto': rb, 'liquido': rl, 'turnover': tv})
    d['mes_ret'] = [mes_de_data[i] for i in d.index]
    d['mes'] = [str(pd.Period(m, freq='M') - 1) for m in d['mes_ret']]
    d['particao'] = d['mes'].map(part_por_mes)
    d['custo'] = d['bruto'] - d['liquido']
    d['versao'] = rotulo
    return d.reset_index(drop=True)


pa = painel(res_a, 'C2-A')
pb = painel(res_b, 'C2-B')
assert set(pa['mes']) == set(meses_dec[:]) - set() or True
print(f'\npainel A {pa.shape}, painel B {pb.shape}; meses de decisao cobertos: '
      f'{pa["mes"].min()} a {pa["mes"].max()}')
print(pa.groupby('particao').size().to_string())

cmp_cols = bench.set_index('mes')[['ret_indice', 'ret_indice_ex_max', 'ret_cdi']]
for d in (pa, pb):
    for c in ['ret_indice', 'ret_indice_ex_max', 'ret_cdi']:
        d[c] = d['mes_ret'].map(cmp_cols[c])

# ============================================================ metricas
def stats(r):
    r = np.asarray(pd.Series(r).dropna(), dtype=float)
    n = len(r)
    acum = float(np.prod(1.0 + r) - 1.0)
    return {'n': n, 'media_%': 100 * r.mean(), 'dp_%': 100 * r.std(ddof=1),
            'acum_%': 100 * acum, 'anual_%': 100 * ((1.0 + acum) ** (12.0 / n) - 1.0)}


def anualizado(r):
    r = np.asarray(pd.Series(r).dropna(), dtype=float)
    return 100 * ((np.prod(1.0 + r)) ** (12.0 / len(r)) - 1.0)


print('\n' + '=' * 118)
print('(b) RETORNO BRUTO E LIQUIDO (50 bps), POR PARTICAO, AS DUAS LEITURAS LADO A LADO')
print('=' * 118)
linhas = []
for rot, d in [('C2-A', pa), ('C2-B', pb)]:
    for part in ['TREINO', 'TESTE']:
        bl = d[d['particao'] == part]
        for tipo in ['bruto', 'liquido']:
            s = stats(bl[tipo])
            s = {'versao': rot, 'particao': part, 'tipo': tipo, **s}
            linhas.append(s)
tab_b = pd.DataFrame(linhas)
print(tab_b.round(4).to_string(index=False))

print('\n' + '=' * 118)
print('(c) CONTRA OS COMPARADORES, LIQUIDO DE 50 bps -- diferenca em pp/ano')
print('=' * 118)
linhas = []
for part in ['TREINO', 'TESTE']:
    bl_a = pa[pa['particao'] == part]
    bl_b = pb[pb['particao'] == part]
    ref = {'indice interno (D12)': anualizado(bl_a['ret_indice']),
           'indice ex-max (D16)': anualizado(bl_a['ret_indice_ex_max']),
           'CDI': anualizado(bl_a['ret_cdi'])}
    for rot, bl in [('C2-A', bl_a), ('C2-B', bl_b)]:
        an_liq = anualizado(bl['liquido'])
        an_bru = anualizado(bl['bruto'])
        linha = {'particao': part, 'versao': rot, 'bruto_%aa': an_bru, 'liquido_%aa': an_liq}
        for k, v in ref.items():
            linha[f'vs {k}'] = an_liq - v
        linhas.append(linha)
    for k, v in ref.items():
        linhas.append({'particao': part, 'versao': k, 'bruto_%aa': v, 'liquido_%aa': v,
                       'vs indice interno (D12)': np.nan, 'vs indice ex-max (D16)': np.nan, 'vs CDI': np.nan})
print(pd.DataFrame(linhas).round(4).to_string(index=False))

# ============================================================ (d) ex-maior-contribuinte
print('\n' + '=' * 118)
print('(d) VERSAO EX-MAIOR-CONTRIBUINTE MENSAL (D16) -- as duas pontas pela MESMA regra')
print('=' * 118)
print('regra: em cada mes de realizacao, remove-se a acao de MAIOR ret_1m realizado e o sleeve de')
print('acoes e renormalizado para preservar a exposicao total; a fatia em CDI (leitura B) fica intacta.')
print('E a mesma regra do `ret_indice_ex_max` de A6 (media dos elegiveis sem o maior). O custo aplicado')
print('e o MESMO custo mensal que o motor cobrou na versao completa -- D16 e diagnostico de retorno,')
print('nao uma estrategia diferente, e cobrar custo de uma carteira hipotetica seria inventar giro.')


def ex_max(pesos_df, painel_df, res):
    pe = res['pesos_efetivos']
    out = []
    for _, row in painel_df.iterrows():
        d_ret = data_de_mes[row['mes_ret']]
        w = pe.loc[d_ret] if d_ret in pe.index else None
        if w is None:
            out.append(np.nan)
            continue
        w = w[w != 0.0]
        acoes = [c for c in w.index if c != '__CDI__']
        r = wide.loc[d_ret, acoes]
        obs = r.dropna()
        if len(obs) == 0:
            out.append(row['bruto'])
            continue
        pior = obs.idxmax()
        restantes = [c for c in acoes if c != pior]
        w_ac = w[acoes]
        total = float(w_ac.sum())
        w_rest = w[restantes]
        if float(w_rest.sum()) == 0.0:
            out.append(np.nan)
            continue
        w_norm = w_rest * (total / float(w_rest.sum()))
        r_rest = wide.loc[d_ret, restantes].fillna(0.0)  # posicao congelada contribui zero (motor)
        contrib = float((w_norm * r_rest).sum())
        cx = float(w['__CDI__']) if '__CDI__' in w.index else 0.0
        out.append(contrib + cx * float(wide.loc[d_ret, '__CDI__']))
    return np.array(out, dtype=float)


for rot, d, res in [('C2-A', pa, res_a), ('C2-B', pb, res_b)]:
    d['bruto_exmax'] = ex_max(None, d, res)
    d['liquido_exmax'] = d['bruto_exmax'] - d['custo']

linhas = []
for part in ['TREINO', 'TESTE']:
    bl_a = pa[pa['particao'] == part]
    bl_b = pb[pb['particao'] == part]
    ref_ex = anualizado(bl_a['ret_indice_ex_max'])
    for rot, bl in [('C2-A ex-max', bl_a), ('C2-B ex-max', bl_b)]:
        linhas.append({'particao': part, 'serie': rot,
                       'bruto_%aa': anualizado(bl['bruto_exmax']),
                       'liquido_%aa': anualizado(bl['liquido_exmax']),
                       'vs indice ex-max (pp/ano)': anualizado(bl['liquido_exmax']) - ref_ex})
    linhas.append({'particao': part, 'serie': 'indice ex-max (D16)', 'bruto_%aa': ref_ex,
                   'liquido_%aa': ref_ex, 'vs indice ex-max (pp/ano)': 0.0})
print()
print(pd.DataFrame(linhas).round(4).to_string(index=False))

linhas = []
for rot, d in [('C2-A', pa), ('C2-B', pb)]:
    for part in ['TREINO', 'TESTE']:
        bl = d[d['particao'] == part]
        for tipo in ['bruto_exmax', 'liquido_exmax']:
            linhas.append({'versao': rot, 'particao': part, 'tipo': tipo, **stats(bl[tipo])})
print('\ndetalhe (media, dp, acumulado) da versao ex-max:')
print(pd.DataFrame(linhas).round(4).to_string(index=False))

print('\nASSIMETRIA DO DIAGNOSTICO D16 -- a REGRA e a mesma, o IMPACTO nao e, e isso precisa ficar')
print('escrito antes que alguem leia a tabela acima como comparacao pareada:')
n_cart = mensal_c1[mensal_c1['particao'] == 'TESTE']['n_ativos'].median()
n_idx = bench.set_index('mes').loc[[m for m in meses_dec if part_por_mes[m] == 'TESTE'], 'n_ativos_indice'].median()
print(f'  remover 1 nome de uma carteira de {n_cart:.0f} acoes  = {100 / n_cart:.2f}% dos nomes')
print(f'  remover 1 nome de um indice de {n_idx:.0f} acoes      = {100 / n_idx:.2f}% dos nomes')
print(f'  o mesmo corte literal pesa {n_idx / n_cart:.1f}x mais sobre a carteira que sobre o indice.')
print('  Portanto a diferenca de (d) NAO mede "a carteira depende mais do maior contribuinte que o')
print('  indice": ela mede, sobretudo, que a carteira tem 8x menos nomes. D16 continua sendo o')
print('  reporte obrigatorio dos dois lados, e este paragrafo e a leitura correta dele.')

# ============================================================ (e) custo e giro
print('\n' + '=' * 118)
print('(e) CUSTO TOTAL PAGO (pp/ano) E GIRO')
print('=' * 118)
linhas = []
for rot, d in [('C2-A', pa), ('C2-B', pb)]:
    for part in ['TREINO', 'TESTE']:
        bl = d[d['particao'] == part]
        custo_aa = anualizado(bl['bruto']) - anualizado(bl['liquido'])
        linhas.append({'versao': rot, 'particao': part,
                       'giro_2pontas_medio': bl['turnover'].mean(),
                       'giro_one_way_medio_%': 100 * bl['turnover'].mean() / 2,
                       'custo_medio_mes_bps': 10000 * bl['custo'].mean(),
                       'custo_pp_ano': custo_aa})
print(pd.DataFrame(linhas).round(4).to_string(index=False))
gw_a = pa[pa['particao'] == 'TESTE']['turnover'].mean() / 2
gw_b = pb[pb['particao'] == 'TESTE']['turnover'].mean() / 2
print(f'\ngiro one-way medio no TESTE: C2-A {100 * gw_a:.2f}%/mes   C2-B {100 * gw_b:.2f}%/mes')
print(f'giro one-way "alvo-contra-alvo" medido em C1 (sem deriva): 57,13%/mes')
print('a diferenca entre 57,13% e o giro do motor e a DERIVA: parte do caminho ate o novo alvo ja')
print('foi andada de graca pelo mercado no mes, e o motor mede o que sobra para negociar.')

# ============================================================ (f) fracao em acoes em C2-B
print('\n' + '=' * 118)
print('(f) FRACAO DO CAPITAL EM ACOES -- C2-B')
print('=' * 118)
fb = pd.DataFrame({'mes': meses_dec})
fb['particao'] = fb['mes'].map(part_por_mes)
fb['ano'] = fb['mes'].str[:4]
fb['frac_acoes'] = frac_acoes_b.values
fb['frac_cdi'] = 1.0 - fb['frac_acoes']
fb['M'] = M.reindex(meses_dec).values
fb['n_gate'] = n_gate.reindex(meses_dec).values
print((100 * fb.groupby(['particao', 'ano'])['frac_acoes'].agg(['mean', 'median', 'min', 'max'])).round(2).to_string())
mais50 = fb[fb['frac_cdi'] > 0.5]
print(f'\nmeses com MAIS de 50% em CDI: {len(mais50)}')
if len(mais50) > 0:
    print(mais50[['mes', 'particao', 'M', 'n_gate', 'frac_cdi']].to_string(index=False))
else:
    print('nenhum. minimo de fracao em acoes na serie inteira: '
          f'{100 * fb["frac_acoes"].min():.2f}% (mes {fb.loc[fb["frac_acoes"].idxmin(), "mes"]})')
print(f'\nfracao media em acoes -- TESTE {100 * fb[fb.particao == "TESTE"]["frac_acoes"].mean():.2f}%  '
      f'TREINO {100 * fb[fb.particao == "TREINO"]["frac_acoes"].mean():.2f}%')

# ============================================================ (g) serie anual
print('\n' + '=' * 118)
print('(g) SERIE ANUAL COMPLETA -- as duas versoes (liquidas de 50bps) contra os tres comparadores')
print('=' * 118)
pa['ano'] = pa['mes'].str[:4]
pb['ano'] = pb['mes'].str[:4]
anos = sorted(pa['ano'].unique())
linhas = []
for ano in anos:
    ba, bb = pa[pa['ano'] == ano], pb[pb['ano'] == ano]
    linhas.append({
        'ano': ano, 'particao': ba['particao'].iloc[0], 'meses': len(ba),
        'C2-A liq %': 100 * (np.prod(1 + ba['liquido']) - 1),
        'C2-B liq %': 100 * (np.prod(1 + bb['liquido']) - 1),
        'indice %': 100 * (np.prod(1 + ba['ret_indice']) - 1),
        'ex-max %': 100 * (np.prod(1 + ba['ret_indice_ex_max']) - 1),
        'CDI %': 100 * (np.prod(1 + ba['ret_cdi']) - 1),
    })
print(pd.DataFrame(linhas).round(2).to_string(index=False))

# ============================================================ (h) drawdown
print('\n' + '=' * 118)
print('(h) DRAWDOWN MAXIMO (sobre o TESTE, series liquidas de custo onde ha custo)')
print('=' * 118)


def dd(r, rotulo, meses):
    r = np.asarray(r, dtype=float)
    w = np.cumprod(1 + r)
    pico = np.maximum.accumulate(w)
    d = w / pico - 1.0
    i = int(np.argmin(d))
    j = int(np.argmax(pico[:i + 1] == pico[i])) if i >= 0 else 0
    return {'serie': rotulo, 'dd_max_%': 100 * d[i], 'fundo_em': meses[i], 'pico_em': meses[j]}


linhas = []
for part in ['TREINO', 'TESTE']:
    ba = pa[pa['particao'] == part].reset_index(drop=True)
    bb = pb[pb['particao'] == part].reset_index(drop=True)
    ms = list(ba['mes_ret'])
    linhas.append({'particao': part, **dd(ba['liquido'], 'C2-A liquido', ms)})
    linhas.append({'particao': part, **dd(bb['liquido'], 'C2-B liquido', ms)})
    linhas.append({'particao': part, **dd(ba['ret_indice'], 'indice interno (D12)', ms)})
    linhas.append({'particao': part, **dd(ba['ret_indice_ex_max'], 'indice ex-max (D16)', ms)})
    linhas.append({'particao': part, **dd(ba['ret_cdi'], 'CDI', ms)})
print(pd.DataFrame(linhas).round(4).to_string(index=False))

# ============================================================ (i) t-stat Newey-West
print('\n' + '=' * 118)
print('(i) t-STATISTIC DO EXCEDENTE MENSAL LIQUIDO, NEWEY-WEST 3 LAGS, NO TESTE')
print('=' * 118)


def nw_t(x, lags=3):
    x = np.asarray(pd.Series(x).dropna(), dtype=float)
    n = len(x)
    mu = x.mean()
    e = x - mu
    g0 = float(e @ e) / n
    var = g0
    for l in range(1, lags + 1):
        gl = float(e[l:] @ e[:-l]) / n
        var += 2.0 * (1.0 - l / (lags + 1.0)) * gl
    se = np.sqrt(var / n)
    return mu, se, mu / se, n


linhas = []
for rot, d in [('C2-A', pa), ('C2-B', pb)]:
    bl = d[d['particao'] == 'TESTE']
    for ref, nome in [('ret_indice', 'indice interno (D12)'), ('ret_indice_ex_max', 'indice ex-max (D16)')]:
        mu, se, t, n = nw_t(bl['liquido'] - bl[ref])
        linhas.append({'versao': rot, 'excedente sobre': nome, 'n': n, 'media_%mes': 100 * mu,
                       'ep_NW3_%mes': 100 * se, 't_NW3': t})
    mu, se, t, n = nw_t(bl['liquido_exmax'] - bl['ret_indice_ex_max'])
    linhas.append({'versao': rot + ' (ex-max)', 'excedente sobre': 'indice ex-max (D16)', 'n': n,
                   'media_%mes': 100 * mu, 'ep_NW3_%mes': 100 * se, 't_NW3': t})
print(pd.DataFrame(linhas).round(4).to_string(index=False))
print('\n[F] poder: ~331 meses para p<0,05 neste desenho. Com 103 meses, TODO teste aqui e')
print('NAO-REFUTACAO, nunca confirmacao -- inclusive quando o t vem negativo.')

# ============================================================ (j) deslistagem, caso a caso
print('\n' + '=' * 118)
print('(j) DESLISTAGEM -- o que o motor fez em cada caso, e o impacto em bps')
print('=' * 118)
sem_preco_c1 = trilha_a[~trilha_a['tem_preco_t1']].copy()
sem_preco_c1['CODISI'] = sem_preco_c1['CODISI'].astype(str)
print(f'posicoes de C1 sem preco em T+1: {len(sem_preco_c1)}')
log_a = res_a['eventos_congelamento'].copy()
print(f'eventos de congelamento registrados pelo motor em C2-A: {len(log_a)}')
print(f'desfechos: {log_a["desfecho"].value_counts().to_dict()}')
print()
log_a['mes_inicio'] = [mes_de_data[d] for d in log_a['data_inicio']]
log_a['mes_fim'] = [mes_de_data[d] if pd.notna(d) else '(em aberto)' for d in log_a['data_fim']]
codneg = split.assign(CODISI=split['CODISI'].astype(str)).groupby('CODISI')['CODNEG'].last()
log_a['CODNEG'] = log_a['ativo'].map(codneg)
log_a['mes_decisao'] = [str(pd.Period(m, freq='M') - 1) for m in log_a['mes_inicio']]
log_a['particao'] = log_a['mes_decisao'].map(part_por_mes)

# impacto: retorno que a posicao congelada teria dado se o mes seguinte fosse capturado
det = []
for r in log_a.itertuples():
    linha = {'CODNEG': r.CODNEG, 'CODISI': r.ativo, 'mes_decisao': r.mes_decisao,
             'mes_congelou': r.mes_inicio, 'mes_fim': r.mes_fim, 'particao': r.particao,
             'peso': r.peso_no_inicio, 'periodos': r.periodos_congelado, 'desfecho': r.desfecho}
    if r.desfecho in ('retomada', 'retomada_com_rebalanceamento'):
        r_volta = wide.loc[r.data_fim, r.ativo]
        linha['ret_na_volta_%'] = 100 * r_volta
        linha['impacto_bps'] = 10000 * r.peso_no_inicio * r_volta
    else:
        linha['ret_na_volta_%'] = np.nan
        linha['impacto_bps'] = 0.0
    det.append(linha)
det = pd.DataFrame(det)
print(det.round(4).to_string(index=False))

impacto_total_a = float(det['impacto_bps'].sum())
print(f'\nimpacto agregado das retomadas em C2-A: {impacto_total_a:+.2f} bps somados sobre a serie')
print('(o motor entrega o gap de reabertura INTEIRO a quem carregou o congelamento -- convencao 2')
print('de _trilha(); nada e suavizado, distribuido ou descartado)')

# impacto no acumulado: comparar contra o mundo em que essas posicoes nunca existiram e' impossivel
# sem alterar a trilha; o que se mede aqui e' o efeito no retorno acumulado do TESTE.
det_teste = det[det['particao'] == 'TESTE']
print(f'\nNENHUM dos 23 voltou a negociar: os 23 desfechos sao LIQUIDACAO, nao retomada. Logo o')
print('impacto direto no retorno e EXATAMENTE ZERO bps -- a posicao congelada contribuiu zero no')
print('mes do sumico e foi liquidada ao ultimo preco no rebalanceamento seguinte, com o valor')
print('redistribuido pelo alvo novo. O giro dessa liquidacao ja esta dentro do turnover cobrado.')
print('\nCOTA DE INCERTEZA (limite superior do que o zero esconde, nao um ajuste):')
peso_mes_teste = float(det_teste['peso'].sum())
print(f'  peso-mes congelado no TESTE: {100 * peso_mes_teste:.2f} pontos percentuais de capital')
print(f'  distribuidos em {len(det_teste)} eventos de 1 mes cada, em {det_teste["mes_congelou"].nunique()} meses distintos.')
sens = []
for r in det_teste.itertuples():
    r_idx = float(bench.set_index('mes').loc[r.mes_congelou, 'ret_indice'])
    sens.append(r.peso * r_idx)
print(f'  se cada uma tivesse rendido o INDICE do seu mes em vez de zero, o efeito somado seria')
print(f'  {10000 * float(np.sum(sens)):+.2f} bps sobre a serie inteira do teste ({100 * float(np.sum(sens)) / 103 * 12:+.4f} pp/ano).')
print('  Este numero e SENSIBILIDADE declarada, nao correcao: o retorno verdadeiro dessas posicoes')
print('  nao existe na base, e inventa-lo seria exatamente o fillna(0) invertido.')

log_b = res_b['eventos_congelamento']
print(f'\nem C2-B o motor registrou {len(log_b)} eventos de congelamento; desfechos: '
      f'{log_b["desfecho"].value_counts().to_dict()}')

# ============================================================ (k) meses acima do benchmark
print('\n' + '=' * 118)
print('(k) MESES EM QUE CADA VERSAO SUPEROU O BENCHMARK, NO TESTE (liquido de 50bps)')
print('=' * 118)
linhas = []
for rot, d in [('C2-A', pa), ('C2-B', pb)]:
    bl = d[d['particao'] == 'TESTE']
    linhas.append({'versao': rot, 'n_meses': len(bl),
                   '> indice interno': int((bl['liquido'] > bl['ret_indice']).sum()),
                   '% ': 100 * float((bl['liquido'] > bl['ret_indice']).mean()),
                   '> indice ex-max': int((bl['liquido'] > bl['ret_indice_ex_max']).sum()),
                   '% .': 100 * float((bl['liquido'] > bl['ret_indice_ex_max']).mean()),
                   '> CDI': int((bl['liquido'] > bl['ret_cdi']).sum()),
                   '% ..': 100 * float((bl['liquido'] > bl['ret_cdi']).mean())})
print(pd.DataFrame(linhas).round(2).to_string(index=False))

# ============================================================ SANIDADE
print('\n' + '=' * 118)
print('SANIDADE')
print('=' * 118)

falhas = []
for rot, d in [('C2-A', pa), ('C2-B', pb)]:
    for part in ['TREINO', 'TESTE']:
        bl = d[d['particao'] == part]
        b, l = anualizado(bl['bruto']), anualizado(bl['liquido'])
        ok = l < b
        print(f'S1 {rot} {part}: bruto {b:.4f}%aa > liquido {l:.4f}%aa ? {ok}')
        if not ok:
            falhas.append(f'S1 {rot} {part}')
        assert (bl['liquido'] <= bl['bruto'] + 1e-15).all(), f'S1 FALHOU mes a mes em {rot} {part}'
assert not falhas, 'S1 FALHOU'
print('   S1 PASSOU')

print(f'S2 C2-B soma (acoes + CDI): maior desvio de 1,0 = {(soma_b - 1.0).abs().max():.3e}')
assert (soma_b - 1.0).abs().max() < 1e-9, 'S2 FALHOU'
print('   S2 PASSOU')

print('S3 posicoes sem preco em T+1 -- nenhuma recebeu retorno 0,0 por PREENCHIMENTO:')
n_nan = 0
for r in sem_preco_c1.itertuples():
    d_ret = data_de_mes[r.mes_ret]
    v = wide.loc[d_ret, r.CODISI]
    if pd.isna(v):
        n_nan += 1
print(f'   dos {len(sem_preco_c1)} casos, {n_nan} continuam NaN na matriz de retornos entregue ao motor')
assert n_nan == len(sem_preco_c1), 'S3 FALHOU: algum retorno foi preenchido'
casos = set(zip(sem_preco_c1['mes_ret'], sem_preco_c1['CODISI']))
logados = set(zip(log_a['mes_inicio'], log_a['ativo']))
print(f'   dos {len(casos)} casos, {len(casos & logados)} aparecem no log de congelamento do motor')
assert casos <= logados, f'S3 FALHOU: casos sem evento no motor: {sorted(casos - logados)}'
print('   S3 PASSOU (marcacao congelada dentro do motor, serie de retorno intocada e ainda NaN)')

print('S4 conta do custo em C2-A:')
print('   NOTA DE METODO, escrita depois de a primeira versao desta sanidade ter FALHADO e o numero')
print('   ter sido investigado em vez de contornado. A formula da sanidade -- giro x 2 x 50bps x 12 --')
print('   e ARITMETICA (soma de 12 custos mensais). Minha primeira implementacao comparou essa')
print('   formula contra a diferenca dos retornos ANUALIZADOS (bruto_aa - liquido_aa), que e')
print('   GEOMETRICA. No TREINO isso deu razao 1,2025 e a sanidade quebrou. Nao e divergencia de')
print('   convencao do motor: e composicao -- a mesma classe de artefato de Jensen que A6c ja tinha')
print('   descartado ("explica 256% do excesso"). As duas medidas sao impressas abaixo, e a prova')
print('   esta no teste do custo CONSTANTE, que reproduz o mesmo desvio sem giro nenhum variando.')
for part in ['TREINO', 'TESTE']:
    bl = pa[pa['particao'] == part]
    giro_ow = bl['turnover'].mean() / 2
    esperado = giro_ow * 2 * (CUSTO_BPS / 10000.0) * 12
    efetivo_arit = float(bl['custo'].mean()) * 12
    efetivo_geom = anualizado(bl['bruto']) / 100 - anualizado(bl['liquido']) / 100
    razao = efetivo_arit / esperado
    # prova: mesmo custo, mas CONSTANTE e igual a media -- se o desvio geometrico persistir,
    # ele nao vem da convencao de giro, vem da composicao.
    r_const = bl['bruto'] - float(bl['custo'].mean())
    geom_const = anualizado(bl['bruto']) / 100 - anualizado(r_const) / 100
    print(f'   {part}: giro one-way {100 * giro_ow:.4f}%/mes x 2 x {CUSTO_BPS:.0f}bps x 12 = '
          f'{100 * esperado:.4f} pp/ano ESPERADO (aritmetico)')
    print(f'           efetivo do motor, aritmetico (media mensal x 12): {100 * efetivo_arit:.4f} pp/ano '
          f'-> razao {razao:.6f}')
    print(f'           efetivo do motor, geometrico (bruto_aa - liquido_aa): {100 * efetivo_geom:.4f} pp/ano '
          f'-> razao {efetivo_geom / esperado:.4f}')
    print(f'           PROVA: com custo CONSTANTE de {10000 * bl["custo"].mean():.2f} bps/mes (giro fixo, '
          f'zero variacao de convencao), o desvio geometrico e {100 * geom_const:.4f} pp/ano '
          f'-> razao {geom_const / esperado:.4f}')
    assert abs(razao - 1.0) <= 0.20, f'S4 FALHOU em {part}: razao aritmetica {razao:.6f}'
print('   S4 PASSOU na medida que a propria formula descreve (aritmetica): razao 1,000000 nas duas')
print('   particoes -- o motor cobra exatamente turnover x bps/10.000, sem convencao escondida.')
print('   O desvio geometrico do TREINO (1,2025) e integralmente reproduzido pelo teste de custo')
print('   constante, o que o identifica como composicao e nao como convencao de giro.')

c1_teste_aa = 5.4772
c2a_teste_aa = anualizado(pa[pa['particao'] == 'TESTE']['bruto'])
print(f'S5 bruto de C2-A no TESTE {c2a_teste_aa:.4f}%aa vs C1 {c1_teste_aa:.4f}%aa; '
      f'diferenca {c2a_teste_aa - c1_teste_aa:+.4f} pp')
assert abs(c2a_teste_aa - c1_teste_aa) <= 0.1, 'S5 FALHOU'
print('   S5 PASSOU')

# ============================================================ gravacao
saida = pd.concat([pa, pb], ignore_index=True)
saida.to_parquet(DIR_INT + r'\c2_backtest.parquet', index=False)
pes_b_out = pes_b.copy()
pes_b_out.index = [mes_de_data[i] for i in pes_b_out.index]
sel_b_out = sel_b[['mes', 'CODISI', 'CODNEG', 'particao', 'modelo', 'score', 'rk', 'K', 'M', 'n_gate', 'peso']]
sel_b_out.to_parquet(DIR_INT + r'\c2_trilha_leituraB.parquet', index=False)
print('\n' + '=' * 118)
print(f'GRAVADO intermediario\\c2_backtest.parquet          {saida.shape}')
print(f'GRAVADO intermediario\\c2_trilha_leituraB.parquet   {sel_b_out.shape}')
print('C2 CONCLUIDA -- as duas leituras do gate reportadas (D27), nenhuma escolhida.')
print('=' * 118)


C2 -- BACKTEST COM O MOTOR LOCAL (src\motor.py), DUAS LEITURAS DO GATE (D27)

(a) PASSO 0 -- ASSINATURA DO MODULO src\motor.py (nao adivinhada: lida por inspect)
arquivo: C:\Users\lucca\quant2026\src\motor.py
funcoes publicas: ['gerar_datas_rebalanceamento', 'aplicar_pesos', 'calcular_turnover', 'aplicar_custos', 'serie_riqueza', 'rodar_backtest']
constantes de modulo: TOL_SOMA_PESOS = 1e-06

----------------------------------------------------------------------------------------------------------------------
gerar_datas_rebalanceamento(indice, frequencia)
----------------------------------------------------------------------------------------------------------------------
    indice  -> SEM DEFAULT (obrigatorio)
    frequencia  -> SEM DEFAULT (obrigatorio)
    docstring:
      Deriva as datas de rebalanceamento a partir do calendário de pregões.
      
      O QUE FAZ: escolhe, dentro do calendário REAL informado, as datas em que a
      carteira é redecidida. Para frequência de calen

## C3 (curva de custo) + D1 (comparadores e diagnosticos)

Celulas de MEDICAO. Nenhum parametro do MODEL_SPEC (congelado 2026-08-15 18:09:30) e
alterado, criado ou reajustado. DIAG-2 e explicitamente PROIBIDO de alimentar selecao de
features (D23), e os IC individuais que ele reporta sao medidos direto no teste, sem
walk-forward -- usa-los para escolher feature seria escolher no proprio teste.

**C3** recalcula o liquido em 0/10/25/50/75/100 bps e resolve o breakeven contra cada
comparador. **D1** mede quatro comparadores especificados ANTES de qualquer um deles ser
rodado: EW do universo elegivel, 200 selecoes aleatorias com a mesma construcao (semente
20260815), selecao invertida, e o IBOV como secundario.

O motor roda UMA vez por trilha a custo zero; o custo entra depois pela aritmetica
identica a `aplicar_custos()` (conferida contra a saida de C2 com desvio maximo de 0,0).
Isso e necessario porque o modulo, por desenho, rejeita `custo_bps` negativo, e o solver
de breakeven precisa atravessar o zero.

Nenhuma dependencia nova foi instalada: `scipy` nao existe nesta `.venv`, entao Spearman
e Pearson sao calculados na propria celula, com a mesma convencao `method='average'` de B2.

Todo resultado aqui e NAO-REFUTACAO: [F] o poder deste desenho exigiria ~331 meses.


In [1]:
import sys
import time

import numpy as np
import pandas as pd

if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8')

pd.set_option('display.width', 270)
pd.set_option('display.max_columns', 80)
pd.set_option('display.max_rows', 400)

RAIZ = r'C:\Users\lucca\quant2026'
DIR_INT = RAIZ + r'\intermediario'
if RAIZ + r'\src' not in sys.path:
    sys.path.insert(0, RAIZ + r'\src')
import motor  # noqa: E402

# ---- correlacoes sem scipy (nao instalado nesta .venv; PROIBIDO instalar fora dela) ----
def _pearson(x, y):
    x = np.asarray(x, dtype=float); y = np.asarray(y, dtype=float)
    x = x - x.mean(); y = y - y.mean()
    den = np.sqrt(float(x @ x) * float(y @ y))
    return float(x @ y) / den if den > 0 else np.nan


def spearman(x, y):
    """Spearman = Pearson sobre os postos, method='average' (mesma convencao de B2/B5)."""
    rx = pd.Series(np.asarray(x, dtype=float)).rank(method='average').to_numpy()
    ry = pd.Series(np.asarray(y, dtype=float)).rank(method='average').to_numpy()
    return _pearson(rx, ry)


def t_de_rho(rho, n):
    if not np.isfinite(rho) or n <= 2 or abs(rho) >= 1:
        return np.nan
    return rho * np.sqrt((n - 2) / (1 - rho ** 2))

SEMENTE = 20260815          # semente fixa e DECLARADA (comparador 2)
N_ALEATORIAS = 200
NIVEIS_BPS = [0, 10, 25, 50, 75, 100]
RANKS = ['mom_1m_rank', 'mom_3m_rank', 'mom_6m_rank', 'mom_12m_rank', 'vol_3m_rank', 'vol_6m_rank']

print('=' * 122)
print('C3 (curva de custo) + D1 (comparadores) -- CELULAS DE MEDICAO. Nenhum parametro do')
print('MODEL_SPEC (congelado 2026-08-15 18:09:30) e alterado, criado ou reajustado aqui.')
print('=' * 122)

# ==================================================================== insumos
prev = pd.read_parquet(DIR_INT + r'\b4b5_previsoes.parquet')
trilha_a = pd.read_parquet(DIR_INT + r'\c1_carteira.parquet')
trilha_b = pd.read_parquet(DIR_INT + r'\c2_trilha_leituraB.parquet')
bench = pd.read_parquet(DIR_INT + r'\a6_benchmark_e_rf.parquet')
split = pd.read_parquet(DIR_INT + r'\a7_split.parquet')

prev['CODISI'] = prev['CODISI'].astype(str)
for d in (trilha_a, trilha_b):
    d['CODISI'] = d['CODISI'].astype(str)
split['CODISI'] = split['CODISI'].astype(str)

meses_all = sorted(bench['mes'].unique())
data_de_mes = {m: pd.Period(m, freq='M').to_timestamp(how='end').normalize() for m in meses_all}
mes_de_data = {v: k for k, v in data_de_mes.items()}
meses_dec = sorted(prev['mes'].unique())
part_por_mes = prev.groupby('mes')['particao'].first().to_dict()
datas_dec = pd.DatetimeIndex([data_de_mes[m] for m in meses_dec])

wide = split.pivot_table(index='mes', columns='CODISI', values='ret_1m', aggfunc='first').reindex(meses_all)
wide.index = pd.DatetimeIndex([data_de_mes[m] for m in meses_all])
cdi_s = bench.set_index('mes')['ret_cdi'].reindex(meses_all)
cdi_s.index = wide.index
wide['__CDI__'] = cdi_s.values
CAIXA = wide['__CDI__']

bench_idx = bench.set_index('mes')
print(f'matriz de retornos {wide.shape}; meses de decisao {len(meses_dec)} '
      f'({meses_dec[0]} a {meses_dec[-1]})')


# ==================================================================== motor
def rodar_trilha(pesos_df, rotulo, silencioso=False):
    """pesos_df: index=mes (str), colunas=CODISI (+ __CDI__). Roda o motor a custo 0."""
    p = pesos_df.copy()
    cols = [c for c in p.columns if (p[c] != 0).any()]
    p = p[cols].fillna(0.0)
    p.index = pd.DatetimeIndex([data_de_mes[m] for m in p.index])
    out = motor.rodar_backtest(p, wide[cols], custo_bps=0.0, retornos_caixa=CAIXA)
    rb, tv = out['retornos_brutos'], out['turnover'].reindex(out['retornos_brutos'].index)
    d = pd.DataFrame({'bruto': rb.values, 'turnover': tv.values},
                     index=[mes_de_data[i] for i in rb.index])
    d.index.name = 'mes_ret'
    d = d.reset_index()
    d['mes'] = [str(pd.Period(m, freq='M') - 1) for m in d['mes_ret']]
    d['particao'] = d['mes'].map(part_por_mes)
    if not silencioso:
        print(f'  [{rotulo}] ativos {len(cols)}  meses {len(d)}  '
              f'congelamentos {len(out["eventos_congelamento"])}  '
              f'descartados {out["rebal_descartados"]}')
    return d, out


def liquido(d, bps):
    return d['bruto'] - d['turnover'] * (bps / 10000.0)


def anual(r):
    r = np.asarray(pd.Series(r).dropna(), dtype=float)
    return 100.0 * ((np.prod(1.0 + r)) ** (12.0 / len(r)) - 1.0)


def nw_t(x, lags=3):
    x = np.asarray(pd.Series(x).dropna(), dtype=float)
    n = len(x)
    mu = x.mean()
    e = x - mu
    var = float(e @ e) / n
    for l in range(1, lags + 1):
        var += 2.0 * (1.0 - l / (lags + 1.0)) * (float(e[l:] @ e[:-l]) / n)
    se = np.sqrt(var / n)
    return mu, se, mu / se


def dd_max(r):
    r = np.asarray(pd.Series(r).dropna(), dtype=float)
    w = np.cumprod(1 + r)
    pico = np.maximum.accumulate(w)
    return 100.0 * float((w / pico - 1.0).min())


# ==================================================================== trilhas A e B
def matriz_pesos(df, col='peso'):
    m = df.pivot_table(index='mes', columns='CODISI', values=col, aggfunc='sum').reindex(meses_dec)
    m = m.fillna(0.0)
    m['__CDI__'] = 0.0
    return m


pes_a = matriz_pesos(trilha_a)
pes_b = matriz_pesos(trilha_b)
n_gate_b = trilha_b.groupby('mes')['n_gate'].first()
M_b = trilha_b.groupby('mes')['M'].first()
pes_b['__CDI__'] = (1.0 - (n_gate_b / M_b).reindex(meses_dec)).values

print('\nrodando o motor a custo ZERO (o custo entra depois, por aplicar_custos/aritmetica identica):')
d_a, out_a = rodar_trilha(pes_a, 'C2-A')
d_b, out_b = rodar_trilha(pes_b, 'C2-B')

# conferencia contra C2 (que rodou com custo_bps=50 dentro do motor)
c2 = pd.read_parquet(DIR_INT + r'\c2_backtest.parquet')
for rot, d in [('C2-A', d_a), ('C2-B', d_b)]:
    ref = c2[c2['versao'] == rot].set_index('mes')
    dv = (liquido(d, 50).values - ref.loc[d['mes'], 'liquido'].values)
    print(f'  conferencia {rot}: maior |liquido(50bps) recalculado - C2| = {np.abs(dv).max():.3e}')
    assert np.abs(dv).max() < 1e-15, 'recalculo de custo diverge do motor'
print('  => a formula direta (bruto - giro*bps/1e4) e IDENTICA a aplicar_custos(). Uso ela')
print('     no solver de breakeven porque o modulo, por desenho, rejeita custo_bps negativo.')

for m in bench_idx.columns:
    pass
for d in (d_a, d_b):
    for c in ['ret_indice', 'ret_indice_ex_max', 'ret_cdi', 'ret_ibov']:
        d[c] = d['mes_ret'].map(bench_idx[c])

# ==================================================================== C3
print('\n' + '=' * 122)
print('CELULA C3 -- CURVA DE CUSTO E BREAKEVEN')
print('=' * 122)

print('\n(a) RETORNO ANUALIZADO LIQUIDO POR NIVEL DE CUSTO, E A DIFERENCA CONTRA CADA COMPARADOR')
linhas = []
for rot, d in [('C2-A', d_a), ('C2-B', d_b)]:
    for part in ['TREINO', 'TESTE']:
        bl = d[d['particao'] == part]
        ref_i, ref_x, ref_c = anual(bl['ret_indice']), anual(bl['ret_indice_ex_max']), anual(bl['ret_cdi'])
        for bps in NIVEIS_BPS:
            a = anual(liquido(bl, bps))
            linhas.append({'versao': rot, 'particao': part, 'custo_bps': bps, 'liquido_%aa': a,
                           'vs indice': a - ref_i, 'vs ex-max': a - ref_x, 'vs CDI': a - ref_c})
tab_a = pd.DataFrame(linhas)
print(tab_a.round(4).to_string(index=False))
print('\nreferencias no periodo: TESTE indice 7,1873 / ex-max 3,0731 / CDI 9,0315 %aa; '
      'TREINO 22,4266 / 15,7325 / 13,1695 %aa')

print('\n(b) BREAKEVEN CONTRA CADA COMPARADOR (o custo_bps em que a carteira IGUALA o comparador)')


def breakeven(bl, col_ref):
    alvo = anual(bl[col_ref])
    f = lambda b: anual(liquido(bl, b)) - alvo
    lo, hi = -3000.0, 3000.0
    if f(lo) < 0:
        return np.nan
    if f(hi) > 0:
        return np.inf
    for _ in range(200):
        mid = (lo + hi) / 2
        if f(mid) > 0:
            lo = mid
        else:
            hi = mid
    return (lo + hi) / 2


linhas = []
for rot, d in [('C2-A', d_a), ('C2-B', d_b)]:
    for part in ['TREINO', 'TESTE']:
        bl = d[d['particao'] == part]
        linha = {'versao': rot, 'particao': part, 'liquido a 0bps %aa': anual(bl['bruto'])}
        for col, nome in [('ret_indice', 'indice interno'), ('ret_indice_ex_max', 'indice ex-max'),
                          ('ret_cdi', 'CDI')]:
            linha[f'breakeven vs {nome} (bps)'] = breakeven(bl, col)
        linhas.append(linha)
print(pd.DataFrame(linhas).round(2).to_string(index=False))
print('\nLEITURA EXPLICITA DOS BREAKEVENS NEGATIVOS:')
for rot, d in [('C2-A', d_a), ('C2-B', d_b)]:
    bl = d[d['particao'] == 'TESTE']
    b0 = anual(bl['bruto'])
    for col, nome in [('ret_indice', 'indice interno'), ('ret_indice_ex_max', 'indice ex-max'),
                      ('ret_cdi', 'CDI')]:
        be = breakeven(bl, col)
        alvo = anual(bl[col])
        if be < 0:
            print(f'  {rot} vs {nome}: breakeven {be:.1f} bps -- NEGATIVO. A carteira ja perde a CUSTO ZERO '
                  f'({b0:.4f}%aa contra {alvo:.4f}%aa). Nao existe nivel de custo que a salve: seria preciso '
                  f'RECEBER {abs(be):.1f} bps por rebalanceamento para empatar.')
        elif np.isfinite(be):
            print(f'  {rot} vs {nome}: breakeven {be:.1f} bps -- positivo: a carteira empata a esse custo.')
print(f'\n[F] marco: breakeven de ~53 bps contra o IBOV. Aqui, contra o indice interno, o breakeven '
      f'e negativo nas duas leituras no TESTE.')

print('\n(c) CUSTO PAGO EM pp/ano E A FRACAO DO RETORNO BRUTO QUE ELE CONSOME')
linhas = []
for rot, d in [('C2-A', d_a), ('C2-B', d_b)]:
    for part in ['TREINO', 'TESTE']:
        bl = d[d['particao'] == part]
        bruto_aa = anual(bl['bruto'])
        for bps in NIVEIS_BPS:
            custo_arit = float((bl['turnover'] * bps / 10000.0).mean()) * 12 * 100
            custo_geo = bruto_aa - anual(liquido(bl, bps))
            linhas.append({'versao': rot, 'particao': part, 'custo_bps': bps,
                           'custo_aritm_pp_ano': custo_arit, 'custo_geom_pp_ano': custo_geo,
                           'bruto_%aa': bruto_aa,
                           '% do bruto consumido (geom)': 100 * custo_geo / bruto_aa if bruto_aa != 0 else np.nan})
print(pd.DataFrame(linhas).round(4).to_string(index=False))
print('as duas colunas de custo estao lado a lado por causa da nota de metodo de C2: a aritmetica e')
print('a soma dos 12 custos mensais; a geometrica inclui composicao (Jensen). Nao sao a mesma coisa.')

print('\n(d) GIRO ONE-WAY DE C2-B POR ANO, E A CORRELACAO GIRO x RETORNO MENSAL')
d_b['ano'] = d_b['mes'].str[:4]
d_a['ano'] = d_a['mes'].str[:4]
gb = d_b.groupby(['particao', 'ano'])['turnover'].agg(['size', 'mean', 'median'])
gb[['mean', 'median']] = 100 * gb[['mean', 'median']] / 2
print(gb.round(2).to_string())
linhas = []
for rot, d in [('C2-A', d_a), ('C2-B', d_b)]:
    for part in ['TREINO', 'TESTE']:
        bl = d[d['particao'] == part]
        rp = _pearson(bl['turnover'], bl['bruto'])
        rs = spearman(bl['turnover'], bl['bruto'])
        rl = _pearson(bl['turnover'], liquido(bl, 50))
        n = len(bl)
        linhas.append({'versao': rot, 'particao': part, 'n': n,
                       'pearson giro x bruto': rp, 't': t_de_rho(rp, n),
                       'spearman giro x bruto': rs, 't.': t_de_rho(rs, n),
                       'pearson giro x liquido50': rl, 't..': t_de_rho(rl, n)})
print('\ncorrelacao entre o giro do mes e o retorno do mesmo mes:')
print(pd.DataFrame(linhas).round(4).to_string(index=False))

# ==================================================================== D1
print('\n' + '=' * 122)
print('CELULA D1 -- COMPARADORES')
print('=' * 122)

# ---------------- COMPARADOR 1: EW do universo elegivel
print('\nCOMPARADOR 1 -- EQUAL-WEIGHT DO UNIVERSO ELEGIVEL (o mais importante do projeto)')
eleg = split.loc[split['elegivel'], ['mes', 'CODISI']]
eleg = eleg[eleg['mes'].isin(meses_dec)]
n_el = eleg.groupby('mes')['CODISI'].size()
eleg = eleg.merge(n_el.rename('n_el'), on='mes')
eleg['peso'] = 1.0 / eleg['n_el']
pes_ew = matriz_pesos(eleg)
print(f'  universo: mediana de {n_el.reindex(meses_dec).median():.0f} elegiveis/mes; '
      f'{pes_ew.shape[1] - 1} ISINs distintos ao longo de {len(meses_dec)} meses')
d_ew, out_ew = rodar_trilha(pes_ew, 'EW universo')
for c in ['ret_indice', 'ret_indice_ex_max', 'ret_cdi']:
    d_ew[c] = d_ew['mes_ret'].map(bench_idx[c])
d_ew['ano'] = d_ew['mes'].str[:4]

linhas = []
for part in ['TREINO', 'TESTE']:
    bl = d_ew[d_ew['particao'] == part]
    for bps in [0, 10, 25, 50]:
        linhas.append({'particao': part, 'custo_bps': bps, 'liquido_%aa': anual(liquido(bl, bps)),
                       'indice_%aa': anual(bl['ret_indice']),
                       'dif pp/ano': anual(liquido(bl, bps)) - anual(bl['ret_indice']),
                       'giro_one_way_%': 100 * bl['turnover'].mean() / 2})
print(pd.DataFrame(linhas).round(4).to_string(index=False))

print('\n  DECOMPOSICAO DA DIFERENCA EW(0bps) x INDICE INTERNO -- ela NAO e zero por construcao:')
bl = d_ew[d_ew['particao'] == 'TESTE']
print(f'  EW a custo 0: {anual(bl["bruto"]):.4f}%aa   indice interno: {anual(bl["ret_indice"]):.4f}%aa   '
      f'diferenca {anual(bl["bruto"]) - anual(bl["ret_indice"]):+.4f} pp/ano')
print('  As duas series NAO sao a mesma coisa, e a diferenca tem duas fontes mensuraveis:')
print('   (1) CONJUNTO: o indice do mes m e a media dos elegiveis EM m; a carteira EW decidida em T')
print('       carrega os elegiveis DE T e captura o retorno de T+1. Quem entra na elegibilidade em')
print('       T+1 esta no indice e nao na carteira, e vice-versa.')
print('   (2) CONGELAMENTO: posicao sem preco em T+1 contribui zero na carteira (motor); no indice')
print('       ela simplesmente nao entra na media.')
ent_sai = []
eleg_set = split.loc[split['elegivel']].groupby('mes')['CODISI'].apply(set)
for m in meses_dec:
    m1 = str(pd.Period(m, freq='M') + 1)
    if m1 not in eleg_set.index:
        continue
    a, b = eleg_set[m], eleg_set[m1]
    ent_sai.append({'mes': m, 'particao': part_por_mes[m], 'n_T': len(a), 'n_T1': len(b),
                    'saem': len(a - b), 'entram': len(b - a),
                    'pct_dif': 100 * len(a ^ b) / len(a | b)})
es = pd.DataFrame(ent_sai)
print('\n  tamanho da divergencia de conjunto (elegiveis em T vs em T+1):')
print(es.groupby('particao')[['n_T', 'saem', 'entram', 'pct_dif']].median().round(2).to_string())
print(f'  no TESTE, mediana de {es[es.particao == "TESTE"]["saem"].median():.0f} saidas e '
      f'{es[es.particao == "TESTE"]["entram"].median():.0f} entradas por mes sobre '
      f'{es[es.particao == "TESTE"]["n_T"].median():.0f} nomes.')
cong_ew = out_ew['eventos_congelamento']
print(f'  congelamentos na carteira EW: {len(cong_ew)} '
      f'(desfechos: {cong_ew["desfecho"].value_counts().to_dict() if len(cong_ew) else "{}"})')

# ---------------- COMPARADOR 2: 200 aleatorias
print(f'\nCOMPARADOR 2 -- {N_ALEATORIAS} SELECOES ALEATORIAS, MESMA CONSTRUCAO (semente {SEMENTE})')
print('  regra: em cada mes, dentro de CADA modelo que passou no gate, sorteia o MESMO numero de')
print('  ativos que a estrategia real selecionou naquele modelo, do pool de candidatos daquele')
print('  modelo naquele mes; peso igual sobre o total, mesmo motor, mesmo custo.')
pool = prev.groupby(['mes', 'modelo'])['CODISI'].apply(list).to_dict()
alvo_n = trilha_a.groupby(['mes', 'modelo']).size().to_dict()
chaves = sorted(alvo_n.keys())
isins_pool = sorted({i for k in chaves for i in pool[k]})
pos_isin = {c: i for i, c in enumerate(isins_pool)}
pos_mes = {m: i for i, m in enumerate(meses_dec)}
print(f'  {len(chaves)} pares (mes, modelo) alocam; pool com {len(isins_pool)} ISINs distintos')

rng = np.random.default_rng(SEMENTE)
t0 = time.time()
res_rand = []
W = np.zeros((len(meses_dec), len(isins_pool)))
cols_rand = isins_pool + ['__CDI__']
for s in range(N_ALEATORIAS):
    W[:] = 0.0
    for (m, mod) in chaves:
        cand = pool[(m, mod)]
        k = alvo_n[(m, mod)]
        esc = rng.choice(len(cand), size=k, replace=False)
        i = pos_mes[m]
        for j in esc:
            W[i, pos_isin[cand[j]]] += 1.0
    tot = W.sum(axis=1, keepdims=True)
    Wn = W / tot
    pw = pd.DataFrame(Wn, index=meses_dec, columns=isins_pool)
    pw['__CDI__'] = 0.0
    d_r, _ = rodar_trilha(pw, f'aleatoria {s}', silencioso=True)
    res_rand.append(d_r[['mes', 'particao', 'bruto', 'turnover']])
    if s == 4:
        print(f'  (5 rodadas em {time.time() - t0:.1f}s -- estimativa total '
              f'{(time.time() - t0) / 5 * N_ALEATORIAS:.0f}s)')
print(f'  {N_ALEATORIAS} rodadas concluidas em {time.time() - t0:.1f}s')

dist = {}
for bps in [0, 50]:
    v = np.array([anual(r['bruto'] - r['turnover'] * bps / 10000.0)
                  for r in [x[x['particao'] == 'TESTE'] for x in res_rand]])
    dist[bps] = v
real_a = {bps: anual(liquido(d_a[d_a['particao'] == 'TESTE'], bps)) for bps in [0, 50]}
real_b = {bps: anual(liquido(d_b[d_b['particao'] == 'TESTE'], bps)) for bps in [0, 50]}
ew_teste = {bps: anual(liquido(d_ew[d_ew['particao'] == 'TESTE'], bps)) for bps in [0, 50]}
linhas = []
for bps in [0, 50]:
    v = dist[bps]
    linhas.append({'custo_bps': bps, 'min': v.min(), 'p5': np.percentile(v, 5), 'p25': np.percentile(v, 25),
                   'mediana': np.median(v), 'p75': np.percentile(v, 75), 'p95': np.percentile(v, 95),
                   'max': v.max(), 'media': v.mean(), 'dp': v.std(ddof=1)})
print('\n  distribuicao do retorno anualizado das 200 no TESTE (%aa):')
print(pd.DataFrame(linhas).round(4).to_string(index=False))
for bps in [0, 50]:
    v = dist[bps]
    pa_ = 100.0 * float((v < real_a[bps]).mean())
    pb_ = 100.0 * float((v < real_b[bps]).mean())
    print(f'  a {bps} bps: C2-A real = {real_a[bps]:.4f}%aa -> percentil {pa_:.1f} das 200;  '
          f'C2-B real = {real_b[bps]:.4f}%aa -> percentil {pb_:.1f}')
print('  LEITURA OBRIGATORIA: isto e NAO-REFUTACAO, nunca confirmacao. O percentil diz onde a')
print('  estrategia cai numa distribuicao de 200 sorteios com a MESMA construcao; nao mede')
print('  capacidade preditiva, e [F] o poder exigiria ~331 meses para p<0,05.')

# ---------------- COMPARADOR 3: selecao invertida
print('\nCOMPARADOR 3 -- SELECAO INVERTIDA (K PIORES por modelo, gate invertido)')
print('  construcao declarada: aplica-se a MESMA maquina de C1 sobre -score. Isso espelha as duas')
print('  regras de uma vez -- K=2 sse (-s)[3] < (-s)[1]/2, e gate media(-s)>0, que e exatamente')
print('  "aloca se a media dos piores for < 0". Nenhuma regra nova foi inventada.')
pv = prev.copy()
pv['score_inv'] = -pv['score']
pv = pv.sort_values(['mes', 'modelo', 'score_inv'], ascending=[True, True, False], kind='mergesort')
pv['rk'] = pv.groupby(['mes', 'modelo']).cumcount()
t3 = pv[pv['rk'] < 3]
pi = t3.pivot_table(index=['mes', 'modelo'], columns='rk', values='score_inv', aggfunc='first')
pi.columns = ['s1', 's2', 's3']
pi = pi.reset_index()
pi['K'] = np.where(pi['s3'] < pi['s1'] * 0.5, 2, 3)
pi['media_K'] = np.where(pi['K'] == 2, (pi['s1'] + pi['s2']) / 2, (pi['s1'] + pi['s2'] + pi['s3']) / 3)
pi['gate'] = pi['media_K'] > 0
sel_i = t3.merge(pi[['mes', 'modelo', 'K', 'gate']], on=['mes', 'modelo'], how='left')
sel_i = sel_i[(sel_i['rk'] < sel_i['K']) & sel_i['gate']].copy()
n_i = sel_i.groupby('mes').size().rename('n')
sel_i = sel_i.merge(n_i, on='mes')
sel_i['peso'] = 1.0 / sel_i['n']
pes_i = matriz_pesos(sel_i)
print(f'  modelos que passam no gate invertido: {int(pi["gate"].sum())} de {len(pi)}; '
      f'o gate NORMAL passava {len(chaves)} de {len(pi)}')
print('  o gate invertido passa MAIS modelos porque, na maioria dos grupos, a media dos 3 PIORES')
print('  scores e negativa -- simetrico do fato de a media dos 3 melhores ser positiva na maioria.')
print(f'  posicoes: {len(sel_i)} (a real tinha {len(trilha_a)})')
meses_i = sorted(sel_i['mes'].unique())
falt = [m for m in meses_dec if m not in set(meses_i)]
print(f'  meses SEM alocacao na invertida (100% CDI): {len(falt)}')
if falt:
    pes_i.loc[falt, '__CDI__'] = 1.0
d_i, out_i = rodar_trilha(pes_i, 'invertida')
for c in ['ret_indice', 'ret_indice_ex_max', 'ret_cdi']:
    d_i[c] = d_i['mes_ret'].map(bench_idx[c])
linhas = []
for part in ['TREINO', 'TESTE']:
    for rot, d in [('REAL C2-A', d_a), ('INVERTIDA', d_i)]:
        bl = d[d['particao'] == part]
        linhas.append({'particao': part, 'versao': rot, 'bruto_%aa': anual(bl['bruto']),
                       'liquido50_%aa': anual(liquido(bl, 50)),
                       'giro_one_way_%': 100 * bl['turnover'].mean() / 2,
                       'media_%mes': 100 * bl['bruto'].mean()})
print(pd.DataFrame(linhas).round(4).to_string(index=False))
bl_r = d_a[d_a['particao'] == 'TESTE']
bl_i = d_i[d_i['particao'] == 'TESTE']
dif = bl_r['bruto'].values - bl_i['bruto'].values
mu, se, t = nw_t(dif)
print(f'\n  REAL menos INVERTIDA, bruto, no TESTE: {100 * mu:+.4f}%/mes, ep NW(3) {100 * se:.4f}, '
      f't = {t:+.4f}')
print(f'  em anualizado: real {anual(bl_r["bruto"]):.4f}%aa contra invertida {anual(bl_i["bruto"]):.4f}%aa '
      f'({anual(bl_r["bruto"]) - anual(bl_i["bruto"]):+.4f} pp/ano)')

# ---------------- COMPARADOR 4: IBOV
print('\nCOMPARADOR 4 -- IBOV (SECUNDARIO: a serie do CSV termina em 2025-12)')
ib = d_a[['mes', 'mes_ret', 'particao', 'ret_ibov']].dropna()
ib_t = ib[ib['particao'] == 'TESTE']
print(f'  meses de TESTE com IBOV: {len(ib_t)} de 103 ({ib_t["mes_ret"].min()} a {ib_t["mes_ret"].max()})')
sub = d_a[d_a['mes_ret'].isin(set(ib_t['mes_ret']))]
sub_b = d_b[d_b['mes_ret'].isin(set(ib_t['mes_ret']))]
sub_ew = d_ew[d_ew['mes_ret'].isin(set(ib_t['mes_ret']))]
print(f'  no MESMO subperiodo ({len(sub)} meses): IBOV {anual(ib_t["ret_ibov"]):.4f}%aa | '
      f'C2-A liq50 {anual(liquido(sub, 50)):.4f} | C2-B liq50 {anual(liquido(sub_b, 50)):.4f} | '
      f'EW liq50 {anual(liquido(sub_ew, 50)):.4f} | indice {anual(sub["ret_indice"]):.4f} | '
      f'CDI {anual(sub["ret_cdi"]):.4f}')
print('  DECLARADO SECUNDARIO: o IBOV e total-return-parcial de fonte externa, o resto do projeto e')
print('  price-only sobre o universo interno (D12). A comparacao e informativa, nao e o benchmark.')

# ==================================================================== DIAGNOSTICOS
print('\n' + '=' * 122)
print('DIAGNOSTICOS -- MEDEM, NAO AJUSTAM')
print('=' * 122)

# ---------------- DIAG-1
print('\nDIAG-1 -- PERSISTENCIA DO SCORE (Spearman cross-seccional entre T e T+1)')
pv2 = prev[['mes', 'CODISI', 'score'] + RANKS].copy()
pv2['mes_prox'] = [str(pd.Period(m, freq='M') + 1) for m in pv2['mes']]
prox = pv2[['mes', 'CODISI', 'score'] + RANKS].rename(
    columns={'mes': 'mes_prox', 'score': 'score_t1', **{r: r + '_t1' for r in RANKS}})
par = pv2.merge(prox, on=['mes_prox', 'CODISI'], how='inner')
linhas = []
for m, g in par.groupby('mes'):
    if len(g) < 10:
        continue
    reg = {'mes': m, 'particao': part_por_mes[m], 'n': len(g),
           'rho_score': spearman(g['score'], g['score_t1'])}
    for r in RANKS:
        reg['rho_' + r] = spearman(g[r], g[r + '_t1'])
    linhas.append(reg)
per = pd.DataFrame(linhas)
cols_rho = ['rho_score'] + ['rho_' + r for r in RANKS]
print('\n  media e dp da autocorrelacao cross-seccional lag-1, por particao:')
res = per.groupby('particao')[cols_rho].agg(['mean', 'std']).T
print(res.round(4).to_string())
print('\n  serie ANUAL de rho_score e rho_mom_12m_rank:')
per['ano'] = per['mes'].str[:4]
print(per.groupby(['particao', 'ano'])[['rho_score', 'rho_mom_12m_rank', 'rho_mom_1m_rank', 'n']]
      .mean().round(4).to_string())
rs_t = per[per['particao'] == 'TESTE']['rho_score']
print(f'\n  TESTE: rho medio do SCORE = {rs_t.mean():.4f} (dp {rs_t.std():.4f}); '
      f'rho medio de mom_12m_rank = {per[per["particao"] == "TESTE"]["rho_mom_12m_rank"].mean():.4f}')
print('  MECANISMO: o score e uma combinacao linear de ranks com COEFICIENTES que mudam todo mes')
print('  (re-treino mensal). Mesmo com features persistentes, um score pouco persistente produz')
print('  giro alto por RUIDO de coeficiente, nao por informacao nova. Isto e medicao, nao ajuste.')

# ---------------- DIAG-2
print('\nDIAG-2 -- CONTRIBUICAO POR FEATURE (PROIBIDO usar para selecionar feature -- D23)')
b3 = pd.read_parquet(DIR_INT + r'\b3_dataset.parquet')


def mes_idx(s):
    return s.str[:4].astype(int) * 12 + s.str[5:7].astype(int)


def idx_mes(i):
    a, m = divmod(int(i) - 1, 12)
    return f'{a:04d}-{m + 1:02d}'


b3['mes_idx'] = mes_idx(b3['mes'])
b3 = b3.sort_values('mes_idx').reset_index(drop=True)
pu = prev.copy()
pu['mes_idx'] = mes_idx(pu['mes'])
pu = pu.sort_values(['mes_idx']).reset_index(drop=True)


def walk_forward(feats, capturar_coef=False):
    """Reexecuta a MESMA estrutura de B4/B5 (janela [T-37,T-2], D01, OLS com intercepto,
    sem selecao) sobre um subconjunto de features. So diagnostico."""
    Xtr_all_m = b3[feats].to_numpy(dtype='float64')
    ytr = b3['alfa_fut'].to_numpy(dtype='float64')
    sub_tr_all = b3['subsetor'].to_numpy()
    mi_tr = b3['mes_idx'].to_numpy()
    Xp_all = pu[feats].to_numpy(dtype='float64')
    sub_p = pu['subsetor'].to_numpy()
    mi_p = pu['mes_idx'].to_numpy()
    score_out = np.full(len(pu), np.nan)
    modelo_out = np.empty(len(pu), dtype=object)
    coef_g, coef_e = [], []
    for T in sorted(set(mi_p)):
        lo = np.searchsorted(mi_tr, T - 37, 'left')
        hi = np.searchsorted(mi_tr, T - 2, 'right')
        Xt, yt, st = Xtr_all_m[lo:hi], ytr[lo:hi], sub_tr_all[lo:hi]
        plo = np.searchsorted(mi_p, T, 'left')
        phi = np.searchsorted(mi_p, T, 'right')
        if phi == plo or len(yt) < 7:
            continue
        Xp, sp = Xp_all[plo:phi], sub_p[plo:phi]
        Xd = np.column_stack([np.ones(len(Xt)), Xt])
        bg = np.linalg.lstsq(Xd, yt, rcond=None)[0]
        if capturar_coef:
            coef_g.append({'T': idx_mes(T), **{f: bg[i + 1] for i, f in enumerate(feats)},
                           'intercepto': bg[0], 'n_obs': len(yt)})
        subu, cu = np.unique(sp, return_counts=True)
        n_at = dict(zip(subu, cu))
        stu, ct = np.unique(st, return_counts=True)
        n_ob = dict(zip(stu, ct))
        Xdp = np.column_stack([np.ones(len(Xp)), Xp])
        sc = Xdp @ bg
        mo = np.full(len(Xp), 'GENERALISTA', dtype=object)
        for s in subu:
            if s == 'SEM_SETOR' or n_at.get(s, 0) < 5 or n_ob.get(s, 0) < 150:
                continue
            m = st == s
            bs = np.linalg.lstsq(np.column_stack([np.ones(int(m.sum())), Xt[m]]), yt[m], rcond=None)[0]
            mp = sp == s
            sc[mp] = Xdp[mp] @ bs
            mo[mp] = s
            if capturar_coef:
                coef_e.append({'T': idx_mes(T), 'subsetor': s,
                               **{f: bs[i + 1] for i, f in enumerate(feats)},
                               'intercepto': bs[0], 'n_obs': int(m.sum())})
        score_out[plo:phi] = sc
        modelo_out[plo:phi] = mo
    return score_out, modelo_out, pd.DataFrame(coef_g), pd.DataFrame(coef_e)


t0 = time.time()
sc6, mo6, cg6, ce6 = walk_forward(RANKS, capturar_coef=True)
print(f'  reexecucao do walk-forward completo (6 features) em {time.time() - t0:.1f}s')
dif_score = np.abs(sc6 - pu['score'].to_numpy())
print(f'  VERIFICACAO: maior |score reexecutado - score congelado em b4b5| = {np.nanmax(dif_score):.3e}')
assert np.nanmax(dif_score) < 1e-10, 'a reexecucao NAO reproduz o parquet congelado -- PARAR'
print(f'  modelos identicos: {int((mo6 == pu["modelo"].to_numpy()).sum())} de {len(pu)}')
assert (mo6 == pu['modelo'].to_numpy()).all(), 'atribuicao de modelo divergiu'
print('  => os coeficientes abaixo sao os DA execucao congelada, nao de um refit diferente.')

print('\n  (i) coeficiente medio e dp por feature, ao longo dos meses de decisao:')
lin = []
for f in RANKS:
    lin.append({'feature': f, 'onde': 'GENERALISTA', 'n_modelos': len(cg6),
                'media': cg6[f].mean(), 'dp': cg6[f].std(),
                '|media|/dp': abs(cg6[f].mean()) / cg6[f].std(),
                '% meses > 0': 100 * float((cg6[f] > 0).mean())})
    lin.append({'feature': f, 'onde': 'ESPECIALISTAS (todos)', 'n_modelos': len(ce6),
                'media': ce6[f].mean(), 'dp': ce6[f].std(),
                '|media|/dp': abs(ce6[f].mean()) / ce6[f].std(),
                '% meses > 0': 100 * float((ce6[f] > 0).mean())})
print(pd.DataFrame(lin).round(6).to_string(index=False))
print('  |media|/dp < 1 significa que o coeficiente muda de mes para mes mais do que o seu proprio')
print('  nivel medio -- exatamente o padrao que D23 cita para PROIBIR selecao por estabilidade.')

print('\n  (ii) IC individual de cada rank (Spearman mensal com alfa_fut), no TESTE:')
pu_t = pu[(pu['particao'] == 'TESTE') & pu['alfa_fut'].notna()]
lin = []
for f in RANKS + ['score']:
    ics = pu_t.groupby('mes').apply(
        lambda g: spearman(g[f], g['alfa_fut']) if len(g) >= 10 else np.nan,
        include_groups=False).dropna()
    mu, se, t = nw_t(ics)
    lin.append({'serie': f, 'n_meses': len(ics), 'IC medio': mu, 'dp': ics.std(),
                't (NW3)': t, '% meses > 0': 100 * float((ics > 0).mean())})
print(pd.DataFrame(lin).round(4).to_string(index=False))
print('  RESSALVA que precisa vir junto do numero: estes IC individuais sao medidos DIRETO no')
print('  periodo de TESTE, sem walk-forward -- nao existe versao "fora da amostra" deles. Sao')
print('  descritivos do teste, e usa-los para escolher feature seria escolher no proprio teste,')
print('  duplamente proibido (D23 proibe selecao de fator por QUALQUER criterio).')
print('  O contraste que eles produzem: mom_12m_rank sozinho tem IC +0,0542 (t=3,84) e vol_6m_rank')
print('  sozinho tem IC -0,0632 (t=-4,33), enquanto o SCORE combinado tem +0,0025 (t=0,19). A')
print('  combinacao por OLS re-treinada todo mes nao preserva o sinal univariado das features.')

print('\n  (iii) IC de um score ajustado SO com momento (4) e SO com volatilidade (2):')
MOM = ['mom_1m_rank', 'mom_3m_rank', 'mom_6m_rank', 'mom_12m_rank']
VOL = ['vol_3m_rank', 'vol_6m_rank']
sc_m, _, _, _ = walk_forward(MOM)
sc_v, _, _, _ = walk_forward(VOL)
pu2 = pu.copy()
pu2['score_mom'] = sc_m
pu2['score_vol'] = sc_v
pu2_t = pu2[(pu2['particao'] == 'TESTE') & pu2['alfa_fut'].notna()]
lin = []
for f, rot in [('score', 'score COMPLETO (6 features, o congelado)'),
               ('score_mom', 'score SO momento (4)'), ('score_vol', 'score SO volatilidade (2)')]:
    ics = pu2_t.groupby('mes').apply(
        lambda g: spearman(g[f], g['alfa_fut']) if len(g) >= 10 else np.nan,
        include_groups=False).dropna()
    mu, se, t = nw_t(ics)
    lin.append({'score': rot, 'n_meses': len(ics), 'IC medio': mu, 'dp': ics.std(), 't (NW3)': t,
                '% meses > 0': 100 * float((ics > 0).mean())})
print(pd.DataFrame(lin).round(4).to_string(index=False))
print('  PROIBIDO usar isto para escolher features (D23). O conjunto de 6 esta congelado desde B1 e')
print('  segue congelado; este numero entra no relatorio como diagnostico declarado, nada mais.')

# ---------------- DIAG-3
print('\nDIAG-3 -- ONDE O RETORNO SE PERDE (carteira bruta x indice interno, TESTE)')
bl = d_a[d_a['particao'] == 'TESTE']
rc, ri = bl['bruto'].to_numpy(), bl['ret_indice'].to_numpy()
mu_c, mu_i = rc.mean(), ri.mean()
lc, li = np.log1p(rc).mean(), np.log1p(ri).mean()
Dc, Di = mu_c - lc, mu_i - li      # arrasto de Jensen de cada serie
print(f'  media aritmetica mensal: carteira {100 * mu_c:.4f}%  indice {100 * mu_i:.4f}%  '
      f'-> selecao = {100 * (mu_c - mu_i):+.4f}%/mes')
print(f'  dp mensal: carteira {100 * rc.std(ddof=1):.4f}%  indice {100 * ri.std(ddof=1):.4f}%')
print(f'  arrasto de Jensen (media aritmetica - media log): carteira {100 * Dc:.4f}%/mes  '
      f'indice {100 * Di:.4f}%/mes')
sel_eff = 12 * (mu_c - mu_i)
conc_eff = -12 * (Dc - Di)
tot = 12 * (lc - li)
print(f'\n  IDENTIDADE EXATA (em log, aditiva): 12*(log-medio carteira - log-medio indice) = '
      f'12*(mu_c - mu_i) - 12*(D_c - D_i)')
print(f'    (a) EFEITO DE SELECAO   = {100 * sel_eff:+.4f} pp/ano (log)')
print(f'    (b) EFEITO DE CONCENTRACAO = {100 * conc_eff:+.4f} pp/ano (log)')
print(f'    soma = {100 * (sel_eff + conc_eff):+.4f}   verificado contra {100 * tot:+.4f}  '
      f'(residuo {100 * (sel_eff + conc_eff - tot):.2e})')
print(f'  em retorno simples: carteira {anual(rc):.4f}%aa - indice {anual(ri):.4f}%aa = '
      f'{anual(rc) - anual(ri):+.4f} pp/ano')
base = np.exp(li * 12) - 1
sel_simples = (np.exp(li * 12 + sel_eff) - 1) - base
conc_simples = (np.exp(li * 12 + sel_eff + conc_eff) - 1) - (np.exp(li * 12 + sel_eff) - 1)
print(f'    decomposicao multiplicativa a partir do indice ({100 * base:.4f}%aa):')
print(f'      + selecao      {100 * sel_simples:+.4f} pp/ano')
print(f'      + concentracao {100 * conc_simples:+.4f} pp/ano')
print(f'      = carteira     {100 * (base + sel_simples + conc_simples):.4f}%aa')
print(f'  APROXIMACAO DE CONTROLE (sigma^2/2): a diferenca de arrasto prevista por variancia e '
      f'{100 * 12 * (rc.var(ddof=1) - ri.var(ddof=1)) / 2:.4f} pp/ano, contra os '
      f'{100 * -conc_eff:.4f} medidos -- mesma ordem, e o medido e o exato.')
n_cart = trilha_a[trilha_a['particao'] == 'TESTE'].groupby('mes').size().median()
n_idx = bench_idx.loc[[m for m in meses_dec if part_por_mes[m] == 'TESTE'], 'n_ativos_indice'].median()
print(f'  contexto: {n_cart:.0f} nomes na carteira contra {n_idx:.0f} no indice.')

# ---------------- DIAG-4
print('\nDIAG-4 -- CONCENTRACAO TEMPORAL (3 melhores e 3 piores meses)')
lin = []
for rot, d, col in [('C2-A bruto', d_a, 'bruto'), ('C2-A liq50', d_a, None),
                    ('C2-B bruto', d_b, 'bruto'), ('C2-B liq50', d_b, None),
                    ('indice interno', d_a, 'ret_indice')]:
    bl = d[d['particao'] == 'TESTE']
    r = bl['bruto'] if col == 'bruto' else (bl[col] if col else liquido(bl, 50))
    r = np.asarray(r, dtype=float)
    acum = np.prod(1 + r) - 1
    ordem = np.argsort(r)
    piores, melhores = ordem[:3], ordem[-3:]
    sem_mel = np.prod(1 + np.delete(r, melhores)) - 1
    sem_pio = np.prod(1 + np.delete(r, piores)) - 1
    lin.append({'serie': rot, 'acum_%': 100 * acum,
                'sem 3 melhores_%': 100 * sem_mel, 'sem 3 piores_%': 100 * sem_pio,
                'meses melhores': ', '.join(bl['mes_ret'].iloc[sorted(melhores)]),
                'meses piores': ', '.join(bl['mes_ret'].iloc[sorted(piores)])})
dg4 = pd.DataFrame(lin)
print(dg4.round(3).to_string(index=False))
bl = d_a[d_a['particao'] == 'TESTE']
r = np.asarray(bl['bruto'])
soma3 = np.sort(r)[-3:].sum()
print(f'\n  em soma aritmetica simples: os 3 melhores meses somam {100 * soma3:.3f} pp contra '
      f'{100 * r.sum():.3f} pp de soma total dos 103 meses -> {100 * soma3 / r.sum():.1f}%')
print(f'  [F] marco: 3 meses = 88,5% do alfa. Aqui a concentracao e medida sobre RETORNO, nao alfa,')
print(f'  e o denominador tem sinal proprio -- o numero acima nao e comparavel 1 para 1 com o [F].')

# ==================================================================== TABELA FINAL
print('\n' + '=' * 122)
print('TABELA FINAL CONSOLIDADA -- TESTE (103 meses, 2018-01 a 2026-07 de decisao)')
print('=' * 122)
rf = d_a[d_a['particao'] == 'TESTE']['ret_cdi'].to_numpy()
idx_ref = d_a[d_a['particao'] == 'TESTE']['ret_indice'].to_numpy()


def bloco(nome, r, giro=None, r_liq=None):
    r = np.asarray(pd.Series(r).dropna(), dtype=float)
    rl = np.asarray(pd.Series(r_liq).dropna(), dtype=float) if r_liq is not None else r
    exc_rf = rl - rf[:len(rl)]
    exc_ix = rl - idx_ref[:len(rl)]
    mu, se, t = nw_t(exc_ix)
    return {'serie': nome, 'bruto_%aa': anual(r), 'liq50_%aa': anual(rl),
            'vol_%aa': 100 * rl.std(ddof=1) * np.sqrt(12),
            'Sharpe (CDI D13)': (exc_rf.mean() * 12) / (exc_rf.std(ddof=1) * np.sqrt(12)),
            'DDmax_%': dd_max(rl),
            'giro_ow_%mes': 100 * giro / 2 if giro is not None else np.nan,
            'exc vs indice %mes': 100 * mu, 't NW3': t}


bl_a = d_a[d_a['particao'] == 'TESTE']
bl_b = d_b[d_b['particao'] == 'TESTE']
bl_e = d_ew[d_ew['particao'] == 'TESTE']
bl_i = d_i[d_i['particao'] == 'TESTE']
med_rand = np.argsort(dist[50])[len(dist[50]) // 2]
r_rand = res_rand[med_rand]
r_rand_t = r_rand[r_rand['particao'] == 'TESTE']

linhas = [
    bloco('C2-A (leitura A)', bl_a['bruto'], bl_a['turnover'].mean(), liquido(bl_a, 50)),
    bloco('C2-B (leitura B)', bl_b['bruto'], bl_b['turnover'].mean(), liquido(bl_b, 50)),
    bloco('EW universo elegivel', bl_e['bruto'], bl_e['turnover'].mean(), liquido(bl_e, 50)),
    bloco('aleatoria MEDIANA (de 200)', r_rand_t['bruto'], r_rand_t['turnover'].mean(),
          r_rand_t['bruto'] - r_rand_t['turnover'] * 50 / 10000),
    bloco('selecao INVERTIDA', bl_i['bruto'], bl_i['turnover'].mean(), liquido(bl_i, 50)),
    bloco('indice interno (D12)', bl_a['ret_indice'], None, bl_a['ret_indice']),
    bloco('indice ex-max (D16)', bl_a['ret_indice_ex_max'], None, bl_a['ret_indice_ex_max']),
    bloco('CDI (D13)', bl_a['ret_cdi'], None, bl_a['ret_cdi']),
]
tab = pd.DataFrame(linhas)
print(tab.round(4).rename(columns={'Sharpe (CDI D13)': 'Sharpe (CDI D13) [NAO ORDENAVEL]'}).to_string(index=False))
print('[NAO ORDENAVEL] (plano item 6, rodada 3): com excesso negativo contra o CDI, mu/sigma cresce')
print('com sigma -- a selecao INVERTIDA rende -3,1569%aa liq50 com Sharpe -0,3125, melhor que o')
print('Sharpe -0,3330 de C2-A, que rende -1,6673%aa (1,49 pp/ano a mais). A razao inverte a')
print('ordenacao em vez de resumi-la (Israelsen 2005). PROIBIDO ordenar series por Sharpe.')
print('O parquet NAO foi reescrito -- o rotulo vive na impressao, a nota de rodape aqui.')
print('CONVENCAO DE GIRO (plano item 7): giro_ow_%mes e UMA perna; a coluna turnover dos parquets')
print('e DUAS pernas (soma |dw|); o custo de bps e cobrado sobre DUAS pernas.')
print('\nIBOV (subperiodo menor, ate 2025-12, DECLARADO SECUNDARIO):')
print(f'  {anual(ib_t["ret_ibov"]):.4f}%aa em {len(ib_t)} meses; no mesmo subperiodo C2-A liq50 = '
      f'{anual(liquido(sub, 50)):.4f}%aa e indice interno = {anual(sub["ret_indice"]):.4f}%aa')
print('\nLEITURA DA COLUNA t NW3, para nao ser mal lida: ela mede o excedente sobre o INDICE INTERNO.')
print('O EW do universo tem excedente de apenas -0,1070%/mes e ainda assim t = -4,38, porque EW e')
print('indice sao quase a MESMA carteira: o excedente tem variancia minuscula, e uma diferenca')
print('pequena e persistente vira t grande. t grande NAO significa diferenca economicamente grande.')
print('\nSharpe: (media do excedente sobre o CDI de D13) x 12 / (dp do excedente x raiz(12)).')
print('Todas as series de estrategia estao LIQUIDAS de 50bps na coluna liq50 e no Sharpe/DD/t.')

# ==================================================================== SANIDADE
print('\n' + '=' * 122)
print('SANIDADE')
print('=' * 122)

ew0 = anual(d_ew[d_ew['particao'] == 'TESTE']['bruto'])
ix0 = anual(d_ew[d_ew['particao'] == 'TESTE']['ret_indice'])
print(f'S1 EW a custo 0 {ew0:.4f}%aa vs indice interno {ix0:.4f}%aa -> diferenca {ew0 - ix0:+.4f} pp/ano')
assert abs(ew0 - ix0) <= 0.5, f'S1 FALHOU: {ew0 - ix0:+.4f} pp/ano (limite 0,5)'
print('   S1 PASSOU')

med_r0 = float(np.median(dist[0]))
print(f'S2 mediana das 200 aleatorias a custo 0 {med_r0:.4f}%aa vs EW do universo {ew0:.4f}%aa '
      f'-> diferenca {med_r0 - ew0:+.4f} pp/ano')
print('   S2 REPORTADA (o enunciado pede reportar a diferenca, nao um limiar). Mecanismo da')
print('   diferenca: as aleatorias sorteiam ~35 nomes DENTRO dos modelos que passaram no gate,')
print('   nao entre os ~295 elegiveis; carregam o arrasto de concentracao de DIAG-3 e o mesmo')
print('   giro alto da estrategia real, enquanto o EW carrega o universo inteiro com giro baixo.')

print('S3 nenhum comparador usa informacao de T+1 na selecao em T:')
# (i) todo selecionado e elegivel em T (propriedade que depende so de t e t-1)
eleg_par = set(zip(split.loc[split['elegivel'], 'mes'], split.loc[split['elegivel'], 'CODISI']))
for rot, df in [('C2-A', trilha_a), ('C2-B', trilha_b), ('INVERTIDA', sel_i)]:
    pares = set(zip(df['mes'], df['CODISI']))
    fora = pares - eleg_par
    print(f'   {rot}: {len(pares)} posicoes, {len(fora)} nao elegiveis em T')
    assert not fora, f'S3 FALHOU em {rot}'
pares_ew = set(zip(eleg['mes'], eleg['CODISI']))
assert not (pares_ew - eleg_par), 'S3 FALHOU no EW'
print(f'   EW: {len(pares_ew)} posicoes, 0 nao elegiveis em T')
falha_r = 0
for r in res_rand[:5]:
    pass
# (ii) reconstrucao PONTUAL: refaz a selecao de 12 meses do teste usando SO a fatia <= T
amostra_meses = [m for m in meses_dec if part_por_mes[m] == 'TESTE'][::9][:12]
div = 0
for m in amostra_meses:
    fatia = prev[prev['mes'] <= m].drop(columns=['alfa_fut'])
    g = fatia[fatia['mes'] == m].sort_values(['modelo', 'score'], ascending=[True, False], kind='mergesort')
    g['rk'] = g.groupby('modelo').cumcount()
    t3m = g[g['rk'] < 3]
    p3 = t3m.pivot_table(index='modelo', columns='rk', values='score', aggfunc='first')
    p3.columns = ['s1', 's2', 's3']
    K = np.where(p3['s3'] < p3['s1'] * 0.5, 2, 3)
    med = np.where(K == 2, (p3['s1'] + p3['s2']) / 2, (p3['s1'] + p3['s2'] + p3['s3']) / 3)
    gate = med > 0
    esc = set()
    for i, mod in enumerate(p3.index):
        if gate[i]:
            esc |= set(t3m[(t3m['modelo'] == mod) & (t3m['rk'] < K[i])]['CODISI'])
    real = set(trilha_a[trilha_a['mes'] == m]['CODISI'])
    div += len(esc ^ real)
print(f'   reconstrucao point-in-time de {len(amostra_meses)} meses do TESTE, com alfa_fut REMOVIDO '
      f'do dado: {div} divergencias de selecao')
assert div == 0, 'S3 FALHOU na reconstrucao point-in-time'
print('   as 200 aleatorias sorteiam do pool de candidatos de T (universo de PREVISAO, D25) e nunca')
print('   tocam alfa_fut nem ret_1m de T+1; a invertida usa -score, funcao apenas do score de T.')
print('   S3 PASSOU')

excesso_cdi = bl_a['ret_cdi'].to_numpy() - rf
print('S4 Sharpe calculado com o CDI de D13 (composto por dias uteis, a6_benchmark_e_rf):')
print(f'   prova direta: excedente do PROPRIO CDI sobre o rf usado -> maior |desvio| = '
      f'{np.abs(excesso_cdi).max():.3e} em {len(rf)} meses (serie identicamente nula, e por isso o')
print('   Sharpe do CDI na tabela sai NaN: 0/0, e nao um numero qualquer).')
assert np.abs(excesso_cdi).max() < 1e-15, 'S4 FALHOU: o rf usado nao e o CDI de D13'
print(f'   fonte do rf: coluna ret_cdi de a6_benchmark_e_rf.parquet, anualizada no periodo em '
      f'{anual(rf):.4f}%aa')
print('   S4 PASSOU')

# ==================================================================== gravacao
saida = []
for rot, d in [('C2-A', d_a), ('C2-B', d_b), ('EW_universo', d_ew), ('INVERTIDA', d_i)]:
    x = d[['mes', 'mes_ret', 'particao', 'bruto', 'turnover']].copy()
    x['versao'] = rot
    for bps in NIVEIS_BPS:
        x[f'liq_{bps}bps'] = liquido(d, bps)
    saida.append(x)
saida = pd.concat(saida, ignore_index=True)
saida.to_parquet(DIR_INT + r'\c3_curva_custo.parquet', index=False)
rnd = pd.DataFrame({'sorteio': np.arange(N_ALEATORIAS), 'anual_0bps': dist[0], 'anual_50bps': dist[50]})
rnd.to_parquet(DIR_INT + r'\d1_aleatorias.parquet', index=False)
tab.to_parquet(DIR_INT + r'\d1_tabela_final.parquet', index=False)
print('\n' + '=' * 122)
print(f'GRAVADO intermediario\\c3_curva_custo.parquet   {saida.shape}')
print(f'GRAVADO intermediario\\d1_aleatorias.parquet    {rnd.shape}')
print(f'GRAVADO intermediario\\d1_tabela_final.parquet  {tab.shape}')
print('C3 e D1 CONCLUIDAS -- medicao apenas, nenhum parametro do MODEL_SPEC tocado.')
print('=' * 122)


C:\Users\lucca\AppData\Local\Temp\claude\C--Users-lucca-quant2026\c01746d5-1664-4816-be0b-b637a4fbf8fc\scratchpad\c3d1.py:121: RuntimeWarning: invalid value encountered in scalar divide
  return mu, se, mu / se
C:\Users\lucca\AppData\Local\Temp\claude\C--Users-lucca-quant2026\c01746d5-1664-4816-be0b-b637a4fbf8fc\scratchpad\c3d1.py:718: RuntimeWarning: invalid value encountered in scalar divide
  'Sharpe (CDI D13)': (exc_rf.mean() * 12) / (exc_rf.std(ddof=1) * np.sqrt(12)),
C3 (curva de custo) + D1 (comparadores) -- CELULAS DE MEDICAO. Nenhum parametro do
MODEL_SPEC (congelado 2026-08-15 18:09:30) e alterado, criado ou reajustado aqui.
matriz de retornos (380, 1873); meses de decisao 296 (2001-12 a 2026-07)

rodando o motor a custo ZERO (o custo entra depois, por aplicar_custos/aritmetica identica):
  [C2-A] ativos 522  meses 296  congelamentos 23  descartados 0
  [C2-B] ativos 523  meses 296  congelamentos 23  descartados 0
  conferencia C2-A: maior |liquido(50bps) recalculado - C2| = 

### Correção pós-conselho (plano item 4) — índice interno declarado NÃO INVESTÍVEL; comparador de manchete → EW point-in-time 6,7162

Decomposição: **+0,5309 de membresia − 0,0598 de faltante = +0,4711 pp/ano** de distância até o gêmeo investível. `elegivel(t)` usa `n_sessoes(t) ≥ 4` — informação do próprio mês do retorno — enquanto a carteira decide em t−1. O sinal do gap **inverte** fora do teste (treino walk-forward −2,4848; amostra cheia −0,8470 pela construção faltante-a-0% e −0,9244 pela faltante-descartado). Célula executada em 2026-08-16 com o interpretador do projeto; output real abaixo.

In [ ]:
# ---- item 4 do plano do conselho (rodada2): indice interno declarado NAO INVESTIVEL ----
# Tres construcoes lado a lado + decomposicao do gap + inversao do sinal fora do teste.
# Autocontido: le apenas parquets de intermediario\. Convencoes identicas as da celula:
#   anual(r) = 100*((prod(1+r))**(12/len(r))-1); TESTE = retornos 2018-02..2026-08 (103m).
import numpy as np
import pandas as pd

_split = pd.read_parquet(r'..\intermediario\a7_split.parquet',
                         columns=['CODISI', 'mes', 'ret_1m', 'elegivel', 'particao'])
_d1 = pd.read_parquet(r'..\intermediario\d1_tabela_final.parquet')


def _anual(r):
    r = np.asarray(r, dtype=float)
    return 100.0 * ((np.prod(1.0 + r)) ** (12.0 / len(r)) - 1.0)


_meses = sorted(_split['mes'].unique())
_eleg = _split.loc[_split['elegivel']].groupby('mes')['CODISI'].apply(set).to_dict()
_rets = {m: g.set_index('CODISI')['ret_1m']
         for m, g in _split[['mes', 'CODISI', 'ret_1m']].groupby('mes')}

_lin = []
for _i, _m in enumerate(_meses[1:], start=1):
    _ant = _meses[_i - 1]
    if _ant not in _eleg or _m not in _eleg:
        continue
    _rm = _rets[_m]
    _r1 = _rm.reindex(sorted(_eleg[_m]))          # membresia do proprio mes (como esta)
    _r0 = _rm.reindex(sorted(_eleg[_ant]))        # membresia point-in-time (elegivel em t-1)
    _lin.append({'mes_ret': _m, 'asis': _r1.dropna().mean(),
                 'pit_drop': _r0.dropna().mean(), 'pit_zero': _r0.fillna(0.0).mean()})
_ser = pd.DataFrame(_lin).set_index('mes_ret')

print('\nINDICE INTERNO -- DECLARACAO DE NAO INVESTIBILIDADE (item 4 do plano do conselho)')
print('  elegivel(t) usa n_sessoes(t)>=4 -- informacao do proprio mes do retorno; a carteira')
print('  decide em t-1. As tres construcoes no TESTE (103 meses de retorno, 2018-02..2026-08):')
_bl = _ser.loc['2018-02':'2026-08']
_a1, _a2, _a3 = _anual(_bl['asis']), _anual(_bl['pit_drop']), _anual(_bl['pit_zero'])
print(f'  (1) indice como esta (elegivel em t)          : {_a1:7.4f} %aa  <- NAO INVESTIVEL')
print(f'  (2) elegivel em t-1, faltante descartado      : {_a2:7.4f} %aa')
print(f'  (3) elegivel em t-1, faltante a 0% (=EW PIT)  : {_a3:7.4f} %aa  <- COMPARADOR DE MANCHETE')
print(f'  decomposicao: membresia (1)-(2) = {_a1 - _a2:+.4f}  |  faltante (2)-(3) = {_a2 - _a3:+.4f}'
      f'  |  distancia ate o gemeo investivel (1)-(3) = {_a1 - _a3:+.4f} pp/ano')

_ew_d1 = float(_d1.loc[_d1['serie'].str.contains('EW universo'), 'bruto_%aa'].iloc[0])
assert abs(_a3 - _ew_d1) < 1e-9, \
    f'reconstrucao PIT ({_a3:.6f}) nao bate a linha EW universo elegivel de d1 ({_ew_d1:.6f})'
print(f'  igualdade exata: (3) {_a3:.6f} == linha "EW universo elegivel" de d1_tabela_final '
      f'{_ew_d1:.6f} (dif {_a3 - _ew_d1:+.1e})')

_tr = _ser.loc['2002-01':'2018-01']
print(f'\n  O SINAL DO GAP NAO E ESTAVEL: no walk-forward de TREINO (193 meses de retorno,')
print(f'  2002-01..2018-01) o gap (1)-(3) INVERTE para {_anual(_tr["asis"]) - _anual(_tr["pit_zero"]):+.4f} pp/ano;'
      f' na amostra cheia ({len(_ser)} meses) e {_anual(_ser["asis"]) - _anual(_ser["pit_zero"]):+.4f}'
      f' ((1)-(3)) a {_anual(_ser["asis"]) - _anual(_ser["pit_drop"]):+.4f} ((1)-(2)).')
print('  E contaminacao DESTA janela, nao premio de sobrevivencia sistematico.')

_ent_n, _tot_n, _r_ent, _r_dem = 0, 0, [], []
for _m in _bl.index:
    _ant = _meses[_meses.index(_m) - 1]
    _rm = _rets[_m]
    _e1, _e0 = _eleg[_m], _eleg[_ant]
    _ent_n += len(_e1 - _e0)
    _tot_n += len(_e1)
    _r_ent.extend(_rm.reindex(sorted(_e1 - _e0)).dropna().tolist())
    _r_dem.extend(_rm.reindex(sorted(_e1 & _e0)).dropna().tolist())
print(f'  mecanismo: {100 * _ent_n / _tot_n:.2f}% das observacoes do indice ({_ent_n / len(_bl):.1f} '
      f'nomes/mes) nao eram elegiveis em t-1 e renderam {100 * np.mean(_r_ent):+.4f}%/mes contra '
      f'{100 * np.mean(_r_dem):+.4f}%/mes dos demais.')
print('  CONSEQUENCIA: o comparador de manchete das tabelas finais passa a ser o EW point-in-time')
print('  (6,7162%aa); o indice interno vira linha secundaria rotulada "nao investivel".')



INDICE INTERNO -- DECLARACAO DE NAO INVESTIBILIDADE (item 4 do plano do conselho)
  elegivel(t) usa n_sessoes(t)>=4 -- informacao do proprio mes do retorno; a carteira
  decide em t-1. As tres construcoes no TESTE (103 meses de retorno, 2018-02..2026-08):
  (1) indice como esta (elegivel em t)          :  7.1873 %aa  <- NAO INVESTIVEL
  (2) elegivel em t-1, faltante descartado      :  6.6564 %aa
  (3) elegivel em t-1, faltante a 0% (=EW PIT)  :  6.7162 %aa  <- COMPARADOR DE MANCHETE
  decomposicao: membresia (1)-(2) = +0.5309  |  faltante (2)-(3) = -0.0598  |  distancia ate o gemeo investivel (1)-(3) = +0.4711 pp/ano
  igualdade exata: (3) 6.716157 == linha "EW universo elegivel" de d1_tabela_final 6.716157 (dif +0.0e+00)

  O SINAL DO GAP NAO E ESTAVEL: no walk-forward de TREINO (193 meses de retorno,
  2002-01..2018-01) o gap (1)-(3) INVERTE para -2.4848 pp/ano; na amostra cheia (344 meses) e -0.8470 ((1)-(3)) a -0.9244 ((1)-(2)).
  E contaminacao DESTA janela, nao premio de sobrevi

## D2 (decomposicao do excedente) + D3 (bootstrap de bloco) + D4 (metricas finais)

Tres celulas de MEDICAO do modelo congelado em 2026-08-15 18:09:30. Nenhum parametro e tocado.

**D2** mede o excedente sobre o indice interno e sobre o ex-max (D16), bruto e liquido de 50 bps,
com t simples e Newey-West em 1, 3 e 6 lags -- os tres impressos, porque a escolha do lag muda o t
e isso tem que ficar visivel. Inclui Jensen, betas de Dimson com 2 defasagens, e uma decomposicao
de Brinson de dois termos cuja identidade e exata (residuo 0,0).

**D3** roda UM unico bootstrap de bloco, com o bloco definido pela autocorrelacao MEDIDA e nao
escolhida, semente `20260815` declarada, mais um CONTROLE sobre o EW do universo com bloco, semente
e n identicos -- o controle existe para testar se a metrica discrimina, nao e uma variante entre
as quais se escolhe. A legenda do que o bootstrap NAO mede e obrigatoria e esta no output literal.

**D4** consolida retorno, risco, drawdown, VaR/CVaR (relatorio descritivo, proibidos de entrar em
qualquer decisao), HHI, capacity por VOLTOT e a versao ex-maior-contribuinte com a correcao de
mesma FRACAO ao lado da versao literal de D16.

`scipy` nao existe nesta `.venv`: Newey-West, HAC de regressao, autocorrelacao e bootstrap estao
implementados na propria celula.

Todo resultado e NAO-REFUTACAO: [F] o poder deste desenho exigiria ~331 meses.


In [1]:
import sys

import numpy as np
import pandas as pd

if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8')

pd.set_option('display.width', 275)
pd.set_option('display.max_columns', 90)
pd.set_option('display.max_rows', 400)

RAIZ = r'C:\Users\lucca\quant2026'
DIR_INT = RAIZ + r'\intermediario'

SEMENTE_BOOTSTRAP = 20260815        # semente FIXA e DECLARADA (D3)
N_BOOT = 10000
CUSTO = 50.0

print('=' * 124)
print('D2 (decomposicao do excedente) + D3 (bootstrap de bloco) + D4 (metricas finais)')
print('CELULAS DE MEDICAO do modelo congelado em 2026-08-15 18:09:30. NADA muda aqui.')
print('scipy NAO existe nesta .venv: tudo o que precisa de estatistica esta implementado abaixo.')
print('=' * 124)


# ============================================================ estatistica na mao
def nw_var(e, lags):
    """Variancia de longo prazo de Newey-West-Bartlett para a MEDIA de uma serie."""
    e = np.asarray(e, dtype=float)
    n = len(e)
    d = e - e.mean()
    v = float(d @ d) / n
    for l in range(1, lags + 1):
        v += 2.0 * (1.0 - l / (lags + 1.0)) * (float(d[l:] @ d[:-l]) / n)
    return v


def t_media(x, lags=None):
    x = np.asarray(pd.Series(x).dropna(), dtype=float)
    n = len(x)
    mu = x.mean()
    if lags is None:
        se = x.std(ddof=1) / np.sqrt(n)
    else:
        se = np.sqrt(nw_var(x, lags) / n)
    return mu, se, mu / se


def ols_nw(y, X, lags=3):
    """OLS com erro-padrao classico e HAC de Newey-West-Bartlett. X SEM intercepto."""
    y = np.asarray(y, dtype=float)
    X = np.asarray(X, dtype=float)
    if X.ndim == 1:
        X = X[:, None]
    Xd = np.column_stack([np.ones(len(y)), X])
    n, k = Xd.shape
    beta = np.linalg.lstsq(Xd, y, rcond=None)[0]
    e = y - Xd @ beta
    XtX_inv = np.linalg.inv(Xd.T @ Xd)
    s2 = float(e @ e) / (n - k)
    se_ols = np.sqrt(np.diag(s2 * XtX_inv))
    S = np.zeros((k, k))
    u = Xd * e[:, None]
    S += u.T @ u
    for l in range(1, lags + 1):
        w = 1.0 - l / (lags + 1.0)
        G = u[l:].T @ u[:-l]
        S += w * (G + G.T)
    V = XtX_inv @ S @ XtX_inv
    se_nw = np.sqrt(np.diag(V))
    ybar = y.mean()
    r2 = 1.0 - float(e @ e) / float(((y - ybar) ** 2).sum())
    return beta, se_ols, se_nw, r2


def acf(x, lag):
    x = np.asarray(x, dtype=float)
    d = x - x.mean()
    return float(d[lag:] @ d[:-lag]) / float(d @ d)


def anual(r):
    r = np.asarray(pd.Series(r).dropna(), dtype=float)
    return 100.0 * ((np.prod(1.0 + r)) ** (12.0 / len(r)) - 1.0)


# ============================================================ insumos
c3 = pd.read_parquet(DIR_INT + r'\c3_curva_custo.parquet')
bench = pd.read_parquet(DIR_INT + r'\a6_benchmark_e_rf.parquet')
split = pd.read_parquet(DIR_INT + r'\a7_split.parquet')
trilha_a = pd.read_parquet(DIR_INT + r'\c1_carteira.parquet')
trilha_b = pd.read_parquet(DIR_INT + r'\c2_trilha_leituraB.parquet')
split['CODISI'] = split['CODISI'].astype(str)
trilha_a['CODISI'] = trilha_a['CODISI'].astype(str)
trilha_b['CODISI'] = trilha_b['CODISI'].astype(str)

bi = bench.set_index('mes')
series = {}
for v in ['C2-A', 'C2-B', 'EW_universo', 'INVERTIDA']:
    d = c3[c3['versao'] == v].sort_values('mes').reset_index(drop=True)
    for c in ['ret_indice', 'ret_indice_ex_max', 'ret_cdi']:
        d[c] = d['mes_ret'].map(bi[c])
    d['liquido'] = d['liq_50bps']
    series[v] = d

TESTE = {k: v[v['particao'] == 'TESTE'].reset_index(drop=True) for k, v in series.items()}
n_teste = len(TESTE['C2-A'])
meses_ret = list(TESTE['C2-A']['mes_ret'])
meses_dec_teste = list(TESTE['C2-A']['mes'])
print(f'\nTESTE: {n_teste} meses de decisao ({meses_dec_teste[0]} a {meses_dec_teste[-1]}), '
      f'realizando em {meses_ret[0]} a {meses_ret[-1]}')

# ============================================================================ D2
print('\n' + '=' * 124)
print('CELULA D2 -- DECOMPOSICAO DO EXCEDENTE')
print('=' * 124)

EXC = {}
for v in ['C2-A', 'C2-B']:
    d = TESTE[v]
    for tipo, col in [('bruto', 'bruto'), ('liquido50', 'liquido')]:
        EXC[(v, tipo, 'indice')] = (d[col] - d['ret_indice']).to_numpy()
        EXC[(v, tipo, 'ex-max')] = (d[col] - d['ret_indice_ex_max']).to_numpy()

print('\n(a) MEDIA, DP E t DO EXCEDENTE MENSAL -- t simples e Newey-West com 1, 3 e 6 lags')
lin = []
for (v, tipo, ref), x in EXC.items():
    mu, se0, t0 = t_media(x)
    reg = {'versao': v, 'tipo': tipo, 'vs': ref, 'n': len(x),
           'media_%mes': 100 * mu, 'dp_%mes': 100 * x.std(ddof=1),
           'ep simples': 100 * se0, 't simples': t0}
    for L in (1, 3, 6):
        _, se, t = t_media(x, L)
        reg[f't NW({L})'] = t
    lin.append(reg)
tab_a = pd.DataFrame(lin)
print(tab_a.round(4).to_string(index=False))
print('\nA ESCOLHA DO LAG MUDA O t, e por isso os tres estao impressos. Nenhum deles foi escolhido')
print('depois de olhar: o enunciado fixou 1, 3 e 6 antes de a celula rodar. Nenhum e "o" t.')

print('\n(b) CONCENTRACAO DO EXCEDENTE ACUMULADO')
print('AVISO DE INTERPRETACAO, que precisa vir antes da tabela: o excedente e uma DIFERENCA de')
print('retornos e sua soma no teste e NEGATIVA nas quatro series contra o indice. "Fracao do')
print('acumulado" dividida por um total negativo nao e interpretavel como participacao. Por isso a')
print('tabela traz a contribuicao em PONTOS PERCENTUAIS e a fracao sobre a SOMA DOS EXCEDENTES')
print('POSITIVOS, que tem denominador bem definido.')
lin = []
for (v, tipo, ref), x in EXC.items():
    o = np.sort(x)
    pos = x[x > 0].sum()
    neg = x[x < 0].sum()
    reg = {'versao': v, 'tipo': tipo, 'vs': ref, 'soma_pp': 100 * x.sum(),
           'soma_positivos_pp': 100 * pos, 'soma_negativos_pp': 100 * neg,
           'melhor_pp': 100 * o[-1], '3 melhores_pp': 100 * o[-3:].sum(),
           '5 melhores_pp': 100 * o[-5:].sum(),
           'pior_pp': 100 * o[0], '3 piores_pp': 100 * o[:3].sum(), '5 piores_pp': 100 * o[:5].sum(),
           '3 mel / positivos %': 100 * o[-3:].sum() / pos,
           '3 pio / negativos %': 100 * o[:3].sum() / neg}
    lin.append(reg)
print(pd.DataFrame(lin).round(3).to_string(index=False))
x = EXC[('C2-A', 'bruto', 'indice')]
o = np.sort(x)
print(f'\n[F] marco: 3 meses = 88,5% DO ALFA. Aqui, em C2-A bruto contra o indice, os 3 melhores')
print(f'meses somam {100 * o[-3:].sum():+.3f} pp e os 3 piores {100 * o[:3].sum():+.3f} pp, sobre uma soma')
print(f'total de {100 * x.sum():+.3f} pp. O numero de marco nao tem equivalente direto aqui porque la o')
print('alfa acumulado era POSITIVO e a fracao fazia sentido; aqui o excedente acumulado e negativo.')

print('\n(c) EXCEDENTE MEDIO REMOVENDO MESES EXTREMOS')
lin = []
for (v, tipo, ref), x in EXC.items():
    o = np.argsort(x)
    sem_mel = np.delete(x, o[-3:])
    sem_pio = np.delete(x, o[:3])
    sem_ext = np.delete(x, np.argsort(np.abs(x))[-6:])
    lin.append({'versao': v, 'tipo': tipo, 'vs': ref,
                'media_%mes': 100 * x.mean(),
                'sem 3 melhores': 100 * sem_mel.mean(),
                'sem 3 piores': 100 * sem_pio.mean(),
                'sem 6 maiores |.|': 100 * sem_ext.mean(),
                't NW3 sem 6 maiores': t_media(sem_ext, 3)[2]})
print(pd.DataFrame(lin).round(4).to_string(index=False))

print('\n(d) % DE MESES EM QUE A CARTEIRA SUPERA CADA COMPARADOR (TESTE, 103 meses)')
lin = []
for v in ['C2-A', 'C2-B', 'EW_universo']:
    d = TESTE[v]
    for tipo, col in [('bruto', 'bruto'), ('liquido50', 'liquido')]:
        lin.append({'versao': v, 'tipo': tipo,
                    '> indice %': 100 * float((d[col] > d['ret_indice']).mean()),
                    '> ex-max %': 100 * float((d[col] > d['ret_indice_ex_max']).mean()),
                    '> CDI %': 100 * float((d[col] > d['ret_cdi']).mean())})
print(pd.DataFrame(lin).round(2).to_string(index=False))

print('\n(e) JENSEN e BETAS DE DIMSON (2 lags)')
print('regressao: (r_carteira - CDI) = alfa + beta*(r_indice - CDI) + erro; Dimson soma os betas')
print('do benchmark contemporaneo e de 2 defasagens. Erro-padrao classico E Newey-West(3) nos dois.')
lin = []
for v in ['C2-A', 'C2-B', 'EW_universo']:
    d = TESTE[v]
    rf = d['ret_cdi'].to_numpy()
    bm = d['ret_indice'].to_numpy() - rf
    for tipo, col in [('bruto', 'bruto'), ('liquido50', 'liquido')]:
        y = d[col].to_numpy() - rf
        b, se_o, se_n, r2 = ols_nw(y, bm, lags=3)
        lin.append({'versao': v, 'tipo': tipo, 'modelo': 'Jensen',
                    'alfa_%mes': 100 * b[0], 't alfa (OLS)': b[0] / se_o[0],
                    't alfa (NW3)': b[0] / se_n[0], 'beta': b[1], 'soma betas': b[1], 'R2': r2})
        X = np.column_stack([bm[2:], bm[1:-1], bm[:-2]])
        yd = y[2:]
        bd, se_od, se_nd, r2d = ols_nw(yd, X, lags=3)
        lin.append({'versao': v, 'tipo': tipo, 'modelo': 'Dimson (2 lags)',
                    'alfa_%mes': 100 * bd[0], 't alfa (OLS)': bd[0] / se_od[0],
                    't alfa (NW3)': bd[0] / se_nd[0], 'beta': bd[1],
                    'soma betas': bd[1] + bd[2] + bd[3], 'R2': r2d})
print(pd.DataFrame(lin).round(4).to_string(index=False))
print('\n[F] marco: Jensen alfa +1,09%/mes com t~2,5, caindo para ~0,83% e t~1,8 com Dimson 2 lags.')
print('Aqui os alfas sao NEGATIVOS nas duas leituras, liquidos e brutos. O padrao de marco (o alfa')
print('encolhe ao acrescentar as defasagens) pode ou nao se repetir; a tabela mostra o que houve.')

print('\n(f) EXCEDENTE POR ANO (excedente medio mensal, em %/mes)')
lin = []
for v in ['C2-A', 'C2-B']:
    d = TESTE[v].copy()
    d['ano'] = d['mes'].str[:4]
    for ano, g in d.groupby('ano'):
        lin.append({'ano': ano, 'meses': len(g), 'versao': v,
                    'bruto - indice': 100 * (g['bruto'] - g['ret_indice']).mean(),
                    'liq50 - indice': 100 * (g['liquido'] - g['ret_indice']).mean(),
                    'liq50 - ex-max': 100 * (g['liquido'] - g['ret_indice_ex_max']).mean(),
                    'liq50 - CDI': 100 * (g['liquido'] - g['ret_cdi']).mean()})
tf = pd.DataFrame(lin).pivot_table(index='ano', columns='versao',
                                   values=['bruto - indice', 'liq50 - indice', 'liq50 - CDI'])
print(tf.round(4).to_string())

print('\n(g) EXCEDENTE POR ORIGEM DOS ATIVOS (especialista vs generalista)')
comp = trilha_a[trilha_a['particao'] == 'TESTE'].copy()
comp['origem'] = np.where(comp['modelo'] == 'GENERALISTA', 'GEN', 'ESP')
por_mes = comp.groupby(['mes', 'origem']).size().unstack(fill_value=0)
por_mes['frac_esp'] = por_mes['ESP'] / (por_mes['ESP'] + por_mes['GEN'])
d = TESTE['C2-A'].set_index('mes')
por_mes['exc_bruto'] = (d['bruto'] - d['ret_indice']).reindex(por_mes.index)
por_mes['exc_liq'] = (d['liquido'] - d['ret_indice']).reindex(por_mes.index)
n_gen_maioria = int((por_mes['frac_esp'] < 0.5).sum())
print(f'  meses em que a MAIORIA dos ativos veio do generalista: {n_gen_maioria} de {len(por_mes)}')
print(f'  fracao de especialistas: min {por_mes["frac_esp"].min():.4f}  mediana '
      f'{por_mes["frac_esp"].median():.4f}  max {por_mes["frac_esp"].max():.4f}')
print('  O CORTE PEDIDO E DEGENERADO nesta amostra: o generalista contribui com 2 a 3 nomes de ~35,')
print('  e nunca chega a maioria. Reporto isso em vez de forcar um grupo vazio, e acrescento o corte')
print('  que a mesma pergunta admite com dado nao-degenerado -- tercis da fracao de especialistas:')
por_mes['tercil'] = pd.qcut(por_mes['frac_esp'], 3, labels=['baixa', 'media', 'alta'])
print(por_mes.groupby('tercil', observed=True).agg(
    meses=('frac_esp', 'size'), frac_esp_media=('frac_esp', 'mean'),
    exc_bruto_pct_mes=('exc_bruto', lambda s: 100 * s.mean()),
    exc_liq_pct_mes=('exc_liq', lambda s: 100 * s.mean())).round(4).to_string())
rho = np.corrcoef(por_mes['frac_esp'], por_mes['exc_bruto'])[0, 1]
print(f'  correlacao de Pearson entre fracao de especialistas e excedente bruto: {rho:+.4f} '
      f'(t = {rho * np.sqrt((len(por_mes) - 2) / (1 - rho ** 2)):+.4f})')

print('\n(h) DECOMPOSICAO DE BRINSON SIMPLIFICADA (exata, dois termos)')
print('  excedente = SOMA_s (w_p,s - w_b,s)*r_b,s   [ALOCACAO: quais SUBSETORES]')
print('            + SOMA_s  w_p,s*(r_p,s - r_b,s)  [SELECAO: quais ACOES dentro do subsetor]')
print('  A identidade e exata: os dois termos somam SOMA_s w_p,s r_p,s - SOMA_s w_b,s r_b,s.')
print('  Subsetor presente na carteira e ausente do benchmark: r_b,s cancela algebricamente nos')
print('  dois termos, e usa-se o retorno do benchmark inteiro naquele mes (escolha declarada, sem')
print('  efeito sobre a identidade).')
sub_de = split.drop_duplicates('CODISI').set_index('CODISI')['subsetor_chave'].to_dict()
ret_de = {(m, c): r for m, c, r in zip(split['mes'], split['CODISI'], split['ret_1m'])}
eleg_por_mes = {m: g for m, g in split[split['elegivel'] & split['ret_valido']].groupby('mes')}
lin_br = []
for v, trilha in [('C2-A', trilha_a), ('C2-B', trilha_b)]:
    tr = trilha[trilha['particao'] == 'TESTE']
    for mes, g in tr.groupby('mes'):
        m1 = str(pd.Period(mes, freq='M') + 1)
        be = eleg_por_mes.get(m1)
        if be is None:
            continue
        rb_total = float(be['ret_1m'].mean())
        wb = be.groupby('subsetor_chave').size() / len(be)
        rb = be.groupby('subsetor_chave')['ret_1m'].mean()
        gg = g.copy()
        gg['sub'] = gg['CODISI'].map(sub_de)
        gg['r'] = [ret_de.get((m1, c), np.nan) for c in gg['CODISI']]
        # posicao congelada contribui ZERO: e a marcacao de posicao parada do motor
        # (_trilha, convencao 6), nao imputacao de dado -- a serie de retorno segue NaN.
        gg['r'] = gg['r'].fillna(0.0)
        # a fatia em CDI da leitura B entra como um "subsetor" proprio, senao os pesos da
        # carteira nao somam 1 e a decomposicao nao reconstroi o retorno do motor.
        w_cdi = 1.0 - float(g['peso'].sum())
        if w_cdi > 1e-12:
            gg = pd.concat([gg, pd.DataFrame([{'sub': '__CDI__', 'peso': w_cdi,
                                               'r': float(bi.loc[m1, 'ret_cdi'])}])],
                           ignore_index=True)
        wp = gg.groupby('sub')['peso'].sum()
        rp = gg.groupby('sub').apply(lambda h: float((h['peso'] * h['r']).sum() / h['peso'].sum()),
                                     include_groups=False)
        subs = sorted(set(wp.index) | set(wb.index))
        wpv = np.array([wp.get(s, 0.0) for s in subs])
        wbv = np.array([wb.get(s, 0.0) for s in subs])
        rbv = np.array([rb.get(s, rb_total) for s in subs])
        rpv = np.array([rp.get(s, 0.0) for s in subs])
        aloc = float(((wpv - wbv) * rbv).sum())
        selec = float((wpv * (rpv - rbv)).sum())
        rp_tot = float((wpv * rpv).sum())
        rb_tot = float((wbv * rbv).sum())
        lin_br.append({'versao': v, 'mes': mes, 'mes_ret': m1, 'alocacao': aloc, 'selecao': selec,
                       'rp': rp_tot, 'rb': rb_tot, 'exc': rp_tot - rb_tot})
br = pd.DataFrame(lin_br)
br['residuo'] = br['exc'] - (br['alocacao'] + br['selecao'])
print('\n  resultado agregado no TESTE (media mensal, em %/mes):')
print(br.groupby('versao').agg(
    meses=('mes', 'size'), alocacao=('alocacao', lambda s: 100 * s.mean()),
    selecao=('selecao', lambda s: 100 * s.mean()), excedente=('exc', lambda s: 100 * s.mean()),
    residuo_max=('residuo', lambda s: float(np.abs(s).max()))).round(6).to_string())
for v in ['C2-A', 'C2-B']:
    b = br[br['versao'] == v]
    mu_a, _, t_a = t_media(b['alocacao'], 3)
    mu_s, _, t_s = t_media(b['selecao'], 3)
    print(f'  {v}: alocacao {100 * mu_a:+.4f}%/mes (t NW3 {t_a:+.3f}) | '
          f'selecao {100 * mu_s:+.4f}%/mes (t NW3 {t_s:+.3f})')
print('\n  NOTA: o excedente reconstruido aqui (rp - rb) usa o benchmark recalculado sobre os')
print('  elegiveis do mes de realizacao; ele reproduz o `ret_indice` de A6 e a carteira do motor')
print('  quando nao ha congelamento, e por isso a media nao coincide digito a digito com (a).')
for v in ['C2-A', 'C2-B']:
    b = br[br['versao'] == v]
    d = TESTE[v]
    dif = b['exc'].to_numpy() - (d['bruto'] - d['ret_indice']).to_numpy()
    print(f'  {v}: maior |excedente Brinson - excedente do motor| = {np.abs(dif).max():.3e} '
          f'(media {np.abs(dif).mean():.3e})')

# ============================================================================ D3
print('\n' + '=' * 124)
print('CELULA D3 -- BOOTSTRAP DE BLOCO, UM SO')
print('=' * 124)

alvo = TESTE['C2-B']['liquido'].to_numpy()
n = len(alvo)
banda = 2.0 / np.sqrt(n)
print(f'\n1. AUTOCORRELACAO da serie de retorno mensal de C2-B LIQUIDO, lags 1 a 12 (n={n}):')
acs = [(l, acf(alvo, l)) for l in range(1, 13)]
for l, a in acs:
    marca = 'DENTRO' if abs(a) < banda else 'FORA'
    print(f'   lag {l:2d}: {a:+.4f}   |acf| {"<" if abs(a) < banda else ">="} '
          f'2/sqrt(n)={banda:.4f}  -> {marca} da banda')

print(f'\n2. BLOCO, definido pela autocorrelacao MEDIDA (banda +/-{banda:.4f}):')
dentro = [l for l, a in acs if abs(a) < banda]
primeiro_dentro = dentro[0] if dentro else None
todos_dentro_a_partir = None
for l in range(1, 13):
    if all(abs(a) < banda for ll, a in acs if ll >= l):
        todos_dentro_a_partir = l
        break
print(f'   leitura (i) menor lag COM |acf| dentro da banda: {primeiro_dentro}')
print(f'   leitura (ii) menor lag A PARTIR DO QUAL todos os lags ate 12 estao dentro: '
      f'{todos_dentro_a_partir}')
if primeiro_dentro == todos_dentro_a_partir:
    BLOCO = int(primeiro_dentro)
    print(f'   as duas leituras coincidem -> BLOCO = {BLOCO}')
else:
    BLOCO = int(todos_dentro_a_partir)
    print(f'   AS DUAS LEITURAS DIVERGEM. "a partir do qual" e literalmente a leitura (ii), e ela e')
    print(f'   a CONSERVADORA (bloco maior preserva mais dependencia). Adotada: BLOCO = {BLOCO}.')
    print('   A escolha esta declarada aqui, antes de qualquer numero do bootstrap aparecer.')
if BLOCO == 1:
    print('   BLOCO = 1: a autocorrelacao ja e insignificante no lag 1, entao a reamostragem e i.i.d.')

exc_alvo = (TESTE['C2-B']['liquido'] - TESTE['C2-B']['ret_indice']).to_numpy()
exc_ew = (TESTE['EW_universo']['liquido'] - TESTE['EW_universo']['ret_indice']).to_numpy()


def bootstrap_bloco(x, bloco, n_rep, semente):
    rng = np.random.default_rng(semente)
    m = len(x)
    n_blocos = int(np.ceil(m / bloco))
    inicios = rng.integers(0, m - bloco + 1, size=(n_rep, n_blocos))
    idx = (inicios[:, :, None] + np.arange(bloco)[None, None, :]).reshape(n_rep, -1)[:, :m]
    return x[idx].mean(axis=1)


print(f'\n3. BOOTSTRAP: {N_BOOT} reamostragens de bloco movel, bloco={BLOCO}, '
      f'SEMENTE = {SEMENTE_BOOTSTRAP} (fixa e declarada).')
print('   serie reamostrada: excedente mensal de C2-B LIQUIDO sobre o indice interno, no TESTE.')
print('   UM unico bootstrap. Nenhuma variante foi rodada, nenhuma foi comparada, nenhuma descartada.')
boot = bootstrap_bloco(exc_alvo, BLOCO, N_BOOT, SEMENTE_BOOTSTRAP)

print('\n4. DISTRIBUICAO DA MEDIA DO EXCEDENTE (%/mes):')
qs = [1, 5, 25, 50, 75, 95, 99]
print(f'   observado na amostra: {100 * exc_alvo.mean():+.4f}%/mes')
for q in qs:
    print(f'   p{q:<2d}: {100 * np.percentile(boot, q):+.4f}')
frac_pos = float((boot > 0).mean())
print(f'   fracao de reamostragens com media POSITIVA: {frac_pos:.4f}')

print('\n5. LEGENDA OBRIGATORIA (texto literal):')
LEGENDA = ('Este bootstrap reamostra a PROPRIA amostra realizada. Ele mede a dispersao do '
           'caminho DADO que a media e a observada; NAO testa se a media e real, nao produz '
           'p-valor de existencia de alfa, e nao constitui evidencia de robustez preditiva. '
           'Em marco havia dois bootstraps no mesmo notebook (0,8904 e 0,9435) e publicou-se o '
           'maior [F]. Aqui ha um, definido antes de rodar.')
print('   "' + LEGENDA + '"')

print('\n6. CONTROLE -- O MESMO bootstrap (mesmo bloco, mesma semente, mesmo n) sobre o EW:')
boot_ew = bootstrap_bloco(exc_ew, BLOCO, N_BOOT, SEMENTE_BOOTSTRAP)
print(f'   EW: excedente observado {100 * exc_ew.mean():+.4f}%/mes; '
      f'fracao de reamostragens positiva {float((boot_ew > 0).mean()):.4f}')
for q in qs:
    print(f'   p{q:<2d}: {100 * np.percentile(boot_ew, q):+.4f}')
frac_ew = float((boot_ew > 0).mean())
print(f'\n   COMPARACAO: C2-B {frac_pos:.4f} contra EW {frac_ew:.4f}.')
print('   O MODO DE FALHA QUE O CONTROLE FOI DESENHADO PARA PEGAR era "o EW tambem da fracao alta,')
print(f'   logo a metrica nao distingue as duas series". ELE NAO OCORREU: as duas fracoes sao BAIXAS')
print(f'   ({frac_pos:.4f} e {frac_ew:.4f}), porque os dois excedentes observados sao negativos')
print(f'   ({100 * exc_alvo.mean():+.4f} e {100 * exc_ew.mean():+.4f} %/mes). O controle nao invalida')
print('   a leitura -- mas tambem nao a valida, porque a legenda do item 5 continua valendo inteira.')
print('   O QUE O CONTROLE DE FATO MOSTRA e outra coisa, e vale registrar: a distribuicao do EW e')
larg_b = np.percentile(boot, 99) - np.percentile(boot, 1)
larg_e = np.percentile(boot_ew, 99) - np.percentile(boot_ew, 1)
print(f'   MUITO mais estreita -- amplitude p1-p99 de {100 * larg_e:.4f} pp/mes contra '
      f'{100 * larg_b:.4f} pp/mes de C2-B,')
print(f'   {larg_b / larg_e:.1f}x menor, porque EW e indice sao quase a mesma carteira e o excedente')
print('   quase nao varia. A mesma fracao "0,00" significa coisas diferentes nas duas series.')

# ============================================================================ D4
print('\n' + '=' * 124)
print('CELULA D4 -- METRICAS FINAIS')
print('=' * 124)

rf = TESTE['C2-A']['ret_cdi'].to_numpy()
print('\nS1 (prova do rf antes de qualquer Sharpe): excedente do CDI sobre o rf usado -> '
      f'maior |desvio| = {np.abs(TESTE["C2-A"]["ret_cdi"].to_numpy() - rf).max():.3e} em {len(rf)} meses')
assert np.abs(TESTE['C2-A']['ret_cdi'].to_numpy() - rf).max() < 1e-15, 'S1 FALHOU'
print('   S1 PASSOU -- ha UM unico rf no projeto, o CDI de D13.')


def metricas(nome, r_bruto, r_liq, meses):
    r = np.asarray(r_liq, dtype=float)
    exc = r - rf
    down = np.minimum(exc, 0.0)
    dd_dev = np.sqrt(float((down ** 2).mean()))
    w = np.cumprod(1 + r)
    pico = np.maximum.accumulate(w)
    ddser = w / pico - 1.0
    i = int(np.argmin(ddser))
    j = int(np.argmax(pico[:i + 1] == pico[i]))
    rec = np.flatnonzero(w[i:] >= pico[i])
    if len(rec):
        recup = f'{int(rec[0])} meses (ate {meses[i + int(rec[0])]})'
    else:
        recup = f'NAO recuperado ate o fim ({len(w) - 1 - i} meses)'
    ordem = np.sort(r)
    k5 = max(1, int(np.floor(0.05 * len(r))))
    var5 = float(np.percentile(r, 5))
    cvar5 = float(ordem[:k5].mean())
    return {'serie': nome,
            'bruto_%aa': anual(r_bruto), 'liq50_%aa': anual(r),
            'vol_%aa': 100 * r.std(ddof=1) * np.sqrt(12),
            'Sharpe': (exc.mean() * 12) / (exc.std(ddof=1) * np.sqrt(12)) if exc.std(ddof=1) > 0 else np.nan,
            'Sortino': (exc.mean() * 12) / (dd_dev * np.sqrt(12)) if dd_dev > 0 else np.nan,
            'DDmax_%': 100 * float(ddser[i]), 'DD inicio': meses[j], 'DD fundo': meses[i],
            'recuperacao': recup,
            'pior mes_%': 100 * float(r.min()), 'melhor mes_%': 100 * float(r.max()),
            'meses+ %': 100 * float((r > 0).mean()),
            'VaR5_%': 100 * var5, 'CVaR5_%': 100 * cvar5}


d = TESTE['C2-A']
linhas = [
    metricas('C2-A (leitura A)', d['bruto'], d['liquido'], meses_ret),
    metricas('C2-B (leitura B)', TESTE['C2-B']['bruto'], TESTE['C2-B']['liquido'], meses_ret),
    metricas('EW universo elegivel', TESTE['EW_universo']['bruto'], TESTE['EW_universo']['liquido'], meses_ret),
    metricas('indice interno (D12)', d['ret_indice'], d['ret_indice'], meses_ret),
    metricas('indice ex-max (D16)', d['ret_indice_ex_max'], d['ret_indice_ex_max'], meses_ret),
    metricas('CDI (D13)', d['ret_cdi'], d['ret_cdi'], meses_ret),
]
tabD4 = pd.DataFrame(linhas)
print('\n(a) RETORNO, RISCO E DRAWDOWN (TESTE, 103 meses; estrategias LIQUIDAS de 50 bps)')
print(tabD4[['serie', 'bruto_%aa', 'liq50_%aa', 'vol_%aa', 'Sharpe', 'Sortino', 'DDmax_%',
             'DD inicio', 'DD fundo', 'recuperacao', 'pior mes_%', 'melhor mes_%',
             'meses+ %']].round(4).to_string(index=False))
print('\nSortino: (media do excedente sobre o CDI)*12 / (semi-desvio abaixo de ZERO do excedente)*raiz(12).')

print('\n(b) VaR e CVaR MENSAIS a 5% -- RELATORIO DESCRITIVO')
print(tabD4[['serie', 'VaR5_%', 'CVaR5_%']].round(4).to_string(index=False))
print('LEGENDA OBRIGATORIA: VaR e CVaR aqui sao ESTATISTICAS DA AMOSTRA REALIZADA de 103 meses, nao')
print('previsoes. Nao foram usados em nenhuma decisao do modelo -- nao entram em selecao, em peso,')
print('em gate nem em dimensionamento. O MODEL_SPEC nao contem risco-alvo, vol targeting nem teto.')

print('\n(c) HHI MEDIO E NUMERO EFETIVO DE ATIVOS')
lin = []
for v, trilha in [('C2-A', trilha_a), ('C2-B', trilha_b)]:
    tr = trilha[trilha['particao'] == 'TESTE']
    # O HHI e da CARTEIRA INTEIRA: a fatia em CDI da leitura B e uma posicao e entra na soma
    # dos quadrados. Sem ela, os pesos nao somam 1 e 1/HHI daria um numero efetivo MAIOR que
    # o numero de nomes -- impossivel por definicao.
    soma_acoes = tr.groupby('mes')['peso'].sum()
    h = tr.groupby('mes')['peso'].apply(lambda w: float((w ** 2).sum())) + (1.0 - soma_acoes) ** 2
    lin.append({'serie': v, 'HHI medio': h.mean(), 'n efetivo medio': (1 / h).mean(),
                'n nomes medio': tr.groupby('mes').size().mean(),
                'peso medio em CDI %': 100 * float((1 - soma_acoes).mean())})
el_t = split[split['elegivel'] & split['mes'].isin(meses_dec_teste)]
nel = el_t.groupby('mes').size()
lin.append({'serie': 'EW universo elegivel', 'HHI medio': float((1 / nel).mean()),
            'n efetivo medio': float(nel.mean()), 'n nomes medio': float(nel.mean())})
nidx = bi.loc[meses_ret, 'n_ativos_indice']
lin.append({'serie': 'indice interno (D12)', 'HHI medio': float((1 / nidx).mean()),
            'n efetivo medio': float(nidx.mean()), 'n nomes medio': float(nidx.mean())})
print(pd.DataFrame(lin).round(6).to_string(index=False))
print('em C2-A o n efetivo iguala EXATAMENTE o n de nomes (35,13 e 35,13), porque o peso e igual --')
print('e a mesma verificacao executavel de C1, item (g). Em C2-B ele e MENOR (28,29 contra 35,13)')
print('por duas razoes somadas: o peso nao e igual entre acoes (cada modelo divide 1/M pelo seu K)')
print('e ~10,1% do capital fica numa unica posicao, o CDI, que entra na soma dos quadrados.')

print('\n(d) CAPACITY -- estimativa por participacao no volume mensal (VOLTOT)')
vol_de = {(m, c): v for m, c, v in zip(split['mes'], split['CODISI'], split['VOLTOT'])}
tr = trilha_a[trilha_a['particao'] == 'TESTE'].copy()
tr['VOLTOT'] = [vol_de.get((m, c), np.nan) for m, c in zip(tr['mes'], tr['CODISI'])]
print(f'  VOLTOT das posicoes do teste: mediana R$ {tr["VOLTOT"].median():,.0f}  '
      f'p10 R$ {tr["VOLTOT"].quantile(.10):,.0f}  min R$ {tr["VOLTOT"].min():,.0f}')
print('  UNIDADE DECLARADA: VOLTOT tratado em REAIS, a mesma convencao de A8/L14. Se o campo bruto')
print('  do COTAHIST carregasse 2 casas implicitas, todos os valores de capacity cairiam 100x.')
lin = []
for p in (0.05, 0.10):
    tr['cap_i'] = p * tr['VOLTOT'] / tr['peso']
    por_mes_cap = tr.groupby('mes')['cap_i'].min()
    i_bind = tr.loc[tr['cap_i'].idxmin()]
    med_vol = tr.groupby('mes')['VOLTOT'].median()
    n_med = tr.groupby('mes').size()
    cap_mediana_vol = (p * med_vol * n_med)
    lin.append({'participacao': f'{p:.0%}',
                'capacity (min do mes) mediana R$': float(por_mes_cap.median()),
                'capacity (min do mes) p10 R$': float(por_mes_cap.quantile(.10)),
                'capacity (min do mes) minimo R$': float(por_mes_cap.min()),
                'capacity pelo VOLTOT MEDIANO R$': float(cap_mediana_vol.median()),
                'ativo mais restritivo': f'{i_bind["CODNEG"]} ({i_bind["mes"]})'})
cap = pd.DataFrame(lin)
print(cap.to_string(index=False, float_format=lambda x: f'{x:,.0f}'))
print('  duas leituras impressas de proposito: a "min do mes" e a restricao REAL (basta um nome')
print('  ilquido para limitar a carteira inteira); a "VOLTOT mediano" e a que o enunciado descreve')
print('  literalmente e ignora o nome mais restritivo. A diferenca entre elas E o resultado.')
print('\n  CRUZAMENTO COM L14 (elegiveis com VOLTOT baixo, no TESTE):')
el_t2 = el_t.copy()
for lim, rot in [(1e6, 'R$ 1 milhao'), (5e6, 'R$ 5 milhoes'), (1e7, 'R$ 10 milhoes')]:
    c = el_t2.groupby('mes').apply(lambda g: int((g['VOLTOT'] < lim).sum()), include_groups=False)
    print(f'   abaixo de {rot}: mediana de {c.median():.0f} elegiveis/mes de '
          f'{nel.median():.0f} ({100 * c.median() / nel.median():.1f}%)')
sel_baixo = tr.groupby('mes').apply(lambda g: int((g['VOLTOT'] < 1e6).sum()), include_groups=False)
print(f'   entre as SELECIONADAS: mediana de {sel_baixo.median():.0f} por mes com VOLTOT < R$1M, '
      f'em {int((sel_baixo > 0).sum())} de {len(sel_baixo)} meses ha ao menos uma')

print('\n(e) VERSAO EX-MAIOR-CONTRIBUINTE (D16) E A CORRECAO QUE TORNA A COMPARACAO JUSTA')
n_cart_med = tr.groupby('mes').size().median()
n_idx_med = float(nidx.median())
k_justo = int(round(n_idx_med / n_cart_med))
print(f'  D16 literal remove 1 nome de cada lado: 1/{n_cart_med:.0f} = {100 / n_cart_med:.2f}% da carteira')
print(f'  contra 1/{n_idx_med:.0f} = {100 / n_idx_med:.2f}% do indice -- o mesmo corte pesa '
      f'{n_idx_med / n_cart_med:.1f}x mais sobre a carteira.')
print(f'  CORRECAO: remover a mesma FRACAO. Para 1 nome da carteira, o equivalente no indice e')
print(f'  round({n_idx_med:.0f}/{n_cart_med:.0f}) = {k_justo} nomes. As duas versoes abaixo.')
ret_por_mes_el = {m: g for m, g in el_t.groupby('mes')}
lin = []
for mes_r in meses_ret:
    g = eleg_por_mes.get(mes_r)
    if g is None:
        lin.append({'mes_ret': mes_r, 'idx': np.nan, 'idx_ex1': np.nan, 'idx_exk': np.nan})
        continue
    r = np.sort(g['ret_1m'].to_numpy())[::-1]
    lin.append({'mes_ret': mes_r, 'idx': r.mean(), 'idx_ex1': r[1:].mean(), 'idx_exk': r[k_justo:].mean()})
ix = pd.DataFrame(lin).set_index('mes_ret')
cart_ex1 = []
for v, trilha in [('C2-A', trilha_a)]:
    trv = trilha[trilha['particao'] == 'TESTE']
    for mes, g in trv.groupby('mes'):
        m1 = str(pd.Period(mes, freq='M') + 1)
        rr = np.array([ret_de.get((m1, c), np.nan) for c in g['CODISI']], dtype=float)
        w = g['peso'].to_numpy()
        obs = ~np.isnan(rr)
        rr0 = np.where(obs, rr, 0.0)
        if obs.sum() == 0:
            cart_ex1.append({'mes_ret': m1, 'cart': float((w * rr0).sum()), 'cart_ex1': np.nan})
            continue
        pior = np.argmax(np.where(obs, rr, -np.inf))
        keep = np.ones(len(w), dtype=bool)
        keep[pior] = False
        wn = w[keep] * (w.sum() / w[keep].sum())
        cart_ex1.append({'mes_ret': m1, 'cart': float((w * rr0).sum()),
                         'cart_ex1': float((wn * rr0[keep]).sum())})
ce = pd.DataFrame(cart_ex1).set_index('mes_ret').reindex(meses_ret)
comp_ex = pd.DataFrame({
    'serie': ['carteira C2-A completa', 'carteira C2-A ex-1 (2,86% dos nomes)',
              'indice completo', f'indice ex-1 (0,34%) -- D16 literal',
              f'indice ex-{k_justo} (~2,9%) -- MESMA FRACAO'],
    'bruto_%aa': [anual(ce['cart']), anual(ce['cart_ex1']), anual(ix['idx']),
                  anual(ix['idx_ex1']), anual(ix['idx_exk'])]})
print(comp_ex.round(4).to_string(index=False))
print('  Com o corte de MESMA FRACAO dos dois lados, a distancia entre carteira e indice na versao')
print('  ex-max deixa de ser um artefato do numero de nomes. D16 LITERAL continua sendo o reporte')
print('  obrigatorio do projeto; esta correcao entra AO LADO dele, nao no lugar.')

print('\n(f) TABELA FINAL CONSOLIDADA DO PROJETO')
extra = []
for nome, serie_key in [('C2-A (leitura A)', 'C2-A'), ('C2-B (leitura B)', 'C2-B'),
                        ('EW universo elegivel', 'EW_universo')]:
    dd_ = TESTE[serie_key]
    exc = (dd_['liquido'] - dd_['ret_indice']).to_numpy()
    mu, se, t = t_media(exc, 3)
    extra.append({'serie': nome, 'giro_ow_%mes': 100 * dd_['turnover'].mean() / 2,
                  'exc vs indice %mes': 100 * mu, 't NW3': t})
for nome in ['indice interno (D12)', 'indice ex-max (D16)', 'CDI (D13)']:
    extra.append({'serie': nome, 'giro_ow_%mes': np.nan, 'exc vs indice %mes': np.nan, 't NW3': np.nan})
final = tabD4.merge(pd.DataFrame(extra), on='serie', how='left')
cols = ['serie', 'bruto_%aa', 'liq50_%aa', 'vol_%aa', 'Sharpe', 'Sortino', 'DDmax_%',
        'meses+ %', 'giro_ow_%mes', 'VaR5_%', 'CVaR5_%', 'exc vs indice %mes', 't NW3']
print(final[cols].round(4).rename(columns={'Sharpe': 'Sharpe [NAO ORDENAVEL]'}).to_string(index=False))
print('[NAO ORDENAVEL] (plano item 6, rodada 3): com excesso negativo contra o CDI, mu/sigma cresce')
print('com sigma -- a selecao INVERTIDA rende -3,1569%aa liq50 com Sharpe -0,3125, melhor que o')
print('Sharpe -0,3330 de C2-A, que rende -1,6673%aa (1,49 pp/ano a mais). A razao inverte a')
print('ordenacao em vez de resumi-la (Israelsen 2005). PROIBIDO ordenar series por Sharpe.')
print('O parquet NAO foi reescrito -- o rotulo vive na impressao, a nota de rodape aqui.')
print('CONVENCAO DE GIRO (plano item 7): giro_ow_%mes e UMA perna; a coluna turnover dos parquets')
print('e DUAS pernas (soma |dw|); o custo de bps e cobrado sobre DUAS pernas.')

# ============================================================================ SANIDADE
print('\n' + '=' * 124)
print('SANIDADE')
print('=' * 124)
print('S1 PASSOU (provada acima, antes de qualquer Sharpe ser calculado).')

res_max = float(br['residuo'].abs().max())
print(f'S2 residuo maximo da decomposicao de Brinson (alocacao + selecao - excedente): {res_max:.3e}')
assert res_max < 1e-9, 'S2 FALHOU'
print('   S2 PASSOU')

print(f'S3 bootstrap: UM so, bloco={BLOCO} definido pela autocorrelacao MEDIDA antes de rodar, '
      f'semente {SEMENTE_BOOTSTRAP} impressa, {N_BOOT} reamostragens.')
print('   Nenhuma variante foi rodada. O unico segundo bootstrap desta celula e o CONTROLE sobre o')
print('   EW, que usa bloco, semente e n IDENTICOS e existe para testar se a metrica discrimina --')
print('   nao e uma variante entre as quais se escolhe.')
print('   S3 PASSOU')

print('\nS4 -- CONFRONTO COM A REGUA DE FATOS [F], um a um:')
checks = [
    ('marco: 3 meses = 88,5% do alfa', 'nao contradito -- e sobre OUTRA base/estrategia; aqui o '
     'excedente acumulado e negativo e a fracao nao e definida do mesmo jeito (item b de D2 diz isso)'),
    ('marco: giro 52,2%/mes one-way; breakeven ~53 bps', 'nao contradito -- aqui giro 58,3%/mes e '
     'breakeven NEGATIVO; sao numeros DESTA base, e a comparacao esta declarada em C1/C3'),
    ('peso igual nas mesmas acoes rendia +0,80 a +0,83%/mes', 'nao contradito -- nao reproduzimos '
     'a carteira de marco; medimos peso igual nas NOSSAS selecoes'),
    ('EW do universo inteiro: +0,37%/mes com giro 5%/mes, sobrevivia ao custo',
     f'CONSISTENTE -- aqui EW rende {anual(TESTE["EW_universo"]["bruto"]) / 12:.3f}%/mes equivalente '
     f'anualizado {anual(TESTE["EW_universo"]["bruto"]):.2f}%aa com giro one-way '
     f'{100 * TESTE["EW_universo"]["turnover"].mean() / 2:.2f}%/mes e sobrevive ao custo '
     f'({anual(TESTE["EW_universo"]["liquido"]):.2f}%aa liquido)'),
    ('clip(-0,8;+2,0) REDUZIA o resultado e nao pode voltar', 'nao contradito -- nenhum clip, '
     'winsorizacao ou dropna foi aplicado em nenhuma celula deste projeto'),
    ('validacao de marco era NEGATIVA (-0,07/-0,15/-0,11%/mes)',
     f'CONSISTENTE em sinal -- aqui o excedente liquido de C2-A e '
     f'{100 * (TESTE["C2-A"]["liquido"] - TESTE["C2-A"]["ret_indice"]).mean():+.4f}%/mes'),
    ('rel_1m_rank == mom_1m_rank, rel_* banida', 'nao contradito -- as 6 features seguem congeladas'),
    ('0 de 5.624 posicoes perderam preco (viés de sobrevivencia)',
     'nao contradito -- aqui 23 de 6.567 perderam, o oposto da assinatura, medido em C1'),
    ('poder: ~331 meses para p<0,05; todo teste e NAO-REFUTACAO',
     'RESPEITADO -- nenhum resultado desta sessao e apresentado como confirmacao'),
    ('Jensen de marco: alfa +1,09%/mes t~2,5; Dimson ~0,83% t~1,8',
     'nao contradito -- sao numeros da base de marco; os desta base estao no item (e) e sao negativos'),
]
for f, veredito in checks:
    print(f'   [F] {f}\n       -> {veredito}')
print('   S4 PASSOU -- nenhum numero desta sessao contradiz um [F].')

# ============================================================================ gravacao
br.to_parquet(DIR_INT + r'\d2_brinson.parquet', index=False)
pd.DataFrame({'media_excedente': boot}).to_parquet(DIR_INT + r'\d3_bootstrap.parquet', index=False)
final.to_parquet(DIR_INT + r'\d4_metricas_finais.parquet', index=False)
print('\n' + '=' * 124)
print(f'GRAVADO intermediario\\d2_brinson.parquet         {br.shape}')
print(f'GRAVADO intermediario\\d3_bootstrap.parquet       ({N_BOOT}, 1)')
print(f'GRAVADO intermediario\\d4_metricas_finais.parquet {final.shape}')
print('D2, D3 e D4 CONCLUIDAS -- medicao do modelo congelado, nenhum parametro tocado.')
print('=' * 124)


D2 (decomposicao do excedente) + D3 (bootstrap de bloco) + D4 (metricas finais)
CELULAS DE MEDICAO do modelo congelado em 2026-08-15 18:09:30. NADA muda aqui.
scipy NAO existe nesta .venv: tudo o que precisa de estatistica esta implementado abaixo.

TESTE: 103 meses de decisao (2018-01 a 2026-07), realizando em 2018-02 a 2026-08

CELULA D2 -- DECOMPOSICAO DO EXCEDENTE

(a) MEDIA, DP E t DO EXCEDENTE MENSAL -- t simples e Newey-West com 1, 3 e 6 lags
versao      tipo     vs   n  media_%mes  dp_%mes  ep simples  t simples  t NW(1)  t NW(3)  t NW(6)
  C2-A     bruto indice 103     -0.1103   2.4664      0.2430    -0.4540  -0.4209  -0.4094  -0.3931
  C2-A     bruto ex-max 103      0.2222   2.4372      0.2401     0.9251   0.8641   0.8421   0.8172
  C2-A liquido50 indice 103     -0.6933   2.4709      0.2435    -2.8478  -2.6481  -2.5817  -2.4846
  C2-A liquido50 ex-max 103     -0.3608   2.4416      0.2406    -1.4999  -1.4058  -1.3741  -1.3376
  C2-B     bruto indice 103      0.0008   2.5909   

### Correção pós-conselho (plano item 5) — a coluna Sharpe NÃO É ORDENÁVEL em `d1_tabela_final` e `d4_metricas_finais`

> **Nota de rodapé (obrigatória onde as duas tabelas forem citadas):** com excesso negativo contra um CDI de 9,03%aa, μ/σ cresce com σ: a seleção INVERTIDA rende −3,1569%aa líq50 com Sharpe −0,3125, "melhor" que o Sharpe −0,3330 de V1 (C2-A), que rende −1,6673%aa — 1,49 pp/ano a mais. Entre as dez séries de excesso negativo do projeto, corr(vol, Sharpe) = −0,43 (−0,41 a −0,57 conforme a composição do conjunto). A razão inverte a ordenação em vez de resumi-la (Israelsen, 2005). Para comparar risco-retorno, use retorno excedente acumulado e drawdown separadamente. **Nenhuma frase do dossiê ou do PDF pode ordenar séries por Sharpe.**

Fontes conferidas em 2026-08-16: `d1_tabela_final.parquet` linhas "C2-A (leitura A)" e "selecao INVERTIDA"; convenção de Sharpe da célula C3+D1 (`(exc_rf.mean()*12)/(exc_rf.std(ddof=1)*sqrt(12))`, excesso sobre CDI por `mes_ret`).

## DIAG-COEF -- anatomia dos coeficientes

Celula de MEDICAO. Reexecuta o walk-forward congelado (a reconstrucao reproduz os 65.273 scores
com desvio **0,000e+00**) e grava um registro por (mes de decisao, modelo, feature) com
coeficiente, erro-padrao, t, numero de observacoes da janela e R2 da regressao.

**A hipotese que esta celula foi escrita para testar -- "a instabilidade de estimacao dos
coeficientes destroi o sinal que existe nas features" -- NAO se sustenta na medicao, mas o
teste NAO TEM PODER (D33).** Os coeficientes tem autocorrelacao lag-1 de 0,96 (generalista) e
0,95 (mediana dos especialistas) -- indistinguiveis do nulo mecanico 0,95-0,98 da janela de 36m
sobreposta (D33; sem comparacao com marco); a baixa persistencia do score vem do movimento das FEATURES, nao
dos coeficientes; e o remedio direto da hipotese (suavizar por 12 janelas) leva o IC de +0,0025
para +0,0013, ou seja, nao melhora.

O item (1e), **reconciliacao com DIAG-1**, nao estava no enunciado e foi acrescentado porque dois
numeros medidos do projeto pareciam se contradizer -- persistencia do score de 0,533 contra
features de 0,90 e coeficientes de 0,96. A conta separa as duas pernas e resolve a contradicao.

Os tres scores alternativos do item 6 sao medidos **APENAS por IC**. Nenhuma carteira, nenhum
backtest, nenhum retorno. A celula nao recomenda arquitetura -- a decisao e do humano e esta fora
dela. O MODEL_SPEC segue congelado.


In [1]:
import sys

import numpy as np
import pandas as pd

if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8')

pd.set_option('display.width', 270)
pd.set_option('display.max_columns', 80)
pd.set_option('display.max_rows', 400)

RAIZ = r'C:\Users\lucca\quant2026'
DIR_INT = RAIZ + r'\intermediario'

RANKS = ['mom_1m_rank', 'mom_3m_rank', 'mom_6m_rank', 'mom_12m_rank', 'vol_3m_rank', 'vol_6m_rank']
MOM = ['mom_1m_rank', 'mom_3m_rank', 'mom_6m_rank', 'mom_12m_rank']
VOL = ['vol_3m_rank', 'vol_6m_rank']
JANELA, MIN_ATIVOS_D01, MIN_OBS_D01 = 36, 5, 150
N_SUAVIZA = 12

print('=' * 126)
print('DIAG-COEF -- ANATOMIA DOS COEFICIENTES. CELULA DE MEDICAO.')
print('Nenhum parametro do MODEL_SPEC (congelado 2026-08-15 18:09:30) e alterado. Nenhuma variante')
print('de arquitetura e rodada. Nenhum backtest, nenhuma carteira, nenhum retorno para os scores')
print('alternativos do item 6 -- so IC. A decisao sobre o que fazer com estes numeros e do humano')
print('e esta FORA desta celula.')
print('=' * 126)

# ==================================================================== helpers
def _pearson(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    ok = np.isfinite(x) & np.isfinite(y)
    x, y = x[ok] - x[ok].mean(), y[ok] - y[ok].mean()
    den = np.sqrt(float(x @ x) * float(y @ y))
    return float(x @ y) / den if den > 0 else np.nan


def spearman(x, y):
    """Spearman = Pearson sobre postos, method='average' -- a MESMA convencao de B2/B5/D1."""
    rx = pd.Series(np.asarray(x, dtype=float)).rank(method='average').to_numpy()
    ry = pd.Series(np.asarray(y, dtype=float)).rank(method='average').to_numpy()
    return _pearson(rx, ry)


def nw_t(x, lags=3):
    x = np.asarray(pd.Series(x).dropna(), dtype=float)
    n = len(x)
    mu = x.mean()
    d = x - mu
    v = float(d @ d) / n
    for l in range(1, lags + 1):
        v += 2.0 * (1.0 - l / (lags + 1.0)) * (float(d[l:] @ d[:-l]) / n)
    return mu, np.sqrt(v / n), mu / np.sqrt(v / n)


def mes_idx(s):
    return s.str[:4].astype(int) * 12 + s.str[5:7].astype(int)


def idx_mes(i):
    a, m = divmod(int(i) - 1, 12)
    return f'{a:04d}-{m + 1:02d}'


def ols_completo(X, y):
    """OLS COM intercepto -- identica a de B4/B5 -- devolvendo tambem ep, t e R2."""
    n = len(y)
    Xd = np.column_stack([np.ones(n), X])
    beta = np.linalg.lstsq(Xd, y, rcond=None)[0]
    e = y - Xd @ beta
    k = Xd.shape[1]
    s2 = float(e @ e) / (n - k)
    XtX_inv = np.linalg.pinv(Xd.T @ Xd)
    se = np.sqrt(np.maximum(np.diag(s2 * XtX_inv), 0.0))
    ss_tot = float(((y - y.mean()) ** 2).sum())
    r2 = 1.0 - float(e @ e) / ss_tot if ss_tot > 0 else np.nan
    return beta, se, r2


# ==================================================================== insumos
prev = pd.read_parquet(DIR_INT + r'\b4b5_previsoes.parquet')
b3 = pd.read_parquet(DIR_INT + r'\b3_dataset.parquet')
prev['CODISI'] = prev['CODISI'].astype(str)
prev['mes_idx'] = mes_idx(prev['mes'])
prev = prev.sort_values('mes_idx').reset_index(drop=True)
b3['mes_idx'] = mes_idx(b3['mes'])
b3 = b3.sort_values('mes_idx').reset_index(drop=True)
print(f'\nprevisoes {prev.shape} | dataset de treino {b3.shape}')

Xtr = b3[RANKS].to_numpy(dtype='float64')
ytr = b3['alfa_fut'].to_numpy(dtype='float64')
sub_tr = b3['subsetor'].to_numpy()
mi_tr = b3['mes_idx'].to_numpy()
Xp_all = prev[RANKS].to_numpy(dtype='float64')
sub_p = prev['subsetor'].to_numpy()
mi_p = prev['mes_idx'].to_numpy()

# ==================================================================== walk-forward com coeficientes
registros = []
score_rec = np.full(len(prev), np.nan)
modelo_rec = np.empty(len(prev), dtype=object)
betas_por_modelo = {}          # (modelo, T) -> vetor beta completo (intercepto + 6)
meses_decisao = sorted(set(mi_p))

for T in meses_decisao:
    lo = np.searchsorted(mi_tr, T - (JANELA + 1), 'left')
    hi = np.searchsorted(mi_tr, T - 2, 'right')
    Xt, yt, st = Xtr[lo:hi], ytr[lo:hi], sub_tr[lo:hi]
    plo = np.searchsorted(mi_p, T, 'left')
    phi = np.searchsorted(mi_p, T, 'right')
    if phi == plo or len(yt) < 7:
        continue
    Xp, sp = Xp_all[plo:phi], sub_p[plo:phi]
    Xdp = np.column_stack([np.ones(len(Xp)), Xp])

    bg, seg, r2g = ols_completo(Xt, yt)
    betas_por_modelo[('GENERALISTA', T)] = bg
    for i, f in enumerate(RANKS):
        registros.append({'T': idx_mes(T), 'T_idx': T, 'modelo': 'GENERALISTA', 'tipo': 'GENERALISTA',
                          'feature': f, 'coef': bg[i + 1], 'ep': seg[i + 1],
                          't': bg[i + 1] / seg[i + 1] if seg[i + 1] > 0 else np.nan,
                          'n_obs': len(yt), 'r2': r2g, 'intercepto': bg[0]})

    sc = Xdp @ bg
    mo = np.full(len(Xp), 'GENERALISTA', dtype=object)
    subu, cu = np.unique(sp, return_counts=True)
    n_at = dict(zip(subu, cu))
    stu, ct = np.unique(st, return_counts=True)
    n_ob = dict(zip(stu, ct))
    for s in subu:
        if s == 'SEM_SETOR' or n_at.get(s, 0) < MIN_ATIVOS_D01 or n_ob.get(s, 0) < MIN_OBS_D01:
            continue
        m = st == s
        bs, ses, r2s = ols_completo(Xt[m], yt[m])
        betas_por_modelo[(s, T)] = bs
        mp = sp == s
        sc[mp] = Xdp[mp] @ bs
        mo[mp] = s
        for i, f in enumerate(RANKS):
            registros.append({'T': idx_mes(T), 'T_idx': T, 'modelo': s, 'tipo': 'ESPECIALISTA',
                              'feature': f, 'coef': bs[i + 1], 'ep': ses[i + 1],
                              't': bs[i + 1] / ses[i + 1] if ses[i + 1] > 0 else np.nan,
                              'n_obs': int(m.sum()), 'r2': r2s, 'intercepto': bs[0]})
    score_rec[plo:phi] = sc
    modelo_rec[plo:phi] = mo

coef = pd.DataFrame(registros)
print(f'coeficientes gravados: {coef.shape} '
      f'({coef.groupby("tipo")["modelo"].size().to_dict()}); '
      f'{coef[["T", "modelo"]].drop_duplicates().shape[0]} regressoes')

# ---- S1
dev = np.nanmax(np.abs(score_rec - prev['score'].to_numpy()))
print(f'\nS1 maior |score reconstruido - b4b5_previsoes| = {dev:.3e}   '
      f'modelos identicos: {int((modelo_rec == prev["modelo"].to_numpy()).sum())} de {len(prev)}')
assert dev == 0.0, 'S1 FALHOU: reconstrucao nao reproduz o parquet congelado'
assert (modelo_rec == prev['modelo'].to_numpy()).all(), 'S1 FALHOU: atribuicao de modelo divergiu'
print('   S1 PASSOU -- os coeficientes desta celula sao os DA execucao congelada.')

# ==================================================================== 1. ESTABILIDADE TEMPORAL
print('\n' + '=' * 126)
print('1. ESTABILIDADE TEMPORAL DOS COEFICIENTES')
print('=' * 126)


def estat_serie(g):
    g = g.sort_values('T_idx')
    c = g['coef'].to_numpy()
    t_idx = g['T_idx'].to_numpy()
    cons = np.flatnonzero(np.diff(t_idx) == 1)
    if len(cons) >= 2:
        a = _pearson(c[cons], c[cons + 1])
        troca = 100.0 * float(np.mean(np.sign(c[cons]) != np.sign(c[cons + 1])))
    else:
        a, troca = np.nan, np.nan
    mu, dp = c.mean(), c.std(ddof=1) if len(c) > 1 else np.nan
    return pd.Series({'meses': len(c), 'pares_consec': len(cons), 'media': mu, 'dp': dp,
                      'CV': dp / abs(mu) if mu != 0 else np.nan, 'autocorr_lag1': a,
                      'troca_sinal_%': troca})


est = coef.groupby(['tipo', 'modelo', 'feature'], observed=True).apply(estat_serie, include_groups=False).reset_index()

print('\n(1d) GENERALISTA isoladamente -- a maior amostra do projeto, logo o mais estavel esperado:')
g = est[est['modelo'] == 'GENERALISTA'].set_index('feature').loc[RANKS]
print(g[['meses', 'media', 'dp', 'CV', 'autocorr_lag1', 'troca_sinal_%']].round(4).to_string())

print('\n(1a-1c) ESPECIALISTAS -- distribuicao entre os modelos (so os com >=12 pares consecutivos):')
esp = est[(est['tipo'] == 'ESPECIALISTA') & (est['pares_consec'] >= 12)]
print(f'  {esp["modelo"].nunique()} subsetores especialistas qualificados, '
      f'{len(esp)} pares (modelo, feature)')
res = esp.groupby('feature', observed=True)[['dp', 'CV', 'autocorr_lag1', 'troca_sinal_%']].agg(
    ['median', 'mean', 'min', 'max'])
print(res.round(4).to_string())

print('\n  CONTEXTO OBRIGATORIO: janelas consecutivas compartilham 35 de 36 meses. A autocorrelacao')
print('  lag-1 do coeficiente DEVERIA ser muito alta -- em marco era 0,80-0,81 [F].')
ac_gen = g['autocorr_lag1']
ac_esp = esp['autocorr_lag1']
print(f'  GENERALISTA: autocorr lag-1 por feature de {ac_gen.min():.4f} a {ac_gen.max():.4f} '
      f'(mediana {ac_gen.median():.4f})')
print(f'  ESPECIALISTAS: mediana {ac_esp.median():.4f}, p10 {ac_esp.quantile(.10):.4f}, '
      f'p90 {ac_esp.quantile(.90):.4f}, min {ac_esp.min():.4f}, max {ac_esp.max():.4f}')
print(f'  fracao de series de especialista com autocorr abaixo de 0,80: '
      f'{100 * float((ac_esp < 0.80).mean()):.1f}%')
print(f'  fracao abaixo de 0,50: {100 * float((ac_esp < 0.50).mean()):.1f}%')

print('\n(1e) RECONCILIACAO COM DIAG-1 -- acrescentada porque dois numeros MEDIDOS deste projeto')
print('     parecem se contradizer, e deixar os dois no dossie sem a conta seria esconder o')
print('     problema: DIAG-1 mediu persistencia do SCORE em 0,5330 no teste, enquanto as features')
print('     tem persistencia de ate 0,9036 e os coeficientes, acima, tem autocorrelacao de 0,96.')
print('     Se coeficiente e feature sao persistentes, por que o score nao e?')
print('     A conta abaixo separa as duas pernas, em cada par de meses consecutivos do TESTE, sobre')
print('     os ativos presentes nos dois meses:')
print('       rho_total    = Spearman(score_T, score_T+1)                  -> tem que bater com DIAG-1')
print('       rho_features = Spearman(score_T, beta_T aplicado as features de T+1)   -> so as features')
print('                      se movem, os coeficientes ficam congelados em T')
print('       rho_coef     = Spearman(beta_T em x_T+1, beta_T+1 em x_T+1)  -> so os coeficientes se')
print('                      movem, as features ficam congeladas em T+1')
mods_arr = prev['modelo'].to_numpy()
meses_teste_dec = sorted(prev.loc[prev['particao'] == 'TESTE', 'mes'].unique())
lin_rec = []
for m in meses_teste_dec[:-1]:
    T = int(mes_idx(pd.Series([m])).iloc[0])
    m1 = idx_mes(T + 1)
    a = prev[prev['mes'] == m]
    b = prev[prev['mes'] == m1]
    j = a[['CODISI', 'score', 'modelo']].merge(
        b[['CODISI', 'score', 'modelo'] + RANKS], on='CODISI', suffixes=('_T', '_T1'))
    if len(j) < 10:
        continue
    X1 = np.column_stack([np.ones(len(j)), j[RANKS].to_numpy()])
    s_bT_x1 = np.full(len(j), np.nan)
    s_bT1_x1 = np.full(len(j), np.nan)
    for i in range(len(j)):
        kT = (j['modelo_T'].iloc[i], T)
        kT1 = (j['modelo_T1'].iloc[i], T + 1)
        if kT in betas_por_modelo:
            s_bT_x1[i] = X1[i] @ betas_por_modelo[kT]
        if kT1 in betas_por_modelo:
            s_bT1_x1[i] = X1[i] @ betas_por_modelo[kT1]
    ok = np.isfinite(s_bT_x1) & np.isfinite(s_bT1_x1)
    if ok.sum() < 10:
        continue
    lin_rec.append({
        'mes': m, 'n': int(ok.sum()),
        'rho_total': spearman(j['score_T'].to_numpy()[ok], j['score_T1'].to_numpy()[ok]),
        'rho_features': spearman(j['score_T'].to_numpy()[ok], s_bT_x1[ok]),
        'rho_coef': spearman(s_bT_x1[ok], s_bT1_x1[ok])})
rec = pd.DataFrame(lin_rec)
print(f'\n     {len(rec)} pares de meses consecutivos do TESTE:')
print(rec[['rho_total', 'rho_features', 'rho_coef']].agg(['mean', 'std', 'min', 'max']).round(4).to_string())
print(f'\n     rho_total medio {rec["rho_total"].mean():.4f} -- bate com os 0,5330 de DIAG-1.')
print(f'     rho_coef medio  {rec["rho_coef"].mean():.4f}: congelando as features, trocar os')
print('     coeficientes de um mes para o outro quase nao reordena a secao transversal.')
print(f'     rho_features medio {rec["rho_features"].mean():.4f}: congelando os coeficientes, o')
print('     movimento das FEATURES sozinho ja reproduz quase toda a perda de persistencia.')
print('     => a baixa persistencia do score NAO vem da instabilidade dos coeficientes. Vem de o')
print('     score ser uma combinacao com pesos pequenos e de sinais mistos, em que as features de')
print('     alta persistencia se cancelam parcialmente e o que sobra e dominado por mom_1m_rank,')
print('     cuja persistencia cross-seccional medida em DIAG-1 e -0,0024 -- praticamente zero, e')
print('     que e justamente a feature com o maior |media|/dp de coeficiente no generalista (1,385).')

# ==================================================================== 2. CONCORDANCIA
print('\n' + '=' * 126)
print('2. CONCORDANCIA ENTRE ESPECIALISTAS SOBRE O SINAL DE CADA FEATURE')
print('=' * 126)
ce = coef[coef['tipo'] == 'ESPECIALISTA']
conc = ce.groupby(['T', 'feature'], observed=True)['coef'].agg(
    n='size', pos=lambda s: int((s > 0).sum())).reset_index()
conc = conc[conc['n'] >= 5]
conc['frac_pos'] = conc['pos'] / conc['n']
conc['maioria_%'] = 100 * np.maximum(conc['frac_pos'], 1 - conc['frac_pos'])
print(f'meses-feature com >=5 especialistas: {len(conc)}')
print('\ndistribuicao da FRACAO de especialistas com coeficiente POSITIVO, por feature:')
print(conc.groupby('feature', observed=True)['frac_pos'].describe(
    percentiles=[.1, .25, .5, .75, .9]).round(4).to_string())
print('\nCONCORDANCIA DA MAIORIA (100% = unanimidade; 50% = empate perfeito, ou seja, sorteio):')
print(conc.groupby('feature', observed=True)['maioria_%'].agg(['mean', 'median', 'min', 'max']).round(2).to_string())
print('\nfracao de meses-feature em que a divisao fica entre 40% e 60% (praticamente empate):')
emp = conc.assign(empate=(conc['frac_pos'] > 0.40) & (conc['frac_pos'] < 0.60))
print((100 * emp.groupby('feature', observed=True)['empate'].mean()).round(2).to_string())
print('\nunanimidade (todos os especialistas com o mesmo sinal), por feature, em % dos meses:')
una = conc.assign(u=(conc['frac_pos'] == 0) | (conc['frac_pos'] == 1))
print((100 * una.groupby('feature', observed=True)['u'].mean()).round(2).to_string())

# ==================================================================== 3. SINAL DO COEF vs SINAL DO IC
print('\n' + '=' * 126)
print('3. SINAL DO COEFICIENTE DO GENERALISTA vs SINAL DO IC INDIVIDUAL DAQUELE MES')
print('=' * 126)
print('IC individual do mes T = Spearman(rank da feature em T, alfa_fut realizado em T), no TESTE.')
print('E o IC DAQUELE MES, nao o do periodo. Note que ele usa o alvo realizado -- e diagnostico')
print('retrospectivo, nunca esteve disponivel em T e nunca entrou em decisao nenhuma.')
pt = prev[(prev['particao'] == 'TESTE') & prev['alfa_fut'].notna()]
ic_mes = []
for m, gg in pt.groupby('mes'):
    if len(gg) < 10:
        continue
    reg = {'T': m}
    for f in RANKS:
        reg[f] = spearman(gg[f], gg['alfa_fut'])
    ic_mes.append(reg)
ic_mes = pd.DataFrame(ic_mes)
cg = coef[coef['modelo'] == 'GENERALISTA'].pivot_table(index='T', columns='feature', values='coef')
comum = sorted(set(ic_mes['T']) & set(cg.index))
icm = ic_mes.set_index('T').loc[comum]
cgm = cg.loc[comum]
lin = []
for f in RANKS:
    ok = np.sign(icm[f].to_numpy()) == np.sign(cgm[f].to_numpy())
    lin.append({'feature': f, 'meses': len(ok), 'concordancia_%': 100 * float(ok.mean()),
                'IC medio do periodo': icm[f].mean(),
                'coef medio do generalista': cgm[f].mean(),
                'sinal do IC medio': int(np.sign(icm[f].mean())),
                'sinal do coef medio': int(np.sign(cgm[f].mean()))})
tab3 = pd.DataFrame(lin)
print()
print(tab3.round(4).to_string(index=False))
print(f'\nconcordancia MEDIA entre as 6 features: {tab3["concordancia_%"].mean():.2f}%')
print('50% e o valor esperado se o sinal estimado fosse sorteado.')

# ==================================================================== 4. SIGNIFICANCIA
print('\n' + '=' * 126)
print('4. SIGNIFICANCIA DOS COEFICIENTES E R2 DAS REGRESSOES')
print('=' * 126)
coef['sig'] = coef['t'].abs() > 2
print('% de coeficientes com |t| > 2, por feature e por tipo de modelo:')
print((100 * coef.pivot_table(index='feature', columns='tipo', values='sig', aggfunc='mean')
       ).round(2).to_string())
print(f'\n% GERAL de coeficientes com |t|>2: {100 * coef["sig"].mean():.2f}% '
      f'({int(coef["sig"].sum())} de {len(coef)})')
print('sob a hipotese nula de coeficiente zero, o esperado seria ~5%.')
r2 = coef.drop_duplicates(['T', 'modelo'])[['T', 'modelo', 'tipo', 'r2', 'n_obs']]
print('\nR2 das regressoes:')
print(r2.groupby('tipo')['r2'].describe(percentiles=[.1, .5, .9]).round(5).to_string())
print(f'\nR2 medio do GENERALISTA: {r2[r2["modelo"] == "GENERALISTA"]["r2"].mean():.5f}')
print(f'R2 medio dos ESPECIALISTAS: {r2[r2["tipo"] == "ESPECIALISTA"]["r2"].mean():.5f}')
print('\nA frase que este item ia testar era "R2 proximo de zero COM coeficientes instaveis e a')
print('assinatura de estimacao sobre ruido". A METADE DELA NAO SE SUSTENTA: os coeficientes NAO sao')
print('instaveis (item 1). O que fica medido e diferente e mais especifico -- o generalista tem R2 de')
print('0,003, isto e, explica 0,3% da variacao do alfa, e o faz de forma ESTAVEL. Estabilidade sem')
print('poder explicativo nao e ruido oscilando: e um sinal minusculo sendo estimado com precisao.')

# ==================================================================== 5. AMOSTRA vs INSTABILIDADE
print('\n' + '=' * 126)
print('5. TAMANHO DE AMOSTRA vs INSTABILIDADE DO COEFICIENTE')
print('=' * 126)
amost = coef.groupby(['tipo', 'modelo', 'feature'], observed=True).agg(
    n_obs_medio=('n_obs', 'mean'), dp_coef=('coef', 'std'), meses=('coef', 'size')).reset_index()
amost = amost[amost['meses'] >= 12]
lin = []
for f in RANKS:
    a = amost[amost['feature'] == f]
    lin.append({'feature': f, 'n_modelos': len(a),
                'corr(n_obs, dp) Pearson': _pearson(a['n_obs_medio'], a['dp_coef']),
                'corr(log n_obs, log dp)': _pearson(np.log(a['n_obs_medio']), np.log(a['dp_coef']))})
print(pd.DataFrame(lin).round(4).to_string(index=False))
todos = amost.copy()
print(f'\npooled (as 6 features juntas, {len(todos)} pares modelo-feature): '
      f'Pearson {_pearson(todos["n_obs_medio"], todos["dp_coef"]):+.4f}, '
      f'log-log {_pearson(np.log(todos["n_obs_medio"]), np.log(todos["dp_coef"])):+.4f}')
print('\nO GENERALISTA e o modelo com a MAIOR amostra. Se a instabilidade fosse subamostragem, ele')
print('seria o mais estavel de todos. Os numeros lado a lado:')
gp = amost[amost['modelo'] == 'GENERALISTA'].set_index('feature').loc[RANKS]
ep = amost[amost['tipo'] == 'ESPECIALISTA'].groupby('feature', observed=True).agg(
    n_obs_medio=('n_obs_medio', 'median'), dp_coef=('dp_coef', 'median'))
cmpp = pd.DataFrame({'n_obs GEN': gp['n_obs_medio'], 'dp GEN': gp['dp_coef'],
                     'n_obs ESP (mediana)': ep['n_obs_medio'], 'dp ESP (mediana)': ep['dp_coef']})
cmpp['razao dp ESP/GEN'] = cmpp['dp ESP (mediana)'] / cmpp['dp GEN']
cmpp['razao n_obs GEN/ESP'] = cmpp['n_obs GEN'] / cmpp['n_obs ESP (mediana)']
print(cmpp.round(4).to_string())

# ==================================================================== 6. CONTRAFACTUAL (SO IC)
print('\n' + '=' * 126)
print('6. TESTE CONTRAFACTUAL DIRETO -- SO IC. NENHUMA CARTEIRA, NENHUM BACKTEST, NENHUM RETORNO.')
print('=' * 126)

# (i) media simples com sinal fixado por HIPOTESE DECLARADA
sinal = {f: (+1.0 if f in MOM else -1.0) for f in RANKS}
print('(i) score_media_simples: media dos 6 ranks com sinal fixado por hipotese DECLARADA --')
print(f'    {sinal}. O sinal e imposto por hipotese economica (momento positivo, volatilidade')
print('    negativa), NAO estimado do dado; e por isso que nao ha coeficiente para ficar instavel.')
prev['score_media_simples'] = sum(sinal[f] * prev[f] for f in RANKS) / len(RANKS)

# (ii) coeficientes suavizados: media das 12 janelas ANTERIORES a T
print(f'\n(ii) score_coef_suavizado: media dos coeficientes das ultimas {N_SUAVIZA} janelas '
      f'ANTERIORES a T.')
print(f'     Leitura literal e conservadora do enunciado: usa as janelas dos meses de decisao')
print(f'     T-{N_SUAVIZA} a T-1, TODAS estritamente anteriores a T. A janela do proprio T fica de')
print('     fora. Cada uma delas ja usava dado ate o seu proprio mes menos 1, logo o dado mais')
print('     recente que entra e o de T-2. Declarado antes de rodar, como em D26.')
score_suav = np.full(len(prev), np.nan)
usados = []
for T in meses_decisao:
    plo = np.searchsorted(mi_p, T, 'left')
    phi = np.searchsorted(mi_p, T, 'right')
    if phi == plo:
        continue
    Xp = Xp_all[plo:phi]
    Xdp = np.column_stack([np.ones(len(Xp)), Xp])
    sp = sub_p[plo:phi]
    mods = prev['modelo'].to_numpy()[plo:phi]
    sc = np.full(len(Xp), np.nan)
    for mod in np.unique(mods):
        anteriores = [betas_por_modelo[(mod, t)] for t in range(T - N_SUAVIZA, T)
                      if (mod, t) in betas_por_modelo]
        mask = mods == mod
        if not anteriores:
            continue
        b = np.mean(np.array(anteriores), axis=0)
        sc[mask] = Xdp[mask] @ b
        usados.append(len(anteriores))
    score_suav[plo:phi] = sc
prev['score_coef_suavizado'] = score_suav
u = np.array(usados)
print(f'     janelas anteriores efetivamente disponiveis por (modelo, mes): mediana {np.median(u):.0f}, '
      f'min {u.min()}, max {u.max()}; {100 * float((u == N_SUAVIZA).mean()):.1f}% com as 12 completas')
print(f'     ativos sem score suavizado (modelo estreando, sem janela anterior): '
      f'{int(np.isnan(score_suav).sum())} de {len(prev)}')

# (iii) so mom_12m_rank
prev['score_so_mom12'] = prev['mom_12m_rank']
print('\n(iii) score_so_mom12: o proprio mom_12m_rank como score. Nenhum coeficiente e estimado.')

# S2
print('\nS2 -- verificacao de que nenhum alternativo usa informacao de T ou posterior:')
print('   (i)   media simples: funcao APENAS dos 6 ranks de T, cujas janelas terminam em T (D19);')
print('         os sinais sao hipotese fixa, nao estimados de dado nenhum. OK.')
max_t_usado = max(t for (_, t) in betas_por_modelo)
viol = 0
for T in meses_decisao:
    for (mod, t) in [(m, t) for (m, t) in betas_por_modelo if t in range(T - N_SUAVIZA, T)]:
        if t >= T:
            viol += 1
print(f'   (ii)  suavizado: janelas usadas para T estao todas em [T-{N_SUAVIZA}, T-1]; '
      f'violacoes (t >= T): {viol}')
assert viol == 0, 'S2 FALHOU'
print('   (iii) so mom_12m_rank: rank de T, janela de 12 meses terminando em T (D19). OK.')
print('   S2 PASSOU')

# S3 + IC pela MESMA funcao
def ic_mensal(df, col):
    ics = []
    for m, gg in df.groupby('mes'):
        if len(gg) < 10 or gg[col].isna().all():
            continue
        h = gg[[col, 'alfa_fut']].dropna()
        if len(h) < 10:
            continue
        ics.append(spearman(h[col], h['alfa_fut']))
    return np.array(ics)


print('\nS3 -- o IC dos alternativos e medido pela MESMA funcao `ic_mensal`/`spearman` que mede o do')
print('score real. Confirmacao executavel: o IC recalculado do score REAL tem que bater com o de')
print('B5/D1 (+0,0025).')
pt2 = prev[(prev['particao'] == 'TESTE') & prev['alfa_fut'].notna()]
lin = []
for col, rot in [('score', 'score REAL (o congelado, 6 features)'),
                 ('score_media_simples', '(i) media simples dos 6 ranks, sinal por hipotese'),
                 ('score_coef_suavizado', f'(ii) coeficientes suavizados ({N_SUAVIZA} janelas anteriores)'),
                 ('score_so_mom12', '(iii) so mom_12m_rank')]:
    ics = ic_mensal(pt2, col)
    mu, se, t = nw_t(ics)
    lin.append({'score': rot, 'n_meses': len(ics), 'IC medio': mu, 'dp': ics.std(ddof=1),
                't (NW3)': t, '% meses > 0': 100 * float((ics > 0).mean())})
tab6 = pd.DataFrame(lin)
print()
print(tab6.round(4).to_string(index=False))
ic_real = tab6.loc[tab6['score'].str.startswith('score REAL'), 'IC medio'].iloc[0]
print(f'\n   IC recalculado do score real = {ic_real:.4f} contra +0,0025 de B5/D1 -- bate.')
assert abs(ic_real - 0.0025) < 0.0005, 'S3 FALHOU: a funcao de IC nao reproduz o numero de B5/D1'
print('   S3 PASSOU')
print(f'\n   (iii) reproduz por construcao o IC de mom_12m_rank medido em DIAG-2 (+0,0542), porque')
print('   Spearman e invariante a transformacao monotona -- e uma conferencia de consistencia.')
print('\nLIMITE DESTA MEDICAO, escrito aqui e nao so no dossie: os tres numeros acima sao IC, e SO')
print('IC. Nenhuma carteira foi construida, nenhum backtest rodou, nenhum retorno foi calculado para')
print('eles. IC nao e retorno -- D24 registrou exatamente isso, e C1-C3 mostraram que a distancia')
print('entre um e outro e grande. Nada aqui recomenda arquitetura; a decisao e do humano e esta fora')
print('desta celula.')

# ==================================================================== 7. VEREDITO
print('\n' + '=' * 126)
print('7. VEREDITO -- as quatro perguntas, com numero')
print('=' * 126)
print('\nP1. Os coeficientes sao instaveis alem do que a mudanca de janela justifica?')
print(f'    Janelas consecutivas compartilham 35 de 36 meses (97,2% do dado). Marco media 0,80-0,81 [F].')
print(f'    GENERALISTA (maior amostra): autocorr lag-1 mediana {ac_gen.median():.4f} '
      f'(faixa {ac_gen.min():.4f} a {ac_gen.max():.4f})')
print(f'    ESPECIALISTAS: mediana {ac_esp.median():.4f}; {100 * float((ac_esp < 0.80).mean()):.1f}% '
      f'abaixo de 0,80 e {100 * float((ac_esp < 0.50).mean()):.1f}% abaixo de 0,50')
print(f'    troca de sinal entre meses consecutivos: generalista mediana '
      f'{g["troca_sinal_%"].median():.2f}%, especialistas mediana {esp["troca_sinal_%"].median():.2f}%')
print('\nP2. Os especialistas concordam entre si sobre o sinal das features?')
for f in RANKS:
    c = conc[conc['feature'] == f]
    print(f'    {f:<14} maioria media {c["maioria_%"].mean():.2f}%  |  '
          f'unanimidade em {100 * float(((c["frac_pos"] == 0) | (c["frac_pos"] == 1)).mean()):.2f}% dos meses  |  '
          f'quase-empate (40-60%) em {100 * float(((c["frac_pos"] > .4) & (c["frac_pos"] < .6)).mean()):.2f}%')
print('\nP3. O modelo estima o sinal certo com que frequencia?')
for r in tab3.itertuples():
    print(f'    {r.feature:<14} concordancia com o IC do proprio mes: {r.concordancia_:.2f}%'
          if hasattr(r, 'concordancia_') else
          f'    {r.feature:<14} concordancia com o IC do proprio mes: {getattr(r, "_3"):.2f}%')
print(f'    media das 6: {tab3["concordancia_%"].mean():.2f}%  (50% = sorteio)')
print('\nP4. A instabilidade e explicada por tamanho de amostra?')
print(f'    correlacao pooled entre n_obs medio do modelo e dp do coeficiente: '
      f'{_pearson(todos["n_obs_medio"], todos["dp_coef"]):+.4f} (log-log '
      f'{_pearson(np.log(todos["n_obs_medio"]), np.log(todos["dp_coef"])):+.4f})')
print(f'    o GENERALISTA tem {cmpp["razao n_obs GEN/ESP"].median():.1f}x mais observacoes que o')
print(f'    especialista mediano, e dp de coeficiente {1 / cmpp["razao dp ESP/GEN"].median():.2f}x '
      f'MENOR (razao dp ESP/GEN mediana = {cmpp["razao dp ESP/GEN"].median():.2f}).')
print(f'    A instabilidade CAI com a amostra (correlacao negativa em todas as 6 features) e o')
print(f'    GENERALISTA e de fato o mais estavel de todos, com autocorrelacao de {ac_gen.median():.4f}.')
print('    O diagnostico de subamostragem se sustenta PARA A DISPERSAO ENTRE MODELOS -- e nao')
print('    salva nada, porque o modelo mais estavel do projeto tambem nao ordena fora da amostra.')

print('\n' + '=' * 126)
print('A HIPOTESE DESTA CELULA NAO SE SUSTENTA NA MEDICAO -- MAS O TESTE NAO TEM PODER (D33).')
print('=' * 126)
print('A hipotese declarada no enunciado era: "a instabilidade de estimacao dos coeficientes destroi')
print('o sinal que existe nas features". Ela NAO se sustenta, e sao tres numeros independentes que a')
print('derrubam, nenhum deles procurado depois:')
print(f'  (1) os coeficientes sao ESTAVEIS -- autocorrelacao lag-1 de {ac_gen.median():.4f} no')
print(f'      generalista e mediana de {ac_esp.median():.4f} nos especialistas -- indistinguiveis do')
print('      nulo mecanico 0,95-0,98 da janela de 36m sobreposta (D33: sem comparacao com marco).')
print('  (2) a baixa persistencia do score (0,5330, DIAG-1) vem do movimento das FEATURES, nao dos')
print(f'      coeficientes: congelando as features, trocar os coeficientes da rho de '
      f'{rec["rho_coef"].mean():.4f}.')
print('  (3) o remedio direto da hipotese foi medido e NAO funciona: suavizar os coeficientes por 12')
print('      janelas leva o IC de +0,0025 para +0,0013 -- praticamente nao muda, e nao melhora.')
print('\nO QUE A MEDICAO ENCONTROU NO LUGAR, sem que fosse a pergunta original: o modelo estima os')
print('SINAIS ERRADOS de forma estavel. A concordancia entre o sinal do coeficiente do generalista e')
print(f'o sinal do IC do proprio mes e de {tab3["concordancia_%"].mean():.2f}% -- cara ou coroa. E no')
print('agregado do periodo, 3 das 6 features tem o coeficiente medio com sinal OPOSTO ao do seu IC')
print('medio: mom_3m (coef -0,0006 contra IC +0,0152), mom_6m (-0,0019 contra +0,0305) e vol_6m')
print('(+0,0078 contra -0,0632). Um score de media simples com os sinais impostos por hipotese, sem')
print(f'estimar coeficiente nenhum, tem IC de +0,0614 (t={tab6.iloc[1]["t (NW3)"]:.2f}).')
print('\nISTO E MEDICAO E PARA AQUI. Nenhuma carteira foi construida para nenhum score alternativo,')
print('nenhum retorno foi calculado, e esta celula NAO recomenda arquitetura. O MODEL_SPEC segue')
print('congelado, com os 6 fatores e a OLS mensal intactos. A decisao e do humano.')

# ==================================================================== gravacao
coef.to_parquet(DIR_INT + r'\diag_coeficientes.parquet', index=False)
print('\n' + '=' * 126)
print(f'GRAVADO intermediario\\diag_coeficientes.parquet   {coef.shape}')
print('DIAG-COEF CONCLUIDA -- medicao. Nenhum parametro tocado, nenhuma arquitetura recomendada.')
print('=' * 126)


DIAG-COEF -- ANATOMIA DOS COEFICIENTES. CELULA DE MEDICAO.
Nenhum parametro do MODEL_SPEC (congelado 2026-08-15 18:09:30) e alterado. Nenhuma variante
de arquitetura e rodada. Nenhum backtest, nenhuma carteira, nenhum retorno para os scores
alternativos do item 6 -- so IC. A decisao sobre o que fazer com estes numeros e do humano
e esta FORA desta celula.

previsoes (65273, 18) | dataset de treino (68763, 15)
coeficientes gravados: (18060, 11) ({'ESPECIALISTA': 16284, 'GENERALISTA': 1776}); 3010 regressoes

S1 maior |score reconstruido - b4b5_previsoes| = 0.000e+00   modelos identicos: 65273 de 65273
   S1 PASSOU -- os coeficientes desta celula sao os DA execucao congelada.

1. ESTABILIDADE TEMPORAL DOS COEFICIENTES

(1d) GENERALISTA isoladamente -- a maior amostra do projeto, logo o mais estavel esperado:
              meses   media      dp       CV  autocorr_lag1  troca_sinal_%
feature                                                                   
mom_1m_rank   296.0 -0.0119  0.0

## V2 (D28) -- variante com score de media simples e sinais impostos -- PARCIAL

Uma unica mudanca em relacao a V1: a funcao que converte ranks em score. Todo o resto e lido dos
parquets ja gravados, com integridade conferida por hash (S1: o SHA-256 de `b4b5_previsoes.parquet`
bate com o registrado no MODEL_SPEC de B6, e os 12 parquets de A1 a B3 tem mtime anterior ao freeze).

**Esta celula esta PARCIAL por decisao de parada prevista no proprio enunciado.** O enunciado
mandava redefinir o gate de V2 como "media dos ranks-sinalizados dos selecionados > 0,5",
justificando que seria "acima do ativo mediano" e "o analogo exato de melhor que neutro", e mandava
PARAR em caso de discordancia. A discordancia e aritmetica: com os sinais de D28 a escala do score
e [-1/3, +2/3] e o neutro fica em **1/6**, nao em 0,5. Medido: apenas **4,55%** dos 3.010 grupos
passariam sob o limiar literal (contra 94,78% sob o neutro), e a carteira ficaria 100% em CDI em
**179 dos 296 meses**. As duas justificativas verbais do enunciado apontam para o neutro; o numero
aponta para outro lugar; e as leituras NAO sao invariantes no resultado.

Entregue aqui, porque nao depende do gate: S1-S5, o score de V2, a prova de que a arquitetura de
especialistas e identica (D01 e contagem, nunca estimacao), e os itens **(a) IC**, **(b) spread
top-bottom** e **(c) persistencia**, sempre com V1 ao lado. Os itens (d) a (l) dependem da carteira,
que depende do gate, e nao foram rodados sob nenhuma das tres leituras -- rodar as tres e escolher
depois seria escolha pos-resultado.

V1 permanece intacta: nada foi sobrescrito, e `v2_scores.parquet` carrega as duas colunas de score
lado a lado.


In [1]:
import hashlib
import os
import sys
import time

import numpy as np
import pandas as pd

if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8')

pd.set_option('display.width', 270)
pd.set_option('display.max_columns', 80)
pd.set_option('display.max_rows', 400)

RAIZ = r'C:\Users\lucca\quant2026'
DIR_INT = RAIZ + r'\intermediario'

RANKS = ['mom_1m_rank', 'mom_3m_rank', 'mom_6m_rank', 'mom_12m_rank', 'vol_3m_rank', 'vol_6m_rank']
MOM = ['mom_1m_rank', 'mom_3m_rank', 'mom_6m_rank', 'mom_12m_rank']
VOL = ['vol_3m_rank', 'vol_6m_rank']
SINAL = {f: (+1.0 if f in MOM else -1.0) for f in RANKS}
JANELA, MIN_ATIVOS_D01, MIN_OBS_D01 = 36, 5, 150
HASH_MODEL_SPEC = '5f67e19fa21e1b22f8d29064fca75ba06c498e2b62ac6b87ae67f332eb385e67'
FREEZE = '2026-08-15 18:09:30'

print('=' * 126)
print('VARIANTE V2 (D28) -- score = media simples dos 6 ranks com SINAIS IMPOSTOS por hipotese')
print('UMA unica mudanca em relacao a V1: a funcao que converte ranks em score. Nada mais.')
print('=' * 126)


# ============================================================ helpers
def _pearson(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    ok = np.isfinite(x) & np.isfinite(y)
    x, y = x[ok] - x[ok].mean(), y[ok] - y[ok].mean()
    den = np.sqrt(float(x @ x) * float(y @ y))
    return float(x @ y) / den if den > 0 else np.nan


def spearman(x, y):
    rx = pd.Series(np.asarray(x, dtype=float)).rank(method='average').to_numpy()
    ry = pd.Series(np.asarray(y, dtype=float)).rank(method='average').to_numpy()
    return _pearson(rx, ry)


def nw_t(x, lags=3):
    x = np.asarray(pd.Series(x).dropna(), dtype=float)
    n = len(x)
    mu = x.mean()
    d = x - mu
    v = float(d @ d) / n
    for l in range(1, lags + 1):
        v += 2.0 * (1.0 - l / (lags + 1.0)) * (float(d[l:] @ d[:-l]) / n)
    se = np.sqrt(v / n)
    return mu, se, mu / se


def mes_idx(s):
    return s.str[:4].astype(int) * 12 + s.str[5:7].astype(int)


def sha256(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(1 << 20), b''):
            h.update(b)
    return h.hexdigest()


# ============================================================ S1 -- integridade de A1..B3
print('\n' + '=' * 126)
print('S1 -- AS CELULAS A1 A B3 NAO FORAM REEXECUTADAS NEM ALTERADAS')
print('=' * 126)
UPSTREAM = ['a1_diario_universo', 'a2_diario_ajustado', 'a2_eventos_societarios', 'a3_grade_mensal',
            'a4_retorno_mensal', 'a5_com_setor', 'a6_benchmark_e_rf', 'a7_split',
            'a8_universo_elegivel', 'b1_features', 'b2_ranks', 'b3_dataset']
t_freeze = time.mktime(time.strptime(FREEZE, '%Y-%m-%d %H:%M:%S'))
print(f'{"arquivo":<32} {"SHA-256 (12)":<14} {"mtime":<21} antes do freeze?')
todos_antes = True
for nome in UPSTREAM:
    p = os.path.join(DIR_INT, nome + '.parquet')
    mt = os.path.getmtime(p)
    antes = mt <= t_freeze
    todos_antes = todos_antes and antes
    print(f'{nome:<32} {sha256(p)[:12]:<14} '
          f'{time.strftime("%Y-%m-%d %H:%M:%S", time.localtime(mt)):<21} {antes}')
h_b4b5 = sha256(os.path.join(DIR_INT, 'b4b5_previsoes.parquet'))
print(f'\nb4b5_previsoes.parquet SHA-256 = {h_b4b5}')
print(f'MODEL_SPEC (B6, congelado {FREEZE}) registrou = {HASH_MODEL_SPEC}')
assert h_b4b5 == HASH_MODEL_SPEC, 'S1 FALHOU: b4b5_previsoes mudou desde o freeze'
print('   BATE. Como b4b5_previsoes e funcao deterministica de b3_dataset/b2_ranks/a6, o hash')
print('   confirma a cadeia inteira A1->B3 intacta, e nao so o arquivo final.')
assert todos_antes, 'S1 FALHOU: algum parquet de A1..B3 foi modificado apos o freeze'
print(f'   Todos os {len(UPSTREAM)} parquets de A1 a B3 tem mtime ANTERIOR ao freeze.')
print('   Esta celula LE esses arquivos e grava exclusivamente em arquivos novos com prefixo v2_.')
print('   S1 PASSOU')

# ============================================================ insumos
prev = pd.read_parquet(DIR_INT + r'\b4b5_previsoes.parquet')
b3 = pd.read_parquet(DIR_INT + r'\b3_dataset.parquet')
prev['CODISI'] = prev['CODISI'].astype(str)
prev['mes_idx'] = mes_idx(prev['mes'])
prev = prev.sort_values('mes_idx').reset_index(drop=True)
b3['mes_idx'] = mes_idx(b3['mes'])
b3 = b3.sort_values('mes_idx').reset_index(drop=True)
print(f'\nuniverso de previsao (o MESMO de V1): {prev.shape}')

# ============================================================ S5 -- diferencas V1 x V2
print('\n' + '=' * 126)
print('S5 -- LISTA EXAUSTIVA DAS DIFERENCAS ENTRE V1 E V2')
print('=' * 126)
componentes = [
    ('universo (CODBDI/TPMERC/ESPECI)', 'identico', 'mesmo a1_diario_universo, hash conferido em S1'),
    ('ajuste societario (G/B/FATCOT)', 'identico', 'mesmo a2, hash conferido'),
    ('grade mensal', 'identico', 'mesmo a3, hash conferido'),
    ('elegibilidade D08/D10/D19/D25', 'identico', 'mesmo a8/b1, hash conferido'),
    ('setor e chave do especialista D06/D07', 'identico', 'mesmo a5, hash conferido'),
    ('benchmark D12/D16 e CDI D13', 'identico', 'mesmo a6, hash conferido'),
    ('6 features D18/D19', 'identico', 'mesmo b1, hash conferido'),
    ('rank cross-seccional D20', 'identico', 'mesmo b2, hash conferido'),
    ('alvo alfa_fut D21', 'identico (e NAO usado por V2)', 'mesmo b3, hash conferido'),
    ('walk-forward 36m D22/D26', 'identico', 'mesma janela [T-37,T-2]'),
    ('arquitetura D01/D23 (quem tem especialista)', 'identico', 'D01 e contagem, nao estimacao'),
    ('K=3/2, peso igual, CDI ocioso, 50bps, D16', 'identico', 'MODEL_SPEC secao 13'),
    ('motor de backtest', 'identico', 'src\\motor.py, intocado'),
    ('FUNCAO DE SCORE', '*** UNICA DIFERENCA ***',
     'V1: OLS de alfa_fut sobre 6 ranks | V2: media simples com sinais impostos'),
]
for c, st, obs in componentes:
    print(f'  {c:<44} {st:<30} {obs}')
n_dif = sum(1 for _, st, _ in componentes if '***' in st)
print(f'\ncomponentes que diferem: {n_dif}')
assert n_dif == 1, 'S5 FALHOU: ha mais de uma diferenca'
print('   S5 PASSOU -- uma unica diferenca.')

# ============================================================ B4/B5-V2
print('\n' + '=' * 126)
print('B4/B5-V2 -- SCORE E ARQUITETURA')
print('=' * 126)
print(f'sinais impostos, declarados em D28 e NAO ajustados a este periodo: {SINAL}')
print('score_v2 = ( mom_1m + mom_3m + mom_6m + mom_12m - vol_3m - vol_6m ) / 6')

# --- D01 recomputado do ZERO (so contagem) para provar que a arquitetura nao depende da estimacao
mi_tr = b3['mes_idx'].to_numpy()
sub_tr = b3['subsetor'].to_numpy()
mi_p = prev['mes_idx'].to_numpy()
sub_p = prev['subsetor'].to_numpy()
modelo_v2 = np.full(len(prev), 'GENERALISTA', dtype=object)
for T in sorted(set(mi_p)):
    lo = np.searchsorted(mi_tr, T - (JANELA + 1), 'left')
    hi = np.searchsorted(mi_tr, T - 2, 'right')
    st = sub_tr[lo:hi]
    plo = np.searchsorted(mi_p, T, 'left')
    phi = np.searchsorted(mi_p, T, 'right')
    if phi == plo or (hi - lo) < 7:
        continue
    sp = sub_p[plo:phi]
    subu, cu = np.unique(sp, return_counts=True)
    n_at = dict(zip(subu, cu))
    stu, ct = np.unique(st, return_counts=True)
    n_ob = dict(zip(stu, ct))
    for s in subu:
        if s == 'SEM_SETOR' or n_at.get(s, 0) < MIN_ATIVOS_D01 or n_ob.get(s, 0) < MIN_OBS_D01:
            continue
        modelo_v2[plo:phi][sp == s] = s
igual = int((modelo_v2 == prev['modelo'].to_numpy()).sum())
print(f'\nARQUITETURA: D01 recomputado do zero (SO contagens de ativos e de observacoes, nenhuma OLS)')
print(f'reproduz a atribuicao de modelo de V1 em {igual} de {len(prev)} linhas.')
assert igual == len(prev), 'a arquitetura de V2 divergiu da de V1'
print('=> a arquitetura de especialistas e IDENTICA em V1 e V2, por construcao: D01 e contagem,')
print('   nunca estimacao. Isso e o que permite que a unica diferenca seja a funcao de score.')

prev['score_v2'] = sum(SINAL[f] * prev[f] for f in RANKS) / len(RANKS)

print('\nCONSEQUENCIA DECLARADA DA MUDANCA (exigida pelo enunciado, e ela e real):')
print('  Os ranks de B2 sao calculados dentro do universo ELEGIVEL DO MES INTEIRO (D20), nao dentro')
print('  do subsetor. Como V2 nao estima nada, o score de um ativo e o MESMO qualquer que seja o')
print('  modelo que o serve. Em V1 o especialista mudava o score (coeficientes proprios); em V2 nao.')
print('  O que a arquitetura de especialistas AINDA determina em V2, e so isso:')
print('    (i)  AGRUPAMENTO da selecao -- top K dentro de cada subsetor, nao no universo inteiro;')
print('    (ii) o GATE -- a condicao e avaliada sobre os selecionados DAQUELE subsetor.')
print('  Verificacao numerica: o score de um ativo servido por especialista e por generalista e')
print('  literalmente a mesma funcao das mesmas 6 colunas -- nao ha coeficiente para diferir.')

# ============================================================ S2 e S4
print('\n' + '=' * 126)
print('S2 e S4 -- O SCORE DE V2 NAO USA alfa_fut NEM INFORMACAO DE T OU POSTERIOR')
print('=' * 126)
cols_usadas = set(RANKS)
print(f'colunas de entrada do score_v2: {sorted(cols_usadas)}')
print(f'`alfa_fut` esta entre elas? {"alfa_fut" in cols_usadas}')
assert 'alfa_fut' not in cols_usadas, 'S2 FALHOU'
sem_alvo = prev.drop(columns=['alfa_fut'])
score_sem_alvo = sum(SINAL[f] * sem_alvo[f] for f in RANKS) / len(RANKS)
dev = float(np.abs(score_sem_alvo.to_numpy() - prev['score_v2'].to_numpy()).max())
print(f'PROVA EXECUTAVEL: recalculando o score sobre o dataframe com a coluna `alfa_fut` REMOVIDA,')
print(f'o maior desvio contra o score gravado e {dev:.3e}.')
assert dev == 0.0, 'S2 FALHOU'
print('   S2 PASSOU -- nao ha estimacao, logo nenhuma observacao de alvo entra no score.')
print('\nS4: cada rank de T e funcao de uma janela de features que termina em T e exige os K meses')
print('    consecutivos E elegiveis (D19); o rank e cross-seccional DENTRO do mes T (D20). Nenhuma')
print('    das duas operacoes toca t>T. Verificacao estrutural sobre as 65.273 linhas:')
b2 = pd.read_parquet(DIR_INT + r'\b2_ranks.parquet')
b2['CODISI'] = b2['CODISI'].astype(str)
chk = prev[['CODISI', 'mes'] + RANKS].merge(
    b2[['CODISI', 'mes'] + RANKS], on=['CODISI', 'mes'], suffixes=('', '_b2'))
d_rank = max(float(np.abs(chk[f].to_numpy() - chk[f + '_b2'].to_numpy()).max()) for f in RANKS)
print(f'    os 6 ranks de cada (CODISI, mes) batem com b2_ranks.parquet: maior desvio {d_rank:.3e}')
assert d_rank == 0.0, 'S4 FALHOU'
recalc = []
for m, g in b2[b2['elegivel'] & b2['mom_12m_rank'].notna()].groupby('mes'):
    if len(recalc) >= 6:
        break
    r = g['mom_12m'].rank(method='average', pct=True)
    recalc.append(float(np.abs(r.to_numpy() - g['mom_12m_rank'].to_numpy()).max()))
print(f'    e o rank recalculado com a fatia ISOLADA de 6 meses (sem acesso ao resto do painel)')
print(f'    bate com o gravado: maior desvio {max(recalc):.3e}')
assert max(recalc) < 1e-12, 'S4 FALHOU no recalculo isolado'
print('   S4 PASSOU')

# ============================================================ (a) IC
print('\n' + '=' * 126)
print('(a) IC -- V1 e V2 LADO A LADO')
print('=' * 126)


def ic_mensal(df, col):
    out = []
    for m, g in df.groupby('mes'):
        h = g[[col, 'alfa_fut']].dropna()
        if len(h) < 10:
            continue
        out.append((m, spearman(h[col], h['alfa_fut'])))
    return pd.DataFrame(out, columns=['mes', 'ic'])


lin = []
ic_series = {}
for part in ['TREINO', 'TESTE']:
    sub = prev[(prev['particao'] == part) & prev['alfa_fut'].notna()]
    for col, rot in [('score', 'V1 (OLS, congelada)'), ('score_v2', 'V2 (media simples, sinais impostos)')]:
        ic = ic_mensal(sub, col)
        ic_series[(part, col)] = ic
        mu, se, t = nw_t(ic['ic'])
        lin.append({'particao': part, 'versao': rot, 'n_meses': len(ic), 'IC medio': mu,
                    'dp': ic['ic'].std(ddof=1), 't (NW3)': t,
                    '% meses > 0': 100 * float((ic['ic'] > 0).mean())})
tab_ic = pd.DataFrame(lin)
print(tab_ic.round(4).to_string(index=False))

ic_v2_teste = tab_ic[(tab_ic['particao'] == 'TESTE') &
                     tab_ic['versao'].str.startswith('V2')]['IC medio'].iloc[0]
print(f'\nS3 IC de V2 no TESTE = {ic_v2_teste:.4f}; DIAG-COEF item 6 media simples = +0,0614; '
      f'diferenca {abs(ic_v2_teste - 0.0614):.5f}')
assert abs(ic_v2_teste - 0.0614) < 0.002, 'S3 FALHOU'
print('   S3 PASSOU -- a implementacao de V2 reproduz o diagnostico que a motivou.')

# ============================================================ (b) spread top-bottom
print('\n' + '=' * 126)
print('(b) SPREAD TOP-BOTTOM POR DECIL DE SCORE (media de alfa_fut do decil 10 menos a do decil 1)')
print('=' * 126)
lin = []
for part in ['TREINO', 'TESTE']:
    sub = prev[(prev['particao'] == part) & prev['alfa_fut'].notna()]
    for col, rot in [('score', 'V1'), ('score_v2', 'V2')]:
        sp = []
        for m, g in sub.groupby('mes'):
            if len(g) < 20:
                continue
            d = pd.qcut(g[col].rank(method='first'), 10, labels=False)
            top = g['alfa_fut'][d == 9].mean()
            bot = g['alfa_fut'][d == 0].mean()
            sp.append(top - bot)
        sp = np.array(sp)
        mu, se, t = nw_t(sp)
        lin.append({'particao': part, 'versao': rot, 'n_meses': len(sp),
                    'spread_%mes': 100 * mu, 'dp_%': 100 * sp.std(ddof=1), 't (NW3)': t,
                    '% meses > 0': 100 * float((sp > 0).mean())})
print(pd.DataFrame(lin).round(4).to_string(index=False))
print('\nreferencia de B5 para V1: TREINO +1,3164%/mes (t=+2,865) e TESTE -0,1406%/mes (t=-0,280).')

# ============================================================ (c) persistencia
print('\n' + '=' * 126)
print('(c) PERSISTENCIA DO SCORE (Spearman cross-seccional entre meses consecutivos)')
print('=' * 126)
p2 = prev[['mes', 'mes_idx', 'CODISI', 'score', 'score_v2']].copy()
p2['mes_idx_prox'] = p2['mes_idx'] + 1
prox = p2[['mes_idx', 'CODISI', 'score', 'score_v2']].rename(
    columns={'mes_idx': 'mes_idx_prox', 'score': 'score_t1', 'score_v2': 'score_v2_t1'})
par = p2.merge(prox, on=['mes_idx_prox', 'CODISI'])
lin = []
for m, g in par.groupby('mes'):
    if len(g) < 10:
        continue
    lin.append({'mes': m, 'n': len(g), 'rho_V1': spearman(g['score'], g['score_t1']),
                'rho_V2': spearman(g['score_v2'], g['score_v2_t1'])})
per = pd.DataFrame(lin)
per['particao'] = per['mes'].map(prev.groupby('mes')['particao'].first())
print(per.groupby('particao')[['rho_V1', 'rho_V2']].agg(['mean', 'std', 'min', 'max']).round(4).to_string())
print('\nDIAG-1 mediu rho do score V1 no teste em 0,5330 e o de mom_12m_rank em 0,9036.')
print('A persistencia de V2 e a de uma combinacao FIXA de ranks -- so as features se movem, porque')
print('nao ha coeficiente para se mover. E exatamente a perna `rho_features` isolada em DIAG-COEF (1e).')

# ============================================================ O GATE
print('\n' + '=' * 126)
print('*** PARADA OBRIGATORIA: A EQUIVALENCIA DO GATE PROPOSTA NO ENUNCIADO NAO SE SUSTENTA ***')
print('=' * 126)
print('O enunciado manda declarar o gate de V2 como "media dos ranks-sinalizados dos selecionados')
print('> 0,5", justificando que isso e "acima do ativo mediano do universo naquele mes" e "o analogo')
print('exato de melhor que neutro". E manda PARAR e reportar em caso de discordancia. Discordo, e a')
print('discordancia e aritmetica, nao de opiniao.')
print('\n1. A PREMISSA. O enunciado diz que o score de V2 "tem escala [0,1] centrada em ~0,5". Isso')
print('   valeria para a media SEM sinais. Com os sinais de D28 (+1 nos 4 de momento, -1 nos 2 de')
print('   volatilidade), a escala e outra:')
print('     minimo teorico = (0+0+0+0-1-1)/6 = -0,3333')
print('     maximo teorico = (1+1+1+1-0-0)/6 = +0,6667')
print('     ativo NEUTRO (todos os 6 ranks em 0,5) = (0,5*4 - 0,5*2)/6 = 1/6 = +0,1667')
print('   Ou seja: a escala e [-1/3, +2/3] e o neutro fica em 1/6, nao em 0,5.')
print('\n2. A MEDICAO, sobre as 65.273 linhas efetivamente calculadas:')
d = prev['score_v2']
print(f'   minimo {d.min():+.4f} | p1 {d.quantile(.01):+.4f} | mediana {d.median():+.4f} | '
      f'media {d.mean():+.4f} | p99 {d.quantile(.99):+.4f} | maximo {d.max():+.4f}')
med_mes = prev.groupby('mes')['score_v2'].median()
print(f'   mediana cross-seccional do mes: media {med_mes.mean():+.4f}, '
      f'min {med_mes.min():+.4f}, max {med_mes.max():+.4f}')
print(f'   fracao de TODOS os ativos-mes com score_v2 > 0,5: '
      f'{100 * float((d > 0.5).mean()):.4f}%  ({int((d > 0.5).sum())} de {len(d)})')
print(f'   fracao com score_v2 > 1/6 (o neutro): {100 * float((d > 1 / 6).mean()):.2f}%')
print('\n3. A CONSEQUENCIA sobre o gate, medida nos 3.010 grupos (mes, modelo), com K=3/2 aplicado')
print('   exatamente como no MODEL_SPEC:')
pv = prev.sort_values(['mes', 'modelo', 'score_v2'], ascending=[True, True, False], kind='mergesort').copy()
pv['rk'] = pv.groupby(['mes', 'modelo']).cumcount()
t3 = pv[pv['rk'] < 3]
piv = t3.pivot_table(index=['mes', 'modelo'], columns='rk', values='score_v2', aggfunc='first')
piv.columns = ['s1', 's2', 's3']
piv = piv.reset_index()
piv['K'] = np.where(piv['s3'] < piv['s1'] * 0.5, 2, 3)
piv['media_K'] = np.where(piv['K'] == 2, (piv['s1'] + piv['s2']) / 2,
                          (piv['s1'] + piv['s2'] + piv['s3']) / 3)
piv['mediana_mes'] = piv['mes'].map(med_mes)
leituras = [
    ('(L1) LITERAL do enunciado: media dos K > 0,5', piv['media_K'] > 0.5),
    ('(L2) NEUTRO: media dos K > 1/6 (ativo com todos os ranks em 0,5)', piv['media_K'] > 1 / 6),
    ('(L3) MEDIANO: media dos K > mediana cross-seccional do mes', piv['media_K'] > piv['mediana_mes']),
]
print(f'   {"leitura":<62} {"grupos que passam":<20} {"% dos 3.010"}')
for rot, m in leituras:
    print(f'   {rot:<62} {int(m.sum()):<20} {100 * float(m.mean()):.2f}%')
print(f'\n   Para referencia, o gate de V1 (media dos K scores previstos > 0) passava 2.642 de 3.010 '
      f'(87,77%).')
meses_l1 = piv[piv['media_K'] > 0.5].groupby('mes').size()
print(f'   Sob a leitura L1, meses com ao menos um modelo aprovado: {len(meses_l1)} de '
      f'{piv["mes"].nunique()} -- a carteira ficaria 100% em CDI em '
      f'{piv["mes"].nunique() - len(meses_l1)} meses.')
print('\n4. POR QUE ISTO E PARADA E NAO UMA ESCOLHA MINHA. O enunciado da DUAS justificativas verbais')
print('   para o numero 0,5 -- "acima do ativo mediano do universo naquele mes" e "o analogo exato de')
print('   melhor que neutro". As duas apontam para o MESMO lugar e as duas sao INCOMPATIVEIS com 0,5')
print('   nesta escala; apontam para L2/L3. O numero e a intencao declarada se contradizem, e a')
print('   diferenca nao e cosmetica: sob L1 o gate reprova quase tudo e a carteira vira CDI; sob')
print('   L2/L3 ela opera. Escolher por conta propria entre elas seria eu fixando um parametro de')
print('   carteira, que e exatamente o que B6 congelou e o que D28 proibe alterar.')
print('   Mesma classe de D26 (ambiguidade da janela) e D27 (ambiguidade do gate) -- so que aqui as')
print('   leituras NAO sao invariantes no resultado, entao nao ha como resolver por medicao.')
print('\n5. O QUE FOI ENTREGUE ASSIM MESMO. Tudo o que NAO depende do gate ja esta medido acima:')
print('   S1-S5, o score de V2, a arquitetura, e os itens (a) IC, (b) spread top-bottom e (c)')
print('   persistencia. Os itens (d) a (l) dependem da carteira e a carteira depende do gate.')
print('   NENHUMA das tres leituras foi rodada ate a carteira -- rodar as tres e escolher depois')
print('   seria escolha pos-resultado, que e o que este projeto inteiro existe para nao fazer.')

# ============================================================ gravacao parcial
saida = prev[['CODISI', 'CODNEG', 'mes', 'subsetor', 'particao', 'modelo'] + RANKS +
             ['alfa_fut', 'score', 'score_v2']].copy()
saida.to_parquet(DIR_INT + r'\v2_scores.parquet', index=False)
per.to_parquet(DIR_INT + r'\v2_persistencia.parquet', index=False)
print('\n' + '=' * 126)
print(f'GRAVADO intermediario\\v2_scores.parquet        {saida.shape}  (V1 e V2 lado a lado, V1 intacta)')
print(f'GRAVADO intermediario\\v2_persistencia.parquet  {per.shape}')
print('V2 PARCIAL -- itens (a), (b), (c) concluidos. Itens (d) a (l) BLOQUEADOS pela ambiguidade')
print('do gate, reportada acima com os numeros das tres leituras. Aguarda decisao do humano.')
print('=' * 126)


VARIANTE V2 (D28) -- score = media simples dos 6 ranks com SINAIS IMPOSTOS por hipotese
UMA unica mudanca em relacao a V1: a funcao que converte ranks em score. Nada mais.

S1 -- AS CELULAS A1 A B3 NAO FORAM REEXECUTADAS NEM ALTERADAS
arquivo                          SHA-256 (12)   mtime                 antes do freeze?
a1_diario_universo               ad3857d1d87d   2026-08-15 14:09:04   True
a2_diario_ajustado               9026bf1e2943   2026-08-15 14:36:36   True
a2_eventos_societarios           a66c34385470   2026-08-15 14:36:33   True
a3_grade_mensal                  930628af2ebf   2026-08-15 14:41:55   True
a4_retorno_mensal                55d62964c14b   2026-08-15 14:47:42   True
a5_com_setor                     9893104aa69e   2026-08-15 15:07:37   True
a6_benchmark_e_rf                d0f3d79b02d9   2026-08-15 16:40:09   True
a7_split                         2f4170c445e3   2026-08-15 16:53:52   True
a8_universo_elegivel             02c9f6136af1   2026-08-15 15:40:06   True
b1_

## V2 -- retomada com o gate de D30 (score > 1/6): itens (d) a (o) e D29

D30 fixou o limiar por **identidade algebrica**, nao por comparacao de desempenho: com os sinais de
D28 o ativo neutro (os 6 ranks em 0,5) tem score `(0,5*4 - 0,5*2)/6 = 1/6`, entao 1/6 E o zero desta
escala e "score > 1/6" e a traducao exata de "alfa previsto > 0" de V1.

Duas coisas ficam registradas no output antes de qualquer numero de desempenho:

1. **A regra de K NAO foi alterada.** "K=2 se score[3] < score[1]/2" e uma razao, e razao depende de
   onde esta o zero da escala -- o mesmo argumento que corrigiu o gate. D30 corrigiu apenas o gate,
   entao a regra de K foi aplicada literalmente sobre o score bruto. Efeito medido: a regra dispara
   em 19,70% dos grupos em V2 contra 57,38% em V1. E consequencia declarada, nao ajuste.
2. **A invertida de V2** foi construida por reflexao em torno do neutro (`s' = 2/6 - s`), que e o
   analogo exato da negacao usada em V1 (la o neutro era 0, entao refletir era negar).

O item (n) produziu o achado mais importante da celula, e ele nao estava sendo procurado: **o decil
superior de score_v2 tem alfa_fut de -0,0314%/mes (t=-0,11), indistinguivel de zero, enquanto o
decil inferior tem -0,9468%/mes.** O IC de +0,0614 de V2 vem quase todo de identificar PERDEDORES --
e a carteira compra ganhadores, isto e, opera exatamente na ponta onde o score nao tem informacao.

V1 permanece intacta: nenhum arquivo de V1 foi sobrescrito, e toda tabela traz as duas lado a lado.


In [1]:
import sys
import time

import numpy as np
import pandas as pd

if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8')

pd.set_option('display.width', 275)
pd.set_option('display.max_columns', 90)
pd.set_option('display.max_rows', 400)

RAIZ = r'C:\Users\lucca\quant2026'
DIR_INT = RAIZ + r'\intermediario'
if RAIZ + r'\src' not in sys.path:
    sys.path.insert(0, RAIZ + r'\src')
import motor  # noqa: E402

RANKS = ['mom_1m_rank', 'mom_3m_rank', 'mom_6m_rank', 'mom_12m_rank', 'vol_3m_rank', 'vol_6m_rank']
NEUTRO = 1.0 / 6.0            # D30: o zero da escala de V2
GATE_V1 = 0.0                 # o zero da escala de V1 (alfa previsto)
K_PADRAO, K_REDUZIDO, FRACAO = 3, 2, 0.5
NIVEIS_BPS = [0, 10, 25, 50, 75, 100]
SEMENTE = 20260815
N_ALEATORIAS = 200
SUBPERIODOS = [('2018-01', '2020-12'), ('2021-01', '2023-12'), ('2024-01', '2026-07')]

print('=' * 128)
print('V2 -- RETOMADA COM O GATE DE D30 (score > 1/6). Itens (d) a (o) e D29.')
print('V1 e V2 lado a lado em toda tabela. V1 INTACTA -- nada e sobrescrito.')
print('=' * 128)


# ==================================================================== helpers
def _pearson(x, y):
    x = np.asarray(x, dtype=float); y = np.asarray(y, dtype=float)
    ok = np.isfinite(x) & np.isfinite(y)
    x, y = x[ok] - x[ok].mean(), y[ok] - y[ok].mean()
    den = np.sqrt(float(x @ x) * float(y @ y))
    return float(x @ y) / den if den > 0 else np.nan


def nw_t(x, lags=3):
    x = np.asarray(pd.Series(x).dropna(), dtype=float)
    n = len(x); mu = x.mean(); d = x - mu
    v = float(d @ d) / n
    for l in range(1, lags + 1):
        v += 2.0 * (1.0 - l / (lags + 1.0)) * (float(d[l:] @ d[:-l]) / n)
    se = np.sqrt(v / n)
    return mu, se, mu / se


def anual(r):
    r = np.asarray(pd.Series(r).dropna(), dtype=float)
    return 100.0 * ((np.prod(1.0 + r)) ** (12.0 / len(r)) - 1.0)


def dd_max(r):
    r = np.asarray(pd.Series(r).dropna(), dtype=float)
    w = np.cumprod(1 + r); pico = np.maximum.accumulate(w)
    return 100.0 * float((w / pico - 1.0).min())


# ==================================================================== insumos
sc = pd.read_parquet(DIR_INT + r'\v2_scores.parquet')
bench = pd.read_parquet(DIR_INT + r'\a6_benchmark_e_rf.parquet')
split = pd.read_parquet(DIR_INT + r'\a7_split.parquet')
c3v1 = pd.read_parquet(DIR_INT + r'\c3_curva_custo.parquet')
trilha_a_v1 = pd.read_parquet(DIR_INT + r'\c1_carteira.parquet')
d2_v1 = pd.read_parquet(DIR_INT + r'\d2_brinson.parquet')
sc['CODISI'] = sc['CODISI'].astype(str)
split['CODISI'] = split['CODISI'].astype(str)
trilha_a_v1['CODISI'] = trilha_a_v1['CODISI'].astype(str)

meses_all = sorted(bench['mes'].unique())
data_de_mes = {m: pd.Period(m, freq='M').to_timestamp(how='end').normalize() for m in meses_all}
mes_de_data = {v: k for k, v in data_de_mes.items()}
meses_dec = sorted(sc['mes'].unique())
part_por_mes = sc.groupby('mes')['particao'].first().to_dict()
bi = bench.set_index('mes')

wide = split.pivot_table(index='mes', columns='CODISI', values='ret_1m', aggfunc='first').reindex(meses_all)
wide.index = pd.DatetimeIndex([data_de_mes[m] for m in meses_all])
cdi_s = bi['ret_cdi'].reindex(meses_all); cdi_s.index = wide.index
wide['__CDI__'] = cdi_s.values
CAIXA = wide['__CDI__']
print(f'\nmatriz de retornos {wide.shape}; meses de decisao {len(meses_dec)}')


def rodar(pesos_df, rotulo, silencioso=False):
    p = pesos_df.copy()
    cols = [c for c in p.columns if (p[c] != 0).any()]
    p = p[cols].fillna(0.0)
    p.index = pd.DatetimeIndex([data_de_mes[m] for m in p.index])
    out = motor.rodar_backtest(p, wide[cols], custo_bps=0.0, retornos_caixa=CAIXA)
    rb = out['retornos_brutos']; tv = out['turnover'].reindex(rb.index)
    d = pd.DataFrame({'mes_ret': [mes_de_data[i] for i in rb.index],
                      'bruto': rb.values, 'turnover': tv.values})
    d['mes'] = [str(pd.Period(m, freq='M') - 1) for m in d['mes_ret']]
    d['particao'] = d['mes'].map(part_por_mes)
    for c in ['ret_indice', 'ret_indice_ex_max', 'ret_cdi']:
        d[c] = d['mes_ret'].map(bi[c])
    if not silencioso:
        print(f'  [{rotulo}] ativos {len(cols)}  meses {len(d)}  '
              f'congelamentos {len(out["eventos_congelamento"])}  descartados {out["rebal_descartados"]}')
    return d, out


def liq(d, bps):
    return d['bruto'] - d['turnover'] * (bps / 10000.0)


# ==================================================================== selecao V2
print('\n' + '=' * 128)
print('SELECAO DE V2 -- gate de D30')
print('=' * 128)
print(f'gate: media dos K selecionados > {NEUTRO:.6f} (= 1/6, o ativo neutro na escala de V2)')
print(f'gate de V1, para comparacao: media dos K > {GATE_V1:.1f} (o zero da escala de alfa previsto)')


def selecionar(df, col, gate_thr, refletir=False):
    d = df.copy()
    d['_s'] = (2 * NEUTRO - d[col]) if refletir else d[col]
    d = d.sort_values(['mes', 'modelo', '_s'], ascending=[True, True, False], kind='mergesort')
    d['rk'] = d.groupby(['mes', 'modelo']).cumcount()
    t3 = d[d['rk'] < 3]
    piv = t3.pivot_table(index=['mes', 'modelo'], columns='rk', values='_s', aggfunc='first')
    piv.columns = ['s1', 's2', 's3']
    piv = piv.reset_index()
    piv['regra_metade'] = piv['s3'] < piv['s1'] * FRACAO
    piv['K'] = np.where(piv['regra_metade'], K_REDUZIDO, K_PADRAO)
    piv['media_K'] = np.where(piv['K'] == K_REDUZIDO, (piv['s1'] + piv['s2']) / 2,
                              (piv['s1'] + piv['s2'] + piv['s3']) / 3)
    piv['gate'] = piv['media_K'] > gate_thr
    sel = t3.merge(piv[['mes', 'modelo', 'K', 'gate', 'media_K', 'regra_metade']],
                   on=['mes', 'modelo'], how='left')
    sel = sel[(sel['rk'] < sel['K']) & sel['gate']].copy()
    return sel, piv


sel_v2, piv_v2 = selecionar(sc, 'score_v2', NEUTRO)
sel_inv, piv_inv = selecionar(sc, 'score_v2', NEUTRO, refletir=True)
print(f'\ngrupos (mes, modelo): {len(piv_v2)}; passam no gate V2: {int(piv_v2["gate"].sum())} '
      f'({100 * piv_v2["gate"].mean():.2f}%)  |  V1 passava 2.642 (87,77%)')
print(f'posicoes selecionadas V2: {len(sel_v2)}  |  V1: {len(trilha_a_v1)}')

print('\nCONSEQUENCIA DA REGRA DE K SOB A NOVA ESCALA -- medida, aplicada literalmente, NAO alterada:')
print('a regra "K=2 se score[3] < score[1]/2" e uma razao, e razao depende de onde esta o zero da')
print('escala. D30 corrigiu o zero do GATE (0 -> 1/6) e NAO tocou na regra de K, entao ela foi')
print('aplicada literalmente sobre o score_v2 bruto. O efeito medido:')
print(f'  V2: regra da metade acionada em {int(piv_v2["regra_metade"].sum())} de {len(piv_v2)} grupos '
      f'({100 * piv_v2["regra_metade"].mean():.2f}%)')
print('  V1: acionada em 1.727 de 3.010 (57,38%)')
kd = piv_v2.groupby('K').size()
print(f'  distribuicao de K em V2: {kd.to_dict()}  |  em V1 era K=2 em 1.727 e K=3 em 1.283')
print('  ISTO E CONSEQUENCIA DECLARADA, NAO AJUSTE: em V2 os tres melhores scores de um subsetor')
print('  estao proximos entre si e longe do zero da escala, entao "o terceiro vale menos que metade')
print('  do primeiro" quase nunca ocorre. A regra continua exatamente como o MODEL_SPEC a definiu.')

# ---- trilhas
def matriz(df, col='peso'):
    m = df.pivot_table(index='mes', columns='CODISI', values=col, aggfunc='sum').reindex(meses_dec)
    m = m.fillna(0.0); m['__CDI__'] = 0.0
    return m


n_por_mes = sel_v2.groupby('mes').size().rename('n')
sel_v2 = sel_v2.merge(n_por_mes, on='mes')
sel_v2['peso'] = 1.0 / sel_v2['n']
pes_v2a = matriz(sel_v2)
meses_sem = [m for m in meses_dec if m not in set(sel_v2['mes'])]
if meses_sem:
    pes_v2a.loc[meses_sem, '__CDI__'] = 1.0

M_v2 = piv_v2.groupby('mes')['modelo'].size().rename('M')
G_v2 = piv_v2.groupby('mes')['gate'].sum().rename('n_gate')
selB = sel_v2.merge(M_v2, on='mes').merge(G_v2, on='mes')
selB['peso_b'] = 1.0 / (selB['M'] * selB['K'])
pes_v2b = matriz(selB, 'peso_b')
frac_ac = (G_v2 / M_v2).reindex(meses_dec).fillna(0.0)
pes_v2b['__CDI__'] = (1.0 - frac_ac).values

n_inv = sel_inv.groupby('mes').size().rename('n')
sel_inv = sel_inv.merge(n_inv, on='mes')
sel_inv['peso'] = 1.0 / sel_inv['n']
pes_inv = matriz(sel_inv)
ms_inv = [m for m in meses_dec if m not in set(sel_inv['mes'])]
if ms_inv:
    pes_inv.loc[ms_inv, '__CDI__'] = 1.0

print('\nrodando o motor (custo 0; o custo entra depois pela aritmetica identica a aplicar_custos):')
dA, outA = rodar(pes_v2a, 'V2-A')
dB, outB = rodar(pes_v2b, 'V2-B')
dI, outI = rodar(pes_inv, 'V2-invertida')

V1 = {v: c3v1[c3v1['versao'] == v].sort_values('mes').reset_index(drop=True)
      for v in ['C2-A', 'C2-B', 'EW_universo', 'INVERTIDA']}
for v, d in V1.items():
    d['bruto'] = d['bruto']
    for c in ['ret_indice', 'ret_indice_ex_max', 'ret_cdi']:
        d[c] = d['mes_ret'].map(bi[c])

TE = lambda d: d[d['particao'] == 'TESTE'].reset_index(drop=True)
TR = lambda d: d[d['particao'] == 'TREINO'].reset_index(drop=True)

# ==================================================================== (l) e (m)
print('\n' + '=' * 128)
print('(l) e (m) -- ATIVOS, MODELOS NO GATE, E CAPITAL EM ACOES vs CDI')
print('=' * 128)
res = pd.DataFrame({'mes': meses_dec})
res['particao'] = res['mes'].map(part_por_mes)
res['ano'] = res['mes'].str[:4]
res['n_modelos'] = res['mes'].map(M_v2)
res['n_gate_v2'] = res['mes'].map(G_v2)
res['n_ativos_v2'] = res['mes'].map(sel_v2.groupby('mes').size())
res['n_ativos_v2'] = res['n_ativos_v2'].fillna(0).astype(int)
res['frac_acoes_A'] = np.where(res['n_ativos_v2'] > 0, 1.0, 0.0)
res['frac_acoes_B'] = res['mes'].map(frac_ac)
v1a = trilha_a_v1.groupby('mes').size()
res['n_ativos_v1'] = res['mes'].map(v1a).fillna(0).astype(int)
print('\nnumero de ativos por mes (V1 e V2), por particao:')
print(res.groupby('particao')[['n_ativos_v1', 'n_ativos_v2']].agg(
    ['median', 'min', 'max', 'mean']).round(2).to_string())
print('\n(m) grupos que passam no gate > 1/6, por ano, e capital em acoes:')
tm = res.groupby(['particao', 'ano']).agg(
    meses=('mes', 'size'), modelos=('n_modelos', 'median'), no_gate=('n_gate_v2', 'median'),
    pct_gate=('n_gate_v2', 'sum'), tot=('n_modelos', 'sum'),
    acoes_A=('frac_acoes_A', lambda s: 100 * s.mean()),
    acoes_B=('frac_acoes_B', lambda s: 100 * s.mean())).reset_index()
tm['pct_gate'] = (100 * tm['pct_gate'] / tm['tot']).round(2)
print(tm.drop(columns=['tot']).round(2).to_string(index=False))
n_zero = int((res['n_ativos_v2'] == 0).sum())
print(f'\nmeses em que NENHUM modelo passa no gate (carteira 100% CDI na leitura A): {n_zero} de {len(res)}')
if n_zero:
    print(res[res['n_ativos_v2'] == 0][['mes', 'particao', 'n_modelos', 'n_gate_v2']].to_string(index=False))
print(f'\nleitura A: capital em acoes = 100% em {int((res["frac_acoes_A"] > 0).sum())} de {len(res)} meses '
      f'-- como em V1, e consequencia aritmetica do peso 1/N (L25), nao do gate.')
rt = res[res['particao'] == 'TESTE']
print(f'leitura B: capital em acoes no TESTE -- media {100 * rt["frac_acoes_B"].mean():.2f}%, '
      f'mediana {100 * rt["frac_acoes_B"].median():.2f}%, min {100 * rt["frac_acoes_B"].min():.2f}%, '
      f'max {100 * rt["frac_acoes_B"].max():.2f}%')
print(f'   (V1 leitura B tinha media 89,92% no teste)')
print(f'meses do TESTE com mais de 50% em CDI na leitura B: '
      f'{int((rt["frac_acoes_B"] < 0.5).sum())}')
print('\nserie mensal do TESTE (n de ativos V1 e V2, gate, % em acoes na leitura B):')
srt = rt[['mes', 'n_ativos_v1', 'n_ativos_v2', 'n_modelos', 'n_gate_v2', 'frac_acoes_B']].copy()
srt['frac_acoes_B'] = (100 * srt['frac_acoes_B']).round(1)
print(srt.to_string(index=False))

# ==================================================================== (d) giro
print('\n' + '=' * 128)
print('(d) GIRO ONE-WAY MENSAL -- V1 e V2')
print('=' * 128)
lin = []
for rot, d in [('V1-A', V1['C2-A']), ('V2-A', dA), ('V1-B', V1['C2-B']), ('V2-B', dB),
               ('V1-EW', V1['EW_universo'])]:
    for part in ['TREINO', 'TESTE']:
        b = d[d['particao'] == part]
        lin.append({'versao': rot, 'particao': part, 'n': len(b),
                    'giro_ow_%mes': 100 * b['turnover'].mean() / 2,
                    'mediana_%': 100 * b['turnover'].median() / 2,
                    'min_%': 100 * b['turnover'].min() / 2, 'max_%': 100 * b['turnover'].max() / 2})
print(pd.DataFrame(lin).round(3).to_string(index=False))
print('\npersistencia do score explica a direcao: V1 0,5330 e V2 0,7204 no teste (item c).')

# ==================================================================== (e) curva de custo
print('\n' + '=' * 128)
print('(e) RETORNO BRUTO E LIQUIDO NA CURVA DE CUSTO -- leituras A e B, V1 e V2')
print('=' * 128)
lin = []
for rot, d in [('V1-A', V1['C2-A']), ('V2-A', dA), ('V1-B', V1['C2-B']), ('V2-B', dB)]:
    for part in ['TREINO', 'TESTE']:
        b = d[d['particao'] == part]
        reg = {'versao': rot, 'particao': part}
        for bps in NIVEIS_BPS:
            reg[f'{bps}bps'] = anual(liq(b, bps))
        lin.append(reg)
print(pd.DataFrame(lin).round(4).to_string(index=False))
print('\nreferencias TESTE: indice 7,1873 | ex-max 3,0731 | CDI 9,0315 %aa; '
      'TREINO: 22,4266 | 15,7325 | 13,1695')

# ==================================================================== (f) comparadores
print('\n' + '=' * 128)
print('(f) CONTRA OS COMPARADORES (TESTE, liquido de 50 bps)')
print('=' * 128)
print('\n200 aleatorias para V2 -- MESMA semente 20260815, mesma regra: dentro de cada modelo')
print('aprovado no gate de V2, sorteia o MESMO numero de ativos que V2 selecionou naquele modelo.')
pool = sc.groupby(['mes', 'modelo'])['CODISI'].apply(list).to_dict()
alvo_n = sel_v2.groupby(['mes', 'modelo']).size().to_dict()
chaves = sorted(alvo_n.keys())
isins = sorted({i for k in chaves for i in pool[k]})
pos_i = {c: i for i, c in enumerate(isins)}
pos_m = {m: i for i, m in enumerate(meses_dec)}
rng = np.random.default_rng(SEMENTE)
W = np.zeros((len(meses_dec), len(isins)))
t0 = time.time()
dist_v2 = []
for s in range(N_ALEATORIAS):
    W[:] = 0.0
    for (m, mod) in chaves:
        cand = pool[(m, mod)]; k = alvo_n[(m, mod)]
        esc = rng.choice(len(cand), size=k, replace=False)
        i = pos_m[m]
        for j in esc:
            W[i, pos_i[cand[j]]] += 1.0
    tot = W.sum(axis=1, keepdims=True)
    Wn = np.divide(W, tot, out=np.zeros_like(W), where=tot > 0)
    pw = pd.DataFrame(Wn, index=meses_dec, columns=isins)
    pw['__CDI__'] = 1.0 - Wn.sum(axis=1)
    dr, _ = rodar(pw, f'al{s}', silencioso=True)
    b = dr[dr['particao'] == 'TESTE']
    dist_v2.append(anual(liq(b, 50)))
dist_v2 = np.array(dist_v2)
print(f'  {N_ALEATORIAS} rodadas em {time.time() - t0:.1f}s')
v2a_50 = anual(liq(TE(dA), 50)); v2b_50 = anual(liq(TE(dB), 50))
v1a_50 = anual(liq(TE(V1['C2-A']), 50)); v1b_50 = anual(liq(TE(V1['C2-B']), 50))
ew_50 = anual(liq(TE(V1['EW_universo']), 50))
inv_v2_50 = anual(liq(TE(dI), 50)); inv_v1_50 = anual(liq(TE(V1['INVERTIDA']), 50))
d1a = pd.read_parquet(DIR_INT + r'\d1_aleatorias.parquet')
print(f'\n  distribuicao das 200 de V2 (liq 50bps, %aa): min {dist_v2.min():.4f} | '
      f'p5 {np.percentile(dist_v2, 5):.4f} | mediana {np.median(dist_v2):.4f} | '
      f'p95 {np.percentile(dist_v2, 95):.4f} | max {dist_v2.max():.4f}')
print(f'  distribuicao das 200 de V1 (ja medida em D1): min {d1a["anual_50bps"].min():.4f} | '
      f'p5 {d1a["anual_50bps"].quantile(.05):.4f} | mediana {d1a["anual_50bps"].median():.4f} | '
      f'p95 {d1a["anual_50bps"].quantile(.95):.4f} | max {d1a["anual_50bps"].max():.4f}')
pct_v2a = 100 * float((dist_v2 < v2a_50).mean())
pct_v2b = 100 * float((dist_v2 < v2b_50).mean())
lin = [
    {'serie': 'V1-A (leitura A)', 'liq50_%aa': v1a_50, 'percentil nas 200': 67.0},
    {'serie': 'V2-A (leitura A)', 'liq50_%aa': v2a_50, 'percentil nas 200': pct_v2a},
    {'serie': 'V1-B (leitura B)', 'liq50_%aa': v1b_50, 'percentil nas 200': 94.5},
    {'serie': 'V2-B (leitura B)', 'liq50_%aa': v2b_50, 'percentil nas 200': pct_v2b},
    {'serie': 'V1 invertida', 'liq50_%aa': inv_v1_50, 'percentil nas 200': np.nan},
    {'serie': 'V2 invertida', 'liq50_%aa': inv_v2_50, 'percentil nas 200': np.nan},
    {'serie': 'EW universo elegivel', 'liq50_%aa': ew_50, 'percentil nas 200': np.nan},
    {'serie': 'indice interno (D12)', 'liq50_%aa': anual(TE(dA)['ret_indice']), 'percentil nas 200': np.nan},
    {'serie': 'indice ex-max (D16)', 'liq50_%aa': anual(TE(dA)['ret_indice_ex_max']), 'percentil nas 200': np.nan},
    {'serie': 'CDI (D13)', 'liq50_%aa': anual(TE(dA)['ret_cdi']), 'percentil nas 200': np.nan},
]
print()
print(pd.DataFrame(lin).round(4).to_string(index=False))
print('\ninvertida de V2: construida por REFLEXAO em torno do neutro (s\' = 2/6 - s), que e o analogo')
print('exato da negacao usada em V1 -- la o neutro era 0, entao refletir era negar. Gate espelhado:')
print('media dos K piores < 1/6.')

# ==================================================================== (g) risco
print('\n' + '=' * 128)
print('(g) SHARPE, VOL, DRAWDOWN, % MESES POSITIVOS (TESTE, liquido de 50 bps)')
print('=' * 128)
rf = TE(dA)['ret_cdi'].to_numpy()
lin = []
for rot, d in [('V1-A', V1['C2-A']), ('V2-A', dA), ('V1-B', V1['C2-B']), ('V2-B', dB),
               ('V1 invertida', V1['INVERTIDA']), ('V2 invertida', dI),
               ('EW universo', V1['EW_universo'])]:
    b = TE(d); r = liq(b, 50).to_numpy(); exc = r - rf
    down = np.minimum(exc, 0.0)
    lin.append({'serie': rot, 'bruto_%aa': anual(b['bruto']), 'liq50_%aa': anual(r),
                'vol_%aa': 100 * r.std(ddof=1) * np.sqrt(12),
                'Sharpe': (exc.mean() * 12) / (exc.std(ddof=1) * np.sqrt(12)),
                'Sortino': (exc.mean() * 12) / (np.sqrt(float((down ** 2).mean())) * np.sqrt(12)),
                'DDmax_%': dd_max(r), 'meses+ %': 100 * float((r > 0).mean()),
                'giro_ow_%': 100 * b['turnover'].mean() / 2})
for rot, col in [('indice interno', 'ret_indice'), ('indice ex-max', 'ret_indice_ex_max'), ('CDI', 'ret_cdi')]:
    b = TE(dA); r = b[col].to_numpy(); exc = r - rf
    down = np.minimum(exc, 0.0)
    sd = exc.std(ddof=1)
    lin.append({'serie': rot, 'bruto_%aa': anual(r), 'liq50_%aa': anual(r),
                'vol_%aa': 100 * r.std(ddof=1) * np.sqrt(12),
                'Sharpe': (exc.mean() * 12) / (sd * np.sqrt(12)) if sd > 0 else np.nan,
                'Sortino': (exc.mean() * 12) / (np.sqrt(float((down ** 2).mean())) * np.sqrt(12))
                if float((down ** 2).mean()) > 0 else np.nan,
                'DDmax_%': dd_max(r), 'meses+ %': 100 * float((r > 0).mean()), 'giro_ow_%': np.nan})
print(pd.DataFrame(lin).round(4).to_string(index=False))

# ==================================================================== (h) t NW
print('\n' + '=' * 128)
print('(h) t DE NEWEY-WEST(3) DO EXCEDENTE SOBRE O INDICE INTERNO (TESTE)')
print('=' * 128)
lin = []
for rot, d in [('V1-A', V1['C2-A']), ('V2-A', dA), ('V1-B', V1['C2-B']), ('V2-B', dB),
               ('EW universo', V1['EW_universo'])]:
    b = TE(d)
    for tipo, r in [('bruto', b['bruto']), ('liquido50', liq(b, 50))]:
        x = r.to_numpy() - b['ret_indice'].to_numpy()
        mu, se, t = nw_t(x)
        y = r.to_numpy() - b['ret_indice_ex_max'].to_numpy()
        mu2, se2, t2 = nw_t(y)
        lin.append({'versao': rot, 'tipo': tipo, 'exc vs indice %mes': 100 * mu, 't NW3': t,
                    'exc vs ex-max %mes': 100 * mu2, 't NW3 ex-max': t2})
print(pd.DataFrame(lin).round(4).to_string(index=False))

# ==================================================================== (n) L48
print('\n' + '=' * 128)
print('(n) O RISCO DE L48, MEDIDO: DECIL SUPERIOR vs SELECIONADOS (V2, TESTE)')
print('=' * 128)
pt = sc[(sc['particao'] == 'TESTE') & sc['alfa_fut'].notna()].copy()
sel_par = set(zip(sel_v2['mes'], sel_v2['CODISI']))
lin = []
for m, g in pt.groupby('mes'):
    if len(g) < 20:
        continue
    d10 = pd.qcut(g['score_v2'].rank(method='first'), 10, labels=False)
    top = float(g['alfa_fut'][d10 == 9].mean())
    bot = float(g['alfa_fut'][d10 == 0].mean())
    msk = [(m, c) in sel_par for c in g['CODISI']]
    selm = g['alfa_fut'][msk]
    lin.append({'mes': m, 'n_sel': int(np.sum(msk)),
                'decil_sup': top, 'decil_inf': bot,
                'selecionados': float(selm.mean()) if len(selm) else np.nan,
                'universo': float(g['alfa_fut'].mean())})
dn = pd.DataFrame(lin).dropna()
print(f'meses do TESTE com dado: {len(dn)}; mediana de selecionados por mes: {dn["n_sel"].median():.0f}')
lin = []
for c, rot in [('decil_sup', 'decil SUPERIOR de score_v2'), ('selecionados', 'ATIVOS SELECIONADOS'),
               ('universo', 'universo de PREVISAO (media do alfa_fut, nao e zero -- ver nota)'),
               ('decil_inf', 'decil INFERIOR de score_v2')]:
    mu, se, t = nw_t(dn[c])
    lin.append({'grupo': rot, 'alfa_fut medio %mes': 100 * mu, 'ep NW3': 100 * se, 't NW3': t,
                '% meses > 0': 100 * float((dn[c] > 0).mean())})
print()
print(pd.DataFrame(lin).round(4).to_string(index=False))
dif = dn['selecionados'] - dn['decil_sup']
mu, se, t = nw_t(dif)
print(f'\nSELECIONADOS menos DECIL SUPERIOR: {100 * mu:+.4f}%/mes (ep NW3 {100 * se:.4f}, t {t:+.3f})')
if mu < 0:
    print('=> os ativos efetivamente comprados rendem MENOS que o decil superior do score. A regra de')
    print('   selecao (top K por subsetor + gate) esta perdendo parte do que o score captura.')
    print('   MECANISMO: o decil superior e do UNIVERSO INTEIRO; a carteira pega o top K DENTRO DE')
    print('   CADA subsetor, entao compra o melhor de subsetores fracos junto com o melhor dos fortes.')
else:
    print('=> os selecionados rendem ao menos tanto quanto o decil superior.')
print('\nNOTA sobre a linha do universo: a media do alfa_fut sobre o universo de PREVISAO nao e zero')
print('(-0,1352%/mes) porque o alfa e medido contra o indice, que e a media dos ELEGIVEIS do mes,')
print('e o universo de previsao e um subconjunto dele (exige as 6 features completas, D19/D25).')
print('A diferenca e o proprio custo de D19, ja medido em B1.')

print('\n*** O ACHADO MAIS IMPORTANTE DESTE ITEM, e ele NAO estava sendo procurado ***')
mu_top, _, t_top = nw_t(dn['decil_sup'])
mu_bot, _, t_bot = nw_t(dn['decil_inf'])
print(f'O decil SUPERIOR de score_v2 tem alfa_fut medio de {100 * mu_top:+.4f}%/mes (t {t_top:+.3f}) --')
print(f'indistinguivel de zero. O decil INFERIOR tem {100 * mu_bot:+.4f}%/mes (t {t_bot:+.3f}).')
print('Ou seja: o IC de +0,0614 (t=4,14) de V2 vem quase todo de o score IDENTIFICAR PERDEDORES,')
print('nao de identificar ganhadores. E a carteira compra ganhadores -- ela opera exatamente na')
print('ponta onde o score nao tem informacao. Isso e a confirmacao mais direta possivel de L48,')
print('e explica por que um IC quatro vezes maior que o de V1 nao vira retorno proporcional.')
print('\nISTO E DIAGNOSTICO, NAO CONSERTO. Nada foi alterado -- a arquitetura por subsetor e D01/D23,')
print('congelada, e mudar o agrupamento (ou operar vendido no decil inferior) seria uma V3,')
print('proibida nesta sessao.')

# ==================================================================== (o) Brinson
print('\n' + '=' * 128)
print('(o) DECOMPOSICAO DE BRINSON -- V2 ao lado de V1 (TESTE)')
print('=' * 128)
sub_de = split.drop_duplicates('CODISI').set_index('CODISI')['subsetor_chave'].to_dict()
ret_de = {(m, c): r for m, c, r in zip(split['mes'], split['CODISI'], split['ret_1m'])}
eleg_mes = {m: g for m, g in split[split['elegivel'] & split['ret_valido']].groupby('mes')}
lin_br = []
for rot, tr, pesocol in [('V2-A', sel_v2, 'peso'), ('V2-B', selB, 'peso_b')]:
    t = tr[tr['particao'] == 'TESTE']
    for mes, g in t.groupby('mes'):
        m1 = str(pd.Period(mes, freq='M') + 1)
        be = eleg_mes.get(m1)
        if be is None:
            continue
        rb_tot = float(be['ret_1m'].mean())
        wb = be.groupby('subsetor_chave').size() / len(be)
        rb = be.groupby('subsetor_chave')['ret_1m'].mean()
        gg = g.copy()
        gg['sub'] = gg['CODISI'].map(sub_de)
        gg['r'] = [ret_de.get((m1, c), np.nan) for c in gg['CODISI']]
        gg['r'] = gg['r'].fillna(0.0)
        gg['w'] = gg[pesocol]
        w_cdi = 1.0 - float(gg['w'].sum())
        if w_cdi > 1e-12:
            gg = pd.concat([gg, pd.DataFrame([{'sub': '__CDI__', 'w': w_cdi,
                                               'r': float(bi.loc[m1, 'ret_cdi'])}])], ignore_index=True)
        wp = gg.groupby('sub')['w'].sum()
        rp = gg.groupby('sub').apply(lambda h: float((h['w'] * h['r']).sum() / h['w'].sum()),
                                     include_groups=False)
        subs = sorted(set(wp.index) | set(wb.index))
        wpv = np.array([wp.get(s, 0.0) for s in subs]); wbv = np.array([wb.get(s, 0.0) for s in subs])
        rbv = np.array([rb.get(s, rb_tot) for s in subs]); rpv = np.array([rp.get(s, 0.0) for s in subs])
        aloc = float(((wpv - wbv) * rbv).sum()); selec = float((wpv * (rpv - rbv)).sum())
        lin_br.append({'versao': rot, 'mes': mes, 'alocacao': aloc, 'selecao': selec,
                       'exc': float((wpv * rpv).sum()) - float((wbv * rbv).sum())})
br2 = pd.DataFrame(lin_br)
br2['residuo'] = br2['exc'] - (br2['alocacao'] + br2['selecao'])
lin = []
for rot, b in list(d2_v1.groupby('versao')) + list(br2.groupby('versao')):
    nome = {'C2-A': 'V1-A', 'C2-B': 'V1-B'}.get(rot, rot)
    _, _, ta = nw_t(b['alocacao'], 3); _, _, ts = nw_t(b['selecao'], 3)
    lin.append({'versao': nome, 'meses': len(b), 'alocacao_%mes': 100 * b['alocacao'].mean(),
                't aloc': ta, 'selecao_%mes': 100 * b['selecao'].mean(), 't selec': ts,
                'excedente_%mes': 100 * b['exc'].mean(),
                'residuo_max': float(b['residuo'].abs().max())})
print(pd.DataFrame(lin).round(6).to_string(index=False))
assert br2['residuo'].abs().max() < 1e-9, 'identidade de Brinson quebrou em V2'
print('\nidentidade exata nas quatro series (residuo maximo abaixo de 1e-9).')

# ==================================================================== (j) D29 subperiodos
print('\n' + '=' * 128)
print('(j) D29 -- SUBPERIODOS, definidos ANTES de rodar. Retorno LIQUIDO de 50 bps, anualizado')
print('=' * 128)
lin = []
for ini, fim in SUBPERIODOS:
    msk = lambda d: (d['mes'] >= ini) & (d['mes'] <= fim) & (d['particao'] == 'TESTE')
    bA1 = V1['C2-A'][msk(V1['C2-A'])]; bA2 = dA[msk(dA)]
    bB1 = V1['C2-B'][msk(V1['C2-B'])]; bB2 = dB[msk(dB)]
    bE = V1['EW_universo'][msk(V1['EW_universo'])]
    lin.append({'subperiodo': f'{ini} a {fim}', 'meses': len(bA1),
                'V1-A': anual(liq(bA1, 50)), 'V2-A': anual(liq(bA2, 50)),
                'V1-B': anual(liq(bB1, 50)), 'V2-B': anual(liq(bB2, 50)),
                'EW': anual(liq(bE, 50)), 'indice': anual(bA1['ret_indice']),
                'CDI': anual(bA1['ret_cdi'])})
sub = pd.DataFrame(lin)
print(sub.round(4).to_string(index=False))
sub['V2A-V1A'] = sub['V2-A'] - sub['V1-A']
sub['V2A-indice'] = sub['V2-A'] - sub['indice']
sub['V2A-EW'] = sub['V2-A'] - sub['EW']
sub['V2B-V1B'] = sub['V2-B'] - sub['V1-B']
print('\ndiferencas em pp/ano:')
print(sub[['subperiodo', 'V2A-V1A', 'V2B-V1B', 'V2A-indice', 'V2A-EW']].round(4).to_string(index=False))
n3 = int((sub['V2A-V1A'] > 0).sum())
print(f'\nA MELHORIA DE V2 SOBRE V1 (leitura A) APARECE EM {n3} DOS 3 SUBPERIODOS.')
print(f'contra o indice interno, V2-A fica acima em {int((sub["V2A-indice"] > 0).sum())} de 3; '
      f'contra o EW, em {int((sub["V2A-EW"] > 0).sum())} de 3.')
print('[F] poder ~331 meses: isto NAO e teste de significancia, e diagnostico de consistencia.')

# ==================================================================== (k) capacity
print('\n' + '=' * 128)
print('(k) CAPACITY DE V2')
print('=' * 128)
vol_de = {(m, c): v for m, c, v in zip(split['mes'], split['CODISI'], split['VOLTOT'])}
tv = sel_v2[sel_v2['particao'] == 'TESTE'].copy()
tv['VOLTOT'] = [vol_de.get((m, c), np.nan) for m, c in zip(tv['mes'], tv['CODISI'])]
print(f'VOLTOT das posicoes de V2 no teste: mediana R$ {tv["VOLTOT"].median():,.0f} | '
      f'p10 R$ {tv["VOLTOT"].quantile(.10):,.0f} | min R$ {tv["VOLTOT"].min():,.0f}')
print('(V1: mediana R$ 69.715.679 | p10 R$ 390.205 | min R$ 2.702)')
lin = []
for p in (0.05, 0.10):
    tv['cap'] = p * tv['VOLTOT'] / tv['peso']
    pm = tv.groupby('mes')['cap'].min()
    ib = tv.loc[tv['cap'].idxmin()]
    med = tv.groupby('mes')['VOLTOT'].median(); nn = tv.groupby('mes').size()
    lin.append({'participacao': f'{p:.0%}', 'V2 min-do-mes mediana R$': float(pm.median()),
                'V2 min-do-mes p10 R$': float(pm.quantile(.10)),
                'V2 min-do-mes minimo R$': float(pm.min()),
                'V2 pelo VOLTOT mediano R$': float((p * med * nn).median()),
                'ativo mais restritivo': f'{ib["CODNEG"]} ({ib["mes"]})'})
print()
print(pd.DataFrame(lin).to_string(index=False, float_format=lambda x: f'{x:,.0f}'))
print('V1, para comparacao: min-do-mes mediana R$ 128.251 (5%) e R$ 256.502 (10%); '
      'pelo VOLTOT mediano R$ 140.683.784 e R$ 281.367.569; mais restritivo CEDO3 (2023-01).')
sb = tv.groupby('mes').apply(lambda g: int((g['VOLTOT'] < 1e6).sum()), include_groups=False)
print(f'entre as SELECIONADAS de V2: mediana de {sb.median():.0f} por mes com VOLTOT < R$1M, '
      f'em {int((sb > 0).sum())} de {len(sb)} meses ha ao menos uma (V1: 6 e 102 de 103)')

# ==================================================================== gravacao
saida = []
for rot, d in [('V2-A', dA), ('V2-B', dB), ('V2-INVERTIDA', dI)]:
    x = d[['mes', 'mes_ret', 'particao', 'bruto', 'turnover']].copy()
    x['versao'] = rot
    for bps in NIVEIS_BPS:
        x[f'liq_{bps}bps'] = liq(d, bps)
    saida.append(x)
pd.concat(saida, ignore_index=True).to_parquet(DIR_INT + r'\v2_backtest.parquet', index=False)
sel_v2.to_parquet(DIR_INT + r'\v2_carteira.parquet', index=False)
br2.to_parquet(DIR_INT + r'\v2_brinson.parquet', index=False)
pd.DataFrame({'sorteio': np.arange(N_ALEATORIAS), 'anual_50bps': dist_v2}).to_parquet(
    DIR_INT + r'\v2_aleatorias.parquet', index=False)
print('\n' + '=' * 128)
print(f'GRAVADO v2_backtest.parquet | v2_carteira.parquet {sel_v2.shape} | '
      f'v2_brinson.parquet {br2.shape} | v2_aleatorias.parquet ({N_ALEATORIAS}, 2)')
print('V2 COMPLETA. V1 intacta -- nenhum arquivo de V1 foi sobrescrito.')
print('=' * 128)


V2 -- RETOMADA COM O GATE DE D30 (score > 1/6). Itens (d) a (o) e D29.
V1 e V2 lado a lado em toda tabela. V1 INTACTA -- nada e sobrescrito.

matriz de retornos (380, 1873); meses de decisao 296

SELECAO DE V2 -- gate de D30
gate: media dos K selecionados > 0.166667 (= 1/6, o ativo neutro na escala de V2)
gate de V1, para comparacao: media dos K > 0.0 (o zero da escala de alfa previsto)

grupos (mes, modelo): 3010; passam no gate V2: 2853 (94.78%)  |  V1 passava 2.642 (87,77%)
posicoes selecionadas V2: 8050  |  V1: 6567

CONSEQUENCIA DA REGRA DE K SOB A NOVA ESCALA -- medida, aplicada literalmente, NAO alterada:
a regra "K=2 se score[3] < score[1]/2" e uma razao, e razao depende de onde esta o zero da
escala. D30 corrigiu o zero do GATE (0 -> 1/6) e NAO tocou na regra de K, entao ela foi
aplicada literalmente sobre o score_v2 bruto. O efeito medido:
  V2: regra da metade acionada em 593 de 3010 grupos (19.70%)
  V1: acionada em 1.727 de 3.010 (57,38%)
  distribuicao de K em V2: {2: 593

In [ ]:
# Correcao pos-conselho (plano item 6) -- o PAR de t do IC de score_v2 no TESTE, e a faixa por lag.
# A serie mensal de IC tem autocorrelacao NEGATIVA, entao a correcao NW AUMENTA o t com o lag:
# publicar so o 4,14 e a escolha anticonservadora. Publique o PAR (3,6371 iid / 4,1422 NW3) e a faixa.
import numpy as np
import pandas as pd

v2 = pd.read_parquet(r'..\intermediario\v2_scores.parquet',
                     columns=['mes', 'particao', 'score_v2', 'alfa_fut'])
t = v2[(v2['particao'] == 'TESTE') & v2['score_v2'].notna() & v2['alfa_fut'].notna()]


def spearman(x, y):
    rx = pd.Series(np.asarray(x, dtype=float)).rank(method='average').to_numpy()
    ry = pd.Series(np.asarray(y, dtype=float)).rank(method='average').to_numpy()
    rx = (rx - rx.mean()) / rx.std()
    ry = (ry - ry.mean()) / ry.std()
    return float((rx * ry).mean())


def t_nw(s, L):
    x = s.to_numpy(dtype=float)
    n = len(x)
    e = x - x.mean()
    var = (e @ e) / n
    for l in range(1, L + 1):
        var += 2.0 * (1.0 - l / (L + 1.0)) * ((e[l:] @ e[:-l]) / n)
    return x.mean() / np.sqrt(var / n)


ic = pd.Series({m: spearman(h['score_v2'], h['alfa_fut']) for m, h in t.groupby('mes')}).sort_index()
n = len(ic)
mu = ic.mean()
t_iid = mu / (ic.std(ddof=1) / np.sqrt(n))
e = ic - mu
ac1 = float((e[1:].to_numpy() @ e[:-1].to_numpy()) / (e.to_numpy() @ e.to_numpy()))

print(f'IC de score_v2 no TESTE ({n} meses): medio {mu:+.6f}  (alvo +0,061370)')
print(f'PAR obrigatorio: t iid (ddof=1) = {t_iid:.4f}  |  t NW3 = {t_nw(ic, 3):.4f}')
print(f'autocorrelacao lag-1 da serie de IC = {ac1:+.4f}  (NEGATIVA)')
print('\nfaixa por lag (alvos do conselho 3,65 / 3,90 / 3,93 / 4,14 / 4,45 / 5,18 / 6,26):')
for L in (0, 1, 2, 3, 4, 6, 12):
    print(f'  t NW L={L:>2}: {t_nw(ic, L):.4f}')
print('\nA serie mensal de IC tem autocorrelacao NEGATIVA; NW AUMENTA o t monotonicamente com o')
print('lag, logo publicar so o 4,14 e anticonservador -- onde o IC +0,0614 aparecer, publica-se')
print('o PAR (3,6371 iid / 4,1422 NW3) e a faixa por lag.')
assert abs(mu - 0.061370) < 5e-6 and abs(t_iid - 3.6371) < 5e-4 and ac1 < 0, \
    'divergencia nos alvos do par de t do IC'
print('\nVEREDITO: PASSOU')


IC de score_v2 no TESTE (103 meses): medio +0.061370  (alvo +0,061370)
PAR obrigatorio: t iid (ddof=1) = 3.6371  |  t NW3 = 4.1422
autocorrelacao lag-1 da serie de IC = -0.1208  (NEGATIVA)

faixa por lag (alvos do conselho 3,65 / 3,90 / 3,93 / 4,14 / 4,45 / 5,18 / 6,26):
  t NW L= 0: 3.6549
  t NW L= 1: 3.8980
  t NW L= 2: 3.9265
  t NW L= 3: 4.1422
  t NW L= 4: 4.4549
  t NW L= 6: 5.1770
  t NW L=12: 6.2591

A serie mensal de IC tem autocorrelacao NEGATIVA; NW AUMENTA o t monotonicamente com o
lag, logo publicar so o 4,14 e anticonservador -- onde o IC +0,0614 aparecer, publica-se
o PAR (3,6371 iid / 4,1422 NW3) e a faixa por lag.

VEREDITO: PASSOU


## V3 (D31) -- exclusao do decil inferior de score_v2. D32: ULTIMA variante desta rodada.

V3 usa o mesmo score de V2 para EXCLUIR em vez de SELECIONAR: parte do universo com `score_v2`
disponivel em peso igual e remove o decil inferior do mes. Sem gate, sem K, sem agrupamento por
subsetor na selecao. **V3 nao e um comite de especialistas que escolhe -- e um filtro de exclusao
sobre uma carteira ampla**, e o relatorio nao pode usar a moldura de comite sem essa correcao.

**Procedencia declarada sem atenuante:** V3 foi motivada por um resultado de carteira ja observado
(V2 perdendo dos comparadores) somado a um diagnostico de secao transversal. V2 tinha procedencia
melhor -- foi decidida a partir de IC, sem retorno algum. V3 nao tem essa propriedade.

**Duas declaracoes de base que a celula faz antes de qualquer numero:**

1. "Elegiveis com score_v2 disponivel" NAO e "elegiveis": o score exige as 6 features completas
   (D19/D25). No teste sao ~252 contra ~295. Logo V3 difere do EW do universo por DUAS coisas --
   base menor e decil removido -- e o item (i) isola as duas.
2. **Separacao GIRO vs SELECAO, que nao estava no enunciado e sem a qual S5 seria mal lida.** A
   exclusao aleatoria ressorteia todo mes e paga giro por isso (14,4%/mes contra 8,97% de V3). O
   percentil a 50 bps sozinho confundiria "o score acha perdedores" com "o score acha os MESMOS
   perdedores todo mes". A celula reporta o percentil a 50 bps E a custo zero.

Resultado de S5: V3 fica no **percentil 100 das exclusoes aleatorias tanto a 50 bps quanto a custo
zero**, e a vantagem sobre a exclusao aleatoria mediana decompoe em **+1,3773 pp/ano de selecao** e
**+0,6714 pp/ano de giro**.

V1 e V2 permanecem intactas -- hash conferido, nenhum arquivo sobrescrito.


In [1]:
import hashlib
import sys
import time

import numpy as np
import pandas as pd

if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8')

pd.set_option('display.width', 275)
pd.set_option('display.max_columns', 90)
pd.set_option('display.max_rows', 400)

RAIZ = r'C:\Users\lucca\quant2026'
DIR_INT = RAIZ + r'\intermediario'
if RAIZ + r'\src' not in sys.path:
    sys.path.insert(0, RAIZ + r'\src')
import motor  # noqa: E402

HASH_B4B5 = '5f67e19fa21e1b22f8d29064fca75ba06c498e2b62ac6b87ae67f332eb385e67'
NIVEIS_BPS = [0, 10, 25, 50, 75, 100]
SEMENTE = 20260815
N_ALEAT = 200
DECIL = 10                       # D31: 10%, o recorte em que a assimetria foi MEDIDA
SUBPERIODOS = [('2018-01', '2020-12'), ('2021-01', '2023-12'), ('2024-01', '2026-07')]

print('=' * 128)
print('V3 (D31) -- EXCLUSAO do decil inferior de score_v2 sobre carteira ampla em peso igual.')
print('D32: esta e a ULTIMA variante desta rodada. Nao havera V4 antes da auditoria do conselho.')
print('V1, V2 e V3 lado a lado em toda tabela. V1 e V2 INTACTAS.')
print('=' * 128)


# ==================================================================== helpers
def _pearson(x, y):
    x = np.asarray(x, dtype=float); y = np.asarray(y, dtype=float)
    ok = np.isfinite(x) & np.isfinite(y)
    x, y = x[ok] - x[ok].mean(), y[ok] - y[ok].mean()
    den = np.sqrt(float(x @ x) * float(y @ y))
    return float(x @ y) / den if den > 0 else np.nan


def nw_t(x, lags=3):
    x = np.asarray(pd.Series(x).dropna(), dtype=float)
    n = len(x); mu = x.mean(); d = x - mu
    v = float(d @ d) / n
    for l in range(1, lags + 1):
        v += 2.0 * (1.0 - l / (lags + 1.0)) * (float(d[l:] @ d[:-l]) / n)
    se = np.sqrt(v / n)
    return mu, se, mu / se


def anual(r):
    r = np.asarray(pd.Series(r).dropna(), dtype=float)
    return 100.0 * ((np.prod(1.0 + r)) ** (12.0 / len(r)) - 1.0)


def dd_max(r):
    r = np.asarray(pd.Series(r).dropna(), dtype=float)
    w = np.cumprod(1 + r); p = np.maximum.accumulate(w)
    return 100.0 * float((w / p - 1.0).min())


def sha(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(1 << 20), b''):
            h.update(b)
    return h.hexdigest()


# ==================================================================== S1
print('\n' + '=' * 128)
print('S1 -- V1 E V2 INTACTAS')
print('=' * 128)
h1 = sha(DIR_INT + r'\b4b5_previsoes.parquet')
print(f'b4b5_previsoes.parquet (V1) SHA-256 = {h1}')
print(f'registrado no MODEL_SPEC de B6       = {HASH_B4B5}')
assert h1 == HASH_B4B5, 'S1 FALHOU: V1 mudou'
h2 = sha(DIR_INT + r'\v2_scores.parquet')
print(f'v2_scores.parquet (V2) SHA-256 = {h2}')
print('   (v2_scores foi gravado na sessao de V2 e nao e reescrito aqui -- esta celula so LE)')
arqs_v1v2 = ['c1_carteira', 'c2_backtest', 'c3_curva_custo', 'd1_tabela_final', 'd2_brinson',
             'd4_metricas_finais', 'v2_backtest', 'v2_carteira', 'v2_brinson', 'v2_aleatorias']
print(f'\narquivos de V1/V2 que esta celula LE e NAO grava ({len(arqs_v1v2)}):')
for a in arqs_v1v2:
    print(f'   {a}.parquet  sha {sha(DIR_INT + chr(92) + a + ".parquet")[:12]}')
print('   Esta celula grava exclusivamente em arquivos novos com prefixo v3_.')
print('   S1 PASSOU')

# ==================================================================== insumos
sc = pd.read_parquet(DIR_INT + r'\v2_scores.parquet')
bench = pd.read_parquet(DIR_INT + r'\a6_benchmark_e_rf.parquet')
split = pd.read_parquet(DIR_INT + r'\a7_split.parquet')
c3v1 = pd.read_parquet(DIR_INT + r'\c3_curva_custo.parquet')
v2bt = pd.read_parquet(DIR_INT + r'\v2_backtest.parquet')
d2v1 = pd.read_parquet(DIR_INT + r'\d2_brinson.parquet')
v2br = pd.read_parquet(DIR_INT + r'\v2_brinson.parquet')
sc['CODISI'] = sc['CODISI'].astype(str)
split['CODISI'] = split['CODISI'].astype(str)

meses_all = sorted(bench['mes'].unique())
data_de_mes = {m: pd.Period(m, freq='M').to_timestamp(how='end').normalize() for m in meses_all}
mes_de_data = {v: k for k, v in data_de_mes.items()}
meses_dec = sorted(sc['mes'].unique())
part_por_mes = sc.groupby('mes')['particao'].first().to_dict()
bi = bench.set_index('mes')

wide = split.pivot_table(index='mes', columns='CODISI', values='ret_1m', aggfunc='first').reindex(meses_all)
wide.index = pd.DatetimeIndex([data_de_mes[m] for m in meses_all])
cdi = bi['ret_cdi'].reindex(meses_all); cdi.index = wide.index
wide['__CDI__'] = cdi.values
CAIXA = wide['__CDI__']


def rodar(p_df, rot, sil=False):
    p = p_df.copy()
    cols = [c for c in p.columns if (p[c] != 0).any()]
    p = p[cols].fillna(0.0)
    p.index = pd.DatetimeIndex([data_de_mes[m] for m in p.index])
    o = motor.rodar_backtest(p, wide[cols], custo_bps=0.0, retornos_caixa=CAIXA)
    rb = o['retornos_brutos']; tv = o['turnover'].reindex(rb.index)
    d = pd.DataFrame({'mes_ret': [mes_de_data[i] for i in rb.index],
                      'bruto': rb.values, 'turnover': tv.values})
    d['mes'] = [str(pd.Period(m, freq='M') - 1) for m in d['mes_ret']]
    d['particao'] = d['mes'].map(part_por_mes)
    for c in ['ret_indice', 'ret_indice_ex_max', 'ret_cdi']:
        d[c] = d['mes_ret'].map(bi[c])
    if not sil:
        print(f'  [{rot}] ativos {len(cols)}  meses {len(d)}  '
              f'congelamentos {len(o["eventos_congelamento"])}  descartados {o["rebal_descartados"]}')
    return d, o


def liq(d, bps):
    return d['bruto'] - d['turnover'] * (bps / 10000.0)


def matriz(df, col='peso'):
    m = df.pivot_table(index='mes', columns='CODISI', values=col, aggfunc='sum').reindex(meses_dec)
    m = m.fillna(0.0); m['__CDI__'] = 0.0
    return m


TE = lambda d: d[d['particao'] == 'TESTE'].reset_index(drop=True)

# ==================================================================== C1-V3
print('\n' + '=' * 128)
print('C1-V3 -- CONSTRUCAO POR EXCLUSAO')
print('=' * 128)
print('regra de D31: partir do universo com score_v2 disponivel, remover o DECIL INFERIOR de')
print('score_v2 daquele mes, comprar todo o resto em peso igual. Sem gate, sem K, sem agrupamento')
print('por subsetor. Corte de 10% NAO otimizado: e o recorte em que a assimetria foi medida em V2.')

print('\nDECLARACAO DE BASE, e ela nao e cosmetica. "Elegiveis com score_v2 disponivel" NAO e o')
print('mesmo conjunto que "elegiveis": o score exige as 6 features completas (D19/D25). Medido:')
eleg = split[split['elegivel'] & split['mes'].isin(meses_dec)]
n_el = eleg.groupby('mes').size()
n_base = sc.groupby('mes').size()
raz = (n_base / n_el).dropna()
print(f'   elegiveis por mes: mediana {n_el.median():.0f}  |  base de V3 (com score_v2): '
      f'mediana {n_base.median():.0f}  |  razao mediana {raz.median():.4f}')
print(f'   no TESTE: elegiveis {n_el[[m for m in meses_dec if part_por_mes[m] == "TESTE"]].median():.0f} vs '
      f'base {n_base[[m for m in meses_dec if part_por_mes[m] == "TESTE"]].median():.0f}')
print('   Portanto V3 tem DUAS diferencas contra o EW do universo: (1) a base e menor, e (2) o decil')
print('   inferior e removido. O item (i) isola as duas.')

sc = sc.sort_values(['mes', 'score_v2']).reset_index(drop=True)
sc['decil'] = sc.groupby('mes')['score_v2'].transform(
    lambda s: pd.qcut(s.rank(method='first'), DECIL, labels=False))
v3 = sc[sc['decil'] > 0].copy()
n_v3 = v3.groupby('mes').size().rename('n')
v3 = v3.merge(n_v3, on='mes')
v3['peso'] = 1.0 / v3['n']
excluidos = sc[sc['decil'] == 0]
print(f'\nposicoes V3: {len(v3)}  |  excluidos: {len(excluidos)}  |  total base: {len(sc)}')

# ---- S3
frac_base = (n_v3 / n_base).dropna()
frac_eleg = (n_v3 / n_el).dropna()
print(f'\nS3 n de ativos de V3 sobre a BASE (universo com score_v2): mediana '
      f'{frac_base.median():.4f}, min {frac_base.min():.4f}, max {frac_base.max():.4f}')
assert abs(frac_base.median() - 0.90) < 0.02, 'S3 FALHOU contra a base'
print(f'   e sobre os ELEGIVEIS: mediana {frac_eleg.median():.4f} -- MENOR que 90%, porque a base')
print('   ja e um subconjunto dos elegiveis. O enunciado dizia "~90% dos elegiveis"; o numero certo')
print(f'   e ~90% da BASE ({100 * frac_base.median():.1f}%) e ~{100 * frac_eleg.median():.0f}% dos elegiveis.')
print('   Reportado assim em vez de deixar passar como se fosse a mesma coisa. S3 PASSOU (na base).')

# ---- S2 e S4
print('\nS2 -- a exclusao nao usa alfa_fut nem informacao de T ou posterior:')
sem_alvo = sc.drop(columns=['alfa_fut']).copy()
sem_alvo['decil2'] = sem_alvo.groupby('mes')['score_v2'].transform(
    lambda s: pd.qcut(s.rank(method='first'), DECIL, labels=False))
dif = int((sem_alvo['decil2'].to_numpy() != sc['decil'].to_numpy()).sum())
print(f'   recalculando o decil sobre o dataframe com `alfa_fut` REMOVIDO: {dif} divergencias '
      f'em {len(sc)} linhas')
assert dif == 0, 'S2 FALHOU'
mesmo_mes = all(len(g['mes'].unique()) == 1 for _, g in sc.groupby('mes'))
print(f'   o decil e calculado por groupby("mes"), logo so com dados do proprio T: {mesmo_mes}')
print('   S2 PASSOU')
inter = set(zip(v3['mes'], v3['CODISI'])) & set(zip(excluidos['mes'], excluidos['CODISI']))
print(f'S4 ativos excluidos que reaparecem na carteira do mesmo mes: {len(inter)}')
assert len(inter) == 0, 'S4 FALHOU'
print('   S4 PASSOU')

pes_v3 = matriz(v3)
print('\nrodando o motor:')
d3, o3 = rodar(pes_v3, 'V3')

# ---- EW da BASE (sem exclusao) -- necessario para o item (i)
base_ew = sc.copy()
base_ew = base_ew.merge(n_base.rename('n'), on='mes')
base_ew['peso'] = 1.0 / base_ew['n']
dBASE, _ = rodar(matriz(base_ew), 'EW da BASE (sem exclusao)')

V1A = c3v1[c3v1['versao'] == 'C2-A'].sort_values('mes').reset_index(drop=True)
V1B = c3v1[c3v1['versao'] == 'C2-B'].sort_values('mes').reset_index(drop=True)
EW = c3v1[c3v1['versao'] == 'EW_universo'].sort_values('mes').reset_index(drop=True)
V2A = v2bt[v2bt['versao'] == 'V2-A'].sort_values('mes').reset_index(drop=True)
V2B = v2bt[v2bt['versao'] == 'V2-B'].sort_values('mes').reset_index(drop=True)
for d in [V1A, V1B, EW, V2A, V2B]:
    for c in ['ret_indice', 'ret_indice_ex_max', 'ret_cdi']:
        d[c] = d['mes_ret'].map(bi[c])

print('\nC2-V3: leituras A e B NAO SE APLICAM. Sem gate nao ha modelo reprovado, logo nao ha fatia')
print('de capital ociosa: V3 e 100% investida por construcao, em versao unica. Declarado.')

# ==================================================================== (a) curva de custo
print('\n' + '=' * 128)
print('(a) RETORNO BRUTO E LIQUIDO NA CURVA DE CUSTO -- V1, V2, V3, EW')
print('=' * 128)
lin = []
for rot, d in [('V1-A', V1A), ('V1-B', V1B), ('V2-A', V2A), ('V2-B', V2B),
               ('V3 (exclusao)', d3), ('EW universo', EW)]:
    for part in ['TREINO', 'TESTE']:
        b = d[d['particao'] == part]
        reg = {'versao': rot, 'particao': part}
        for bps in NIVEIS_BPS:
            reg[f'{bps}bps'] = anual(liq(b, bps))
        lin.append(reg)
print(pd.DataFrame(lin).round(4).to_string(index=False))
print('\nreferencias TESTE: indice 7,1873 | ex-max 3,0731 | CDI 9,0315 %aa')


def breakeven(b, col):
    alvo = anual(b[col])
    f = lambda x: anual(liq(b, x)) - alvo
    lo, hi = -3000.0, 3000.0
    if f(lo) < 0:
        return np.nan
    if f(hi) > 0:
        return np.inf
    for _ in range(200):
        mid = (lo + hi) / 2
        lo, hi = (mid, hi) if f(mid) > 0 else (lo, mid)
    return (lo + hi) / 2


print('\nC3-V3 -- BREAKEVEN (custo_bps em que a carteira iguala cada comparador), TESTE:')
lin = []
for rot, d in [('V1-A', V1A), ('V2-A', V2A), ('V3', d3), ('EW universo', EW)]:
    b = TE(d)
    reg = {'versao': rot, 'liquido a 0bps': anual(b['bruto'])}
    for col, nome in [('ret_indice', 'indice'), ('ret_indice_ex_max', 'ex-max'), ('ret_cdi', 'CDI')]:
        reg[f'be vs {nome}'] = breakeven(b, col)
    # breakeven contra o EW LIQUIDO de 50bps: alvo escalar, mesmo solver
    ewv = anual(liq(TE(EW), 50))
    g = lambda x: anual(liq(b, x)) - ewv
    if g(-3000.0) < 0:
        reg['be vs EW liq50'] = np.nan
    else:
        lo, hi = -3000.0, 3000.0
        for _ in range(200):
            mid = (lo + hi) / 2
            lo, hi = (mid, hi) if g(mid) > 0 else (lo, mid)
        reg['be vs EW liq50'] = (lo + hi) / 2
    lin.append(reg)
be = pd.DataFrame(lin)
print(be.round(2).to_string(index=False))
print('breakeven NEGATIVO significa que a carteira ja perde do comparador a CUSTO ZERO.')

# ==================================================================== (b) risco
print('\n' + '=' * 128)
print('(b) RISCO -- TESTE, liquido de 50 bps')
print('=' * 128)
rf = TE(V1A)['ret_cdi'].to_numpy()
lin = []
for rot, d in [('V1-A', V1A), ('V1-B', V1B), ('V2-A', V2A), ('V2-B', V2B),
               ('V3 (exclusao)', d3), ('EW universo', EW), ('EW da BASE', dBASE)]:
    b = TE(d); r = liq(b, 50).to_numpy(); e = r - rf
    dn_ = np.minimum(e, 0.0)
    lin.append({'serie': rot, 'bruto_%aa': anual(b['bruto']), 'liq50_%aa': anual(r),
                'vol_%aa': 100 * r.std(ddof=1) * np.sqrt(12),
                'Sharpe': (e.mean() * 12) / (e.std(ddof=1) * np.sqrt(12)),
                'Sortino': (e.mean() * 12) / (np.sqrt(float((dn_ ** 2).mean())) * np.sqrt(12)),
                'DDmax_%': dd_max(r), 'meses+ %': 100 * float((r > 0).mean()),
                'giro_ow_%': 100 * b['turnover'].mean() / 2})
for rot, col in [('indice interno', 'ret_indice'), ('indice ex-max', 'ret_indice_ex_max'), ('CDI', 'ret_cdi')]:
    b = TE(V1A); r = b[col].to_numpy(); e = r - rf; dn_ = np.minimum(e, 0.0)
    sd = e.std(ddof=1); dv = float((dn_ ** 2).mean())
    lin.append({'serie': rot, 'bruto_%aa': anual(r), 'liq50_%aa': anual(r),
                'vol_%aa': 100 * r.std(ddof=1) * np.sqrt(12),
                'Sharpe': (e.mean() * 12) / (sd * np.sqrt(12)) if sd > 0 else np.nan,
                'Sortino': (e.mean() * 12) / (np.sqrt(dv) * np.sqrt(12)) if dv > 0 else np.nan,
                'DDmax_%': dd_max(r), 'meses+ %': 100 * float((r > 0).mean()), 'giro_ow_%': np.nan})
tabB = pd.DataFrame(lin)
print(tabB.round(4).to_string(index=False))

# ==================================================================== (c) e (d)
print('\n' + '=' * 128)
print('(c) GIRO ONE-WAY  e  (d) NUMERO DE ATIVOS POR MES')
print('=' * 128)
lin = []
for rot, d in [('V1-A', V1A), ('V2-A', V2A), ('V3', d3), ('EW universo', EW), ('EW da BASE', dBASE)]:
    for part in ['TREINO', 'TESTE']:
        b = d[d['particao'] == part]
        lin.append({'versao': rot, 'particao': part, 'giro_ow_%mes': 100 * b['turnover'].mean() / 2,
                    'mediana_%': 100 * b['turnover'].median() / 2,
                    'min_%': 100 * b['turnover'].min() / 2, 'max_%': 100 * b['turnover'].max() / 2})
print(pd.DataFrame(lin).round(3).to_string(index=False))
print('\nESPERADO no enunciado: V3 gira pouco, proximo do EW (6,4%). MEDIDO acima.')
gv3 = TE(d3)['turnover'].mean() / 2
print(f'V3 no TESTE: {100 * gv3:.3f}%/mes contra {100 * TE(EW)["turnover"].mean() / 2:.3f}% do EW e '
      f'{100 * TE(V2A)["turnover"].mean() / 2:.3f}% de V2-A.')
c1v1 = pd.read_parquet(DIR_INT + r'\c1_carteira.parquet')
v2c = pd.read_parquet(DIR_INT + r'\v2_carteira.parquet')
nat = pd.DataFrame({'mes': meses_dec})
nat['particao'] = nat['mes'].map(part_por_mes)
nat['V1'] = nat['mes'].map(c1v1.groupby('mes').size())
nat['V2'] = nat['mes'].map(v2c.groupby('mes').size())
nat['V3'] = nat['mes'].map(n_v3)
nat['base'] = nat['mes'].map(n_base)
nat['elegiveis'] = nat['mes'].map(n_el)
print('\nnumero de ativos por mes:')
print(nat.groupby('particao')[['V1', 'V2', 'V3', 'base', 'elegiveis']].agg(
    ['median', 'min', 'max']).round(1).to_string())

# ==================================================================== (e) excedente
print('\n' + '=' * 128)
print('(e) EXCEDENTE SOBRE O INDICE INTERNO E SOBRE O EW, com t de Newey-West')
print('=' * 128)
# o EW comparavel e o do MESMO recorte de meses (TESTE), nao os primeiros 103 da serie inteira
EW_TE = TE(EW)
ew_bruto_te = EW_TE['bruto'].to_numpy()
ew_liq_te = liq(EW_TE, 50).to_numpy()
lin = []
for rot, d in [('V1-A', V1A), ('V2-A', V2A), ('V3', d3)]:
    b = TE(d)
    assert list(b['mes']) == list(EW_TE['mes']), 'meses desalinhados entre a versao e o EW'
    for tipo, r in [('bruto', b['bruto'].to_numpy()), ('liquido50', liq(b, 50).to_numpy())]:
        x = r - b['ret_indice'].to_numpy()
        mu, se, t = nw_t(x, 3)
        _, _, t1 = nw_t(x, 1); _, _, t6 = nw_t(x, 6)
        y = (r - ew_liq_te) if tipo == 'liquido50' else (r - ew_bruto_te)
        mu2, se2, t2 = nw_t(y, 3)
        lin.append({'versao': rot, 'tipo': tipo, 'exc vs indice %mes': 100 * mu,
                    't NW1': t1, 't NW3': t, 't NW6': t6,
                    'exc vs EW %mes': 100 * mu2, 't NW3 vs EW': t2})
print(pd.DataFrame(lin).round(4).to_string(index=False))

# ==================================================================== (f) aleatorias
print('\n' + '=' * 128)
print('(f) PERCENTIL NAS 200 ALEATORIAS E NAS 200 EXCLUSOES ALEATORIAS')
print('=' * 128)
print('COMPARADOR NOVO E OBRIGATORIO -- EXCLUSAO ALEATORIA: 200 carteiras que removem 10% da MESMA')
print('base ao acaso em cada mes e compram o resto em peso igual. E o unico teste que separa')
print('"o score identifica perdedores" de "remover 10% de qualquer coisa ajuda".')
idx_por_mes = {m: g.index.to_numpy() for m, g in sc.groupby('mes')}
isins_all = sorted(sc['CODISI'].unique())
pos_i = {c: i for i, c in enumerate(isins_all)}
pos_m = {m: i for i, m in enumerate(meses_dec)}
cod_arr = sc['CODISI'].to_numpy()
rng = np.random.default_rng(SEMENTE)
W = np.zeros((len(meses_dec), len(isins_all)))
t0 = time.time()
dist_exc = []
for s in range(N_ALEAT):
    W[:] = 0.0
    for m, idxs in idx_por_mes.items():
        n = len(idxs)
        k = int(round(n / DECIL))
        fora = rng.choice(n, size=k, replace=False)
        keep = np.setdiff1d(np.arange(n), fora, assume_unique=False)
        i = pos_m[m]
        w = 1.0 / len(keep)
        for j in keep:
            W[i, pos_i[cod_arr[idxs[j]]]] = w
    pw = pd.DataFrame(W, index=meses_dec, columns=isins_all)
    pw['__CDI__'] = 0.0
    dr, _ = rodar(pw, f'ex{s}', sil=True)
    b = TE(dr)
    dist_exc.append((anual(liq(b, 0)), anual(liq(b, 50)), 100 * b['turnover'].mean() / 2))
dexc = pd.DataFrame(dist_exc, columns=['bruto', 'liq50', 'giro_ow'])
dist_exc = dexc['liq50'].to_numpy()
dist_exc_bruto = dexc['bruto'].to_numpy()
print(f'\n  {N_ALEAT} exclusoes aleatorias em {time.time() - t0:.1f}s')
v3_50 = anual(liq(TE(d3), 50)); v3_00 = anual(liq(TE(d3), 0))
print(f'  distribuicao (liq 50bps, %aa): min {dist_exc.min():.4f} | p5 {np.percentile(dist_exc, 5):.4f} | '
      f'p25 {np.percentile(dist_exc, 25):.4f} | mediana {np.median(dist_exc):.4f} | '
      f'p75 {np.percentile(dist_exc, 75):.4f} | p90 {np.percentile(dist_exc, 90):.4f} | '
      f'p95 {np.percentile(dist_exc, 95):.4f} | max {dist_exc.max():.4f}')
pct_v3 = 100 * float((dist_exc < v3_50).mean())
print(f'  V3 = {v3_50:.4f}%aa  ->  PERCENTIL {pct_v3:.1f} da distribuicao de exclusoes aleatorias')

print('\n  *** SEPARACAO OBRIGATORIA, e ela nao estava no enunciado: GIRO ou SELECAO? ***')
print('  A exclusao aleatoria RESSORTEIA todo mes, entao ela nao so exclui outra coisa -- ela')
print('  exclui algo DIFERENTE a cada mes, e paga giro por isso. A exclusao pelo score e')
print('  persistente (o score_v2 tem rho 0,72). Sem separar, o percentil a 50 bps confunde')
print('  "o score acha perdedores" com "o score acha os MESMOS perdedores todo mes".')
print(f'  giro one-way no TESTE: V3 {100 * TE(d3)["turnover"].mean() / 2:.3f}%/mes  |  '
      f'exclusoes aleatorias mediana {dexc["giro_ow"].median():.3f}%/mes  |  '
      f'EW da BASE {100 * TE(dBASE)["turnover"].mean() / 2:.3f}%/mes')
pct_v3_bruto = 100 * float((dist_exc_bruto < v3_00).mean())
print(f'\n  A CUSTO ZERO, onde o giro nao pesa:')
print(f'  distribuicao BRUTA: min {dist_exc_bruto.min():.4f} | p5 {np.percentile(dist_exc_bruto, 5):.4f} | '
      f'mediana {np.median(dist_exc_bruto):.4f} | p90 {np.percentile(dist_exc_bruto, 90):.4f} | '
      f'p95 {np.percentile(dist_exc_bruto, 95):.4f} | max {dist_exc_bruto.max():.4f}')
print(f'  V3 bruto = {v3_00:.4f}%aa  ->  PERCENTIL {pct_v3_bruto:.1f}')
print(f'  Decomposicao da vantagem de V3 sobre a exclusao aleatoria MEDIANA:')
print(f'     a 0 bps  (so selecao):        {v3_00 - np.median(dist_exc_bruto):+.4f} pp/ano')
print(f'     a 50 bps (selecao + giro):    {v3_50 - np.median(dist_exc):+.4f} pp/ano')
print(f'     parcela atribuivel ao GIRO:   '
      f'{(v3_50 - np.median(dist_exc)) - (v3_00 - np.median(dist_exc_bruto)):+.4f} pp/ano')
d1a = pd.read_parquet(DIR_INT + r'\d1_aleatorias.parquet')
v2a_ = pd.read_parquet(DIR_INT + r'\v2_aleatorias.parquet')
print(f'\n  Para referencia posicional apenas (construcao DIFERENTE, nao e teste valido para V3):')
print(f'  200 SELECOES aleatorias de V1: mediana {d1a["anual_50bps"].median():.4f}; de V2: '
      f'{v2a_["anual_50bps"].median():.4f}. V3 ({v3_50:.4f}) esta acima das duas distribuicoes')
print(f'  inteiras (max de V1 {d1a["anual_50bps"].max():.4f}, max de V2 {v2a_["anual_50bps"].max():.4f}),')
print('  mas isso compara carteiras de ~40 nomes com uma de ~227 -- e diferenca de construcao, nao')
print('  de selecao. O teste valido de V3 e a distribuicao de EXCLUSOES acima.')

print('\n' + '-' * 128)
print('S5 -- V3 SUPERA O PERCENTIL 90 DAS EXCLUSOES ALEATORIAS?')
print('-' * 128)
p90 = float(np.percentile(dist_exc, 90))
print(f'   V3 = {v3_50:.4f}%aa   |   p90 das exclusoes aleatorias = {p90:.4f}%aa')
p90b = float(np.percentile(dist_exc_bruto, 90))
print(f'   A CUSTO ZERO: V3 = {v3_00:.4f}%aa  |  p90 bruto = {p90b:.4f}%aa  -> '
      f'percentil {pct_v3_bruto:.1f}')
if v3_50 > p90:
    print(f'   SIM a 50 bps. V3 esta no percentil {pct_v3:.1f}.')
    if v3_00 > p90b:
        print(f'   E SIM tambem a custo ZERO (percentil {pct_v3_bruto:.1f}), onde o giro nao pesa --')
        print('   entao a vantagem NAO e apenas persistencia da exclusao: o score exclui ativos que')
        print('   de fato renderam menos, e nao so os mesmos ativos todo mes.')
    else:
        print(f'   MAS NAO a custo zero (percentil {pct_v3_bruto:.1f}, abaixo do p90 bruto). COM TODAS AS')
        print('   LETRAS: a vantagem de V3 sobre a exclusao aleatoria vem de GIRO, nao de selecao --')
        print('   o score exclui os MESMOS ativos todo mes, e isso custa menos que ressortear.')
    print('   Em qualquer caso e NAO-REFUTACAO: [F] o poder exigiria ~331 meses, e esta e a TERCEIRA')
    print('   arquitetura testada sobre o mesmo periodo (L53).')
else:
    print(f'   NAO. V3 esta no percentil {pct_v3:.1f}, ABAIXO do p90. COM TODAS AS LETRAS:')
    print('   A EXCLUSAO PELO SCORE NAO SE DISTINGUE DE EXCLUIR AO ACASO. O ganho de V3 sobre o EW')
    print('   da base vem de remover 10% de qualquer coisa, nao de remover o que o score aponta.')

# ==================================================================== (i) decomposicao
print('\n' + '=' * 128)
print('(i) DECOMPOSICAO DE V3 -- quanto vem do universo amplo e quanto da exclusao')
print('=' * 128)
med_exc = float(np.median(dist_exc)); med_exc_b = float(np.median(dist_exc_bruto))
ew50, ew00 = anual(liq(TE(EW), 50)), anual(liq(TE(EW), 0))
bs50, bs00 = anual(liq(TE(dBASE), 50)), anual(liq(TE(dBASE), 0))
lin = [
    {'etapa': '1. EW do universo ELEGIVEL (o comparador de D1)', 'bruto_%aa': ew00,
     'liq50_%aa': ew50, 'delta bruto': np.nan, 'delta liq50': np.nan},
    {'etapa': '2. EW da BASE de V3 (so com score_v2 disponivel)', 'bruto_%aa': bs00,
     'liq50_%aa': bs50, 'delta bruto': bs00 - ew00, 'delta liq50': bs50 - ew50},
    {'etapa': '3. BASE menos decil ALEATORIO (mediana das 200)', 'bruto_%aa': med_exc_b,
     'liq50_%aa': med_exc, 'delta bruto': med_exc_b - bs00, 'delta liq50': med_exc - bs50},
    {'etapa': '4. V3 = BASE menos decil PELO SCORE', 'bruto_%aa': v3_00,
     'liq50_%aa': v3_50, 'delta bruto': v3_00 - med_exc_b, 'delta liq50': v3_50 - med_exc},
]
print(pd.DataFrame(lin).round(4).to_string(index=False))
print('\nAs colunas BRUTO e LIQ50 estao lado a lado de proposito: a diferenca entre elas e giro, e sem')
print('as duas a etapa 3->4 confundiria selecao com persistencia da exclusao.')
print('\nleitura das quatro linhas, na ordem:')
print(f'  (1->2) restringir o universo aos que tem as 6 features: {bs00 - ew00:+.4f} pp/ano bruto '
      f'({bs50 - ew50:+.4f} liquido)')
print('        -- e o custo de D19, ja conhecido, e NAO tem nada a ver com o score.')
print(f'  (2->3) remover 10% AO ACASO da base: {med_exc_b - bs00:+.4f} pp/ano bruto '
      f'({med_exc - bs50:+.4f} liquido). A diferenca entre os dois numeros e o giro de ressortear.')
print(f'  (3->4) remover 10% PELO SCORE em vez de ao acaso: {v3_00 - med_exc_b:+.4f} pp/ano BRUTO '
      f'({v3_50 - med_exc:+.4f} liquido).')
print(f'        O numero BRUTO ({v3_00 - med_exc_b:+.4f}) e a contribuicao atribuivel a QUAL ativo o')
print(f'        score exclui. Os {(v3_50 - med_exc) - (v3_00 - med_exc_b):+.4f} pp/ano restantes vem de a')
print('        exclusao ser PERSISTENTE e portanto barata de manter -- que e uma propriedade real da')
print('        estrategia, mas nao e capacidade preditiva.')

# ==================================================================== Brinson
print('\n' + '=' * 128)
print('BRINSON -- V3 ao lado de V1 e V2 (TESTE)')
print('=' * 128)
sub_de = split.drop_duplicates('CODISI').set_index('CODISI')['subsetor_chave'].to_dict()
ret_de = {(m, c): r for m, c, r in zip(split['mes'], split['CODISI'], split['ret_1m'])}
eleg_mes = {m: g for m, g in split[split['elegivel'] & split['ret_valido']].groupby('mes')}
lin_br = []
tr = v3[v3['particao'] == 'TESTE']
for mes, g in tr.groupby('mes'):
    m1 = str(pd.Period(mes, freq='M') + 1)
    be_ = eleg_mes.get(m1)
    if be_ is None:
        continue
    rb_tot = float(be_['ret_1m'].mean())
    wb = be_.groupby('subsetor_chave').size() / len(be_)
    rb = be_.groupby('subsetor_chave')['ret_1m'].mean()
    gg = g.copy()
    gg['sub'] = gg['CODISI'].map(sub_de)
    gg['r'] = [ret_de.get((m1, c), np.nan) for c in gg['CODISI']]
    gg['r'] = gg['r'].fillna(0.0)
    wp = gg.groupby('sub')['peso'].sum()
    rp = gg.groupby('sub').apply(lambda h: float((h['peso'] * h['r']).sum() / h['peso'].sum()),
                                 include_groups=False)
    subs = sorted(set(wp.index) | set(wb.index))
    wpv = np.array([wp.get(s, 0.0) for s in subs]); wbv = np.array([wb.get(s, 0.0) for s in subs])
    rbv = np.array([rb.get(s, rb_tot) for s in subs]); rpv = np.array([rp.get(s, 0.0) for s in subs])
    lin_br.append({'versao': 'V3', 'mes': mes, 'alocacao': float(((wpv - wbv) * rbv).sum()),
                   'selecao': float((wpv * (rpv - rbv)).sum()),
                   'exc': float((wpv * rpv).sum()) - float((wbv * rbv).sum())})
br3 = pd.DataFrame(lin_br)
br3['residuo'] = br3['exc'] - (br3['alocacao'] + br3['selecao'])
lin = []
for nome, b in [('V1-A', d2v1[d2v1['versao'] == 'C2-A']), ('V1-B', d2v1[d2v1['versao'] == 'C2-B']),
                ('V2-A', v2br[v2br['versao'] == 'V2-A']), ('V2-B', v2br[v2br['versao'] == 'V2-B']),
                ('V3', br3)]:
    _, _, ta = nw_t(b['alocacao'], 3); _, _, ts = nw_t(b['selecao'], 3)
    lin.append({'versao': nome, 'meses': len(b), 'alocacao_%mes': 100 * b['alocacao'].mean(),
                't aloc': ta, 'selecao_%mes': 100 * b['selecao'].mean(), 't selec': ts,
                'excedente_%mes': 100 * b['exc'].mean(), 'residuo_max': float(b['residuo'].abs().max())})
print(pd.DataFrame(lin).round(6).to_string(index=False))
assert br3['residuo'].abs().max() < 1e-9, 'identidade de Brinson quebrou em V3'

# ==================================================================== (g) subperiodos
print('\n' + '=' * 128)
print('(g) D29-V3 -- SUBPERIODOS (liquido de 50 bps, anualizado)')
print('=' * 128)
lin = []
for ini, fim in SUBPERIODOS:
    f = lambda d: d[(d['mes'] >= ini) & (d['mes'] <= fim) & (d['particao'] == 'TESTE')]
    lin.append({'subperiodo': f'{ini} a {fim}', 'meses': len(f(V1A)),
                'V1-A': anual(liq(f(V1A), 50)), 'V2-A': anual(liq(f(V2A), 50)),
                'V3': anual(liq(f(d3), 50)), 'EW': anual(liq(f(EW), 50)),
                'indice': anual(f(V1A)['ret_indice']), 'CDI': anual(f(V1A)['ret_cdi'])})
sp = pd.DataFrame(lin)
print(sp.round(4).to_string(index=False))
sp['V3-indice'] = sp['V3'] - sp['indice']; sp['V3-EW'] = sp['V3'] - sp['EW']
print('\ndiferencas em pp/ano:')
print(sp[['subperiodo', 'V3-indice', 'V3-EW']].round(4).to_string(index=False))
print(f'\nV3 acima do indice em {int((sp["V3-indice"] > 0).sum())} de 3 subperiodos; '
      f'acima do EW em {int((sp["V3-EW"] > 0).sum())} de 3.')

# ==================================================================== (h) capacity
print('\n' + '=' * 128)
print('(h) CAPACITY DE V3')
print('=' * 128)
vol_de = {(m, c): v for m, c, v in zip(split['mes'], split['CODISI'], split['VOLTOT'])}
tv = v3[v3['particao'] == 'TESTE'].copy()
tv['VOLTOT'] = [vol_de.get((m, c), np.nan) for m, c in zip(tv['mes'], tv['CODISI'])]
lin = []
for p in (0.05, 0.10):
    tv['cap'] = p * tv['VOLTOT'] / tv['peso']
    pm = tv.groupby('mes')['cap'].min()
    ib = tv.loc[tv['cap'].idxmin()]
    med = tv.groupby('mes')['VOLTOT'].median(); nn = tv.groupby('mes').size()
    lin.append({'participacao': f'{p:.0%}', 'V3 min-do-mes mediana R$': float(pm.median()),
                'V3 min-do-mes p10 R$': float(pm.quantile(.10)),
                'V3 pelo VOLTOT mediano R$': float((p * med * nn).median()),
                'mais restritivo': f'{ib["CODNEG"]} ({ib["mes"]})'})
print(pd.DataFrame(lin).to_string(index=False, float_format=lambda x: f'{x:,.0f}'))
print('V1: min-do-mes mediana R$ 128.251 (5%) | V2: R$ 216.710 (5%)')
print(f'V3 carrega ~{n_v3[[m for m in meses_dec if part_por_mes[m] == "TESTE"]].median():.0f} nomes, '
      'entao o peso individual e ~6x menor que o de V1/V2 e a capacity sobe na mesma proporcao.')

# ==================================================================== gravacao
x = d3[['mes', 'mes_ret', 'particao', 'bruto', 'turnover']].copy()
x['versao'] = 'V3'
for bps in NIVEIS_BPS:
    x[f'liq_{bps}bps'] = liq(d3, bps)
x.to_parquet(DIR_INT + r'\v3_backtest.parquet', index=False)
v3.to_parquet(DIR_INT + r'\v3_carteira.parquet', index=False)
br3.to_parquet(DIR_INT + r'\v3_brinson.parquet', index=False)
dexc.assign(sorteio=np.arange(N_ALEAT)).to_parquet(
    DIR_INT + r'\v3_exclusoes_aleatorias.parquet', index=False)
print('\n' + '=' * 128)
print(f'GRAVADO v3_backtest.parquet | v3_carteira.parquet {v3.shape} | v3_brinson.parquet {br3.shape} | '
      f'v3_exclusoes_aleatorias.parquet ({N_ALEAT}, 4: bruto, liq50, giro, sorteio)')
print('V3 CONCLUIDA. V1 e V2 intactas. D32: ULTIMA variante desta rodada.')
print('=' * 128)


V3 (D31) -- EXCLUSAO do decil inferior de score_v2 sobre carteira ampla em peso igual.
D32: esta e a ULTIMA variante desta rodada. Nao havera V4 antes da auditoria do conselho.
V1, V2 e V3 lado a lado em toda tabela. V1 e V2 INTACTAS.

S1 -- V1 E V2 INTACTAS
b4b5_previsoes.parquet (V1) SHA-256 = 5f67e19fa21e1b22f8d29064fca75ba06c498e2b62ac6b87ae67f332eb385e67
registrado no MODEL_SPEC de B6       = 5f67e19fa21e1b22f8d29064fca75ba06c498e2b62ac6b87ae67f332eb385e67
v2_scores.parquet (V2) SHA-256 = c887216bcd6bb5ed86f508a689738a4eb29779f7ebabf948a5e3e84df44305ac
   (v2_scores foi gravado na sessao de V2 e nao e reescrito aqui -- esta celula so LE)

arquivos de V1/V2 que esta celula LE e NAO grava (10):
   c1_carteira.parquet  sha 6a945491dffe
   c2_backtest.parquet  sha 5b6f689f2b20
   c3_curva_custo.parquet  sha 925143cbc791
   d1_tabela_final.parquet  sha 4eaf93ab5b69
   d2_brinson.parquet  sha e2cb55047da7
   d4_metricas_finais.parquet  sha 80aa856ba8c1
   v2_backtest.parquet  sha 8919ae

### Correção pós-conselho (plano item 3) — cascata de V3 reancorada no EW da própria base; manchete +2,05 → +0,87 (t NW3 +0,864); giro é CUSTO (−0,52), não bônus

A exclusão re-sorteada mensalmente (placebo) gira 14,43%/mês one-way e não é uma alternativa de investimento; a alternativa passiva real é não excluir nada (EW da base, giro 4,92%/mês). Contra ela V3 gira **mais** (8,97%) e o termo de giro vira **custo**: seleção +1,3937 bruto − 0,5217 de giro = **+0,8720 pp/ano líquido (t NW3 +0,864)**; a custo zero, +0,0892%/mês (t +1,581); contra o EW elegível, **−0,2940 pp/ano**. O +2,05 permanece válido apenas contra o placebo. Célula executada em 2026-08-16; output real abaixo.

In [ ]:
# Correcao pos-conselho (plano item 3) -- cascata de V3 reancorada no EW da PROPRIA BASE.
# Tres ancoras lado a lado: placebo re-sorteado (a existente), EW da base de V3, EW dos elegiveis.
# Manchete: +2,05 (vs placebo) -> +0,87 (t NW3 +0,864) vs EW da base; giro e CUSTO (-0,52), nao bonus.
# Autocontido: le parquets de ..\intermediario\ e re-roda motor.rodar_backtest SO EM MEMORIA (nada gravado).
import sys

import numpy as np
import pandas as pd

if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8')

DIR_INT = r'..\intermediario'
if r'..\src' not in sys.path:
    sys.path.insert(0, r'..\src')
import motor  # noqa: E402


def nw_t(x, lags=3):
    x = np.asarray(pd.Series(x).dropna(), dtype=float)
    n = len(x); mu = x.mean(); d = x - mu
    v = float(d @ d) / n
    for l in range(1, lags + 1):
        v += 2.0 * (1.0 - l / (lags + 1.0)) * (float(d[l:] @ d[:-l]) / n)
    se = np.sqrt(v / n)
    return mu, se, mu / se


def anual(r):
    r = np.asarray(pd.Series(r).dropna(), dtype=float)
    return 100.0 * ((np.prod(1.0 + r)) ** (12.0 / len(r)) - 1.0)


# ---- insumos (identicos a celula de V3/D31) ----
sc = pd.read_parquet(DIR_INT + r'\v2_scores.parquet')
bench = pd.read_parquet(DIR_INT + r'\a6_benchmark_e_rf.parquet')
split = pd.read_parquet(DIR_INT + r'\a7_split.parquet')
c3v1 = pd.read_parquet(DIR_INT + r'\c3_curva_custo.parquet')
v3bt = pd.read_parquet(DIR_INT + r'\v3_backtest.parquet')
vexc = pd.read_parquet(DIR_INT + r'\v3_exclusoes_aleatorias.parquet')
sc['CODISI'] = sc['CODISI'].astype(str)
split['CODISI'] = split['CODISI'].astype(str)

meses_all = sorted(bench['mes'].unique())
data_de_mes = {m: pd.Period(m, freq='M').to_timestamp(how='end').normalize() for m in meses_all}
mes_de_data = {v: k for k, v in data_de_mes.items()}
meses_dec = sorted(sc['mes'].unique())
part_por_mes = sc.groupby('mes')['particao'].first().to_dict()
bi = bench.set_index('mes')

wide = split.pivot_table(index='mes', columns='CODISI', values='ret_1m', aggfunc='first').reindex(meses_all)
wide.index = pd.DatetimeIndex([data_de_mes[m] for m in meses_all])
cdi = bi['ret_cdi'].reindex(meses_all); cdi.index = wide.index
wide['__CDI__'] = cdi.values
CAIXA = wide['__CDI__']


def rodar(p_df, rot):
    p = p_df.copy()
    cols = [c for c in p.columns if (p[c] != 0).any()]
    p = p[cols].fillna(0.0)
    p.index = pd.DatetimeIndex([data_de_mes[m] for m in p.index])
    o = motor.rodar_backtest(p, wide[cols], custo_bps=0.0, retornos_caixa=CAIXA)
    rb = o['retornos_brutos']; tv = o['turnover'].reindex(rb.index)
    d = pd.DataFrame({'mes_ret': [mes_de_data[i] for i in rb.index],
                      'bruto': rb.values, 'turnover': tv.values})
    d['mes'] = [str(pd.Period(m, freq='M') - 1) for m in d['mes_ret']]
    d['particao'] = d['mes'].map(part_por_mes)
    print(f'  [{rot}] ativos {len(cols)}  meses {len(d)}')
    return d


def liq(d, bps):
    return d['bruto'] - d['turnover'] * (bps / 10000.0)


def matriz(df, col='peso'):
    m = df.pivot_table(index='mes', columns='CODISI', values=col, aggfunc='sum').reindex(meses_dec)
    m = m.fillna(0.0); m['__CDI__'] = 0.0
    return m


TE = lambda d: d[d['particao'] == 'TESTE'].sort_values('mes').reset_index(drop=True)

# ---- EW da BASE de V3 (identico ao dBASE da celula de V3) ----
n_base = sc.groupby('mes').size()
base_ew = sc.merge(n_base.rename('n'), on='mes')
base_ew['peso'] = 1.0 / base_ew['n']
dBASE = rodar(matriz(base_ew), 'EW da BASE (sem exclusao)')

# ---- series do TESTE, alinhadas por mes ----
b_base = TE(dBASE)
b_v3 = TE(v3bt)
b_ew = TE(c3v1[c3v1['versao'] == 'EW_universo'])
assert list(b_base['mes']) == list(b_v3['mes']) == list(b_ew['mes']), 'meses desalinhados'
assert len(b_v3) == 103, f'TESTE deveria ter 103 meses, tem {len(b_v3)}'

bs00, bs50 = anual(b_base['bruto']), anual(liq(b_base, 50))
v3_00, v3_50 = anual(b_v3['bruto']), anual(liq(b_v3, 50))
ew00, ew50 = anual(b_ew['bruto']), anual(liq(b_ew, 50))
med_pl_b = float(vexc['bruto'].median())
med_pl_50 = float(vexc['liq50'].median())

sel_base = v3_00 - bs00
sel_plac = v3_00 - med_pl_b
liq_base = v3_50 - bs50
giro_base = liq_base - sel_base
exc_ew = v3_50 - ew50

x_liq = (liq(b_v3, 50) - liq(b_base, 50)).to_numpy()
x_bru = (b_v3['bruto'] - b_base['bruto']).to_numpy()
mu_l, _, t_l = nw_t(x_liq, 3)
mu_b, _, t_b = nw_t(x_bru, 3)

giro_v3 = 100 * b_v3['turnover'].mean() / 2
giro_bs = 100 * b_base['turnover'].mean() / 2
giro_pl = float(vexc['giro_ow'].median())

print('\n=== nos da cascata (TESTE, 103 meses, %aa geometrico) ===')
print(f'EW elegivel     bruto {ew00:.4f}  liq50 {ew50:.4f}   (alvo liq50 5,9073)')
print(f'EW da BASE      bruto {bs00:.4f}  liq50 {bs50:.4f}   (alvo 5,3589 / 4,7413)')
print(f'placebo mediana bruto {med_pl_b:.4f}  liq50 {med_pl_50:.4f}')
print(f'V3              bruto {v3_00:.4f}  liq50 {v3_50:.4f}   (alvo 6,7526 / 5,6133)')

print('\n=== decomposicao reancorada ===')
print(f'selecao bruta vs BASE     {sel_base:+.4f}  (alvo +1,3937)')
print(f'selecao bruta vs PLACEBO  {sel_plac:+.4f}  (alvo +1,377)')
print(f'giro vs BASE              {giro_base:+.4f}  (alvo -0,5217)')
print(f'liquido vs BASE           {liq_base:+.4f}  (alvo +0,8720)   t NW3 {t_l:+.3f} (alvo +0,864)')
print(f'excedente mensal a custo zero vs BASE  {100 * mu_b:+.4f} %/mes (alvo +0,0892)  '
      f't NW3 {t_b:+.3f} (alvo +1,581)')
print(f'vs EW ELEGIVEL (liq50)    {exc_ew:+.4f}  (alvo -0,2940)')
print(f'\ngiro one-way TESTE: V3 {giro_v3:.2f}%/mes | base {giro_bs:.2f}%/mes (alvo 4,92) | '
      f'placebo {giro_pl:.2f}%/mes (alvo 14,43)')

print('\n=== aritmetica de sanidade ===')
print(f'{v3_50:.4f} - {bs50:.4f} = {v3_50 - bs50:+.4f}  (deve ser +0,8720)')
print(f'{v3_00:.4f} - {bs00:.4f} = {v3_00 - bs00:+.4f}  (deve ser +1,3937)')
print(f'{liq_base:+.4f} - {sel_base:+.4f} = {giro_base:+.4f}  (deve ser -0,5217)')

ok = (abs(bs00 - 5.3589) < 5e-4 and abs(bs50 - 4.7413) < 5e-4
      and abs(sel_base - 1.3937) < 5e-4 and abs(sel_plac - 1.377) < 5e-3
      and abs(giro_base + 0.5217) < 5e-4 and abs(liq_base - 0.8720) < 5e-4
      and abs(100 * mu_b - 0.0892) < 5e-4 and abs(exc_ew + 0.2940) < 5e-4
      and abs(t_l - 0.864) < 5e-3 and abs(t_b - 1.581) < 5e-3)
assert ok, 'DIVERGENCIA nos alvos da cascata reancorada -- ver impressao acima'
print('\nVEREDITO: PASSOU -- todos os alvos reproduzidos')
print('\nNOTA ESTRUTURAL (mantida do conselho): os degraus sao diferencas de retornos anuais')
print('GEOMETRICOS -- nao sao aditivos mes a mes e os passos nao comutam (0,01-0,12 pp/ano).')
print('A expressao "bonus de giro" SAI do texto: contra a base, o giro e custo (-0,5217 pp/ano).')


  [EW da BASE (sem exclusao)] ativos 746  meses 296

=== nos da cascata (TESTE, 103 meses, %aa geometrico) ===
EW elegivel     bruto 6.7162  liq50 5.9073   (alvo liq50 5,9073)
EW da BASE      bruto 5.3589  liq50 4.7413   (alvo 5,3589 / 4,7413)
placebo mediana bruto 5.3754  liq50 3.5647
V3              bruto 6.7526  liq50 5.6133   (alvo 6,7526 / 5,6133)

=== decomposicao reancorada ===
selecao bruta vs BASE     +1.3938  (alvo +1,3937)
selecao bruta vs PLACEBO  +1.3773  (alvo +1,377)
giro vs BASE              -0.5217  (alvo -0,5217)
liquido vs BASE           +0.8721  (alvo +0,8720)   t NW3 +0.864 (alvo +0,864)
excedente mensal a custo zero vs BASE  +0.0892 %/mes (alvo +0,0892)  t NW3 +1.581 (alvo +1,581)
vs EW ELEGIVEL (liq50)    -0.2940  (alvo -0,2940)

giro one-way TESTE: V3 8.97%/mes | base 4.92%/mes (alvo 4,92) | placebo 14.43%/mes (alvo 14,43)

=== aritmetica de sanidade ===
5.6133 - 4.7413 = +0.8721  (deve ser +0,8720)
6.7526 - 5.3589 = +1.3938  (deve ser +1,3937)
+0.8721 - +1.3938

## E1 -- figuras do dossie (10 PNG, outputs\, 150 dpi)

Cada figura le direto dos parquets gravados -- nenhum numero digitado a mao (S1, impresso figura a
figura). Eixos de retorno/percentual incluem o zero; as duas curvas acumuladas (F3, F5) tem eixo Y
comecando em 0 (S2). F5 e F6 confirmados usando o mesmo custo de 50 bps (S3).

**Achado de verificacao em F1, nao pedido pelo enunciado:** a primeira tentativa de cobertura
setorial (media direta sobre linhas mes-a-mes) deu 19,7%/99,5%, divergindo do [F] de A5
(19,2%/98,3%). Investigado antes de aceitar como arredondamento: a metodologia certa dedup por
CODISI dentro do ano; recalculada assim, bate com o [F] em 0,02pp -- com um `assert` que pararia a
celula se voltasse a divergir.

F5 e a figura central do projeto: curva acumulada do teste, base 100, liquida de 50bps, com V1-A,
V2-A, V3, EW elegivel, indice interno e CDI -- as tres arquiteturas testadas contra os tres
comparadores, no mesmo periodo e com o mesmo motor.


In [1]:
import sys

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8')

pd.set_option('display.width', 250)

RAIZ = r'C:\Users\lucca\quant2026'
DIR_INT = RAIZ + r'\intermediario'
DIR_OUT = RAIZ + r'\outputs'
if RAIZ + r'\src' not in sys.path:
    sys.path.insert(0, RAIZ + r'\src')
import motor  # noqa: E402

# ------------------------------------------------------------------ estilo sobrio, legivel em P&B
plt.rcParams.update({
    'figure.dpi': 150, 'savefig.dpi': 150, 'font.size': 10.5, 'font.family': 'DejaVu Sans',
    'axes.edgecolor': '#333333', 'axes.labelcolor': '#1a1a1a', 'text.color': '#1a1a1a',
    'xtick.color': '#333333', 'ytick.color': '#333333', 'axes.grid': True,
    'grid.color': '#d9d9d9', 'grid.linewidth': 0.6, 'axes.axisbelow': True,
    'legend.frameon': False, 'axes.spines.top': False, 'axes.spines.right': False,
})
COR = {'V1': '#4d4d4d', 'V2': '#2c6e91', 'V3': '#a4405e', 'EW': '#5a8a3c',
       'indice': '#111111', 'cdi': '#b08a2e', 'ibov': '#8858a8', 'base': '#8c8c8c'}
LS = {'V1': '-', 'V2': '--', 'V3': '-.', 'EW': ':', 'indice': '-', 'cdi': '--', 'ibov': ':'}
MK = {'V1': 'o', 'V2': 's', 'V3': '^', 'EW': 'D'}

print('=' * 122)
print('E1 -- FIGURAS DO DOSSIE. PNG, 150 dpi, sem titulo na imagem, legendas em portugues.')
print('S1: cada figura imprime de qual(is) parquet(s) leu. Nenhum numero digitado a mao.')
print('=' * 122)


# ==================================================================== helpers
def _pearson(x, y):
    x = np.asarray(x, dtype=float); y = np.asarray(y, dtype=float)
    ok = np.isfinite(x) & np.isfinite(y)
    x, y = x[ok] - x[ok].mean(), y[ok] - y[ok].mean()
    den = np.sqrt(float(x @ x) * float(y @ y))
    return float(x @ y) / den if den > 0 else np.nan


def spearman(x, y):
    rx = pd.Series(np.asarray(x, dtype=float)).rank(method='average').to_numpy()
    ry = pd.Series(np.asarray(y, dtype=float)).rank(method='average').to_numpy()
    return _pearson(rx, ry)


def nw_t(x, lags=3):
    x = np.asarray(pd.Series(x).dropna(), dtype=float)
    n = len(x); mu = x.mean(); d = x - mu
    v = float(d @ d) / n
    for l in range(1, lags + 1):
        v += 2.0 * (1.0 - l / (lags + 1.0)) * (float(d[l:] @ d[:-l]) / n)
    return mu, np.sqrt(v / n), mu / np.sqrt(v / n)


def anual(r):
    r = np.asarray(pd.Series(r).dropna(), dtype=float)
    return 100.0 * ((np.prod(1.0 + r)) ** (12.0 / len(r)) - 1.0)


def salvar(fig, nome):
    caminho = DIR_OUT + '\\' + nome
    fig.savefig(caminho, bbox_inches='tight', facecolor='white')
    plt.close(fig)
    import os
    tam = os.path.getsize(caminho)
    print(f'  GRAVADO {caminho}  ({tam / 1024:.1f} KB)')
    return caminho


arquivos_gerados = []

# ==================================================================== insumos gerais
bench = pd.read_parquet(DIR_INT + r'\a6_benchmark_e_rf.parquet')
split = pd.read_parquet(DIR_INT + r'\a7_split.parquet')
bi = bench.set_index('mes')
meses_all = sorted(bench['mes'].unique())
data_de_mes = {m: pd.Period(m, freq='M').to_timestamp(how='end').normalize() for m in meses_all}
mes_de_data = {v: k for k, v in data_de_mes.items()}

# ==================================================================== F1 -- cobertura setorial
print('\n[F1] fonte: intermediario\\a5_com_setor.parquet')
a5 = pd.read_parquet(DIR_INT + r'\a5_com_setor.parquet')
a5['ano'] = a5['mes'].str[:4].astype(int)
a5['coberto'] = a5['subsetor_chave'] != 'SEM_SETOR'
# dedup por CODISI dentro do ano (um ativo conta uma vez, nao uma vez por mes) -- metodologia de A5,
# reconferida aqui: reproduz os numeros [F] (19,2% / 48,0% / 79,0% / 98,3% em ativos; 72,7% / 99,8% em
# volume) com desvio 0,00pp. A media direta sobre as linhas mes-a-mes (sem dedup) diverge ~0,5-1,2pp
# nas pontas porque pondera ativos de vida longa mais vezes -- descartada em favor da que bate o [F].
g = a5.groupby(['ano', 'CODISI']).agg(coberto=('coberto', 'first'), VOLTOT=('VOLTOT', 'sum')).reset_index()
cov_n = g.groupby('ano')['coberto'].mean() * 100
cov_v = g.groupby('ano').apply(
    lambda d: 100 * float(d.loc[d['coberto'], 'VOLTOT'].sum() / d['VOLTOT'].sum()), include_groups=False)
print(f'  1995: nomes {cov_n.loc[1995]:.2f}% (vol {cov_v.loc[1995]:.2f}%) | '
      f'2026: nomes {cov_n.loc[2026]:.2f}% (vol {cov_v.loc[2026]:.2f}%)')
print(f'  conferencia contra [F] (A5): 19,2/98,3 em ativos e 72,7/99,8 em volume -- '
      f'desvio {abs(cov_n.loc[1995] - 19.2):.2f}/{abs(cov_n.loc[2026] - 98.3):.2f}pp')
assert abs(cov_n.loc[1995] - 19.2) < 0.1 and abs(cov_n.loc[2026] - 98.3) < 0.1, \
    'cobertura recalculada nao bate com o [F] de A5 -- PARAR'

fig, ax = plt.subplots(figsize=(8.5, 4.6))
ax.plot(cov_n.index, cov_n.values, color=COR['V1'], ls='-', marker='o', ms=3.2, lw=1.6,
        label='cobertura em nº de ativos')
ax.plot(cov_v.index, cov_v.values, color=COR['EW'], ls='--', marker='s', ms=3.2, lw=1.6,
        label='cobertura em volume negociado')
ax.annotate(f'{cov_n.loc[1995]:.1f}%', xy=(1995, cov_n.loc[1995]), xytext=(1998, cov_n.loc[1995] - 12),
            arrowprops=dict(arrowstyle='->', color='#555555', lw=0.9), fontsize=9)
ax.annotate(f'{cov_n.loc[2026]:.1f}%', xy=(2026, cov_n.loc[2026]), xytext=(2015, cov_n.loc[2026] - 15),
            arrowprops=dict(arrowstyle='->', color='#555555', lw=0.9), fontsize=9)
ax.set_ylim(0, 105)
ax.set_xlabel('ano'); ax.set_ylabel('cobertura do mapa setorial (%)')
ax.legend(loc='lower right')
salvar(fig, 'F1_cobertura_setorial.png')
arquivos_gerados.append(('F1_cobertura_setorial.png',
    'Cobertura do mapa setorial por ano, em número de ativos e em volume negociado, 1995–2026',
    'a5_com_setor.parquet',
    'Mostra que o mapa setorial é o de hoje e só alcança o passado onde a empresa sobreviveu — curva de sobrevivência, e razão de existir o generalista.',
    'Não mostra a qualidade da classificação, só a cobertura.'))

# ==================================================================== F2 -- eventos societarios
print('\n[F2] fonte: intermediario\\a2_eventos_societarios.parquet')
a2e = pd.read_parquet(DIR_INT + r'\a2_eventos_societarios.parquet')
a2e['ano'] = a2e['data'].dt.year
TIPOS = ['ganha_G', 'ganha_B', 'fatcot', 'extensao_defasagem_1']
comp = pd.DataFrame(index=sorted(a2e['ano'].unique()))
for t in TIPOS:
    comp[t] = a2e[a2e['canal'].str.contains(t, regex=False)].groupby('ano').size()
comp = comp.fillna(0).astype(int)
print(f'  total de eventos: {a2e.shape[0]}; por tipo (canal contem o rotulo, combos contam nos dois): '
      f'{comp.sum().to_dict()}')

fig, ax = plt.subplots(figsize=(9.5, 4.6))
bottom = np.zeros(len(comp))
cores2 = ['#4d4d4d', '#2c6e91', '#a4405e', '#c9a227']
haches = ['', '///', '...', 'xx']
for t, c, h in zip(TIPOS, cores2, haches):
    ax.bar(comp.index, comp[t], bottom=bottom, color=c, hatch=h, edgecolor='white', linewidth=0.4,
           label=t, width=0.8)
    bottom += comp[t].to_numpy()
ax.set_xlabel('ano'); ax.set_ylabel('nº de eventos')
ax.legend(loc='upper left', ncol=2)
salvar(fig, 'F2_eventos_societarios.png')
arquivos_gerados.append(('F2_eventos_societarios.png',
    'Eventos societários detectados por ano e por tipo (grupamento, bonificação, mudança de FATCOT, extensão de defasagem)',
    'a2_eventos_societarios.parquet',
    'Mostra a densidade de desdobramentos, grupamentos e bonificações capturados pelo ajuste societário ao longo do tempo.',
    'Não mostra proventos em dinheiro (D/J/R/S), que o COTAHIST não marca e que não são neutralizados (L08).'))

# ==================================================================== F3 -- indice vs CDI vs IBOV
print('\n[F3] fonte: intermediario\\a6_benchmark_e_rf.parquet')
f3 = bench[(bench['mes'] >= '2010-01') & (bench['mes'] <= '2025-12')].sort_values('mes').reset_index(drop=True)
print(f'  {len(f3)} meses, {f3["mes"].min()} a {f3["mes"].max()}')
w_idx = 100 * np.cumprod(1 + f3['ret_indice'].to_numpy())
w_cdi = 100 * np.cumprod(1 + f3['ret_cdi'].to_numpy())
w_ibov = 100 * np.cumprod(1 + f3['ret_ibov'].fillna(0).to_numpy())
w_ibov[f3['ret_ibov'].isna().to_numpy()] = np.nan
x = np.arange(len(f3))

fig, ax = plt.subplots(figsize=(9.5, 4.8))
ax.plot(x, w_idx, color=COR['indice'], ls='-', lw=1.7, label='índice interno (D12), peso igual')
ax.plot(x, w_cdi, color=COR['cdi'], ls='--', lw=1.5, label='CDI (D13)')
ax.plot(x, w_ibov, color=COR['ibov'], ls=':', lw=1.7, label='IBOV (secundário, fonte externa)')
ticks = [i for i, m in enumerate(f3['mes']) if m.endswith('-01') and int(m[:4]) % 3 == 0]
ax.set_xticks(ticks); ax.set_xticklabels([f3['mes'].iloc[i][:4] for i in ticks])
ax.set_ylim(0, None)
ax.set_xlabel('ano'); ax.set_ylabel('riqueza acumulada (base 100 em 2010-01)')
ax.legend(loc='upper left')
salvar(fig, 'F3_indice_cdi_ibov.png')
arquivos_gerados.append(('F3_indice_cdi_ibov.png',
    'Índice interno, CDI e IBOV, riqueza acumulada em base 100, 2010–2025',
    'a6_benchmark_e_rf.parquet',
    'Mostra a trajetória de longo prazo do benchmark interno price-only contra a taxa livre de risco e o IBOV.',
    'Não mostra custo de transação nem seleção de ativos — é o benchmark, não uma estratégia.'))

# ==================================================================== F4 -- IC mensal e especialistas
print('\n[F4] fontes: intermediario\\b4b5_previsoes.parquet e intermediario\\v2_scores.parquet')
prev = pd.read_parquet(DIR_INT + r'\b4b5_previsoes.parquet')
v2s = pd.read_parquet(DIR_INT + r'\v2_scores.parquet')


def ic_por_mes(df, col):
    out = []
    for m, g in df.groupby('mes'):
        h = g[[col, 'alfa_fut']].dropna()
        if len(h) < 10:
            continue
        out.append({'mes': m, 'ic': spearman(h[col], h['alfa_fut'])})
    return pd.DataFrame(out).sort_values('mes').reset_index(drop=True)


ic_v1 = ic_por_mes(prev, 'score')
ic_v2 = ic_por_mes(v2s, 'score_v2')
n_esp = prev[prev['modelo'] != 'GENERALISTA'].groupby('mes')['modelo'].nunique()
meses_dec = sorted(prev['mes'].unique())
part_por_mes = prev.groupby('mes')['particao'].first().to_dict()
teste_meses = [m for m in meses_dec if part_por_mes[m] == 'TESTE']
ic_v1_t = ic_v1.set_index('mes').reindex(meses_dec)['ic']
ic_v2_t = ic_v2.set_index('mes').reindex(meses_dec)['ic']
mm_v1 = ic_v1_t.rolling(12, min_periods=6).mean()
mm_v2 = ic_v2_t.rolling(12, min_periods=6).mean()
nesp_s = n_esp.reindex(meses_dec)
idx_teste = [meses_dec.index(m) for m in teste_meses]
print(f'  {len(teste_meses)} meses de teste; IC V1 medio no teste {ic_v1_t.reindex(teste_meses).mean():.4f}; '
      f'IC V2 medio {ic_v2_t.reindex(teste_meses).mean():.4f}')

x = np.arange(len(teste_meses))
fig, ax1 = plt.subplots(figsize=(10, 5))
ax1.plot(x, mm_v1.reindex(teste_meses).to_numpy(), color=COR['V1'], ls='-', lw=1.7,
         label='IC V1 (média móvel 12m)')
ax1.plot(x, mm_v2.reindex(teste_meses).to_numpy(), color=COR['V2'], ls='--', lw=1.7,
         label='IC V2 (média móvel 12m)')
ax1.axhline(0, color='#999999', lw=0.8)
ax1.set_ylabel('IC (Spearman, média móvel de 12 meses)')
ax1.set_xlabel('mês de decisão (período de teste)')
ticks = [i for i, m in enumerate(teste_meses) if m.endswith('-01')]
ax1.set_xticks(ticks); ax1.set_xticklabels([teste_meses[i][:4] for i in ticks])
ax2 = ax1.twinx()
ax2.bar(x, nesp_s.reindex(teste_meses).to_numpy(), color=COR['EW'], alpha=0.18, width=0.9,
        label='nº de especialistas ativos (eixo direito)')
ax2.set_ylabel('nº de especialistas ativos')
ax2.set_ylim(0, nesp_s.max() * 2.2)
ax2.grid(False)
l1, lb1 = ax1.get_legend_handles_labels(); l2, lb2 = ax2.get_legend_handles_labels()
ax1.legend(l1 + l2, lb1 + lb2, loc='upper left', fontsize=9)
salvar(fig, 'F4_ic_mensal_especialistas.png')
arquivos_gerados.append(('F4_ic_mensal_especialistas.png',
    'IC mensal de V1 e V2 (média móvel de 12 meses) e nº de especialistas ativos, período de teste',
    'b4b5_previsoes.parquet e v2_scores.parquet',
    'Mostra a capacidade de ordenação de cada versão mês a mês e como a arquitetura se adensa (mais especialistas) ao longo do teste.',
    'Não mostra retorno de carteira, que depende de seleção, peso e custo — ver F5.'))

# ==================================================================== dados de backtest para F5, F6, F10
print('\ncarregando series de backtest para F5/F6/F9/F10: c3_curva_custo, v2_backtest, v3_backtest')
c3 = pd.read_parquet(DIR_INT + r'\c3_curva_custo.parquet')
v2bt = pd.read_parquet(DIR_INT + r'\v2_backtest.parquet')
v3bt = pd.read_parquet(DIR_INT + r'\v3_backtest.parquet')

SERIES = {
    'V1': c3[c3['versao'] == 'C2-A'].sort_values('mes').reset_index(drop=True),
    'V2': v2bt[v2bt['versao'] == 'V2-A'].sort_values('mes').reset_index(drop=True),
    'V3': v3bt[v3bt['versao'] == 'V3'].sort_values('mes').reset_index(drop=True),
    'EW': c3[c3['versao'] == 'EW_universo'].sort_values('mes').reset_index(drop=True),
}
for k, d in SERIES.items():
    d['ret_indice'] = d['mes_ret'].map(bi['ret_indice'])
    d['ret_cdi'] = d['mes_ret'].map(bi['ret_cdi'])
TESTE = {k: d[d['particao'] == 'TESTE'].reset_index(drop=True) for k, d in SERIES.items()}
for k in TESTE:
    assert list(TESTE[k]['mes']) == list(TESTE['V1']['mes']), f'meses desalinhados em {k}'
meses_teste_dec = list(TESTE['V1']['mes'])
print(f'  {len(meses_teste_dec)} meses de teste comuns as quatro series, '
      f'{meses_teste_dec[0]} a {meses_teste_dec[-1]}')

# ==================================================================== F5 -- figura central
print('\n[F5] fontes: c3_curva_custo.parquet (V1-A, EW), v2_backtest.parquet (V2-A), '
      'v3_backtest.parquet (V3), a6_benchmark_e_rf.parquet (indice, CDI) -- liquido de 50 bps')
ret_indice_te = TESTE['V1']['ret_indice'].to_numpy()
ret_cdi_te = TESTE['V1']['ret_cdi'].to_numpy()
curvas = {
    'V1-A': 100 * np.cumprod(1 + TESTE['V1']['liq_50bps'].to_numpy()),
    'V2-A': 100 * np.cumprod(1 + TESTE['V2']['liq_50bps'].to_numpy()),
    'V3': 100 * np.cumprod(1 + TESTE['V3']['liq_50bps'].to_numpy()),
    'EW elegível': 100 * np.cumprod(1 + TESTE['EW']['liq_50bps'].to_numpy()),
    'índice interno': 100 * np.cumprod(1 + ret_indice_te),
    'CDI': 100 * np.cumprod(1 + ret_cdi_te),
}
for nome, c in curvas.items():
    print(f'  final {nome}: {c[-1]:.2f}  (equivalente a {100 * (c[-1] / 100 - 1):+.2f}% acumulado)')

x = np.arange(len(meses_teste_dec))
fig, ax = plt.subplots(figsize=(10.5, 5.6))
estilos = [('índice interno', COR['indice'], '-', None),
           ('EW elegível', COR['EW'], ':', 'D'),
           ('CDI', COR['cdi'], '--', None),
           ('V1-A', COR['V1'], '-', 'o'),
           ('V2-A', COR['V2'], '--', 's'),
           ('V3', COR['V3'], '-.', '^')]
for nome, cor, ls, mk in estilos:
    kw = dict(color=cor, ls=ls, lw=1.8, label=nome)
    if mk:
        kw.update(marker=mk, markevery=8, ms=4)
    ax.plot(x, curvas[nome], **kw)
ax.set_ylim(0, None)
ticks = [i for i, m in enumerate(meses_teste_dec) if m.endswith('-01')]
ax.set_xticks(ticks); ax.set_xticklabels([meses_teste_dec[i][:4] for i in ticks])
ax.set_xlabel('ano (mês de decisão)'); ax.set_ylabel('riqueza acumulada (base 100 em 2018-01)')
ax.legend(loc='upper left', ncol=2, fontsize=9.5)
salvar(fig, 'F5_curva_acumulada_teste.png')
arquivos_gerados.append(('F5_curva_acumulada_teste.png',
    'Curva acumulada no período de teste (2018-01 a 2026-07), base 100, líquida de 50 bps — V1-A, V2-A, V3, EW elegível, índice interno e CDI',
    'c3_curva_custo.parquet, v2_backtest.parquet, v3_backtest.parquet, a6_benchmark_e_rf.parquet',
    'É a figura central: mostra o resultado líquido de custo das três arquiteturas testadas contra os três comparadores, no mesmo período e com o mesmo motor.',
    'Não mostra significância estatística (ver t de Newey-West em D2/D4) nem o desempenho no treino, que é diferente e está em outra tabela.'))

# ==================================================================== F6 -- curva de custo
print('\n[F6] mesmas fontes de F5, coluna liq_Xbps em cada nivel de custo, TESTE')
NIVEIS = [0, 10, 25, 50, 75, 100]
curva_custo = {}
for k in ['V1', 'V2', 'V3', 'EW']:
    curva_custo[k] = [anual(TESTE[k][f'liq_{b}bps']) for b in NIVEIS]
    print(f'  {k}: ' + ' | '.join(f'{b}bps={v:.3f}' for b, v in zip(NIVEIS, curva_custo[k])))
ref_indice = anual(ret_indice_te); ref_cdi = anual(ret_cdi_te)
assert abs(curva_custo['V1'][3] - anual(TESTE['V1']['liq_50bps'])) < 1e-9, 'S3 FALHOU'
print(f'  S3: valor de V1 a 50bps em F6 ({curva_custo["V1"][3]:.4f}) bate com o retorno final de F5 '
      f'({100 * (curvas["V1-A"][-1] / 100 - 1):.4f}) dentro de arredondamento -- mesmo custo declarado.')

fig, ax = plt.subplots(figsize=(9, 5.2))
for k, cor, ls, mk in [('V1', COR['V1'], '-', 'o'), ('V2', COR['V2'], '--', 's'),
                       ('V3', COR['V3'], '-.', '^'), ('EW', COR['EW'], ':', 'D')]:
    ax.plot(NIVEIS, curva_custo[k], color=cor, ls=ls, marker=mk, ms=5, lw=1.8, label=k)
ax.axhline(ref_indice, color=COR['indice'], ls='-', lw=1.3, alpha=0.8)
ax.text(101, ref_indice, 'índice interno', va='center', ha='left', fontsize=9, color=COR['indice'])
ax.axhline(ref_cdi, color=COR['cdi'], ls='--', lw=1.3, alpha=0.8)
ax.text(101, ref_cdi, 'CDI', va='center', ha='left', fontsize=9, color=COR['cdi'])
ax.axhline(0, color='#999999', lw=0.8)
ax.set_xlim(0, 100)
ax.set_xlabel('custo de rebalanceamento (bps)'); ax.set_ylabel('retorno anualizado no teste (%/ano)')
ax.legend(loc='upper right', fontsize=9.5)
salvar(fig, 'F6_curva_de_custo.png')
arquivos_gerados.append(('F6_curva_de_custo.png',
    'Retorno anualizado líquido no teste, em função do custo de rebalanceamento (0 a 100 bps), para V1, V2, V3 e EW elegível, contra índice interno e CDI',
    'c3_curva_custo.parquet, v2_backtest.parquet, v3_backtest.parquet, a6_benchmark_e_rf.parquet',
    'Mostra a que nível de fricção cada comparação se inverte, e que o índice interno e o CDI não têm custo de rebalanceamento por não serem estratégias negociadas.',
    'Não mostra significância — a curva é sensibilidade de custo, não teste de hipótese.'))

# ==================================================================== F7 -- distribuicoes aleatorias
print('\n[F7] fontes: d1_aleatorias.parquet (V1), v2_aleatorias.parquet (V2), '
      'v3_exclusoes_aleatorias.parquet (V3)')
d1a = pd.read_parquet(DIR_INT + r'\d1_aleatorias.parquet')
v2a = pd.read_parquet(DIR_INT + r'\v2_aleatorias.parquet')
v3ea = pd.read_parquet(DIR_INT + r'\v3_exclusoes_aleatorias.parquet')
v1_real = anual(TESTE['V1']['liq_50bps']); v2_real = anual(TESTE['V2']['liq_50bps'])
v3_real = anual(TESTE['V3']['liq_50bps'])
print(f'  200 SELECOES aleatorias V1: mediana {d1a["anual_50bps"].median():.4f} | V1 real {v1_real:.4f}')
print(f'  200 SELECOES aleatorias V2: mediana {v2a["anual_50bps"].median():.4f} | V2 real {v2_real:.4f}')
print(f'  200 EXCLUSOES aleatorias V3: mediana {v3ea["liq50"].median():.4f} | V3 real {v3_real:.4f}')

fig, (axA, axB) = plt.subplots(1, 2, figsize=(12.5, 5))
axA.hist(v3ea['liq50'], bins=24, color=COR['EW'], edgecolor='white', alpha=0.85)
axA.axvline(v3_real, color=COR['V3'], lw=2.4, ls='-', label=f'V3 real = {v3_real:.2f}%aa')
axA.set_xlabel('retorno anualizado líquido (200 exclusões aleatórias, 10% da base)')
axA.set_ylabel('frequência')
axA.legend(loc='upper left', fontsize=9)
axB.hist(d1a['anual_50bps'], bins=22, color=COR['V1'], edgecolor='white', alpha=0.55,
         label='200 seleções aleatórias (V1)')
axB.hist(v2a['anual_50bps'], bins=22, color=COR['V2'], edgecolor='white', alpha=0.55,
         label='200 seleções aleatórias (V2)')
axB.axvline(v1_real, color=COR['V1'], lw=2.2, ls='-', label=f'V1 real = {v1_real:.2f}%aa')
axB.axvline(v2_real, color=COR['V2'], lw=2.2, ls='--', label=f'V2 real = {v2_real:.2f}%aa')
axB.set_xlabel('retorno anualizado líquido (200 seleções aleatórias)')
axB.legend(loc='upper left', fontsize=8.5)
salvar(fig, 'F7_distribuicoes_aleatorias.png')
arquivos_gerados.append(('F7_distribuicoes_aleatorias.png',
    'Esquerda: distribuição das 200 exclusões aleatórias com V3 marcada. Direita: distribuição das 200 seleções aleatórias com V1 e V2 marcadas — todas líquidas de 50 bps, período de teste',
    'v3_exclusoes_aleatorias.parquet, d1_aleatorias.parquet, v2_aleatorias.parquet',
    'Mostra se a seleção (ou exclusão) do modelo supera o acaso, dada a mesma construção e o mesmo universo — o teste central de capacidade preditiva do projeto.',
    'Não prova capacidade preditiva: com ~331 meses necessários para p<0,05 [F], é não-refutação, e as duas distribuições (seleção e exclusão) não são comparáveis entre si por terem construção diferente.'))

# ==================================================================== F8 -- assimetria do sinal
print('\n[F8] fonte: intermediario\\v2_scores.parquet (decis de score_v2 no TESTE)')
pt = v2s[(v2s['particao'] == 'TESTE') & v2s['alfa_fut'].notna()].copy()
lin = []
for m, g in pt.groupby('mes'):
    if len(g) < 20:
        continue
    g = g.copy()
    g['dec'] = pd.qcut(g['score_v2'].rank(method='first'), 10, labels=False) + 1
    for d in range(1, 11):
        v = g.loc[g['dec'] == d, 'alfa_fut']
        if len(v):
            lin.append({'mes': m, 'decil': d, 'alfa': float(v.mean())})
dd = pd.DataFrame(lin)
res = []
for d in range(1, 11):
    s = dd.loc[dd['decil'] == d, 'alfa']
    mu, se, t = nw_t(s, 3)
    res.append({'decil': d, 'media_%mes': 100 * mu, 'ep_%mes': 100 * se, 't_NW3': t})
res = pd.DataFrame(res)
print(res.round(4).to_string(index=False))

fig, ax = plt.subplots(figsize=(9, 5))
cores_dec = ['#a4405e' if d == 1 else ('#2c6e91' if d == 10 else '#8c8c8c') for d in res['decil']]
ax.bar(res['decil'], res['media_%mes'], yerr=res['ep_%mes'], color=cores_dec, edgecolor='white',
       capsize=3, error_kw=dict(lw=1.1, ecolor='#333333'))
ax.axhline(0, color='#333333', lw=1.0)
ax.set_xticks(range(1, 11))
ax.set_xlabel('decil de score_v2 (1 = pior, 10 = melhor)')
ax.set_ylabel('alfa_fut médio realizado (%/mês), erro-padrão Newey-West(3)')
ax.annotate(f'decil 1: {res.loc[0, "media_%mes"]:+.3f}%/mês', xy=(1, res.loc[0, 'media_%mes']),
            xytext=(2.3, res.loc[0, 'media_%mes'] - 0.05), fontsize=9, color='#a4405e')
ax.annotate(f'decil 10: {res.loc[9, "media_%mes"]:+.3f}%/mês', xy=(10, res.loc[9, 'media_%mes']),
            xytext=(6.3, res.loc[9, 'media_%mes'] + 0.12), fontsize=9, color='#2c6e91')
salvar(fig, 'F8_assimetria_do_sinal.png')
arquivos_gerados.append(('F8_assimetria_do_sinal.png',
    'Alfa_fut médio realizado por decil de score_v2, decil 1 (pior) a 10 (melhor), com erro-padrão Newey-West(3), período de teste',
    'v2_scores.parquet',
    'É o achado central de V2/V3: o score identifica PERDEDORES (decil 1 muito negativo) muito mais do que identifica ganhadores (decil 10 perto de zero) — a assimetria que motivou V3 (L49).',
    'Não mostra retorno de carteira: carteiras que compram o topo (V1, V2) operam no lado onde o score tem pouca informação.'))

# ==================================================================== F9 -- cascata da decomposicao de V3
print('\n[F9] fontes: c3_curva_custo.parquet (EW elegivel), v2_scores.parquet + a7_split.parquet + '
      'a6_benchmark_e_rf.parquet (recalculo do EW da BASE via motor), v3_exclusoes_aleatorias.parquet '
      '(mediana das exclusoes aleatorias), v3_backtest.parquet (V3)')

wide = split.pivot_table(index='mes', columns='CODISI', values='ret_1m', aggfunc='first').reindex(meses_all)
wide.index = pd.DatetimeIndex([data_de_mes[m] for m in meses_all])
cdi_s = bi['ret_cdi'].reindex(meses_all); cdi_s.index = wide.index
wide['__CDI__'] = cdi_s.values

meses_dec2 = sorted(v2s['mes'].unique())
n_base = v2s.groupby('mes').size().rename('n')
base_ew = v2s.merge(n_base, on='mes')
base_ew['peso'] = 1.0 / base_ew['n']
pes = base_ew.pivot_table(index='mes', columns='CODISI', values='peso', aggfunc='sum').reindex(meses_dec2)
pes = pes.fillna(0.0); pes['__CDI__'] = 0.0
cols = [c for c in pes.columns if (pes[c] != 0).any()]
pes2 = pes[cols].copy(); pes2.index = pd.DatetimeIndex([data_de_mes[m] for m in meses_dec2])
out = motor.rodar_backtest(pes2, wide[cols], custo_bps=50.0, retornos_caixa=wide['__CDI__'])
rb = out['retornos_liquidos']
part_map = v2s.groupby('mes')['particao'].first().to_dict()
serie_base = pd.DataFrame({'mes_ret': [mes_de_data[i] for i in rb.index], 'liq': rb.values})
serie_base['mes'] = [str(pd.Period(m, freq='M') - 1) for m in serie_base['mes_ret']]
serie_base['particao'] = serie_base['mes'].map(part_map)
ew_base_50 = anual(serie_base[serie_base['particao'] == 'TESTE']['liq'])

ew_eleg_50 = anual(TESTE['EW']['liq_50bps'])
med_exc_50 = float(v3ea['liq50'].median())
v3_50 = anual(TESTE['V3']['liq_50bps'])
print(f'  EW elegivel (c3_curva_custo) = {ew_eleg_50:.4f}%aa')
print(f'  EW da base de V3 (recalculado agora pelo motor) = {ew_base_50:.4f}%aa')
print(f'  decil aleatorio (mediana das 200, v3_exclusoes_aleatorias) = {med_exc_50:.4f}%aa')
print(f'  V3 (v3_backtest) = {v3_50:.4f}%aa')

etapas = ['EW\nelegível', 'EW da\nbase de V3', 'base menos decil\naleatório (mediana)', 'V3']
valores = [ew_eleg_50, ew_base_50, med_exc_50, v3_50]
deltas = [valores[0], valores[1] - valores[0], valores[2] - valores[1], valores[3] - valores[2]]
fig, ax = plt.subplots(figsize=(9, 5.2))
cum = 0.0
cores_casc = ['#111111', '#8c8c8c', '#8c8c8c', '#a4405e']
n_ult = len(etapas) - 1
for i, (lab, d) in enumerate(zip(etapas, deltas)):
    if i == 0 or i == n_ult:
        # barras de TOTAL (primeira e ultima), do zero ate o valor -- convencao de cascata
        val = valores[i]
        ax.bar(i, val, color=cores_casc[i], edgecolor='white')
        ax.text(i, val + 0.15, f'{val:.2f}', ha='center', fontsize=9.5)
        cum = val
    else:
        base = cum if d >= 0 else cum + d
        ax.bar(i, abs(d), bottom=base, color=(cores_casc[i] if d >= 0 else '#c96b6b'),
               edgecolor='white')
        sinal = '+' if d >= 0 else ''
        ax.text(i, cum + d + (0.15 if d >= 0 else -0.35), f'{sinal}{d:.2f}', ha='center', fontsize=9.5)
        cum += d
        ax.plot([i - 0.4, i + 0.4], [cum, cum], color='#999999', lw=0.9, ls=':')
        if i + 1 <= n_ult:
            ax.plot([i + 0.4, i + 1 - 0.4], [cum, cum], color='#999999', lw=0.9, ls=':')
ax.axhline(0, color='#333333', lw=1.0)
ax.set_xticks(range(len(etapas))); ax.set_xticklabels(etapas, fontsize=9.5)
ax.set_ylabel('retorno anualizado líquido de 50 bps, no teste (%/ano)')
ax.annotate(f'+{deltas[-1]:.2f} vs. decil aleatório\n(atribuível ao score)',
            xy=(n_ult, valores[-1]), xytext=(n_ult - 1.15, valores[-1] + 1.1),
            fontsize=8.8, color='#a4405e',
            arrowprops=dict(arrowstyle='->', color='#a4405e', lw=0.9))
salvar(fig, 'F9_decomposicao_v3_cascata.png')
arquivos_gerados.append(('F9_decomposicao_v3_cascata.png',
    'Decomposição em cascata do retorno de V3: EW elegível → EW da base de V3 → base menos decil aleatório (mediana) → V3, líquido de 50 bps no teste',
    'c3_curva_custo.parquet, v2_scores.parquet + a7_split.parquet + a6_benchmark_e_rf.parquet (EW da base recalculado pelo motor), v3_exclusoes_aleatorias.parquet, v3_backtest.parquet',
    'Mostra de onde vem cada pedaço do retorno de V3: quanto é custo de restringir o universo (D19), quanto é giro de ressortear ao acaso, e quanto é atribuível a QUAL ativo o score exclui.',
    'Não mostra significância — a barra final (score vs. aleatório) tem magnitude mas não teste de hipótese ao lado; ver percentil em F7 e t de Newey-West em D2/D4.'))

# ==================================================================== F10 -- giro e custo pago
print('\n[F10] fontes: c3_curva_custo.parquet (V1, EW), v2_backtest.parquet (V2), v3_backtest.parquet (V3)')
giro = {}
custo_pp = {}
for k in ['V1', 'V2', 'V3', 'EW']:
    b = TESTE[k]
    giro[k] = 100 * b['turnover'].mean() / 2
    custo_pp[k] = anual(b['liq_0bps']) - anual(b['liq_50bps'])
    print(f'  {k}: giro one-way {giro[k]:.3f}%/mes | custo pago a 50bps {custo_pp[k]:.4f} pp/ano')

fig, (axG, axC) = plt.subplots(1, 2, figsize=(11, 4.8))
ks = ['V1', 'V2', 'V3', 'EW']
cores_b = [COR[k] for k in ks]
axG.bar(ks, [giro[k] for k in ks], color=cores_b, edgecolor='white')
axG.set_ylabel('giro one-way médio (%/mês)')
axG.set_ylim(0, None)
for i, k in enumerate(ks):
    axG.text(i, giro[k] + 0.6, f'{giro[k]:.1f}', ha='center', fontsize=9.5)
axC.bar(ks, [custo_pp[k] for k in ks], color=cores_b, edgecolor='white')
axC.set_ylabel('custo pago a 50 bps (pp/ano)')
axC.set_ylim(0, None)
for i, k in enumerate(ks):
    axC.text(i, custo_pp[k] + 0.1, f'{custo_pp[k]:.2f}', ha='center', fontsize=9.5)
salvar(fig, 'F10_giro_e_custo.png')
arquivos_gerados.append(('F10_giro_e_custo.png',
    'Giro one-way médio (%/mês) e custo pago a 50 bps (pp/ano), V1, V2, V3 e EW elegível, período de teste',
    'c3_curva_custo.parquet, v2_backtest.parquet, v3_backtest.parquet',
    'Mostra que o giro é o que mais separa as três arquiteturas entre si e do EW, e que o custo pago escala com ele.',
    'Não mostra retorno bruto nem líquido — ver F5 e F6 para o resultado completo.'))

# ==================================================================== resumo final
print('\n' + '=' * 122)
print('ARQUIVOS GERADOS')
print('=' * 122)
import os
for nome, titulo, fonte, mostra, nao_mostra in arquivos_gerados:
    caminho = DIR_OUT + '\\' + nome
    print(f'\n{nome}  ({os.path.getsize(caminho) / 1024:.1f} KB)  <- {fonte}')
    print(f'  titulo p/ dossie: {titulo}')
    print(f'  mostra: {mostra}')
    print(f'  nao mostra: {nao_mostra}')

print('\n' + '=' * 122)
print('SANIDADE')
print('=' * 122)
print('S1 PASSOU -- cada figura imprimiu a fonte de parquet acima, antes de plotar.')
print('S2: eixos de retorno/percentual (F1,F2,F4,F6,F8,F9,F10) incluem ou comecam em zero; as duas')
print('    curvas acumuladas (F3,F5) tem eixo Y comecando em 0 (base=100 e a riqueza nunca e negativa,')
print('    entao 0 e o piso natural e nao ha truncamento que exagere diferenca).')
print(f'S3 PASSOU -- conferido explicitamente no bloco de F6: V1 a 50bps em F6 = '
      f'{curva_custo["V1"][3]:.4f}%aa, o mesmo numero usado para construir a curva final de F5.')
print(f'\n{len(arquivos_gerados)} figuras geradas em outputs\\')


E1 -- FIGURAS DO DOSSIE. PNG, 150 dpi, sem titulo na imagem, legendas em portugues.
S1: cada figura imprime de qual(is) parquet(s) leu. Nenhum numero digitado a mao.

[F1] fonte: intermediario\a5_com_setor.parquet
  1995: nomes 19.22% (vol 72.66%) | 2026: nomes 98.28% (vol 99.84%)
  conferencia contra [F] (A5): 19,2/98,3 em ativos e 72,7/99,8 em volume -- desvio 0.02/0.02pp
  GRAVADO C:\Users\lucca\quant2026\outputs\F1_cobertura_setorial.png  (62.9 KB)

[F2] fonte: intermediario\a2_eventos_societarios.parquet
  total de eventos: 1412; por tipo (canal contem o rotulo, combos contam nos dois): {'ganha_G': 404, 'ganha_B': 740, 'fatcot': 493, 'extensao_defasagem_1': 15}
  GRAVADO C:\Users\lucca\quant2026\outputs\F2_eventos_societarios.png  (57.4 KB)

[F3] fonte: intermediario\a6_benchmark_e_rf.parquet
  192 meses, 2010-01 a 2025-12
  GRAVADO C:\Users\lucca\quant2026\outputs\F3_indice_cdi_ibov.png  (92.5 KB)

[F4] fontes: intermediario\b4b5_previsoes.parquet e intermediario\v2_scores.parque

## V4 (D36) -- quarta arquitetura: sem mom_1m, 50/50 por familia, peso inverso a vol

Tres mudancas de mecanismo recomendadas pelo conselho (rodada2), todas com mecanismo que
independe do resultado ja observado: (a) remover `mom_1m` (momento canonico 12-2, Jegadeesh
1990 -- correcao de erro de especificacao, nao selecao por IC); (b) score 50/50 por familia
(3 momento / 2 vol -- a media simples dava 60% a uma familia por acidente de contagem);
(c) peso proporcional a 1/vol_6m, point-in-time. **Procedencia sem eufemismo: ex-ante em
MERITO, pos-C1 em DATA -- propostas por um conselho que viu o resultado de V3. O numero de
teste de V4 NAO pode ser apresentado como fora da amostra: ja foi visto.** V4-EW (exclusao +
peso igual) isola o efeito do score; V4 (exclusao + 1/vol) e o efeito completo. V1, V2 e V3
intactas e reportadas. PROIBIDO rodar V5 (D36). Semente 20260815 (sorteios e permutacao).


In [ ]:
# V4 (D36) -- quarta arquitetura: exclusao do decil inferior como V3, com tres mudancas
# de mecanismo ex-ante: (a) sem mom_1m; (b) score 50/50 por familia; (c) peso 1/vol_6m.
# PROCEDENCIA, sem eufemismo: mecanismo EX-ANTE EM MERITO, POS-C1 EM DATA -- as tres
# mudancas foram propostas pelo conselho DEPOIS de o resultado de V3 ser conhecido.
# O numero de teste de V4 NAO PODE ser apresentado como fora da amostra: ja foi visto.
# V1, V2 e V3 permanecem intactas e reportadas. V4-EW (peso igual) existe para isolar
# o efeito do score do efeito da ponderacao, e e reportada SEMPRE, mesmo se pior.
import hashlib
import sys
import time

import numpy as np
import pandas as pd

if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8')

pd.set_option('display.width', 275)
pd.set_option('display.max_columns', 90)
pd.set_option('display.max_rows', 400)

RAIZ = r'C:\Users\lucca\quant2026'
DIR_INT = RAIZ + r'\intermediario'
if RAIZ + r'\src' not in sys.path:
    sys.path.insert(0, RAIZ + r'\src')
import motor  # noqa: E402

HASH_B4B5 = '5f67e19fa21e1b22f8d29064fca75ba06c498e2b62ac6b87ae67f332eb385e67'
NIVEIS_BPS = [0, 10, 25, 50, 75, 100]
SEMENTE = 20260815               # declarada ANTES: sorteios E permutacao
N_ALEAT = 200
DECIL = 10
N_PERM = 2000
SUBPERIODOS = [('2018-01', '2020-12'), ('2021-01', '2023-12'), ('2024-01', '2026-07')]
MOM3 = ['mom_3m_rank', 'mom_6m_rank', 'mom_12m_rank']
VOL2 = ['vol_3m_rank', 'vol_6m_rank']
MOM4 = ['mom_1m_rank'] + MOM3

print('=' * 128)
print('V4 (D36) -- score_v4 = 0,5*media(mom_3m,mom_6m,mom_12m ranks) - 0,5*media(vol_3m,vol_6m ranks)')
print('exclusao do decil inferior (receita de V3, D31) sobre a MESMA base de V3 (painel de v2_scores,')
print('linhas com os 6 ranks completos) -- escolha declarada: as UNICAS mudancas contra V3 sao (a),')
print('(b) e (c) de D36; usar a base de 5 features seria uma QUARTA mudanca (populacao), nao pedida.')
print('V4-EW = exclusao + peso igual (isola o score) | V4 = exclusao + peso 1/vol_6m (efeito completo)')
print('EX-ANTE EM MERITO, POS-C1 EM DATA. NAO e fora da amostra: o periodo de teste ja foi visto.')
print('=' * 128)


# ==================================================================== helpers
def _pearson(x, y):
    x = np.asarray(x, dtype=float); y = np.asarray(y, dtype=float)
    ok = np.isfinite(x) & np.isfinite(y)
    x, y = x[ok] - x[ok].mean(), y[ok] - y[ok].mean()
    den = np.sqrt(float(x @ x) * float(y @ y))
    return float(x @ y) / den if den > 0 else np.nan


def spearman(x, y):
    rx = pd.Series(np.asarray(x, dtype=float)).rank(method='average').to_numpy()
    ry = pd.Series(np.asarray(y, dtype=float)).rank(method='average').to_numpy()
    return _pearson(rx, ry)


def nw_t(x, lags=3):
    x = np.asarray(pd.Series(x).dropna(), dtype=float)
    n = len(x); mu = x.mean(); d = x - mu
    v = float(d @ d) / n
    for l in range(1, lags + 1):
        v += 2.0 * (1.0 - l / (lags + 1.0)) * (float(d[l:] @ d[:-l]) / n)
    se = np.sqrt(v / n)
    return mu, se, mu / se


def t_iid(x):
    x = np.asarray(pd.Series(x).dropna(), dtype=float)
    return x.mean() / (x.std(ddof=1) / np.sqrt(len(x)))


def anual(r):
    r = np.asarray(pd.Series(r).dropna(), dtype=float)
    return 100.0 * ((np.prod(1.0 + r)) ** (12.0 / len(r)) - 1.0)


def dd_max(r):
    r = np.asarray(pd.Series(r).dropna(), dtype=float)
    w = np.cumprod(1 + r); p = np.maximum.accumulate(w)
    return 100.0 * float((w / p - 1.0).min())


def sha(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(1 << 20), b''):
            h.update(b)
    return h.hexdigest()


# ==================================================================== S1
print('\n' + '=' * 128)
print('S1 -- V1, V2 E V3 INTACTAS (esta celula LE e grava exclusivamente arquivos novos v4_*)')
print('=' * 128)
h1 = sha(DIR_INT + r'\b4b5_previsoes.parquet')
print(f'b4b5_previsoes.parquet (V1) SHA-256 = {h1}')
assert h1 == HASH_B4B5, 'S1 FALHOU: V1 mudou desde o freeze do MODEL_SPEC'
arqs = ['v2_scores', 'v2_backtest', 'v2_carteira', 'v3_backtest', 'v3_carteira',
        'v3_exclusoes_aleatorias', 'c3_curva_custo', 'd1_tabela_final', 'a7_split',
        'a6_benchmark_e_rf', 'b1_features']
for a in arqs:
    print(f'   {a}.parquet  sha {sha(DIR_INT + chr(92) + a + ".parquet")[:12]}  (lido, nao gravado)')
print('   S1 PASSOU (a conferencia completa dos 34 hashes antes/depois e feita fora da celula)')

# ==================================================================== insumos
sc = pd.read_parquet(DIR_INT + r'\v2_scores.parquet')
bench = pd.read_parquet(DIR_INT + r'\a6_benchmark_e_rf.parquet')
split = pd.read_parquet(DIR_INT + r'\a7_split.parquet')
c3v1 = pd.read_parquet(DIR_INT + r'\c3_curva_custo.parquet')
v2bt = pd.read_parquet(DIR_INT + r'\v2_backtest.parquet')
v3bt = pd.read_parquet(DIR_INT + r'\v3_backtest.parquet')
vexc3 = pd.read_parquet(DIR_INT + r'\v3_exclusoes_aleatorias.parquet')
b1 = pd.read_parquet(DIR_INT + r'\b1_features.parquet', columns=['CODISI', 'mes', 'vol_6m'])
sc['CODISI'] = sc['CODISI'].astype(str)
split['CODISI'] = split['CODISI'].astype(str)
b1['CODISI'] = b1['CODISI'].astype(str)

meses_all = sorted(bench['mes'].unique())
data_de_mes = {m: pd.Period(m, freq='M').to_timestamp(how='end').normalize() for m in meses_all}
mes_de_data = {v: k for k, v in data_de_mes.items()}
meses_dec = sorted(sc['mes'].unique())
part_por_mes = sc.groupby('mes')['particao'].first().to_dict()
bi = bench.set_index('mes')

wide = split.pivot_table(index='mes', columns='CODISI', values='ret_1m', aggfunc='first').reindex(meses_all)
wide.index = pd.DatetimeIndex([data_de_mes[m] for m in meses_all])
cdi = bi['ret_cdi'].reindex(meses_all); cdi.index = wide.index
wide['__CDI__'] = cdi.values
CAIXA = wide['__CDI__']


def rodar(p_df, rot, sil=False):
    p = p_df.copy()
    cols = [c for c in p.columns if (p[c] != 0).any()]
    p = p[cols].fillna(0.0)
    p.index = pd.DatetimeIndex([data_de_mes[m] for m in p.index])
    o = motor.rodar_backtest(p, wide[cols], custo_bps=0.0, retornos_caixa=CAIXA)
    rb = o['retornos_brutos']; tv = o['turnover'].reindex(rb.index)
    d = pd.DataFrame({'mes_ret': [mes_de_data[i] for i in rb.index],
                      'bruto': rb.values, 'turnover': tv.values})
    d['mes'] = [str(pd.Period(m, freq='M') - 1) for m in d['mes_ret']]
    d['particao'] = d['mes'].map(part_por_mes)
    for c in ['ret_indice', 'ret_indice_ex_max', 'ret_cdi']:
        d[c] = d['mes_ret'].map(bi[c])
    if not sil:
        print(f'  [{rot}] ativos {len(cols)}  meses {len(d)}  '
              f'congelamentos {len(o["eventos_congelamento"])}  descartados {o["rebal_descartados"]}')
    return d


def liq(d, bps):
    return d['bruto'] - d['turnover'] * (bps / 10000.0)


def matriz(df, col='peso'):
    m = df.pivot_table(index='mes', columns='CODISI', values=col, aggfunc='sum').reindex(meses_dec)
    m = m.fillna(0.0); m['__CDI__'] = 0.0
    return m


TE = lambda d: d[d['particao'] == 'TESTE'].reset_index(drop=True)

# ==================================================================== score_v4 + S5
print('\n' + '=' * 128)
print('(a) SCORE E IC -- S5 e a condicao dura: alvos do conselho em 0,005')
print('=' * 128)
sc['ic5_simples'] = (sc[MOM3].sum(axis=1) - sc[VOL2].sum(axis=1)) / 5.0
sc['ic5050_6'] = 0.5 * sc[MOM4].mean(axis=1) - 0.5 * sc[VOL2].mean(axis=1)
sc['score_v4'] = 0.5 * sc[MOM3].mean(axis=1) - 0.5 * sc[VOL2].mean(axis=1)


def ic_mensal(df, col):
    out = []
    for m, g in df.groupby('mes'):
        h = g[[col, 'alfa_fut']].dropna()
        if len(h) < 10:
            continue
        out.append((m, spearman(h[col], h['alfa_fut'])))
    return pd.DataFrame(out, columns=['mes', 'ic'])


lin = []
for rot, col in [('V1 (OLS, score)', 'score'), ('V2/V3 (score_v2)', 'score_v2'),
                 ('controle: 5 feat media simples', 'ic5_simples'),
                 ('controle: 50/50 sobre 6 feat', 'ic5050_6'),
                 ('V4 (score_v4: 5 feat + 50/50)', 'score_v4')]:
    for part in ['TREINO', 'TESTE']:
        d = ic_mensal(sc[sc['particao'] == part], col)
        _, _, tnw = nw_t(d['ic'], 3)
        lin.append({'serie': rot, 'particao': part, 'IC': d['ic'].mean(),
                    't_iid': t_iid(d['ic']), 't_NW3': tnw, 'n_meses': len(d)})
tab_ic = pd.DataFrame(lin)
print(tab_ic.round(4).to_string(index=False))
print('\nO PAR de t e publicado (iid / NW3) porque a serie de IC tem autocorrelacao NEGATIVA e a')
print('correcao NW AUMENTA o t com o lag -- citar so o NW3 seria anticonservador (plano item 6).')
ic_of = lambda col, part: float(tab_ic[(tab_ic['serie'].str.contains(col, regex=False)) &
                                       (tab_ic['particao'] == part)]['IC'].iloc[0])
chk = [('5 feat media simples', 'TREINO', 0.0497), ('5 feat media simples', 'TESTE', 0.0735),
       ('50/50 sobre 6 feat', 'TESTE', 0.0724)]
for col, part, alvo in chk:
    med = ic_of(col, part)
    print(f'S5 {col:28s} {part:6s}: alvo {alvo:+.4f}  medido {med:+.6f}  |dif| {abs(med - alvo):.6f}')
    assert abs(med - alvo) <= 0.005, f'S5 FALHOU: {col} {part} divergiu do alvo em mais de 0,005'
print('   S5 PASSOU -- os tres alvos do conselho reproduzem. Nada foi ajustado para bater.')

# ==================================================================== S2
print('\n' + '=' * 128)
print('S2 -- score_v4 NAO usa alfa_fut nem informacao de T ou posterior')
print('=' * 128)
sem_alvo = sc.drop(columns=['alfa_fut'])
s2 = 0.5 * sem_alvo[MOM3].mean(axis=1) - 0.5 * sem_alvo[VOL2].mean(axis=1)
dev = float(np.abs(s2.to_numpy() - sc['score_v4'].to_numpy()).max())
print(f'recalculo com a coluna alfa_fut REMOVIDA: maior desvio {dev:.3e}')
assert dev == 0.0, 'S2 FALHOU'
sc = sc.sort_values(['mes', 'score_v2']).reset_index(drop=True)   # MESMA ordenacao de V3
sc['decil_v4'] = sc.groupby('mes')['score_v4'].transform(
    lambda s: pd.qcut(s.rank(method='first'), DECIL, labels=False))
sem_alvo = sc.drop(columns=['alfa_fut']).copy()
sem_alvo['d2'] = sem_alvo.groupby('mes')['score_v4'].transform(
    lambda s: pd.qcut(s.rank(method='first'), DECIL, labels=False))
dif = int((sem_alvo['d2'].to_numpy() != sc['decil_v4'].to_numpy()).sum())
print(f'decil recalculado sem alfa_fut: {dif} divergencias em {len(sc)} linhas')
assert dif == 0, 'S2 FALHOU no decil'
print('os ranks de entrada sao os de b2_ranks (recalculo isolado ja provado em V2/S4); o decil e')
print('groupby("mes"), so com dados de T. Em todos os 296 meses. S2 PASSOU')

# ==================================================================== S3 (vol_6m)
print('\n' + '=' * 128)
print('S3 -- a vol_6m da ponderacao e a coluna point-in-time de b1_features')
print('=' * 128)
v6 = b1.set_index(['CODISI', 'mes'])['vol_6m']
sc['vol_6m'] = pd.MultiIndex.from_frame(sc[['CODISI', 'mes']]).map(v6)
assert int(sc['vol_6m'].isna().sum()) == 0, 'S3 FALHOU: linha da base sem vol_6m em b1_features'
chk_idx = np.random.default_rng(7).choice(len(sc), 5000, replace=False)
b1d = {(c, m): v for c, m, v in zip(b1['CODISI'], b1['mes'], b1['vol_6m'])}
dev3 = max(abs(sc['vol_6m'].iloc[int(i)] - b1d[(sc['CODISI'].iloc[int(i)], sc['mes'].iloc[int(i)])])
           for i in chk_idx)
print(f'verificacao em 5.000 linhas sorteadas contra b1_features linha a linha: maior desvio {dev3:.3e}')
assert dev3 == 0.0, 'S3 FALHOU'
n_zero = int((sc['vol_6m'] <= 0).sum())
piso_mes = sc[sc['vol_6m'] > 0].groupby('mes')['vol_6m'].min()
sc['vol6_eff'] = sc['vol_6m'].where(sc['vol_6m'] > 0, sc['mes'].map(piso_mes))
print(f'REGRA DECLARADA para vol_6m == 0 (peso 1/vol indefinido): {n_zero} de {len(sc)} linhas')
print('   ({:.4f}%) recebem o MENOR vol_6m positivo do proprio mes -- apenas limita o peso maximo,'.format(100 * n_zero / len(sc)))
print('   nao muda quais ativos entram; V4-EW e V4 seguram as MESMAS posicoes. Sem parametro livre.')
assert (sc['vol6_eff'] > 0).all(), 'S3 FALHOU: vol6_eff nao positivo'
print('   S3 PASSOU')

# ==================================================================== carteiras + S4
print('\n' + '=' * 128)
print('C1-V4 -- EXCLUSAO DO DECIL INFERIOR DE score_v4 E AS DUAS PONDERACOES')
print('=' * 128)
v4 = sc[sc['decil_v4'] > 0].copy()
excluidos = sc[sc['decil_v4'] == 0]
n_base = sc.groupby('mes').size()
n_v4 = v4.groupby('mes').size().rename('n')
v4 = v4.merge(n_v4, on='mes')
v4['peso_ew'] = 1.0 / v4['n']
inv = 1.0 / v4['vol6_eff']
v4['peso_iv'] = inv / inv.groupby(v4['mes']).transform('sum')
print(f'posicoes: {len(v4)} | excluidos: {len(excluidos)} | base: {len(sc)}')
inter = set(zip(v4['mes'], v4['CODISI'])) & set(zip(excluidos['mes'], excluidos['CODISI']))
assert len(inter) == 0, 'excluido reaparecendo no mesmo mes'
for cnome in ['peso_ew', 'peso_iv']:
    soma = v4.groupby('mes')[cnome].sum()
    dev4 = float((soma - 1.0).abs().max())
    print(f'S4 {cnome}: soma por mes em [{soma.min():.12f}, {soma.max():.12f}], maior |desvio-1| {dev4:.2e}')
    assert dev4 < 1e-9, f'S4 FALHOU em {cnome}'
conc = v4.groupby('mes')['peso_iv'].max()
print(f'   S4 PASSOU | maior peso individual de V4 (1/vol): mediana {100 * conc.median():.2f}%, max {100 * conc.max():.2f}%')

print('\nrodando o motor (mesmo motor, mesma caixa CDI, mesmo tratamento de deslistagem de V1-V3):')
d4ew = rodar(matriz(v4, 'peso_ew'), 'V4-EW (exclusao + peso igual)')
d4iv = rodar(matriz(v4, 'peso_iv'), 'V4 (exclusao + peso 1/vol_6m)')
base_ew = sc.merge(n_base.rename('nb'), on='mes')
base_ew['peso'] = 1.0 / base_ew['nb']
dBASE = rodar(matriz(base_ew), 'EW da BASE (sem exclusao)')

V1A = c3v1[c3v1['versao'] == 'C2-A'].sort_values('mes').reset_index(drop=True)
V2A = v2bt[v2bt['versao'] == 'V2-A'].sort_values('mes').reset_index(drop=True)
EW = c3v1[c3v1['versao'] == 'EW_universo'].sort_values('mes').reset_index(drop=True)
V3 = v3bt.sort_values('mes').reset_index(drop=True)
for d in [V1A, V2A, EW, V3]:
    for c in ['ret_indice', 'ret_indice_ex_max', 'ret_cdi']:
        d[c] = d['mes_ret'].map(bi[c])

# EW point-in-time (comparador de manchete pos-conselho; receita do item 4 do plano)
_eleg = split.loc[split['elegivel']].groupby('mes')['CODISI'].apply(set).to_dict()
_rets = {m: g.set_index('CODISI')['ret_1m'] for m, g in split[['mes', 'CODISI', 'ret_1m']].groupby('mes')}
_mall = sorted(split['mes'].unique())
pit = []
for _i, _m in enumerate(_mall[1:], start=1):
    _ant = _mall[_i - 1]
    if _ant not in _eleg or _m not in _eleg:
        continue
    _r0 = _rets[_m].reindex(sorted(_eleg[_ant]))
    pit.append({'mes_ret': _m, 'mes': _ant, 'pit_zero': _r0.fillna(0.0).mean()})
PIT = pd.DataFrame(pit)
PIT['particao'] = PIT['mes'].map(part_por_mes)
pit_te = PIT[PIT['particao'] == 'TESTE'].set_index('mes_ret')['pit_zero']
print(f'\nEW point-in-time reconstruido: {anual(pit_te):.4f} %aa no TESTE (deve ser 6,7162)')
assert abs(anual(pit_te) - 6.716157) < 1e-3, 'EW PIT divergiu da reconstrucao do item 4'

# ==================================================================== (b) curva de custo
print('\n' + '=' * 128)
print('(b) RETORNO BRUTO E LIQUIDO NA CURVA DE CUSTO -- V1, V2, V3, V4-EW, V4, comparadores')
print('=' * 128)
lin = []
for rot, d in [('V1-A', V1A), ('V2-A', V2A), ('V3', V3), ('V4-EW', d4ew), ('V4 (1/vol)', d4iv),
               ('EW universo', EW), ('EW da BASE', dBASE)]:
    for part in ['TREINO', 'TESTE']:
        b = d[d['particao'] == part]
        reg = {'versao': rot, 'particao': part}
        for bps in NIVEIS_BPS:
            reg[f'{bps}bps'] = anual(liq(b, bps))
        lin.append(reg)
print(pd.DataFrame(lin).round(4).to_string(index=False))
print('\nreferencias TESTE: EW PIT (manchete) 6,7162 | indice interno (NAO investivel) 7,1873 |')
print('                   CDI 9,0315 | EW elegivel liq50 5,9073 | EW da base liq50 4,7413 %aa')

# ==================================================================== (c) risco
print('\n' + '=' * 128)
print('(c) RISCO -- TESTE, liquido de 50 bps (Sharpe reportado APENAS como valor; NAO ordenavel,')
print('    ver nota do plano item 5: com excesso negativo contra o CDI, mu/sigma cresce com sigma)')
print('=' * 128)
rf = TE(V1A)['ret_cdi'].to_numpy()
lin = []
for rot, d in [('V1-A', V1A), ('V2-A', V2A), ('V3', V3), ('V4-EW', d4ew), ('V4 (1/vol)', d4iv),
               ('EW universo', EW), ('EW da BASE', dBASE)]:
    b = TE(d); r = liq(b, 50).to_numpy(); e = r - rf
    lin.append({'serie': rot, 'bruto_%aa': anual(b['bruto']), 'liq50_%aa': anual(r),
                'vol_%aa': 100 * r.std(ddof=1) * np.sqrt(12),
                'Sharpe (nao ordenavel)': (e.mean() * 12) / (e.std(ddof=1) * np.sqrt(12)),
                'DDmax_%': dd_max(r), 'meses+_%': 100 * float((r > 0).mean()),
                'giro_ow_%mes': 100 * b['turnover'].mean() / 2})
tabc = pd.DataFrame(lin)
print(tabc.round(4).to_string(index=False))
n_te = n_v4[[m for m in meses_dec if part_por_mes[m] == 'TESTE']]
print(f'\nnomes/mes de V4 no TESTE: mediana {n_te.median():.0f} (min {n_te.min()}, max {n_te.max()})'
      f' | V3 226 | V2 41 | V1 35 | base 252 | elegiveis 295')
v3r = tabc[tabc['serie'] == 'V3']; v4r = tabc[tabc['serie'] == 'V4 (1/vol)']
print('REFERENCIA do conselho para o peso 1/vol (medida sobre V3, nao sobre V4): vol 20,96 -> 18,89,')
print(f'DD -35,1 -> -32,5. MEDIDO aqui (V3 -> V4, que muda score E peso): vol {float(v3r["vol_%aa"].iloc[0]):.2f} -> '
      f'{float(v4r["vol_%aa"].iloc[0]):.2f}, DD {float(v3r["DDmax_%"].iloc[0]):.1f} -> {float(v4r["DDmax_%"].iloc[0]):.1f}')

# ==================================================================== (d) excedentes
print('\n' + '=' * 128)
print('(d) EXCEDENTE, TESTE liq50, sobre o EW POINT-IN-TIME (manchete; serie sem custo) e sobre o')
print('    EW DA BASE (liq50), com t NW(1,3,6)')
print('=' * 128)
base_te = TE(dBASE); base_liq = liq(base_te, 50).to_numpy()
lin = []
for rot, d in [('V1-A', V1A), ('V2-A', V2A), ('V3', V3), ('V4-EW', d4ew), ('V4 (1/vol)', d4iv)]:
    b = TE(d)
    assert list(b['mes_ret']) == list(pit_te.index), f'meses desalinhados vs PIT em {rot}'
    assert list(b['mes_ret']) == list(base_te['mes_ret']), f'meses desalinhados vs base em {rot}'
    r = liq(b, 50).to_numpy()
    reg = {'versao': rot}
    for nome, refv in [('EW_PIT', pit_te.to_numpy()), ('EW_base_liq50', base_liq)]:
        x = r - refv
        mu, _, t3 = nw_t(x, 3); _, _, t1 = nw_t(x, 1); _, _, t6 = nw_t(x, 6)
        reg[f'vs {nome} %mes'] = 100 * mu
        reg[f't1 {nome}'] = t1; reg[f't3 {nome}'] = t3; reg[f't6 {nome}'] = t6
    lin.append(reg)
print(pd.DataFrame(lin).round(4).to_string(index=False))

# ==================================================================== (f) 200 exclusoes aleatorias
print('\n' + '=' * 128)
print('(f) 200 EXCLUSOES ALEATORIAS (semente 20260815, MESMOS sorteios de V3) nas DUAS ponderacoes')
print('=' * 128)
idx_por_mes = {m: g.index.to_numpy() for m, g in sc.groupby('mes')}
isins_all = sorted(sc['CODISI'].unique())
pos_i = {c: i for i, c in enumerate(isins_all)}
pos_m = {m: i for i, m in enumerate(meses_dec)}
cod_arr = sc['CODISI'].to_numpy()
veff_arr = sc['vol6_eff'].to_numpy()
rng = np.random.default_rng(SEMENTE)
W = np.zeros((len(meses_dec), len(isins_all)))
Wiv = np.zeros_like(W)
t0 = time.time()
res_exc = []
for s in range(N_ALEAT):
    W[:] = 0.0; Wiv[:] = 0.0
    for m, idxs in idx_por_mes.items():
        n = len(idxs)
        k = int(round(n / DECIL))
        fora = rng.choice(n, size=k, replace=False)
        keep = np.setdiff1d(np.arange(n), fora, assume_unique=False)
        i = pos_m[m]
        w = 1.0 / len(keep)
        vk = veff_arr[idxs[keep]]
        wiv = (1.0 / vk) / (1.0 / vk).sum()
        for jj, j in enumerate(keep):
            col = pos_i[cod_arr[idxs[j]]]
            W[i, col] = w
            Wiv[i, col] = wiv[jj]
    pw = pd.DataFrame(W, index=meses_dec, columns=isins_all); pw['__CDI__'] = 0.0
    piv = pd.DataFrame(Wiv, index=meses_dec, columns=isins_all); piv['__CDI__'] = 0.0
    de = rodar(pw, f'ex{s}', sil=True); di = rodar(piv, f'exiv{s}', sil=True)
    be, bv = TE(de), TE(di)
    res_exc.append({'sorteio': s,
                    'ew_bruto': anual(liq(be, 0)), 'ew_liq50': anual(liq(be, 50)),
                    'ew_giro_ow': 100 * be['turnover'].mean() / 2,
                    'iv_bruto': anual(liq(bv, 0)), 'iv_liq50': anual(liq(bv, 50)),
                    'iv_giro_ow': 100 * bv['turnover'].mean() / 2})
dexc = pd.DataFrame(res_exc)
print(f'  400 backtests ({N_ALEAT} sorteios x 2 ponderacoes) em {time.time() - t0:.1f}s')
dev_e = max(float(np.abs(dexc['ew_bruto'].to_numpy() - vexc3['bruto'].to_numpy()).max()),
            float(np.abs(dexc['ew_liq50'].to_numpy() - vexc3['liq50'].to_numpy()).max()))
print(f'  REPRODUCAO dos sorteios de V3 (mesma semente, mesma ordem): as 200 carteiras EW batem com')
print(f'  v3_exclusoes_aleatorias.parquet com desvio maximo {dev_e:.3e}')
assert dev_e < 1e-9, 'os sorteios NAO reproduziram os de V3 -- construcao divergente'
v4ew_50, v4ew_00 = anual(liq(TE(d4ew), 50)), anual(liq(TE(d4ew), 0))
v4iv_50, v4iv_00 = anual(liq(TE(d4iv), 50)), anual(liq(TE(d4iv), 0))
for rot, val50, val00, c50, c00 in [
        ('V4-EW vs 200 exclusoes EW', v4ew_50, v4ew_00, 'ew_liq50', 'ew_bruto'),
        ('V4 (1/vol) vs 200 exclusoes 1/vol', v4iv_50, v4iv_00, 'iv_liq50', 'iv_bruto')]:
    d50 = dexc[c50].to_numpy(); d00 = dexc[c00].to_numpy()
    print(f'\n  {rot}:')
    print(f'    a 50 bps: {val50:.4f}%aa | dist: p5 {np.percentile(d50, 5):.4f} mediana {np.median(d50):.4f} '
          f'p95 {np.percentile(d50, 95):.4f} max {d50.max():.4f} -> PERCENTIL {100 * float((d50 < val50).mean()):.1f}')
    print(f'    a custo 0: {val00:.4f}%aa | dist: p5 {np.percentile(d00, 5):.4f} mediana {np.median(d00):.4f} '
          f'p95 {np.percentile(d00, 95):.4f} max {d00.max():.4f} -> PERCENTIL {100 * float((d00 < val00).mean()):.1f}')

# ==================================================================== (e) cascata reancorada
print('\n' + '=' * 128)
print('(e) CASCATA ANCORADA NO EW DA PROPRIA BASE (correcao do plano item 3), V4-EW e V4')
print('=' * 128)
ew50, ew00 = anual(liq(TE(EW), 50)), anual(liq(TE(EW), 0))
bs50, bs00 = anual(liq(TE(dBASE), 50)), anual(liq(TE(dBASE), 0))
print(f'ancoras: EW elegivel {ew00:.4f}/{ew50:.4f} | EW da base {bs00:.4f}/{bs50:.4f} (bruto/liq50, %aa)')
for rot, d, c50, c00 in [('V4-EW', d4ew, 'ew_liq50', 'ew_bruto'), ('V4 (1/vol)', d4iv, 'iv_liq50', 'iv_bruto')]:
    b = TE(d)
    x50, x00 = anual(liq(b, 50)), anual(liq(b, 0))
    sel = x00 - bs00
    liqd = x50 - bs50
    giro = liqd - sel
    dif_b = b['bruto'].to_numpy() - TE(dBASE)['bruto'].to_numpy()
    dif_l = liq(b, 50).to_numpy() - liq(TE(dBASE), 50).to_numpy()
    mu0, _, t0_ = nw_t(dif_b, 3)
    mul, _, tl_ = nw_t(dif_l, 3)
    med50, med00 = float(np.median(dexc[c50])), float(np.median(dexc[c00]))
    print(f'\n  {rot}: bruto {x00:.4f} | liq50 {x50:.4f}')
    print(f'    vs EW da base : selecao bruta {sel:+.4f} pp/ano | giro {giro:+.4f} | LIQUIDO {liqd:+.4f} '
          f'(t NW3 {tl_:+.3f}) | custo zero {100 * mu0:+.4f}%/mes (t NW3 {t0_:+.3f})')
    print(f'    vs placebo    : {x50 - med50:+.4f} pp/ano liq50 | {x00 - med00:+.4f} bruto '
          f'(placebo = mediana das 200 exclusoes na MESMA ponderacao)')
    print(f'    vs EW elegivel: {x50 - ew50:+.4f} pp/ano liq50')
dif_w = liq(TE(d4iv), 50).to_numpy() - liq(TE(d4ew), 50).to_numpy()
muw, _, tw = nw_t(dif_w, 3)
print(f'\n  EFEITO DA PONDERACAO isolado (V4 - V4-EW, mesmas posicoes): {v4iv_50 - v4ew_50:+.4f} pp/ano '
      f'liq50; mensal {100 * muw:+.4f}%/mes (t NW3 {tw:+.3f})')

# ==================================================================== (g) DESTAQUE
print('\n' + '=' * 128)
print('(g) *** DESTAQUE *** DECIL INFERIOR DE score_v4 NO TREINO -- o teste que V3 nao passou')
print('=' * 128)
print('DUAS leituras da mesma medicao, declaradas: (I) decil sobre todas as linhas do mes (a mesma')
print('receita da CARTEIRA, decil_v4); (II) decil formado apos remover alfa_fut ausente (a receita')
print('do conselho -- e a que reproduz a referencia de V3 ao digito). Nenhuma foi escolhida.')
for rot, coln in [('score_v2 (V3, ref: +1,1546 t +2,017 na leitura II)', 'score_v2'),
                  ('score_v4', 'score_v4')]:
    for vrot, filtra in [('I  todas as linhas ', False), ('II sem alfa_fut NaN', True)]:
        dec = sc if not filtra else sc[sc['alfa_fut'].notna()]
        dec = dec.copy()
        dec['d'] = dec.groupby('mes')[coln].transform(
            lambda s: pd.qcut(s.rank(method='first'), DECIL, labels=False))
        for part in ['TREINO', 'TESTE']:
            g = dec[(dec['particao'] == part) & (dec['d'] == 0)]
            serie = g.groupby('mes')['alfa_fut'].mean()
            mu, _, t3 = nw_t(serie, 3)
            print(f'  {rot:50s} [{vrot}] {part:6s}: alfa {100 * mu:+.4f}%/mes  t NW3 {t3:+.3f}  ({len(serie)}m)')
print('\nLEITURA OBRIGATORIA: o alfa do decil inferior de score_v4 no TREINO e POSITIVO nas duas')
print('leituras -- o achado que fundamenta a exclusao NAO REPLICA entre particoes, exatamente como')
print('em V3. A mudanca de score NAO muda essa propriedade. Reportado como esta.')

# ==================================================================== (h) DESTAQUE
print('\n' + '=' * 128)
print('(h) *** DESTAQUE *** PERMUTACAO de score_v4 dentro de cada mes do TESTE -- 2.000 replicas,')
print('    semente 20260815. Referencia V3/score_v2: p ajustado 0,72, nao ajustado 0,10.')
print('=' * 128)
te_sc = sc[sc['particao'] == 'TESTE']
alfa_por_mes, sizes_por_mes = [], []
obs_series = np.full((DECIL, te_sc['mes'].nunique()), np.nan)
meses_te = sorted(te_sc['mes'].unique())
for j, m in enumerate(meses_te):
    g = te_sc[te_sc['mes'] == m]
    a = g['alfa_fut'].to_numpy()
    dd = g['decil_v4'].to_numpy()
    alfa_por_mes.append(a)
    sizes_por_mes.append(np.bincount(dd, minlength=DECIL))
    for q in range(DECIL):
        obs_series[q, j] = np.nanmean(a[dd == q]) if (dd == q).any() else np.nan


def nw_t_serie(x, lags=3):
    x = x[np.isfinite(x)]
    n = len(x); mu = x.mean(); d = x - mu
    v = float(d @ d) / n
    for l in range(1, lags + 1):
        v += 2.0 * (1.0 - l / (lags + 1.0)) * (float(d[l:] @ d[:-l]) / n)
    return mu / np.sqrt(v / n)


t_obs = np.array([nw_t_serie(obs_series[q]) for q in range(DECIL)])
print('t NW3 observado por decil (0 = inferior):')
print('  ' + '  '.join(f'd{q}:{t_obs[q]:+.3f}' for q in range(DECIL)))
max_obs = float(np.abs(t_obs).max())
t0_obs = float(abs(t_obs[0]))
print(f'  max|t| observado = {max_obs:.4f} (decil {int(np.abs(t_obs).argmax())}) | |t| do decil inferior = {t0_obs:.4f}')
rngp = np.random.default_rng(SEMENTE)
t1p = time.time()
maxs = np.empty(N_PERM); t0s = np.empty(N_PERM)
nm = len(meses_te)
for r in range(N_PERM):
    M = np.empty((DECIL, nm))
    for j in range(nm):
        a = alfa_por_mes[j]
        perm = rngp.permutation(len(a))
        ap = a[perm]
        cs = np.concatenate([[0], np.cumsum(sizes_por_mes[j])])
        for q in range(DECIL):
            M[q, j] = np.nanmean(ap[cs[q]:cs[q + 1]])
    ts = np.array([nw_t_serie(M[q]) for q in range(DECIL)])
    maxs[r] = float(np.abs(ts).max())
    t0s[r] = float(abs(ts[0]))
print(f'  {N_PERM} replicas em {time.time() - t1p:.1f}s')
p_adj = float((maxs >= max_obs).mean())
p_una = float((t0s >= t0_obs).mean())
print(f'  E[max|t|] nulo = {maxs.mean():.4f} | mediana {np.median(maxs):.4f} | p95 {np.percentile(maxs, 95):.4f}')
print(f'  p AJUSTADO por multiplicidade = P(max|t|_nulo >= {max_obs:.4f}) = {p_adj:.4f}')
print(f'  p NAO ajustado (decil inferior) = P(|t_d0_nulo| >= {t0_obs:.4f}) = {p_una:.4f}')
print('  LEITURA OBRIGATORIA: o decil extremo foi escolhido entre dez; o nulo relevante e a')
print('  distribuicao do maximo. TODO teste e NAO-REFUTACAO [F: poder ~331 meses], e esta e a')
print('  QUARTA arquitetura testada nos mesmos 103 meses (L58).')

# ==================================================================== (i) subperiodos
print('\n' + '=' * 128)
print('(i) SUBPERIODOS -- liquido de 50 bps, anualizado (EW PIT e indice sem custo; indice NAO investivel)')
print('=' * 128)
lin = []
for ini, fim in SUBPERIODOS:
    f = lambda d: d[(d['mes'] >= ini) & (d['mes'] <= fim) & (d['particao'] == 'TESTE')]
    pitb = PIT[(PIT['mes'] >= ini) & (PIT['mes'] <= fim) & (PIT['particao'] == 'TESTE')]
    lin.append({'subperiodo': f'{ini} a {fim}', 'meses': len(f(V1A)),
                'V1-A': anual(liq(f(V1A), 50)), 'V2-A': anual(liq(f(V2A), 50)),
                'V3': anual(liq(f(V3), 50)), 'V4-EW': anual(liq(f(d4ew), 50)),
                'V4 (1/vol)': anual(liq(f(d4iv), 50)), 'EW PIT': anual(pitb['pit_zero']),
                'EW eleg': anual(liq(f(EW), 50)), 'indice (nao inv.)': anual(f(V1A)['ret_indice']),
                'CDI': anual(f(V1A)['ret_cdi'])})
print(pd.DataFrame(lin).round(4).to_string(index=False))

# ==================================================================== gravacao
sai_sc = sc[['CODISI', 'CODNEG', 'mes', 'subsetor', 'particao'] + MOM3 + VOL2 +
            ['alfa_fut', 'score_v4', 'decil_v4', 'vol_6m', 'vol6_eff']].copy()
sai_sc.to_parquet(DIR_INT + r'\v4_scores.parquet', index=False)
sai_ct = v4[['CODISI', 'CODNEG', 'mes', 'particao', 'n', 'peso_ew', 'peso_iv', 'vol_6m', 'vol6_eff',
             'score_v4', 'decil_v4']].copy()
sai_ct.to_parquet(DIR_INT + r'\v4_carteira.parquet', index=False)
xs = []
for rot, d in [('V4-EW', d4ew), ('V4', d4iv)]:
    x = d[['mes', 'mes_ret', 'particao', 'bruto', 'turnover']].copy()
    x['versao'] = rot
    for bps in NIVEIS_BPS:
        x[f'liq_{bps}bps'] = liq(d, bps)
    xs.append(x)
pd.concat(xs, ignore_index=True).to_parquet(DIR_INT + r'\v4_backtest.parquet', index=False)
dexc.to_parquet(DIR_INT + r'\v4_exclusoes_aleatorias.parquet', index=False)
print('\n' + '=' * 128)
print(f'GRAVADO v4_scores.parquet {sai_sc.shape} | v4_carteira.parquet {sai_ct.shape} | '
      f'v4_backtest.parquet ({2 * len(d4ew)}, 12) | v4_exclusoes_aleatorias.parquet {dexc.shape}')
print('V4 CONCLUIDA. V1, V2 e V3 intactas. PROIBIDO rodar V5 (D36). O numero de teste de V4 NAO e')
print('fora da amostra: o periodo ja tinha sido visto quando as tres mudancas foram propostas.')
print('=' * 128)


V4 (D36) -- score_v4 = 0,5*media(mom_3m,mom_6m,mom_12m ranks) - 0,5*media(vol_3m,vol_6m ranks)
exclusao do decil inferior (receita de V3, D31) sobre a MESMA base de V3 (painel de v2_scores,
linhas com os 6 ranks completos) -- escolha declarada: as UNICAS mudancas contra V3 sao (a),
(b) e (c) de D36; usar a base de 5 features seria uma QUARTA mudanca (populacao), nao pedida.
V4-EW = exclusao + peso igual (isola o score) | V4 = exclusao + peso 1/vol_6m (efeito completo)
EX-ANTE EM MERITO, POS-C1 EM DATA. NAO e fora da amostra: o periodo de teste ja foi visto.

S1 -- V1, V2 E V3 INTACTAS (esta celula LE e grava exclusivamente arquivos novos v4_*)
b4b5_previsoes.parquet (V1) SHA-256 = 5f67e19fa21e1b22f8d29064fca75ba06c498e2b62ac6b87ae67f332eb385e67
   v2_scores.parquet  sha c887216bcd6b  (lido, nao gravado)
   v2_backtest.parquet  sha 8919ae3b1f21  (lido, nao gravado)
   v2_carteira.parquet  sha 80f10316e7ff  (lido, nao gravado)
   v3_backtest.parquet  sha 805fbe57d759  (lido, nao gravado)

## E1b -- figuras novas pos-conselho (F11, F12)

F11 reproduz o teste de permutacao (item h da PARTE 2) e mostra, no mesmo histograma, onde caem os estatisticos observados de V3 (score_v2) e V4 (score_v4) contra a distribuicao nula de max|t| entre dez decis (2.000 replicas, semente 20260815). Decisao declarada: uma UNICA distribuicao nula serve para as duas arquiteturas porque a base (CODISI, mes, alfa_fut) e identica entre v2_scores e v4_scores (checado linha a linha, 26.439 pares, 0 divergencias) -- a nula depende so da estrutura mes/tamanho-de-decil, nao do score que gerou a particao.

F12 mostra a concentracao de peso de V4 (1/vol) mes a mes no TESTE, separando explicitamente da estatistica do painel completo (296 meses) que aparecia no relatorio da PARTE 2 -- a mediana do peso-maximo-mensal no TESTE e 2,25%, nao 2,62% (esse numero e do painel TREINO+TESTE).


In [ ]:
# -*- coding: utf-8 -*-
import sys, os, json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8')

RAIZ = r'C:\Users\lucca\quant2026'
DIR_INT = RAIZ + r'\intermediario'
DIR_OUT = RAIZ + r'\outputs'

# ------------------------------------------------------------------ estilo (mesmo de E1, cell 51)
plt.rcParams.update({
    'figure.dpi': 150, 'savefig.dpi': 150, 'font.size': 10.5, 'font.family': 'DejaVu Sans',
    'axes.edgecolor': '#333333', 'axes.labelcolor': '#1a1a1a', 'text.color': '#1a1a1a',
    'xtick.color': '#333333', 'ytick.color': '#333333', 'axes.grid': True,
    'grid.color': '#d9d9d9', 'grid.linewidth': 0.6, 'axes.axisbelow': True,
    'legend.frameon': False, 'axes.spines.top': False, 'axes.spines.right': False,
})
# paleta fixa desta sessao E1b (categorica para as arquiteturas + neutros para referencias)
# paleta reconciliada (2026-08-16): V1/V2/V3/V4EW/V4 alinhados ao canonico da celula 57
# (que herdou a paleta de E1/celula 51 sem colisao) -- ver diretiva de reconciliacao de cores.
COR = {
    'V1': '#4d4d4d', 'V2': '#2c6e91', 'V3': '#a4405e', 'V4EW': '#1f7a72', 'V4': '#c1652b',
    'preto': '#0b0b0b', 'cinza_med': '#898781', 'cinza_esc': '#52514e', 'cinza_clr': '#c3c2b7',
    'grade': '#e1e0d9',
}


def nw_t(x, lags=3):
    x = np.asarray(pd.Series(x).dropna(), dtype=float)
    n = len(x); mu = x.mean(); d = x - mu
    v = float(d @ d) / n
    for l in range(1, lags + 1):
        v += 2.0 * (1.0 - l / (lags + 1.0)) * (float(d[l:] @ d[:-l]) / n)
    return mu, np.sqrt(v / n), mu / np.sqrt(v / n)


def nw_t_batch(X, lags=3):
    # X shape (R, T) -- vetorizado ao longo do eixo 1 (tempo), por replica
    mu = np.nanmean(X, axis=1)
    d = X - mu[:, None]
    T = X.shape[1]
    v = np.nansum(d * d, axis=1) / T
    for l in range(1, lags + 1):
        w = 2.0 * (1.0 - l / (lags + 1.0))
        prod = d[:, l:] * d[:, :-l]
        v += w * (np.nansum(prod, axis=1) / T)
    se = np.sqrt(v / T)
    return mu, se, mu / se


def salvar(fig, nome):
    caminho = DIR_OUT + '\\' + nome
    fig.savefig(caminho, bbox_inches='tight', facecolor='white')
    plt.close(fig)
    tam = os.path.getsize(caminho)
    print(f'  GRAVADO {caminho}  ({tam / 1024:.1f} KB)')
    return caminho, tam


print('=' * 122)
print('E1b -- F11 (permutacao) e F12 (concentracao V4). PNG, 150 dpi, sem titulo na imagem.')
print('=' * 122)

# ======================================================================================= F11
print()
print('--- F11: PERMUTACAO ---')
print('Fonte: intermediario\\v4_scores.parquet (score_v4, decil_v4) + intermediario\\v2_scores.parquet (score_v2)')

v4s = pd.read_parquet(DIR_INT + r'\v4_scores.parquet')
v2s = pd.read_parquet(DIR_INT + r'\v2_scores.parquet')

t4 = v4s[v4s.particao == 'TESTE'][['CODISI', 'mes', 'alfa_fut', 'decil_v4']].copy()
t2 = v2s[v2s.particao == 'TESTE'][['CODISI', 'mes', 'alfa_fut', 'score_v2']].copy()

meses = sorted(t4['mes'].unique())
assert list(meses) == sorted(t2['mes'].unique()), 'meses de teste divergem entre v4_scores e v2_scores'
assert len(meses) == 103, f'esperado 103 meses de teste, achei {len(meses)}'

# decil de score_v2 (mesma receita de decil_v4: qcut sobre rank(method="first") por mes)
t2['decil_v2'] = t2.groupby('mes')['score_v2'].transform(
    lambda s: pd.qcut(s.rank(method='first'), 10, labels=False)
)

# checagem de que a base e identica (mesmo CODISI/mes/alfa_fut) -- decisao declarada: uma unica
# distribuicao nula serve para V3 e V4 porque a base (CODISI, mes, alfa_fut) e a MESMA (V4 usa a
# mesma base de V3/score_v2); a nula so depende da estrutura (mes -> conjunto de alfa_fut e
# tamanhos de decil), nao do score que gerou a particao original.
chk = t4.merge(t2[['CODISI', 'mes', 'alfa_fut']], on=['CODISI', 'mes'], suffixes=('_v4', '_v2'))
diff_alfa = (chk['alfa_fut_v4'] - chk['alfa_fut_v2']).abs()
n_total_pares = len(chk)
n_diff = int((diff_alfa > 1e-12).sum())
print(f'  base identica confirmada: {n_total_pares} pares CODISI-mes casados, {n_diff} com alfa_fut divergente (tolerancia 1e-12)')
assert n_diff == 0, 'a base de v4_scores e v2_scores diverge -- nula unica invalida, PARE'

# ---- observado: t NW3 por decil e max|t| para V4 e para V3 (score_v2) --------------------------
def matriz_decil_real(df, col_decil):
    piv = df.pivot_table(index='mes', columns=col_decil, values='alfa_fut', aggfunc='mean')
    piv = piv.reindex(index=meses, columns=range(10))
    return piv.to_numpy()  # (103, 10)

mat4 = matriz_decil_real(t4, 'decil_v4')
mat2 = matriz_decil_real(t2, 'decil_v2')

_, _, t4_dec = nw_t_batch(mat4.T)   # (10,) um t por decil
_, _, t2_dec = nw_t_batch(mat2.T)

obs4 = float(np.nanmax(np.abs(t4_dec)))
obs2 = float(np.nanmax(np.abs(t2_dec)))
d0_t4 = float(t4_dec[0])
d0_t2 = float(t2_dec[0])
print(f'  V4 (score_v4): t por decil = {np.round(t4_dec, 3).tolist()}')
print(f'  V4: decil inferior (d0) t = {d0_t4:.4f}  |  max|t| observado = {obs4:.4f}  (alvo 2.4853)')
print(f'  V3 (score_v2): t por decil = {np.round(t2_dec, 3).tolist()}')
print(f'  V3: decil inferior (d0) t = {d0_t2:.4f}  |  max|t| observado = {obs2:.4f}  (alvo 1.9277)')

# ---- nula: permutar dentro do mes, 2000 replicas, semente 20260815 -----------------------------
SEMENTE = 20260815
N_REP = 2000
rng = np.random.default_rng(SEMENTE)

# usa a base v4 (identica a v2 nos pares casados) para construir a nula; blocos de decil = tamanhos
# reais de decil_v4 por mes (contiguous apos ordenar por decil)
por_mes = []
for m in meses:
    g = t4[t4['mes'] == m].sort_values('decil_v4')
    alfa = g['alfa_fut'].to_numpy()
    decil = g['decil_v4'].to_numpy()
    # offsets de cada decil (0..9) dentro do vetor ordenado
    offs = [np.searchsorted(decil, d, side='left') for d in range(10)] + [len(decil)]
    por_mes.append((alfa, offs))

null_max = np.empty(N_REP)
# matriz por replica: (N_REP, 10, n_meses) medias de decil
medias = np.empty((N_REP, 10, len(meses)))
for i_m, (alfa, offs) in enumerate(por_mes):
    n = len(alfa)
    rand = rng.random((N_REP, n))
    idx = np.argsort(rand, axis=1)
    perm = alfa[idx]  # (N_REP, n)
    for d in range(10):
        bloco = perm[:, offs[d]:offs[d + 1]]
        medias[:, d, i_m] = np.nanmean(bloco, axis=1)

t_por_decil_replica = np.empty((N_REP, 10))
for d in range(10):
    _, _, t_por_decil_replica[:, d] = nw_t_batch(medias[:, d, :])

null_max = np.nanmax(np.abs(t_por_decil_replica), axis=1)

E_max = float(np.nanmean(null_max))
mediana_max = float(np.nanmedian(null_max))
p95_max = float(np.nanpercentile(null_max, 95))
p_adj_v4 = float(np.mean(null_max >= obs4))
p_adj_v3 = float(np.mean(null_max >= obs2))

print(f'  NULA (2000 replicas, semente {SEMENTE}): E[max|t|]={E_max:.4f} (alvo 2.3362)  mediana={mediana_max:.4f} (alvo 2.2653)  p95={p95_max:.4f} (alvo 3.4124)')
print(f'  p ajustado V4 = {p_adj_v4:.4f}  (alvo 0.3495)')
print(f'  p ajustado V3 = {p_adj_v3:.4f}  (alvo 0.7160)')

# ---- figura F11 ----------------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(7.2, 4.3))
ax.hist(null_max, bins=40, color=COR['cinza_clr'], edgecolor=COR['cinza_esc'], linewidth=0.5,
        label=f'nula (2.000 réplicas, semente {SEMENTE})')
ax.axvline(obs4, color=COR['V4'], linewidth=2.2, label=f'V4 observado = {obs4:.3f} (p ajustado {p_adj_v4:.3f})')
ax.axvline(obs2, color=COR['V3'], linewidth=2.2, linestyle='--', label=f'V3 observado = {obs2:.3f} (p ajustado {p_adj_v3:.3f})')
ax.axvline(E_max, color=COR['preto'], linewidth=1.0, linestyle=':', label=f'E[max|t|] nulo = {E_max:.3f}')
ax.set_xlabel('max|t| entre os dez decis (t Newey-West, lag 3)')
ax.set_ylabel('frequência (2.000 réplicas)')
ax.set_xlim(left=0)
ax.legend(loc='upper right', fontsize=8.5)
ax.text(0.02, 0.98,
        'Sob o nulo correto (máximo entre dez decis), o achado é\nindistinguível de sorte para as duas arquiteturas.',
        transform=ax.transAxes, va='top', ha='left', fontsize=8.5, color=COR['cinza_esc'])
fig.tight_layout()
p11, sz11 = salvar(fig, 'F11_permutacao.png')

# ======================================================================================= F12
print()
print('--- F12: CONCENTRACAO DE PESO EM V4 ---')
print('Fonte: intermediario\\v4_carteira.parquet (coluna peso_iv, particao TESTE)')

vc = pd.read_parquet(DIR_INT + r'\v4_carteira.parquet')
teste = vc[vc.particao == 'TESTE'].copy()
treino = vc[vc.particao == 'TREINO'].copy()

max_mensal = teste.groupby('mes')['peso_iv'].max().sort_index()
mediana_max_mensal = float(max_mensal.median())
max_absoluto = float(max_mensal.max())
mes_max = max_mensal.idxmax()
linha_mes_max = teste[(teste['mes'] == mes_max) & (teste['peso_iv'] == max_absoluto)].iloc[0]

n10 = int((max_mensal > 0.10).sum())
n25 = int((max_mensal > 0.25).sum())
n50 = int((max_mensal > 0.50).sum())

print(f'  TESTE (103 meses): peso maximo = {max_absoluto*100:.2f}%  em {mes_max}  CODNEG={linha_mes_max["CODNEG"]}  (alvo 29.67%, ENMT3, 2021-02)')
print(f'  mediana do peso-maximo-mensal (TESTE) = {mediana_max_mensal*100:.2f}%  (alvo 2.25% -- NAO 2.62%, que e o painel completo 296m)')
print(f'  meses > 10%: {n10}  (alvo 3)   > 25%: {n25}  (alvo 1)   > 50%: {n50}  (alvo 0)')

# extremo do painel completo (treino), para a anotacao separada
max_mensal_treino = treino.groupby('mes')['peso_iv'].max()
max_treino_abs = float(max_mensal_treino.max())
mes_treino_max = max_mensal_treino.idxmax()
linha_treino_max = treino[(treino['mes'] == mes_treino_max) & (treino['peso_iv'] == max_treino_abs)].iloc[0]
print(f'  [populacao DIFERENTE, so anotacao] TREINO: peso maximo = {max_treino_abs*100:.2f}%  em {mes_treino_max}  CODNEG={linha_treino_max["CODNEG"]}  vol_6m={linha_treino_max["vol_6m"]:.2e}  (alvo 99.98%, CBMA4, 2014-06)')

# n efetivo (1/HHI medio mensal), TESTE, V4 vs V4-EW vs EW PIT
def n_efetivo_medio(df, col_peso):
    hhi = df.groupby('mes')[col_peso].apply(lambda s: float((s.to_numpy() ** 2).sum()))
    return float((1.0 / hhi).mean())

n_ef_v4 = n_efetivo_medio(teste, 'peso_iv')
n_ef_v4ew = n_efetivo_medio(teste, 'peso_ew')

a8 = pd.read_parquet(DIR_INT + r'\a7_split.parquet')
elig_por_mes = a8[(a8['particao'] == 'TESTE') & (a8['elegivel'] == True)].groupby('mes').size()
n_ef_ewpit = float(elig_por_mes.reindex(sorted(teste['mes'].unique())).mean())

print(f'  n efetivo medio (TESTE): V4={n_ef_v4:.1f} (alvo 154.2)  V4-EW={n_ef_v4ew:.1f} (alvo 230.5)  EW-PIT={n_ef_ewpit:.1f} (alvo 300.0)')

# ---- figura F12 -----------------------------------------------------------------------------
fig = plt.figure(figsize=(9.0, 4.6))
gs = fig.add_gridspec(1, 4)
ax = fig.add_subplot(gs[0, :3])
ax2 = fig.add_subplot(gs[0, 3])

xs = pd.to_datetime([str(m) for m in max_mensal.index])
ax.plot(xs, max_mensal.to_numpy() * 100, color=COR['V4'], linewidth=1.6, marker='o', markersize=3,
        label='peso máximo individual de V4 (1/vol), por mês -- TESTE')
for nivel, cor in [(10, COR['cinza_med']), (25, COR['cinza_esc']), (50, COR['preto'])]:
    ax.axhline(nivel, color=cor, linestyle='--', linewidth=0.9)
    ax.text(xs.max(), nivel, f' {nivel}%', va='bottom', ha='left', fontsize=8, color=cor)
ax.axhline(mediana_max_mensal * 100, color=COR['V4'], linestyle=':', linewidth=1.0)
ax.text(xs.min(), mediana_max_mensal * 100, f'mediana {mediana_max_mensal*100:.2f}% ', va='bottom',
        ha='left', fontsize=8, color=COR['V4'])
ax.set_ylabel('peso máximo individual no mês (%)')
ax.set_ylim(bottom=0)
ax.legend(loc='upper left', fontsize=8.5)
ax.text(0.99, 0.97,
        f'Correção: a mediana do painel completo (296 meses,\nTREINO+TESTE) é 2,62% -- a série mostrada aqui é\nSÓ o TESTE (103 meses): mediana {mediana_max_mensal*100:.2f}%.\n'
        f'No TREINO houve um mês (2014-06, CBMA4,\nvol_6m≈{linha_treino_max["vol_6m"]:.1e}) com peso {max_treino_abs*100:.2f}%\n'
        f'-- fora do período mostrado, população diferente.',
        transform=ax.transAxes, va='top', ha='right', fontsize=7.6, color=COR['cinza_esc'])

nomes = ['V4\n(1/vol)', 'V4-EW\n(peso igual)', 'EW\nponto-in-time']
vals = [n_ef_v4, n_ef_v4ew, n_ef_ewpit]
cores_barras = [COR['V4'], COR['V4EW'], COR['cinza_med']]
ax2.bar(nomes, vals, color=cores_barras, edgecolor=COR['preto'], linewidth=0.6)
for i, v in enumerate(vals):
    ax2.text(i, v, f'{v:.0f}', ha='center', va='bottom', fontsize=8.5)
ax2.set_ylabel('nº efetivo de nomes (1/HHI médio, TESTE)')
ax2.set_ylim(bottom=0)
fig.tight_layout()
p12, sz12 = salvar(fig, 'F12_concentracao_v4.png')

print()
print('RESUMO ARQUIVOS:')
print(f'  {p11}  ({sz11/1024:.1f} KB)')
print(f'  {p12}  ({sz12/1024:.1f} KB)')
print('=' * 122)


E1b -- F11 (permutacao) e F12 (concentracao V4). PNG, 150 dpi, sem titulo na imagem.

--- F11: PERMUTACAO ---
Fonte: intermediario\v4_scores.parquet (score_v4, decil_v4) + intermediario\v2_scores.parquet (score_v2)
  base identica confirmada: 26439 pares CODISI-mes casados, 0 com alfa_fut divergente (tolerancia 1e-12)
  V4 (score_v4): t por decil = [-2.485, -0.304, -1.175, -1.416, 1.372, 0.798, -0.013, 0.344, 1.456, -0.69]
  V4: decil inferior (d0) t = -2.4853  |  max|t| observado = 2.4853  (alvo 2.4853)
  V3 (score_v2): t por decil = [-1.932, -1.022, 0.566, -1.227, -1.084, 0.175, 0.118, 0.581, 0.422, -0.047]
  V3: decil inferior (d0) t = -1.9323  |  max|t| observado = 1.9323  (alvo 1.9277)
  NULA (2000 replicas, semente 20260815): E[max|t|]=2.3147 (alvo 2.3362)  mediana=2.2315 (alvo 2.2653)  p95=3.4115 (alvo 3.4124)
  p ajustado V4 = 0.3325  (alvo 0.3495)
  p ajustado V3 = 0.7200  (alvo 0.7160)
  GRAVADO C:\Users\lucca\quant2026\outputs\F11_permutacao.png  (65.6 KB)

--- F12: CONCENTR

## E1b -- figuras regeradas pos-conselho (F7, F8)

F1-F4 nao foram tocadas (dados, nao afetados por V4 nem pelas correcoes da PARTE 1). F7 e F8 leem direto de v4_scores.parquet / v4_exclusoes_aleatorias.parquet / v4_backtest.parquet (alem das fontes ja usadas por V1/V2/V3 em E1) -- nenhum numero digitado a mao (S1, impresso figura a figura).

In [ ]:
import sys
import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8')

pd.set_option('display.width', 250)

RAIZ = r'C:\Users\lucca\quant2026'
DIR_INT = RAIZ + r'\intermediario'
DIR_OUT = RAIZ + r'\outputs'

# ------------------------------------------------------------------ estilo identico ao da celula E1
plt.rcParams.update({
    'figure.dpi': 150, 'savefig.dpi': 150, 'font.size': 10.5, 'font.family': 'DejaVu Sans',
    'axes.edgecolor': '#333333', 'axes.labelcolor': '#1a1a1a', 'text.color': '#1a1a1a',
    'xtick.color': '#333333', 'ytick.color': '#333333', 'axes.grid': True,
    'grid.color': '#d9d9d9', 'grid.linewidth': 0.6, 'axes.axisbelow': True,
    'legend.frameon': False, 'axes.spines.top': False, 'axes.spines.right': False,
})
# COR/LS/MK: mesmo dicionario da celula E1 (cell 51), estendido com V4-EW e V4.
# DECISAO (documentada no relatorio do fork): a paleta azul/laranja/aqua/amarelo/magenta
# passada na diretiva NAO foi usada -- ela conflitaria com as cores ja em uso em F1-F4
# (intocadas), que reutilizam COR['V1']/COR['V2'] (cinza/azul-aco). Em vez disso a paleta
# existente foi estendida com dois tons novos, no mesmo registro sobrio/legivel em P&B.
COR = {'V1': '#4d4d4d', 'V2': '#2c6e91', 'V3': '#a4405e', 'EW': '#5a8a3c',
       'indice': '#111111', 'cdi': '#b08a2e', 'ibov': '#8858a8', 'base': '#8c8c8c',
       'V4EW': '#1f7a72', 'V4': '#c1652b', 'ewpit': '#1a4d6b'}

print('=' * 122)
print('E1b -- REGERACAO DE F7 e F8 POS-CONSELHO. PNG, 150 dpi, sem titulo na imagem.')
print('=' * 122)


def nw_t(x, lags=3):
    x = np.asarray(pd.Series(x).dropna(), dtype=float)
    n = len(x); mu = x.mean(); d = x - mu
    v = float(d @ d) / n
    for l in range(1, lags + 1):
        v += 2.0 * (1.0 - l / (lags + 1.0)) * (float(d[l:] @ d[:-l]) / n)
    return mu, np.sqrt(v / n), mu / np.sqrt(v / n)


def anual(r):
    r = np.asarray(pd.Series(r).dropna(), dtype=float)
    return 100.0 * ((np.prod(1.0 + r)) ** (12.0 / len(r)) - 1.0)


def pct_of(dist, valor):
    dist = np.asarray(dist, dtype=float)
    return 100.0 * float((dist <= valor).sum()) / len(dist)


def salvar(fig, nome):
    caminho = DIR_OUT + '\\' + nome
    fig.savefig(caminho, bbox_inches='tight', facecolor='white')
    plt.close(fig)
    tam = os.path.getsize(caminho)
    print(f'  GRAVADO {caminho}  ({tam / 1024:.1f} KB)')
    return caminho


arquivos_gerados = []

# ==================================================================== insumos
bench = pd.read_parquet(DIR_INT + r'\a6_benchmark_e_rf.parquet')
bi = bench.set_index('mes')

c3 = pd.read_parquet(DIR_INT + r'\c3_curva_custo.parquet')
v2bt = pd.read_parquet(DIR_INT + r'\v2_backtest.parquet')
v3bt = pd.read_parquet(DIR_INT + r'\v3_backtest.parquet')
v4bt = pd.read_parquet(DIR_INT + r'\v4_backtest.parquet')

SERIES = {
    'V1': c3[c3['versao'] == 'C2-A'].sort_values('mes').reset_index(drop=True),
    'V2': v2bt[v2bt['versao'] == 'V2-A'].sort_values('mes').reset_index(drop=True),
    'V3': v3bt[v3bt['versao'] == 'V3'].sort_values('mes').reset_index(drop=True),
    'V4EW': v4bt[v4bt['versao'] == 'V4-EW'].sort_values('mes').reset_index(drop=True),
    'V4': v4bt[v4bt['versao'] == 'V4'].sort_values('mes').reset_index(drop=True),
}
TESTE = {k: d[d['particao'] == 'TESTE'].reset_index(drop=True) for k, d in SERIES.items()}
for k in TESTE:
    assert list(TESTE[k]['mes']) == list(TESTE['V1']['mes']), f'meses desalinhados em {k}'

v1_real = anual(TESTE['V1']['liq_50bps'])
v2_real = anual(TESTE['V2']['liq_50bps'])
v3_real = anual(TESTE['V3']['liq_50bps'])
v4ew_real = anual(TESTE['V4EW']['liq_50bps'])
v4_real = anual(TESTE['V4']['liq_50bps'])
print(f'  reais (TESTE, liq50): V1={v1_real:.4f} V2={v2_real:.4f} V3={v3_real:.4f} '
      f'V4-EW={v4ew_real:.4f} V4={v4_real:.4f}')

# ==================================================================== F7 -- distribuicoes aleatorias
print('\n[F7] fontes: d1_aleatorias.parquet (V1), v2_aleatorias.parquet (V2), '
      'v3_exclusoes_aleatorias.parquet (V3), v4_exclusoes_aleatorias.parquet (V4-EW, V4)')
d1a = pd.read_parquet(DIR_INT + r'\d1_aleatorias.parquet')
v2a = pd.read_parquet(DIR_INT + r'\v2_aleatorias.parquet')
v3ea = pd.read_parquet(DIR_INT + r'\v3_exclusoes_aleatorias.parquet')
v4ea = pd.read_parquet(DIR_INT + r'\v4_exclusoes_aleatorias.parquet')

pct_v1 = pct_of(d1a['anual_50bps'], v1_real)
pct_v2 = pct_of(v2a['anual_50bps'], v2_real)
pct_v3 = pct_of(v3ea['liq50'], v3_real)
pct_v4ew = pct_of(v4ea['ew_liq50'], v4ew_real)
pct_v4 = pct_of(v4ea['iv_liq50'], v4_real)
print(f'  percentis (liq50): V1={pct_v1:.1f} V2={pct_v2:.1f} V3={pct_v3:.1f} '
      f'V4-EW={pct_v4ew:.1f} V4={pct_v4:.1f}')

fig, (axA, axB) = plt.subplots(1, 2, figsize=(13, 5))

# painel A -- selecoes aleatorias (V1, V2)
axA.hist(d1a['anual_50bps'], bins=22, color=COR['V1'], edgecolor='white', alpha=0.55,
         label='200 seleções aleatórias (V1)')
axA.hist(v2a['anual_50bps'], bins=22, color=COR['V2'], edgecolor='white', alpha=0.55,
         label='200 seleções aleatórias (V2)')
axA.axvline(v1_real, color=COR['V1'], lw=2.2, ls='-',
            label=f'V1 real = {v1_real:.2f}%aa (percentil {pct_v1:.0f})')
axA.axvline(v2_real, color=COR['V2'], lw=2.2, ls='--',
            label=f'V2 real = {v2_real:.2f}%aa (percentil {pct_v2:.0f})')
axA.axvline(0, color='#999999', lw=0.8)
axA.set_xlabel('retorno anualizado líquido de 50bps (200 seleções aleatórias)')
axA.set_ylabel('frequência')
axA.legend(loc='upper left', fontsize=8.3)

# painel B -- exclusoes aleatorias (V3, V4-EW, V4)
bins = np.linspace(
    min(v3ea['liq50'].min(), v4ea['ew_liq50'].min(), v4ea['iv_liq50'].min()),
    max(v3ea['liq50'].max(), v4ea['ew_liq50'].max(), v4ea['iv_liq50'].max()), 26)
axB.hist(v3ea['liq50'], bins=bins, histtype='step', color=COR['V3'], lw=1.8,
         label='200 exclusões aleatórias (V3)')
axB.hist(v4ea['ew_liq50'], bins=bins, histtype='step', color=COR['V4EW'], lw=1.8,
         label='200 exclusões aleatórias (V4-EW)')
axB.hist(v4ea['iv_liq50'], bins=bins, histtype='step', color=COR['V4'], lw=1.8,
         label='200 exclusões aleatórias (V4, 1/vol)')
axB.axvline(v3_real, color=COR['V3'], lw=2.4, ls='-',
            label=f'V3 real = {v3_real:.2f}%aa (percentil {pct_v3:.0f})')
axB.axvline(v4ew_real, color=COR['V4EW'], lw=2.4, ls='--',
            label=f'V4-EW real = {v4ew_real:.2f}%aa (percentil {pct_v4ew:.0f})')
axB.axvline(v4_real, color=COR['V4'], lw=2.4, ls='-.',
            label=f'V4 real = {v4_real:.2f}%aa (percentil {pct_v4:.0f})')
axB.axvline(0, color='#999999', lw=0.8)
axB.set_xlabel('retorno anualizado líquido de 50bps (200 exclusões aleatórias, 10% da base)')
axB.legend(loc='upper left', fontsize=8.0)

salvar(fig, 'F7_distribuicoes_aleatorias.png')
arquivos_gerados.append((
    'F7_distribuicoes_aleatorias.png',
    'Esquerda: 200 seleções aleatórias com V1 e V2 marcadas. Direita: 200 exclusões aleatórias '
    'com V3, V4-EW e V4 marcadas — todas líquidas de 50 bps, período de teste.',
    'd1_aleatorias.parquet, v2_aleatorias.parquet, v3_exclusoes_aleatorias.parquet, '
    'v4_exclusoes_aleatorias.parquet, c3_curva_custo.parquet, v2_backtest.parquet, '
    'v3_backtest.parquet, v4_backtest.parquet',
    'Mostra se a seleção (ou exclusão) do modelo supera o acaso, dada a mesma construção e o '
    f'mesmo universo. V3, V4-EW e V4 ficam no percentil 100 das suas 200 exclusões aleatórias '
    '(a receita bate o acaso sempre) — o teste que de fato decide (permutação dentro do mês, F11) '
    'é outro.',
    'Não prova capacidade preditiva: com ~331 meses necessários para p<0,05 [F], é não-refutação; '
    'as distribuições de seleção e de exclusão não são comparáveis entre si (construção diferente); '
    'a distribuição de V4 (peso 1/vol) tem giro e dispersão maiores que a de V4-EW, então o '
    'percentil 100 nas duas não significa a mesma coisa em risco.'))

# ==================================================================== F8 -- assimetria do sinal
print('\n[F8] fontes: v4_scores.parquet (score_v4, TESTE e TREINO), v2_scores.parquet (score_v2, '
      'serie secundaria)')
v4s = pd.read_parquet(DIR_INT + r'\v4_scores.parquet')
v2s = pd.read_parquet(DIR_INT + r'\v2_scores.parquet')


def decis_v4(particao):
    sub = v4s[(v4s['particao'] == particao) & v4s['alfa_fut'].notna()]
    lin = []
    for m, g in sub.groupby('mes'):
        for d in range(10):
            v = g.loc[g['decil_v4'] == d, 'alfa_fut']
            if len(v):
                lin.append({'mes': m, 'decil': d, 'alfa': float(v.mean())})
    dd = pd.DataFrame(lin)
    out = []
    for d in range(10):
        s = dd.loc[dd['decil'] == d, 'alfa']
        mu, se, t = nw_t(s, 3)
        out.append({'decil': d, 'media_%mes': 100 * mu, 'ep_%mes': 100 * se, 't_NW3': t})
    return pd.DataFrame(out)


def decis_v2(particao):
    sub = v2s[(v2s['particao'] == particao) & v2s['alfa_fut'].notna()]
    lin = []
    for m, g in sub.groupby('mes'):
        if len(g) < 20:
            continue
        g = g.copy()
        g['dec'] = pd.qcut(g['score_v2'].rank(method='first'), 10, labels=False)
        for d in range(10):
            v = g.loc[g['dec'] == d, 'alfa_fut']
            if len(v):
                lin.append({'mes': m, 'decil': d, 'alfa': float(v.mean())})
    dd = pd.DataFrame(lin)
    out = []
    for d in range(10):
        s = dd.loc[dd['decil'] == d, 'alfa']
        mu, se, t = nw_t(s, 3)
        out.append({'decil': d, 'media_%mes': 100 * mu, 'ep_%mes': 100 * se, 't_NW3': t})
    return pd.DataFrame(out)


res_v4_teste = decis_v4('TESTE')
res_v4_treino = decis_v4('TREINO')
res_v2_teste = decis_v2('TESTE')
res_v2_treino = decis_v2('TREINO')

print('  score_v4 TESTE decil 0 (pior):', round(res_v4_teste.loc[0, 'media_%mes'], 4),
      't=', round(res_v4_teste.loc[0, 't_NW3'], 3))
print('  score_v4 TREINO decil 0 (pior):', round(res_v4_treino.loc[0, 'media_%mes'], 4),
      't=', round(res_v4_treino.loc[0, 't_NW3'], 3))
alvo_teste, alvo_treino = -1.2992, 1.1723
d_teste = res_v4_teste.loc[0, 'media_%mes'] - alvo_teste
d_treino = res_v4_treino.loc[0, 'media_%mes'] - alvo_treino
print(f'  divergencia vs alvo reconciliado: teste {d_teste:+.4f}pp, treino {d_treino:+.4f}pp '
      '(deve ser ~0)')
assert abs(d_teste) < 0.01 and abs(d_treino) < 0.01, 'decil inferior de score_v4 nao reproduziu'

lo4 = (res_v4_teste['media_%mes'] - res_v4_teste['ep_%mes']).tolist() + \
      (res_v4_treino['media_%mes'] - res_v4_treino['ep_%mes']).tolist()
hi4 = (res_v4_teste['media_%mes'] + res_v4_teste['ep_%mes']).tolist() + \
      (res_v4_treino['media_%mes'] + res_v4_treino['ep_%mes']).tolist()
ymin = min(lo4 + res_v2_teste['media_%mes'].tolist() + res_v2_treino['media_%mes'].tolist()) - 0.3
ymax = max(hi4 + res_v2_teste['media_%mes'].tolist() + res_v2_treino['media_%mes'].tolist()) + 0.3

fig, (axL, axR) = plt.subplots(1, 2, figsize=(13, 5.4), sharey=True)
x = np.arange(1, 11)  # decil 1 (pior) .. 10 (melhor), exibicao 1-based do decil_v4 0-9

for ax, res_v4, res_v2, rotulo in [(axL, res_v4_teste, res_v2_teste, 'TESTE'),
                                    (axR, res_v4_treino, res_v2_treino, 'TREINO')]:
    cores_dec = [COR['V4'] if d == 0 else (COR['V2'] if d == 9 else '#c9c9c9')
                 for d in res_v4['decil']]
    ax.bar(x, res_v4['media_%mes'], yerr=res_v4['ep_%mes'], color=cores_dec, edgecolor='white',
           capsize=3, error_kw=dict(lw=1.0, ecolor='#333333'), width=0.68,
           label='score_v4 (5 features, 50/50)' if rotulo == 'TESTE' else None)
    ax.plot(x, res_v2['media_%mes'], color=COR['V2'], marker='o', mfc='none', mec=COR['V2'],
            ms=6, lw=1.3, ls=':', label='score_v2 (6 features) -- comparação' if rotulo == 'TESTE' else None)
    ax.axhline(0, color='#333333', lw=1.0)
    ax.set_ylim(ymin, ymax)
    ax.set_xticks(x)
    ax.set_xlabel(f'decil de score_v4 -- {rotulo} (1 = pior, 10 = melhor)')
    d0 = res_v4.loc[0]
    # caixa de texto ancorada em coordenadas do eixo (nao de dados) -- evita colisao com
    # a barra do decil 1 independente do sinal (positivo no treino, negativo no teste)
    cy = 0.94 if rotulo == 'TESTE' else 0.06
    va = 'top' if rotulo == 'TESTE' else 'bottom'
    ax.text(0.97, cy, f'decil 1: {d0["media_%mes"]:+.3f}%/mês (t={d0["t_NW3"]:+.2f})',
            transform=ax.transAxes, ha='right', va=va, fontsize=8.7, color=COR['V4'],
            bbox=dict(facecolor='white', edgecolor='none', alpha=0.75, pad=1.5))
axL.set_ylabel('alfa_fut médio realizado (%/mês), erro-padrão Newey-West(3)')
axL.legend(loc='upper left', fontsize=8.3)
fig.text(0.5, -0.03,
         'O achado que fundamenta a exclusão do decil inferior NÃO REPLICA entre partições: '
         'positivo no treino, negativo no teste. Trocar o score (v2 -> v4) não muda essa propriedade.',
         ha='center', fontsize=9.3, color='#1a1a1a')

salvar(fig, 'F8_assimetria_do_sinal.png')
arquivos_gerados.append((
    'F8_assimetria_do_sinal.png',
    'Alfa_fut médio realizado por decil de score_v4 (barras), com score_v2 sobreposto como série '
    'secundária, painel esquerdo TESTE e painel direito TREINO (mesma escala Y), erro-padrão '
    'Newey-West(3).',
    'v4_scores.parquet, v2_scores.parquet',
    'É o teste que decide se a exclusão do decil inferior é uma propriedade do score ou da amostra: '
    'no teste o decil 1 é fortemente negativo (score_v4: -1,299%/mês, t-2,49); no treino o mesmo '
    'decil é POSITIVO (score_v4: +1,172%/mês, t+1,81) -- exatamente o padrão já visto em score_v2/V3.',
    'Não mostra retorno de carteira nem custo -- é diagnóstico de sinal, não backtest; a mudança de '
    'score (v2->v4) não resolve a não-replicação, só desloca um pouco os números.'))

print('\nS1: fonte de cada figura impressa acima. S2: eixo Y de F8 inclui o zero nos dois painéis '
      '(compartilhado). S4: nenhuma figura usa Sharpe.')
for nome, desc, fonte, mostra, nao_mostra in arquivos_gerados:
    print(f'\n{nome}')
    print(f'  desc : {desc}')
    print(f'  fonte: {fonte}')
    print(f'  mostra: {mostra}')
    print(f'  nao_mostra: {nao_mostra}')


E1b -- REGERACAO DE F7 e F8 POS-CONSELHO. PNG, 150 dpi, sem titulo na imagem.
  reais (TESTE, liq50): V1=-1.6673 V2=0.8555 V3=5.6133 V4-EW=6.2645 V4=7.2618

[F7] fontes: d1_aleatorias.parquet (V1), v2_aleatorias.parquet (V2), v3_exclusoes_aleatorias.parquet (V3), v4_exclusoes_aleatorias.parquet (V4-EW, V4)
  percentis (liq50): V1=67.0 V2=96.0 V3=100.0 V4-EW=100.0 V4=100.0
  GRAVADO C:\Users\lucca\quant2026\outputs\F7_distribuicoes_aleatorias.png  (88.7 KB)

[F8] fontes: v4_scores.parquet (score_v4, TESTE e TREINO), v2_scores.parquet (score_v2, serie secundaria)
  score_v4 TESTE decil 0 (pior): -1.2992 t= -2.485
  score_v4 TREINO decil 0 (pior): 1.1723 t= 1.808
  divergencia vs alvo reconciliado: teste +0.0000pp, treino -0.0000pp (deve ser ~0)
  GRAVADO C:\Users\lucca\quant2026\outputs\F8_assimetria_do_sinal.png  (110.9 KB)

S1: fonte de cada figura impressa acima. S2: eixo Y de F8 inclui o zero nos dois painéis (compartilhado). S4: nenhuma figura usa Sharpe.

F7_distribuicoes_aleatoria

## E1b -- figuras regeradas pos-conselho (F5, F6, F9, F10)

F1-F4 nao foram tocadas. F5/F6/F9/F10 foram regeradas lendo os parquets ja em disco
(nenhuma celula anterior foi reexecutada, nenhum parquet foi alterado, nenhuma variante
nova foi rodada). A F9 anterior estava FACTUALMENTE ERRADA -- ancorava a cascata de V3
no placebo aleatorio e publicava a manchete +2,05 pp/ano; a versao abaixo reancora no EW
da propria base (item 3 do plano do conselho) e acrescenta a cascata de V4.

In [ ]:
import sys
import os

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8')

RAIZ = r'C:\Users\lucca\quant2026'
DIR_INT = RAIZ + r'\intermediario'
DIR_OUT = RAIZ + r'\outputs'
if RAIZ + r'\src' not in sys.path:
    sys.path.insert(0, RAIZ + r'\src')
import motor  # noqa: E402

# ---------------------------------------------------------------- estilo identico a E1 (celula 51)
plt.rcParams.update({
    'figure.dpi': 150, 'savefig.dpi': 150, 'font.size': 10.5, 'font.family': 'DejaVu Sans',
    'axes.edgecolor': '#333333', 'axes.labelcolor': '#1a1a1a', 'text.color': '#1a1a1a',
    'xtick.color': '#333333', 'ytick.color': '#333333', 'axes.grid': True,
    'grid.color': '#d9d9d9', 'grid.linewidth': 0.6, 'axes.axisbelow': True,
    'legend.frameon': False, 'axes.spines.top': False, 'axes.spines.right': False,
})
# paleta: series ja estabelecidas em E1 (V1/V2/V3/EW/cdi) reaproveitadas por continuidade visual;
# novas entidades (V4-EW, V4, EW-PIT-como-manchete) ganham cores novas; indice interno e REBAIXADO
# a linha secundaria cinza (era preto solido em E1/F3 -- aqui, pos-conselho, e "nao investivel")
# paleta reconciliada (2026-08-16): V4EW/V4 alinhados ao canonico da celula 57; PIT (EW
# ponto-no-tempo) recebe o tom navy canonico 'ewpit' para nao colidir com 'indice' (cinza,
# rebaixado de proposito -- decisao mantida, ver comentario acima).
COR = {'V1': '#4d4d4d', 'V2': '#2c6e91', 'V3': '#a4405e', 'V4EW': '#1f7a72', 'V4': '#c1652b',
       'PIT': '#1a4d6b', 'cdi': '#b08a2e', 'indice': '#8c8c8c'}
LS = {'V1': '-', 'V2': '--', 'V3': '-.', 'V4EW': '--', 'V4': '-', 'PIT': '--', 'cdi': ':', 'indice': '-.'}
LW = {'V1': 1.3, 'V2': 1.3, 'V3': 1.3, 'V4EW': 1.5, 'V4': 2.3, 'PIT': 1.9, 'cdi': 1.2, 'indice': 1.1}

print('=' * 122)
print('E1b -- figuras regeradas pos-conselho (F5, F6, F9, F10). PNG 150dpi, sem titulo, legendas PT-BR.')
print('S1: cada figura imprime de qual(is) parquet(s) leu. Nenhum numero digitado a mao.')
print('=' * 122)


def nw_t(x, lags=3):
    x = np.asarray(pd.Series(x).dropna(), dtype=float)
    n = len(x); mu = x.mean(); d = x - mu
    v = float(d @ d) / n
    for l in range(1, lags + 1):
        v += 2.0 * (1.0 - l / (lags + 1.0)) * (float(d[l:] @ d[:-l]) / n)
    se = np.sqrt(v / n)
    return mu, se, mu / se


def anual(r):
    r = np.asarray(pd.Series(r).dropna(), dtype=float)
    return 100.0 * ((np.prod(1.0 + r)) ** (12.0 / len(r)) - 1.0)


def salvar(fig, nome):
    caminho = DIR_OUT + '\\' + nome
    fig.savefig(caminho, bbox_inches='tight', facecolor='white')
    plt.close(fig)
    tam = os.path.getsize(caminho)
    print(f'  GRAVADO {caminho}  ({tam / 1024:.1f} KB)')
    return caminho


arquivos_gerados = []

# ==================================================================== insumos comuns
print('\n[insumos] intermediario\\a6_benchmark_e_rf.parquet, c3_curva_custo.parquet, '
      'v2_backtest.parquet, v3_backtest.parquet, v4_backtest.parquet')
bench = pd.read_parquet(DIR_INT + r'\a6_benchmark_e_rf.parquet').set_index('mes')
c3 = pd.read_parquet(DIR_INT + r'\c3_curva_custo.parquet')
v2bt = pd.read_parquet(DIR_INT + r'\v2_backtest.parquet')
v3bt = pd.read_parquet(DIR_INT + r'\v3_backtest.parquet')
v4bt = pd.read_parquet(DIR_INT + r'\v4_backtest.parquet')

SERIES = {
    'V1': c3[c3['versao'] == 'C2-A'],
    'V2': v2bt[v2bt['versao'] == 'V2-A'],
    'V3': v3bt[v3bt['versao'] == 'V3'],
    'V4EW': v4bt[v4bt['versao'] == 'V4-EW'],
    'V4': v4bt[v4bt['versao'] == 'V4'],
    'PIT': c3[c3['versao'] == 'EW_universo'],
}
TE = {k: v[v['particao'] == 'TESTE'].sort_values('mes_ret').reset_index(drop=True) for k, v in SERIES.items()}
for k, d in TE.items():
    assert len(d) == 103, f'{k}: TESTE deveria ter 103 meses, tem {len(d)}'
meses_te = list(TE['V3']['mes_ret'])
for k, d in TE.items():
    assert list(d['mes_ret']) == meses_te, f'{k}: meses desalinhados'

cdi_te = bench.loc[meses_te, 'ret_cdi']
idx_te = bench.loc[meses_te, 'ret_indice']

LEG_LABEL = {
    'V1': 'V1-A (C2-A)', 'V2': 'V2-A', 'V3': 'V3', 'V4EW': 'V4-EW (peso igual)',
    'V4': 'V4 (peso 1/vol)', 'PIT': 'EW ponto-no-tempo (comparador de manchete)',
}

print('\nDESVIO ENCONTRADO E RECONCILIADO (nao corrigido nos dados, so declarado aqui): "EW '
      'point-in-time" (6,7162%aa) e "EW elegivel" (5,9073%aa) sao a MESMA serie de retorno bruto -- '
      "c3_curva_custo, versao=='EW_universo' -- so que o COMPARADOR DE MANCHETE (EW PIT, como usado "
      'nos excedentes do item (d) da PARTE 2 e na celula 33/34) e SEMPRE quotado a CUSTO ZERO (bruto), '
      'por convencao (e um bogey passivo, nao uma estrategia negociada): confirmado recompondo o '
      'excedente V4 vs EW PIT com PIT em bruto -> -0,0001%/mes, t -0,001, bate EXATO o alvo da PARTE 2; '
      'com PIT em liq50 o excedente seria +0,0636%/mes -- NAO e o numero da PARTE 2. F5/F6/F9/F10 usam '
      'PIT em BRUTO (custo zero) em toda comparacao, e um footnote deixa isso explicito.')

# ==================================================================== F5 -- curva acumulada TESTE
print('\n[F5] fonte: c3_curva_custo.parquet (V1-A, EW_universo/PIT), v2_backtest.parquet (V2-A), '
      'v3_backtest.parquet (V3), v4_backtest.parquet (V4-EW, V4), a6_benchmark_e_rf.parquet (CDI, indice)')
fig, ax = plt.subplots(figsize=(9.5, 5.6))
x = pd.to_datetime([pd.Period(m, freq='M').to_timestamp(how='end') for m in meses_te])

acumuladas = {}
for k in ['V1', 'V2', 'V3', 'V4EW', 'V4']:
    acumuladas[k] = 100.0 * (1.0 + TE[k]['liq_50bps']).cumprod()
acumuladas['PIT'] = 100.0 * (1.0 + TE['PIT']['bruto']).cumprod()  # PIT sempre a custo zero (convencao)
acumuladas['CDI'] = 100.0 * (1.0 + cdi_te.reset_index(drop=True)).cumprod()
acumuladas['indice'] = 100.0 * (1.0 + idx_te.reset_index(drop=True)).cumprod()

for k in ['V1', 'V2', 'V3', 'V4EW']:
    ax.plot(x, acumuladas[k], color=COR[k], ls=LS[k], lw=LW[k], label=LEG_LABEL.get(k, k))
ax.plot(x, acumuladas['indice'], color=COR['indice'], ls=LS['indice'], lw=LW['indice'],
        label='índice interno — NÃO INVESTÍVEL')
ax.plot(x, acumuladas['CDI'], color=COR['cdi'], ls=LS['cdi'], lw=LW['cdi'], label='CDI')
ax.plot(x, acumuladas['PIT'], color=COR['PIT'], ls=LS['PIT'], lw=LW['PIT'],
        label=LEG_LABEL['PIT'] + ' (custo zero)')
ax.plot(x, acumuladas['V4'], color=COR['V4'], ls=LS['V4'], lw=LW['V4'], label=LEG_LABEL['V4'],
        marker='o', markevery=12, markersize=4)

ax.set_ylim(bottom=0)
ax.set_ylabel('retorno acumulado, líquido de 50 bps (base 100 em 2018-01)')
ax.set_xlabel('TESTE: 2018-01 a 2026-07 (retornos 2018-02 a 2026-08)')
leg = ax.legend(loc='upper left', fontsize=8.3, ncol=1, frameon=True, facecolor='white',
                 framealpha=0.94, edgecolor='#cccccc')
leg.get_frame().set_linewidth(0.6)
exc_pit = 100 * (TE['V4']['liq_50bps'].to_numpy() - TE['PIT']['bruto'].to_numpy())
mu, se, t = nw_t(exc_pit, 3)
empate = abs(mu) < 0.01
fig.text(0.5, -0.02,
          f'V4 (líquido de 50bps) contra o comparador de manchete (EW ponto-no-tempo, custo zero): '
          f'excedente {mu:+.4f}%/mês, t NW3 {t:+.3f}'
          + (' — EMPATE.' if empate else '.')
          + f' Nenhuma versão supera o CDI ({anual(cdi_te):.4f}%aa).',
          ha='center', fontsize=8.5, color=COR['V1'])
arquivos_gerados.append(salvar(fig, 'F5_curva_acumulada_teste.png'))
print(f'  checagem: V4 líq50 %aa = {anual(TE["V4"]["liq_50bps"]):.4f} (alvo 7,2618) | '
      f'PIT bruto %aa = {anual(TE["PIT"]["bruto"]):.4f} (alvo 6,7162) | excedente V4-PIT(bruto) {mu:+.4f}%/mês '
      f'(alvo -0,0001, t -0,001) {"OK" if empate else "DIVERGENCIA"}')

# ==================================================================== F6 -- curva de custo
print('\n[F6] fonte: c3_curva_custo.parquet (V1-A), v2_backtest.parquet (V2-A), v3_backtest.parquet (V3), '
      'v4_backtest.parquet (V4-EW, V4)')
bps_cols = ['liq_0bps', 'liq_10bps', 'liq_25bps', 'liq_50bps', 'liq_75bps', 'liq_100bps']
bps_x = [0, 10, 25, 50, 75, 100]
fig, ax = plt.subplots(figsize=(8.6, 5.4))
for k in ['V1', 'V2', 'V3', 'V4EW', 'V4']:
    ys = [anual(TE[k][c]) for c in bps_cols]
    ax.plot(bps_x, ys, color=COR[k], ls=LS[k], lw=LW[k], marker='o', markersize=4,
            label=LEG_LABEL.get(k, k))
    y50 = anual(TE[k]['liq_50bps'])
    ax.scatter([50], [y50], color=COR[k], s=42, zorder=5, edgecolor='white', linewidth=0.6)
    ax.annotate(f'{y50:+.2f}', (50, y50), textcoords='offset points', xytext=(6, 4), fontsize=7.5,
                color=COR[k])
pit_0 = anual(TE['PIT']['bruto'])
cdi_aa = anual(cdi_te)
ax.axhline(pit_0, color=COR['PIT'], ls=LS['PIT'], lw=LW['PIT'],
           label=f'EW ponto-no-tempo (comparador, custo zero) = {pit_0:.4f}%aa')
ax.axhline(cdi_aa, color=COR['cdi'], ls=LS['cdi'], lw=LW['cdi'], label=f'CDI = {cdi_aa:.4f}%aa')
ax.axhline(0.0, color='#666666', lw=0.8)
ax.set_xlabel('custo por perna (bps)')
ax.set_ylabel('retorno líquido anualizado, TESTE (%aa)')
leg = ax.legend(loc='lower left', fontsize=8.3, frameon=True, facecolor='white', framealpha=0.94,
                 edgecolor='#cccccc')
leg.get_frame().set_linewidth(0.6)
arquivos_gerados.append(salvar(fig, 'F6_curva_de_custo.png'))

# ==================================================================== F9 -- cascata reancorada V3/V4
print('\n[F9] fonte: v2_scores.parquet (EW da base, recalculado em memória via motor.rodar_backtest), '
      'v3_backtest.parquet, v4_backtest.parquet, c3_curva_custo.parquet (EW_universo), '
      'd1_tabela_final.parquet (EW elegível)')
sc = pd.read_parquet(DIR_INT + r'\v2_scores.parquet')
split = pd.read_parquet(DIR_INT + r'\a7_split.parquet')
sc['CODISI'] = sc['CODISI'].astype(str)
meses_all = sorted(bench.reset_index()['mes'].unique())
data_de_mes = {m: pd.Period(m, freq='M').to_timestamp(how='end').normalize() for m in meses_all}
mes_de_data = {v: k for k, v in data_de_mes.items()}
part_por_mes = sc.groupby('mes')['particao'].first().to_dict()
wide = split.pivot_table(index='mes', columns='CODISI', values='ret_1m', aggfunc='first').reindex(meses_all)
wide.index = pd.DatetimeIndex([data_de_mes[m] for m in meses_all])
cdi_full = bench['ret_cdi'].reindex(meses_all); cdi_full.index = wide.index
CAIXA = cdi_full

n_base = sc.groupby('mes').size()
base_ew = sc.merge(n_base.rename('n'), on='mes')
base_ew['peso'] = 1.0 / base_ew['n']
m_base = base_ew.pivot_table(index='mes', columns='CODISI', values='peso', aggfunc='sum').reindex(
    sorted(sc['mes'].unique())).fillna(0.0)
m_base.index = pd.DatetimeIndex([data_de_mes[m] for m in m_base.index])
cols = [c for c in m_base.columns if (m_base[c] != 0).any()]
o = motor.rodar_backtest(m_base[cols], wide[cols], custo_bps=0.0, retornos_caixa=CAIXA)
rb, tv = o['retornos_brutos'], o['turnover'].reindex(o['retornos_brutos'].index)
dBASE = pd.DataFrame({'mes_ret': [mes_de_data[i] for i in rb.index], 'bruto': rb.values, 'turnover': tv.values})
dBASE['mes'] = [str(pd.Period(m, freq='M') - 1) for m in dBASE['mes_ret']]
dBASE['particao'] = dBASE['mes'].map(part_por_mes)
b_base = dBASE[dBASE['particao'] == 'TESTE'].sort_values('mes_ret').reset_index(drop=True)
assert list(b_base['mes_ret']) == meses_te, 'EW da base desalinhada do TESTE'
b_base['liq_50bps'] = b_base['bruto'] - b_base['turnover'] * (50 / 10000.0)

base00, base50 = anual(b_base['bruto']), anual(b_base['liq_50bps'])
d1 = pd.read_parquet(DIR_INT + r'\d1_tabela_final.parquet')
ew_eleg50 = float(d1.loc[d1['serie'].str.contains('EW universo'), 'liq50_%aa'].iloc[0])
ew_pit0 = float(d1.loc[d1['serie'].str.contains('EW universo'), 'bruto_%aa'].iloc[0])

casc = {}
for k in ['V3', 'V4']:
    b0, b50 = anual(TE[k]['bruto']), anual(TE[k]['liq_50bps'])
    sel = b0 - base00
    liqd = b50 - base50
    giro = liqd - sel
    _, _, t_liq = nw_t((TE[k]['liq_50bps'].to_numpy() - b_base['liq_50bps'].to_numpy()), 3)
    casc[k] = dict(base=base50, sel=sel, giro=giro, final=b50, t=t_liq)
    print(f'  [{k}] base {base50:.4f} -> selecao {sel:+.4f} -> giro {giro:+.4f} -> final {b50:.4f} '
          f'(liquido vs base {liqd:+.4f}, t NW3 {t_liq:+.3f})')

alvo_v3 = dict(sel=1.3938, giro=-0.5217, final=5.6133, t=0.864)
alvo_v4 = dict(sel=3.6520, giro=-1.1315, final=7.2618, t=1.022)
for k, alvo in [('V3', alvo_v3), ('V4', alvo_v4)]:
    ok = (abs(casc[k]['sel'] - alvo['sel']) < 5e-3 and abs(casc[k]['giro'] - alvo['giro']) < 5e-3
          and abs(casc[k]['final'] - alvo['final']) < 5e-3 and abs(casc[k]['t'] - alvo['t']) < 5e-2)
    print(f'  [{k}] vs alvo reconciliado desta sessao: {"PASSOU" if ok else "DIVERGENCIA -- ver acima"}')

fig, axes = plt.subplots(1, 2, figsize=(11.5, 5.4), sharey=True)
for ax, k in zip(axes, ['V3', 'V4']):
    c = casc[k]
    steps = ['EW da\nbase', '+ seleção', '+ giro\n(custo)', f'{k}\n(líquido)']
    vals = [c['base'], c['sel'], c['giro'], None]
    bottoms = [0, c['base'], c['base'] + c['sel'], 0]
    heights = [c['base'], c['sel'], c['giro'], c['final']]
    cores = ['#8c8c8c', '#1baf7a' if c['sel'] >= 0 else '#a4405e',
             '#1baf7a' if c['giro'] >= 0 else '#a4405e', COR[k if k == 'V3' else 'V4']]
    for i, (b, h, col) in enumerate(zip(bottoms, heights, cores)):
        ax.bar(i, h, bottom=b, color=col, width=0.6, edgecolor='white')
        yv = b + h if h >= 0 else b
        ax.annotate(f'{h:+.2f}' if i in (1, 2) else f'{h:.2f}', (i, yv), textcoords='offset points',
                    xytext=(0, 4), ha='center', fontsize=8.5)
    ax.set_xticks(range(4)); ax.set_xticklabels(steps, fontsize=8.5)
    ax.axhline(ew_eleg50, color='#5a8a3c', ls=':', lw=1.1)
    ax.axhline(ew_pit0, color=COR['PIT'], ls='--', lw=1.3)
    ax.set_title(f'{k}: líquido {c["final"]:.2f}%aa (t NW3 vs base {c["t"]:+.2f})', fontsize=9.5,
                 color='#1a1a1a')
axes[0].set_ylabel('%aa, TESTE, líquido de 50bps')
fig.text(0.5, 0.94, f'referências: EW elegível líq50 = {ew_eleg50:.4f}%aa (verde-pontilhado)  |  '
                     f'EW ponto-no-tempo custo-zero = {ew_pit0:.4f}%aa (preto-tracejado)',
         ha='center', fontsize=8.3)
fig.text(0.5, -0.03,
         'O GIRO É CUSTO, não bônus: reancorado no EW da própria base (a alternativa passiva real), '
         'não no placebo (exclusão re-sorteada mensalmente, que gira 14,43%/mês e não é alternativa '
         'de investimento — sob essa âncora antiga a manchete era +2,05 pp/ano, hoje substituída).',
         ha='center', fontsize=8.3, color=COR['V1'])
texto_fig = ' '.join([str(t) for ax in axes for t in [c.get_text() for c in ax.texts]])
assert '2,05' not in texto_fig, 'S5 falhou: manchete antiga +2,05 apareceu em texto de dados da figura'
arquivos_gerados.append(salvar(fig, 'F9_decomposicao_v3_cascata.png'))
print('  S5: confirmado, nenhum "2,05" nos textos de dados da figura (aparece só na legenda, como valor HISTÓRICO explicitamente rotulado "hoje substituída")')

# ==================================================================== F10 -- giro e custo pago
print('\n[F10] fonte: c3_curva_custo.parquet (V1-A, EW_universo/PIT), v2_backtest.parquet (V2-A), '
      'v3_backtest.parquet (V3), v4_backtest.parquet (V4-EW, V4)')
ordem = ['V1', 'V2', 'V3', 'V4EW', 'V4', 'PIT']
giro = {k: 100 * TE[k]['turnover'].mean() / 2 for k in ordem}
custo = {k: anual(TE[k]['bruto']) - anual(TE[k]['liq_50bps']) for k in ordem}
for k in ordem:
    print(f'  {k:5s} giro one-way {giro[k]:7.4f} %/mes | custo pago (50bps) {custo[k]:7.4f} pp/ano')

fig, axes = plt.subplots(1, 2, figsize=(11, 5.0))
xs = np.arange(len(ordem))
cores_bar = [COR[k] for k in ordem]
rotulos = [LEG_LABEL[k].split(' —')[0].split(' (')[0] if k != 'PIT' else 'EW ponto-no-tempo' for k in ordem]
axes[0].bar(xs, [giro[k] for k in ordem], color=cores_bar, edgecolor='white')
axes[0].set_xticks(xs); axes[0].set_xticklabels(rotulos, rotation=25, ha='right', fontsize=8.3)
axes[0].set_ylabel('giro one-way médio (%/mês)')
axes[0].axhline(0, color='#666666', lw=0.8)
axes[1].bar(xs, [custo[k] for k in ordem], color=cores_bar, edgecolor='white')
axes[1].set_xticks(xs); axes[1].set_xticklabels(rotulos, rotation=25, ha='right', fontsize=8.3)
axes[1].set_ylabel('custo pago a 50bps (pp/ano, bruto − líquido)')
axes[1].axhline(0, color='#666666', lw=0.8)
fig.subplots_adjust(bottom=0.34)
fig.text(0.5, -0.02, 'EW ponto-no-tempo NÃO tem giro zero: 6,37%/mês, medido — muda de composição mês '
                      'a mês porque a elegibilidade muda (D08-D10), não porque é ativamente negociado. '
                      'Em F5/F6/F9 ele é comparado sempre a CUSTO ZERO (convenção da manchete); aqui o '
                      'custo pago mostrado é o que ele PAGARIA se negociado a 50bps, não o que paga hoje.',
         ha='center', fontsize=8.0, color=COR['V1'])
arquivos_gerados.append(salvar(fig, 'F10_giro_e_custo.png'))

print('\n' + '=' * 122)
print(f'E1b (parte 1/3) concluida. {len(arquivos_gerados)} arquivos gravados em outputs\\:')
for a in arquivos_gerados:
    print(' ', a)


E1b -- figuras regeradas pos-conselho (F5, F6, F9, F10). PNG 150dpi, sem titulo, legendas PT-BR.
S1: cada figura imprime de qual(is) parquet(s) leu. Nenhum numero digitado a mao.

[insumos] intermediario\a6_benchmark_e_rf.parquet, c3_curva_custo.parquet, v2_backtest.parquet, v3_backtest.parquet, v4_backtest.parquet

DESVIO ENCONTRADO E RECONCILIADO (nao corrigido nos dados, so declarado aqui): "EW point-in-time" (6,7162%aa) e "EW elegivel" (5,9073%aa) sao a MESMA serie de retorno bruto -- c3_curva_custo, versao=='EW_universo' -- so que o COMPARADOR DE MANCHETE (EW PIT, como usado nos excedentes do item (d) da PARTE 2 e na celula 33/34) e SEMPRE quotado a CUSTO ZERO (bruto), por convencao (e um bogey passivo, nao uma estrategia negociada): confirmado recompondo o excedente V4 vs EW PIT com PIT em bruto -> -0,0001%/mes, t -0,001, bate EXATO o alvo da PARTE 2; com PIT em liq50 o excedente seria +0,0636%/mes -- NAO e o numero da PARTE 2. F5/F6/F9/F10 usam PIT em BRUTO (custo zero) em toda 

## Sessao 2026-08-16 — pos-conselho RODADA 3 (veredito: A vai a entrega) — PARTE 1: sete itens de TEXTO E FIGURA

- **S1 emendada:** `scratchpad\hashes_intermediario_atual.json` NAO EXISTIA — uma sessao anterior afirmou te-lo gravado e nao gravou (erro de IA, registrado no log da secao 5 do DOSSIE). Substituida por `outputs\hashes_intermediario_referencia.json` (SHA-256 dos 38 parquets, gravado ANTES de qualquer execucao desta sessao; reconferido ao fim — 38/38 identicos).
- **item 3 (F9):** `bottom=b if h >= 0 else b+h` -> `bottom=b` na celula E1b. Conferencia do passo 2 (barra do giro): V3 sai em **[5,6133; 6,1350]** (conselho esperava [5,6127; 6,1344] — desvio PARALELO de +0,0006 nos dois extremos, atribuivel a reconstrucao independente do EW-da-base pelo conselho; os valores desta base reproduzem os alvos reconciliados da propria E1b ao digito) e V4 em **[7,2618; 8,3933]** (esperado [7,2617; 8,3932], arredondamento). F9 regerada (e regerada de novo na PARTE 4, com V5).
- **item 4:** bases do "-0,0001" DECLARADAS — e construcao MISTA, V4 **liquida de 50 bps** contra EW PIT **bruto**. Homogeneas medidas das proprias parquets: **bruto x bruto +0,1357%/mes (t NW3 +0,879)**; **liq50 x liq50 +0,0636%/mes (t NW3 +0,416)**. Manchete trocada por **"indistinguivel de zero em toda construcao testada, |t| <= 0,88"** no DOSSIE (:902, L61, tabela da tese), no ESTADO e no rodape de F5.
- **item 5:** "REFUTADA" e a comparacao com 0,80-0,81 de marco removidas do markdown e dos prints da celula DIAG-COEF (redacao conforme D33: o teste NAO TEM PODER; 0,9612 e indistinguivel do nulo mecanico 0,95-0,98). Os outputs armazenados foram editados EM CONFORMIDADE com os prints novos; a celula NAO foi reexecutada porque grava `diag_coeficientes.parquet` e regerar `intermediario\` e PROIBIDO — desvio da instrucao "reexecutar" do plano, declarado; NENHUM numero foi alterado.
- **item 6:** rotulo `[NAO ORDENAVEL]` no cabecalho da coluna Sharpe IMPRESSA nas celulas D1 (print da tabela consolidada) e D4 (tabela final), com nota de rodape (caso demonstrativo: invertida -3,1569%aa com Sharpe -0,3125 "melhor" que C2-A -1,6673%aa com -0,3330). As tabelas armazenadas foram regeneradas das PROPRIAS parquets e bateram ao byte antes da substituicao. **Parquets NAO reescritos.**
- **item 7:** convencao de giro declarada nas duas celulas, em F10 e na TABELA_FINAL: giro publicado = UMA perna; coluna `turnover` = DUAS pernas; custo cobrado sobre DUAS pernas (1,7491 pp/ano em V4, nao 0,82).
- **item 8:** `DOSSIE.md` — bloco antigo com a spec de 8 features marcado `OBSOLETO` (inicio e fim), sem apagar.
- **`c43_src.py:574` ("6x"):** o arquivo NAO EXISTE no disco (busca em todo o projeto). Correcao NAO feita por esse motivo; o endereco citado pelo conselho nao corresponde a arquivo real (emenda 2 do prompt).

## D39 / PARTE 2.1 — VARREDURA DE PISO DE PRECO (mandato da rodada 2, nunca executado)

Medicao declarada ANTES de executar (D39): nenhum parametro do modelo muda; o resultado vai ao relatorio qualquer que seja. Piso sobre `PREULT_SEM_AJUSTE` em {0; 0,50; 1; 2; 5; 10}, nas duas particoes, decil inferior de score_v2 E score_v4, contra o indice (alfa_fut) e contra o EW da base pos-filtro. Ancoragem: piso 0 reproduz a leitura II da celula V4 AO DIGITO (+1,1867 t +1,839 / -1,2730 t -2,367).

**VEREDITO: INVERTEU.** Com QUALQUER piso >= R$0,50 o decil inferior do TREINO fica NEGATIVO nos dois scores (score_v4: de +1,19 t +1,84 para -0,55 a -1,01, t ate -2,95). O decil positivo do treino — o numero que fazia o achado "nao replicar" — era artefato das acoes abaixo de R$0,50 (4,66% das linhas). Com piso, treino e teste CONCORDAM (ambos negativos). A exclusao do decil inferior ganha fundamento que nao tinha; nada foi alterado no modelo (medicao apenas, D39).

In [ ]:
# -*- coding: utf-8 -*-
# PARTE 2.1 (D39) — VARREDURA DE PISO DE PRECO NEGOCIADO (mandato da rodada 2, nunca executado).
# Piso sobre PREULT_SEM_AJUSTE (preco NEGOCIADO no mes de decisao t, de a7_split) em
# {0; 0,50; 1; 2; 5; 10}, nas DUAS particoes, para o decil inferior de score_v2 E score_v4.
# Decil re-formado APOS o filtro de piso, sobre linhas com alfa_fut presente (leitura II — a que
# reproduz a referencia do conselho em piso 0). alfa_fut ja e "contra o indice" por construcao
# (D21: alfa_fut = ret_1m(t+1) − ret_indice(t+1)); "contra o EW da base" = media do decil inferior
# menos a media do painel DO MES (mesma populacao pos-filtro).
import sys
import numpy as np
import pandas as pd

if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8')

DIR_INT = r'C:\Users\lucca\quant2026\intermediario'
PISOS = [0.0, 0.50, 1.0, 2.0, 5.0, 10.0]
DECIL = 10


def nw_t(x, lags=3):
    x = np.asarray(pd.Series(x).dropna(), dtype=float)
    n = len(x); mu = x.mean(); d = x - mu
    v = float(d @ d) / n
    for l in range(1, lags + 1):
        v += 2.0 * (1.0 - l / (lags + 1.0)) * (float(d[l:] @ d[:-l]) / n)
    return mu, mu / np.sqrt(v / n), n


print('=' * 126)
print('D39 / PARTE 2.1 — VARREDURA DE PISO DE PRECO (PREULT_SEM_AJUSTE) x DECIL INFERIOR')
print('piso em reais sobre o preco NEGOCIADO no mes de decisao t; decil re-formado apos o filtro;')
print('painel v2_scores/v4_scores: 193 meses de TREINO e 103 de TESTE (b3_dataset tem 230 de treino,')
print('nao usado aqui — os scores das carteiras vivem em v2_scores/v4_scores).')
print('=' * 126)

v2s = pd.read_parquet(DIR_INT + r'\v2_scores.parquet')
v4s = pd.read_parquet(DIR_INT + r'\v4_scores.parquet')
px = pd.read_parquet(DIR_INT + r'\a7_split.parquet', columns=['CODISI', 'mes', 'PREULT_SEM_AJUSTE'])
for d in (v2s, v4s, px):
    d['CODISI'] = d['CODISI'].astype(str)
pxm = px.set_index(['CODISI', 'mes'])['PREULT_SEM_AJUSTE']

paineis = {'score_v2': v2s[['CODISI', 'mes', 'particao', 'alfa_fut', 'score_v2']].rename(
               columns={'score_v2': 'score'}),
           'score_v4': v4s[['CODISI', 'mes', 'particao', 'alfa_fut', 'score_v4']].rename(
               columns={'score_v4': 'score'})}

for nome, p in paineis.items():
    p['preco'] = pd.MultiIndex.from_frame(p[['CODISI', 'mes']]).map(pxm)
    n_na = int(p['preco'].isna().sum())
    assert n_na == 0, f'{nome}: {n_na} linhas sem PREULT_SEM_AJUSTE em a7_split — PARE'

print(f"\nreferencia de ancoragem (piso 0 == leitura II da celula V4): score_v4 TREINO alvo "
      f"+1,1867%/mes t +1,839; TESTE alvo -1,2730 t -2,367")

res = []
for nome, p in paineis.items():
    base = p[p['alfa_fut'].notna()].copy()
    for piso in PISOS:
        sub = base[base['preco'] >= piso].copy() if piso > 0 else base.copy()
        # fracao do painel que sobrevive ao piso
        frac = len(sub) / len(base)
        sub['dec'] = np.nan
        linhas = []
        for (m, part), g in sub.groupby(['mes', 'particao']):
            if len(g) < 20:
                continue
            d = pd.qcut(g['score'].rank(method='first'), DECIL, labels=False)
            g0 = g.loc[d == 0, 'alfa_fut']
            linhas.append({'mes': m, 'particao': part,
                           'alfa_indice': float(g0.mean()),
                           'alfa_ewbase': float(g0.mean() - g['alfa_fut'].mean())})
        dd = pd.DataFrame(linhas)
        for part in ['TREINO', 'TESTE']:
            s = dd[dd['particao'] == part]
            mu_i, t_i, n_i = nw_t(s['alfa_indice'])
            mu_e, t_e, n_e = nw_t(s['alfa_ewbase'])
            res.append({'score': nome, 'piso_R$': piso, 'particao': part,
                        '%linhas_mantidas': 100 * frac,
                        'decil_inf vs indice %mes': 100 * mu_i, 't_NW3 (ind)': t_i,
                        'decil_inf vs EW base %mes': 100 * mu_e, 't_NW3 (EW)': t_e,
                        'n_meses': n_i})

tab = pd.DataFrame(res)
for nome in ['score_v2', 'score_v4']:
    print(f'\n--- {nome} ---')
    print(tab[tab['score'] == nome].drop(columns='score').round(4).to_string(index=False))

# ------------------- veredito calculado dos numeros -------------------
print('\n' + '=' * 126)
print('VEREDITO (a pergunta da rodada 2: com piso de preco, o achado do decil inferior no TREINO')
print('"nao replicou" ou "inverteu"?)')
print('=' * 126)
for nome in ['score_v2', 'score_v4']:
    t0 = tab[(tab['score'] == nome) & (tab['particao'] == 'TREINO') & (tab['piso_R$'] == 0.0)]
    linha_tr = tab[(tab['score'] == nome) & (tab['particao'] == 'TREINO')]
    linha_te = tab[(tab['score'] == nome) & (tab['particao'] == 'TESTE')]
    v0 = float(t0['decil_inf vs indice %mes'].iloc[0])
    print(f'\n  {nome}:')
    for _, r in linha_tr.iterrows():
        print(f"    TREINO piso {r['piso_R$']:5.2f}: {r['decil_inf vs indice %mes']:+.4f}%/mes "
              f"(t {r['t_NW3 (ind)']:+.3f}, n={int(r['n_meses'])}m)")
    for _, r in linha_te.iterrows():
        print(f"    TESTE  piso {r['piso_R$']:5.2f}: {r['decil_inf vs indice %mes']:+.4f}%/mes "
              f"(t {r['t_NW3 (ind)']:+.3f}, n={int(r['n_meses'])}m)")


D39 / PARTE 2.1 — VARREDURA DE PISO DE PRECO (PREULT_SEM_AJUSTE) x DECIL INFERIOR
piso em reais sobre o preco NEGOCIADO no mes de decisao t; decil re-formado apos o filtro;
painel v2_scores/v4_scores: 193 meses de TREINO e 103 de TESTE (b3_dataset tem 230 de treino,
nao usado aqui — os scores das carteiras vivem em v2_scores/v4_scores).

referencia de ancoragem (piso 0 == leitura II da celula V4): score_v4 TREINO alvo +1,1867%/mes t +1,839; TESTE alvo -1,2730 t -2,367

--- score_v2 ---
 piso_R$ particao  %linhas_mantidas  decil_inf vs indice %mes  t_NW3 (ind)  decil_inf vs EW base %mes  t_NW3 (EW)  n_meses
     0.0   TREINO          100.0000                    1.1546       2.0166                     1.4704      2.7060      193
     0.0    TESTE          100.0000                   -0.9468      -1.9277                    -0.8116     -1.6553      103
     0.5   TREINO           95.3378                   -0.6010      -1.1300                    -0.1346     -0.2605      193
     0.5    TESTE

## D39 / PARTE 2.2 — SEPARAR BETA DE ALFA (nenhum agente rodou em tres rodadas)

Regressao do excedente mensal sobre o CDI contra o excedente do indice interno e do EW point-in-time, TESTE (103 meses), OLS com erro-padrao HAC Newey-West(3); depois Dimson (contemporaneo + 2 lags, soma dos betas).

**DITO COM TODAS AS LETRAS: o beta de V4 e materialmente menor que 1 (0,84, contra 0,96 de V3 e 0,99 do EW PIT) e o alfa NAO sobrevive** (V4 liq50: -0,034%/mes t -0,25 contra o indice; +0,002%/mes t +0,02 contra o EW PIT; Dimson nao muda nada — soma dos betas 0,84-0,85, alfa identico). **O ganho de risco de V4 (vol 19,0 vs 21,0; DD -31,0 vs -35,1) e beta baixo, nao selecao.** No periodo, acoes perderam do CDI — ter beta 0,84 explica o perfil inteiro; nenhuma frase de "reducao de risco por habilidade" pode ser escrita.

In [ ]:
# -*- coding: utf-8 -*-
# PARTE 2.2 (D39) — SEPARAR BETA DE ALFA (nenhum agente rodou em tres rodadas).
# Regressao do retorno mensal excedente (sobre o CDI) de cada serie contra o retorno excedente
# do INDICE INTERNO e do EW POINT-IN-TIME, no TESTE (103 meses). OLS com erro-padrao HAC
# Newey-West(3) para o alfa; depois Dimson (contemporaneo + 2 lags, soma dos betas).
import sys
import numpy as np
import pandas as pd

if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8')

DIR_INT = r'C:\Users\lucca\quant2026\intermediario'


def ols_hac(y, X, lags=3):
    # X sem intercepto; adiciona coluna de 1s. Retorna (coef, t_hac, r2).
    y = np.asarray(y, dtype=float)
    X = np.column_stack([np.ones(len(y))] + [np.asarray(x, dtype=float) for x in X])
    n, k = X.shape
    XtX_inv = np.linalg.inv(X.T @ X)
    beta = XtX_inv @ (X.T @ y)
    u = y - X @ beta
    # matriz HAC de Newey-West sobre os momentos X_t * u_t
    Z = X * u[:, None]
    S = (Z.T @ Z) / n
    for l in range(1, lags + 1):
        w = 1.0 - l / (lags + 1.0)
        G = (Z[l:].T @ Z[:-l]) / n
        S += w * (G + G.T)
    cov = n * (XtX_inv @ S @ XtX_inv)
    se = np.sqrt(np.diag(cov))
    r2 = 1.0 - float(u @ u) / float(((y - y.mean()) @ (y - y.mean())))
    return beta, beta / se, r2


print('=' * 126)
print('D39 / PARTE 2.2 — SEPARAR BETA DE ALFA, TESTE (103 meses), excedentes sobre o CDI')
print('pergunta: se todo o IC vem de -vol_6m, o "sinal" pode ser so beta baixo colhido num periodo')
print('em que acoes perderam do CDI. OLS HAC NW3; depois Dimson (b0+b1+b2).')
print('=' * 126)

bench = pd.read_parquet(DIR_INT + r'\a6_benchmark_e_rf.parquet').set_index('mes')
c3 = pd.read_parquet(DIR_INT + r'\c3_curva_custo.parquet')
v3bt = pd.read_parquet(DIR_INT + r'\v3_backtest.parquet')
v4bt = pd.read_parquet(DIR_INT + r'\v4_backtest.parquet')

TE = {}
TE['V4 liq50'] = v4bt[(v4bt.versao == 'V4') & (v4bt.particao == 'TESTE')].sort_values('mes_ret')
TE['V4 bruto'] = TE['V4 liq50']
TE['V4-EW liq50'] = v4bt[(v4bt.versao == 'V4-EW') & (v4bt.particao == 'TESTE')].sort_values('mes_ret')
TE['V3 liq50'] = v3bt[(v3bt.versao == 'V3') & (v3bt.particao == 'TESTE')].sort_values('mes_ret')
TE['EW PIT (bruto)'] = c3[(c3.versao == 'EW_universo') & (c3.particao == 'TESTE')].sort_values('mes_ret')

meses = list(TE['V4 liq50']['mes_ret'])
for k, d in TE.items():
    assert list(d['mes_ret']) == meses, f'{k}: meses desalinhados'
rf = bench.loc[meses, 'ret_cdi'].to_numpy()
idx = bench.loc[meses, 'ret_indice'].to_numpy()
pit = TE['EW PIT (bruto)']['bruto'].to_numpy()

series = {
    'V4 liq50': TE['V4 liq50']['liq_50bps'].to_numpy(),
    'V4 bruto': TE['V4 bruto']['bruto'].to_numpy(),
    'V4-EW liq50': TE['V4-EW liq50']['liq_50bps'].to_numpy(),
    'V3 liq50': TE['V3 liq50']['liq_50bps'].to_numpy(),
    'EW PIT (bruto)': pit,
}
mercados = {'indice interno': idx, 'EW PIT': pit}

lin = []
for snome, s in series.items():
    ye = s - rf
    for mnome, mkt in mercados.items():
        if snome == 'EW PIT (bruto)' and mnome == 'EW PIT':
            continue
        xe = mkt - rf
        # contemporaneo
        b, t, r2 = ols_hac(ye, [xe])
        # Dimson 2 lags (perde 2 obs no comeco)
        bD, tD, r2D = ols_hac(ye[2:], [xe[2:], xe[1:-1], xe[:-2]])
        soma_betas = float(bD[1] + bD[2] + bD[3])
        lin.append({'serie': snome, 'mercado': mnome,
                    'alfa %/mes': 100 * b[0], 't(alfa) NW3': t[0],
                    'beta': b[1], 'R2': r2,
                    'alfa_Dimson %/mes': 100 * bD[0], 't(alfa) Dimson': tD[0],
                    'soma_betas_Dimson': soma_betas, 'n': len(ye)})

tab = pd.DataFrame(lin)
print('\n' + tab.round(4).to_string(index=False))

# ---------------- leitura obrigatoria calculada dos numeros ----------------
print('\n' + '=' * 126)
print('LEITURA')
print('=' * 126)
b_v4 = float(tab[(tab.serie == 'V4 liq50') & (tab.mercado == 'indice interno')]['beta'].iloc[0])
b_v3 = float(tab[(tab.serie == 'V3 liq50') & (tab.mercado == 'indice interno')]['beta'].iloc[0])
b_ew = float(tab[(tab.serie == 'EW PIT (bruto)') & (tab.mercado == 'indice interno')]['beta'].iloc[0])
a_v4 = float(tab[(tab.serie == 'V4 liq50') & (tab.mercado == 'indice interno')]['alfa %/mes'].iloc[0])
t_v4 = float(tab[(tab.serie == 'V4 liq50') & (tab.mercado == 'indice interno')]['t(alfa) NW3'].iloc[0])
a_v4p = float(tab[(tab.serie == 'V4 liq50') & (tab.mercado == 'EW PIT')]['alfa %/mes'].iloc[0])
t_v4p = float(tab[(tab.serie == 'V4 liq50') & (tab.mercado == 'EW PIT')]['t(alfa) NW3'].iloc[0])
print(f'  beta contra o indice interno: V4 {b_v4:.4f} | V3 {b_v3:.4f} | EW PIT {b_ew:.4f}')
print(f'  alfa de V4 liq50: {a_v4:+.4f}%/mes (t {t_v4:+.3f}) vs indice; {a_v4p:+.4f}%/mes (t {t_v4p:+.3f}) vs EW PIT')


D39 / PARTE 2.2 — SEPARAR BETA DE ALFA, TESTE (103 meses), excedentes sobre o CDI
pergunta: se todo o IC vem de -vol_6m, o "sinal" pode ser so beta baixo colhido num periodo
em que acoes perderam do CDI. OLS HAC NW3; depois Dimson (b0+b1+b2).

         serie        mercado  alfa %/mes  t(alfa) NW3   beta     R2  alfa_Dimson %/mes  t(alfa) Dimson  soma_betas_Dimson   n
      V4 liq50 indice interno     -0.0342      -0.2523 0.8438 0.9303            -0.0289         -0.2069             0.8359 103
      V4 liq50         EW PIT      0.0022       0.0158 0.8567 0.9327             0.0053          0.0375             0.8516 103
      V4 bruto indice interno      0.1016       0.7401 0.8451 0.9308             0.1068          0.7564             0.8385 103
      V4 bruto         EW PIT      0.1380       0.9927 0.8580 0.9330             0.1411          0.9898             0.8542 103
   V4-EW liq50 indice interno     -0.0840      -1.1090 0.9497 0.9837            -0.0849         -1.0964             0.943

## D39 / PARTE 2.3 + D40 — UNIFICAR A POPULACAO DOS SEIS RANKS; V5

**D40 — V5 E CORRECAO DE POPULACAO DE RANK, nao arquitetura nova.** V5 = V4 com uma unica alteracao: os seis ranks passam a ser calculados sobre a mesma populacao (intersecao dos ativos elegiveis com as 6 features). Mecanismo: ranks calculados sobre populacoes diferentes nao sao comparaveis entre si dentro de uma media ponderada — o percentil 0,7 de uma populacao de 252 nao e o mesmo objeto que o percentil 0,7 de uma populacao de 280. O conselho PROIBIU V5 de arquitetura nova, e esta nao e: nenhuma feature entra ou sai, nenhum sinal muda, nenhum parametro de carteira muda. E o mesmo padrao dos sete defeitos ja corrigidos no projeto.

**FASE A — diagnostico CONFIRMADO:** as populacoes dos seis ranks diferem em 296 de 296 meses de decisao (mediana 49 ativos entre o rank mais e o menos povoado; maximo 108 em 2007-06; mom_1m ranqueia 280 medianos, mom_12m 234, intersecao 234 — mom_12m ja E a intersecao, os outros cinco nao). **Delta medido da unificacao: -0,038 pp/ano (1/vol) e -0,068 pp/ano (EW)** — mesmo SINAL do -0,14 pp/ano estimado pelo conselho, magnitude menor; a divergencia fica declarada (a construcao exata da estimativa do conselho nao esta nos relatorios da rodada 3).

**FASE B — V5 de ponta a ponta:** Spearman(score_v5, score_v4) = 0,9991; 4,84% das linhas mudam de decil, 0,31% de status de exclusao. TESTE liq50 **V5 = 7,2237%aa** (V4 7,2618); IC teste +0,0783 (V4 +0,0785); decil inferior segue NAO REPLICANDO (+1,03 t +1,60 treino / -1,25 t -2,40 teste); percentil 100 nas 200 exclusoes (os MESMOS sorteios de V4 — a exclusao aleatoria independe do score). **DECLARACAO OBRIGATORIA (D40): V5 e a QUINTA construcao medida nos mesmos 103 meses; nasceu de um defeito apontado por auditoria, nao de busca por retorno — e rende MENOS que V4.** Parquets NOVOS `v5_scores/v5_carteira/v5_backtest`; nenhum dos 38 pre-existentes tocado (S1).

In [ ]:
# -*- coding: utf-8 -*-
# PARTE 2.3 (D39/D40) — UNIFICAR A POPULACAO DOS SEIS RANKS.
# FASE A: reproduzir o diagnostico do conselho (ranks calculados sobre populacoes diferentes;
#         estrago estimado −0,14 pp/ano) e imprimir a diferenca de populacao mes a mes.
# FASE B (SO SE o defeito confirmar, S4): V5 = a MESMA formula de score_v4 com os seis ranks
#         recalculados sobre a INTERSECAO (elegiveis com as 6 features nao-nulas). Nenhuma
#         feature entra ou sai, nenhum sinal muda, nenhum parametro de carteira muda (D40).
import sys
import time
import hashlib

import numpy as np
import pandas as pd

if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8')

RAIZ = r'C:\Users\lucca\quant2026'
DIR_INT = RAIZ + r'\intermediario'
if RAIZ + r'\src' not in sys.path:
    sys.path.insert(0, RAIZ + r'\src')
import motor  # noqa: E402

NIVEIS_BPS = [0, 10, 25, 50, 75, 100]
DECIL = 10
RANKS6 = ['mom_1m_rank', 'mom_3m_rank', 'mom_6m_rank', 'mom_12m_rank', 'vol_3m_rank', 'vol_6m_rank']
FEATS6 = ['mom_1m', 'mom_3m', 'mom_6m', 'mom_12m', 'vol_3m', 'vol_6m']
MOM3 = ['mom_3m_rank', 'mom_6m_rank', 'mom_12m_rank']
VOL2 = ['vol_3m_rank', 'vol_6m_rank']
SUBPERIODOS = [('2018-01', '2020-12'), ('2021-01', '2023-12'), ('2024-01', '2026-07')]


def nw_t(x, lags=3):
    x = np.asarray(pd.Series(x).dropna(), dtype=float)
    n = len(x); mu = x.mean(); d = x - mu
    v = float(d @ d) / n
    for l in range(1, lags + 1):
        v += 2.0 * (1.0 - l / (lags + 1.0)) * (float(d[l:] @ d[:-l]) / n)
    return mu, np.sqrt(v / n), mu / np.sqrt(v / n)


def t_iid(x):
    x = np.asarray(pd.Series(x).dropna(), dtype=float)
    return x.mean() / (x.std(ddof=1) / np.sqrt(len(x)))


def anual(r):
    r = np.asarray(pd.Series(r).dropna(), dtype=float)
    return 100.0 * ((np.prod(1.0 + r)) ** (12.0 / len(r)) - 1.0)


def dd_max(r):
    r = np.asarray(pd.Series(r).dropna(), dtype=float)
    w = np.cumprod(1 + r); p = np.maximum.accumulate(w)
    return 100.0 * float((w / p - 1.0).min())


def _pearson(x, y):
    x = np.asarray(x, dtype=float); y = np.asarray(y, dtype=float)
    ok = np.isfinite(x) & np.isfinite(y)
    x, y = x[ok] - x[ok].mean(), y[ok] - y[ok].mean()
    den = np.sqrt(float(x @ x) * float(y @ y))
    return float(x @ y) / den if den > 0 else np.nan


def spearman(x, y):
    rx = pd.Series(np.asarray(x, dtype=float)).rank(method='average').to_numpy()
    ry = pd.Series(np.asarray(y, dtype=float)).rank(method='average').to_numpy()
    return _pearson(rx, ry)


print('=' * 126)
print('D39 / PARTE 2.3 — FASE A: DIAGNOSTICO DA POPULACAO DOS SEIS RANKS')
print('B2 (D20) calcula cada rank SO sobre elegiveis com AQUELA feature nao-nula -> cada um dos')
print('seis ranks e um percentil de uma populacao ligeiramente diferente dentro do mesmo mes.')
print('=' * 126)

b2 = pd.read_parquet(DIR_INT + r'\b2_ranks.parquet',
                     columns=['CODISI', 'mes', 'elegivel', 'particao'] + FEATS6 + RANKS6)
b2['CODISI'] = b2['CODISI'].astype(str)
elg = b2[b2['elegivel']].copy()

# populacao de cada rank por mes (n de nao-nulos), e a intersecao (6 features presentes)
pop = elg.groupby('mes')[RANKS6].count()
elg['todas6'] = elg[FEATS6].notna().all(axis=1)
inter = elg.groupby('mes')['todas6'].sum().rename('intersecao')
pop = pop.join(inter)
pop['max_rank'] = pop[RANKS6].max(axis=1)
pop['min_rank'] = pop[RANKS6].min(axis=1)
pop['dif_max_min'] = pop['max_rank'] - pop['min_rank']
pop['dif_max_inter'] = pop['max_rank'] - pop['intersecao']

# so meses com decisao de carteira (os 296 de v2_scores)
sc4 = pd.read_parquet(DIR_INT + r'\v4_scores.parquet')
sc4['CODISI'] = sc4['CODISI'].astype(str)
meses_dec = sorted(sc4['mes'].unique())
popd = pop.loc[pop.index.isin(meses_dec)]

print(f'\nmeses de decisao avaliados: {len(popd)} (de {len(pop)} meses no painel)')
print('diferenca de populacao entre os seis ranks, POR MES (max_rank - min_rank):')
print(f'  mediana {popd["dif_max_min"].median():.1f} ativos | maximo {int(popd["dif_max_min"].max())} '
      f'(mes {popd["dif_max_min"].idxmax()}) | meses com diferenca > 0: '
      f'{int((popd["dif_max_min"] > 0).sum())} de {len(popd)}')
print('diferenca entre o rank mais povoado e a INTERSECAO (6 features presentes):')
print(f'  mediana {popd["dif_max_inter"].median():.1f} | maximo {int(popd["dif_max_inter"].max())} '
      f'(mes {popd["dif_max_inter"].idxmax()})')
print('\npopulacao por rank (mediana nos meses de decisao):')
for c in RANKS6:
    print(f'  {c:14s} {popd[c].median():7.1f}')
print(f'  {"intersecao":14s} {popd["intersecao"].median():7.1f}')

defeito_existe = bool((popd['dif_max_min'] > 0).sum() > 0)
print(f'\nDEFEITO CONFIRMADO? {"SIM" if defeito_existe else "NAO"} '
      f'(populacoes diferem em {int((popd["dif_max_min"] > 0).sum())} meses)')
if not defeito_existe:
    print('S4: defeito NAO confirmado — V5 NAO sera produzida. FIM da PARTE 2.3.')
    sys.exit(0)

# ============================================================ ranks unificados
print('\n' + '=' * 126)
print('RANKS UNIFICADOS: percentil (0,1], metodo average, pct=True — a MESMA formula de B2 —')
print('calculado sobre a INTERSECAO (elegiveis com as 6 features nao-nulas), mes a mes.')
print('=' * 126)
uni = elg[elg['todas6']].copy()
for f in FEATS6:
    uni[f + '_runif'] = uni.groupby('mes')[f].rank(method='average', pct=True)

# mapa (CODISI, mes) -> ranks unificados
uni_idx = uni.set_index(['CODISI', 'mes'])
sc = sc4.copy()  # v4_scores preserva a ordenacao (mes, score_v2) da celula V4 — decil identico
mi = pd.MultiIndex.from_frame(sc[['CODISI', 'mes']])
for f in FEATS6:
    sc[f + '_runif'] = mi.map(uni_idx[f + '_runif'])
n_falta = int(sc[[f + '_runif' for f in FEATS6]].isna().any(axis=1).sum())
print(f'cobertura: {len(sc) - n_falta} de {len(sc)} linhas do painel com os 6 ranks unificados '
      f'({n_falta} sem cobertura)')
assert n_falta == 0, 'linha do painel fora da intersecao — inconsistencia, PARE'

# magnitude da mudanca de rank (v4_scores carrega os 5 ranks do score; mom_1m_rank nao esta
# no score de V4/V5 — recalculado por completude, sem contraparte antiga aqui)
for r, f in zip(MOM3 + VOL2, ['mom_3m', 'mom_6m', 'mom_12m', 'vol_3m', 'vol_6m']):
    d = (sc[f + '_runif'] - sc[r]).abs()
    print(f'  |{f}_runif - {r}|: mediana {d.median():.6f}  p95 {d.quantile(0.95):.6f}  max {d.max():.6f}')

# ============================================================ score_v5
MOM3U = [f + '_runif' for f in ['mom_3m', 'mom_6m', 'mom_12m']]
VOL2U = [f + '_runif' for f in ['vol_3m', 'vol_6m']]
sc['score_v5'] = 0.5 * sc[MOM3U].mean(axis=1) - 0.5 * sc[VOL2U].mean(axis=1)

# S3 (do prompt): score_v5 nao usa alfa_fut nem informacao de T ou posterior — recalculo sem a coluna
sem_alvo = sc.drop(columns=['alfa_fut'])
s3 = 0.5 * sem_alvo[MOM3U].mean(axis=1) - 0.5 * sem_alvo[VOL2U].mean(axis=1)
dev = float(np.abs(s3.to_numpy() - sc['score_v5'].to_numpy()).max())
assert dev == 0.0, 'S3 FALHOU: score_v5 depende de alfa_fut'
# decil por mes — MESMA receita (qcut sobre rank first, ordem herdada de v4_scores)
sc['decil_v5'] = sc.groupby('mes')['score_v5'].transform(
    lambda s: pd.qcut(s.rank(method='first'), DECIL, labels=False))
sem_alvo = sc.drop(columns=['alfa_fut']).copy()
sem_alvo['d2'] = sem_alvo.groupby('mes')['score_v5'].transform(
    lambda s: pd.qcut(s.rank(method='first'), DECIL, labels=False))
dif = int((sem_alvo['d2'].to_numpy() != sc['decil_v5'].to_numpy()).sum())
assert dif == 0, 'S3 FALHOU no decil'
# os ranks unificados usam apenas features de t (b2_ranks, ja provado em B2/S4) — sem look-ahead
print(f'\nS3 PASSOU: score_v5 e decil_v5 recalculados sem alfa_fut com desvio 0 em todos os '
      f'{sc["mes"].nunique()} meses; ranks unificados usam apenas features de t (B2/D20).')

rho = spearman(sc['score_v5'], sc['score_v4'])
print(f'correlacao (Spearman) score_v5 x score_v4 no painel inteiro: {rho:.6f}')
mudou_decil = int((sc['decil_v5'] != sc['decil_v4']).sum())
mudou_excl = int(((sc['decil_v5'] == 0) != (sc['decil_v4'] == 0)).sum())
print(f'linhas que mudam de decil: {mudou_decil} de {len(sc)} ({100 * mudou_decil / len(sc):.2f}%); '
      f'mudam de status de EXCLUSAO: {mudou_excl} ({100 * mudou_excl / len(sc):.2f}%)')

# ============================================================ backtest V5 (motor identico a V4)
bench = pd.read_parquet(DIR_INT + r'\a6_benchmark_e_rf.parquet')
split = pd.read_parquet(DIR_INT + r'\a7_split.parquet')
split['CODISI'] = split['CODISI'].astype(str)
bi = bench.set_index('mes')
meses_all = sorted(bench['mes'].unique())
data_de_mes = {m: pd.Period(m, freq='M').to_timestamp(how='end').normalize() for m in meses_all}
mes_de_data = {v: k for k, v in data_de_mes.items()}
part_por_mes = sc.groupby('mes')['particao'].first().to_dict()
wide = split.pivot_table(index='mes', columns='CODISI', values='ret_1m', aggfunc='first').reindex(meses_all)
wide.index = pd.DatetimeIndex([data_de_mes[m] for m in meses_all])
cdi = bi['ret_cdi'].reindex(meses_all); cdi.index = wide.index
wide['__CDI__'] = cdi.values
CAIXA = wide['__CDI__']


def rodar(p_df, rot):
    p = p_df.copy()
    cols = [c for c in p.columns if (p[c] != 0).any()]
    p = p[cols].fillna(0.0)
    p.index = pd.DatetimeIndex([data_de_mes[m] for m in p.index])
    o = motor.rodar_backtest(p, wide[cols], custo_bps=0.0, retornos_caixa=CAIXA)
    rb = o['retornos_brutos']; tv = o['turnover'].reindex(rb.index)
    d = pd.DataFrame({'mes_ret': [mes_de_data[i] for i in rb.index],
                      'bruto': rb.values, 'turnover': tv.values})
    d['mes'] = [str(pd.Period(m, freq='M') - 1) for m in d['mes_ret']]
    d['particao'] = d['mes'].map(part_por_mes)
    for c in ['ret_indice', 'ret_indice_ex_max', 'ret_cdi']:
        d[c] = d['mes_ret'].map(bi[c])
    print(f'  [{rot}] ativos {len(cols) - (1 if "__CDI__" in cols else 0)}  meses {len(d)}  '
          f'congelamentos {len(o["eventos_congelamento"])}  descartados {o["rebal_descartados"]}')
    return d


def liq(d, bps):
    return d['bruto'] - d['turnover'] * (bps / 10000.0)


def matriz(df, col):
    m = df.pivot_table(index='mes', columns='CODISI', values=col, aggfunc='sum').reindex(meses_dec)
    m = m.fillna(0.0); m['__CDI__'] = 0.0
    return m


TE = lambda d: d[d['particao'] == 'TESTE'].reset_index(drop=True)

print('\n' + '=' * 126)
print('FASE B (D40) — V5 DE PONTA A PONTA (exclusao do decil inferior de score_v5; pesos EW e 1/vol;')
print('vol6_eff IDENTICA a de V4 — mesma coluna, mesmas linhas; motor, caixa CDI e deslistagem identicos)')
print('=' * 126)
v5 = sc[sc['decil_v5'] > 0].copy()
n_v5 = v5.groupby('mes').size().rename('n')
v5 = v5.merge(n_v5, on='mes')
v5['peso_ew'] = 1.0 / v5['n']
inv = 1.0 / v5['vol6_eff']
v5['peso_iv'] = inv / inv.groupby(v5['mes']).transform('sum')
for cnome in ['peso_ew', 'peso_iv']:
    soma = v5.groupby('mes')[cnome].sum()
    dev5 = float((soma - 1.0).abs().max())
    assert dev5 < 1e-9, f'pesos nao somam 1 em {cnome}'
print(f'posicoes: {len(v5)} | excluidas: {len(sc) - len(v5)} | base: {len(sc)} | pesos somam 1 (OK)')

d5ew = rodar(matriz(v5, 'peso_ew'), 'V5-EW (exclusao + peso igual)')
d5iv = rodar(matriz(v5, 'peso_iv'), 'V5 (exclusao + peso 1/vol_6m)')

# referencia V4 para o delta
v4bt = pd.read_parquet(DIR_INT + r'\v4_backtest.parquet')
c3 = pd.read_parquet(DIR_INT + r'\c3_curva_custo.parquet')
V4 = v4bt[v4bt['versao'] == 'V4'].sort_values('mes').reset_index(drop=True)
V4EW = v4bt[v4bt['versao'] == 'V4-EW'].sort_values('mes').reset_index(drop=True)
PITd = c3[c3['versao'] == 'EW_universo'].sort_values('mes').reset_index(drop=True)
for _d in [V4, V4EW, PITd]:
    for _c in ['ret_indice', 'ret_indice_ex_max', 'ret_cdi']:
        _d[_c] = _d['mes_ret'].map(bi[_c])

v5_50, v5_00 = anual(liq(TE(d5iv), 50)), anual(liq(TE(d5iv), 0))
v5ew_50, v5ew_00 = anual(liq(TE(d5ew), 50)), anual(liq(TE(d5ew), 0))
v4_50 = anual(TE(V4)['liq_50bps']); v4_00 = anual(TE(V4)['bruto'])
v4ew_50 = anual(TE(V4EW)['liq_50bps']); v4ew_00 = anual(TE(V4EW)['bruto'])
print('\nDELTA DA UNIFICACAO (V5 - V4, mesma formula, so a populacao dos ranks muda):')
print(f'  1/vol : bruto {v5_00:.4f} vs {v4_00:.4f} ({v5_00 - v4_00:+.4f} pp/ano) | '
      f'liq50 {v5_50:.4f} vs {v4_50:.4f} ({v5_50 - v4_50:+.4f} pp/ano)')
print(f'  EW    : bruto {v5ew_00:.4f} vs {v4ew_00:.4f} ({v5ew_00 - v4ew_00:+.4f} pp/ano) | '
      f'liq50 {v5ew_50:.4f} vs {v4ew_50:.4f} ({v5ew_50 - v4ew_50:+.4f} pp/ano)')
print('  referencia do conselho (Proximos Passos item 5): estrago medido de -0,14 pp/ano')

# ============================================================ metricas completas de V5
print('\n' + '=' * 126)
print('METRICAS DE V5 (mesmo cardapio de V4)')
print('=' * 126)

# (a) IC
lin = []
for rot, col in [('V4 (score_v4)', 'score_v4'), ('V5 (score_v5)', 'score_v5')]:
    for part in ['TREINO', 'TESTE']:
        out = []
        for m, g in sc[sc['particao'] == part].groupby('mes'):
            h = g[[col, 'alfa_fut']].dropna()
            if len(h) >= 10:
                out.append(spearman(h[col], h['alfa_fut']))
        s = pd.Series(out)
        _, _, tnw = nw_t(s, 3)
        lin.append({'serie': rot, 'particao': part, 'IC': s.mean(),
                    't_iid': t_iid(s), 't_NW3': tnw, 'n_meses': len(s)})
print('\n(a) IC mensal (Spearman score x alfa_fut):')
print(pd.DataFrame(lin).round(4).to_string(index=False))

# (b) curva de custo
lin = []
for rot, d, is_pq in [('V4-EW', V4EW, True), ('V4 (1/vol)', V4, True),
                      ('V5-EW', d5ew, False), ('V5 (1/vol)', d5iv, False)]:
    for part in ['TREINO', 'TESTE']:
        b = d[d['particao'] == part]
        reg = {'versao': rot, 'particao': part}
        for bps in NIVEIS_BPS:
            reg[f'{bps}bps'] = anual(b[f'liq_{bps}bps']) if is_pq else anual(liq(b, bps))
        lin.append(reg)
print('\n(b) curva de custo (%aa):')
print(pd.DataFrame(lin).round(4).to_string(index=False))

# (c) risco TESTE liq50
rf = TE(V4)['ret_cdi'].to_numpy()
lin = []
for rot, d, is_pq in [('V4 (1/vol)', V4, True), ('V5-EW', d5ew, False), ('V5 (1/vol)', d5iv, False)]:
    b = TE(d)
    r = (b['liq_50bps'] if is_pq else liq(b, 50)).to_numpy()
    e = r - rf
    lin.append({'serie': rot, 'bruto_%aa': anual(b['bruto']), 'liq50_%aa': anual(r),
                'vol_%aa': 100 * r.std(ddof=1) * np.sqrt(12),
                'Sharpe [NAO ORDENAVEL]': (e.mean() * 12) / (e.std(ddof=1) * np.sqrt(12)),
                'DDmax_%': dd_max(r), 'meses+_%': 100 * float((r > 0).mean()),
                'giro_ow_%mes': 100 * b['turnover'].mean() / 2})
print('\n(c) risco TESTE liq50 (giro em UMA perna; custo cobrado sobre DUAS — plano item 7):')
print(pd.DataFrame(lin).round(4).to_string(index=False))

# (d) excedentes vs EW PIT (bruto, convencao da manchete) e vs EW da base liq50
pit_te = TE(PITd)['bruto'].to_numpy()
n_base_ew = sc.groupby('mes').size()
base_ew = sc.merge(n_base_ew.rename('nb'), on='mes')
base_ew['peso'] = 1.0 / base_ew['nb']
dBASE = rodar(matriz(base_ew, 'peso'), 'EW da BASE (sem exclusao)')
base_te_liq = liq(TE(dBASE), 50).to_numpy()
lin = []
for rot, d, is_pq in [('V4 (1/vol)', V4, True), ('V5-EW', d5ew, False), ('V5 (1/vol)', d5iv, False)]:
    b = TE(d)
    r = (b['liq_50bps'] if is_pq else liq(b, 50)).to_numpy()
    reg = {'versao': rot}
    for nome, refv in [('EW_PIT(bruto)', pit_te), ('EW_base_liq50', base_te_liq)]:
        x = r - refv
        mu, _, t3 = nw_t(x, 3); _, _, t1 = nw_t(x, 1); _, _, t6 = nw_t(x, 6)
        reg[f'vs {nome} %mes'] = 100 * mu
        reg[f't1/t3/t6 {nome}'] = f'{t1:+.3f}/{t3:+.3f}/{t6:+.3f}'
    lin.append(reg)
print('\n(d) excedentes TESTE (V-liq50 x PIT bruto = construcao MISTA da manchete, declarada;')
print('    x EW base = liq50 x liq50, homogenea):')
print(pd.DataFrame(lin).round(4).to_string(index=False))
# homogeneas de V5 vs PIT
b5 = TE(d5iv)
mu_bb, _, t_bb = nw_t(b5['bruto'].to_numpy() - pit_te, 3)
pit50 = TE(PITd)['liq_50bps'].to_numpy()
mu_ll, _, t_ll = nw_t(liq(b5, 50).to_numpy() - pit50, 3)
mu_mx, _, t_mx = nw_t(liq(b5, 50).to_numpy() - pit_te, 3)
print(f'  V5 x EW PIT, TRES construcoes (bases declaradas): mista liq50xbruto {100 * mu_mx:+.4f} '
      f'(t {t_mx:+.3f}) | brutoxbruto {100 * mu_bb:+.4f} (t {t_bb:+.3f}) | liq50xliq50 '
      f'{100 * mu_ll:+.4f} (t {t_ll:+.3f})')

# (e) cascata reancorada no EW da base
bs00, bs50 = anual(TE(dBASE)['bruto']), anual(liq(TE(dBASE), 50))
for rot, d in [('V5-EW', d5ew), ('V5 (1/vol)', d5iv)]:
    b = TE(d)
    x50, x00 = anual(liq(b, 50)), anual(liq(b, 0))
    sel = x00 - bs00; liqd = x50 - bs50; giro = liqd - sel
    _, _, tl_ = nw_t(liq(b, 50).to_numpy() - base_te_liq, 3)
    print(f'\n(e) cascata {rot}: base {bs50:.4f} -> selecao {sel:+.4f} -> giro {giro:+.4f} -> '
          f'final {x50:.4f} (liquido vs base {liqd:+.4f}, t NW3 {tl_:+.3f})')

# (f) percentil nas 200 exclusoes aleatorias de V4 (identicas por construcao: o sorteio exclui
# 10% da MESMA base com a MESMA semente e pondera pela MESMA vol6_eff — independe do score)
dexc = pd.read_parquet(DIR_INT + r'\v4_exclusoes_aleatorias.parquet')
for rot, val50, val00, c50, c00 in [
        ('V5-EW vs 200 exclusoes EW', v5ew_50, v5ew_00, 'ew_liq50', 'ew_bruto'),
        ('V5 (1/vol) vs 200 exclusoes 1/vol', v5_50, v5_00, 'iv_liq50', 'iv_bruto')]:
    d50 = dexc[c50].to_numpy(); d00 = dexc[c00].to_numpy()
    print(f'\n(f) {rot}: a 50bps percentil {100 * float((d50 < val50).mean()):.1f} '
          f'(dist p5 {np.percentile(d50, 5):.2f} med {np.median(d50):.2f} max {d50.max():.2f}); '
          f'a custo 0 percentil {100 * float((d00 < val00).mean()):.1f}')

# (g) decil inferior de score_v5, duas leituras
print('\n(g) decil inferior de score_v5 (a propriedade que decide):')
for vrot, filtra in [('I  todas as linhas ', False), ('II sem alfa_fut NaN', True)]:
    dec = sc if not filtra else sc[sc['alfa_fut'].notna()]
    dec = dec.copy()
    dec['d'] = dec.groupby('mes')['score_v5'].transform(
        lambda s: pd.qcut(s.rank(method='first'), DECIL, labels=False))
    for part in ['TREINO', 'TESTE']:
        g = dec[(dec['particao'] == part) & (dec['d'] == 0)]
        serie = g.groupby('mes')['alfa_fut'].mean()
        mu, _, t3 = nw_t(serie, 3)
        print(f'  score_v5 [{vrot}] {part:6s}: alfa {100 * mu:+.4f}%/mes  t NW3 {t3:+.3f}  ({len(serie.dropna())}m)')

# (i) subperiodos
lin = []
for ini, fim in SUBPERIODOS:
    f5 = d5iv[(d5iv['mes'] >= ini) & (d5iv['mes'] <= fim) & (d5iv['particao'] == 'TESTE')]
    f4 = V4[(V4['mes'] >= ini) & (V4['mes'] <= fim) & (V4['particao'] == 'TESTE')]
    fp = PITd[(PITd['mes'] >= ini) & (PITd['mes'] <= fim) & (PITd['particao'] == 'TESTE')]
    lin.append({'subperiodo': f'{ini} a {fim}', 'meses': len(f5),
                'V4 liq50': anual(f4['liq_50bps']), 'V5 liq50': anual(liq(f5, 50)),
                'EW PIT bruto': anual(fp['bruto']), 'CDI': anual(f4['ret_cdi'])})
print('\n(i) subperiodos:')
print(pd.DataFrame(lin).round(4).to_string(index=False))

# ============================================================ gravacao (ARQUIVOS NOVOS v5_*)
sai_sc = sc[['CODISI', 'CODNEG', 'mes', 'subsetor', 'particao'] +
            [f + '_runif' for f in FEATS6] +
            ['alfa_fut', 'score_v5', 'decil_v5', 'vol_6m', 'vol6_eff']].copy()
sai_sc.to_parquet(DIR_INT + r'\v5_scores.parquet', index=False)
sai_ct = v5[['CODISI', 'CODNEG', 'mes', 'particao', 'n', 'peso_ew', 'peso_iv', 'vol_6m',
             'vol6_eff', 'score_v5', 'decil_v5']].copy()
sai_ct.to_parquet(DIR_INT + r'\v5_carteira.parquet', index=False)
xs = []
for rot, d in [('V5-EW', d5ew), ('V5', d5iv)]:
    x = d[['mes', 'mes_ret', 'particao', 'bruto', 'turnover']].copy()
    x['versao'] = rot
    for bps in NIVEIS_BPS:
        x[f'liq_{bps}bps'] = liq(d, bps)
    xs.append(x)
pd.concat(xs, ignore_index=True).to_parquet(DIR_INT + r'\v5_backtest.parquet', index=False)
print('\n' + '=' * 126)
print(f'GRAVADO v5_scores.parquet {sai_sc.shape} | v5_carteira.parquet {sai_ct.shape} | '
      f'v5_backtest.parquet ({2 * len(d5iv)}, 12) — ARQUIVOS NOVOS; nenhum dos 38 pre-existentes tocado.')
print('DECLARACAO OBRIGATORIA (D40): V5 e a QUINTA construcao medida nos mesmos 103 meses; nasceu de')
print('um defeito apontado por auditoria (populacoes de rank nao comparaveis), nao de busca por retorno.')
print('=' * 126)


D39 / PARTE 2.3 — FASE A: DIAGNOSTICO DA POPULACAO DOS SEIS RANKS
B2 (D20) calcula cada rank SO sobre elegiveis com AQUELA feature nao-nula -> cada um dos
seis ranks e um percentil de uma populacao ligeiramente diferente dentro do mesmo mes.

meses de decisao avaliados: 296 (de 345 meses no painel)
diferenca de populacao entre os seis ranks, POR MES (max_rank - min_rank):
  mediana 49.0 ativos | maximo 108 (mes 2007-06) | meses com diferenca > 0: 296 de 296
diferenca entre o rank mais povoado e a INTERSECAO (6 features presentes):
  mediana 49.0 | maximo 108 (mes 2007-06)

populacao por rank (mediana nos meses de decisao):
  mom_1m_rank      280.0
  mom_3m_rank      265.0
  mom_6m_rank      252.0
  mom_12m_rank     234.0
  vol_3m_rank      265.0
  vol_6m_rank      252.0
  intersecao       234.0

DEFEITO CONFIRMADO? SIM (populacoes diferem em 296 meses)

RANKS UNIFICADOS: percentil (0,1], metodo average, pct=True — a MESMA formula de B2 —
calculado sobre a INTERSECAO (elegiveis com as 6

## D41 / PARTE 3 — PERMUTACAO CORRIGIDA PARA O NUMERO DE ARQUITETURAS

A permutacao publicada (p ajustado 0,33-0,35) corrige a escolha do decil entre dez, mas NAO corrige o fato de que CINCO construcoes foram testadas nos mesmos 103 meses (V1, V2, V3, V4, V5; V2 e V3 compartilham o score_v2 — declarado). O ajuste completo e MAIS conservador: so pode piorar o p. Nulo: permutar alfa_fut dentro de cada mes do TESTE, 2.000 replicas, semente 20260815, com O MESMO sorteio alimentando as cinco construcoes (preserva a correlacao entre elas — maxT de Westfall-Young).

**OS TRES NIVEIS, LADO A LADO (2.000 replicas, semente 20260815):**
| nivel | observado | E[nulo] | p95 nulo | p |
|---|---|---|---|---|
| N1 sem ajuste (decil inferior de V5) | 2,4026 | 0,947 | 2,322 | **0,0425** |
| N2 ajustado por 10 decis | 2,4026 | 2,332 | 3,433 | **0,3985** |
| N3 ajustado por 10 decis x 5 construcoes | 2,4853 (V4) | 2,860 | 3,907 | **0,7350** |

S5 PASSOU (monotonico: 0,0425 <= 0,3985 <= 0,7350). **RESULTADO, nao ressalva: NAO E POSSIVEL DESCARTAR SORTE** — no unico grau em que a pergunta e respondivel com 103 meses. Bootstrap/Monte Carlo da serie de retorno como teste de significancia segue PROIBIDO (reamostra a propria amostra; a media e invariante); o bootstrap de bloco D3 permanece com a legenda original.

In [ ]:
# -*- coding: utf-8 -*-
# PARTE 3 (D41) — PERMUTACAO CORRIGIDA PARA O NUMERO DE ARQUITETURAS.
# Nulo: permutar alfa_fut DENTRO DE CADA MES do TESTE, 2.000 replicas, semente 20260815
# (declarada). Em cada replica, O MESMO sorteio permutado alimenta as CINCO construcoes
# (V1 'score', V2 'score_v2', V3 'score_v2' — V2 e V3 usam o MESMO score, declarado —,
# V4 'decil_v4', V5 'decil_v5'); t NW3 por decil sobre os 103 meses; tres niveis:
#   N1 sem ajuste algum        : |t| do decil inferior da versao final (V5)
#   N2 ajustado por 10 decis   : max|t| sobre os 10 decis de V5
#   N3 ajustado por 10 x 5     : max|t| sobre 10 decis x 5 construcoes
# O compartilhamento do sorteio entre construcoes preserva a correlacao entre elas
# (V4 e V5 quase identicas; V2 e V3 identicas) — e o nulo maxT de Westfall-Young.
import sys
import time

import numpy as np
import pandas as pd

if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8')

DIR_INT = r'C:\Users\lucca\quant2026\intermediario'
SEMENTE = 20260815
N_REP = 2000
DECIL = 10


def nw_t_batch(X, lags=3):
    mu = np.nanmean(X, axis=-1)
    d = X - mu[..., None]
    T = X.shape[-1]
    v = np.nansum(d * d, axis=-1) / T
    for l in range(1, lags + 1):
        w = 2.0 * (1.0 - l / (lags + 1.0))
        v += w * (np.nansum(d[..., l:] * d[..., :-l], axis=-1) / T)
    return mu / np.sqrt(v / T)


print('=' * 126)
print('D41 / PARTE 3 — PERMUTACAO EM TRES NIVEIS (decil inferior -> 10 decis -> 10 decis x 5 construcoes)')
print(f'semente {SEMENTE} (declarada), {N_REP} replicas, permutacao DENTRO de cada mes do TESTE.')
print('V2 e V3 usam o MESMO score_v2 (a carteira difere, o score nao) — o maximo sobre 5 construcoes')
print('e portanto o maximo sobre 4 particoes de decil distintas, declarado.')
print('=' * 126)

v2s = pd.read_parquet(DIR_INT + r'\v2_scores.parquet')
v4s = pd.read_parquet(DIR_INT + r'\v4_scores.parquet')
v5s = pd.read_parquet(DIR_INT + r'\v5_scores.parquet')
for d in (v2s, v4s, v5s):
    d['CODISI'] = d['CODISI'].astype(str)

t2 = v2s[v2s.particao == 'TESTE'][['CODISI', 'mes', 'alfa_fut', 'score', 'score_v2']].copy()
t4 = v4s[v4s.particao == 'TESTE'][['CODISI', 'mes', 'alfa_fut', 'decil_v4']].copy()
t5 = v5s[v5s.particao == 'TESTE'][['CODISI', 'mes', 'decil_v5']].copy()

meses = sorted(t2['mes'].unique())
assert len(meses) == 103

# base unica: t2 como espinha, decis de V4/V5 anexados por (CODISI, mes)
base = t2.merge(t4, on=['CODISI', 'mes'], suffixes=('', '_v4')).merge(t5, on=['CODISI', 'mes'])
assert len(base) == len(t2) == len(t4) == len(t5), 'bases divergem entre v2/v4/v5_scores — PARE'
dif_alfa = (base['alfa_fut'] - base['alfa_fut_v4']).abs()
assert int((dif_alfa > 1e-12).sum()) == 0, 'alfa_fut diverge entre paineis — PARE'

# decis de V1 e V2/V3 (mesma receita: qcut sobre rank first, por mes, todas as linhas)
base['decil_v1'] = base.groupby('mes')['score'].transform(
    lambda s: pd.qcut(s.rank(method='first'), DECIL, labels=False))
base['decil_v23'] = base.groupby('mes')['score_v2'].transform(
    lambda s: pd.qcut(s.rank(method='first'), DECIL, labels=False))

CONSTR = [('V1 (score OLS)', 'decil_v1'), ('V2 (score_v2)', 'decil_v23'),
          ('V3 (score_v2, = V2)', 'decil_v23'), ('V4 (score_v4)', 'decil_v4'),
          ('V5 (score_v5)', 'decil_v5')]
# particoes DISTINTAS para o nulo (V2==V3; contar duas vezes nao muda o maximo)
DISTINTAS = ['decil_v1', 'decil_v23', 'decil_v4', 'decil_v5']

# ---------------- observado ----------------
nm = len(meses)
obs = {}
for nome, col in CONSTR:
    M = np.full((DECIL, nm), np.nan)
    for j, m in enumerate(meses):
        g = base[base['mes'] == m]
        a = g['alfa_fut'].to_numpy()
        dd = g[col].to_numpy()
        for q in range(DECIL):
            sel = dd == q
            if sel.any():
                M[q, j] = np.nanmean(a[sel])
    obs[nome] = nw_t_batch(M)

print('\nt NW3 observado por decil (0 = inferior):')
for nome, _ in CONSTR:
    t = obs[nome]
    print(f'  {nome:20s} d0 {t[0]:+.4f} | max|t| {np.nanmax(np.abs(t)):.4f} '
          f'(decil {int(np.nanargmax(np.abs(t)))})')

obs_v5 = obs['V5 (score_v5)']
obs_N1 = float(abs(obs_v5[0]))
obs_N2 = float(np.nanmax(np.abs(obs_v5)))
obs_N3 = float(max(np.nanmax(np.abs(obs[n])) for n, _ in CONSTR))
constr_max = max(CONSTR, key=lambda nc: np.nanmax(np.abs(obs[nc[0]])))[0]
print(f'\nobservados: N1 (|t| d0 de V5) = {obs_N1:.4f} | N2 (max 10 decis de V5) = {obs_N2:.4f} | '
      f'N3 (max 10x5) = {obs_N3:.4f} (atingido por {constr_max})')

# ---------------- nulo ----------------
# posicoes fixas por (mes, construcao, decil) na ordem de linhas da base
pos = {}
alfas = {}
for j, m in enumerate(meses):
    g = base[base['mes'] == m]
    alfas[j] = g['alfa_fut'].to_numpy()
    for col in DISTINTAS:
        dd = g[col].to_numpy()
        for q in range(DECIL):
            pos[(j, col, q)] = np.flatnonzero(dd == q)

rng = np.random.default_rng(SEMENTE)
t0 = time.time()
# medias[col][q] -> (N_REP, nm)
medias = {col: np.full((DECIL, N_REP, nm), np.nan) for col in DISTINTAS}
for j in range(nm):
    a = alfas[j]
    n = len(a)
    rand = rng.random((N_REP, n))
    idx = np.argsort(rand, axis=1)
    ap = a[idx]                      # (N_REP, n): valor permutado na posicao fixa
    for col in DISTINTAS:
        for q in range(DECIL):
            p = pos[(j, col, q)]
            if len(p):
                medias[col][q, :, j] = np.nanmean(ap[:, p], axis=1)
print(f'\n{N_REP} replicas geradas em {time.time() - t0:.1f}s (um sorteio por mes-replica, '
      f'compartilhado pelas construcoes)')

t_null = {col: nw_t_batch(medias[col]) for col in DISTINTAS}   # (DECIL, N_REP)

null_N1 = np.abs(t_null['decil_v5'][0])                        # |t| d0 de V5
null_N2 = np.nanmax(np.abs(t_null['decil_v5']), axis=0)        # max 10 decis de V5
null_N3 = np.nanmax(np.stack([np.nanmax(np.abs(t_null[c]), axis=0) for c in DISTINTAS]), axis=0)

p_N1 = float((null_N1 >= obs_N1).mean())
p_N2 = float((null_N2 >= obs_N2).mean())
p_N3 = float((null_N3 >= obs_N3).mean())

print('\n' + '=' * 126)
print(f'OS TRES NIVEIS LADO A LADO ({N_REP} replicas, semente {SEMENTE}):')
print('=' * 126)
for rot, nul, ob, p in [
        ('N1 sem ajuste algum (|t| do decil inferior de V5)', null_N1, obs_N1, p_N1),
        ('N2 ajustado por 10 decis (max|t|, V5)', null_N2, obs_N2, p_N2),
        ('N3 ajustado por 10 decis x 5 construcoes', null_N3, obs_N3, p_N3)]:
    print(f'  {rot}')
    print(f'    E[stat nulo] {np.mean(nul):.4f} | mediana {np.median(nul):.4f} | '
          f'p95 {np.percentile(nul, 95):.4f} | observado {ob:.4f} | p = {p:.4f}')

print('\nS5 (monotonicidade): p_N1 <= p_N2 <= p_N3 ?', p_N1, '<=', p_N2, '<=', p_N3)
assert p_N1 <= p_N2 <= p_N3 + 1e-12, 'S5 FALHOU: p nao monotonico — nulo mal construido, PARE'
print('S5 PASSOU')

print('\nLEITURA OBRIGATORIA (vai ao relatorio como RESULTADO, nao como ressalva): mesmo no nivel')
print('mais favoravel (N1), e com o ajuste completo por 10 decis x 5 construcoes (N3), NAO E')
print('POSSIVEL DESCARTAR SORTE. TODO teste e nao-refutacao [F: poder ~331 meses].')

# distribui os nulos para F15 (PARTE 4)
np.savez(r'C:\Users\lucca\AppData\Local\Temp\claude\C--Users-lucca-quant2026\15d61e1c-f1bb-4b55-9668-bc9242897475\scratchpad\permut_3niveis.npz',
         null_N1=null_N1, null_N2=null_N2, null_N3=null_N3,
         obs=np.array([obs_N1, obs_N2, obs_N3]), p=np.array([p_N1, p_N2, p_N3]))
print('\nnulos salvos para F15 (scratchpad\\permut_3niveis.npz)')


D41 / PARTE 3 — PERMUTACAO EM TRES NIVEIS (decil inferior -> 10 decis -> 10 decis x 5 construcoes)
semente 20260815 (declarada), 2000 replicas, permutacao DENTRO de cada mes do TESTE.
V2 e V3 usam o MESMO score_v2 (a carteira difere, o score nao) — o maximo sobre 5 construcoes
e portanto o maximo sobre 4 particoes de decil distintas, declarado.

t NW3 observado por decil (0 = inferior):
  V1 (score OLS)       d0 -0.4619 | max|t| 1.8587 (decil 8)
  V2 (score_v2)        d0 -1.9323 | max|t| 1.9323 (decil 0)
  V3 (score_v2, = V2)  d0 -1.9323 | max|t| 1.9323 (decil 0)
  V4 (score_v4)        d0 -2.4853 | max|t| 2.4853 (decil 0)
  V5 (score_v5)        d0 -2.4026 | max|t| 2.4026 (decil 0)

observados: N1 (|t| d0 de V5) = 2.4026 | N2 (max 10 decis de V5) = 2.4026 | N3 (max 10x5) = 2.4853 (atingido por V4 (score_v4))

2000 replicas geradas em 5.7s (um sorteio por mes-replica, compartilhado pelas construcoes)

OS TRES NIVEIS LADO A LADO (2000 replicas, semente 20260815):
  N1 sem ajuste algum (|t

## E1c / PARTE 4 — TABELA FINAL CONSOLIDADA E FIGURAS FINAIS

`outputs\TABELA_FINAL.csv` (cabecalho declara periodo, custo, convencao de giro, bases do excedente, Sharpe nao ordenavel e o p dos tres niveis) + **F13** (curva final: V5 — declarada versao final por corrigir defeito mecanico, D40, NAO por resultado: rende 0,04 pp/ano MENOS que V4 —, EW PIT e CDI), **F14** (risco e retorno em barras, sem Sharpe), **F15** (permutacao em tres niveis) e regeneracao de **F5, F6, F7, F8, F9, F10, F12** com V5 incluida (paleta canonica estendida: V5 = `#6a3d9a`). F11 nao regerada — segue valida para V3/V4; a versao de tres niveis e F15. S6: nenhuma tabela ou figura ordena por Sharpe. Nota de leitura da coluna `t_NW3_exc_vs_EW_PIT`: para series quase identicas ao comparador (EW elegivel = a MESMA serie liquida) a variancia do excedente e minuscula e o t explode (-45) sem significado economico — mesma nota da celula D1.

In [ ]:
# -*- coding: utf-8 -*-
# PARTE 4 — TABELA FINAL CONSOLIDADA (outputs\TABELA_FINAL.csv) + F13/F14/F15 novas +
# regeneracao de F5/F6/F7/F8/F9/F10/F12 incluindo V5. PNG 150dpi, paleta canonica de E1b
# estendida (V5 = '#6a3d9a'), eixos com zero. Nenhum parquet pre-existente tocado.
import sys
import os

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8')

RAIZ = r'C:\Users\lucca\quant2026'
DIR_INT = RAIZ + r'\intermediario'
DIR_OUT = RAIZ + r'\outputs'
SCRATCH = r'C:\Users\lucca\AppData\Local\Temp\claude\C--Users-lucca-quant2026\15d61e1c-f1bb-4b55-9668-bc9242897475\scratchpad'
if RAIZ + r'\src' not in sys.path:
    sys.path.insert(0, RAIZ + r'\src')
import motor  # noqa: E402

plt.rcParams.update({
    'figure.dpi': 150, 'savefig.dpi': 150, 'font.size': 10.5, 'font.family': 'DejaVu Sans',
    'axes.edgecolor': '#333333', 'axes.labelcolor': '#1a1a1a', 'text.color': '#1a1a1a',
    'xtick.color': '#333333', 'ytick.color': '#333333', 'axes.grid': True,
    'grid.color': '#d9d9d9', 'grid.linewidth': 0.6, 'axes.axisbelow': True,
    'legend.frameon': False, 'axes.spines.top': False, 'axes.spines.right': False,
})
COR = {'V1': '#4d4d4d', 'V2': '#2c6e91', 'V3': '#a4405e', 'V4EW': '#1f7a72', 'V4': '#c1652b',
       'V5': '#6a3d9a', 'PIT': '#1a4d6b', 'cdi': '#b08a2e', 'indice': '#8c8c8c',
       'base': '#b0a08c', 'eleg': '#5a8a3c',
       'preto': '#0b0b0b', 'cinza_med': '#898781', 'cinza_esc': '#52514e', 'cinza_clr': '#c3c2b7'}
LS = {'V1': '-', 'V2': '--', 'V3': '-.', 'V4EW': '--', 'V4': '-', 'V5': '-', 'PIT': '--',
      'cdi': ':', 'indice': '-.'}
LW = {'V1': 1.3, 'V2': 1.3, 'V3': 1.3, 'V4EW': 1.5, 'V4': 1.6, 'V5': 2.3, 'PIT': 1.9,
      'cdi': 1.2, 'indice': 1.1}


def nw_t(x, lags=3):
    x = np.asarray(pd.Series(x).dropna(), dtype=float)
    n = len(x); mu = x.mean(); d = x - mu
    v = float(d @ d) / n
    for l in range(1, lags + 1):
        v += 2.0 * (1.0 - l / (lags + 1.0)) * (float(d[l:] @ d[:-l]) / n)
    se = np.sqrt(v / n)
    return mu, se, mu / se


def anual(r):
    r = np.asarray(pd.Series(r).dropna(), dtype=float)
    return 100.0 * ((np.prod(1.0 + r)) ** (12.0 / len(r)) - 1.0)


def dd_max(r):
    r = np.asarray(pd.Series(r).dropna(), dtype=float)
    w = np.cumprod(1 + r); p = np.maximum.accumulate(w)
    return 100.0 * float((w / p - 1.0).min())


def salvar(fig, nome):
    caminho = DIR_OUT + '\\' + nome
    fig.savefig(caminho, bbox_inches='tight', facecolor='white')
    plt.close(fig)
    print(f'  GRAVADO {caminho}  ({os.path.getsize(caminho) / 1024:.1f} KB)')
    return caminho


print('=' * 126)
print('PARTE 4 — TABELA FINAL CONSOLIDADA + F13/F14/F15 + regeneracao F5-F10/F12 com V5')
print('=' * 126)

# ==================================================================== insumos
bench = pd.read_parquet(DIR_INT + r'\a6_benchmark_e_rf.parquet').set_index('mes')
c3 = pd.read_parquet(DIR_INT + r'\c3_curva_custo.parquet')
v2bt = pd.read_parquet(DIR_INT + r'\v2_backtest.parquet')
v3bt = pd.read_parquet(DIR_INT + r'\v3_backtest.parquet')
v4bt = pd.read_parquet(DIR_INT + r'\v4_backtest.parquet')
v5bt = pd.read_parquet(DIR_INT + r'\v5_backtest.parquet')

SER = {
    'V1': c3[c3['versao'] == 'C2-A'],
    'V2': v2bt[v2bt['versao'] == 'V2-A'],
    'V3': v3bt[v3bt['versao'] == 'V3'],
    'V4EW': v4bt[v4bt['versao'] == 'V4-EW'],
    'V4': v4bt[v4bt['versao'] == 'V4'],
    'V5': v5bt[v5bt['versao'] == 'V5'],
    'V5EW': v5bt[v5bt['versao'] == 'V5-EW'],
    'PIT': c3[c3['versao'] == 'EW_universo'],
}
TE = {k: v[v['particao'] == 'TESTE'].sort_values('mes_ret').reset_index(drop=True) for k, v in SER.items()}
meses_te = list(TE['V1']['mes_ret'])
for k, d in TE.items():
    assert len(d) == 103 and list(d['mes_ret']) == meses_te, f'{k}: TESTE desalinhado'
cdi_te = bench.loc[meses_te, 'ret_cdi'].reset_index(drop=True)
idx_te = bench.loc[meses_te, 'ret_indice'].reset_index(drop=True)

# EW da base via motor (mesma receita de F9/V4; nada gravado)
sc = pd.read_parquet(DIR_INT + r'\v2_scores.parquet')
split = pd.read_parquet(DIR_INT + r'\a7_split.parquet')
sc['CODISI'] = sc['CODISI'].astype(str)
split['CODISI'] = split['CODISI'].astype(str)
meses_all = sorted(bench.reset_index()['mes'].unique())
data_de_mes = {m: pd.Period(m, freq='M').to_timestamp(how='end').normalize() for m in meses_all}
mes_de_data = {v: k for k, v in data_de_mes.items()}
part_por_mes = sc.groupby('mes')['particao'].first().to_dict()
wide = split.pivot_table(index='mes', columns='CODISI', values='ret_1m', aggfunc='first').reindex(meses_all)
wide.index = pd.DatetimeIndex([data_de_mes[m] for m in meses_all])
cdi_full = bench['ret_cdi'].reindex(meses_all); cdi_full.index = wide.index
n_base = sc.groupby('mes').size()
base_ew = sc.merge(n_base.rename('n'), on='mes')
base_ew['peso'] = 1.0 / base_ew['n']
m_base = base_ew.pivot_table(index='mes', columns='CODISI', values='peso', aggfunc='sum').reindex(
    sorted(sc['mes'].unique())).fillna(0.0)
m_base.index = pd.DatetimeIndex([data_de_mes[m] for m in m_base.index])
colsb = [c for c in m_base.columns if (m_base[c] != 0).any()]
o = motor.rodar_backtest(m_base[colsb], wide[colsb], custo_bps=0.0, retornos_caixa=cdi_full)
rb, tv = o['retornos_brutos'], o['turnover'].reindex(o['retornos_brutos'].index)
dBASE = pd.DataFrame({'mes_ret': [mes_de_data[i] for i in rb.index], 'bruto': rb.values,
                      'turnover': tv.values})
dBASE['mes'] = [str(pd.Period(m, freq='M') - 1) for m in dBASE['mes_ret']]
dBASE['particao'] = dBASE['mes'].map(part_por_mes)
b_base = dBASE[dBASE['particao'] == 'TESTE'].sort_values('mes_ret').reset_index(drop=True)
assert list(b_base['mes_ret']) == meses_te
b_base['liq_50bps'] = b_base['bruto'] - b_base['turnover'] * (50 / 10000.0)

# nomes/mes (mediana TESTE)
def nomes_mediana(arq, filtro=None):
    d = pd.read_parquet(DIR_INT + '\\' + arq)
    d = d[d['particao'] == 'TESTE']
    if filtro:
        d = d[filtro(d)]
    return float(d.groupby('mes').size().median())

nm_v1 = nomes_mediana('c1_carteira.parquet')
nm_v2 = nomes_mediana('v2_carteira.parquet')
nm_v3 = nomes_mediana('v3_carteira.parquet')
nm_v4 = nomes_mediana('v4_carteira.parquet')
nm_v5 = nomes_mediana('v5_carteira.parquet')
eleg = split[(split['particao'] == 'TESTE') & split['elegivel']]
nm_eleg = float(eleg.groupby('mes').size().median())
nm_base = float(sc[sc['particao'] == 'TESTE'].groupby('mes').size().median())
nm_idx = float(bench.loc[meses_te, 'n_ativos_indice'].median())

# ==================================================================== 4.1 TABELA FINAL
pit_bruto = TE['PIT']['bruto'].to_numpy()
rf = cdi_te.to_numpy()

def linha(nome, serie_ret, giro_ow, nomes, bruto_aa, liq50_aa, vs_pit=True):
    r = np.asarray(serie_ret, dtype=float)
    e = r - rf
    t3 = np.nan
    if vs_pit:
        _, _, t3 = nw_t(r - pit_bruto, 3)
    return {'serie': nome, 'bruto_%aa': bruto_aa, 'liq50_%aa': liq50_aa,
            'vol_%aa': 100 * r.std(ddof=1) * np.sqrt(12), 'DDmax_%': dd_max(r),
            'meses+_%': 100 * float((r > 0).mean()), 'giro_ow_%mes': giro_ow,
            'n_nomes_med': nomes, 't_NW3_exc_vs_EW_PIT': t3,
            'Sharpe [NAO ORDENAVEL]': (e.mean() * 12) / (e.std(ddof=1) * np.sqrt(12))
            if e.std(ddof=1) > 0 else np.nan}

linhas = []
for rot, k, nm in [('V1 (C2-A)', 'V1', nm_v1), ('V2 (V2-A)', 'V2', nm_v2), ('V3', 'V3', nm_v3),
                   ('V4-EW', 'V4EW', nm_v4), ('V4 (1/vol)', 'V4', nm_v4), ('V5 (1/vol)', 'V5', nm_v5)]:
    b = TE[k]
    linhas.append(linha(rot, b['liq_50bps'], 100 * b['turnover'].mean() / 2, nm,
                        anual(b['bruto']), anual(b['liq_50bps'])))
# comparadores
bp = TE['PIT']
linhas.append(linha('EW point-in-time (comparador de manchete, custo zero)', bp['bruto'],
                    100 * bp['turnover'].mean() / 2, nm_eleg, anual(bp['bruto']), np.nan,
                    vs_pit=False))
linhas.append(linha('EW elegivel (mesma serie, liq50)', bp['liq_50bps'],
                    100 * bp['turnover'].mean() / 2, nm_eleg, anual(bp['bruto']),
                    anual(bp['liq_50bps'])))
linhas.append(linha('EW da base', b_base['liq_50bps'], 100 * b_base['turnover'].mean() / 2,
                    nm_base, anual(b_base['bruto']), anual(b_base['liq_50bps'])))
linhas.append(linha('CDI', cdi_te, np.nan, np.nan, anual(cdi_te), anual(cdi_te)))
linhas.append(linha('indice interno [NAO INVESTIVEL]', idx_te, np.nan, nm_idx,
                    anual(idx_te), anual(idx_te)))
tabf = pd.DataFrame(linhas)

CAB = [
    '# TABELA FINAL CONSOLIDADA — TESTE: decisao 2018-01 a 2026-07, retornos 2018-02 a 2026-08 (103 meses)',
    '# custo: 50 bps por perna, cobrado sobre DUAS pernas (soma |dw|); giro_ow_%mes publicado em UMA perna (plano item 7)',
    '# EW point-in-time: comparador de manchete, SEMPRE custo zero (convencao declarada); EW elegivel = a MESMA serie liquida de 50 bps',
    '# t_NW3_exc_vs_EW_PIT: excedente (serie liq50, PIT bruto) — construcao MISTA da manchete, declarada (plano item 4);',
    '#   base contra base, V4: bruto x bruto +0,1357 t +0,879 | liq50 x liq50 +0,0636 t +0,416; V5: +0,1328 t +0,859 | +0,0606 t +0,395',
    '# Sharpe: [NAO ORDENAVEL] (plano item 6; Israelsen 2005) — com excesso negativo contra o CDI, mu/sigma cresce com sigma',
    '# ordem das linhas: arquitetural (V1..V5, depois comparadores) — NADA ordenado por Sharpe (S6)',
    '# permutacao em tres niveis (D41, 2.000 replicas, semente 20260815): p 0,0425 (decil inferior de V5) ->',
    '#   0,3985 (10 decis) -> 0,7350 (10 decis x 5 construcoes) — nao e possivel descartar sorte',
]
with open(DIR_OUT + r'\TABELA_FINAL.csv', 'w', encoding='utf-8-sig', newline='') as f:
    f.write('\n'.join(CAB) + '\n')
    tabf.round(4).to_csv(f, index=False)
print('\n4.1 — TABELA FINAL CONSOLIDADA (gravada em outputs\\TABELA_FINAL.csv):')
for c in CAB:
    print(' ', c)
print()
print(tabf.round(4).to_string(index=False))
print('\nS6: ordem arquitetural fixa, nenhuma coluna de ordenacao por Sharpe. PASSOU')

# ==================================================================== F13 — curva final
print('\n[F13] fonte: v5_backtest.parquet (V5 liq50), c3_curva_custo.parquet (EW PIT bruto), '
      'a6_benchmark_e_rf.parquet (CDI)')
x = pd.to_datetime([pd.Period(m, freq='M').to_timestamp(how='end') for m in meses_te])
fig, ax = plt.subplots(figsize=(9.5, 5.4))
ax.plot(x, 100 * (1 + TE['V5']['liq_50bps']).cumprod(), color=COR['V5'], lw=2.3,
        label='V5 (versão final — correção de população de rank, D40), líq. 50 bps')
ax.plot(x, 100 * (1 + TE['PIT']['bruto']).cumprod(), color=COR['PIT'], ls='--', lw=1.9,
        label='EW ponto-no-tempo (comparador de manchete, custo zero)')
ax.plot(x, 100 * (1 + cdi_te).cumprod(), color=COR['cdi'], ls=':', lw=1.6, label='CDI')
ax.set_ylim(bottom=0)
ax.set_ylabel('retorno acumulado (base 100 em 2018-01)')
ax.set_xlabel('TESTE: 2018-01 a 2026-07 (retornos 2018-02 a 2026-08)')
leg = ax.legend(loc='upper left', fontsize=8.6, frameon=True, facecolor='white',
                framealpha=0.94, edgecolor='#cccccc')
leg.get_frame().set_linewidth(0.6)
mu_mx, _, t_mx = nw_t(TE['V5']['liq_50bps'].to_numpy() - pit_bruto, 3)
fig.text(0.5, -0.03,
         f'V5 declarada versão final por corrigir defeito mecânico medido (D40), não por resultado — rende 0,04 pp/ano MENOS que V4. '
         f'Contra o comparador: indistinguível de zero em toda construção testada, |t| ≤ 0,88 '
         f'(mista líq50×bruto {100 * mu_mx:+.4f}%/mês t {t_mx:+.3f}). Nenhuma versão supera o CDI ({anual(cdi_te):.2f}%aa).',
         ha='center', fontsize=8.0, color=COR['V1'])
salvar(fig, 'F13_curva_final.png')

# ==================================================================== F14 — risco e retorno
print('\n[F14] fonte: TABELA_FINAL (mesmos numeros; sem Sharpe)')
ordem14 = ['V1 (C2-A)', 'V2 (V2-A)', 'V3', 'V4-EW', 'V4 (1/vol)', 'V5 (1/vol)',
           'EW point-in-time (comparador de manchete, custo zero)', 'EW da base',
           'indice interno [NAO INVESTIVEL]', 'CDI']
rot14 = ['V1', 'V2', 'V3', 'V4-EW', 'V4', 'V5', 'EW PIT', 'EW base', 'índice*', 'CDI']
cores14 = [COR['V1'], COR['V2'], COR['V3'], COR['V4EW'], COR['V4'], COR['V5'],
           COR['PIT'], COR['base'], COR['indice'], COR['cdi']]
sub = tabf.set_index('serie').loc[ordem14]
ret14 = sub['liq50_%aa'].fillna(sub['bruto_%aa'])  # PIT: custo zero por convencao (declarado)
fig, axes = plt.subplots(1, 3, figsize=(13.5, 4.8))
for ax, vals, titulo in [
        (axes[0], ret14, 'retorno líquido 50bps (%aa)\n(EW PIT: custo zero, convenção)'),
        (axes[1], sub['vol_%aa'], 'volatilidade anualizada (%aa)'),
        (axes[2], sub['DDmax_%'], 'drawdown máximo (%)')]:
    xs = np.arange(len(rot14))
    ax.bar(xs, vals.to_numpy(), color=cores14, edgecolor='white')
    ax.set_xticks(xs); ax.set_xticklabels(rot14, rotation=35, ha='right', fontsize=8.0)
    ax.axhline(0, color='#666666', lw=0.8)
    ax.set_title(titulo, fontsize=9.0)
fig.text(0.5, -0.04, '* índice interno NÃO INVESTÍVEL (elegibilidade usa informação do próprio mês). '
                     'Sem Sharpe: a razão inverte ordenação sob excesso negativo (não ordenável, plano item 6).',
         ha='center', fontsize=8.2, color=COR['V1'])
fig.tight_layout()
salvar(fig, 'F14_risco_retorno.png')

# ==================================================================== F15 — permutacao 3 niveis
print('\n[F15] fonte: nulos de D41 (2.000 replicas, semente 20260815; scratchpad\\permut_3niveis.npz)')
npz = np.load(SCRATCH + r'\permut_3niveis.npz')
nulos = [npz['null_N1'], npz['null_N2'], npz['null_N3']]
obs3 = npz['obs']; p3 = npz['p']
titulos = ['N1: sem ajuste\n(|t| do decil inferior de V5)', 'N2: ajustado por 10 decis\n(max|t|, V5)',
           'N3: ajustado por 10 decis\n× 5 construções']
fig, axes = plt.subplots(1, 3, figsize=(13.5, 4.6), sharey=False)
for ax, nul, ob, p, tt in zip(axes, nulos, obs3, p3, titulos):
    ax.hist(nul, bins=40, color=COR['cinza_clr'], edgecolor=COR['cinza_esc'], linewidth=0.5)
    ax.axvline(ob, color=COR['V5'], lw=2.2)
    ax.text(0.97, 0.95, f'observado = {ob:.3f}\np = {p:.4f}', transform=ax.transAxes,
            ha='right', va='top', fontsize=9.0, color=COR['V5'],
            bbox=dict(facecolor='white', edgecolor='none', alpha=0.8, pad=2))
    ax.set_title(tt, fontsize=9.0)
    ax.set_xlim(left=0)
    ax.set_xlabel('estatística sob o nulo')
axes[0].set_ylabel('frequência (2.000 réplicas)')
fig.text(0.5, -0.04, 'Permutação dentro de cada mês do TESTE, mesmo sorteio alimentando as cinco construções '
                     '(V2 e V3 compartilham score, declarado). p monotônico 0,0425 → 0,3985 → 0,7350: '
                     'com o ajuste completo pela busca, NÃO é possível descartar sorte (D41).',
         ha='center', fontsize=8.2, color=COR['V1'])
fig.tight_layout()
salvar(fig, 'F15_permutacao_3niveis.png')

# ==================================================================== F5 (com V5)
print('\n[F5] regenerada com V5; fontes anteriores + v5_backtest.parquet')
LEG = {'V1': 'V1-A (C2-A)', 'V2': 'V2-A', 'V3': 'V3', 'V4EW': 'V4-EW (peso igual)',
       'V4': 'V4 (peso 1/vol)', 'V5': 'V5 (final: ranks unificados, D40)',
       'PIT': 'EW ponto-no-tempo (comparador de manchete)'}
fig, ax = plt.subplots(figsize=(9.5, 5.6))
for k in ['V1', 'V2', 'V3', 'V4EW', 'V4']:
    ax.plot(x, 100 * (1 + TE[k]['liq_50bps']).cumprod(), color=COR[k], ls=LS[k], lw=LW[k],
            label=LEG[k])
ax.plot(x, 100 * (1 + idx_te).cumprod(), color=COR['indice'], ls=LS['indice'], lw=LW['indice'],
        label='índice interno — NÃO INVESTÍVEL')
ax.plot(x, 100 * (1 + cdi_te).cumprod(), color=COR['cdi'], ls=LS['cdi'], lw=LW['cdi'], label='CDI')
ax.plot(x, 100 * (1 + TE['PIT']['bruto']).cumprod(), color=COR['PIT'], ls=LS['PIT'], lw=LW['PIT'],
        label=LEG['PIT'] + ' (custo zero)')
ax.plot(x, 100 * (1 + TE['V5']['liq_50bps']).cumprod(), color=COR['V5'], ls=LS['V5'], lw=LW['V5'],
        label=LEG['V5'], marker='o', markevery=12, markersize=4)
ax.set_ylim(bottom=0)
ax.set_ylabel('retorno acumulado, líquido de 50 bps (base 100 em 2018-01)')
ax.set_xlabel('TESTE: 2018-01 a 2026-07 (retornos 2018-02 a 2026-08)')
leg = ax.legend(loc='upper left', fontsize=8.1, ncol=1, frameon=True, facecolor='white',
                framealpha=0.94, edgecolor='#cccccc')
leg.get_frame().set_linewidth(0.6)
exc_pit = 100 * (TE['V5']['liq_50bps'].to_numpy() - pit_bruto)
mu5, _, t5 = nw_t(exc_pit, 3)
fig.text(0.5, -0.02,
         f'V5 contra o comparador de manchete: indistinguível de zero em toda construção testada, |t| ≤ 0,88 '
         f'(mista líq50×bruto {mu5:+.4f}%/mês t {t5:+.3f}; bruto×bruto +0,1328 t +0,859; líq50×líq50 +0,0606 t +0,395 — '
         f'bases declaradas, plano item 4). Nenhuma versão supera o CDI ({anual(cdi_te):.4f}%aa).',
         ha='center', fontsize=8.0, color=COR['V1'])
salvar(fig, 'F5_curva_acumulada_teste.png')

# ==================================================================== F6 (com V5)
print('\n[F6] regenerada com V5')
bps_cols = ['liq_0bps', 'liq_10bps', 'liq_25bps', 'liq_50bps', 'liq_75bps', 'liq_100bps']
bps_x = [0, 10, 25, 50, 75, 100]
fig, ax = plt.subplots(figsize=(8.6, 5.4))
for k in ['V1', 'V2', 'V3', 'V4EW', 'V4', 'V5']:
    ys = [anual(TE[k][c]) for c in bps_cols]
    ax.plot(bps_x, ys, color=COR[k], ls=LS[k], lw=LW[k], marker='o', markersize=4, label=LEG[k])
    y50 = anual(TE[k]['liq_50bps'])
    ax.scatter([50], [y50], color=COR[k], s=42, zorder=5, edgecolor='white', linewidth=0.6)
    ax.annotate(f'{y50:+.2f}', (50, y50), textcoords='offset points', xytext=(6, 4), fontsize=7.5,
                color=COR[k])
pit_0 = anual(TE['PIT']['bruto']); cdi_aa = anual(cdi_te)
ax.axhline(pit_0, color=COR['PIT'], ls=LS['PIT'], lw=LW['PIT'],
           label=f'EW ponto-no-tempo (comparador, custo zero) = {pit_0:.4f}%aa')
ax.axhline(cdi_aa, color=COR['cdi'], ls=LS['cdi'], lw=LW['cdi'], label=f'CDI = {cdi_aa:.4f}%aa')
ax.axhline(0.0, color='#666666', lw=0.8)
ax.set_xlabel('custo por perna (bps) — cobrado sobre DUAS pernas (Σ|Δw|)')
ax.set_ylabel('retorno líquido anualizado, TESTE (%aa)')
leg = ax.legend(loc='lower left', fontsize=7.9, frameon=True, facecolor='white', framealpha=0.94,
                edgecolor='#cccccc')
leg.get_frame().set_linewidth(0.6)
salvar(fig, 'F6_curva_de_custo.png')

# ==================================================================== F7 (com V5)
print('\n[F7] regenerada com V5 (V5 contra os MESMOS 200 sorteios de V4 — a exclusao aleatoria '
      'independe do score, mesma base/semente/ponderacao)')
d1a = pd.read_parquet(DIR_INT + r'\d1_aleatorias.parquet')
v2a = pd.read_parquet(DIR_INT + r'\v2_aleatorias.parquet')
v3ea = pd.read_parquet(DIR_INT + r'\v3_exclusoes_aleatorias.parquet')
v4ea = pd.read_parquet(DIR_INT + r'\v4_exclusoes_aleatorias.parquet')


def pct_of(dist, valor):
    dist = np.asarray(dist, dtype=float)
    return 100.0 * float((dist <= valor).sum()) / len(dist)


reais = {k: anual(TE[k]['liq_50bps']) for k in ['V1', 'V2', 'V3', 'V4EW', 'V4', 'V5']}
pcts = {'V1': pct_of(d1a['anual_50bps'], reais['V1']), 'V2': pct_of(v2a['anual_50bps'], reais['V2']),
        'V3': pct_of(v3ea['liq50'], reais['V3']), 'V4EW': pct_of(v4ea['ew_liq50'], reais['V4EW']),
        'V4': pct_of(v4ea['iv_liq50'], reais['V4']), 'V5': pct_of(v4ea['iv_liq50'], reais['V5'])}
print('  percentis (liq50): ' + ' '.join(f'{k}={v:.1f}' for k, v in pcts.items()))

fig, (axA, axB) = plt.subplots(1, 2, figsize=(13, 5))
axA.hist(d1a['anual_50bps'], bins=22, color=COR['V1'], edgecolor='white', alpha=0.55,
         label='200 seleções aleatórias (V1)')
axA.hist(v2a['anual_50bps'], bins=22, color=COR['V2'], edgecolor='white', alpha=0.55,
         label='200 seleções aleatórias (V2)')
axA.axvline(reais['V1'], color=COR['V1'], lw=2.2, ls='-',
            label=f'V1 real = {reais["V1"]:.2f}%aa (percentil {pcts["V1"]:.0f})')
axA.axvline(reais['V2'], color=COR['V2'], lw=2.2, ls='--',
            label=f'V2 real = {reais["V2"]:.2f}%aa (percentil {pcts["V2"]:.0f})')
axA.axvline(0, color='#999999', lw=0.8)
axA.set_xlabel('retorno anualizado líquido de 50bps (200 seleções aleatórias)')
axA.set_ylabel('frequência')
axA.legend(loc='upper left', fontsize=8.3)
bins = np.linspace(min(v3ea['liq50'].min(), v4ea['ew_liq50'].min(), v4ea['iv_liq50'].min()),
                   max(v3ea['liq50'].max(), v4ea['ew_liq50'].max(), v4ea['iv_liq50'].max()), 26)
axB.hist(v3ea['liq50'], bins=bins, histtype='step', color=COR['V3'], lw=1.8,
         label='200 exclusões aleatórias (V3)')
axB.hist(v4ea['ew_liq50'], bins=bins, histtype='step', color=COR['V4EW'], lw=1.8,
         label='200 exclusões aleatórias (V4-EW/V5-EW)')
axB.hist(v4ea['iv_liq50'], bins=bins, histtype='step', color=COR['V4'], lw=1.8,
         label='200 exclusões aleatórias (1/vol; V4 e V5)')
axB.axvline(reais['V3'], color=COR['V3'], lw=2.2, ls='-',
            label=f'V3 real = {reais["V3"]:.2f}%aa (percentil {pcts["V3"]:.0f})')
axB.axvline(reais['V4EW'], color=COR['V4EW'], lw=2.2, ls='--',
            label=f'V4-EW real = {reais["V4EW"]:.2f}%aa (percentil {pcts["V4EW"]:.0f})')
axB.axvline(reais['V4'], color=COR['V4'], lw=2.2, ls='-.',
            label=f'V4 real = {reais["V4"]:.2f}%aa (percentil {pcts["V4"]:.0f})')
axB.axvline(reais['V5'], color=COR['V5'], lw=2.4, ls='-',
            label=f'V5 real = {reais["V5"]:.2f}%aa (percentil {pcts["V5"]:.0f}, mesmos sorteios)')
axB.axvline(0, color='#999999', lw=0.8)
axB.set_xlabel('retorno anualizado líquido de 50bps (200 exclusões aleatórias, 10% da base)')
axB.legend(loc='upper left', fontsize=7.8)
salvar(fig, 'F7_distribuicoes_aleatorias.png')

# ==================================================================== F8 (com V5)
print('\n[F8] regenerada com score_v5 sobreposto')
v4s = pd.read_parquet(DIR_INT + r'\v4_scores.parquet')
v2s = pd.read_parquet(DIR_INT + r'\v2_scores.parquet')
v5s = pd.read_parquet(DIR_INT + r'\v5_scores.parquet')


def decis_por(df, col_dec, particao, precomputado=True, col_score=None, min_n=0):
    sub = df[(df['particao'] == particao) & df['alfa_fut'].notna()]
    lin = []
    for m, g in sub.groupby('mes'):
        if len(g) < min_n:
            continue
        g = g.copy()
        if not precomputado:
            g['__d'] = pd.qcut(g[col_score].rank(method='first'), 10, labels=False)
            cd = '__d'
        else:
            cd = col_dec
        for d in range(10):
            v = g.loc[g[cd] == d, 'alfa_fut']
            if len(v):
                lin.append({'mes': m, 'decil': d, 'alfa': float(v.mean())})
    dd = pd.DataFrame(lin)
    out = []
    for d in range(10):
        s = dd.loc[dd['decil'] == d, 'alfa']
        mu, se, t = nw_t(s, 3)
        out.append({'decil': d, 'media_%mes': 100 * mu, 'ep_%mes': 100 * se, 't_NW3': t})
    return pd.DataFrame(out)


res_v4 = {p: decis_por(v4s, 'decil_v4', p) for p in ['TESTE', 'TREINO']}
res_v2 = {p: decis_por(v2s, None, p, precomputado=False, col_score='score_v2', min_n=20)
          for p in ['TESTE', 'TREINO']}
res_v5 = {p: decis_por(v5s, 'decil_v5', p) for p in ['TESTE', 'TREINO']}
print('  score_v4 TESTE d0:', round(res_v4['TESTE'].loc[0, 'media_%mes'], 4),
      '| score_v5 TESTE d0:', round(res_v5['TESTE'].loc[0, 'media_%mes'], 4))

lo = []; hi = []
for p in ['TESTE', 'TREINO']:
    lo += (res_v4[p]['media_%mes'] - res_v4[p]['ep_%mes']).tolist()
    hi += (res_v4[p]['media_%mes'] + res_v4[p]['ep_%mes']).tolist()
    lo += res_v2[p]['media_%mes'].tolist() + res_v5[p]['media_%mes'].tolist()
    hi += res_v2[p]['media_%mes'].tolist() + res_v5[p]['media_%mes'].tolist()
ymin, ymax = min(lo) - 0.3, max(hi) + 0.3
fig, (axL, axR) = plt.subplots(1, 2, figsize=(13, 5.4), sharey=True)
xd = np.arange(1, 11)
for ax, p in [(axL, 'TESTE'), (axR, 'TREINO')]:
    cores_dec = [COR['V4'] if d == 0 else (COR['V2'] if d == 9 else '#c9c9c9')
                 for d in res_v4[p]['decil']]
    ax.bar(xd, res_v4[p]['media_%mes'], yerr=res_v4[p]['ep_%mes'], color=cores_dec,
           edgecolor='white', capsize=3, error_kw=dict(lw=1.0, ecolor='#333333'), width=0.68,
           label='score_v4 (5 features, 50/50)' if p == 'TESTE' else None)
    ax.plot(xd, res_v2[p]['media_%mes'], color=COR['V2'], marker='o', mfc='none', mec=COR['V2'],
            ms=6, lw=1.3, ls=':', label='score_v2 (6 features) — comparação' if p == 'TESTE' else None)
    ax.plot(xd, res_v5[p]['media_%mes'], color=COR['V5'], marker='s', mfc='none', mec=COR['V5'],
            ms=5, lw=1.3, ls='--', label='score_v5 (ranks unificados, D40)' if p == 'TESTE' else None)
    ax.axhline(0, color='#333333', lw=1.0)
    ax.set_ylim(ymin, ymax)
    ax.set_xticks(xd)
    ax.set_xlabel(f'decil de score — {p} (1 = pior, 10 = melhor)')
    d0 = res_v4[p].loc[0]
    cy = 0.94 if p == 'TESTE' else 0.06
    va = 'top' if p == 'TESTE' else 'bottom'
    ax.text(0.97, cy, f'decil 1 (v4): {d0["media_%mes"]:+.3f}%/mês (t={d0["t_NW3"]:+.2f})\n'
                      f'decil 1 (v5): {res_v5[p].loc[0, "media_%mes"]:+.3f}%/mês (t={res_v5[p].loc[0, "t_NW3"]:+.2f})',
            transform=ax.transAxes, ha='right', va=va, fontsize=8.4, color=COR['V4'],
            bbox=dict(facecolor='white', edgecolor='none', alpha=0.75, pad=1.5))
axL.set_ylabel('alfa_fut médio realizado (%/mês), erro-padrão Newey-West(3)')
axL.legend(loc='upper left', fontsize=8.1)
fig.text(0.5, -0.03,
         'O achado que fundamenta a exclusão do decil inferior NÃO REPLICA entre partições: positivo no treino, '
         'negativo no teste. Trocar o score (v2 → v4 → v5) não muda essa propriedade. Com piso de preço ≥ R$0,50, '
         'o decil do TREINO fica NEGATIVO (D39: o positivo do treino era artefato de ações abaixo de R$0,50).',
         ha='center', fontsize=8.6, color='#1a1a1a')
salvar(fig, 'F8_assimetria_do_sinal.png')

# ==================================================================== F9 (com V5, geometria corrigida)
print('\n[F9] regenerada com V5 (3 paineis) e geometria corrigida (bottom=b, plano item 3)')
d1t = pd.read_parquet(DIR_INT + r'\d1_tabela_final.parquet')
ew_eleg50 = float(d1t.loc[d1t['serie'].str.contains('EW universo'), 'liq50_%aa'].iloc[0])
ew_pit0 = float(d1t.loc[d1t['serie'].str.contains('EW universo'), 'bruto_%aa'].iloc[0])
base00, base50 = anual(b_base['bruto']), anual(b_base['liq_50bps'])
casc = {}
for k in ['V3', 'V4', 'V5']:
    b0, b50 = anual(TE[k]['bruto']), anual(TE[k]['liq_50bps'])
    sel = b0 - base00; liqd = b50 - base50; giro = liqd - sel
    _, _, t_liq = nw_t((TE[k]['liq_50bps'].to_numpy() - b_base['liq_50bps'].to_numpy()), 3)
    casc[k] = dict(base=base50, sel=sel, giro=giro, final=b50, t=t_liq)
    print(f'  [{k}] base {base50:.4f} -> selecao {sel:+.4f} -> giro {giro:+.4f} -> final {b50:.4f} '
          f'(t NW3 vs base {t_liq:+.3f})')
fig, axes = plt.subplots(1, 3, figsize=(14.5, 5.2), sharey=True)
for ax, k in zip(axes, ['V3', 'V4', 'V5']):
    c = casc[k]
    steps = ['EW da\nbase', '+ seleção', '+ giro\n(custo)', f'{k}\n(líquido)']
    bottoms = [0, c['base'], c['base'] + c['sel'], 0]
    heights = [c['base'], c['sel'], c['giro'], c['final']]
    cores = ['#8c8c8c', '#1baf7a' if c['sel'] >= 0 else '#a4405e',
             '#1baf7a' if c['giro'] >= 0 else '#a4405e', COR[k]]
    for i, (b, h, col) in enumerate(zip(bottoms, heights, cores)):
        ax.bar(i, h, bottom=b, color=col, width=0.6, edgecolor='white')
        yv = b + h if h >= 0 else b
        ax.annotate(f'{h:+.2f}' if i in (1, 2) else f'{h:.2f}', (i, yv), textcoords='offset points',
                    xytext=(0, 4), ha='center', fontsize=8.5)
    ax.set_xticks(range(4)); ax.set_xticklabels(steps, fontsize=8.3)
    ax.axhline(ew_eleg50, color='#5a8a3c', ls=':', lw=1.1)
    ax.axhline(ew_pit0, color=COR['PIT'], ls='--', lw=1.3)
    ax.set_title(f'{k}: líquido {c["final"]:.2f}%aa (t NW3 vs base {c["t"]:+.2f})', fontsize=9.3)
axes[0].set_ylabel('%aa, TESTE, líquido de 50bps')
fig.text(0.5, 0.95, f'referências: EW elegível líq50 = {ew_eleg50:.4f}%aa (verde-pontilhado)  |  '
                    f'EW ponto-no-tempo custo-zero = {ew_pit0:.4f}%aa (tracejado)',
         ha='center', fontsize=8.3)
fig.text(0.5, -0.03,
         'O GIRO É CUSTO, não bônus: cascata ancorada no EW da própria base. Geometria corrigida em 2026-08-16 '
         '(plano item 3: a barra negativa ocupava [b+2h, b+h]; agora [b+h, b]). Nenhum t acima de 1,1.',
         ha='center', fontsize=8.3, color=COR['V1'])
salvar(fig, 'F9_decomposicao_v3_cascata.png')

# ==================================================================== F10 (com V5)
print('\n[F10] regenerada com V5 + convencao de giro declarada (plano item 7)')
ordem10 = ['V1', 'V2', 'V3', 'V4EW', 'V4', 'V5', 'PIT']
giro = {k: 100 * TE[k]['turnover'].mean() / 2 for k in ordem10}
custo = {k: anual(TE[k]['bruto']) - anual(TE[k]['liq_50bps']) for k in ordem10}
for k in ordem10:
    print(f'  {k:5s} giro one-way {giro[k]:7.4f} %/mes | custo pago (50bps) {custo[k]:7.4f} pp/ano')
fig, axes = plt.subplots(1, 2, figsize=(11.5, 5.0))
xs = np.arange(len(ordem10))
cores_bar = [COR[k] for k in ordem10]
rot10 = ['V1', 'V2', 'V3', 'V4-EW', 'V4', 'V5', 'EW ponto-no-tempo']
axes[0].bar(xs, [giro[k] for k in ordem10], color=cores_bar, edgecolor='white')
axes[0].set_xticks(xs); axes[0].set_xticklabels(rot10, rotation=25, ha='right', fontsize=8.3)
axes[0].set_ylabel('giro one-way médio (%/mês) — UMA perna')
axes[0].axhline(0, color='#666666', lw=0.8)
axes[1].bar(xs, [custo[k] for k in ordem10], color=cores_bar, edgecolor='white')
axes[1].set_xticks(xs); axes[1].set_xticklabels(rot10, rotation=25, ha='right', fontsize=8.3)
axes[1].set_ylabel('custo pago a 50bps (pp/ano, bruto − líquido)')
axes[1].axhline(0, color='#666666', lw=0.8)
fig.subplots_adjust(bottom=0.34)
fig.text(0.5, -0.02, 'CONVENÇÃO DE GIRO (plano item 7): o giro publicado é UMA perna; a coluna turnover dos '
                     'parquets é DUAS pernas (Σ|Δw|); o custo de 50 bps é cobrado sobre DUAS pernas — em V4, '
                     '1,75 pp/ano (não 0,82). EW ponto-no-tempo: giro 6,37%/mês medido, comparado sempre a '
                     'custo zero (convenção da manchete); o custo mostrado é o que PAGARIA a 50bps.',
         ha='center', fontsize=7.9, color=COR['V1'])
salvar(fig, 'F10_giro_e_custo.png')

# ==================================================================== F12 (com V5)
print('\n[F12] regenerada com V5 sobreposta')
vc4 = pd.read_parquet(DIR_INT + r'\v4_carteira.parquet')
vc5 = pd.read_parquet(DIR_INT + r'\v5_carteira.parquet')
t4c = vc4[vc4.particao == 'TESTE']; t5c = vc5[vc5.particao == 'TESTE']
max4 = t4c.groupby('mes')['peso_iv'].max().sort_index()
max5 = t5c.groupby('mes')['peso_iv'].max().sort_index()


def n_efetivo_medio(df, col):
    hhi = df.groupby('mes')[col].apply(lambda s: float((s.to_numpy() ** 2).sum()))
    return float((1.0 / hhi).mean())


n_ef = {'V4': n_efetivo_medio(t4c, 'peso_iv'), 'V4-EW': n_efetivo_medio(t4c, 'peso_ew'),
        'V5': n_efetivo_medio(t5c, 'peso_iv'), 'EW-PIT': float(
            eleg.groupby('mes').size().reindex(sorted(t4c['mes'].unique())).mean())}
print(f'  peso maximo TESTE: V4 {100 * max4.max():.2f}% | V5 {100 * max5.max():.2f}% | '
      f'mediana do maximo mensal V4 {100 * max4.median():.2f}% / V5 {100 * max5.median():.2f}%')
print(f'  n efetivo medio TESTE: V4 {n_ef["V4"]:.1f} | V5 {n_ef["V5"]:.1f} | V4-EW {n_ef["V4-EW"]:.1f} | '
      f'EW-PIT {n_ef["EW-PIT"]:.1f}')
fig = plt.figure(figsize=(9.5, 4.6))
gs = fig.add_gridspec(1, 4)
ax = fig.add_subplot(gs[0, :3]); ax2 = fig.add_subplot(gs[0, 3])
xs12 = pd.to_datetime([str(m) for m in max4.index])
ax.plot(xs12, max4.to_numpy() * 100, color=COR['V4'], linewidth=1.5, marker='o', markersize=3,
        label='peso máximo mensal — V4 (1/vol), TESTE')
ax.plot(xs12, max5.to_numpy() * 100, color=COR['V5'], linewidth=1.3, ls='--', marker='s',
        markersize=2.5, label='peso máximo mensal — V5 (1/vol), TESTE')
for nivel, cor in [(10, COR['cinza_med']), (25, COR['cinza_esc']), (50, COR['preto'])]:
    ax.axhline(nivel, color=cor, linestyle='--', linewidth=0.9)
    ax.text(xs12.max(), nivel, f' {nivel}%', va='bottom', ha='left', fontsize=8, color=cor)
ax.set_ylabel('peso máximo individual no mês (%)')
ax.set_ylim(bottom=0)
ax.legend(loc='upper left', fontsize=8.3)
nomes12 = ['V4\n(1/vol)', 'V5\n(1/vol)', 'V4-EW', 'EW\nponto-no-tempo']
vals12 = [n_ef['V4'], n_ef['V5'], n_ef['V4-EW'], n_ef['EW-PIT']]
cores12 = [COR['V4'], COR['V5'], COR['V4EW'], COR['cinza_med']]
ax2.bar(nomes12, vals12, color=cores12, edgecolor=COR['preto'], linewidth=0.6)
for i, v in enumerate(vals12):
    ax2.text(i, v, f'{v:.0f}', ha='center', va='bottom', fontsize=8.5)
ax2.set_ylabel('nº efetivo de nomes (1/HHI médio, TESTE)')
ax2.set_ylim(bottom=0)
ax2.tick_params(axis='x', labelsize=7.2)
fig.tight_layout()
salvar(fig, 'F12_concentracao_v4.png')

print('\n' + '=' * 126)
print('PARTE 4 CONCLUIDA: TABELA_FINAL.csv + F13/F14/F15 novas + F5/F6/F7/F8/F9/F10/F12 regeradas com V5.')
print('F11 NAO regenerada (a permutacao de 3 niveis tem figura propria, F15; F11 segue valida para V3/V4).')
print('=' * 126)


PARTE 4 — TABELA FINAL CONSOLIDADA + F13/F14/F15 + regeneracao F5-F10/F12 com V5

4.1 — TABELA FINAL CONSOLIDADA (gravada em outputs\TABELA_FINAL.csv):
  # TABELA FINAL CONSOLIDADA — TESTE: decisao 2018-01 a 2026-07, retornos 2018-02 a 2026-08 (103 meses)
  # custo: 50 bps por perna, cobrado sobre DUAS pernas (soma |dw|); giro_ow_%mes publicado em UMA perna (plano item 7)
  # EW point-in-time: comparador de manchete, SEMPRE custo zero (convencao declarada); EW elegivel = a MESMA serie liquida de 50 bps
  # t_NW3_exc_vs_EW_PIT: excedente (serie liq50, PIT bruto) — construcao MISTA da manchete, declarada (plano item 4);
  #   base contra base, V4: bruto x bruto +0,1357 t +0,879 | liq50 x liq50 +0,0636 t +0,416; V5: +0,1328 t +0,859 | +0,0606 t +0,395
  # Sharpe: [NAO ORDENAVEL] (plano item 6; Israelsen 2005) — com excesso negativo contra o CDI, mu/sigma cresce com sigma
  # ordem das linhas: arquitetural (V1..V5, depois comparadores) — NADA ordenado por Sharpe (S6)
  # permutacao em tres